# Implementation Framework — Reordered Execution Notebook

This notebook reorganizes the original blocks into the correct execution order for the **core IMDb methodology pipeline** and separates the **optional MongoDB validation blocks**.

**Note:** legacy references to `V25`/`mongo_payload_m1` were not included in the main execution flow because the corresponding generator block is not present in the provided notebook.

## Part A — Core methodology pipeline

Run the following cells in order. They create the conceptual scenario, compute the analytical matrix, infer activation classes, and build the experiment catalog and execution template.

### Notes from the original notebook

- `V0` to `V5`: structural logic kept.
- Blocks `1`, `2`, `3` were adapted for the real IMDb scenario via DuckDB.
- `V18` and later blocks handle sharedness and the final analytical matrix.
- The main benchmark-planning output is produced in `V21A`, `V21B`, `V22`, `V22B`, `V22C`, and `V22D`.

In [32]:
# =========================================================
# BLOCK 1 — REAL FIBEN BUNDLE VIA DUCKDB
# =========================================================
#
# Purpose:
# This block defines the main DuckDB bundle used by the FIBEN
# methodology adaptation.
#
# This block does not load the dataset yet.
# It only defines a reusable structure that will store:
# - dataset metadata;
# - the selected scale factor;
# - the DuckDB connection;
# - raw DuckDB views;
# - conceptual/semantic DuckDB views;
# - helper methods for inspection.
#
# The actual CSV registration happens in BLOCK 2.
# =========================================================

from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List
import pandas as pd


@dataclass
class FIBENDuckDBBundle:
    """
    Main DuckDB bundle for the real FIBEN scenario.

    This object stores the DuckDB connection and the logical mapping
    between dataset aliases and DuckDB views.

    The bundle separates two kinds of views:

    1. raw_views:
       Views directly registered from the materialized CSV files.

    2. semantic_views:
       Views derived from raw tables and aligned with the conceptual
       entities used by the methodology.
    """

    dataset_name: str
    scale_label: str
    sf_dir: Path
    con: Any
    raw_views: Dict[str, str] = field(default_factory=dict)
    semantic_views: Dict[str, str] = field(default_factory=dict)

    def view_names(self) -> List[str]:
        """
        Return all registered DuckDB view names.

        This method is useful for quickly checking which raw and
        semantic views are available in the current execution.
        """
        return sorted(
            list(self.raw_views.values()) +
            list(self.semantic_views.values())
        )

    def aliases(self) -> List[str]:
        """
        Return all logical aliases registered in the bundle.

        Aliases are simpler names used inside the notebook, while
        DuckDB views are the actual objects queried by SQL.
        """
        return sorted(
            list(self.raw_views.keys()) +
            list(self.semantic_views.keys())
        )

    def summary(self) -> pd.DataFrame:
        """
        Build a summary table for all registered views.

        For each view, the method tries to collect:
        - scope: raw or semantic;
        - alias;
        - DuckDB view name;
        - number of rows;
        - list of columns.

        If a view cannot be inspected, the error is stored in the
        columns field instead of stopping the notebook.
        """
        rows = []

        for scope, mapping in [
            ("raw", self.raw_views),
            ("semantic", self.semantic_views),
        ]:
            for alias, view_name in mapping.items():
                try:
                    desc_df = self.con.execute(f"DESCRIBE {view_name}").df()
                    columns = desc_df["column_name"].tolist()

                    n_rows = self.con.execute(
                        f"SELECT COUNT(*) AS n FROM {view_name}"
                    ).fetchone()[0]

                except Exception as e:
                    columns = [f"ERROR: {e}"]
                    n_rows = None

                rows.append({
                    "scope": scope,
                    "alias": alias,
                    "view_name": view_name,
                    "rows": n_rows,
                    "columns": columns,
                })

        if not rows:
            return pd.DataFrame(
                columns=["scope", "alias", "view_name", "rows", "columns"]
            )

        return (
            pd.DataFrame(rows)
            .sort_values(["scope", "alias"])
            .reset_index(drop=True)
        )

    def preview(self, alias: str, limit: int = 5) -> pd.DataFrame:
        """
        Return a small sample from a registered view.

        The method first searches in semantic_views.
        If the alias is not found there, it searches in raw_views.
        """
        if alias in self.semantic_views:
            view_name = self.semantic_views[alias]
        elif alias in self.raw_views:
            view_name = self.raw_views[alias]
        else:
            raise KeyError(
                f"Alias '{alias}' was not found. "
                f"Available aliases: {self.aliases()}"
            )

        return self.con.execute(
            f"SELECT * FROM {view_name} LIMIT {int(limit)}"
        ).df()

In [33]:
# =========================================================
# BLOCK 0B — REPRODUCIBILITY, README, AND EXPORT UTILITIES
# =========================================================
#
# Purpose:
# This block centralizes all reproducibility utilities used
# during the FIBEN methodology adaptation.
#
# It creates a stable output folder and provides helper
# functions to:
# - update the README directly from Jupyter;
# - save DataFrames as CSV files;
# - save dictionaries/lists as JSON files;
# - save execution notes block by block;
# - avoid duplicate README sections when a block is re-executed.
#
# Important:
# DuckDB connections cannot be saved as CSV.
# Therefore, this block saves the metadata needed to recreate
# the DuckDB bundle, such as paths, scale labels, view inventory,
# and summary tables.
# =========================================================

from pathlib import Path
from datetime import datetime
import json
import pandas as pd


# ---------------------------------------------------------
# 1. Define the main output folders
# ---------------------------------------------------------

FIBEN_PROJECT_OUTPUT_DIR = Path(
    "/home/jovyan/privado/framework evaluation approachs/framework with dataset fiben"
)

FIBEN_README_PATH = FIBEN_PROJECT_OUTPUT_DIR / "README_FIBEN_Implementation_Framework.md"

FIBEN_EXPORTS_DIR = FIBEN_PROJECT_OUTPUT_DIR / "exports"
FIBEN_VARIABLES_DIR = FIBEN_PROJECT_OUTPUT_DIR / "variables"
FIBEN_LOGS_DIR = FIBEN_PROJECT_OUTPUT_DIR / "logs"
FIBEN_DUCKDB_DIR = FIBEN_PROJECT_OUTPUT_DIR / "duckdb"

for folder in [
    FIBEN_PROJECT_OUTPUT_DIR,
    FIBEN_EXPORTS_DIR,
    FIBEN_VARIABLES_DIR,
    FIBEN_LOGS_DIR,
    FIBEN_DUCKDB_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------
# 2. JSON serialization helper
# ---------------------------------------------------------

def make_json_serializable(obj):
    """
    Convert common Python objects into JSON-serializable values.

    This is useful for saving metadata dictionaries that may contain:
    - Path objects;
    - pandas/numpy scalar values;
    - lists;
    - dictionaries.
    """
    if isinstance(obj, Path):
        return str(obj)

    if isinstance(obj, dict):
        return {str(k): make_json_serializable(v) for k, v in obj.items()}

    if isinstance(obj, list):
        return [make_json_serializable(v) for v in obj]

    if isinstance(obj, tuple):
        return [make_json_serializable(v) for v in obj]

    if hasattr(obj, "item"):
        try:
            return obj.item()
        except Exception:
            pass

    return obj


# ---------------------------------------------------------
# 3. Save DataFrame helper
# ---------------------------------------------------------

def save_dataframe(df: pd.DataFrame, relative_path: str, index: bool = False) -> Path:
    """
    Save a pandas DataFrame as CSV inside the FIBEN output folder.

    Parameters
    ----------
    df:
        DataFrame to save.

    relative_path:
        Relative path inside FIBEN_PROJECT_OUTPUT_DIR.
        Example: 'variables/block02/raw_inventory.csv'

    index:
        Whether to save the DataFrame index.

    Returns
    -------
    Path
        Full path of the saved CSV file.
    """
    output_path = FIBEN_PROJECT_OUTPUT_DIR / relative_path
    output_path.parent.mkdir(parents=True, exist_ok=True)

    df.to_csv(output_path, index=index)

    print(f"Saved DataFrame: {output_path}")
    return output_path


# ---------------------------------------------------------
# 4. Save JSON helper
# ---------------------------------------------------------

def save_json(obj, relative_path: str) -> Path:
    """
    Save a Python object as JSON inside the FIBEN output folder.

    Parameters
    ----------
    obj:
        Dictionary, list, or JSON-compatible object.

    relative_path:
        Relative path inside FIBEN_PROJECT_OUTPUT_DIR.
        Example: 'variables/block02/fiben_paths.json'

    Returns
    -------
    Path
        Full path of the saved JSON file.
    """
    output_path = FIBEN_PROJECT_OUTPUT_DIR / relative_path
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with output_path.open("w", encoding="utf-8") as f:
        json.dump(
            make_json_serializable(obj),
            f,
            indent=2,
            ensure_ascii=False,
        )

    print(f"Saved JSON: {output_path}")
    return output_path


# ---------------------------------------------------------
# 5. README initialization
# ---------------------------------------------------------

def initialize_fiben_readme() -> None:
    """
    Create the FIBEN README file if it does not exist yet.

    The README is intentionally updated block by block during
    notebook execution.
    """
    if FIBEN_README_PATH.exists():
        return

    initial_text = f"""# FIBEN Implementation Framework — Execution Guide

This README documents the adaptation of the methodology previously tested on IMDb to the FIBEN dataset.

The goal is to reuse the generic methodology pipeline as much as possible, while replacing only the dataset-specific layer:

1. dataset registration in DuckDB;
2. conceptual views;
3. conceptual schema;
4. workload definition;
5. observed cardinality and sharedness extraction;
6. MongoDB materialization logic;
7. concrete benchmark queries.

The methodology computes:

- selected root;
- Rc;
- D;
- Re;
- DeltaR;
- DeltaRratio;
- semantic relationship profile;
- update volatility;
- observed sharedness.

The activation logic uses the generic G0–G9 configuration families.

The benchmark compares:

- primary queries;
- secondary affected queries;
- control queries.

## Execution log

This README is updated directly from the Jupyter notebook.

Created at: {datetime.now().isoformat(timespec="seconds")}

"""
    FIBEN_README_PATH.write_text(initial_text, encoding="utf-8")
    print(f"Created README: {FIBEN_README_PATH}")


# ---------------------------------------------------------
# 6. README section updater
# ---------------------------------------------------------

def update_readme_section(section_title: str, section_body: str) -> None:
    """
    Insert or replace one README section.

    The function uses hidden markers to avoid duplicated sections
    when a notebook block is executed more than once.

    Parameters
    ----------
    section_title:
        Human-readable section title.

    section_body:
        Markdown content of the section.
    """
    initialize_fiben_readme()

    start_marker = f"<!-- START: {section_title} -->"
    end_marker = f"<!-- END: {section_title} -->"

    new_section = f"""
{start_marker}
## {section_title}

{section_body.strip()}

_Last updated: {datetime.now().isoformat(timespec="seconds")}_

{end_marker}
"""

    current_text = FIBEN_README_PATH.read_text(encoding="utf-8")

    if start_marker in current_text and end_marker in current_text:
        before = current_text.split(start_marker)[0]
        after = current_text.split(end_marker)[1]
        updated_text = before + new_section + after
    else:
        updated_text = current_text.rstrip() + "\n\n" + new_section + "\n"

    FIBEN_README_PATH.write_text(updated_text, encoding="utf-8")

    print(f"Updated README section: {section_title}")
    print(f"README path: {FIBEN_README_PATH}")


# ---------------------------------------------------------
# 7. Save a lightweight execution log
# ---------------------------------------------------------

def write_execution_log(block_name: str, notes: dict) -> Path:
    """
    Save execution notes for one notebook block.

    This helps recover the state of the experiment after a restart.
    """
    safe_block_name = (
        block_name.lower()
        .replace(" ", "_")
        .replace("—", "_")
        .replace("-", "_")
    )

    log_payload = {
        "block_name": block_name,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "notes": make_json_serializable(notes),
    }

    return save_json(
        log_payload,
        f"logs/{safe_block_name}_execution_log.json",
    )


# ---------------------------------------------------------
# 8. Initialize README immediately
# ---------------------------------------------------------

initialize_fiben_readme()

print("FIBEN reproducibility utilities are ready.")
print(f"Main output folder: {FIBEN_PROJECT_OUTPUT_DIR}")

FIBEN reproducibility utilities are ready.
Main output folder: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben


In [34]:
# =========================================================
# BLOCK 2 — REGISTER FIBEN SCALE-FACTOR CSVs IN DUCKDB
# =========================================================
#
# Purpose:
# This block registers the final materialized FIBEN CSV files
# as raw DuckDB views.
#
# This block replaces the IMDb-specific TSV registration logic.
#
# Important:
# - The original selected FIBEN files were headerless.
# - The FIBEN scale-factor preparation pipeline recovered the
#   positional schemas and produced final materialized CSV files
#   with explicit headers.
# - Therefore, this block reads the final materialized CSV files,
#   not the original raw headerless files.
#
# Output:
# - A persistent DuckDB database file;
# - one raw DuckDB view per CSV file;
# - a FIBENDuckDBBundle object;
# - CSV/JSON reproducibility exports;
# - an updated README section.
#
# Conceptual/semantic views are not created here.
# They will be created in BLOCK 3.
# =========================================================

import duckdb
import re
from pathlib import Path
from typing import Dict, Optional
import pandas as pd


# ---------------------------------------------------------
# 1. Define the available FIBEN scale-factor folders
# ---------------------------------------------------------
#
# Adjust these paths to your Jupyter environment.
#
# Expected structure:
#
# fiben_sf_artifacts/
#   sf1_materialized/
#     tables/
#   scaled_corp_rooted/
#     SF10/
#       tables/
#   scaled_corp_rooted/
#     SF30/
#       tables/
# ---------------------------------------------------------

FIBEN_SF_PATHS: Dict[str, str] = {
    "SF1": "/home/jovyan/privado/fiben/fiben_sf_artifacts/sf1_materialized/tables",
    "SF10": "/home/jovyan/privado/fiben/fiben_sf_artifacts/scaled_corp_rooted/SF10/tables",
    "SF30": "/home/jovyan/privado/fiben/fiben_sf_artifacts/scaled_corp_rooted/SF30/tables",
}


# ---------------------------------------------------------
# 2. Select the scale factor for the current notebook run
# ---------------------------------------------------------
#
# Start with SF1.
# Only move to SF10 and SF30 after the full methodology works
# correctly on SF1.
# ---------------------------------------------------------

FIBEN_SELECTED_SCALE = "SF1"


# ---------------------------------------------------------
# 3. Helper: sanitize file names into DuckDB-safe aliases
# ---------------------------------------------------------

def sanitize_identifier(name: str) -> str:
    """
    Convert a file name or table name into a safe DuckDB identifier.

    Examples
    --------
    'CORPORATION.csv'     -> 'corporation'
    'listed-security.csv' -> 'listed_security'
    '123_table.csv'       -> 't_123_table'
    """
    name = str(name).strip().lower()
    name = re.sub(r"[^a-zA-Z0-9_]+", "_", name)
    name = re.sub(r"_+", "_", name)
    name = name.strip("_")

    if not name:
        raise ValueError("The generated identifier is empty.")

    if name[0].isdigit():
        name = f"t_{name}"

    return name


# ---------------------------------------------------------
# 4. Helper: quote paths for SQL safely
# ---------------------------------------------------------

def sql_string(value: str) -> str:
    """
    Return a SQL-safe string literal.

    This avoids SQL errors when a file path contains single quotes.
    """
    return "'" + str(value).replace("'", "''") + "'"


# ---------------------------------------------------------
# 5. Inspect the CSV files available for one scale factor
# ---------------------------------------------------------

def list_fiben_csv_files(sf_dir: Path) -> pd.DataFrame:
    """
    List all CSV files available in the selected FIBEN scale-factor folder.

    This function does not load the full data into memory.
    It only records file metadata needed for reproducibility.
    """
    sf_dir = Path(sf_dir)

    if not sf_dir.exists():
        raise FileNotFoundError(f"Scale-factor folder not found: {sf_dir}")

    csv_files = sorted(sf_dir.glob("*.csv"))

    rows = []

    for path in csv_files:
        rows.append({
            "file_name": path.name,
            "file_stem": path.stem,
            "alias": sanitize_identifier(path.stem),
            "size_mb": round(path.stat().st_size / (1024 * 1024), 4),
            "path": str(path),
        })

    return pd.DataFrame(rows)


# ---------------------------------------------------------
# 6. Create one DuckDB view from one CSV file
# ---------------------------------------------------------

def create_fiben_csv_view(
    con,
    csv_path: Path,
    view_name: str,
    header: bool = True,
) -> None:
    """
    Register one FIBEN CSV file as a DuckDB view.

    The function uses DuckDB's read_csv_auto because the final
    materialized FIBEN files already contain explicit headers.

    Parameters
    ----------
    con:
        Active DuckDB connection.

    csv_path:
        Path to the CSV file.

    view_name:
        DuckDB view name to create.

    header:
        Whether the CSV file contains a header row.
        For the final materialized FIBEN files, this should be True.
    """
    csv_path = Path(csv_path)

    if not csv_path.exists():
        raise FileNotFoundError(f"CSV file not found: {csv_path}")

    path_sql = sql_string(str(csv_path))

    sql = f"""
    CREATE OR REPLACE VIEW {view_name} AS
    SELECT *
    FROM read_csv_auto(
        {path_sql},
        header={str(header).lower()},
        sample_size=20480,
        ignore_errors=false
    );
    """

    con.execute(sql)


# ---------------------------------------------------------
# 7. Open DuckDB and register all FIBEN raw CSV views
# ---------------------------------------------------------

def open_fiben_duckdb_for_scale(
    scale_label: str,
    sf_paths: Optional[Dict[str, str]] = None,
) -> FIBENDuckDBBundle:
    """
    Open a persistent DuckDB database and register the selected
    FIBEN scale-factor CSV files as raw views.

    This function creates only raw views.
    Conceptual/semantic views will be created in BLOCK 3.

    Parameters
    ----------
    scale_label:
        Scale factor label, for example: 'SF1', 'SF10', or 'SF30'.

    sf_paths:
        Optional dictionary mapping scale labels to table folders.
        If not provided, FIBEN_SF_PATHS is used.

    Returns
    -------
    FIBENDuckDBBundle
        Bundle containing metadata, DuckDB connection, and raw view names.
    """
    if sf_paths is None:
        sf_paths = FIBEN_SF_PATHS

    if scale_label not in sf_paths:
        raise KeyError(
            f"Unknown FIBEN scale label: {scale_label}. "
            f"Available labels: {sorted(sf_paths.keys())}"
        )

    sf_dir = Path(sf_paths[scale_label])

    if not sf_dir.exists():
        raise FileNotFoundError(
            f"FIBEN scale-factor folder not found: {sf_dir}\n"
            "Please update FIBEN_SF_PATHS with the correct local path."
        )

    csv_inventory_df = list_fiben_csv_files(sf_dir)

    if csv_inventory_df.empty:
        raise FileNotFoundError(
            f"No CSV files were found in the selected folder: {sf_dir}"
        )

    # Use a persistent DuckDB file instead of an in-memory database.
    # This helps preserve the registered views and metadata across
    # notebook restarts.
    duckdb_file = FIBEN_DUCKDB_DIR / f"fiben_{scale_label.lower()}.duckdb"
    con = duckdb.connect(database=str(duckdb_file))

    try:
        con.execute("PRAGMA disable_progress_bar")
    except Exception:
        pass

    raw_views = {}

    for _, row in csv_inventory_df.iterrows():
        alias = row["alias"]
        csv_path = Path(row["path"])
        view_name = f"fiben_raw_{alias}"

        create_fiben_csv_view(
            con=con,
            csv_path=csv_path,
            view_name=view_name,
            header=True,
        )

        raw_views[alias] = view_name

    bundle = FIBENDuckDBBundle(
        dataset_name="FIBEN",
        scale_label=scale_label,
        sf_dir=sf_dir,
        con=con,
        raw_views=raw_views,
        semantic_views={},
    )

    print(f"FIBEN scale factor loaded: {scale_label}")
    print(f"Folder: {sf_dir}")
    print(f"DuckDB file: {duckdb_file}")
    print(f"Raw CSV files registered: {len(raw_views)}")

    return bundle


# ---------------------------------------------------------
# 8. Create the FIBEN DuckDB bundle for the selected scale
# ---------------------------------------------------------

fiben_data_bundle = open_fiben_duckdb_for_scale(FIBEN_SELECTED_SCALE)


# ---------------------------------------------------------
# 9. Inspect registered raw views
# ---------------------------------------------------------

fiben_raw_inventory_df = list_fiben_csv_files(
    Path(FIBEN_SF_PATHS[FIBEN_SELECTED_SCALE])
)

block02_summary_df = fiben_data_bundle.summary()

display(fiben_raw_inventory_df)
display(block02_summary_df)


# ---------------------------------------------------------
# 10. Save BLOCK 2 outputs for reproducibility
# ---------------------------------------------------------
#
# These files allow the experiment to be inspected and resumed
# after a notebook restart.
# ---------------------------------------------------------

save_dataframe(
    fiben_raw_inventory_df,
    "variables/block02/fiben_raw_inventory.csv",
)

save_dataframe(
    block02_summary_df,
    "variables/block02/fiben_duckdb_bundle_summary.csv",
)

save_json(
    {
        "dataset_name": fiben_data_bundle.dataset_name,
        "selected_scale": fiben_data_bundle.scale_label,
        "scale_factor_folder": fiben_data_bundle.sf_dir,
        "registered_raw_views": fiben_data_bundle.raw_views,
        "registered_semantic_views": fiben_data_bundle.semantic_views,
        "available_aliases": fiben_data_bundle.aliases(),
        "available_view_names": fiben_data_bundle.view_names(),
        "fiben_sf_paths": FIBEN_SF_PATHS,
        "duckdb_file": str(FIBEN_DUCKDB_DIR / f"fiben_{FIBEN_SELECTED_SCALE.lower()}.duckdb"),
    },
    "variables/block02/fiben_duckdb_bundle_metadata.json",
)

write_execution_log(
    block_name="BLOCK 2 — REGISTER FIBEN SCALE-FACTOR CSVs IN DUCKDB",
    notes={
        "status": "completed",
        "selected_scale": FIBEN_SELECTED_SCALE,
        "raw_csv_files_registered": len(fiben_data_bundle.raw_views),
        "raw_inventory_csv": "variables/block02/fiben_raw_inventory.csv",
        "bundle_summary_csv": "variables/block02/fiben_duckdb_bundle_summary.csv",
        "bundle_metadata_json": "variables/block02/fiben_duckdb_bundle_metadata.json",
        "duckdb_file": str(FIBEN_DUCKDB_DIR / f"fiben_{FIBEN_SELECTED_SCALE.lower()}.duckdb"),
    },
)

block02_readme_body = f"""
This block registers the final materialized FIBEN CSV files as raw DuckDB views.

Selected scale factor:

    {FIBEN_SELECTED_SCALE}

Selected folder:

    {Path(FIBEN_SF_PATHS[FIBEN_SELECTED_SCALE])}

Persistent DuckDB file:

    {FIBEN_DUCKDB_DIR / f"fiben_{FIBEN_SELECTED_SCALE.lower()}.duckdb"}

The block creates one raw DuckDB view per CSV file using the prefix:

    fiben_raw_

Number of registered raw CSV files:

    {len(fiben_data_bundle.raw_views)}

Generated reproducibility files:

    variables/block02/fiben_raw_inventory.csv
    variables/block02/fiben_duckdb_bundle_summary.csv
    variables/block02/fiben_duckdb_bundle_metadata.json
    logs/block_2___register_fiben_scale_factor_csvs_in_duckdb_execution_log.json

Important note:

This block only registers raw CSV tables. It does not create conceptual or semantic views yet.
The conceptual FIBEN views are created in BLOCK 3.
"""

update_readme_section(
    section_title="BLOCK 2 — Register FIBEN Scale-Factor CSVs in DuckDB",
    section_body=block02_readme_body,
)

FIBEN scale factor loaded: SF1
Folder: /home/jovyan/privado/fiben/fiben_sf_artifacts/sf1_materialized/tables
DuckDB file: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/duckdb/fiben_sf1.duckdb
Raw CSV files registered: 14


,file_name,file_stem,alias,size_mb,path
0,CORPORATION.csv,CORPORATION,corporation,0.0883,/home/jovyan/privado/fiben/fiben_sf_artifacts/...
1,COUNTRY.csv,COUNTRY,country,0.0048,/home/jovyan/privado/fiben/fiben_sf_artifacts/...
2,DISCLOSURE.csv,DISCLOSURE,disclosure,31.5834,/home/jovyan/privado/fiben/fiben_sf_artifacts/...
3,ELEMENTOFFINANCIALSTATEMENT.csv,ELEMENTOFFINANCIALSTATEMENT,elementoffinancialstatement,65.4888,/home/jovyan/privado/fiben/fiben_sf_artifacts/...
4,ELEMENTSOFFINANCIALREPORT.csv,ELEMENTSOFFINANCIALREPORT,elementsoffinancialreport,130.4384,/home/jovyan/privado/fiben/fiben_sf_artifacts/...
5,FINANCIALREPORT.csv,FINANCIALREPORT,financialreport,1.5662,/home/jovyan/privado/fiben/fiben_sf_artifacts/...
6,FINANCIALSERVICEACCOUNT.csv,FINANCIALSERVICEACCOUNT,financialserviceaccount,7.0501,/home/jovyan/privado/fiben/fiben_sf_artifacts/...
7,HOLDING.csv,HOLDING,holding,27.1699,/home/jovyan/privado/fiben/fiben_sf_artifacts/...
8,INDUSTRYSECTORCLASSIFIER.csv,INDUSTRYSECTORCLASSIFIER,industrysectorclassifier,0.0315,/home/jovyan/privado/fiben/fiben_sf_artifacts/...
9,LISTEDSECURITY.csv,LISTEDSECURITY,listedsecurity,0.1748,/home/jovyan/privado/fiben/fiben_sf_artifacts/...


,scope,alias,view_name,rows,columns
0,raw,corporation,fiben_raw_corporation,2324,"[CORPORATIONID, ISCLASSIFIEDBY, ISDOMICILEDIN,..."
1,raw,country,fiben_raw_country,249,"[COUNTRYID, HASNAME, HASNUMERICCOUNTRYCODE]"
2,raw,disclosure,fiben_raw_disclosure,208158,"[DISCLOSUREID, HASMETRICYEARCALENDAR, HASMETRI..."
3,raw,elementoffinancialstatement,fiben_raw_elementoffinancialstatement,443794,"[ELEMENTOFFINANCIALSTATEMENTID, HASMETRICYEARC..."
4,raw,elementsoffinancialreport,fiben_raw_elementsoffinancialreport,8120084,"[ELEMENTSOFFINANCIALREPORTID, ISMEMBEROF]"
5,raw,financialreport,fiben_raw_financialreport,48301,"[FINANCIALREPORTID, ISPROVIDEDBY, HASUNIQUEIDE..."
6,raw,financialserviceaccount,fiben_raw_financialserviceaccount,97270,"[FINANCIALSERVICEACCOUNTID, ISOWNEDBY, ISMANAG..."
7,raw,holding,fiben_raw_holding,534473,"[HOLDINGID, ISHELDBY, REFERSTO, HASDESCRIPTION..."
8,raw,industrysectorclassifier,fiben_raw_industrysectorclassifier,452,"[INDUSTRYSECTORCLASSIFIERID, ISDEFINEDBY, HASU..."
9,raw,listedsecurity,fiben_raw_listedsecurity,2745,"[LISTEDSECURITYID, HASLASTTRADEDVALUE, HASLIST..."


Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block02/fiben_raw_inventory.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block02/fiben_duckdb_bundle_summary.csv
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block02/fiben_duckdb_bundle_metadata.json
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/logs/block_2___register_fiben_scale_factor_csvs_in_duckdb_execution_log.json
Updated README section: BLOCK 2 — Register FIBEN Scale-Factor CSVs in DuckDB
README path: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/README_FIBEN_Implementation_Framework.md


In [27]:
fiben_data_bundle.preview("corporation", limit=5)

,CORPORATIONID,ISCLASSIFIEDBY,ISDOMICILEDIN,HASTICKERSYMBOL,HASLEGALNAME
0,1000,7372,237,ACCL,Accelrys
1,1001,6331,237,AIG,American International Group Inc
2,1002,6021,237,FMBI,First Midwest Bancorp
3,1003,4412,237,BALT,Baltic Trading Limited
4,1004,7359,237,ELRC,Electro Rent


In [35]:
# =========================================================
# BLOCK 3 — DERIVED CONCEPTUAL VIEWS FOR FIBEN
# =========================================================
#
# Purpose:
# This block creates the first conceptual/semantic view layer
# for the FIBEN dataset.
#
# The previous block registered the final materialized CSV files
# as raw DuckDB views using the prefix:
#
#     fiben_raw_
#
# This block maps those raw views to canonical conceptual entity
# names used by the methodology.
#
# Important:
# This first version creates pass-through conceptual views:
#
#     CREATE VIEW fiben_corporations AS SELECT * FROM fiben_raw_...
#
# The goal is to stabilize the conceptual layer before defining
# the workload and relationships in the next block.
#
# Later, if needed, these views can be refined to rename columns,
# cast types, or derive calculated fields.
# =========================================================

import re
from pathlib import Path
from typing import Dict, List, Optional
import pandas as pd


# ---------------------------------------------------------
# 1. Helper: normalize names for matching
# ---------------------------------------------------------

def normalize_name_for_matching(value: str) -> str:
    """
    Normalize a table, alias, or entity name for robust matching.

    Examples
    --------
    'Listed Security'   -> 'listedsecurity'
    'listed_security'   -> 'listedsecurity'
    'LISTED-SECURITY'   -> 'listedsecurity'
    """
    return re.sub(r"[^a-zA-Z0-9]+", "", str(value).lower())


# ---------------------------------------------------------
# 2. Helper: inspect columns from one DuckDB view
# ---------------------------------------------------------

def describe_duckdb_view(con, view_name: str) -> pd.DataFrame:
    """
    Return DuckDB column metadata for one view.

    The output includes:
    - column name;
    - column type;
    - nullability;
    - default value;
    - primary key flag, when available.
    """
    desc_df = con.execute(f"DESCRIBE {view_name}").df()
    desc_df.insert(0, "view_name", view_name)
    return desc_df


# ---------------------------------------------------------
# 3. Build a raw column dictionary
# ---------------------------------------------------------

def build_raw_column_dictionary(bundle: FIBENDuckDBBundle) -> pd.DataFrame:
    """
    Build a column dictionary for all raw FIBEN views.

    This is useful because the next methodology blocks depend on
    entity names, relationship columns, identifiers, and filters.
    """
    frames = []

    for raw_alias, raw_view in sorted(bundle.raw_views.items()):
        desc_df = describe_duckdb_view(bundle.con, raw_view)
        desc_df.insert(0, "raw_alias", raw_alias)
        frames.append(desc_df)

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)


fiben_raw_column_dictionary_df = build_raw_column_dictionary(fiben_data_bundle)

display(fiben_raw_column_dictionary_df)


# ---------------------------------------------------------
# 4. Define candidate mappings from conceptual entities
#    to physical raw table aliases
# ---------------------------------------------------------
#
# The keys are canonical conceptual entities.
# The values are possible raw aliases generated from CSV file names.
#
# This dictionary is intentionally flexible because the exact
# physical table names may vary depending on the FIBEN extraction
# or materialization process.
# ---------------------------------------------------------

FIBEN_CONCEPTUAL_ENTITY_CANDIDATES: Dict[str, List[str]] = {
    "corporations": [
        "corporation",
        "corporations",
        "corp",
        "company",
        "companies",
    ],
    "reports": [
        "report",
        "reports",
        "financial_report",
        "financial_reports",
        "filing",
        "filings",
    ],
    "report_elements": [
        "report_element",
        "report_elements",
        "reportelement",
        "reportelements",
        "report_el",
        "report_els",
    ],
    "statement_elements": [
        "statement_element",
        "statement_elements",
        "statementelement",
        "statementelements",
        "statement_el",
        "statement_els",
    ],
    "disclosures": [
        "disclosure",
        "disclosures",
    ],
    "securities": [
        "security",
        "securities",
    ],
    "listed_securities": [
        "listed_security",
        "listed_securities",
        "listedsecurity",
        "listedsecurities",
    ],
    "accounts": [
        "account",
        "accounts",
    ],
    "industries": [
        "industry",
        "industries",
        "industry_classification",
        "industry_classifications",
    ],
    "classifications": [
        "classification",
        "classifications",
        "corporation_classification",
        "corporation_classifications",
        "isclassifiedby",
    ],
}


# ---------------------------------------------------------
# 5. Resolve raw aliases for conceptual entities
# ---------------------------------------------------------

def resolve_raw_alias_for_entity(
    canonical_entity: str,
    candidate_aliases: List[str],
    available_raw_aliases: List[str],
) -> Optional[str]:
    """
    Resolve the best raw alias for one conceptual entity.

    Matching strategy:
    1. exact match after normalization;
    2. exact match against the canonical entity name;
    3. no fuzzy substring match by default, to avoid wrong matches
       such as mapping 'securities' to 'listed_securities'.
    """
    normalized_available = {
        normalize_name_for_matching(alias): alias
        for alias in available_raw_aliases
    }

    # Try candidate aliases first.
    for candidate in candidate_aliases:
        normalized_candidate = normalize_name_for_matching(candidate)
        if normalized_candidate in normalized_available:
            return normalized_available[normalized_candidate]

    # Try canonical entity itself.
    normalized_canonical = normalize_name_for_matching(canonical_entity)
    if normalized_canonical in normalized_available:
        return normalized_available[normalized_canonical]

    return None


def build_conceptual_view_mapping(
    bundle: FIBENDuckDBBundle,
    conceptual_candidates: Dict[str, List[str]],
) -> pd.DataFrame:
    """
    Build a mapping between conceptual entities and raw DuckDB views.

    The result explicitly marks each conceptual entity as:
    - matched;
    - unmatched.
    """
    available_raw_aliases = sorted(bundle.raw_views.keys())
    rows = []

    for canonical_entity, candidates in conceptual_candidates.items():
        matched_alias = resolve_raw_alias_for_entity(
            canonical_entity=canonical_entity,
            candidate_aliases=candidates,
            available_raw_aliases=available_raw_aliases,
        )

        if matched_alias is None:
            rows.append({
                "canonical_entity": canonical_entity,
                "status": "unmatched",
                "raw_alias": None,
                "raw_view": None,
                "semantic_view": f"fiben_{canonical_entity}",
                "candidate_aliases": candidates,
            })
        else:
            rows.append({
                "canonical_entity": canonical_entity,
                "status": "matched",
                "raw_alias": matched_alias,
                "raw_view": bundle.raw_views[matched_alias],
                "semantic_view": f"fiben_{canonical_entity}",
                "candidate_aliases": candidates,
            })

    return pd.DataFrame(rows)


fiben_conceptual_view_mapping_df = build_conceptual_view_mapping(
    bundle=fiben_data_bundle,
    conceptual_candidates=FIBEN_CONCEPTUAL_ENTITY_CANDIDATES,
)

display(fiben_conceptual_view_mapping_df)


# ---------------------------------------------------------
# 6. Create conceptual/semantic views
# ---------------------------------------------------------

def create_fiben_conceptual_views(
    bundle: FIBENDuckDBBundle,
    mapping_df: pd.DataFrame,
) -> Dict[str, str]:
    """
    Create conceptual FIBEN views from matched raw views.

    Each matched entity becomes a semantic view:

        fiben_<canonical_entity>

    The view is initially a pass-through view:

        SELECT * FROM raw_view

    Returns
    -------
    Dict[str, str]
        Mapping from canonical entity name to semantic view name.
    """
    semantic_views = {}

    matched_df = mapping_df[mapping_df["status"] == "matched"].copy()

    for _, row in matched_df.iterrows():
        canonical_entity = row["canonical_entity"]
        raw_view = row["raw_view"]
        semantic_view = row["semantic_view"]

        sql = f"""
        CREATE OR REPLACE VIEW {semantic_view} AS
        SELECT *
        FROM {raw_view};
        """

        bundle.con.execute(sql)
        semantic_views[canonical_entity] = semantic_view

    return semantic_views


created_semantic_views = create_fiben_conceptual_views(
    bundle=fiben_data_bundle,
    mapping_df=fiben_conceptual_view_mapping_df,
)

# Update the bundle with the semantic views created in this block.
fiben_data_bundle.semantic_views.update(created_semantic_views)


# ---------------------------------------------------------
# 7. Inspect semantic views created in this block
# ---------------------------------------------------------

fiben_semantic_views_summary_df = fiben_data_bundle.summary()

display(fiben_semantic_views_summary_df)

matched_entities = (
    fiben_conceptual_view_mapping_df
    .query("status == 'matched'")["canonical_entity"]
    .tolist()
)

unmatched_entities = (
    fiben_conceptual_view_mapping_df
    .query("status == 'unmatched'")["canonical_entity"]
    .tolist()
)

print("Matched conceptual entities:")
print(matched_entities)

print("\nUnmatched conceptual entities:")
print(unmatched_entities)


# ---------------------------------------------------------
# 8. Save BLOCK 3 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    fiben_raw_column_dictionary_df,
    "variables/block03/fiben_raw_column_dictionary.csv",
)

save_dataframe(
    fiben_conceptual_view_mapping_df,
    "variables/block03/fiben_conceptual_view_mapping.csv",
)

save_dataframe(
    fiben_semantic_views_summary_df,
    "variables/block03/fiben_semantic_views_summary.csv",
)

save_json(
    {
        "dataset_name": fiben_data_bundle.dataset_name,
        "selected_scale": fiben_data_bundle.scale_label,
        "matched_entities": matched_entities,
        "unmatched_entities": unmatched_entities,
        "created_semantic_views": created_semantic_views,
        "semantic_views_in_bundle": fiben_data_bundle.semantic_views,
        "raw_views_in_bundle": fiben_data_bundle.raw_views,
        "conceptual_entity_candidates": FIBEN_CONCEPTUAL_ENTITY_CANDIDATES,
    },
    "variables/block03/fiben_semantic_metadata.json",
)

write_execution_log(
    block_name="BLOCK 3 — DERIVED CONCEPTUAL VIEWS FOR FIBEN",
    notes={
        "status": "completed",
        "selected_scale": fiben_data_bundle.scale_label,
        "matched_entities": matched_entities,
        "unmatched_entities": unmatched_entities,
        "n_created_semantic_views": len(created_semantic_views),
        "raw_column_dictionary_csv": "variables/block03/fiben_raw_column_dictionary.csv",
        "conceptual_mapping_csv": "variables/block03/fiben_conceptual_view_mapping.csv",
        "semantic_summary_csv": "variables/block03/fiben_semantic_views_summary.csv",
        "semantic_metadata_json": "variables/block03/fiben_semantic_metadata.json",
    },
)


# ---------------------------------------------------------
# 9. Update README section for BLOCK 3
# ---------------------------------------------------------

block03_readme_body = f"""
This block creates the first conceptual/semantic view layer for the FIBEN dataset.

The previous block registered final materialized CSV files as raw DuckDB views.
This block maps those raw views to canonical conceptual entity names used by the methodology.

Matched conceptual entities:

    {matched_entities}

Unmatched conceptual entities:

    {unmatched_entities}

Created semantic views:

    {created_semantic_views}

Generated reproducibility files:

    variables/block03/fiben_raw_column_dictionary.csv
    variables/block03/fiben_conceptual_view_mapping.csv
    variables/block03/fiben_semantic_views_summary.csv
    variables/block03/fiben_semantic_metadata.json

Important note:

The semantic views created in this block are pass-through views.
They currently use SELECT * FROM the matched raw views.

This is intentional. The goal is to stabilize the conceptual layer before defining:
- conceptual relationships;
- query workload;
- selected root per query;
- update targets;
- observed cardinality;
- observed sharedness.

Column renaming and type normalization can be added later if required by the workload or benchmark runner.
"""

update_readme_section(
    section_title="BLOCK 3 — Derived Conceptual Views for FIBEN",
    section_body=block03_readme_body,
)

,raw_alias,view_name,column_name,column_type,null,key,default,extra
0,corporation,fiben_raw_corporation,CORPORATIONID,BIGINT,YES,None,None,None
1,corporation,fiben_raw_corporation,ISCLASSIFIEDBY,BIGINT,YES,None,None,None
2,corporation,fiben_raw_corporation,ISDOMICILEDIN,BIGINT,YES,None,None,None
3,corporation,fiben_raw_corporation,HASTICKERSYMBOL,VARCHAR,YES,None,None,None
4,corporation,fiben_raw_corporation,HASLEGALNAME,VARCHAR,YES,None,None,None
...,...,...,...,...,...,...,...,...
81,securitiestransaction,fiben_raw_securitiestransaction,HASSETTLEMENTDATE,VARCHAR,YES,None,None,None
82,securitiestransaction,fiben_raw_securitiestransaction,HASCOUNT,DOUBLE,YES,None,None,None
83,security,fiben_raw_security,SECURITYID,BIGINT,YES,None,None,None
84,security,fiben_raw_security,ISTRADEDON,BIGINT,YES,None,None,None


,canonical_entity,status,raw_alias,raw_view,semantic_view,candidate_aliases
0,corporations,matched,corporation,fiben_raw_corporation,fiben_corporations,"[corporation, corporations, corp, company, com..."
1,reports,matched,financialreport,fiben_raw_financialreport,fiben_reports,"[report, reports, financial_report, financial_..."
2,report_elements,unmatched,None,None,fiben_report_elements,"[report_element, report_elements, reportelemen..."
3,statement_elements,unmatched,None,None,fiben_statement_elements,"[statement_element, statement_elements, statem..."
4,disclosures,matched,disclosure,fiben_raw_disclosure,fiben_disclosures,"[disclosure, disclosures]"
5,securities,matched,security,fiben_raw_security,fiben_securities,"[security, securities]"
6,listed_securities,matched,listedsecurity,fiben_raw_listedsecurity,fiben_listed_securities,"[listed_security, listed_securities, listedsec..."
7,accounts,unmatched,None,None,fiben_accounts,"[account, accounts]"
8,industries,unmatched,None,None,fiben_industries,"[industry, industries, industry_classification..."
9,classifications,unmatched,None,None,fiben_classifications,"[classification, classifications, corporation_..."


,scope,alias,view_name,rows,columns
0,raw,corporation,fiben_raw_corporation,2324,"[CORPORATIONID, ISCLASSIFIEDBY, ISDOMICILEDIN,..."
1,raw,country,fiben_raw_country,249,"[COUNTRYID, HASNAME, HASNUMERICCOUNTRYCODE]"
2,raw,disclosure,fiben_raw_disclosure,208158,"[DISCLOSUREID, HASMETRICYEARCALENDAR, HASMETRI..."
3,raw,elementoffinancialstatement,fiben_raw_elementoffinancialstatement,443794,"[ELEMENTOFFINANCIALSTATEMENTID, HASMETRICYEARC..."
4,raw,elementsoffinancialreport,fiben_raw_elementsoffinancialreport,8120084,"[ELEMENTSOFFINANCIALREPORTID, ISMEMBEROF]"
5,raw,financialreport,fiben_raw_financialreport,48301,"[FINANCIALREPORTID, ISPROVIDEDBY, HASUNIQUEIDE..."
6,raw,financialserviceaccount,fiben_raw_financialserviceaccount,97270,"[FINANCIALSERVICEACCOUNTID, ISOWNEDBY, ISMANAG..."
7,raw,holding,fiben_raw_holding,534473,"[HOLDINGID, ISHELDBY, REFERSTO, HASDESCRIPTION..."
8,raw,industrysectorclassifier,fiben_raw_industrysectorclassifier,452,"[INDUSTRYSECTORCLASSIFIERID, ISDEFINEDBY, HASU..."
9,raw,listedsecurity,fiben_raw_listedsecurity,2745,"[LISTEDSECURITYID, HASLASTTRADEDVALUE, HASLIST..."


Matched conceptual entities:
['corporations', 'reports', 'disclosures', 'securities', 'listed_securities']

Unmatched conceptual entities:
['report_elements', 'statement_elements', 'accounts', 'industries', 'classifications']
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block03/fiben_raw_column_dictionary.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block03/fiben_conceptual_view_mapping.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block03/fiben_semantic_views_summary.csv
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block03/fiben_semantic_metadata.json
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/logs/block_3___derived_conceptual_views_for_fiben_execution_log.json
Updated README section: B

In [29]:
fiben_conceptual_view_mapping_df

,canonical_entity,status,raw_alias,raw_view,semantic_view,candidate_aliases
0,corporations,matched,corporation,fiben_raw_corporation,fiben_corporations,"[corporation, corporations, corp, company, com..."
1,reports,matched,financialreport,fiben_raw_financialreport,fiben_reports,"[report, reports, financial_report, financial_..."
2,report_elements,unmatched,None,None,fiben_report_elements,"[report_element, report_elements, reportelemen..."
3,statement_elements,unmatched,None,None,fiben_statement_elements,"[statement_element, statement_elements, statem..."
4,disclosures,matched,disclosure,fiben_raw_disclosure,fiben_disclosures,"[disclosure, disclosures]"
5,securities,matched,security,fiben_raw_security,fiben_securities,"[security, securities]"
6,listed_securities,matched,listedsecurity,fiben_raw_listedsecurity,fiben_listed_securities,"[listed_security, listed_securities, listedsec..."
7,accounts,unmatched,None,None,fiben_accounts,"[account, accounts]"
8,industries,unmatched,None,None,fiben_industries,"[industry, industries, industry_classification..."
9,classifications,unmatched,None,None,fiben_classifications,"[classification, classifications, corporation_..."


In [30]:
fiben_data_bundle.preview("corporations", limit=5)

,CORPORATIONID,ISCLASSIFIEDBY,ISDOMICILEDIN,HASTICKERSYMBOL,HASLEGALNAME
0,1000,7372,237,ACCL,Accelrys
1,1001,6331,237,AIG,American International Group Inc
2,1002,6021,237,FMBI,First Midwest Bancorp
3,1003,4412,237,BALT,Baltic Trading Limited
4,1004,7359,237,ELRC,Electro Rent


In [37]:
# =========================================================
# BLOCK 0A — CORE METHODOLOGY DATA STRUCTURES
# =========================================================
#
# Purpose:
# This block defines the core data structures used by the
# methodology notebook.
#
# These classes are dataset-independent.
# They can be reused for IMDb, FIBEN, or any future dataset.
#
# The FIBEN scenario block depends on these classes to define:
# - the conceptual scenario;
# - the recommendation fragment;
# - the physical materialization plan;
# - the benchmark workload specification.
#
# Important:
# This block does not load data.
# This block does not create DuckDB views.
# It only defines reusable Python dataclasses.
# =========================================================

from dataclasses import dataclass, field
from typing import Dict, List, Any, Optional
from pathlib import Path
import pandas as pd


# ---------------------------------------------------------
# 1. Conceptual scenario specification
# ---------------------------------------------------------

@dataclass
class ScenarioSpec:
    """
    Dataset-independent conceptual scenario.

    A scenario contains:
    - a name;
    - a textual description;
    - a list of conceptual entities;
    - a list of conceptual relationships;
    - a list of workload queries.

    The methodology blocks use this object to compute:
    - Rc;
    - root candidates;
    - selected root;
    - D;
    - Re;
    - DeltaR;
    - DeltaRratio;
    - semantic relationship profile;
    - update volatility;
    - observed sharedness.
    """

    name: str
    description: str
    entities: List[str]
    relationships: List[Dict[str, Any]]
    workload_queries: List[Dict[str, Any]]


# ---------------------------------------------------------
# 2. Recommendation fragment specification
# ---------------------------------------------------------

@dataclass
class FragmentSpec:
    """
    Logical fragment recommended by a model-selection method.

    In this notebook, a fragment is used to describe which conceptual
    entities are grouped together before generating candidate document
    configurations.
    """

    name: str
    entities: List[str]
    model: str
    notes: str = ""


@dataclass
class RecommendationSpec:
    """
    Dataset-independent recommendation specification.

    This stores the output of a conceptual/model-selection step.
    """

    recommendation_name: str
    source_method: str
    source_dataset: str
    fragments: List[FragmentSpec] = field(default_factory=list)


# ---------------------------------------------------------
# 3. Physical database specification
# ---------------------------------------------------------

@dataclass
class PhysicalDBSpec:
    """
    Physical database target used by the materialization plan.

    For this project, the main target is MongoDB.
    """

    model: str
    engine: str
    host: str
    port: int
    database_name: str
    username: Optional[str] = None
    password: Optional[str] = None


@dataclass
class MaterializationPlan:
    """
    Physical materialization plan.

    It maps each conceptual fragment to a concrete physical database
    specification.
    """

    scenario_name: str
    recommendation_name: str
    fragment_to_db: Dict[str, PhysicalDBSpec]


# ---------------------------------------------------------
# 4. Benchmark workload specification
# ---------------------------------------------------------

@dataclass
class BenchmarkQuery:
    """
    One benchmark query specification.

    This object stores the conceptual query metadata before the
    concrete query is implemented in the execution runner.
    """

    name: str
    query_type: str
    entities_involved: List[str]
    abstract_query: str
    expected_fragment: str
    expected_db_model: str


@dataclass
class WorkloadSpec:
    """
    Complete benchmark workload specification.

    The workload groups the benchmark queries and stores the default
    number of repetitions.
    """

    name: str
    description: str
    queries: List[BenchmarkQuery]
    repetitions: int = 10


# ---------------------------------------------------------
# 5. Optional reproducibility export
# ---------------------------------------------------------
#
# If the reproducibility utilities from BLOCK 0B are already loaded,
# this block records the data-structure definitions in the README
# and in a metadata JSON file.
# ---------------------------------------------------------

core_methodology_structures = {
    "defined_classes": [
        "ScenarioSpec",
        "FragmentSpec",
        "RecommendationSpec",
        "PhysicalDBSpec",
        "MaterializationPlan",
        "BenchmarkQuery",
        "WorkloadSpec",
    ],
    "purpose": (
        "Dataset-independent dataclasses used by the methodology notebook "
        "to define conceptual scenarios, recommendation fragments, physical "
        "materialization plans, and benchmark workloads."
    ),
}

if "save_json" in globals():
    save_json(
        core_methodology_structures,
        "variables/block00a/core_methodology_data_structures.json",
    )

if "write_execution_log" in globals():
    write_execution_log(
        block_name="BLOCK 0A — CORE METHODOLOGY DATA STRUCTURES",
        notes={
            "status": "completed",
            "defined_classes": core_methodology_structures["defined_classes"],
            "metadata_json": "variables/block00a/core_methodology_data_structures.json",
        },
    )

if "update_readme_section" in globals():
    block00a_readme_body = """
This block defines the dataset-independent Python dataclasses used by the methodology notebook.

Defined classes:

    ScenarioSpec
    FragmentSpec
    RecommendationSpec
    PhysicalDBSpec
    MaterializationPlan
    BenchmarkQuery
    WorkloadSpec

These structures are reused by the FIBEN scenario block to define:

    conceptual entities;
    conceptual relationships;
    workload queries;
    recommendation fragments;
    physical materialization plan;
    benchmark workload metadata.

This block does not load data and does not create DuckDB views.
It only defines reusable methodology structures.
"""

    update_readme_section(
        section_title="BLOCK 0A — Core Methodology Data Structures",
        section_body=block00a_readme_body,
    )

print("Core methodology data structures are ready.")

Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block00a/core_methodology_data_structures.json
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/logs/block_0a___core_methodology_data_structures_execution_log.json
Updated README section: BLOCK 0A — Core Methodology Data Structures
README path: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/README_FIBEN_Implementation_Framework.md
Core methodology data structures are ready.


In [39]:
# =========================================================
# INSTANCE 1 — REAL FIBEN SCENARIO
# =========================================================
#
# Purpose:
# This block defines the conceptual scenario and workload used
# by the methodology for the FIBEN dataset.
#
# This block replaces the previous IMDb scenario.
#
# Important design decision:
# In the user-facing workload, the term "company" is represented
# in the conceptual model as "Corporation", because the FIBEN
# scale-factor generation process is corporation-rooted.
#
# The workload contains the 10 concrete FIBEN queries used to
# define the conceptual schema.
#
# Compatibility note:
# Some downstream generic blocks still refer to variables named
# imdb_scenario and imdb_workload_mongo.
# To avoid rewriting all generic blocks now, this block creates
# compatibility aliases:
#
#     imdb_scenario = fiben_scenario
#     imdb_workload_mongo = fiben_workload_mongo
#
# The actual scenario is FIBEN.
# =========================================================

import pandas as pd


# ---------------------------------------------------------
# 1. Required objects
# ---------------------------------------------------------

required_names = [
    "fiben_data_bundle",
    "ScenarioSpec",
    "RecommendationSpec",
    "FragmentSpec",
    "MaterializationPlan",
    "PhysicalDBSpec",
    "WorkloadSpec",
    "BenchmarkQuery",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first the setup blocks, BLOCK 1, BLOCK 2, and BLOCK 3."
        )


# ---------------------------------------------------------
# 2. Conceptual schema entities
# ---------------------------------------------------------
#
# The entities below are derived from the FIBEN workload.
#
# Company in the natural-language workload is modeled as
# Corporation in the conceptual schema.
# ---------------------------------------------------------

fiben_conceptual_entities = [
    "Corporation",
    "Industry",
    "Country",
    "Security",
    "ListedSecurity",
    "Person",
    "FinancialServiceAccount",
    "Holding",
    "Transaction",
    "BuyTransaction",
    "SellTransaction",
    "FinancialReport",
    "ReportElement",
    "StatementElement",
    "Disclosure",
]


# ---------------------------------------------------------
# 3. Conceptual relationships
# ---------------------------------------------------------
#
# These relationships represent the conceptual paths required
# by the 10 FIBEN workload queries.
#
# The relationship direction is chosen to support traversal and
# root-selection analysis in the methodology. The graph is later
# treated as an undirected conceptual adjacency graph by the
# generic blocks.
# ---------------------------------------------------------

fiben_conceptual_relationships = [
    # Corporation profile descriptors
    {
        "source": "Corporation",
        "target": "Industry",
        "name": "corporation_has_industry",
        "semantic_type": "descriptor",
        "notes": "Connects a corporation to its industry classification."
    },
    {
        "source": "Corporation",
        "target": "Country",
        "name": "corporation_has_country",
        "semantic_type": "descriptor",
        "notes": "Connects a corporation to its country."
    },

    # Securities and listing structure
    {
        "source": "Corporation",
        "target": "ListedSecurity",
        "name": "corporation_has_listed_security",
        "semantic_type": "association",
        "notes": "Connects a corporation to listed securities."
    },
    {
        "source": "ListedSecurity",
        "target": "Security",
        "name": "listed_security_represents_security",
        "semantic_type": "association",
        "notes": "Connects a listed security to the underlying security."
    },

    # Person/account/holding investment structure
    {
        "source": "Person",
        "target": "FinancialServiceAccount",
        "name": "person_owns_financial_service_account",
        "semantic_type": "ownership",
        "notes": "Connects a person to a financial service account."
    },
    {
        "source": "FinancialServiceAccount",
        "target": "Holding",
        "name": "account_has_holding",
        "semantic_type": "containment",
        "notes": "Connects an account to the holdings inside it."
    },
    {
        "source": "Holding",
        "target": "ListedSecurity",
        "name": "holding_refers_to_listed_security",
        "semantic_type": "association",
        "notes": "Connects a holding to the listed security being held."
    },

    # Transactions
    {
        "source": "FinancialServiceAccount",
        "target": "Transaction",
        "name": "account_records_transaction",
        "semantic_type": "containment",
        "notes": "Connects an account to its transactions."
    },
    {
        "source": "Transaction",
        "target": "ListedSecurity",
        "name": "transaction_refers_to_listed_security",
        "semantic_type": "association",
        "notes": "Connects a transaction to the listed security traded."
    },
    {
        "source": "BuyTransaction",
        "target": "Transaction",
        "name": "buy_transaction_is_transaction",
        "semantic_type": "subtype",
        "notes": "Models buy transactions as a subtype of transaction."
    },
    {
        "source": "SellTransaction",
        "target": "Transaction",
        "name": "sell_transaction_is_transaction",
        "semantic_type": "subtype",
        "notes": "Models sell transactions as a subtype of transaction."
    },

    # Financial reporting structure
    {
        "source": "Corporation",
        "target": "FinancialReport",
        "name": "corporation_has_financial_report",
        "semantic_type": "containment",
        "notes": "Connects a corporation to its financial reports."
    },
    {
        "source": "FinancialReport",
        "target": "ReportElement",
        "name": "financial_report_contains_report_element",
        "semantic_type": "containment",
        "notes": "Connects a financial report to report-level metric data."
    },
    {
        "source": "ReportElement",
        "target": "StatementElement",
        "name": "report_element_has_statement_element",
        "semantic_type": "containment",
        "notes": "Connects report elements to statement elements."
    },
    {
        "source": "FinancialReport",
        "target": "Disclosure",
        "name": "financial_report_contains_disclosure",
        "semantic_type": "containment",
        "notes": "Connects a financial report to disclosure data."
    },
]


# ---------------------------------------------------------
# 4. FIBEN scenario with the 10 concrete workload queries
# ---------------------------------------------------------

fiben_scenario = ScenarioSpec(
    name="FIBEN_Q1_Q10",
    description=(
        "Conceptual FIBEN scenario derived from the workload used to define "
        "the conceptual schema. The scenario covers corporation profile lookup, "
        "shallow rooted retrieval, associative retrieval, deep traversal, "
        "containment retrieval, filtered search, aggregation/ranking, comparison, "
        "and update/insertion with relationship creation."
    ),
    entities=fiben_conceptual_entities,
    relationships=fiben_conceptual_relationships,
    workload_queries=[
        {
            "name": "Q1_CompanyProfileIBM",
            "type": "select",
            "generic_class": "QG1",
            "query_family": "Local lookup",
            "entities": ["Corporation"],
            "abstract_query": "Show the profile of company IBM.",
            "notes": (
                "Local lookup over Corporation. The concrete company name IBM is "
                "treated as a query parameter."
            ),
        },
        {
            "name": "Q2_CompanyWithIndustryCountryAndListedSecurities",
            "type": "select",
            "generic_class": "QG2",
            "query_family": "Shallow rooted retrieval",
            "entities": ["Corporation", "Industry", "Country", "ListedSecurity"],
            "abstract_query": (
                "Show IBM with its industry, country, and listed securities."
            ),
            "notes": (
                "Shallow rooted retrieval centered on Corporation with direct "
                "descriptor and associated entities."
            ),
        },
        {
            "name": "Q3_SecuritiesHeldInEachFinancialServiceAccount",
            "type": "select",
            "generic_class": "QG3",
            "query_family": "Associative retrieval",
            "entities": ["FinancialServiceAccount", "Holding", "ListedSecurity", "Security"],
            "abstract_query": (
                "Show the securities held in each financial service account."
            ),
            "notes": (
                "Associative retrieval from financial service accounts through "
                "holdings to listed securities and underlying securities."
            ),
        },
        {
            "name": "Q4_CompaniesReachedFromPersonThroughAccountHoldingListedSecurity",
            "type": "select",
            "generic_class": "QG4",
            "query_family": "Deep traversal",
            "entities": [
                "Person",
                "FinancialServiceAccount",
                "Holding",
                "ListedSecurity",
                "Corporation",
            ],
            "abstract_query": (
                "Show the companies reached from a person through account, "
                "holding, and listed security."
            ),
            "notes": (
                "Deep traversal from Person to Corporation through the account "
                "and holding structure."
            ),
        },
        {
            "name": "Q5_ReportsAndMetricDataOfCompany",
            "type": "select",
            "generic_class": "QG5",
            "query_family": "Containment retrieval",
            "entities": [
                "Corporation",
                "FinancialReport",
                "ReportElement",
                "StatementElement",
                "Disclosure",
            ],
            "abstract_query": (
                "Show the financial reports of a company and the metric data "
                "contained in each report."
            ),
            "notes": (
                "Containment retrieval from Corporation to FinancialReport and "
                "its contained report elements, statement elements, and disclosures."
            ),
        },
        {
            "name": "Q6_TechUSListedSecuritiesWithHighLastTradedValue",
            "type": "select",
            "generic_class": "QG6",
            "query_family": "Filtered search / recommendation",
            "entities": ["Corporation", "Industry", "Country", "ListedSecurity"],
            "abstract_query": (
                "Find listed securities of technology companies in the US with "
                "high last traded value."
            ),
            "notes": (
                "Filtered search over listed securities using corporation industry, "
                "country, and traded-value predicates."
            ),
        },
        {
            "name": "Q7_PersonsWhoBoughtMoreIBMThanSold",
            "type": "select",
            "generic_class": "QG7",
            "query_family": "Aggregation / ranking",
            "entities": [
                "Person",
                "FinancialServiceAccount",
                "BuyTransaction",
                "SellTransaction",
                "Transaction",
                "ListedSecurity",
                "Corporation",
            ],
            "abstract_query": (
                "Who bought more IBM stocks than they sold?"
            ),
            "notes": (
                "Aggregation comparing buy and sell quantities per person for "
                "the IBM listed security."
            ),
        },
        {
            "name": "Q8_IBMTransactionsBelowAverageSellingPrice",
            "type": "select",
            "generic_class": "QG8",
            "query_family": "Aggregation / ranking",
            "entities": [
                "Transaction",
                "SellTransaction",
                "ListedSecurity",
                "Corporation",
            ],
            "abstract_query": (
                "Show me each transaction for IBM whose price is less than the "
                "average selling price."
            ),
            "notes": (
                "Comparison query that computes an average selling price and "
                "returns IBM transactions below that value."
            ),
        },
        {
            "name": "Q9_PersonsWhoBoughtAndSoldSameStock",
            "type": "select",
            "generic_class": "QG9",
            "query_family": "Associative retrieval + comparison",
            "entities": [
                "Person",
                "FinancialServiceAccount",
                "BuyTransaction",
                "SellTransaction",
                "Transaction",
                "ListedSecurity",
            ],
            "abstract_query": (
                "Show me everyone who bought and sold the same stock."
            ),
            "notes": (
                "Associative comparison query identifying persons with both buy "
                "and sell activity for the same listed security."
            ),
        },
        {
            "name": "Q10_CreateAccountHoldingAndBuyTransaction",
            "type": "insert",
            "generic_class": "QG10",
            "query_family": "Update / insertion with relationship creation",
            "entities": [
                "Person",
                "FinancialServiceAccount",
                "Holding",
                "ListedSecurity",
                "Transaction",
                "BuyTransaction",
            ],
            "abstract_query": (
                "Create a financial service account for a person, add a holding "
                "for a listed security, and register a buy transaction with price "
                "and quantity."
            ),
            "notes": (
                "Insertion workload with relationship creation across person, "
                "account, holding, listed security, and buy transaction."
            ),
        },
    ],
)


# ---------------------------------------------------------
# 5. Methodological recommendation fragment
# ---------------------------------------------------------

hubara_fiben_recommendation = RecommendationSpec(
    recommendation_name="FIBEN_Q1_Q10_Workload",
    source_method="Generic documentary methodology instantiated on FIBEN",
    source_dataset="FIBEN",
    fragments=[
        FragmentSpec(
            name="FIBENFragment",
            entities=fiben_conceptual_entities,
            model="document_candidate",
            notes=(
                "FIBEN conceptual fragment used to extract methodology variables, "
                "infer activation classes, generate candidate MongoDB configurations, "
                "and later run benchmark experiments."
            ),
        )
    ],
)


# ---------------------------------------------------------
# 6. Physical baseline materialization plan
# ---------------------------------------------------------
#
# This plan is only descriptive at this stage.
# The actual MongoDB materialization script will be adapted later.
# ---------------------------------------------------------

hubara_fiben_mongo_materialization = MaterializationPlan(
    scenario_name="FIBEN_Q1_Q10",
    recommendation_name="FIBEN_Mongo_Baseline",
    fragment_to_db={
        "FIBENFragment": PhysicalDBSpec(
            model="document",
            engine="MongoDB",
            host="127.0.0.1",
            port=27018,
            database_name="fiben_methodology_mongo_db",
            username="mongo",
            password="mongo",
        ),
    },
)


# ---------------------------------------------------------
# 7. Benchmark workload specification
# ---------------------------------------------------------

fiben_workload_mongo = WorkloadSpec(
    name="FIBEN_Methodology_Workload_Q1_Q10",
    description=(
        "Complete FIBEN workload used to instantiate the generic methodology "
        "classes over the FIBEN conceptual model."
    ),
    queries=[
        BenchmarkQuery(
            name=q["name"],
            query_type=q["type"],
            entities_involved=q["entities"],
            abstract_query=q["abstract_query"],
            expected_fragment="FIBENFragment",
            expected_db_model="document",
        )
        for q in fiben_scenario.workload_queries
    ],
    repetitions=10,
)


# ---------------------------------------------------------
# 8. Compatibility aliases for downstream generic blocks
# ---------------------------------------------------------
#
# The generic blocks V0–V22D still refer to some IMDb variable names.
# These aliases allow those blocks to run without rewriting them now.
# ---------------------------------------------------------

imdb_scenario = fiben_scenario
hubara_imdb_recommendation = hubara_fiben_recommendation
hubara_imdb_mongo_materialization = hubara_fiben_mongo_materialization
imdb_workload_mongo = fiben_workload_mongo


# ---------------------------------------------------------
# 9. Inspection DataFrames
# ---------------------------------------------------------

fiben_conceptual_entities_df = pd.DataFrame({
    "entity": fiben_conceptual_entities
})

fiben_conceptual_relationships_df = pd.DataFrame(
    fiben_conceptual_relationships
)

fiben_workload_core_df = pd.DataFrame(
    fiben_scenario.workload_queries
)

fiben_workload_benchmark_df = pd.DataFrame([
    {
        "name": q.name,
        "query_type": q.query_type,
        "entities_involved": q.entities_involved,
        "expected_fragment": q.expected_fragment,
        "expected_db_model": q.expected_db_model,
        "abstract_query": q.abstract_query,
    }
    for q in fiben_workload_mongo.queries
])

fiben_query_class_summary_df = fiben_workload_core_df[
    [
        "name",
        "generic_class",
        "query_family",
        "type",
        "entities",
        "notes",
    ]
].copy()


print("FIBEN scenario with workload Q1–Q10 was created successfully.")

print("\nConceptual entities:")
display(fiben_conceptual_entities_df)

print("\nConceptual relationships:")
display(fiben_conceptual_relationships_df)

print("\nMethodological query summary:")
display(fiben_query_class_summary_df)

print("\nBenchmark workload:")
display(fiben_workload_benchmark_df)


# ---------------------------------------------------------
# 10. Save INSTANCE 1 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    fiben_conceptual_entities_df,
    "variables/instance01/fiben_conceptual_entities.csv",
)

save_dataframe(
    fiben_conceptual_relationships_df,
    "variables/instance01/fiben_conceptual_relationships.csv",
)

save_dataframe(
    fiben_workload_core_df,
    "variables/instance01/fiben_workload_core.csv",
)

save_dataframe(
    fiben_workload_benchmark_df,
    "variables/instance01/fiben_workload_benchmark.csv",
)

save_dataframe(
    fiben_query_class_summary_df,
    "variables/instance01/fiben_query_class_summary.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "entities": fiben_conceptual_entities,
        "relationships": fiben_conceptual_relationships,
        "workload_queries": fiben_scenario.workload_queries,
        "recommendation_name": hubara_fiben_recommendation.recommendation_name,
        "materialization_recommendation": hubara_fiben_mongo_materialization.recommendation_name,
        "benchmark_workload_name": fiben_workload_mongo.name,
        "compatibility_aliases": {
            "imdb_scenario": "fiben_scenario",
            "imdb_workload_mongo": "fiben_workload_mongo",
            "hubara_imdb_recommendation": "hubara_fiben_recommendation",
            "hubara_imdb_mongo_materialization": "hubara_fiben_mongo_materialization",
        },
    },
    "variables/instance01/fiben_scenario_metadata.json",
)

write_execution_log(
    block_name="INSTANCE 1 — REAL FIBEN SCENARIO",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_entities": len(fiben_conceptual_entities),
        "n_relationships": len(fiben_conceptual_relationships),
        "n_workload_queries": len(fiben_scenario.workload_queries),
        "entities_csv": "variables/instance01/fiben_conceptual_entities.csv",
        "relationships_csv": "variables/instance01/fiben_conceptual_relationships.csv",
        "workload_core_csv": "variables/instance01/fiben_workload_core.csv",
        "workload_benchmark_csv": "variables/instance01/fiben_workload_benchmark.csv",
        "query_class_summary_csv": "variables/instance01/fiben_query_class_summary.csv",
        "scenario_metadata_json": "variables/instance01/fiben_scenario_metadata.json",
    },
)


# ---------------------------------------------------------
# 11. Update README section for INSTANCE 1
# ---------------------------------------------------------

instance01_readme_body = f"""
This block defines the real FIBEN conceptual scenario and workload.

The scenario replaces the previous IMDb scenario while preserving compatibility
with the generic downstream methodology blocks.

Conceptual scenario name:

    {fiben_scenario.name}

Number of conceptual entities:

    {len(fiben_conceptual_entities)}

Number of conceptual relationships:

    {len(fiben_conceptual_relationships)}

Number of workload queries:

    {len(fiben_scenario.workload_queries)}

Important modeling decision:

    The term "company" in the natural-language workload is represented as
    "Corporation" in the conceptual schema.

Workload queries:

    Q1  - Show the profile of company IBM.
    Q2  - Show IBM with its industry, country, and listed securities.
    Q3  - Show the securities held in each financial service account.
    Q4  - Show the companies reached from a person through account, holding, and listed security.
    Q5  - Show the financial reports of a company and the metric data contained in each report.
    Q6  - Find listed securities of technology companies in the US with high last traded value.
    Q7  - Who bought more IBM stocks than they sold?
    Q8  - Show me each transaction for IBM whose price is less than the average selling price.
    Q9  - Show me everyone who bought and sold the same stock.
    Q10 - Create a financial service account for a person, add a holding for a listed security, and register a buy transaction with price and quantity.

Generated reproducibility files:

    variables/instance01/fiben_conceptual_entities.csv
    variables/instance01/fiben_conceptual_relationships.csv
    variables/instance01/fiben_workload_core.csv
    variables/instance01/fiben_workload_benchmark.csv
    variables/instance01/fiben_query_class_summary.csv
    variables/instance01/fiben_scenario_metadata.json

Compatibility aliases created for downstream blocks:

    imdb_scenario = fiben_scenario
    imdb_workload_mongo = fiben_workload_mongo
    hubara_imdb_recommendation = hubara_fiben_recommendation
    hubara_imdb_mongo_materialization = hubara_fiben_mongo_materialization

These aliases are temporary and allow the generic methodology blocks V0–V22D
to run without being rewritten immediately.
"""

update_readme_section(
    section_title="INSTANCE 1 — Real FIBEN Scenario",
    section_body=instance01_readme_body,
)

FIBEN scenario with workload Q1–Q10 was created successfully.

Conceptual entities:


,entity
0,Corporation
1,Industry
2,Country
3,Security
4,ListedSecurity
5,Person
6,FinancialServiceAccount
7,Holding
8,Transaction
9,BuyTransaction



Conceptual relationships:


,source,target,name,semantic_type,notes
0,Corporation,Industry,corporation_has_industry,descriptor,Connects a corporation to its industry classif...
1,Corporation,Country,corporation_has_country,descriptor,Connects a corporation to its country.
2,Corporation,ListedSecurity,corporation_has_listed_security,association,Connects a corporation to listed securities.
3,ListedSecurity,Security,listed_security_represents_security,association,Connects a listed security to the underlying s...
4,Person,FinancialServiceAccount,person_owns_financial_service_account,ownership,Connects a person to a financial service account.
5,FinancialServiceAccount,Holding,account_has_holding,containment,Connects an account to the holdings inside it.
6,Holding,ListedSecurity,holding_refers_to_listed_security,association,Connects a holding to the listed security bein...
7,FinancialServiceAccount,Transaction,account_records_transaction,containment,Connects an account to its transactions.
8,Transaction,ListedSecurity,transaction_refers_to_listed_security,association,Connects a transaction to the listed security ...
9,BuyTransaction,Transaction,buy_transaction_is_transaction,subtype,Models buy transactions as a subtype of transa...



Methodological query summary:


,name,generic_class,query_family,type,entities,notes
0,Q1_CompanyProfileIBM,QG1,Local lookup,select,[Corporation],Local lookup over Corporation. The concrete co...
1,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,"[Corporation, Industry, Country, ListedSecurity]",Shallow rooted retrieval centered on Corporati...
2,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,"[FinancialServiceAccount, Holding, ListedSecur...",Associative retrieval from financial service a...
3,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,"[Person, FinancialServiceAccount, Holding, Lis...",Deep traversal from Person to Corporation thro...
4,Q5_ReportsAndMetricDataOfCompany,QG5,Containment retrieval,select,"[Corporation, FinancialReport, ReportElement, ...",Containment retrieval from Corporation to Fina...
5,Q6_TechUSListedSecuritiesWithHighLastTradedValue,QG6,Filtered search / recommendation,select,"[Corporation, Industry, Country, ListedSecurity]",Filtered search over listed securities using c...
6,Q7_PersonsWhoBoughtMoreIBMThanSold,QG7,Aggregation / ranking,select,"[Person, FinancialServiceAccount, BuyTransacti...",Aggregation comparing buy and sell quantities ...
7,Q8_IBMTransactionsBelowAverageSellingPrice,QG8,Aggregation / ranking,select,"[Transaction, SellTransaction, ListedSecurity,...",Comparison query that computes an average sell...
8,Q9_PersonsWhoBoughtAndSoldSameStock,QG9,Associative retrieval + comparison,select,"[Person, FinancialServiceAccount, BuyTransacti...",Associative comparison query identifying perso...
9,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,"[Person, FinancialServiceAccount, Holding, Lis...",Insertion workload with relationship creation ...



Benchmark workload:


,name,query_type,entities_involved,expected_fragment,expected_db_model,abstract_query
0,Q1_CompanyProfileIBM,select,[Corporation],FIBENFragment,document,Show the profile of company IBM.
1,Q2_CompanyWithIndustryCountryAndListedSecurities,select,"[Corporation, Industry, Country, ListedSecurity]",FIBENFragment,document,"Show IBM with its industry, country, and liste..."
2,Q3_SecuritiesHeldInEachFinancialServiceAccount,select,"[FinancialServiceAccount, Holding, ListedSecur...",FIBENFragment,document,Show the securities held in each financial ser...
3,Q4_CompaniesReachedFromPersonThroughAccountHol...,select,"[Person, FinancialServiceAccount, Holding, Lis...",FIBENFragment,document,Show the companies reached from a person throu...
4,Q5_ReportsAndMetricDataOfCompany,select,"[Corporation, FinancialReport, ReportElement, ...",FIBENFragment,document,Show the financial reports of a company and th...
5,Q6_TechUSListedSecuritiesWithHighLastTradedValue,select,"[Corporation, Industry, Country, ListedSecurity]",FIBENFragment,document,Find listed securities of technology companies...
6,Q7_PersonsWhoBoughtMoreIBMThanSold,select,"[Person, FinancialServiceAccount, BuyTransacti...",FIBENFragment,document,Who bought more IBM stocks than they sold?
7,Q8_IBMTransactionsBelowAverageSellingPrice,select,"[Transaction, SellTransaction, ListedSecurity,...",FIBENFragment,document,Show me each transaction for IBM whose price i...
8,Q9_PersonsWhoBoughtAndSoldSameStock,select,"[Person, FinancialServiceAccount, BuyTransacti...",FIBENFragment,document,Show me everyone who bought and sold the same ...
9,Q10_CreateAccountHoldingAndBuyTransaction,insert,"[Person, FinancialServiceAccount, Holding, Lis...",FIBENFragment,document,Create a financial service account for a perso...


Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/instance01/fiben_conceptual_entities.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/instance01/fiben_conceptual_relationships.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/instance01/fiben_workload_core.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/instance01/fiben_workload_benchmark.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/instance01/fiben_query_class_summary.csv
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/instance01/fiben_scenario_metadata.json
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/logs/instance_1___rea

In [40]:
# =========================================================
# BLOCK V0 — PREPARE THE CONCEPTUAL WORKLOAD
# =========================================================
#
# Purpose:
# This block converts the conceptual workload stored in the
# FIBEN scenario into structured DataFrames.
#
# This is the first generic methodology block after defining
# the dataset-specific scenario.
#
# Input:
# - fiben_scenario
# - imdb_scenario compatibility alias
#
# Main output:
# - workload_conceptual_df
#
# Additional reproducibility outputs:
# - workload_entity_incidence_df
# - workload_summary_by_generic_class_df
# - workload_summary_by_query_type_df
#
# Important:
# The downstream blocks still use the variable name
# workload_conceptual_df. Therefore, this name is preserved.
# =========================================================

import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

if "fiben_scenario" not in globals():
    raise NameError(
        "The object 'fiben_scenario' is not defined. "
        "Run first INSTANCE 1 — REAL FIBEN SCENARIO."
    )

if "imdb_scenario" not in globals():
    raise NameError(
        "The compatibility alias 'imdb_scenario' is not defined. "
        "Run first INSTANCE 1 — REAL FIBEN SCENARIO."
    )

if imdb_scenario is not fiben_scenario:
    print(
        "Warning: imdb_scenario and fiben_scenario are not the same object. "
        "The block will use fiben_scenario as the source of truth."
    )


# ---------------------------------------------------------
# 2. Convert scenario workload into a DataFrame
# ---------------------------------------------------------
#
# The original generic blocks expect a DataFrame called:
#
#     workload_conceptual_df
#
# We keep this variable name to avoid breaking downstream code.
# ---------------------------------------------------------

workload_conceptual_df = pd.DataFrame(fiben_scenario.workload_queries).copy()


# ---------------------------------------------------------
# 3. Standardize column names
# ---------------------------------------------------------
#
# The methodology uses explicit names:
# - query_name;
# - query_type;
# - entities_touched.
# ---------------------------------------------------------

workload_conceptual_df = workload_conceptual_df.rename(columns={
    "name": "query_name",
    "type": "query_type",
    "entities": "entities_touched",
})


# ---------------------------------------------------------
# 4. Ensure required columns exist
# ---------------------------------------------------------

required_columns = [
    "query_name",
    "query_type",
    "generic_class",
    "query_family",
    "entities_touched",
    "abstract_query",
]

missing_columns = [
    col for col in required_columns
    if col not in workload_conceptual_df.columns
]

if missing_columns:
    raise ValueError(
        "The workload is missing required columns: "
        f"{missing_columns}"
    )


# ---------------------------------------------------------
# 5. Normalize entities_touched
# ---------------------------------------------------------
#
# This guarantees that each query has a list of touched entities.
# ---------------------------------------------------------

def ensure_list(value):
    """
    Convert a value into a list.

    This helper protects the methodology when a cell is stored as:
    - a list;
    - a tuple;
    - a single string;
    - a missing value.
    """
    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    if pd.isna(value):
        return []

    return [value]


workload_conceptual_df["entities_touched"] = (
    workload_conceptual_df["entities_touched"]
    .apply(ensure_list)
)


# ---------------------------------------------------------
# 6. Compute R(Qi): number of entities touched by each query
# ---------------------------------------------------------
#
# In this notebook, R(Qi) is represented as:
#
#     n_entities_touched
#
# It is the number of conceptual entities explicitly touched by
# each workload query.
# ---------------------------------------------------------

workload_conceptual_df["n_entities_touched"] = (
    workload_conceptual_df["entities_touched"]
    .apply(len)
)


# ---------------------------------------------------------
# 7. Create an entity-incidence table
# ---------------------------------------------------------
#
# This table expands the workload so each row represents:
#
#     query -> touched entity
#
# It is useful for validation and for later methodology steps.
# ---------------------------------------------------------

entity_incidence_rows = []

for _, row in workload_conceptual_df.iterrows():
    for entity in row["entities_touched"]:
        entity_incidence_rows.append({
            "query_name": row["query_name"],
            "generic_class": row["generic_class"],
            "query_family": row["query_family"],
            "query_type": row["query_type"],
            "entity": entity,
        })

workload_entity_incidence_df = pd.DataFrame(entity_incidence_rows)


# ---------------------------------------------------------
# 8. Create summary tables
# ---------------------------------------------------------

workload_summary_by_generic_class_df = (
    workload_conceptual_df
    .groupby(["generic_class", "query_family"], dropna=False)
    .agg(
        n_queries=("query_name", "count"),
        avg_entities_touched=("n_entities_touched", "mean"),
        min_entities_touched=("n_entities_touched", "min"),
        max_entities_touched=("n_entities_touched", "max"),
    )
    .reset_index()
)

workload_summary_by_query_type_df = (
    workload_conceptual_df
    .groupby(["query_type"], dropna=False)
    .agg(
        n_queries=("query_name", "count"),
        avg_entities_touched=("n_entities_touched", "mean"),
        min_entities_touched=("n_entities_touched", "min"),
        max_entities_touched=("n_entities_touched", "max"),
    )
    .reset_index()
)


# ---------------------------------------------------------
# 9. Display outputs
# ---------------------------------------------------------

print("FIBEN conceptual workload prepared successfully.")

print("\nConceptual workload:")
display(workload_conceptual_df)

print("\nQuery-to-entity incidence table:")
display(workload_entity_incidence_df)

print("\nSummary by generic class:")
display(workload_summary_by_generic_class_df)

print("\nSummary by query type:")
display(workload_summary_by_query_type_df)


# ---------------------------------------------------------
# 10. Save BLOCK V0 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    workload_conceptual_df,
    "variables/block_v00/workload_conceptual.csv",
)

save_dataframe(
    workload_entity_incidence_df,
    "variables/block_v00/workload_entity_incidence.csv",
)

save_dataframe(
    workload_summary_by_generic_class_df,
    "variables/block_v00/workload_summary_by_generic_class.csv",
)

save_dataframe(
    workload_summary_by_query_type_df,
    "variables/block_v00/workload_summary_by_query_type.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "n_queries": len(workload_conceptual_df),
        "n_unique_entities_touched": int(
            workload_entity_incidence_df["entity"].nunique()
        ),
        "query_names": workload_conceptual_df["query_name"].tolist(),
        "generic_classes": workload_conceptual_df["generic_class"].tolist(),
        "query_types": workload_conceptual_df["query_type"].tolist(),
        "output_files": {
            "workload_conceptual_csv": "variables/block_v00/workload_conceptual.csv",
            "workload_entity_incidence_csv": "variables/block_v00/workload_entity_incidence.csv",
            "summary_by_generic_class_csv": "variables/block_v00/workload_summary_by_generic_class.csv",
            "summary_by_query_type_csv": "variables/block_v00/workload_summary_by_query_type.csv",
        },
    },
    "variables/block_v00/workload_conceptual_metadata.json",
)

write_execution_log(
    block_name="BLOCK V0 — PREPARE THE CONCEPTUAL WORKLOAD",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_queries": len(workload_conceptual_df),
        "n_unique_entities_touched": int(
            workload_entity_incidence_df["entity"].nunique()
        ),
        "workload_conceptual_csv": "variables/block_v00/workload_conceptual.csv",
        "workload_entity_incidence_csv": "variables/block_v00/workload_entity_incidence.csv",
        "summary_by_generic_class_csv": "variables/block_v00/workload_summary_by_generic_class.csv",
        "summary_by_query_type_csv": "variables/block_v00/workload_summary_by_query_type.csv",
        "metadata_json": "variables/block_v00/workload_conceptual_metadata.json",
    },
)


# ---------------------------------------------------------
# 11. Update README section for BLOCK V0
# ---------------------------------------------------------

block_v00_readme_body = f"""
This block prepares the conceptual workload used by the FIBEN methodology execution.

The source scenario is:

    {fiben_scenario.name}

The block converts the workload queries into the DataFrame:

    workload_conceptual_df

This DataFrame is required by the downstream methodology blocks.

Number of workload queries:

    {len(workload_conceptual_df)}

Number of unique touched entities:

    {int(workload_entity_incidence_df["entity"].nunique())}

Main variable created:

    workload_conceptual_df

Additional variables created:

    workload_entity_incidence_df
    workload_summary_by_generic_class_df
    workload_summary_by_query_type_df

Generated reproducibility files:

    variables/block_v00/workload_conceptual.csv
    variables/block_v00/workload_entity_incidence.csv
    variables/block_v00/workload_summary_by_generic_class.csv
    variables/block_v00/workload_summary_by_query_type.csv
    variables/block_v00/workload_conceptual_metadata.json

Methodological meaning:

    This block computes R(Qi) as n_entities_touched.
    R(Qi) represents how many conceptual entities are touched by each query.

Important note:

    The variable name workload_conceptual_df is preserved because later generic
    blocks depend on it.
"""

update_readme_section(
    section_title="BLOCK V0 — Prepare the Conceptual Workload",
    section_body=block_v00_readme_body,
)

FIBEN conceptual workload prepared successfully.

Conceptual workload:


,query_name,query_type,generic_class,query_family,entities_touched,abstract_query,notes,n_entities_touched
0,Q1_CompanyProfileIBM,select,QG1,Local lookup,[Corporation],Show the profile of company IBM.,Local lookup over Corporation. The concrete co...,1
1,Q2_CompanyWithIndustryCountryAndListedSecurities,select,QG2,Shallow rooted retrieval,"[Corporation, Industry, Country, ListedSecurity]","Show IBM with its industry, country, and liste...",Shallow rooted retrieval centered on Corporati...,4
2,Q3_SecuritiesHeldInEachFinancialServiceAccount,select,QG3,Associative retrieval,"[FinancialServiceAccount, Holding, ListedSecur...",Show the securities held in each financial ser...,Associative retrieval from financial service a...,4
3,Q4_CompaniesReachedFromPersonThroughAccountHol...,select,QG4,Deep traversal,"[Person, FinancialServiceAccount, Holding, Lis...",Show the companies reached from a person throu...,Deep traversal from Person to Corporation thro...,5
4,Q5_ReportsAndMetricDataOfCompany,select,QG5,Containment retrieval,"[Corporation, FinancialReport, ReportElement, ...",Show the financial reports of a company and th...,Containment retrieval from Corporation to Fina...,5
5,Q6_TechUSListedSecuritiesWithHighLastTradedValue,select,QG6,Filtered search / recommendation,"[Corporation, Industry, Country, ListedSecurity]",Find listed securities of technology companies...,Filtered search over listed securities using c...,4
6,Q7_PersonsWhoBoughtMoreIBMThanSold,select,QG7,Aggregation / ranking,"[Person, FinancialServiceAccount, BuyTransacti...",Who bought more IBM stocks than they sold?,Aggregation comparing buy and sell quantities ...,7
7,Q8_IBMTransactionsBelowAverageSellingPrice,select,QG8,Aggregation / ranking,"[Transaction, SellTransaction, ListedSecurity,...",Show me each transaction for IBM whose price i...,Comparison query that computes an average sell...,4
8,Q9_PersonsWhoBoughtAndSoldSameStock,select,QG9,Associative retrieval + comparison,"[Person, FinancialServiceAccount, BuyTransacti...",Show me everyone who bought and sold the same ...,Associative comparison query identifying perso...,6
9,Q10_CreateAccountHoldingAndBuyTransaction,insert,QG10,Update / insertion with relationship creation,"[Person, FinancialServiceAccount, Holding, Lis...",Create a financial service account for a perso...,Insertion workload with relationship creation ...,6



Query-to-entity incidence table:


,query_name,generic_class,query_family,query_type,entity
0,Q1_CompanyProfileIBM,QG1,Local lookup,select,Corporation
1,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Corporation
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Industry
3,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Country
4,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,ListedSecurity
5,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount
6,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,Holding
7,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,ListedSecurity
8,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,Security
9,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Person



Summary by generic class:


,generic_class,query_family,n_queries,avg_entities_touched,min_entities_touched,max_entities_touched
0,QG1,Local lookup,1,1.0,1,1
1,QG10,Update / insertion with relationship creation,1,6.0,6,6
2,QG2,Shallow rooted retrieval,1,4.0,4,4
3,QG3,Associative retrieval,1,4.0,4,4
4,QG4,Deep traversal,1,5.0,5,5
5,QG5,Containment retrieval,1,5.0,5,5
6,QG6,Filtered search / recommendation,1,4.0,4,4
7,QG7,Aggregation / ranking,1,7.0,7,7
8,QG8,Aggregation / ranking,1,4.0,4,4
9,QG9,Associative retrieval + comparison,1,6.0,6,6



Summary by query type:


,query_type,n_queries,avg_entities_touched,min_entities_touched,max_entities_touched
0,insert,1,6.000000,6,6
1,select,9,4.444444,1,7


Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v00/workload_conceptual.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v00/workload_entity_incidence.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v00/workload_summary_by_generic_class.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v00/workload_summary_by_query_type.csv
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v00/workload_conceptual_metadata.json
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/logs/block_v0___prepare_the_conceptual_workload_execution_log.json
Updated README section: BLOCK V0 — Prepare the Conceptual Workload
README path: /home/jovyan/p

In [43]:
# =========================================================
# BLOCK V1 — ORGANIZE THE CONCEPTUAL RELATIONSHIPS
# =========================================================
#
# Purpose:
# This block organizes the conceptual relationships of the
# FIBEN scenario into DataFrames and graph structures.
#
# Input:
# - fiben_scenario.relationships
#
# Main outputs preserved for downstream compatibility:
# - conceptual_relationships_df
# - conceptual_edges
# - conceptual_adj
#
# Additional outputs:
# - conceptual_bidirectional_edges_df
# - conceptual_adjacency_df
# - conceptual_relationship_summary_df
#
# Methodological role:
# These structures are used by later blocks to compute:
# - root candidates;
# - conceptual distances D(E, er, Qi);
# - reachable entities Re(Qi, er, D);
# - DeltaR and DeltaRratio.
# =========================================================

from collections import defaultdict
import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

if "fiben_scenario" not in globals():
    raise NameError(
        "The object 'fiben_scenario' is not defined. "
        "Run first INSTANCE 1 — REAL FIBEN SCENARIO."
    )

if "imdb_scenario" not in globals():
    raise NameError(
        "The compatibility alias 'imdb_scenario' is not defined. "
        "Run first INSTANCE 1 — REAL FIBEN SCENARIO."
    )

if imdb_scenario is not fiben_scenario:
    print(
        "Warning: imdb_scenario and fiben_scenario are not the same object. "
        "The block will use fiben_scenario as the source of truth."
    )


# ---------------------------------------------------------
# 2. Build the conceptual relationships DataFrame
# ---------------------------------------------------------
#
# The downstream generic blocks expect a variable called:
#
#     conceptual_relationships_df
#
# Therefore, this name is preserved.
# ---------------------------------------------------------

conceptual_relationships_df = pd.DataFrame(
    fiben_scenario.relationships
).copy()


# ---------------------------------------------------------
# 3. Validate required columns
# ---------------------------------------------------------

required_relationship_columns = ["source", "target", "name"]

missing_relationship_columns = [
    col for col in required_relationship_columns
    if col not in conceptual_relationships_df.columns
]

if missing_relationship_columns:
    raise ValueError(
        "The conceptual relationship list is missing required columns: "
        f"{missing_relationship_columns}"
    )


# ---------------------------------------------------------
# 4. Normalize optional columns
# ---------------------------------------------------------

if "semantic_type" not in conceptual_relationships_df.columns:
    conceptual_relationships_df["semantic_type"] = "unspecified"

if "notes" not in conceptual_relationships_df.columns:
    conceptual_relationships_df["notes"] = ""

conceptual_relationships_df["semantic_type"] = (
    conceptual_relationships_df["semantic_type"]
    .fillna("unspecified")
    .astype(str)
)

conceptual_relationships_df["notes"] = (
    conceptual_relationships_df["notes"]
    .fillna("")
    .astype(str)
)


# ---------------------------------------------------------
# 5. Create stable edge identifiers
# ---------------------------------------------------------
#
# The original notebook used edge_id in this format:
#
#     Source -- Target [relationship_name]
#
# We keep the same format for compatibility.
# ---------------------------------------------------------

conceptual_relationships_df["edge_id"] = conceptual_relationships_df.apply(
    lambda row: f'{row["source"]} -- {row["target"]} [{row["name"]}]',
    axis=1,
)

conceptual_relationships_df["source_target_pair"] = (
    conceptual_relationships_df["source"].astype(str)
    + " -> "
    + conceptual_relationships_df["target"].astype(str)
)


# ---------------------------------------------------------
# 6. Validate relationship entities against the scenario entity list
# ---------------------------------------------------------

scenario_entities = set(fiben_scenario.entities)

relationship_entities = set(
    conceptual_relationships_df["source"].tolist()
    + conceptual_relationships_df["target"].tolist()
)

entities_only_in_relationships = sorted(
    relationship_entities - scenario_entities
)

entities_without_relationships = sorted(
    scenario_entities - relationship_entities
)

if entities_only_in_relationships:
    print("Warning: entities appear in relationships but not in scenario.entities:")
    print(entities_only_in_relationships)

if entities_without_relationships:
    print("Warning: scenario entities without relationships:")
    print(entities_without_relationships)


# ---------------------------------------------------------
# 7. Create the conceptual edge list
# ---------------------------------------------------------
#
# This object is expected by later generic blocks.
# ---------------------------------------------------------

conceptual_edges = conceptual_relationships_df[
    ["source", "target", "name", "semantic_type", "edge_id"]
].to_dict(orient="records")


# ---------------------------------------------------------
# 8. Build an undirected conceptual adjacency graph
# ---------------------------------------------------------
#
# The methodology uses conceptual connectivity.
# Therefore, edges are inserted in both directions.
# ---------------------------------------------------------

conceptual_adj = defaultdict(list)

bidirectional_edge_rows = []

for edge in conceptual_edges:
    src = edge["source"]
    tgt = edge["target"]
    rel_name = edge["name"]
    semantic_type = edge["semantic_type"]
    edge_id = edge["edge_id"]

    # Forward direction
    conceptual_adj[src].append((tgt, rel_name, edge_id))

    bidirectional_edge_rows.append({
        "from_entity": src,
        "to_entity": tgt,
        "relationship_name": rel_name,
        "semantic_type": semantic_type,
        "edge_id": edge_id,
        "direction": "forward",
    })

    # Reverse direction
    conceptual_adj[tgt].append((src, rel_name, edge_id))

    bidirectional_edge_rows.append({
        "from_entity": tgt,
        "to_entity": src,
        "relationship_name": rel_name,
        "semantic_type": semantic_type,
        "edge_id": edge_id,
        "direction": "reverse",
    })


conceptual_bidirectional_edges_df = pd.DataFrame(
    bidirectional_edge_rows
)


# ---------------------------------------------------------
# 9. Create an adjacency table for inspection
# ---------------------------------------------------------

adjacency_rows = []

for entity, neighbors in sorted(conceptual_adj.items()):
    for neighbor, relationship_name, edge_id in neighbors:
        adjacency_rows.append({
            "entity": entity,
            "neighbor": neighbor,
            "relationship_name": relationship_name,
            "edge_id": edge_id,
        })

conceptual_adjacency_df = pd.DataFrame(adjacency_rows)


# ---------------------------------------------------------
# 10. Create relationship summary tables
# ---------------------------------------------------------

conceptual_relationship_summary_df = (
    conceptual_relationships_df
    .groupby("semantic_type", dropna=False)
    .agg(
        n_relationships=("name", "count"),
        source_entities=("source", lambda s: sorted(set(s))),
        target_entities=("target", lambda s: sorted(set(s))),
    )
    .reset_index()
    .sort_values("semantic_type")
    .reset_index(drop=True)
)

conceptual_entity_degree_df = (
    conceptual_adjacency_df
    .groupby("entity", as_index=False)
    .agg(
        degree=("neighbor", "count"),
        neighbors=("neighbor", lambda s: sorted(set(s))),
        relationships=("relationship_name", lambda s: sorted(set(s))),
    )
    .sort_values(["degree", "entity"], ascending=[False, True])
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 11. Display outputs
# ---------------------------------------------------------

print("FIBEN conceptual relationships organized successfully.")

print("\nConceptual relationships:")
display(conceptual_relationships_df)

print("\nBidirectional conceptual edges:")
display(conceptual_bidirectional_edges_df)

print("\nConceptual adjacency:")
display(conceptual_adjacency_df)

print("\nRelationship summary by semantic type:")
display(conceptual_relationship_summary_df)

print("\nConceptual entity degree:")
display(conceptual_entity_degree_df)

print("\nEntities present in the conceptual graph:")
print(sorted(conceptual_adj.keys()))


# ---------------------------------------------------------
# 12. Save BLOCK V1 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    conceptual_relationships_df,
    "variables/block_v01/conceptual_relationships.csv",
)

save_dataframe(
    conceptual_bidirectional_edges_df,
    "variables/block_v01/conceptual_bidirectional_edges.csv",
)

save_dataframe(
    conceptual_adjacency_df,
    "variables/block_v01/conceptual_adjacency.csv",
)

save_dataframe(
    conceptual_relationship_summary_df,
    "variables/block_v01/conceptual_relationship_summary.csv",
)

save_dataframe(
    conceptual_entity_degree_df,
    "variables/block_v01/conceptual_entity_degree.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "n_conceptual_relationships": len(conceptual_relationships_df),
        "n_bidirectional_edges": len(conceptual_bidirectional_edges_df),
        "n_entities_in_graph": len(conceptual_adj.keys()),
        "entities_in_graph": sorted(conceptual_adj.keys()),
        "entities_only_in_relationships": entities_only_in_relationships,
        "entities_without_relationships": entities_without_relationships,
        "semantic_types": sorted(
            conceptual_relationships_df["semantic_type"].dropna().unique().tolist()
        ),
        "output_files": {
            "conceptual_relationships_csv": "variables/block_v01/conceptual_relationships.csv",
            "conceptual_bidirectional_edges_csv": "variables/block_v01/conceptual_bidirectional_edges.csv",
            "conceptual_adjacency_csv": "variables/block_v01/conceptual_adjacency.csv",
            "conceptual_relationship_summary_csv": "variables/block_v01/conceptual_relationship_summary.csv",
            "conceptual_entity_degree_csv": "variables/block_v01/conceptual_entity_degree.csv",
        },
    },
    "variables/block_v01/conceptual_relationships_metadata.json",
)

write_execution_log(
    block_name="BLOCK V1 — ORGANIZE THE CONCEPTUAL RELATIONSHIPS",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_conceptual_relationships": len(conceptual_relationships_df),
        "n_bidirectional_edges": len(conceptual_bidirectional_edges_df),
        "n_entities_in_graph": len(conceptual_adj.keys()),
        "conceptual_relationships_csv": "variables/block_v01/conceptual_relationships.csv",
        "conceptual_bidirectional_edges_csv": "variables/block_v01/conceptual_bidirectional_edges.csv",
        "conceptual_adjacency_csv": "variables/block_v01/conceptual_adjacency.csv",
        "conceptual_relationship_summary_csv": "variables/block_v01/conceptual_relationship_summary.csv",
        "conceptual_entity_degree_csv": "variables/block_v01/conceptual_entity_degree.csv",
        "metadata_json": "variables/block_v01/conceptual_relationships_metadata.json",
    },
)


# ---------------------------------------------------------
# 13. Update README section for BLOCK V1
# ---------------------------------------------------------

block_v01_readme_body = f"""
This block organizes the conceptual relationships of the FIBEN scenario.

The source scenario is:

    {fiben_scenario.name}

Main variables created:

    conceptual_relationships_df
    conceptual_edges
    conceptual_adj

Additional variables created:

    conceptual_bidirectional_edges_df
    conceptual_adjacency_df
    conceptual_relationship_summary_df
    conceptual_entity_degree_df

Number of conceptual relationships:

    {len(conceptual_relationships_df)}

Number of bidirectional graph edges:

    {len(conceptual_bidirectional_edges_df)}

Number of entities in the conceptual graph:

    {len(conceptual_adj.keys())}

Semantic relationship types:

    {sorted(conceptual_relationships_df["semantic_type"].dropna().unique().tolist())}

Generated reproducibility files:

    variables/block_v01/conceptual_relationships.csv
    variables/block_v01/conceptual_bidirectional_edges.csv
    variables/block_v01/conceptual_adjacency.csv
    variables/block_v01/conceptual_relationship_summary.csv
    variables/block_v01/conceptual_entity_degree.csv
    variables/block_v01/conceptual_relationships_metadata.json

Methodological meaning:

    The conceptual relationships define the graph used to compute connectivity
    and conceptual distance between entities.

    Although the conceptual model stores source and target directions, the
    methodology treats the graph as undirected for distance calculations.

Why this is important:

    Later blocks use conceptual_adj and conceptual_edges to compute:
    - root candidates;
    - D(E, er, Qi);
    - Re(Qi, er, D);
    - DeltaR;
    - DeltaRratio.
"""

update_readme_section(
    section_title="BLOCK V1 — Organize the Conceptual Relationships",
    section_body=block_v01_readme_body,
)

FIBEN conceptual relationships organized successfully.

Conceptual relationships:


,source,target,name,semantic_type,notes,edge_id,source_target_pair
0,Corporation,Industry,corporation_has_industry,descriptor,Connects a corporation to its industry classif...,Corporation -- Industry [corporation_has_indus...,Corporation -> Industry
1,Corporation,Country,corporation_has_country,descriptor,Connects a corporation to its country.,Corporation -- Country [corporation_has_country],Corporation -> Country
2,Corporation,ListedSecurity,corporation_has_listed_security,association,Connects a corporation to listed securities.,Corporation -- ListedSecurity [corporation_has...,Corporation -> ListedSecurity
3,ListedSecurity,Security,listed_security_represents_security,association,Connects a listed security to the underlying s...,ListedSecurity -- Security [listed_security_re...,ListedSecurity -> Security
4,Person,FinancialServiceAccount,person_owns_financial_service_account,ownership,Connects a person to a financial service account.,Person -- FinancialServiceAccount [person_owns...,Person -> FinancialServiceAccount
5,FinancialServiceAccount,Holding,account_has_holding,containment,Connects an account to the holdings inside it.,FinancialServiceAccount -- Holding [account_ha...,FinancialServiceAccount -> Holding
6,Holding,ListedSecurity,holding_refers_to_listed_security,association,Connects a holding to the listed security bein...,Holding -- ListedSecurity [holding_refers_to_l...,Holding -> ListedSecurity
7,FinancialServiceAccount,Transaction,account_records_transaction,containment,Connects an account to its transactions.,FinancialServiceAccount -- Transaction [accoun...,FinancialServiceAccount -> Transaction
8,Transaction,ListedSecurity,transaction_refers_to_listed_security,association,Connects a transaction to the listed security ...,Transaction -- ListedSecurity [transaction_ref...,Transaction -> ListedSecurity
9,BuyTransaction,Transaction,buy_transaction_is_transaction,subtype,Models buy transactions as a subtype of transa...,BuyTransaction -- Transaction [buy_transaction...,BuyTransaction -> Transaction



Bidirectional conceptual edges:


,from_entity,to_entity,relationship_name,semantic_type,edge_id,direction
0,Corporation,Industry,corporation_has_industry,descriptor,Corporation -- Industry [corporation_has_indus...,forward
1,Industry,Corporation,corporation_has_industry,descriptor,Corporation -- Industry [corporation_has_indus...,reverse
2,Corporation,Country,corporation_has_country,descriptor,Corporation -- Country [corporation_has_country],forward
3,Country,Corporation,corporation_has_country,descriptor,Corporation -- Country [corporation_has_country],reverse
4,Corporation,ListedSecurity,corporation_has_listed_security,association,Corporation -- ListedSecurity [corporation_has...,forward
5,ListedSecurity,Corporation,corporation_has_listed_security,association,Corporation -- ListedSecurity [corporation_has...,reverse
6,ListedSecurity,Security,listed_security_represents_security,association,ListedSecurity -- Security [listed_security_re...,forward
7,Security,ListedSecurity,listed_security_represents_security,association,ListedSecurity -- Security [listed_security_re...,reverse
8,Person,FinancialServiceAccount,person_owns_financial_service_account,ownership,Person -- FinancialServiceAccount [person_owns...,forward
9,FinancialServiceAccount,Person,person_owns_financial_service_account,ownership,Person -- FinancialServiceAccount [person_owns...,reverse



Conceptual adjacency:


...


Relationship summary by semantic type:


,semantic_type,n_relationships,source_entities,target_entities
0,association,4,"[Corporation, Holding, ListedSecurity, Transac...","[ListedSecurity, Security]"
1,containment,6,"[Corporation, FinancialReport, FinancialServic...","[Disclosure, FinancialReport, Holding, ReportE..."
2,descriptor,2,[Corporation],"[Country, Industry]"
3,ownership,1,[Person],[FinancialServiceAccount]
4,subtype,2,"[BuyTransaction, SellTransaction]",[Transaction]



Conceptual entity degree:


,entity,degree,neighbors,relationships
0,Corporation,4,"[Country, FinancialReport, Industry, ListedSec...","[corporation_has_country, corporation_has_fina..."
1,ListedSecurity,4,"[Corporation, Holding, Security, Transaction]","[corporation_has_listed_security, holding_refe..."
2,Transaction,4,"[BuyTransaction, FinancialServiceAccount, List...","[account_records_transaction, buy_transaction_..."
3,FinancialReport,3,"[Corporation, Disclosure, ReportElement]","[corporation_has_financial_report, financial_r..."
4,FinancialServiceAccount,3,"[Holding, Person, Transaction]","[account_has_holding, account_records_transact..."
5,Holding,2,"[FinancialServiceAccount, ListedSecurity]","[account_has_holding, holding_refers_to_listed..."
6,ReportElement,2,"[FinancialReport, StatementElement]","[financial_report_contains_report_element, rep..."
7,BuyTransaction,1,[Transaction],[buy_transaction_is_transaction]
8,Country,1,[Corporation],[corporation_has_country]
9,Disclosure,1,[FinancialReport],[financial_report_contains_disclosure]



Entities present in the conceptual graph:
['BuyTransaction', 'Corporation', 'Country', 'Disclosure', 'FinancialReport', 'FinancialServiceAccount', 'Holding', 'Industry', 'ListedSecurity', 'Person', 'ReportElement', 'Security', 'SellTransaction', 'StatementElement', 'Transaction']
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v01/conceptual_relationships.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v01/conceptual_bidirectional_edges.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v01/conceptual_adjacency.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v01/conceptual_relationship_summary.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v01/con

In [44]:
# =========================================================
# BLOCK V2 — EXTRACT Rc(Qi) EXACTLY
# =========================================================
#
# Purpose:
# This block computes Rc(Qi) for each FIBEN workload query.
#
# Rc(Qi) means:
# the minimum number of conceptual relationships required to
# connect all conceptual entities touched by query Qi.
#
# Example:
# If a query touches only Corporation:
#
#     Rc(Q1) = 0
#
# If a query touches Corporation, FinancialReport, and ReportElement,
# Rc(Qi) is the minimum number of conceptual edges needed to connect
# those entities in the conceptual graph.
#
# Main output preserved for downstream compatibility:
# - rc_df
#
# Additional outputs:
# - rc_selected_edges_long_df
# - rc_summary_df
# - rc_disconnected_queries_df
#
# Methodological role:
# Rc(Qi) is the baseline conceptual relationship cost before selecting
# a document root and embedding depth.
# Later blocks compare Rc(Qi) with Re(Qi, er, D) to compute DeltaR.
# =========================================================

from itertools import combinations
from collections import defaultdict, deque
import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "workload_conceptual_df",
    "conceptual_edges",
    "conceptual_relationships_df",
    "fiben_scenario",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first INSTANCE 1, BLOCK V0, and BLOCK V1."
        )


# ---------------------------------------------------------
# 2. Helper: ensure list format
# ---------------------------------------------------------

def ensure_entity_list(value):
    """
    Convert a value into a clean list of entity names.

    This protects the methodology when entities are stored as:
    - a list;
    - a tuple;
    - a single string;
    - a missing value.
    """
    if isinstance(value, list):
        return [str(v) for v in value]

    if isinstance(value, tuple):
        return [str(v) for v in value]

    if value is None:
        return []

    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    return [str(value)]


# ---------------------------------------------------------
# 3. Helper: check whether terminals are connected
# ---------------------------------------------------------

def terminals_are_connected(selected_edges, terminals):
    """
    Return True if all terminal entities are connected in the
    subgraph formed by selected_edges.

    The conceptual graph is treated as undirected for connectivity.
    """
    terminals = list(dict.fromkeys(terminals))

    if len(terminals) <= 1:
        return True

    adjacency = defaultdict(set)

    for edge in selected_edges:
        adjacency[edge["source"]].add(edge["target"])
        adjacency[edge["target"]].add(edge["source"])

    start = terminals[0]
    visited = {start}
    queue = deque([start])

    while queue:
        current = queue.popleft()

        for neighbor in adjacency[current]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    return all(t in visited for t in terminals)


# ---------------------------------------------------------
# 4. Helper: extract Rc for one query
# ---------------------------------------------------------

def extract_rc_for_query(terminals, conceptual_edges):
    """
    Compute Rc(Qi) for one query.

    Parameters
    ----------
    terminals:
        Conceptual entities touched by the query.

    conceptual_edges:
        Conceptual relationship edges available in the scenario.

    Returns
    -------
    dict
        Dictionary with:
        - rc;
        - selected_edge_ids;
        - status.
    """
    terminals = list(dict.fromkeys(ensure_entity_list(terminals)))

    if len(terminals) <= 1:
        return {
            "rc": 0,
            "selected_edge_ids": [],
            "status": "single_or_no_terminal",
        }

    all_graph_entities = set()

    for edge in conceptual_edges:
        all_graph_entities.add(edge["source"])
        all_graph_entities.add(edge["target"])

    missing_terminals = sorted(set(terminals) - all_graph_entities)

    if missing_terminals:
        return {
            "rc": None,
            "selected_edge_ids": [],
            "status": f"missing_terminals: {missing_terminals}",
        }

    # Test edge subsets in increasing size.
    # The first subset connecting all terminals is the exact minimum.
    for k in range(1, len(conceptual_edges) + 1):
        for subset in combinations(conceptual_edges, k):
            nodes_in_subset = set()

            for edge in subset:
                nodes_in_subset.add(edge["source"])
                nodes_in_subset.add(edge["target"])

            # Pruning:
            # if the subset does not contain all terminals, it cannot connect them.
            if not set(terminals).issubset(nodes_in_subset):
                continue

            if terminals_are_connected(subset, terminals):
                return {
                    "rc": k,
                    "selected_edge_ids": [edge["edge_id"] for edge in subset],
                    "status": "connected",
                }

    return {
        "rc": None,
        "selected_edge_ids": [],
        "status": "not_connected",
    }


# ---------------------------------------------------------
# 5. Apply Rc extraction to each query
# ---------------------------------------------------------

rc_rows = []
rc_selected_edge_rows = []

for _, row in workload_conceptual_df.iterrows():
    query_name = row["query_name"]
    query_type = row["query_type"]
    generic_class = row.get("generic_class", None)
    query_family = row.get("query_family", None)
    terminals = ensure_entity_list(row["entities_touched"])

    rc_info = extract_rc_for_query(
        terminals=terminals,
        conceptual_edges=conceptual_edges,
    )

    selected_edge_ids = rc_info["selected_edge_ids"]

    rc_rows.append({
        "query_name": query_name,
        "generic_class": generic_class,
        "query_family": query_family,
        "query_type": query_type,
        "entities_touched": terminals,
        "n_entities_touched": len(terminals),
        "Rc": rc_info["rc"],
        "Rc_selected_edges": selected_edge_ids,
        "Rc_status": rc_info["status"],
    })

    for edge_position, edge_id in enumerate(selected_edge_ids, start=1):
        rc_selected_edge_rows.append({
            "query_name": query_name,
            "generic_class": generic_class,
            "query_family": query_family,
            "edge_position": edge_position,
            "edge_id": edge_id,
        })


rc_df = pd.DataFrame(rc_rows)
rc_selected_edges_long_df = pd.DataFrame(rc_selected_edge_rows)


# ---------------------------------------------------------
# 6. Build summary tables
# ---------------------------------------------------------

rc_summary_df = (
    rc_df
    .groupby(["generic_class", "query_family"], dropna=False)
    .agg(
        n_queries=("query_name", "count"),
        avg_Rc=("Rc", "mean"),
        min_Rc=("Rc", "min"),
        max_Rc=("Rc", "max"),
        avg_entities_touched=("n_entities_touched", "mean"),
    )
    .reset_index()
)

rc_disconnected_queries_df = rc_df[
    rc_df["Rc"].isna() | (rc_df["Rc_status"] != "connected")
].copy()

# For single-entity queries, Rc = 0 is expected and not a problem.
rc_problem_queries_df = rc_df[
    rc_df["Rc"].isna()
].copy()


# ---------------------------------------------------------
# 7. Display outputs
# ---------------------------------------------------------

print("Rc(Qi) extraction completed.")

print("\nRc by query:")
display(rc_df)

print("\nSelected Rc edges in long format:")
display(rc_selected_edges_long_df)

print("\nRc summary by generic class:")
display(rc_summary_df)

if not rc_problem_queries_df.empty:
    print("\nWarning: some queries could not be connected in the conceptual graph.")
    display(rc_problem_queries_df)
else:
    print("\nAll multi-entity queries were connected successfully.")


# ---------------------------------------------------------
# 8. Save BLOCK V2 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    rc_df,
    "variables/block_v02/rc_by_query.csv",
)

save_dataframe(
    rc_selected_edges_long_df,
    "variables/block_v02/rc_selected_edges_long.csv",
)

save_dataframe(
    rc_summary_df,
    "variables/block_v02/rc_summary_by_generic_class.csv",
)

save_dataframe(
    rc_disconnected_queries_df,
    "variables/block_v02/rc_disconnected_queries.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "n_queries": len(rc_df),
        "n_queries_with_rc_zero": int((rc_df["Rc"] == 0).sum()),
        "n_queries_with_missing_rc": int(rc_df["Rc"].isna().sum()),
        "max_rc": None if rc_df["Rc"].dropna().empty else int(rc_df["Rc"].dropna().max()),
        "rc_status_counts": rc_df["Rc_status"].value_counts(dropna=False).to_dict(),
        "output_files": {
            "rc_by_query_csv": "variables/block_v02/rc_by_query.csv",
            "rc_selected_edges_long_csv": "variables/block_v02/rc_selected_edges_long.csv",
            "rc_summary_by_generic_class_csv": "variables/block_v02/rc_summary_by_generic_class.csv",
            "rc_disconnected_queries_csv": "variables/block_v02/rc_disconnected_queries.csv",
        },
    },
    "variables/block_v02/rc_metadata.json",
)

write_execution_log(
    block_name="BLOCK V2 — EXTRACT Rc(Qi) EXACTLY",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_queries": len(rc_df),
        "n_queries_with_rc_zero": int((rc_df["Rc"] == 0).sum()),
        "n_queries_with_missing_rc": int(rc_df["Rc"].isna().sum()),
        "rc_by_query_csv": "variables/block_v02/rc_by_query.csv",
        "rc_selected_edges_long_csv": "variables/block_v02/rc_selected_edges_long.csv",
        "rc_summary_by_generic_class_csv": "variables/block_v02/rc_summary_by_generic_class.csv",
        "rc_disconnected_queries_csv": "variables/block_v02/rc_disconnected_queries.csv",
        "metadata_json": "variables/block_v02/rc_metadata.json",
    },
)


# ---------------------------------------------------------
# 9. Update README section for BLOCK V2
# ---------------------------------------------------------

max_rc_value = None if rc_df["Rc"].dropna().empty else int(rc_df["Rc"].dropna().max())

block_v02_readme_body = f"""
This block computes Rc(Qi) for each FIBEN workload query.

Rc(Qi) is the minimum number of conceptual relationships required to connect
all conceptual entities touched by a query.

Main variable created:

    rc_df

Additional variables created:

    rc_selected_edges_long_df
    rc_summary_df
    rc_disconnected_queries_df

Number of queries processed:

    {len(rc_df)}

Number of queries with Rc = 0:

    {int((rc_df["Rc"] == 0).sum())}

Number of queries with missing Rc:

    {int(rc_df["Rc"].isna().sum())}

Maximum Rc value:

    {max_rc_value}

Generated reproducibility files:

    variables/block_v02/rc_by_query.csv
    variables/block_v02/rc_selected_edges_long.csv
    variables/block_v02/rc_summary_by_generic_class.csv
    variables/block_v02/rc_disconnected_queries.csv
    variables/block_v02/rc_metadata.json

Methodological meaning:

    Rc(Qi) represents the original conceptual relationship cost of each query
    before selecting a document root and embedding depth.

    Later blocks compare Rc(Qi) with the number of relationships that remain
    outside the selected document configuration. This comparison is used to
    compute DeltaR and DeltaRratio.

Important note:

    Single-entity queries have Rc = 0 because no relationship traversal is
    required to connect their touched entities.
"""

update_readme_section(
    section_title="BLOCK V2 — Extract Rc(Qi) Exactly",
    section_body=block_v02_readme_body,
)

Rc(Qi) extraction completed.

Rc by query:


,query_name,generic_class,query_family,query_type,entities_touched,n_entities_touched,Rc,Rc_selected_edges,Rc_status
0,Q1_CompanyProfileIBM,QG1,Local lookup,select,[Corporation],1,0,[],single_or_no_terminal
1,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,"[Corporation, Industry, Country, ListedSecurity]",4,3,[Corporation -- Industry [corporation_has_indu...,connected
2,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,"[FinancialServiceAccount, Holding, ListedSecur...",4,3,[ListedSecurity -- Security [listed_security_r...,connected
3,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,"[Person, FinancialServiceAccount, Holding, Lis...",5,4,[Corporation -- ListedSecurity [corporation_ha...,connected
4,Q5_ReportsAndMetricDataOfCompany,QG5,Containment retrieval,select,"[Corporation, FinancialReport, ReportElement, ...",5,4,[Corporation -- FinancialReport [corporation_h...,connected
5,Q6_TechUSListedSecuritiesWithHighLastTradedValue,QG6,Filtered search / recommendation,select,"[Corporation, Industry, Country, ListedSecurity]",4,3,[Corporation -- Industry [corporation_has_indu...,connected
6,Q7_PersonsWhoBoughtMoreIBMThanSold,QG7,Aggregation / ranking,select,"[Person, FinancialServiceAccount, BuyTransacti...",7,6,[Corporation -- ListedSecurity [corporation_ha...,connected
7,Q8_IBMTransactionsBelowAverageSellingPrice,QG8,Aggregation / ranking,select,"[Transaction, SellTransaction, ListedSecurity,...",4,3,[Corporation -- ListedSecurity [corporation_ha...,connected
8,Q9_PersonsWhoBoughtAndSoldSameStock,QG9,Associative retrieval + comparison,select,"[Person, FinancialServiceAccount, BuyTransacti...",6,5,[Person -- FinancialServiceAccount [person_own...,connected
9,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,"[Person, FinancialServiceAccount, Holding, Lis...",6,5,[Person -- FinancialServiceAccount [person_own...,connected



Selected Rc edges in long format:


,query_name,generic_class,query_family,edge_position,edge_id
0,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,1,Corporation -- Industry [corporation_has_indus...
1,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,2,Corporation -- Country [corporation_has_country]
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,3,Corporation -- ListedSecurity [corporation_has...
3,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,1,ListedSecurity -- Security [listed_security_re...
4,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,2,FinancialServiceAccount -- Holding [account_ha...
5,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,3,Holding -- ListedSecurity [holding_refers_to_l...
6,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,1,Corporation -- ListedSecurity [corporation_has...
7,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,2,Person -- FinancialServiceAccount [person_owns...
8,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,3,FinancialServiceAccount -- Holding [account_ha...
9,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,4,Holding -- ListedSecurity [holding_refers_to_l...



Rc summary by generic class:


,generic_class,query_family,n_queries,avg_Rc,min_Rc,max_Rc,avg_entities_touched
0,QG1,Local lookup,1,0.0,0,0,1.0
1,QG10,Update / insertion with relationship creation,1,5.0,5,5,6.0
2,QG2,Shallow rooted retrieval,1,3.0,3,3,4.0
3,QG3,Associative retrieval,1,3.0,3,3,4.0
4,QG4,Deep traversal,1,4.0,4,4,5.0
5,QG5,Containment retrieval,1,4.0,4,4,5.0
6,QG6,Filtered search / recommendation,1,3.0,3,3,4.0
7,QG7,Aggregation / ranking,1,6.0,6,6,7.0
8,QG8,Aggregation / ranking,1,3.0,3,3,4.0
9,QG9,Associative retrieval + comparison,1,5.0,5,5,6.0



All multi-entity queries were connected successfully.
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v02/rc_by_query.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v02/rc_selected_edges_long.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v02/rc_summary_by_generic_class.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v02/rc_disconnected_queries.csv
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v02/rc_metadata.json
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/logs/block_v2___extract_rc(qi)_exactly_execution_log.json
Updated README section: BLOCK V2 — Extract Rc(Qi) Exactly
README path: /home/jovyan/priva

In [46]:
# =========================================================
# BLOCK V3 — INITIAL ROOT CANDIDATES er
# =========================================================
#
# Purpose:
# This block defines the initial candidate document roots er
# for each FIBEN workload query.
#
# In this implementation, the initial root candidates are the
# conceptual entities explicitly touched by each query.
#
# Example:
# If Q5 touches:
#
#     Corporation, FinancialReport, ReportElement,
#     StatementElement, Disclosure
#
# then the initial root candidates for Q5 are those entities.
#
# Main outputs preserved for downstream compatibility:
# - root_candidates_df
# - root_candidates_by_query
#
# Additional outputs:
# - root_candidate_summary_df
# - root_candidate_entity_frequency_df
#
# Methodological role:
# These candidates will be used in the next block to choose one
# selected root per query.
# =========================================================

import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "workload_conceptual_df",
    "rc_df",
    "conceptual_entity_degree_df",
    "fiben_scenario",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first INSTANCE 1, BLOCK V0, BLOCK V1, and BLOCK V2."
        )


# ---------------------------------------------------------
# 2. Helper: ensure entity list format
# ---------------------------------------------------------

def normalize_entity_list(value):
    """
    Convert a value into a clean list of conceptual entities.

    This protects the methodology when entities are stored as:
    - a list;
    - a tuple;
    - a single string;
    - a missing value.
    """
    if isinstance(value, list):
        return [str(v) for v in value]

    if isinstance(value, tuple):
        return [str(v) for v in value]

    if value is None:
        return []

    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    return [str(value)]


# ---------------------------------------------------------
# 3. Prepare entity degree lookup
# ---------------------------------------------------------
#
# Entity degree is not used to decide the final root here.
# It is only recorded as useful metadata for inspection.
# ---------------------------------------------------------

entity_degree_lookup = {}

if "conceptual_entity_degree_df" in globals() and not conceptual_entity_degree_df.empty:
    entity_degree_lookup = dict(
        zip(
            conceptual_entity_degree_df["entity"],
            conceptual_entity_degree_df["degree"],
        )
    )


# ---------------------------------------------------------
# 4. Generate initial root candidates
# ---------------------------------------------------------

root_candidate_rows = []

for _, row in workload_conceptual_df.iterrows():
    query_name = row["query_name"]
    query_type = row["query_type"]
    generic_class = row.get("generic_class", None)
    query_family = row.get("query_family", None)
    entities_touched = normalize_entity_list(row["entities_touched"])

    # Remove duplicates while preserving order.
    candidate_roots = list(dict.fromkeys(entities_touched))

    for root_position, candidate_root in enumerate(candidate_roots, start=1):
        root_candidate_rows.append({
            "query_name": query_name,
            "generic_class": generic_class,
            "query_family": query_family,
            "query_type": query_type,
            "candidate_root": candidate_root,
            "root_position": root_position,
            "is_touched_entity": True,
            "n_entities_touched": len(entities_touched),
            "candidate_entity_degree": entity_degree_lookup.get(candidate_root, 0),
            "entities_touched": entities_touched,
        })


root_candidates_df = pd.DataFrame(root_candidate_rows)


# ---------------------------------------------------------
# 5. Create a dictionary for quick access
# ---------------------------------------------------------
#
# This object maps:
#
#     query_name -> [candidate roots]
#
# It is useful for later blocks.
# ---------------------------------------------------------

root_candidates_by_query = (
    root_candidates_df
    .groupby("query_name")["candidate_root"]
    .apply(list)
    .to_dict()
)


# ---------------------------------------------------------
# 6. Add Rc information to the candidate table
# ---------------------------------------------------------

rc_lookup_df = rc_df[
    ["query_name", "Rc", "Rc_status"]
].copy()

root_candidates_df = root_candidates_df.merge(
    rc_lookup_df,
    on="query_name",
    how="left",
)


# ---------------------------------------------------------
# 7. Build summary tables
# ---------------------------------------------------------

root_candidate_summary_df = (
    root_candidates_df
    .groupby(["query_name", "generic_class", "query_family", "query_type"], dropna=False)
    .agg(
        n_candidate_roots=("candidate_root", "count"),
        candidate_roots=("candidate_root", list),
        n_entities_touched=("n_entities_touched", "max"),
        Rc=("Rc", "max"),
        Rc_status=("Rc_status", "first"),
    )
    .reset_index()
)

root_candidate_entity_frequency_df = (
    root_candidates_df
    .groupby("candidate_root", as_index=False)
    .agg(
        n_queries_as_candidate=("query_name", "nunique"),
        query_names=("query_name", lambda s: sorted(set(s))),
        avg_candidate_entity_degree=("candidate_entity_degree", "mean"),
    )
    .sort_values(
        ["n_queries_as_candidate", "candidate_root"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 8. Display outputs
# ---------------------------------------------------------

print("Initial root candidates er were generated successfully.")

print("\nRoot candidates by query:")
display(root_candidates_df)

print("\nRoot candidate summary:")
display(root_candidate_summary_df)

print("\nRoot candidate entity frequency:")
display(root_candidate_entity_frequency_df)


# ---------------------------------------------------------
# 9. Save BLOCK V3 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    root_candidates_df,
    "variables/block_v03/root_candidates.csv",
)

save_dataframe(
    root_candidate_summary_df,
    "variables/block_v03/root_candidate_summary.csv",
)

save_dataframe(
    root_candidate_entity_frequency_df,
    "variables/block_v03/root_candidate_entity_frequency.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "n_queries": int(root_candidate_summary_df["query_name"].nunique()),
        "n_root_candidate_rows": len(root_candidates_df),
        "root_candidates_by_query": root_candidates_by_query,
        "unique_candidate_roots": sorted(root_candidates_df["candidate_root"].unique().tolist()),
        "output_files": {
            "root_candidates_csv": "variables/block_v03/root_candidates.csv",
            "root_candidate_summary_csv": "variables/block_v03/root_candidate_summary.csv",
            "root_candidate_entity_frequency_csv": "variables/block_v03/root_candidate_entity_frequency.csv",
        },
    },
    "variables/block_v03/root_candidates_metadata.json",
)

write_execution_log(
    block_name="BLOCK V3 — INITIAL ROOT CANDIDATES er",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_queries": int(root_candidate_summary_df["query_name"].nunique()),
        "n_root_candidate_rows": len(root_candidates_df),
        "root_candidates_csv": "variables/block_v03/root_candidates.csv",
        "root_candidate_summary_csv": "variables/block_v03/root_candidate_summary.csv",
        "root_candidate_entity_frequency_csv": "variables/block_v03/root_candidate_entity_frequency.csv",
        "metadata_json": "variables/block_v03/root_candidates_metadata.json",
    },
)


# ---------------------------------------------------------
# 10. Update README section for BLOCK V3
# ---------------------------------------------------------

block_v03_readme_body = f"""
This block defines the initial candidate document roots er for each FIBEN workload query.

Main variable created:

    root_candidates_df

Additional variables created:

    root_candidates_by_query
    root_candidate_summary_df
    root_candidate_entity_frequency_df

Number of queries processed:

    {int(root_candidate_summary_df["query_name"].nunique())}

Number of root candidate rows:

    {len(root_candidates_df)}

Unique candidate roots:

    {sorted(root_candidates_df["candidate_root"].unique().tolist())}

Generated reproducibility files:

    variables/block_v03/root_candidates.csv
    variables/block_v03/root_candidate_summary.csv
    variables/block_v03/root_candidate_entity_frequency.csv
    variables/block_v03/root_candidates_metadata.json

Methodological meaning:

    er represents a candidate document root entity for a query.

    In this first candidate generation step, every entity explicitly touched
    by a query is considered a possible document root.

Why this is important:

    The next block selects one root per query. Later blocks compute the
    conceptual distance D(E, er, Qi) from each selected root to the other
    entities touched by the query.
"""

update_readme_section(
    section_title="BLOCK V3 — Initial Root Candidates er",
    section_body=block_v03_readme_body,
)

Initial root candidates er were generated successfully.

Root candidates by query:


,query_name,generic_class,query_family,query_type,candidate_root,root_position,is_touched_entity,n_entities_touched,candidate_entity_degree,entities_touched,Rc,Rc_status
0,Q1_CompanyProfileIBM,QG1,Local lookup,select,Corporation,1,True,1,4,[Corporation],0,single_or_no_terminal
1,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Corporation,1,True,4,4,"[Corporation, Industry, Country, ListedSecurity]",3,connected
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Industry,2,True,4,1,"[Corporation, Industry, Country, ListedSecurity]",3,connected
3,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Country,3,True,4,1,"[Corporation, Industry, Country, ListedSecurity]",3,connected
4,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,ListedSecurity,4,True,4,4,"[Corporation, Industry, Country, ListedSecurity]",3,connected
5,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount,1,True,4,3,"[FinancialServiceAccount, Holding, ListedSecur...",3,connected
6,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,Holding,2,True,4,2,"[FinancialServiceAccount, Holding, ListedSecur...",3,connected
7,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,ListedSecurity,3,True,4,4,"[FinancialServiceAccount, Holding, ListedSecur...",3,connected
8,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,Security,4,True,4,1,"[FinancialServiceAccount, Holding, ListedSecur...",3,connected
9,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Person,1,True,5,1,"[Person, FinancialServiceAccount, Holding, Lis...",4,connected



Root candidate summary:


,query_name,generic_class,query_family,query_type,n_candidate_roots,candidate_roots,n_entities_touched,Rc,Rc_status
0,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,6,"[Person, FinancialServiceAccount, Holding, Lis...",6,5,connected
1,Q1_CompanyProfileIBM,QG1,Local lookup,select,1,[Corporation],1,0,single_or_no_terminal
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,4,"[Corporation, Industry, Country, ListedSecurity]",4,3,connected
3,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,4,"[FinancialServiceAccount, Holding, ListedSecur...",4,3,connected
4,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,5,"[Person, FinancialServiceAccount, Holding, Lis...",5,4,connected
5,Q5_ReportsAndMetricDataOfCompany,QG5,Containment retrieval,select,5,"[Corporation, FinancialReport, ReportElement, ...",5,4,connected
6,Q6_TechUSListedSecuritiesWithHighLastTradedValue,QG6,Filtered search / recommendation,select,4,"[Corporation, Industry, Country, ListedSecurity]",4,3,connected
7,Q7_PersonsWhoBoughtMoreIBMThanSold,QG7,Aggregation / ranking,select,7,"[Person, FinancialServiceAccount, BuyTransacti...",7,6,connected
8,Q8_IBMTransactionsBelowAverageSellingPrice,QG8,Aggregation / ranking,select,4,"[Transaction, SellTransaction, ListedSecurity,...",4,3,connected
9,Q9_PersonsWhoBoughtAndSoldSameStock,QG9,Associative retrieval + comparison,select,6,"[Person, FinancialServiceAccount, BuyTransacti...",6,5,connected



Root candidate entity frequency:


,candidate_root,n_queries_as_candidate,query_names,avg_candidate_entity_degree
0,ListedSecurity,8,"[Q10_CreateAccountHoldingAndBuyTransaction, Q2...",4.0
1,Corporation,7,"[Q1_CompanyProfileIBM, Q2_CompanyWithIndustryC...",4.0
2,FinancialServiceAccount,5,"[Q10_CreateAccountHoldingAndBuyTransaction, Q3...",3.0
3,Person,4,"[Q10_CreateAccountHoldingAndBuyTransaction, Q4...",1.0
4,Transaction,4,"[Q10_CreateAccountHoldingAndBuyTransaction, Q7...",4.0
5,BuyTransaction,3,"[Q10_CreateAccountHoldingAndBuyTransaction, Q7...",1.0
6,Holding,3,"[Q10_CreateAccountHoldingAndBuyTransaction, Q3...",2.0
7,SellTransaction,3,"[Q7_PersonsWhoBoughtMoreIBMThanSold, Q8_IBMTra...",1.0
8,Country,2,[Q2_CompanyWithIndustryCountryAndListedSecurit...,1.0
9,Industry,2,[Q2_CompanyWithIndustryCountryAndListedSecurit...,1.0


Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v03/root_candidates.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v03/root_candidate_summary.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v03/root_candidate_entity_frequency.csv
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v03/root_candidates_metadata.json
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/logs/block_v3___initial_root_candidates_er_execution_log.json
Updated README section: BLOCK V3 — Initial Root Candidates er
README path: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/README_FIBEN_Implementation_Framework.md


In [47]:
# =========================================================
# BLOCK V4 — DEFINE THE ROOT CHOSEN BY QUERY
# =========================================================
#
# Purpose:
# This block selects one document root entity for each FIBEN
# workload query.
#
# Input:
# - workload_conceptual_df
# - root_candidates_df
# - root_candidates_by_query
#
# Main outputs preserved for downstream compatibility:
# - selected_root_df
# - selected_root_by_query
# - root_chosen_by_query
#
# Methodological role:
# The selected root is used by later blocks to compute:
#
#     D(E, er, Qi)
#
# where:
# - E is an entity touched by query Qi;
# - er is the selected root entity;
# - Qi is the query.
#
# The selected root also guides the interpretation of document
# configurations generated later in the methodology.
# =========================================================

import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "workload_conceptual_df",
    "root_candidates_df",
    "root_candidates_by_query",
    "fiben_scenario",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first INSTANCE 1, BLOCK V0, BLOCK V1, BLOCK V2, and BLOCK V3."
        )


# ---------------------------------------------------------
# 2. Define selected roots for the FIBEN workload
# ---------------------------------------------------------
#
# This is a methodological decision.
#
# The selected root is not necessarily the only possible root.
# It is the root chosen for the first execution of the methodology.
#
# Alternative roots can be tested later as additional configurations.
# ---------------------------------------------------------

selected_root_decisions = {
    "Q1_CompanyProfileIBM": {
        "selected_root": "Corporation",
        "selection_strategy": "query_subject",
        "selection_reason": (
            "The query is a local lookup over the profile of one company, "
            "represented as Corporation in the conceptual schema."
        ),
    },
    "Q2_CompanyWithIndustryCountryAndListedSecurities": {
        "selected_root": "Corporation",
        "selection_strategy": "query_subject",
        "selection_reason": (
            "The query starts from IBM and retrieves directly related "
            "descriptor and listed-security information."
        ),
    },
    "Q3_SecuritiesHeldInEachFinancialServiceAccount": {
        "selected_root": "FinancialServiceAccount",
        "selection_strategy": "container_entity",
        "selection_reason": (
            "The query asks for securities held inside each financial "
            "service account, so the account is the natural container root."
        ),
    },
    "Q4_CompaniesReachedFromPersonThroughAccountHoldingListedSecurity": {
        "selected_root": "Person",
        "selection_strategy": "traversal_start",
        "selection_reason": (
            "The query starts from a person and traverses account, holding, "
            "and listed security to reach companies."
        ),
    },
    "Q5_ReportsAndMetricDataOfCompany": {
        "selected_root": "Corporation",
        "selection_strategy": "query_subject",
        "selection_reason": (
            "The query starts from a company and retrieves its financial "
            "reports and contained metric data."
        ),
    },
    "Q6_TechUSListedSecuritiesWithHighLastTradedValue": {
        "selected_root": "ListedSecurity",
        "selection_strategy": "result_entity",
        "selection_reason": (
            "The query returns listed securities filtered by company industry, "
            "country, and traded-value predicates."
        ),
    },
    "Q7_PersonsWhoBoughtMoreIBMThanSold": {
        "selected_root": "Person",
        "selection_strategy": "result_entity",
        "selection_reason": (
            "The query returns persons after aggregating buy and sell activity "
            "for IBM stock."
        ),
    },
    "Q8_IBMTransactionsBelowAverageSellingPrice": {
        "selected_root": "Transaction",
        "selection_strategy": "result_entity",
        "selection_reason": (
            "The query returns transactions for IBM whose price is below the "
            "average selling price."
        ),
    },
    "Q9_PersonsWhoBoughtAndSoldSameStock": {
        "selected_root": "Person",
        "selection_strategy": "result_entity",
        "selection_reason": (
            "The query returns persons who bought and sold the same stock."
        ),
    },
    "Q10_CreateAccountHoldingAndBuyTransaction": {
        "selected_root": "Person",
        "selection_strategy": "update_owner",
        "selection_reason": (
            "The insertion creates a financial service account, holding, and "
            "buy transaction associated with a person."
        ),
    },
}


# ---------------------------------------------------------
# 3. Validate that every workload query has one selected root
# ---------------------------------------------------------

workload_query_names = set(workload_conceptual_df["query_name"].tolist())
decision_query_names = set(selected_root_decisions.keys())

missing_root_decisions = sorted(workload_query_names - decision_query_names)
extra_root_decisions = sorted(decision_query_names - workload_query_names)

if missing_root_decisions:
    raise ValueError(
        "Some workload queries do not have a selected root decision: "
        f"{missing_root_decisions}"
    )

if extra_root_decisions:
    print(
        "Warning: some selected root decisions do not correspond to workload queries:"
    )
    print(extra_root_decisions)


# ---------------------------------------------------------
# 4. Build selected root table
# ---------------------------------------------------------

selected_root_rows = []

for _, row in workload_conceptual_df.iterrows():
    query_name = row["query_name"]
    decision = selected_root_decisions[query_name]

    entities_touched = row["entities_touched"]
    candidate_roots = root_candidates_by_query.get(query_name, [])

    selected_root = decision["selected_root"]

    selected_root_rows.append({
        "query_name": query_name,
        "generic_class": row.get("generic_class", None),
        "query_family": row.get("query_family", None),
        "query_type": row.get("query_type", None),
        "selected_root": selected_root,
        "selection_strategy": decision["selection_strategy"],
        "selection_reason": decision["selection_reason"],
        "entities_touched": entities_touched,
        "candidate_roots": candidate_roots,
        "selected_root_is_candidate": selected_root in candidate_roots,
        "selected_root_is_touched": selected_root in entities_touched,
        "n_entities_touched": len(entities_touched),
    })

selected_root_df = pd.DataFrame(selected_root_rows)


# ---------------------------------------------------------
# 5. Add Rc information when available
# ---------------------------------------------------------

if "rc_df" in globals():
    selected_root_df = selected_root_df.merge(
        rc_df[["query_name", "Rc", "Rc_status"]],
        on="query_name",
        how="left",
    )


# ---------------------------------------------------------
# 6. Validate selected roots against candidates
# ---------------------------------------------------------

invalid_selected_roots_df = selected_root_df[
    selected_root_df["selected_root_is_candidate"] == False
].copy()

if not invalid_selected_roots_df.empty:
    print(
        "Warning: some selected roots are not listed as initial candidates."
    )
    display(invalid_selected_roots_df)


# ---------------------------------------------------------
# 7. Create dictionaries expected by downstream blocks
# ---------------------------------------------------------

selected_root_by_query = dict(
    zip(
        selected_root_df["query_name"],
        selected_root_df["selected_root"],
    )
)

# Compatibility alias used by some versions of the notebook.
root_chosen_by_query = selected_root_by_query

# Additional aliases for safety and readability.
root_selection_df = selected_root_df
chosen_roots_df = selected_root_df


# ---------------------------------------------------------
# 8. Create summary tables
# ---------------------------------------------------------

selected_root_summary_df = (
    selected_root_df
    .groupby(["selected_root", "selection_strategy"], dropna=False)
    .agg(
        n_queries=("query_name", "count"),
        query_names=("query_name", lambda s: sorted(set(s))),
        avg_entities_touched=("n_entities_touched", "mean"),
    )
    .reset_index()
    .sort_values(["n_queries", "selected_root"], ascending=[False, True])
    .reset_index(drop=True)
)

selected_root_by_generic_class_df = (
    selected_root_df
    .groupby(["generic_class", "query_family", "selected_root"], dropna=False)
    .agg(
        n_queries=("query_name", "count"),
        query_names=("query_name", lambda s: sorted(set(s))),
    )
    .reset_index()
    .sort_values(["generic_class", "selected_root"])
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 9. Display outputs
# ---------------------------------------------------------

print("Selected roots were defined successfully.")

print("\nSelected root by query:")
display(selected_root_df)

print("\nSelected root summary:")
display(selected_root_summary_df)

print("\nSelected root by generic class:")
display(selected_root_by_generic_class_df)

if invalid_selected_roots_df.empty:
    print("\nAll selected roots are valid initial candidates.")


# ---------------------------------------------------------
# 10. Save BLOCK V4 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    selected_root_df,
    "variables/block_v04/selected_root_by_query.csv",
)

save_dataframe(
    selected_root_summary_df,
    "variables/block_v04/selected_root_summary.csv",
)

save_dataframe(
    selected_root_by_generic_class_df,
    "variables/block_v04/selected_root_by_generic_class.csv",
)

save_dataframe(
    invalid_selected_roots_df,
    "variables/block_v04/invalid_selected_roots.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "selected_root_by_query": selected_root_by_query,
        "selected_root_decisions": selected_root_decisions,
        "n_queries": len(selected_root_df),
        "n_invalid_selected_roots": len(invalid_selected_roots_df),
        "unique_selected_roots": sorted(
            selected_root_df["selected_root"].unique().tolist()
        ),
        "output_files": {
            "selected_root_by_query_csv": "variables/block_v04/selected_root_by_query.csv",
            "selected_root_summary_csv": "variables/block_v04/selected_root_summary.csv",
            "selected_root_by_generic_class_csv": "variables/block_v04/selected_root_by_generic_class.csv",
            "invalid_selected_roots_csv": "variables/block_v04/invalid_selected_roots.csv",
        },
    },
    "variables/block_v04/selected_root_metadata.json",
)

write_execution_log(
    block_name="BLOCK V4 — DEFINE THE ROOT CHOSEN BY QUERY",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_queries": len(selected_root_df),
        "n_invalid_selected_roots": len(invalid_selected_roots_df),
        "selected_root_by_query_csv": "variables/block_v04/selected_root_by_query.csv",
        "selected_root_summary_csv": "variables/block_v04/selected_root_summary.csv",
        "selected_root_by_generic_class_csv": "variables/block_v04/selected_root_by_generic_class.csv",
        "invalid_selected_roots_csv": "variables/block_v04/invalid_selected_roots.csv",
        "metadata_json": "variables/block_v04/selected_root_metadata.json",
    },
)


# ---------------------------------------------------------
# 11. Update README section for BLOCK V4
# ---------------------------------------------------------

block_v04_readme_body = f"""
This block selects one document root entity for each FIBEN workload query.

Main variable created:

    selected_root_df

Compatibility variables created:

    selected_root_by_query
    root_chosen_by_query
    root_selection_df
    chosen_roots_df

Number of queries processed:

    {len(selected_root_df)}

Unique selected roots:

    {sorted(selected_root_df["selected_root"].unique().tolist())}

Number of invalid selected roots:

    {len(invalid_selected_roots_df)}

Generated reproducibility files:

    variables/block_v04/selected_root_by_query.csv
    variables/block_v04/selected_root_summary.csv
    variables/block_v04/selected_root_by_generic_class.csv
    variables/block_v04/invalid_selected_roots.csv
    variables/block_v04/selected_root_metadata.json

Selected root decisions:

    Q1  -> Corporation
    Q2  -> Corporation
    Q3  -> FinancialServiceAccount
    Q4  -> Person
    Q5  -> Corporation
    Q6  -> ListedSecurity
    Q7  -> Person
    Q8  -> Transaction
    Q9  -> Person
    Q10 -> Person

Methodological meaning:

    The selected root is the document root candidate used to calculate
    conceptual distances from the root to all entities touched by each query.

    The next block computes D(E, er, Qi), where er is the selected root.
"""

update_readme_section(
    section_title="BLOCK V4 — Define the Root Chosen by Query",
    section_body=block_v04_readme_body,
)

Selected roots were defined successfully.

Selected root by query:


,query_name,generic_class,query_family,query_type,selected_root,selection_strategy,selection_reason,entities_touched,candidate_roots,selected_root_is_candidate,selected_root_is_touched,n_entities_touched,Rc,Rc_status
0,Q1_CompanyProfileIBM,QG1,Local lookup,select,Corporation,query_subject,The query is a local lookup over the profile o...,[Corporation],[Corporation],True,True,1,0,single_or_no_terminal
1,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Corporation,query_subject,The query starts from IBM and retrieves direct...,"[Corporation, Industry, Country, ListedSecurity]","[Corporation, Industry, Country, ListedSecurity]",True,True,4,3,connected
2,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount,container_entity,The query asks for securities held inside each...,"[FinancialServiceAccount, Holding, ListedSecur...","[FinancialServiceAccount, Holding, ListedSecur...",True,True,4,3,connected
3,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Person,traversal_start,The query starts from a person and traverses a...,"[Person, FinancialServiceAccount, Holding, Lis...","[Person, FinancialServiceAccount, Holding, Lis...",True,True,5,4,connected
4,Q5_ReportsAndMetricDataOfCompany,QG5,Containment retrieval,select,Corporation,query_subject,The query starts from a company and retrieves ...,"[Corporation, FinancialReport, ReportElement, ...","[Corporation, FinancialReport, ReportElement, ...",True,True,5,4,connected
5,Q6_TechUSListedSecuritiesWithHighLastTradedValue,QG6,Filtered search / recommendation,select,ListedSecurity,result_entity,The query returns listed securities filtered b...,"[Corporation, Industry, Country, ListedSecurity]","[Corporation, Industry, Country, ListedSecurity]",True,True,4,3,connected
6,Q7_PersonsWhoBoughtMoreIBMThanSold,QG7,Aggregation / ranking,select,Person,result_entity,The query returns persons after aggregating bu...,"[Person, FinancialServiceAccount, BuyTransacti...","[Person, FinancialServiceAccount, BuyTransacti...",True,True,7,6,connected
7,Q8_IBMTransactionsBelowAverageSellingPrice,QG8,Aggregation / ranking,select,Transaction,result_entity,The query returns transactions for IBM whose p...,"[Transaction, SellTransaction, ListedSecurity,...","[Transaction, SellTransaction, ListedSecurity,...",True,True,4,3,connected
8,Q9_PersonsWhoBoughtAndSoldSameStock,QG9,Associative retrieval + comparison,select,Person,result_entity,The query returns persons who bought and sold ...,"[Person, FinancialServiceAccount, BuyTransacti...","[Person, FinancialServiceAccount, BuyTransacti...",True,True,6,5,connected
9,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,Person,update_owner,The insertion creates a financial service acco...,"[Person, FinancialServiceAccount, Holding, Lis...","[Person, FinancialServiceAccount, Holding, Lis...",True,True,6,5,connected



Selected root summary:


,selected_root,selection_strategy,n_queries,query_names,avg_entities_touched
0,Corporation,query_subject,3,"[Q1_CompanyProfileIBM, Q2_CompanyWithIndustryC...",3.333333
1,Person,result_entity,2,"[Q7_PersonsWhoBoughtMoreIBMThanSold, Q9_Person...",6.500000
2,FinancialServiceAccount,container_entity,1,[Q3_SecuritiesHeldInEachFinancialServiceAccount],4.000000
3,ListedSecurity,result_entity,1,[Q6_TechUSListedSecuritiesWithHighLastTradedVa...,4.000000
4,Person,traversal_start,1,[Q4_CompaniesReachedFromPersonThroughAccountHo...,5.000000
5,Person,update_owner,1,[Q10_CreateAccountHoldingAndBuyTransaction],6.000000
6,Transaction,result_entity,1,[Q8_IBMTransactionsBelowAverageSellingPrice],4.000000



Selected root by generic class:


,generic_class,query_family,selected_root,n_queries,query_names
0,QG1,Local lookup,Corporation,1,[Q1_CompanyProfileIBM]
1,QG10,Update / insertion with relationship creation,Person,1,[Q10_CreateAccountHoldingAndBuyTransaction]
2,QG2,Shallow rooted retrieval,Corporation,1,[Q2_CompanyWithIndustryCountryAndListedSecurit...
3,QG3,Associative retrieval,FinancialServiceAccount,1,[Q3_SecuritiesHeldInEachFinancialServiceAccount]
4,QG4,Deep traversal,Person,1,[Q4_CompaniesReachedFromPersonThroughAccountHo...
5,QG5,Containment retrieval,Corporation,1,[Q5_ReportsAndMetricDataOfCompany]
6,QG6,Filtered search / recommendation,ListedSecurity,1,[Q6_TechUSListedSecuritiesWithHighLastTradedVa...
7,QG7,Aggregation / ranking,Person,1,[Q7_PersonsWhoBoughtMoreIBMThanSold]
8,QG8,Aggregation / ranking,Transaction,1,[Q8_IBMTransactionsBelowAverageSellingPrice]
9,QG9,Associative retrieval + comparison,Person,1,[Q9_PersonsWhoBoughtAndSoldSameStock]



All selected roots are valid initial candidates.
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v04/selected_root_by_query.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v04/selected_root_summary.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v04/selected_root_by_generic_class.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v04/invalid_selected_roots.csv
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v04/selected_root_metadata.json
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/logs/block_v4___define_the_root_chosen_by_query_execution_log.json
Updated README section: BLOCK V4 — Define the Root Chosen by Q

In [48]:
# =========================================================
# BLOCK V5 — CALCULATE D(E, er, Qi)
# =========================================================
#
# Purpose:
# This block computes the conceptual distance D(E, er, Qi)
# between the selected root er and each entity E touched by
# query Qi.
#
# Input:
# - workload_conceptual_df
# - selected_root_df
# - selected_root_by_query
# - conceptual_adj
#
# Main outputs preserved for downstream compatibility:
# - distance_df
# - d_df
#
# Additional outputs:
# - distance_summary_by_query_df
# - distance_unreachable_entities_df
#
# Methodological role:
# D(E, er, Qi) is used later to determine which entities are
# reachable from the selected root at a given embedding depth D.
#
# Example:
# If selected_root = Corporation:
#
#     D(Corporation, Corporation, Qi) = 0
#     D(FinancialReport, Corporation, Qi) = 1
#     D(ReportElement, Corporation, Qi) = 2
# =========================================================

from collections import deque
import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "workload_conceptual_df",
    "selected_root_df",
    "selected_root_by_query",
    "conceptual_adj",
    "fiben_scenario",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first INSTANCE 1 and BLOCKS V0–V4."
        )


# ---------------------------------------------------------
# 2. Helper: normalize entity list
# ---------------------------------------------------------

def normalize_entity_list_for_distance(value):
    """
    Convert a value into a clean list of conceptual entities.

    This protects the block when entities are stored as:
    - list;
    - tuple;
    - single string;
    - missing value.
    """
    if isinstance(value, list):
        return [str(v) for v in value]

    if isinstance(value, tuple):
        return [str(v) for v in value]

    if value is None:
        return []

    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    return [str(value)]


# ---------------------------------------------------------
# 3. Shortest-path search in the conceptual graph
# ---------------------------------------------------------

def shortest_path_in_conceptual_graph(start_entity: str, target_entity: str, adjacency) -> dict:
    """
    Compute the shortest path between two conceptual entities.

    The conceptual graph is treated as undirected because conceptual
    reachability is used for methodological distance analysis.

    Parameters
    ----------
    start_entity:
        Selected root entity er.

    target_entity:
        Touched entity E.

    adjacency:
        Conceptual adjacency dictionary created in BLOCK V1.

    Returns
    -------
    dict
        Dictionary with:
        - distance;
        - path_entities;
        - path_edge_ids;
        - path_relationships;
        - status.
    """
    if start_entity == target_entity:
        return {
            "distance": 0,
            "path_entities": [start_entity],
            "path_edge_ids": [],
            "path_relationships": [],
            "status": "connected",
        }

    if start_entity not in adjacency:
        return {
            "distance": None,
            "path_entities": [],
            "path_edge_ids": [],
            "path_relationships": [],
            "status": "missing_start_entity",
        }

    if target_entity not in adjacency:
        return {
            "distance": None,
            "path_entities": [],
            "path_edge_ids": [],
            "path_relationships": [],
            "status": "missing_target_entity",
        }

    queue = deque()
    queue.append({
        "entity": start_entity,
        "path_entities": [start_entity],
        "path_edge_ids": [],
        "path_relationships": [],
    })

    visited = {start_entity}

    while queue:
        current = queue.popleft()
        current_entity = current["entity"]

        for neighbor, relationship_name, edge_id in adjacency[current_entity]:
            if neighbor in visited:
                continue

            next_path_entities = current["path_entities"] + [neighbor]
            next_path_edge_ids = current["path_edge_ids"] + [edge_id]
            next_path_relationships = current["path_relationships"] + [relationship_name]

            if neighbor == target_entity:
                return {
                    "distance": len(next_path_edge_ids),
                    "path_entities": next_path_entities,
                    "path_edge_ids": next_path_edge_ids,
                    "path_relationships": next_path_relationships,
                    "status": "connected",
                }

            visited.add(neighbor)

            queue.append({
                "entity": neighbor,
                "path_entities": next_path_entities,
                "path_edge_ids": next_path_edge_ids,
                "path_relationships": next_path_relationships,
            })

    return {
        "distance": None,
        "path_entities": [],
        "path_edge_ids": [],
        "path_relationships": [],
        "status": "unreachable",
    }


# ---------------------------------------------------------
# 4. Compute D(E, er, Qi) for every query/entity pair
# ---------------------------------------------------------

distance_rows = []

for _, row in workload_conceptual_df.iterrows():
    query_name = row["query_name"]
    generic_class = row.get("generic_class", None)
    query_family = row.get("query_family", None)
    query_type = row.get("query_type", None)

    entities_touched = normalize_entity_list_for_distance(
        row["entities_touched"]
    )

    selected_root = selected_root_by_query.get(query_name)

    if selected_root is None:
        raise ValueError(
            f"No selected root was found for query: {query_name}"
        )

    for entity in entities_touched:
        path_info = shortest_path_in_conceptual_graph(
            start_entity=selected_root,
            target_entity=entity,
            adjacency=conceptual_adj,
        )

        distance_rows.append({
            "query_name": query_name,
            "generic_class": generic_class,
            "query_family": query_family,
            "query_type": query_type,
            "selected_root": selected_root,
            "entity": entity,
            "D": path_info["distance"],
            "distance_status": path_info["status"],
            "path_entities": path_info["path_entities"],
            "path_edge_ids": path_info["path_edge_ids"],
            "path_relationships": path_info["path_relationships"],
            "path_length": (
                None
                if path_info["distance"] is None
                else int(path_info["distance"])
            ),
        })


distance_df = pd.DataFrame(distance_rows)

# Compatibility alias.
# Some versions of the notebook use d_df.
d_df = distance_df.copy()


# ---------------------------------------------------------
# 5. Add Rc information when available
# ---------------------------------------------------------

if "rc_df" in globals():
    distance_df = distance_df.merge(
        rc_df[["query_name", "Rc", "Rc_status"]],
        on="query_name",
        how="left",
    )

    d_df = distance_df.copy()


# ---------------------------------------------------------
# 6. Build summary tables
# ---------------------------------------------------------

distance_summary_by_query_df = (
    distance_df
    .groupby(
        ["query_name", "generic_class", "query_family", "query_type", "selected_root"],
        dropna=False,
    )
    .agg(
        n_entities=("entity", "count"),
        max_D=("D", "max"),
        min_D=("D", "min"),
        avg_D=("D", "mean"),
        n_unreachable=("distance_status", lambda s: int((s != "connected").sum())),
        entities=("entity", lambda s: list(s)),
    )
    .reset_index()
)

distance_unreachable_entities_df = distance_df[
    distance_df["distance_status"] != "connected"
].copy()

distance_by_depth_df = (
    distance_df
    .dropna(subset=["D"])
    .groupby(["query_name", "selected_root", "D"], dropna=False)
    .agg(
        n_entities_at_depth=("entity", "count"),
        entities_at_depth=("entity", lambda s: sorted(set(s))),
    )
    .reset_index()
    .sort_values(["query_name", "D"])
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 7. Display outputs
# ---------------------------------------------------------

print("D(E, er, Qi) calculation completed.")

print("\nDistance by query/entity:")
display(distance_df)

print("\nDistance summary by query:")
display(distance_summary_by_query_df)

print("\nEntities by depth:")
display(distance_by_depth_df)

if not distance_unreachable_entities_df.empty:
    print("\nWarning: some entities are unreachable from the selected root.")
    display(distance_unreachable_entities_df)
else:
    print("\nAll touched entities are reachable from their selected roots.")


# ---------------------------------------------------------
# 8. Save BLOCK V5 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    distance_df,
    "variables/block_v05/distance_by_query_entity.csv",
)

save_dataframe(
    d_df,
    "variables/block_v05/d_df.csv",
)

save_dataframe(
    distance_summary_by_query_df,
    "variables/block_v05/distance_summary_by_query.csv",
)

save_dataframe(
    distance_by_depth_df,
    "variables/block_v05/distance_by_depth.csv",
)

save_dataframe(
    distance_unreachable_entities_df,
    "variables/block_v05/distance_unreachable_entities.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "n_query_entity_pairs": len(distance_df),
        "n_queries": int(distance_summary_by_query_df["query_name"].nunique()),
        "n_unreachable_pairs": len(distance_unreachable_entities_df),
        "max_distance": (
            None
            if distance_df["D"].dropna().empty
            else int(distance_df["D"].dropna().max())
        ),
        "selected_root_by_query": selected_root_by_query,
        "output_files": {
            "distance_by_query_entity_csv": "variables/block_v05/distance_by_query_entity.csv",
            "d_df_csv": "variables/block_v05/d_df.csv",
            "distance_summary_by_query_csv": "variables/block_v05/distance_summary_by_query.csv",
            "distance_by_depth_csv": "variables/block_v05/distance_by_depth.csv",
            "distance_unreachable_entities_csv": "variables/block_v05/distance_unreachable_entities.csv",
        },
    },
    "variables/block_v05/distance_metadata.json",
)

write_execution_log(
    block_name="BLOCK V5 — CALCULATE D(E, er, Qi)",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_query_entity_pairs": len(distance_df),
        "n_queries": int(distance_summary_by_query_df["query_name"].nunique()),
        "n_unreachable_pairs": len(distance_unreachable_entities_df),
        "distance_by_query_entity_csv": "variables/block_v05/distance_by_query_entity.csv",
        "d_df_csv": "variables/block_v05/d_df.csv",
        "distance_summary_by_query_csv": "variables/block_v05/distance_summary_by_query.csv",
        "distance_by_depth_csv": "variables/block_v05/distance_by_depth.csv",
        "distance_unreachable_entities_csv": "variables/block_v05/distance_unreachable_entities.csv",
        "metadata_json": "variables/block_v05/distance_metadata.json",
    },
)


# ---------------------------------------------------------
# 9. Update README section for BLOCK V5
# ---------------------------------------------------------

max_distance_value = (
    None
    if distance_df["D"].dropna().empty
    else int(distance_df["D"].dropna().max())
)

block_v05_readme_body = f"""
This block computes D(E, er, Qi), the conceptual distance between the selected
root er and each entity E touched by query Qi.

Main variables created:

    distance_df
    d_df

Additional variables created:

    distance_summary_by_query_df
    distance_by_depth_df
    distance_unreachable_entities_df

Number of query/entity pairs:

    {len(distance_df)}

Number of queries processed:

    {int(distance_summary_by_query_df["query_name"].nunique())}

Maximum conceptual distance:

    {max_distance_value}

Number of unreachable query/entity pairs:

    {len(distance_unreachable_entities_df)}

Generated reproducibility files:

    variables/block_v05/distance_by_query_entity.csv
    variables/block_v05/d_df.csv
    variables/block_v05/distance_summary_by_query.csv
    variables/block_v05/distance_by_depth.csv
    variables/block_v05/distance_unreachable_entities.csv
    variables/block_v05/distance_metadata.json

Methodological meaning:

    D(E, er, Qi) measures how far each touched entity is from the selected
    document root in the conceptual graph.

    This value is later used to estimate which entities can be reached by
    a document configuration with a given embedding depth.

Example:

    If selected_root = Corporation, then:
    D(Corporation, Corporation, Qi) = 0
    D(FinancialReport, Corporation, Qi) = 1
    D(ReportElement, Corporation, Qi) = 2
"""

update_readme_section(
    section_title="BLOCK V5 — Calculate D(E, er, Qi)",
    section_body=block_v05_readme_body,
)

D(E, er, Qi) calculation completed.

Distance by query/entity:


,query_name,generic_class,query_family,query_type,selected_root,entity,D,distance_status,path_entities,path_edge_ids,path_relationships,path_length,Rc,Rc_status
0,Q1_CompanyProfileIBM,QG1,Local lookup,select,Corporation,Corporation,0,connected,[Corporation],[],[],0,0,single_or_no_terminal
1,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Corporation,Corporation,0,connected,[Corporation],[],[],0,3,connected
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Corporation,Industry,1,connected,"[Corporation, Industry]",[Corporation -- Industry [corporation_has_indu...,[corporation_has_industry],1,3,connected
3,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Corporation,Country,1,connected,"[Corporation, Country]",[Corporation -- Country [corporation_has_count...,[corporation_has_country],1,3,connected
4,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Corporation,ListedSecurity,1,connected,"[Corporation, ListedSecurity]",[Corporation -- ListedSecurity [corporation_ha...,[corporation_has_listed_security],1,3,connected
5,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount,FinancialServiceAccount,0,connected,[FinancialServiceAccount],[],[],0,3,connected
6,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount,Holding,1,connected,"[FinancialServiceAccount, Holding]",[FinancialServiceAccount -- Holding [account_h...,[account_has_holding],1,3,connected
7,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount,ListedSecurity,2,connected,"[FinancialServiceAccount, Holding, ListedSecur...",[FinancialServiceAccount -- Holding [account_h...,"[account_has_holding, holding_refers_to_listed...",2,3,connected
8,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount,Security,3,connected,"[FinancialServiceAccount, Holding, ListedSecur...",[FinancialServiceAccount -- Holding [account_h...,"[account_has_holding, holding_refers_to_listed...",3,3,connected
9,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Person,Person,0,connected,[Person],[],[],0,4,connected



Distance summary by query:


,query_name,generic_class,query_family,query_type,selected_root,n_entities,max_D,min_D,avg_D,n_unreachable,entities
0,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,Person,6,3,0,1.833333,0,"[Person, FinancialServiceAccount, Holding, Lis..."
1,Q1_CompanyProfileIBM,QG1,Local lookup,select,Corporation,1,0,0,0.000000,0,[Corporation]
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Corporation,4,1,0,0.750000,0,"[Corporation, Industry, Country, ListedSecurity]"
3,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount,4,3,0,1.500000,0,"[FinancialServiceAccount, Holding, ListedSecur..."
4,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Person,5,4,0,2.000000,0,"[Person, FinancialServiceAccount, Holding, Lis..."
5,Q5_ReportsAndMetricDataOfCompany,QG5,Containment retrieval,select,Corporation,5,3,0,1.600000,0,"[Corporation, FinancialReport, ReportElement, ..."
6,Q6_TechUSListedSecuritiesWithHighLastTradedValue,QG6,Filtered search / recommendation,select,ListedSecurity,4,2,0,1.250000,0,"[Corporation, Industry, Country, ListedSecurity]"
7,Q7_PersonsWhoBoughtMoreIBMThanSold,QG7,Aggregation / ranking,select,Person,7,4,0,2.285714,0,"[Person, FinancialServiceAccount, BuyTransacti..."
8,Q8_IBMTransactionsBelowAverageSellingPrice,QG8,Aggregation / ranking,select,Transaction,4,2,0,1.000000,0,"[Transaction, SellTransaction, ListedSecurity,..."
9,Q9_PersonsWhoBoughtAndSoldSameStock,QG9,Associative retrieval + comparison,select,Person,6,3,0,2.000000,0,"[Person, FinancialServiceAccount, BuyTransacti..."



Entities by depth:


,query_name,selected_root,D,n_entities_at_depth,entities_at_depth
0,Q10_CreateAccountHoldingAndBuyTransaction,Person,0,1,[Person]
1,Q10_CreateAccountHoldingAndBuyTransaction,Person,1,1,[FinancialServiceAccount]
2,Q10_CreateAccountHoldingAndBuyTransaction,Person,2,2,"[Holding, Transaction]"
3,Q10_CreateAccountHoldingAndBuyTransaction,Person,3,2,"[BuyTransaction, ListedSecurity]"
4,Q1_CompanyProfileIBM,Corporation,0,1,[Corporation]
5,Q2_CompanyWithIndustryCountryAndListedSecurities,Corporation,0,1,[Corporation]
6,Q2_CompanyWithIndustryCountryAndListedSecurities,Corporation,1,3,"[Country, Industry, ListedSecurity]"
7,Q3_SecuritiesHeldInEachFinancialServiceAccount,FinancialServiceAccount,0,1,[FinancialServiceAccount]
8,Q3_SecuritiesHeldInEachFinancialServiceAccount,FinancialServiceAccount,1,1,[Holding]
9,Q3_SecuritiesHeldInEachFinancialServiceAccount,FinancialServiceAccount,2,1,[ListedSecurity]



All touched entities are reachable from their selected roots.
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v05/distance_by_query_entity.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v05/d_df.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v05/distance_summary_by_query.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v05/distance_by_depth.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v05/distance_unreachable_entities.csv
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v05/distance_metadata.json
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset 

In [50]:
# =========================================================
# BLOCK V6 — OBSERVED CARDINALITY (LIGHT VERSION)
# =========================================================
#
# Purpose:
# This block computes a lightweight observed cardinality profile
# for the FIBEN conceptual relationships.
#
# The previous blocks used only the conceptual schema.
# This block starts looking at the physical FIBEN data loaded
# into DuckDB.
#
# Important:
# This is a light and defensive version.
#
# It tries to:
# - map each conceptual entity to a DuckDB view;
# - inspect available columns;
# - infer possible join columns;
# - compute observed join statistics when possible;
# - mark unresolved relationships as not_computed instead of failing.
#
# Main outputs:
# - observed_cardinality_df
# - relationship_cardinality_df
# - cardinality_observed_df
#
# These aliases are intentionally duplicated for compatibility with
# later notebook blocks.
# =========================================================

import re
import pandas as pd
from typing import Dict, List, Optional, Tuple


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "fiben_data_bundle",
    "conceptual_relationships_df",
    "fiben_scenario",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first BLOCKS 1–3 and BLOCKS V0–V5."
        )


# ---------------------------------------------------------
# 2. Configuration
# ---------------------------------------------------------
#
# For large scale factors, full joins can be expensive.
# This light block can use a row limit for each table.
#
# Start with a limited sample.
# Later, after the pipeline is stable, set this to None to use
# the full data.
# ---------------------------------------------------------

FIBEN_CARDINALITY_ROW_LIMIT = 200_000

# If True, the block prints each attempted join pair.
FIBEN_CARDINALITY_VERBOSE = False


# ---------------------------------------------------------
# 3. Helper functions
# ---------------------------------------------------------

def normalize_for_match(value: str) -> str:
    """
    Normalize names for robust matching.
    """
    return re.sub(r"[^a-zA-Z0-9]+", "", str(value).lower())


def split_camel_case(value: str) -> List[str]:
    """
    Split CamelCase names into lowercase tokens.

    Example
    -------
    FinancialServiceAccount -> ['financial', 'service', 'account']
    """
    tokens = re.sub("([a-z0-9])([A-Z])", r"\1 \2", str(value)).split()
    return [t.lower() for t in tokens]


def column_looks_like_identifier(column_name: str) -> bool:
    """
    Return True if a column name looks like an identifier or key.
    """
    c = str(column_name).lower()
    return (
        c == "id"
        or c.endswith("_id")
        or c.endswith("id")
        or c.endswith("_key")
        or c.endswith("key")
        or c.endswith("_code")
        or c.endswith("code")
        or "identifier" in c
    )


def sql_identifier(name: str) -> str:
    """
    Quote a DuckDB identifier safely.
    """
    return '"' + str(name).replace('"', '""') + '"'


def entity_to_alias_candidates(entity: str) -> List[str]:
    """
    Generate possible physical aliases for one conceptual entity.

    This function is intentionally broad because FIBEN physical
    table names may differ from conceptual entity names.
    """
    compact = normalize_for_match(entity)
    tokens = split_camel_case(entity)
    snake = "_".join(tokens)

    base = [
        snake,
        snake + "s",
        compact,
        compact + "s",
    ]

    custom = {
        "Corporation": [
            "corporation", "corporations", "corp", "company", "companies"
        ],
        "Industry": [
            "industry", "industries", "classification", "classifications"
        ],
        "Country": [
            "country", "countries"
        ],
        "Security": [
            "security", "securities"
        ],
        "ListedSecurity": [
            "listed_security", "listed_securities",
            "listedsecurity", "listedsecurities"
        ],
        "Person": [
            "person", "persons", "people", "investor", "investors"
        ],
        "FinancialServiceAccount": [
            "financial_service_account",
            "financial_service_accounts",
            "financialserviceaccount",
            "financialserviceaccounts",
            "account",
            "accounts"
        ],
        "Holding": [
            "holding", "holdings"
        ],
        "Transaction": [
            "transaction", "transactions"
        ],
        "BuyTransaction": [
            "buy_transaction", "buy_transactions",
            "buytransaction", "buytransactions",
            "buy", "buys"
        ],
        "SellTransaction": [
            "sell_transaction", "sell_transactions",
            "selltransaction", "selltransactions",
            "sell", "sells"
        ],
        "FinancialReport": [
            "financial_report", "financial_reports",
            "report", "reports"
        ],
        "ReportElement": [
            "report_element", "report_elements",
            "reportelement", "reportelements"
        ],
        "StatementElement": [
            "statement_element", "statement_elements",
            "statementelement", "statementelements"
        ],
        "Disclosure": [
            "disclosure", "disclosures"
        ],
    }

    candidates = base + custom.get(entity, [])

    # Remove duplicates while preserving order.
    return list(dict.fromkeys(candidates))


def resolve_view_for_entity(entity: str, bundle: FIBENDuckDBBundle) -> dict:
    """
    Resolve the best DuckDB view for a conceptual entity.

    Search order:
    1. semantic views created in BLOCK 3;
    2. raw views registered in BLOCK 2.
    """
    candidates = entity_to_alias_candidates(entity)

    normalized_semantic = {
        normalize_for_match(alias): (alias, view)
        for alias, view in bundle.semantic_views.items()
    }

    normalized_raw = {
        normalize_for_match(alias): (alias, view)
        for alias, view in bundle.raw_views.items()
    }

    for candidate in candidates:
        norm = normalize_for_match(candidate)
        if norm in normalized_semantic:
            alias, view = normalized_semantic[norm]
            return {
                "entity": entity,
                "status": "matched_semantic",
                "alias": alias,
                "view_name": view,
                "candidate_used": candidate,
            }

    for candidate in candidates:
        norm = normalize_for_match(candidate)
        if norm in normalized_raw:
            alias, view = normalized_raw[norm]
            return {
                "entity": entity,
                "status": "matched_raw",
                "alias": alias,
                "view_name": view,
                "candidate_used": candidate,
            }

    return {
        "entity": entity,
        "status": "unmatched",
        "alias": None,
        "view_name": None,
        "candidate_used": None,
    }


def get_view_columns(con, view_name: str) -> List[str]:
    """
    Return the column names of a DuckDB view.
    """
    desc = con.execute(f"DESCRIBE {view_name}").df()
    return desc["column_name"].tolist()


def entity_key_tokens(entity: str) -> List[str]:
    """
    Return useful tokens for identifying key columns related to one entity.
    """
    tokens = split_camel_case(entity)

    synonyms = {
        "Corporation": ["corporation", "corp", "company"],
        "Industry": ["industry", "classification"],
        "Country": ["country"],
        "Security": ["security"],
        "ListedSecurity": ["listed", "listedsecurity", "listed_security", "security"],
        "Person": ["person", "people", "investor"],
        "FinancialServiceAccount": ["account", "financialserviceaccount", "financial_service_account"],
        "Holding": ["holding"],
        "Transaction": ["transaction", "trade"],
        "BuyTransaction": ["buy", "transaction"],
        "SellTransaction": ["sell", "transaction"],
        "FinancialReport": ["report", "financialreport", "financial_report"],
        "ReportElement": ["reportelement", "report_element", "element"],
        "StatementElement": ["statementelement", "statement_element", "statement"],
        "Disclosure": ["disclosure"],
    }

    return list(dict.fromkeys(tokens + synonyms.get(entity, [])))


def find_entity_id_columns(entity: str, columns: List[str]) -> List[str]:
    """
    Find likely identifier columns for a conceptual entity in a view.
    """
    tokens = entity_key_tokens(entity)
    normalized_tokens = [normalize_for_match(t) for t in tokens]

    candidates = []

    for col in columns:
        col_norm = normalize_for_match(col)

        if col_norm in ["id", normalize_for_match(entity) + "id"]:
            candidates.append(col)
            continue

        if column_looks_like_identifier(col):
            if any(tok in col_norm for tok in normalized_tokens):
                candidates.append(col)

    # Generic id is a fallback.
    for col in columns:
        if normalize_for_match(col) == "id" and col not in candidates:
            candidates.append(col)

    return list(dict.fromkeys(candidates))


def infer_join_column_pairs(
    source_entity: str,
    target_entity: str,
    source_columns: List[str],
    target_columns: List[str],
) -> List[Tuple[str, str, str]]:
    """
    Infer possible join column pairs between two entity views.

    Returns a list of tuples:

        (source_column, target_column, inference_reason)

    The inference is intentionally conservative.
    """
    pairs = []

    source_cols_norm = {
        normalize_for_match(c): c
        for c in source_columns
    }

    target_cols_norm = {
        normalize_for_match(c): c
        for c in target_columns
    }

    # 1. Same identifier-like column exists in both tables.
    for norm_col, source_col in source_cols_norm.items():
        if norm_col in target_cols_norm and column_looks_like_identifier(source_col):
            pairs.append((
                source_col,
                target_cols_norm[norm_col],
                "same_identifier_column_name"
            ))

    # 2. Source table has a foreign key-like column pointing to target.
    target_id_cols = find_entity_id_columns(target_entity, target_columns)
    target_tokens = [normalize_for_match(t) for t in entity_key_tokens(target_entity)]

    for source_col in source_columns:
        source_col_norm = normalize_for_match(source_col)

        if not column_looks_like_identifier(source_col):
            continue

        if any(tok in source_col_norm for tok in target_tokens):
            for target_id_col in target_id_cols:
                pairs.append((
                    source_col,
                    target_id_col,
                    "source_fk_to_target"
                ))

    # 3. Target table has a foreign key-like column pointing to source.
    source_id_cols = find_entity_id_columns(source_entity, source_columns)
    source_tokens = [normalize_for_match(t) for t in entity_key_tokens(source_entity)]

    for target_col in target_columns:
        target_col_norm = normalize_for_match(target_col)

        if not column_looks_like_identifier(target_col):
            continue

        if any(tok in target_col_norm for tok in source_tokens):
            for source_id_col in source_id_cols:
                pairs.append((
                    source_id_col,
                    target_col,
                    "target_fk_to_source"
                ))

    # Remove duplicates while preserving order.
    unique_pairs = []
    seen = set()

    for pair in pairs:
        key = (pair[0], pair[1])
        if key not in seen:
            unique_pairs.append(pair)
            seen.add(key)

    return unique_pairs


def limited_view_sql(view_name: str, limit: Optional[int]) -> str:
    """
    Return a SQL subquery for either the full view or a limited view.
    """
    if limit is None:
        return f"(SELECT * FROM {view_name})"

    return f"(SELECT * FROM {view_name} LIMIT {int(limit)})"


def try_compute_join_statistics(
    con,
    source_view: str,
    target_view: str,
    source_col: str,
    target_col: str,
    row_limit: Optional[int],
) -> dict:
    """
    Compute lightweight observed join statistics for one candidate join.

    The result is not a full database constraint analysis.
    It is an observed profile over the selected data scope.
    """
    source_sql = limited_view_sql(source_view, row_limit)
    target_sql = limited_view_sql(target_view, row_limit)

    sql = f"""
    WITH
    s AS (
        SELECT {sql_identifier(source_col)} AS source_join_key
        FROM {source_sql}
        WHERE {sql_identifier(source_col)} IS NOT NULL
    ),
    t AS (
        SELECT {sql_identifier(target_col)} AS target_join_key
        FROM {target_sql}
        WHERE {sql_identifier(target_col)} IS NOT NULL
    ),
    joined AS (
        SELECT
            s.source_join_key,
            t.target_join_key
        FROM s
        INNER JOIN t
            ON s.source_join_key = t.target_join_key
    )
    SELECT
        (SELECT COUNT(*) FROM s) AS source_non_null_rows,
        (SELECT COUNT(*) FROM t) AS target_non_null_rows,
        (SELECT COUNT(DISTINCT source_join_key) FROM s) AS source_distinct_keys,
        (SELECT COUNT(DISTINCT target_join_key) FROM t) AS target_distinct_keys,
        COUNT(*) AS joined_rows,
        COUNT(DISTINCT source_join_key) AS joined_distinct_source_keys,
        COUNT(DISTINCT target_join_key) AS joined_distinct_target_keys
    FROM joined;
    """

    return con.execute(sql).df().iloc[0].to_dict()


def classify_light_cardinality(stats: dict) -> str:
    """
    Classify relationship cardinality using observed join statistics.

    This classification is approximate because it is based on detected
    join columns and optionally limited data.
    """
    joined_rows = stats.get("joined_rows", 0)
    joined_source = stats.get("joined_distinct_source_keys", 0)
    joined_target = stats.get("joined_distinct_target_keys", 0)

    if joined_rows == 0:
        return "no_observed_matches"

    if joined_source == 0 or joined_target == 0:
        return "unknown"

    avg_targets_per_source = joined_rows / joined_source
    avg_sources_per_target = joined_rows / joined_target

    source_many = avg_targets_per_source > 1.05
    target_many = avg_sources_per_target > 1.05

    if not source_many and not target_many:
        return "observed_1_to_1"

    if source_many and not target_many:
        return "observed_1_to_N"

    if not source_many and target_many:
        return "observed_N_to_1"

    return "observed_N_to_M"


# ---------------------------------------------------------
# 4. Resolve physical views for conceptual entities
# ---------------------------------------------------------

entity_view_mapping_rows = []

for entity in fiben_scenario.entities:
    entity_view_mapping_rows.append(
        resolve_view_for_entity(entity, fiben_data_bundle)
    )

fiben_entity_view_mapping_df = pd.DataFrame(entity_view_mapping_rows)

display(fiben_entity_view_mapping_df)


# ---------------------------------------------------------
# 5. Compute observed cardinality relationship by relationship
# ---------------------------------------------------------

observed_cardinality_rows = []
candidate_join_rows = []

entity_view_lookup = {
    row["entity"]: row
    for _, row in fiben_entity_view_mapping_df.iterrows()
}

for _, rel in conceptual_relationships_df.iterrows():
    source_entity = rel["source"]
    target_entity = rel["target"]
    relationship_name = rel["name"]
    semantic_type = rel.get("semantic_type", "unspecified")
    edge_id = rel.get(
        "edge_id",
        f"{source_entity} -- {target_entity} [{relationship_name}]"
    )

    source_info = entity_view_lookup.get(source_entity)
    target_info = entity_view_lookup.get(target_entity)

    if source_info is None or target_info is None:
        observed_cardinality_rows.append({
            "edge_id": edge_id,
            "relationship_name": relationship_name,
            "semantic_type": semantic_type,
            "source_entity": source_entity,
            "target_entity": target_entity,
            "source_view": None,
            "target_view": None,
            "selected_source_column": None,
            "selected_target_column": None,
            "join_inference_reason": None,
            "observed_cardinality": "not_computed",
            "computation_status": "missing_entity_mapping",
            "row_limit": FIBEN_CARDINALITY_ROW_LIMIT,
            "joined_rows": None,
            "source_non_null_rows": None,
            "target_non_null_rows": None,
            "source_distinct_keys": None,
            "target_distinct_keys": None,
            "joined_distinct_source_keys": None,
            "joined_distinct_target_keys": None,
        })
        continue

    source_view = source_info["view_name"]
    target_view = target_info["view_name"]

    if source_view is None or target_view is None:
        observed_cardinality_rows.append({
            "edge_id": edge_id,
            "relationship_name": relationship_name,
            "semantic_type": semantic_type,
            "source_entity": source_entity,
            "target_entity": target_entity,
            "source_view": source_view,
            "target_view": target_view,
            "selected_source_column": None,
            "selected_target_column": None,
            "join_inference_reason": None,
            "observed_cardinality": "not_computed",
            "computation_status": "missing_view",
            "row_limit": FIBEN_CARDINALITY_ROW_LIMIT,
            "joined_rows": None,
            "source_non_null_rows": None,
            "target_non_null_rows": None,
            "source_distinct_keys": None,
            "target_distinct_keys": None,
            "joined_distinct_source_keys": None,
            "joined_distinct_target_keys": None,
        })
        continue

    source_columns = get_view_columns(fiben_data_bundle.con, source_view)
    target_columns = get_view_columns(fiben_data_bundle.con, target_view)

    join_pairs = infer_join_column_pairs(
        source_entity=source_entity,
        target_entity=target_entity,
        source_columns=source_columns,
        target_columns=target_columns,
    )

    for source_col, target_col, reason in join_pairs:
        candidate_join_rows.append({
            "edge_id": edge_id,
            "relationship_name": relationship_name,
            "source_entity": source_entity,
            "target_entity": target_entity,
            "source_view": source_view,
            "target_view": target_view,
            "source_column": source_col,
            "target_column": target_col,
            "inference_reason": reason,
        })

    if not join_pairs:
        observed_cardinality_rows.append({
            "edge_id": edge_id,
            "relationship_name": relationship_name,
            "semantic_type": semantic_type,
            "source_entity": source_entity,
            "target_entity": target_entity,
            "source_view": source_view,
            "target_view": target_view,
            "selected_source_column": None,
            "selected_target_column": None,
            "join_inference_reason": None,
            "observed_cardinality": "not_computed",
            "computation_status": "no_join_columns_detected",
            "row_limit": FIBEN_CARDINALITY_ROW_LIMIT,
            "joined_rows": None,
            "source_non_null_rows": None,
            "target_non_null_rows": None,
            "source_distinct_keys": None,
            "target_distinct_keys": None,
            "joined_distinct_source_keys": None,
            "joined_distinct_target_keys": None,
        })
        continue

    # Try candidate joins until one produces observed matches.
    selected_result = None
    selected_error = None

    for source_col, target_col, reason in join_pairs:
        if FIBEN_CARDINALITY_VERBOSE:
            print(
                f"Trying join {source_entity}.{source_col} "
                f"= {target_entity}.{target_col}"
            )

        try:
            stats = try_compute_join_statistics(
                con=fiben_data_bundle.con,
                source_view=source_view,
                target_view=target_view,
                source_col=source_col,
                target_col=target_col,
                row_limit=FIBEN_CARDINALITY_ROW_LIMIT,
            )

            stats = {
                k: (None if pd.isna(v) else v)
                for k, v in stats.items()
            }

            cardinality_label = classify_light_cardinality(stats)

            selected_result = {
                "selected_source_column": source_col,
                "selected_target_column": target_col,
                "join_inference_reason": reason,
                "observed_cardinality": cardinality_label,
                "computation_status": "computed",
                **stats,
            }

            # Prefer a join with observed matches.
            if stats.get("joined_rows", 0) > 0:
                break

        except Exception as e:
            selected_error = f"{type(e).__name__}: {e}"
            selected_result = None
            continue

    if selected_result is None:
        observed_cardinality_rows.append({
            "edge_id": edge_id,
            "relationship_name": relationship_name,
            "semantic_type": semantic_type,
            "source_entity": source_entity,
            "target_entity": target_entity,
            "source_view": source_view,
            "target_view": target_view,
            "selected_source_column": None,
            "selected_target_column": None,
            "join_inference_reason": None,
            "observed_cardinality": "not_computed",
            "computation_status": f"join_failed: {selected_error}",
            "row_limit": FIBEN_CARDINALITY_ROW_LIMIT,
            "joined_rows": None,
            "source_non_null_rows": None,
            "target_non_null_rows": None,
            "source_distinct_keys": None,
            "target_distinct_keys": None,
            "joined_distinct_source_keys": None,
            "joined_distinct_target_keys": None,
        })
    else:
        observed_cardinality_rows.append({
            "edge_id": edge_id,
            "relationship_name": relationship_name,
            "semantic_type": semantic_type,
            "source_entity": source_entity,
            "target_entity": target_entity,
            "source_view": source_view,
            "target_view": target_view,
            "row_limit": FIBEN_CARDINALITY_ROW_LIMIT,
            **selected_result,
        })


observed_cardinality_df = pd.DataFrame(observed_cardinality_rows)
relationship_cardinality_df = observed_cardinality_df.copy()
cardinality_observed_df = observed_cardinality_df.copy()

candidate_join_columns_df = pd.DataFrame(candidate_join_rows)


# ---------------------------------------------------------
# 6. Summary tables
# ---------------------------------------------------------

observed_cardinality_summary_df = (
    observed_cardinality_df
    .groupby(["observed_cardinality", "computation_status"], dropna=False)
    .agg(
        n_relationships=("relationship_name", "count"),
        relationships=("relationship_name", lambda s: sorted(set(s))),
    )
    .reset_index()
    .sort_values(["observed_cardinality", "computation_status"])
    .reset_index(drop=True)
)

uncomputed_cardinality_df = observed_cardinality_df[
    observed_cardinality_df["computation_status"] != "computed"
].copy()


# ---------------------------------------------------------
# 7. Display outputs
# ---------------------------------------------------------

print("Observed cardinality light profiling completed.")

print("\nEntity-to-view mapping:")
display(fiben_entity_view_mapping_df)

print("\nCandidate join columns:")
display(candidate_join_columns_df)

print("\nObserved cardinality by relationship:")
display(observed_cardinality_df)

print("\nObserved cardinality summary:")
display(observed_cardinality_summary_df)

if not uncomputed_cardinality_df.empty:
    print("\nRelationships not computed in the light cardinality profile:")
    display(uncomputed_cardinality_df)
else:
    print("\nAll relationships received an observed cardinality profile.")


# ---------------------------------------------------------
# 8. Save BLOCK V6 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    fiben_entity_view_mapping_df,
    "variables/block_v06/fiben_entity_view_mapping.csv",
)

save_dataframe(
    candidate_join_columns_df,
    "variables/block_v06/candidate_join_columns.csv",
)

save_dataframe(
    observed_cardinality_df,
    "variables/block_v06/observed_cardinality_by_relationship.csv",
)

save_dataframe(
    observed_cardinality_summary_df,
    "variables/block_v06/observed_cardinality_summary.csv",
)

save_dataframe(
    uncomputed_cardinality_df,
    "variables/block_v06/uncomputed_cardinality_relationships.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "row_limit": FIBEN_CARDINALITY_ROW_LIMIT,
        "n_relationships": len(observed_cardinality_df),
        "n_computed_relationships": int(
            (observed_cardinality_df["computation_status"] == "computed").sum()
        ),
        "n_uncomputed_relationships": len(uncomputed_cardinality_df),
        "observed_cardinality_counts": (
            observed_cardinality_df["observed_cardinality"]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "output_files": {
            "entity_view_mapping_csv": "variables/block_v06/fiben_entity_view_mapping.csv",
            "candidate_join_columns_csv": "variables/block_v06/candidate_join_columns.csv",
            "observed_cardinality_csv": "variables/block_v06/observed_cardinality_by_relationship.csv",
            "observed_cardinality_summary_csv": "variables/block_v06/observed_cardinality_summary.csv",
            "uncomputed_cardinality_csv": "variables/block_v06/uncomputed_cardinality_relationships.csv",
        },
    },
    "variables/block_v06/observed_cardinality_metadata.json",
)

write_execution_log(
    block_name="BLOCK V6 — OBSERVED CARDINALITY LIGHT VERSION",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "row_limit": FIBEN_CARDINALITY_ROW_LIMIT,
        "n_relationships": len(observed_cardinality_df),
        "n_computed_relationships": int(
            (observed_cardinality_df["computation_status"] == "computed").sum()
        ),
        "n_uncomputed_relationships": len(uncomputed_cardinality_df),
        "entity_view_mapping_csv": "variables/block_v06/fiben_entity_view_mapping.csv",
        "candidate_join_columns_csv": "variables/block_v06/candidate_join_columns.csv",
        "observed_cardinality_csv": "variables/block_v06/observed_cardinality_by_relationship.csv",
        "observed_cardinality_summary_csv": "variables/block_v06/observed_cardinality_summary.csv",
        "uncomputed_cardinality_csv": "variables/block_v06/uncomputed_cardinality_relationships.csv",
        "metadata_json": "variables/block_v06/observed_cardinality_metadata.json",
    },
)


# ---------------------------------------------------------
# 9. Update README section for BLOCK V6
# ---------------------------------------------------------

block_v06_readme_body = f"""
This block computes a lightweight observed cardinality profile for the FIBEN
conceptual relationships.

Main variables created:

    observed_cardinality_df
    relationship_cardinality_df
    cardinality_observed_df

Additional variables created:

    fiben_entity_view_mapping_df
    candidate_join_columns_df
    observed_cardinality_summary_df
    uncomputed_cardinality_df

Number of conceptual relationships processed:

    {len(observed_cardinality_df)}

Number of computed relationships:

    {int((observed_cardinality_df["computation_status"] == "computed").sum())}

Number of uncomputed relationships:

    {len(uncomputed_cardinality_df)}

Row limit used per table:

    {FIBEN_CARDINALITY_ROW_LIMIT}

Generated reproducibility files:

    variables/block_v06/fiben_entity_view_mapping.csv
    variables/block_v06/candidate_join_columns.csv
    variables/block_v06/observed_cardinality_by_relationship.csv
    variables/block_v06/observed_cardinality_summary.csv
    variables/block_v06/uncomputed_cardinality_relationships.csv
    variables/block_v06/observed_cardinality_metadata.json

Methodological meaning:

    This block estimates the observed cardinality pattern of each conceptual
    relationship using the physical FIBEN data available in DuckDB.

    The result is used as supporting evidence for later semantic and structural
    interpretation of document configurations.

Important limitation:

    This is a light version. It infers join columns automatically and may not
    compute every relationship if physical key names are ambiguous.

    Relationships marked as not_computed should be reviewed manually before
    running the final benchmark stage.
"""

update_readme_section(
    section_title="BLOCK V6 — Observed Cardinality Light Version",
    section_body=block_v06_readme_body,
)

,entity,status,alias,view_name,candidate_used
0,Corporation,matched_semantic,corporations,fiben_corporations,corporations
1,Industry,unmatched,None,None,None
2,Country,matched_raw,country,fiben_raw_country,country
3,Security,matched_semantic,securities,fiben_securities,securities
4,ListedSecurity,matched_semantic,listed_securities,fiben_listed_securities,listed_securities
5,Person,matched_raw,person,fiben_raw_person,person
6,FinancialServiceAccount,matched_raw,financialserviceaccount,fiben_raw_financialserviceaccount,financial_service_account
7,Holding,matched_raw,holding,fiben_raw_holding,holding
8,Transaction,unmatched,None,None,None
9,BuyTransaction,unmatched,None,None,None


Observed cardinality light profiling completed.

Entity-to-view mapping:


,entity,status,alias,view_name,candidate_used
0,Corporation,matched_semantic,corporations,fiben_corporations,corporations
1,Industry,unmatched,None,None,None
2,Country,matched_raw,country,fiben_raw_country,country
3,Security,matched_semantic,securities,fiben_securities,securities
4,ListedSecurity,matched_semantic,listed_securities,fiben_listed_securities,listed_securities
5,Person,matched_raw,person,fiben_raw_person,person
6,FinancialServiceAccount,matched_raw,financialserviceaccount,fiben_raw_financialserviceaccount,financial_service_account
7,Holding,matched_raw,holding,fiben_raw_holding,holding
8,Transaction,unmatched,None,None,None
9,BuyTransaction,unmatched,None,None,None



Candidate join columns:


,edge_id,relationship_name,source_entity,target_entity,source_view,target_view,source_column,target_column,inference_reason
0,ListedSecurity -- Security [listed_security_re...,listed_security_represents_security,ListedSecurity,Security,fiben_listed_securities,fiben_securities,LISTEDSECURITYID,SECURITYID,source_fk_to_target
1,FinancialReport -- Disclosure [financial_repor...,financial_report_contains_disclosure,FinancialReport,Disclosure,fiben_reports,fiben_disclosures,HASUNIQUEIDENTIFIER,HASUNIQUEIDENTIFIER,same_identifier_column_name


...


Observed cardinality summary:


,observed_cardinality,computation_status,n_relationships,relationships
0,no_observed_matches,computed,1,[financial_report_contains_disclosure]
1,not_computed,missing_view,7,"[account_records_transaction, buy_transaction_..."
2,not_computed,no_join_columns_detected,6,"[account_has_holding, corporation_has_country,..."
3,observed_1_to_1,computed,1,[listed_security_represents_security]



Relationships not computed in the light cardinality profile:


,edge_id,relationship_name,semantic_type,source_entity,target_entity,source_view,target_view,selected_source_column,selected_target_column,join_inference_reason,observed_cardinality,computation_status,row_limit,joined_rows,source_non_null_rows,target_non_null_rows,source_distinct_keys,target_distinct_keys,joined_distinct_source_keys,joined_distinct_target_keys
0,Corporation -- Industry [corporation_has_indus...,corporation_has_industry,descriptor,Corporation,Industry,fiben_corporations,None,None,None,None,not_computed,missing_view,200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Corporation -- Country [corporation_has_country],corporation_has_country,descriptor,Corporation,Country,fiben_corporations,fiben_raw_country,None,None,None,not_computed,no_join_columns_detected,200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Corporation -- ListedSecurity [corporation_has...,corporation_has_listed_security,association,Corporation,ListedSecurity,fiben_corporations,fiben_listed_securities,None,None,None,not_computed,no_join_columns_detected,200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Person -- FinancialServiceAccount [person_owns...,person_owns_financial_service_account,ownership,Person,FinancialServiceAccount,fiben_raw_person,fiben_raw_financialserviceaccount,None,None,None,not_computed,no_join_columns_detected,200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,FinancialServiceAccount -- Holding [account_ha...,account_has_holding,containment,FinancialServiceAccount,Holding,fiben_raw_financialserviceaccount,fiben_raw_holding,None,None,None,not_computed,no_join_columns_detected,200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Holding -- ListedSecurity [holding_refers_to_l...,holding_refers_to_listed_security,association,Holding,ListedSecurity,fiben_raw_holding,fiben_listed_securities,None,None,None,not_computed,no_join_columns_detected,200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,FinancialServiceAccount -- Transaction [accoun...,account_records_transaction,containment,FinancialServiceAccount,Transaction,fiben_raw_financialserviceaccount,None,None,None,None,not_computed,missing_view,200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Transaction -- ListedSecurity [transaction_ref...,transaction_refers_to_listed_security,association,Transaction,ListedSecurity,None,fiben_listed_securities,None,None,None,not_computed,missing_view,200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,BuyTransaction -- Transaction [buy_transaction...,buy_transaction_is_transaction,subtype,BuyTransaction,Transaction,None,None,None,None,None,not_computed,missing_view,200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,SellTransaction -- Transaction [sell_transacti...,sell_transaction_is_transaction,subtype,SellTransaction,Transaction,None,None,None,None,None,not_computed,missing_view,200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06/fiben_entity_view_mapping.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06/candidate_join_columns.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06/observed_cardinality_by_relationship.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06/observed_cardinality_summary.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06/uncomputed_cardinality_relationships.csv
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06/observed_cardinality_metadata.json
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fib

In [51]:
observed_cardinality_df[
    [
        "relationship_name",
        "source_entity",
        "target_entity",
        "selected_source_column",
        "selected_target_column",
        "observed_cardinality",
        "computation_status",
        "joined_rows",
    ]
]

,relationship_name,source_entity,target_entity,selected_source_column,selected_target_column,observed_cardinality,computation_status,joined_rows
0,corporation_has_industry,Corporation,Industry,None,None,not_computed,missing_view,NaN
1,corporation_has_country,Corporation,Country,None,None,not_computed,no_join_columns_detected,NaN
2,corporation_has_listed_security,Corporation,ListedSecurity,None,None,not_computed,no_join_columns_detected,NaN
3,listed_security_represents_security,ListedSecurity,Security,LISTEDSECURITYID,SECURITYID,observed_1_to_1,computed,2745.0
4,person_owns_financial_service_account,Person,FinancialServiceAccount,None,None,not_computed,no_join_columns_detected,NaN
5,account_has_holding,FinancialServiceAccount,Holding,None,None,not_computed,no_join_columns_detected,NaN
6,holding_refers_to_listed_security,Holding,ListedSecurity,None,None,not_computed,no_join_columns_detected,NaN
7,account_records_transaction,FinancialServiceAccount,Transaction,None,None,not_computed,missing_view,NaN
8,transaction_refers_to_listed_security,Transaction,ListedSecurity,None,None,not_computed,missing_view,NaN
9,buy_transaction_is_transaction,BuyTransaction,Transaction,None,None,not_computed,missing_view,NaN


In [52]:
uncomputed_cardinality_df[
    [
        "relationship_name",
        "source_entity",
        "target_entity",
        "source_view",
        "target_view",
        "computation_status",
    ]
]

,relationship_name,source_entity,target_entity,source_view,target_view,computation_status
0,corporation_has_industry,Corporation,Industry,fiben_corporations,None,missing_view
1,corporation_has_country,Corporation,Country,fiben_corporations,fiben_raw_country,no_join_columns_detected
2,corporation_has_listed_security,Corporation,ListedSecurity,fiben_corporations,fiben_listed_securities,no_join_columns_detected
4,person_owns_financial_service_account,Person,FinancialServiceAccount,fiben_raw_person,fiben_raw_financialserviceaccount,no_join_columns_detected
5,account_has_holding,FinancialServiceAccount,Holding,fiben_raw_financialserviceaccount,fiben_raw_holding,no_join_columns_detected
6,holding_refers_to_listed_security,Holding,ListedSecurity,fiben_raw_holding,fiben_listed_securities,no_join_columns_detected
7,account_records_transaction,FinancialServiceAccount,Transaction,fiben_raw_financialserviceaccount,None,missing_view
8,transaction_refers_to_listed_security,Transaction,ListedSecurity,None,fiben_listed_securities,missing_view
9,buy_transaction_is_transaction,BuyTransaction,Transaction,None,None,missing_view
10,sell_transaction_is_transaction,SellTransaction,Transaction,None,None,missing_view


In [53]:
# =========================================================
# BLOCK V6A — INSPECT FIBEN PHYSICAL MAPPING AND JOIN KEYS
# =========================================================
#
# Purpose:
# This diagnostic block inspects the physical FIBEN views and
# their columns before refining the observed cardinality block.
#
# It helps identify:
# - which physical views exist;
# - which columns are available in each view;
# - which columns look like identifiers or foreign keys;
# - which conceptual entities are missing physical mappings;
# - which conceptual relationships need manual join hints.
#
# This block does not change the methodology.
# It only produces diagnostic tables for refining BLOCK V6.
# =========================================================

import re
import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "fiben_data_bundle",
    "conceptual_relationships_df",
    "fiben_scenario",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first BLOCKS 1–3 and BLOCKS V0–V5."
        )


# ---------------------------------------------------------
# 2. Helper functions
# ---------------------------------------------------------

def normalize_for_diagnostic(value: str) -> str:
    """
    Normalize names for physical mapping diagnostics.
    """
    return re.sub(r"[^a-zA-Z0-9]+", "", str(value).lower())


def diagnostic_column_looks_like_key(column_name: str) -> bool:
    """
    Return True if a column looks like a primary key, foreign key,
    code, identifier, or relationship column.
    """
    c = str(column_name).lower()

    key_patterns = [
        "id",
        "_id",
        "key",
        "_key",
        "code",
        "_code",
        "identifier",
        "account",
        "corp",
        "corporation",
        "company",
        "country",
        "industry",
        "security",
        "listed",
        "person",
        "holding",
        "transaction",
        "report",
        "element",
        "statement",
        "disclosure",
    ]

    return any(pattern in c for pattern in key_patterns)


def describe_view_columns_for_diagnostic(con, view_name: str) -> pd.DataFrame:
    """
    Return column metadata for one DuckDB view.
    """
    df = con.execute(f"DESCRIBE {view_name}").df()
    df.insert(0, "view_name", view_name)
    df["normalized_column"] = df["column_name"].apply(normalize_for_diagnostic)
    df["looks_like_key"] = df["column_name"].apply(diagnostic_column_looks_like_key)
    return df


# ---------------------------------------------------------
# 3. Build physical view inventory
# ---------------------------------------------------------

physical_view_rows = []

for scope, mapping in [
    ("raw", fiben_data_bundle.raw_views),
    ("semantic", fiben_data_bundle.semantic_views),
]:
    for alias, view_name in sorted(mapping.items()):
        try:
            n_rows = fiben_data_bundle.con.execute(
                f"SELECT COUNT(*) AS n FROM {view_name}"
            ).fetchone()[0]

            columns = fiben_data_bundle.con.execute(
                f"DESCRIBE {view_name}"
            ).df()["column_name"].tolist()

            status = "ok"

        except Exception as e:
            n_rows = None
            columns = []
            status = f"error: {type(e).__name__}: {e}"

        physical_view_rows.append({
            "scope": scope,
            "alias": alias,
            "view_name": view_name,
            "n_rows": n_rows,
            "n_columns": len(columns),
            "columns": columns,
            "status": status,
        })

fiben_physical_view_inventory_df = pd.DataFrame(physical_view_rows)


# ---------------------------------------------------------
# 4. Build full column dictionary
# ---------------------------------------------------------

column_frames = []

for _, row in fiben_physical_view_inventory_df.iterrows():
    if row["status"] != "ok":
        continue

    view_name = row["view_name"]
    alias = row["alias"]
    scope = row["scope"]

    desc_df = describe_view_columns_for_diagnostic(
        fiben_data_bundle.con,
        view_name,
    )

    desc_df.insert(0, "scope", scope)
    desc_df.insert(1, "alias", alias)

    column_frames.append(desc_df)

if column_frames:
    fiben_physical_column_dictionary_df = pd.concat(
        column_frames,
        ignore_index=True,
    )
else:
    fiben_physical_column_dictionary_df = pd.DataFrame()


# ---------------------------------------------------------
# 5. Extract likely key columns
# ---------------------------------------------------------

if not fiben_physical_column_dictionary_df.empty:
    fiben_likely_key_columns_df = (
        fiben_physical_column_dictionary_df[
            fiben_physical_column_dictionary_df["looks_like_key"] == True
        ]
        .copy()
        .sort_values(["scope", "alias", "column_name"])
        .reset_index(drop=True)
    )
else:
    fiben_likely_key_columns_df = pd.DataFrame()


# ---------------------------------------------------------
# 6. Diagnose missing conceptual entity mappings
# ---------------------------------------------------------

conceptual_entities_df = pd.DataFrame({
    "conceptual_entity": fiben_scenario.entities
})

available_aliases = sorted(
    list(fiben_data_bundle.raw_views.keys()) +
    list(fiben_data_bundle.semantic_views.keys())
)

available_views = {
    **fiben_data_bundle.raw_views,
    **fiben_data_bundle.semantic_views,
}

entity_mapping_diagnostic_rows = []

for entity in fiben_scenario.entities:
    normalized_entity = normalize_for_diagnostic(entity)

    possible_alias_matches = []

    for alias in available_aliases:
        normalized_alias = normalize_for_diagnostic(alias)

        if normalized_entity in normalized_alias or normalized_alias in normalized_entity:
            possible_alias_matches.append(alias)

    entity_mapping_diagnostic_rows.append({
        "conceptual_entity": entity,
        "normalized_entity": normalized_entity,
        "possible_alias_matches": possible_alias_matches,
        "n_possible_alias_matches": len(possible_alias_matches),
    })

fiben_entity_mapping_diagnostic_df = pd.DataFrame(
    entity_mapping_diagnostic_rows
)


# ---------------------------------------------------------
# 7. Diagnose relationships that need manual review
# ---------------------------------------------------------

if "observed_cardinality_df" in globals():
    v6_relationship_review_source_df = observed_cardinality_df.copy()
else:
    v6_relationship_review_source_df = conceptual_relationships_df.copy()
    v6_relationship_review_source_df["computation_status"] = "not_run_yet"
    v6_relationship_review_source_df["source_view"] = None
    v6_relationship_review_source_df["target_view"] = None

relationships_need_review_df = v6_relationship_review_source_df[
    v6_relationship_review_source_df["computation_status"].astype(str).isin([
        "missing_view",
        "no_join_columns_detected",
        "not_run_yet",
    ])
    | v6_relationship_review_source_df["computation_status"].astype(str).str.startswith("join_failed")
].copy()


# ---------------------------------------------------------
# 8. Display diagnostics
# ---------------------------------------------------------

print("FIBEN physical mapping diagnostics completed.")

print("\nPhysical view inventory:")
display(fiben_physical_view_inventory_df)

print("\nLikely key columns:")
display(fiben_likely_key_columns_df)

print("\nConceptual entity mapping diagnostic:")
display(fiben_entity_mapping_diagnostic_df)

print("\nRelationships needing manual review:")
display(relationships_need_review_df)


# ---------------------------------------------------------
# 9. Save BLOCK V6A outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    fiben_physical_view_inventory_df,
    "variables/block_v06a/fiben_physical_view_inventory.csv",
)

save_dataframe(
    fiben_physical_column_dictionary_df,
    "variables/block_v06a/fiben_physical_column_dictionary.csv",
)

save_dataframe(
    fiben_likely_key_columns_df,
    "variables/block_v06a/fiben_likely_key_columns.csv",
)

save_dataframe(
    fiben_entity_mapping_diagnostic_df,
    "variables/block_v06a/fiben_entity_mapping_diagnostic.csv",
)

save_dataframe(
    relationships_need_review_df,
    "variables/block_v06a/relationships_need_manual_review.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "n_physical_views": len(fiben_physical_view_inventory_df),
        "n_column_rows": len(fiben_physical_column_dictionary_df),
        "n_likely_key_columns": len(fiben_likely_key_columns_df),
        "n_relationships_need_review": len(relationships_need_review_df),
        "output_files": {
            "physical_view_inventory_csv": "variables/block_v06a/fiben_physical_view_inventory.csv",
            "physical_column_dictionary_csv": "variables/block_v06a/fiben_physical_column_dictionary.csv",
            "likely_key_columns_csv": "variables/block_v06a/fiben_likely_key_columns.csv",
            "entity_mapping_diagnostic_csv": "variables/block_v06a/fiben_entity_mapping_diagnostic.csv",
            "relationships_need_manual_review_csv": "variables/block_v06a/relationships_need_manual_review.csv",
        },
    },
    "variables/block_v06a/fiben_physical_mapping_diagnostic_metadata.json",
)

write_execution_log(
    block_name="BLOCK V6A — INSPECT FIBEN PHYSICAL MAPPING AND JOIN KEYS",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_physical_views": len(fiben_physical_view_inventory_df),
        "n_likely_key_columns": len(fiben_likely_key_columns_df),
        "n_relationships_need_review": len(relationships_need_review_df),
        "physical_view_inventory_csv": "variables/block_v06a/fiben_physical_view_inventory.csv",
        "physical_column_dictionary_csv": "variables/block_v06a/fiben_physical_column_dictionary.csv",
        "likely_key_columns_csv": "variables/block_v06a/fiben_likely_key_columns.csv",
        "entity_mapping_diagnostic_csv": "variables/block_v06a/fiben_entity_mapping_diagnostic.csv",
        "relationships_need_manual_review_csv": "variables/block_v06a/relationships_need_manual_review.csv",
        "metadata_json": "variables/block_v06a/fiben_physical_mapping_diagnostic_metadata.json",
    },
)


# ---------------------------------------------------------
# 10. Update README section for BLOCK V6A
# ---------------------------------------------------------

block_v06a_readme_body = f"""
This diagnostic block inspects the physical FIBEN mapping before refining
the observed cardinality computation.

Main variables created:

    fiben_physical_view_inventory_df
    fiben_physical_column_dictionary_df
    fiben_likely_key_columns_df
    fiben_entity_mapping_diagnostic_df
    relationships_need_review_df

Number of physical views inspected:

    {len(fiben_physical_view_inventory_df)}

Number of likely key columns found:

    {len(fiben_likely_key_columns_df)}

Number of relationships needing manual review:

    {len(relationships_need_review_df)}

Generated reproducibility files:

    variables/block_v06a/fiben_physical_view_inventory.csv
    variables/block_v06a/fiben_physical_column_dictionary.csv
    variables/block_v06a/fiben_likely_key_columns.csv
    variables/block_v06a/fiben_entity_mapping_diagnostic.csv
    variables/block_v06a/relationships_need_manual_review.csv
    variables/block_v06a/fiben_physical_mapping_diagnostic_metadata.json

Methodological meaning:

    This block does not change the conceptual methodology.
    It helps connect the conceptual FIBEN model to the physical CSV tables
    and columns used by DuckDB.

Why this is necessary:

    Some conceptual entities may not have a direct physical table.
    Some relationships require explicit join-column hints because automatic
    inference is not always reliable.
"""

update_readme_section(
    section_title="BLOCK V6A — Inspect FIBEN Physical Mapping and Join Keys",
    section_body=block_v06a_readme_body,
)

FIBEN physical mapping diagnostics completed.

Physical view inventory:


,scope,alias,view_name,n_rows,n_columns,columns,status
0,raw,corporation,fiben_raw_corporation,2324,5,"[CORPORATIONID, ISCLASSIFIEDBY, ISDOMICILEDIN,...",ok
1,raw,country,fiben_raw_country,249,3,"[COUNTRYID, HASNAME, HASNUMERICCOUNTRYCODE]",ok
2,raw,disclosure,fiben_raw_disclosure,208158,15,"[DISCLOSUREID, HASMETRICYEARCALENDAR, HASMETRI...",ok
3,raw,elementoffinancialstatement,fiben_raw_elementoffinancialstatement,443794,15,"[ELEMENTOFFINANCIALSTATEMENTID, HASMETRICYEARC...",ok
4,raw,elementsoffinancialreport,fiben_raw_elementsoffinancialreport,8120084,2,"[ELEMENTSOFFINANCIALREPORTID, ISMEMBEROF]",ok
5,raw,financialreport,fiben_raw_financialreport,48301,3,"[FINANCIALREPORTID, ISPROVIDEDBY, HASUNIQUEIDE...",ok
6,raw,financialserviceaccount,fiben_raw_financialserviceaccount,97270,5,"[FINANCIALSERVICEACCOUNTID, ISOWNEDBY, ISMANAG...",ok
7,raw,holding,fiben_raw_holding,534473,5,"[HOLDINGID, ISHELDBY, REFERSTO, HASDESCRIPTION...",ok
8,raw,industrysectorclassifier,fiben_raw_industrysectorclassifier,452,6,"[INDUSTRYSECTORCLASSIFIERID, ISDEFINEDBY, HASU...",ok
9,raw,listedsecurity,fiben_raw_listedsecurity,2745,5,"[LISTEDSECURITYID, HASLASTTRADEDVALUE, HASLIST...",ok



Likely key columns:


,scope,alias,view_name,column_name,column_type,null,key,default,extra,normalized_column,looks_like_key
0,raw,corporation,fiben_raw_corporation,CORPORATIONID,BIGINT,YES,None,None,None,corporationid,True
1,raw,country,fiben_raw_country,COUNTRYID,BIGINT,YES,None,None,None,countryid,True
2,raw,country,fiben_raw_country,HASNUMERICCOUNTRYCODE,BIGINT,YES,None,None,None,hasnumericcountrycode,True
3,raw,disclosure,fiben_raw_disclosure,DISCLOSUREID,BIGINT,YES,None,None,None,disclosureid,True
4,raw,disclosure,fiben_raw_disclosure,HASUNIQUEIDENTIFIER,VARCHAR,YES,None,None,None,hasuniqueidentifier,True
5,raw,elementoffinancialstatement,fiben_raw_elementoffinancialstatement,ELEMENTOFFINANCIALSTATEMENTID,BIGINT,YES,None,None,None,elementoffinancialstatementid,True
6,raw,elementoffinancialstatement,fiben_raw_elementoffinancialstatement,HASUNIQUEIDENTIFIER,VARCHAR,YES,None,None,None,hasuniqueidentifier,True
7,raw,elementsoffinancialreport,fiben_raw_elementsoffinancialreport,ELEMENTSOFFINANCIALREPORTID,BIGINT,YES,None,None,None,elementsoffinancialreportid,True
8,raw,financialreport,fiben_raw_financialreport,FINANCIALREPORTID,BIGINT,YES,None,None,None,financialreportid,True
9,raw,financialreport,fiben_raw_financialreport,HASUNIQUEIDENTIFIER,VARCHAR,YES,None,None,None,hasuniqueidentifier,True



Conceptual entity mapping diagnostic:


,conceptual_entity,normalized_entity,possible_alias_matches,n_possible_alias_matches
0,Corporation,corporation,"[corporation, corporations]",2
1,Industry,industry,[industrysectorclassifier],1
2,Country,country,[country],1
3,Security,security,"[listedsecurity, security]",2
4,ListedSecurity,listedsecurity,"[listedsecurity, security]",2
5,Person,person,[person],1
6,FinancialServiceAccount,financialserviceaccount,[financialserviceaccount],1
7,Holding,holding,[holding],1
8,Transaction,transaction,[securitiestransaction],1
9,BuyTransaction,buytransaction,[],0



Relationships needing manual review:


,edge_id,relationship_name,semantic_type,source_entity,target_entity,source_view,target_view,selected_source_column,selected_target_column,join_inference_reason,observed_cardinality,computation_status,row_limit,joined_rows,source_non_null_rows,target_non_null_rows,source_distinct_keys,target_distinct_keys,joined_distinct_source_keys,joined_distinct_target_keys
0,Corporation -- Industry [corporation_has_indus...,corporation_has_industry,descriptor,Corporation,Industry,fiben_corporations,None,None,None,None,not_computed,missing_view,200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Corporation -- Country [corporation_has_country],corporation_has_country,descriptor,Corporation,Country,fiben_corporations,fiben_raw_country,None,None,None,not_computed,no_join_columns_detected,200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Corporation -- ListedSecurity [corporation_has...,corporation_has_listed_security,association,Corporation,ListedSecurity,fiben_corporations,fiben_listed_securities,None,None,None,not_computed,no_join_columns_detected,200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Person -- FinancialServiceAccount [person_owns...,person_owns_financial_service_account,ownership,Person,FinancialServiceAccount,fiben_raw_person,fiben_raw_financialserviceaccount,None,None,None,not_computed,no_join_columns_detected,200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,FinancialServiceAccount -- Holding [account_ha...,account_has_holding,containment,FinancialServiceAccount,Holding,fiben_raw_financialserviceaccount,fiben_raw_holding,None,None,None,not_computed,no_join_columns_detected,200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Holding -- ListedSecurity [holding_refers_to_l...,holding_refers_to_listed_security,association,Holding,ListedSecurity,fiben_raw_holding,fiben_listed_securities,None,None,None,not_computed,no_join_columns_detected,200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,FinancialServiceAccount -- Transaction [accoun...,account_records_transaction,containment,FinancialServiceAccount,Transaction,fiben_raw_financialserviceaccount,None,None,None,None,not_computed,missing_view,200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Transaction -- ListedSecurity [transaction_ref...,transaction_refers_to_listed_security,association,Transaction,ListedSecurity,None,fiben_listed_securities,None,None,None,not_computed,missing_view,200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,BuyTransaction -- Transaction [buy_transaction...,buy_transaction_is_transaction,subtype,BuyTransaction,Transaction,None,None,None,None,None,not_computed,missing_view,200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,SellTransaction -- Transaction [sell_transacti...,sell_transaction_is_transaction,subtype,SellTransaction,Transaction,None,None,None,None,None,not_computed,missing_view,200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06a/fiben_physical_view_inventory.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06a/fiben_physical_column_dictionary.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06a/fiben_likely_key_columns.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06a/fiben_entity_mapping_diagnostic.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06a/relationships_need_manual_review.csv
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06a/fiben_physical_mapping_diagnostic_metadata.json
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framew

In [54]:
fiben_entity_mapping_diagnostic_df

,conceptual_entity,normalized_entity,possible_alias_matches,n_possible_alias_matches
0,Corporation,corporation,"[corporation, corporations]",2
1,Industry,industry,[industrysectorclassifier],1
2,Country,country,[country],1
3,Security,security,"[listedsecurity, security]",2
4,ListedSecurity,listedsecurity,"[listedsecurity, security]",2
5,Person,person,[person],1
6,FinancialServiceAccount,financialserviceaccount,[financialserviceaccount],1
7,Holding,holding,[holding],1
8,Transaction,transaction,[securitiestransaction],1
9,BuyTransaction,buytransaction,[],0


In [55]:
fiben_likely_key_columns_df[
    ["scope", "alias", "view_name", "column_name", "column_type"]
]

,scope,alias,view_name,column_name,column_type
0,raw,corporation,fiben_raw_corporation,CORPORATIONID,BIGINT
1,raw,country,fiben_raw_country,COUNTRYID,BIGINT
2,raw,country,fiben_raw_country,HASNUMERICCOUNTRYCODE,BIGINT
3,raw,disclosure,fiben_raw_disclosure,DISCLOSUREID,BIGINT
4,raw,disclosure,fiben_raw_disclosure,HASUNIQUEIDENTIFIER,VARCHAR
5,raw,elementoffinancialstatement,fiben_raw_elementoffinancialstatement,ELEMENTOFFINANCIALSTATEMENTID,BIGINT
6,raw,elementoffinancialstatement,fiben_raw_elementoffinancialstatement,HASUNIQUEIDENTIFIER,VARCHAR
7,raw,elementsoffinancialreport,fiben_raw_elementsoffinancialreport,ELEMENTSOFFINANCIALREPORTID,BIGINT
8,raw,financialreport,fiben_raw_financialreport,FINANCIALREPORTID,BIGINT
9,raw,financialreport,fiben_raw_financialreport,HASUNIQUEIDENTIFIER,VARCHAR


In [56]:
relationships_need_review_df[
    ["relationship_name", "source_entity", "target_entity", "source_view", "target_view", "computation_status"]
]

,relationship_name,source_entity,target_entity,source_view,target_view,computation_status
0,corporation_has_industry,Corporation,Industry,fiben_corporations,None,missing_view
1,corporation_has_country,Corporation,Country,fiben_corporations,fiben_raw_country,no_join_columns_detected
2,corporation_has_listed_security,Corporation,ListedSecurity,fiben_corporations,fiben_listed_securities,no_join_columns_detected
4,person_owns_financial_service_account,Person,FinancialServiceAccount,fiben_raw_person,fiben_raw_financialserviceaccount,no_join_columns_detected
5,account_has_holding,FinancialServiceAccount,Holding,fiben_raw_financialserviceaccount,fiben_raw_holding,no_join_columns_detected
6,holding_refers_to_listed_security,Holding,ListedSecurity,fiben_raw_holding,fiben_listed_securities,no_join_columns_detected
7,account_records_transaction,FinancialServiceAccount,Transaction,fiben_raw_financialserviceaccount,None,missing_view
8,transaction_refers_to_listed_security,Transaction,ListedSecurity,None,fiben_listed_securities,missing_view
9,buy_transaction_is_transaction,BuyTransaction,Transaction,None,None,missing_view
10,sell_transaction_is_transaction,SellTransaction,Transaction,None,None,missing_view


In [57]:
# =========================================================
# BLOCK V6B — MANUAL FIBEN ENTITY VIEW OVERRIDES
# =========================================================
#
# Purpose:
# This block fixes the physical mapping between conceptual FIBEN
# entities and DuckDB views.
#
# The automatic mapping from BLOCK 3 did not resolve all entities.
# The diagnostic BLOCK V6A showed that several conceptual entities
# exist under different physical table names.
#
# This block creates canonical semantic views for those entities and
# stores manual entity-to-view overrides.
#
# Important:
# BuyTransaction and SellTransaction are modeled as conceptual subtypes.
# In the current physical FIBEN files, they appear to be represented by
# the same physical securities transaction table.
#
# If a transaction-type discriminator column is identified later, these
# two views can be refined with WHERE filters.
# =========================================================

from pathlib import Path
import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "fiben_data_bundle",
    "fiben_scenario",
    "fiben_physical_view_inventory_df",
    "fiben_physical_column_dictionary_df",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first BLOCK V6A — INSPECT FIBEN PHYSICAL MAPPING AND JOIN KEYS."
        )


# ---------------------------------------------------------
# 2. Helper: check that a DuckDB view exists
# ---------------------------------------------------------

def duckdb_view_exists(con, view_name: str) -> bool:
    """
    Return True if a DuckDB view/table can be queried.

    This is a practical existence check used before creating
    semantic views from raw views.
    """
    try:
        con.execute(f"SELECT * FROM {view_name} LIMIT 1").df()
        return True
    except Exception:
        return False


# ---------------------------------------------------------
# 3. Define manual semantic view specifications
# ---------------------------------------------------------
#
# Each entry maps a canonical semantic view name to an existing
# physical/raw view.
#
# These mappings are based on the diagnostics produced by BLOCK V6A.
# ---------------------------------------------------------

FIBEN_SEMANTIC_VIEW_SPECS = {
    # Existing conceptual entities with clearer semantic aliases
    "fiben_industries": "fiben_raw_industrysectorclassifier",
    "fiben_countries": "fiben_raw_country",
    "fiben_persons": "fiben_raw_person",
    "fiben_financial_service_accounts": "fiben_raw_financialserviceaccount",
    "fiben_holdings": "fiben_raw_holding",
    "fiben_transactions": "fiben_raw_securitiestransaction",
    "fiben_report_elements": "fiben_raw_elementsoffinancialreport",
    "fiben_statement_elements": "fiben_raw_elementoffinancialstatement",

    # Conceptual subtypes represented physically by securities transactions.
    # These are pass-through aliases for now.
    "fiben_buy_transactions": "fiben_raw_securitiestransaction",
    "fiben_sell_transactions": "fiben_raw_securitiestransaction",
}


# ---------------------------------------------------------
# 4. Create semantic views
# ---------------------------------------------------------

created_or_replaced_semantic_views = []
missing_source_views = []

for semantic_view, source_view in FIBEN_SEMANTIC_VIEW_SPECS.items():
    if not duckdb_view_exists(fiben_data_bundle.con, source_view):
        missing_source_views.append({
            "semantic_view": semantic_view,
            "source_view": source_view,
            "status": "missing_source_view",
        })
        continue

    sql = f"""
    CREATE OR REPLACE VIEW {semantic_view} AS
    SELECT *
    FROM {source_view};
    """

    fiben_data_bundle.con.execute(sql)

    created_or_replaced_semantic_views.append({
        "semantic_view": semantic_view,
        "source_view": source_view,
        "status": "created_or_replaced",
    })


fiben_semantic_view_creation_df = pd.DataFrame(
    created_or_replaced_semantic_views + missing_source_views
)


# ---------------------------------------------------------
# 5. Update the bundle semantic view registry
# ---------------------------------------------------------
#
# The keys are conceptual aliases used by the methodology.
# The values are the DuckDB semantic views.
# ---------------------------------------------------------

FIBEN_ENTITY_VIEW_OVERRIDES = {
    "Corporation": "fiben_corporations",
    "Industry": "fiben_industries",
    "Country": "fiben_countries",
    "Security": "fiben_securities",
    "ListedSecurity": "fiben_listed_securities",
    "Person": "fiben_persons",
    "FinancialServiceAccount": "fiben_financial_service_accounts",
    "Holding": "fiben_holdings",
    "Transaction": "fiben_transactions",
    "BuyTransaction": "fiben_buy_transactions",
    "SellTransaction": "fiben_sell_transactions",
    "FinancialReport": "fiben_reports",
    "ReportElement": "fiben_report_elements",
    "StatementElement": "fiben_statement_elements",
    "Disclosure": "fiben_disclosures",
}

# Register semantic aliases in the bundle.
# These names are normalized plural aliases, not necessarily class names.
fiben_data_bundle.semantic_views.update({
    "industries": "fiben_industries",
    "countries": "fiben_countries",
    "persons": "fiben_persons",
    "financial_service_accounts": "fiben_financial_service_accounts",
    "holdings": "fiben_holdings",
    "transactions": "fiben_transactions",
    "buy_transactions": "fiben_buy_transactions",
    "sell_transactions": "fiben_sell_transactions",
    "report_elements": "fiben_report_elements",
    "statement_elements": "fiben_statement_elements",
})


# ---------------------------------------------------------
# 6. Validate entity overrides
# ---------------------------------------------------------

entity_override_validation_rows = []

for entity, view_name in FIBEN_ENTITY_VIEW_OVERRIDES.items():
    exists = duckdb_view_exists(fiben_data_bundle.con, view_name)

    if exists:
        try:
            n_rows = fiben_data_bundle.con.execute(
                f"SELECT COUNT(*) AS n FROM {view_name}"
            ).fetchone()[0]

            columns = fiben_data_bundle.con.execute(
                f"DESCRIBE {view_name}"
            ).df()["column_name"].tolist()

            status = "ok"

        except Exception as e:
            n_rows = None
            columns = []
            status = f"error: {type(e).__name__}: {e}"
    else:
        n_rows = None
        columns = []
        status = "missing_view"

    entity_override_validation_rows.append({
        "conceptual_entity": entity,
        "override_view": view_name,
        "view_exists": exists,
        "status": status,
        "n_rows": n_rows,
        "n_columns": len(columns),
        "columns": columns,
    })

fiben_entity_view_override_validation_df = pd.DataFrame(
    entity_override_validation_rows
)


# ---------------------------------------------------------
# 7. Inspect important physical columns for manual joins
# ---------------------------------------------------------
#
# This table is intentionally broad. It helps us define manual join
# hints in the next V6 revision.
# ---------------------------------------------------------

important_entities = [
    "Corporation",
    "Industry",
    "Country",
    "Security",
    "ListedSecurity",
    "Person",
    "FinancialServiceAccount",
    "Holding",
    "Transaction",
    "FinancialReport",
    "ReportElement",
    "StatementElement",
    "Disclosure",
]

important_column_rows = []

for entity in important_entities:
    view_name = FIBEN_ENTITY_VIEW_OVERRIDES.get(entity)

    if view_name is None or not duckdb_view_exists(fiben_data_bundle.con, view_name):
        continue

    desc_df = fiben_data_bundle.con.execute(
        f"DESCRIBE {view_name}"
    ).df()

    for _, col in desc_df.iterrows():
        important_column_rows.append({
            "conceptual_entity": entity,
            "view_name": view_name,
            "column_name": col["column_name"],
            "column_type": col["column_type"],
        })

fiben_entity_override_columns_df = pd.DataFrame(important_column_rows)


# ---------------------------------------------------------
# 8. Display outputs
# ---------------------------------------------------------

print("Manual FIBEN entity view overrides were created successfully.")

print("\nSemantic views created or replaced:")
display(fiben_semantic_view_creation_df)

print("\nEntity view override validation:")
display(fiben_entity_view_override_validation_df)

print("\nColumns available in override views:")
display(fiben_entity_override_columns_df)


# ---------------------------------------------------------
# 9. Save BLOCK V6B outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    fiben_semantic_view_creation_df,
    "variables/block_v06b/fiben_semantic_view_creation.csv",
)

save_dataframe(
    fiben_entity_view_override_validation_df,
    "variables/block_v06b/fiben_entity_view_override_validation.csv",
)

save_dataframe(
    fiben_entity_override_columns_df,
    "variables/block_v06b/fiben_entity_override_columns.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "semantic_view_specs": FIBEN_SEMANTIC_VIEW_SPECS,
        "entity_view_overrides": FIBEN_ENTITY_VIEW_OVERRIDES,
        "n_semantic_views_created_or_replaced": int(
            (fiben_semantic_view_creation_df["status"] == "created_or_replaced").sum()
        ),
        "n_missing_source_views": int(
            (fiben_semantic_view_creation_df["status"] == "missing_source_view").sum()
        ),
        "n_entity_overrides": len(FIBEN_ENTITY_VIEW_OVERRIDES),
        "n_valid_entity_overrides": int(
            (fiben_entity_view_override_validation_df["view_exists"] == True).sum()
        ),
        "output_files": {
            "semantic_view_creation_csv": "variables/block_v06b/fiben_semantic_view_creation.csv",
            "entity_view_override_validation_csv": "variables/block_v06b/fiben_entity_view_override_validation.csv",
            "entity_override_columns_csv": "variables/block_v06b/fiben_entity_override_columns.csv",
        },
    },
    "variables/block_v06b/fiben_entity_view_overrides_metadata.json",
)

write_execution_log(
    block_name="BLOCK V6B — MANUAL FIBEN ENTITY VIEW OVERRIDES",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_entity_overrides": len(FIBEN_ENTITY_VIEW_OVERRIDES),
        "n_valid_entity_overrides": int(
            (fiben_entity_view_override_validation_df["view_exists"] == True).sum()
        ),
        "semantic_view_creation_csv": "variables/block_v06b/fiben_semantic_view_creation.csv",
        "entity_view_override_validation_csv": "variables/block_v06b/fiben_entity_view_override_validation.csv",
        "entity_override_columns_csv": "variables/block_v06b/fiben_entity_override_columns.csv",
        "metadata_json": "variables/block_v06b/fiben_entity_view_overrides_metadata.json",
    },
)


# ---------------------------------------------------------
# 10. Update README section for BLOCK V6B
# ---------------------------------------------------------

block_v06b_readme_body = f"""
This block creates manual entity-to-view overrides for the FIBEN physical mapping.

The diagnostic BLOCK V6A showed that some conceptual entities were not mapped
automatically because the physical FIBEN table names differ from the conceptual
entity names.

Main variable created:

    FIBEN_ENTITY_VIEW_OVERRIDES

Additional variables created:

    FIBEN_SEMANTIC_VIEW_SPECS
    fiben_semantic_view_creation_df
    fiben_entity_view_override_validation_df
    fiben_entity_override_columns_df

Number of entity overrides:

    {len(FIBEN_ENTITY_VIEW_OVERRIDES)}

Number of valid entity overrides:

    {int((fiben_entity_view_override_validation_df["view_exists"] == True).sum())}

Generated reproducibility files:

    variables/block_v06b/fiben_semantic_view_creation.csv
    variables/block_v06b/fiben_entity_view_override_validation.csv
    variables/block_v06b/fiben_entity_override_columns.csv
    variables/block_v06b/fiben_entity_view_overrides_metadata.json

Important modeling note:

    BuyTransaction and SellTransaction are currently semantic aliases over
    the SecuritiesTransaction physical table.

    If a transaction-type discriminator column is identified later, these
    views should be refined with WHERE filters.

Why this block is necessary:

    The observed cardinality block needs a reliable mapping from conceptual
    entities to physical DuckDB views before computing join statistics.
"""

update_readme_section(
    section_title="BLOCK V6B — Manual FIBEN Entity View Overrides",
    section_body=block_v06b_readme_body,
)

Manual FIBEN entity view overrides were created successfully.

Semantic views created or replaced:


,semantic_view,source_view,status
0,fiben_industries,fiben_raw_industrysectorclassifier,created_or_replaced
1,fiben_countries,fiben_raw_country,created_or_replaced
2,fiben_persons,fiben_raw_person,created_or_replaced
3,fiben_financial_service_accounts,fiben_raw_financialserviceaccount,created_or_replaced
4,fiben_holdings,fiben_raw_holding,created_or_replaced
5,fiben_transactions,fiben_raw_securitiestransaction,created_or_replaced
6,fiben_report_elements,fiben_raw_elementsoffinancialreport,created_or_replaced
7,fiben_statement_elements,fiben_raw_elementoffinancialstatement,created_or_replaced
8,fiben_buy_transactions,fiben_raw_securitiestransaction,created_or_replaced
9,fiben_sell_transactions,fiben_raw_securitiestransaction,created_or_replaced



Entity view override validation:


,conceptual_entity,override_view,view_exists,status,n_rows,n_columns,columns
0,Corporation,fiben_corporations,True,ok,2324,5,"[CORPORATIONID, ISCLASSIFIEDBY, ISDOMICILEDIN,..."
1,Industry,fiben_industries,True,ok,452,6,"[INDUSTRYSECTORCLASSIFIERID, ISDEFINEDBY, HASU..."
2,Country,fiben_countries,True,ok,249,3,"[COUNTRYID, HASNAME, HASNUMERICCOUNTRYCODE]"
3,Security,fiben_securities,True,ok,2745,3,"[SECURITYID, ISTRADEDON, ISPROVIDEDBY]"
4,ListedSecurity,fiben_listed_securities,True,ok,2745,5,"[LISTEDSECURITYID, HASLASTTRADEDVALUE, HASLIST..."
5,Person,fiben_persons,True,ok,50100,9,"[PERSONID, HASPLACEOFBIRTH, HASCITIZENSHIP, HA..."
6,FinancialServiceAccount,fiben_financial_service_accounts,True,ok,97270,5,"[FINANCIALSERVICEACCOUNTID, ISOWNEDBY, ISMANAG..."
7,Holding,fiben_holdings,True,ok,534473,5,"[HOLDINGID, ISHELDBY, REFERSTO, HASDESCRIPTION..."
8,Transaction,fiben_transactions,True,ok,500000,7,"[SECURITIESTRANSACTIONID, ISFACILITATEDBY, HAS..."
9,BuyTransaction,fiben_buy_transactions,True,ok,500000,7,"[SECURITIESTRANSACTIONID, ISFACILITATEDBY, HAS..."



Columns available in override views:


,conceptual_entity,view_name,column_name,column_type
0,Corporation,fiben_corporations,CORPORATIONID,BIGINT
1,Corporation,fiben_corporations,ISCLASSIFIEDBY,BIGINT
2,Corporation,fiben_corporations,ISDOMICILEDIN,BIGINT
3,Corporation,fiben_corporations,HASTICKERSYMBOL,VARCHAR
4,Corporation,fiben_corporations,HASLEGALNAME,VARCHAR
...,...,...,...,...
78,Disclosure,fiben_disclosures,HASMETRICVALUE,DOUBLE
79,Disclosure,fiben_disclosures,HASMETRICYEARFISCAL,BIGINT
80,Disclosure,fiben_disclosures,HASURL,VARCHAR
81,Disclosure,fiben_disclosures,HASMETRICCONTEXTDIMENSIONS,VARCHAR


Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06b/fiben_semantic_view_creation.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06b/fiben_entity_view_override_validation.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06b/fiben_entity_override_columns.csv
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06b/fiben_entity_view_overrides_metadata.json
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/logs/block_v6b___manual_fiben_entity_view_overrides_execution_log.json
Updated README section: BLOCK V6B — Manual FIBEN Entity View Overrides
README path: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/README_FIBEN_Implementation_Framework.md


In [58]:
fiben_entity_view_override_validation_df[
    ["conceptual_entity", "override_view", "view_exists", "n_rows", "n_columns"]
]

,conceptual_entity,override_view,view_exists,n_rows,n_columns
0,Corporation,fiben_corporations,True,2324,5
1,Industry,fiben_industries,True,452,6
2,Country,fiben_countries,True,249,3
3,Security,fiben_securities,True,2745,3
4,ListedSecurity,fiben_listed_securities,True,2745,5
5,Person,fiben_persons,True,50100,9
6,FinancialServiceAccount,fiben_financial_service_accounts,True,97270,5
7,Holding,fiben_holdings,True,534473,5
8,Transaction,fiben_transactions,True,500000,7
9,BuyTransaction,fiben_buy_transactions,True,500000,7


In [59]:
fiben_entity_override_columns_df[
    ["conceptual_entity", "view_name", "column_name", "column_type"]
]

,conceptual_entity,view_name,column_name,column_type
0,Corporation,fiben_corporations,CORPORATIONID,BIGINT
1,Corporation,fiben_corporations,ISCLASSIFIEDBY,BIGINT
2,Corporation,fiben_corporations,ISDOMICILEDIN,BIGINT
3,Corporation,fiben_corporations,HASTICKERSYMBOL,VARCHAR
4,Corporation,fiben_corporations,HASLEGALNAME,VARCHAR
...,...,...,...,...
78,Disclosure,fiben_disclosures,HASMETRICVALUE,DOUBLE
79,Disclosure,fiben_disclosures,HASMETRICYEARFISCAL,BIGINT
80,Disclosure,fiben_disclosures,HASURL,VARCHAR
81,Disclosure,fiben_disclosures,HASMETRICCONTEXTDIMENSIONS,VARCHAR


In [64]:
output_path = FIBEN_PROJECT_OUTPUT_DIR / "variables/block_v06b/fiben_entity_override_columns_full.csv"

fiben_entity_override_columns_df[
    ["conceptual_entity", "view_name", "column_name", "column_type"]
].to_csv(output_path, index=False)

print(output_path)

/home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06b/fiben_entity_override_columns_full.csv


In [65]:
# =========================================================
# BLOCK V6C — MANUAL FIBEN RELATIONSHIP JOIN HINTS
# =========================================================
#
# Purpose:
# This block defines manual join hints for the FIBEN conceptual
# relationships.
#
# The previous diagnostic blocks showed that automatic join inference
# is not sufficient because the FIBEN physical columns use ontology-like
# names such as:
#
#     ISCLASSIFIEDBY
#     ISDOMICILEDIN
#     ISOWNEDBY
#     ISHELDBY
#     REFERSTO
#     ISPROVIDEDBY
#
# This block stores explicit relationship-to-column mappings that will
# be used by the revised observed cardinality block.
# =========================================================

import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "fiben_data_bundle",
    "fiben_scenario",
    "conceptual_relationships_df",
    "FIBEN_ENTITY_VIEW_OVERRIDES",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first BLOCK V6B — MANUAL FIBEN ENTITY VIEW OVERRIDES."
        )


# ---------------------------------------------------------
# 2. Manual relationship join hints
# ---------------------------------------------------------
#
# Supported hint types:
#
# 1. direct:
#    source_view.source_column = target_view.target_column
#
# 2. bridge:
#    source_view.source_column = bridge_view.source_to_bridge_bridge_column
#    bridge_view.bridge_to_target_bridge_column = target_view.target_column
#
# 3. subtype_alias:
#    conceptual subtype represented by the same physical transaction table.
# ---------------------------------------------------------

FIBEN_MANUAL_RELATIONSHIP_JOIN_HINTS = {
    "corporation_has_industry": {
        "hint_type": "direct",
        "source_entity": "Corporation",
        "target_entity": "Industry",
        "source_view": "fiben_corporations",
        "target_view": "fiben_industries",
        "source_column": "ISCLASSIFIEDBY",
        "target_column": "INDUSTRYSECTORCLASSIFIERID",
        "confidence": "high",
        "notes": "Corporation points to IndustrySectorClassifier through ISCLASSIFIEDBY.",
    },

    "corporation_has_country": {
        "hint_type": "direct",
        "source_entity": "Corporation",
        "target_entity": "Country",
        "source_view": "fiben_corporations",
        "target_view": "fiben_countries",
        "source_column": "ISDOMICILEDIN",
        "target_column": "COUNTRYID",
        "confidence": "high",
        "notes": "Corporation points to Country through ISDOMICILEDIN.",
    },

    "corporation_has_listed_security": {
        "hint_type": "bridge",
        "source_entity": "Corporation",
        "target_entity": "ListedSecurity",
        "source_view": "fiben_corporations",
        "bridge_view": "fiben_securities",
        "target_view": "fiben_listed_securities",
        "source_column": "CORPORATIONID",
        "source_to_bridge_bridge_column": "ISPROVIDEDBY",
        "bridge_to_target_bridge_column": "ISTRADEDON",
        "target_column": "LISTEDSECURITYID",
        "confidence": "medium",
        "notes": "Derived path Corporation -> Security -> ListedSecurity.",
    },

    "listed_security_represents_security": {
        "hint_type": "direct",
        "source_entity": "ListedSecurity",
        "target_entity": "Security",
        "source_view": "fiben_listed_securities",
        "target_view": "fiben_securities",
        "source_column": "LISTEDSECURITYID",
        "target_column": "ISTRADEDON",
        "confidence": "medium",
        "notes": "Security appears to point to ListedSecurity through ISTRADEDON.",
    },

    "person_owns_financial_service_account": {
        "hint_type": "direct",
        "source_entity": "Person",
        "target_entity": "FinancialServiceAccount",
        "source_view": "fiben_persons",
        "target_view": "fiben_financial_service_accounts",
        "source_column": "PERSONID",
        "target_column": "ISOWNEDBY",
        "confidence": "high",
        "notes": "FinancialServiceAccount points to Person through ISOWNEDBY.",
    },

    "account_has_holding": {
        "hint_type": "direct",
        "source_entity": "FinancialServiceAccount",
        "target_entity": "Holding",
        "source_view": "fiben_financial_service_accounts",
        "target_view": "fiben_holdings",
        "source_column": "FINANCIALSERVICEACCOUNTID",
        "target_column": "ISHELDBY",
        "confidence": "high",
        "notes": "Holding points to FinancialServiceAccount through ISHELDBY.",
    },

    "holding_refers_to_listed_security": {
        "hint_type": "direct",
        "source_entity": "Holding",
        "target_entity": "ListedSecurity",
        "source_view": "fiben_holdings",
        "target_view": "fiben_listed_securities",
        "source_column": "REFERSTO",
        "target_column": "LISTEDSECURITYID",
        "confidence": "high",
        "notes": "Holding points to ListedSecurity through REFERSTO.",
    },

    "account_records_transaction": {
        "hint_type": "direct",
        "source_entity": "FinancialServiceAccount",
        "target_entity": "Transaction",
        "source_view": "fiben_financial_service_accounts",
        "target_view": "fiben_transactions",
        "source_column": "FINANCIALSERVICEACCOUNTID",
        "target_column": "ISFACILITATEDBY",
        "confidence": "high",
        "notes": "SecuritiesTransaction points to FinancialServiceAccount through ISFACILITATEDBY.",
    },

    "transaction_refers_to_listed_security": {
        "hint_type": "direct",
        "source_entity": "Transaction",
        "target_entity": "ListedSecurity",
        "source_view": "fiben_transactions",
        "target_view": "fiben_listed_securities",
        "source_column": "REFERSTO",
        "target_column": "LISTEDSECURITYID",
        "confidence": "high",
        "notes": "SecuritiesTransaction points to ListedSecurity through REFERSTO.",
    },

    "buy_transaction_is_transaction": {
        "hint_type": "subtype_alias",
        "source_entity": "BuyTransaction",
        "target_entity": "Transaction",
        "source_view": "fiben_buy_transactions",
        "target_view": "fiben_transactions",
        "source_column": "SECURITIESTRANSACTIONID",
        "target_column": "SECURITIESTRANSACTIONID",
        "confidence": "medium",
        "notes": "BuyTransaction is currently a conceptual alias over SecuritiesTransaction.",
    },

    "sell_transaction_is_transaction": {
        "hint_type": "subtype_alias",
        "source_entity": "SellTransaction",
        "target_entity": "Transaction",
        "source_view": "fiben_sell_transactions",
        "target_view": "fiben_transactions",
        "source_column": "SECURITIESTRANSACTIONID",
        "target_column": "SECURITIESTRANSACTIONID",
        "confidence": "medium",
        "notes": "SellTransaction is currently a conceptual alias over SecuritiesTransaction.",
    },

    "corporation_has_financial_report": {
        "hint_type": "direct",
        "source_entity": "Corporation",
        "target_entity": "FinancialReport",
        "source_view": "fiben_corporations",
        "target_view": "fiben_reports",
        "source_column": "CORPORATIONID",
        "target_column": "ISPROVIDEDBY",
        "confidence": "high",
        "notes": "FinancialReport points to Corporation through ISPROVIDEDBY.",
    },

    "financial_report_contains_report_element": {
        "hint_type": "direct",
        "source_entity": "FinancialReport",
        "target_entity": "ReportElement",
        "source_view": "fiben_reports",
        "target_view": "fiben_report_elements",
        "source_column": "FINANCIALREPORTID",
        "target_column": "ISMEMBEROF",
        "confidence": "high",
        "notes": "ReportElement points to FinancialReport through ISMEMBEROF.",
    },

    "report_element_has_statement_element": {
        "hint_type": "direct",
        "source_entity": "ReportElement",
        "target_entity": "StatementElement",
        "source_view": "fiben_report_elements",
        "target_view": "fiben_statement_elements",
        "source_column": "ELEMENTSOFFINANCIALREPORTID",
        "target_column": "ELEMENTOFFINANCIALSTATEMENTID",
        "confidence": "low",
        "notes": "Tentative mapping. Needs validation with observed matches.",
    },

    "financial_report_contains_disclosure": {
        "hint_type": "direct",
        "source_entity": "FinancialReport",
        "target_entity": "Disclosure",
        "source_view": "fiben_reports",
        "target_view": "fiben_disclosures",
        "source_column": "HASUNIQUEIDENTIFIER",
        "target_column": "HASUNIQUEIDENTIFIER",
        "confidence": "low",
        "notes": "Tentative mapping because Disclosure has no obvious report foreign key.",
    },
}


# ---------------------------------------------------------
# 3. Helper functions for validation
# ---------------------------------------------------------

def duckdb_view_exists(con, view_name: str) -> bool:
    """
    Return True if the DuckDB view can be queried.
    """
    try:
        con.execute(f"SELECT * FROM {view_name} LIMIT 1").df()
        return True
    except Exception:
        return False


def duckdb_column_exists(con, view_name: str, column_name: str) -> bool:
    """
    Return True if a column exists in a DuckDB view.
    """
    try:
        columns = con.execute(f"DESCRIBE {view_name}").df()["column_name"].tolist()
        return column_name in columns
    except Exception:
        return False


# ---------------------------------------------------------
# 4. Validate manual hints
# ---------------------------------------------------------

hint_validation_rows = []

for relationship_name, hint in FIBEN_MANUAL_RELATIONSHIP_JOIN_HINTS.items():
    hint_type = hint["hint_type"]

    if hint_type in ["direct", "subtype_alias"]:
        source_view = hint["source_view"]
        target_view = hint["target_view"]
        source_column = hint["source_column"]
        target_column = hint["target_column"]

        source_view_exists = duckdb_view_exists(fiben_data_bundle.con, source_view)
        target_view_exists = duckdb_view_exists(fiben_data_bundle.con, target_view)

        source_column_exists = (
            duckdb_column_exists(fiben_data_bundle.con, source_view, source_column)
            if source_view_exists else False
        )

        target_column_exists = (
            duckdb_column_exists(fiben_data_bundle.con, target_view, target_column)
            if target_view_exists else False
        )

        is_valid = (
            source_view_exists
            and target_view_exists
            and source_column_exists
            and target_column_exists
        )

        hint_validation_rows.append({
            "relationship_name": relationship_name,
            "hint_type": hint_type,
            "source_entity": hint["source_entity"],
            "target_entity": hint["target_entity"],
            "source_view": source_view,
            "target_view": target_view,
            "source_column": source_column,
            "target_column": target_column,
            "bridge_view": None,
            "source_view_exists": source_view_exists,
            "target_view_exists": target_view_exists,
            "bridge_view_exists": None,
            "source_column_exists": source_column_exists,
            "target_column_exists": target_column_exists,
            "bridge_columns_exist": None,
            "confidence": hint["confidence"],
            "is_valid": is_valid,
            "notes": hint["notes"],
        })

    elif hint_type == "bridge":
        source_view = hint["source_view"]
        bridge_view = hint["bridge_view"]
        target_view = hint["target_view"]

        source_view_exists = duckdb_view_exists(fiben_data_bundle.con, source_view)
        bridge_view_exists = duckdb_view_exists(fiben_data_bundle.con, bridge_view)
        target_view_exists = duckdb_view_exists(fiben_data_bundle.con, target_view)

        source_column_exists = (
            duckdb_column_exists(fiben_data_bundle.con, source_view, hint["source_column"])
            if source_view_exists else False
        )

        bridge_col_1_exists = (
            duckdb_column_exists(
                fiben_data_bundle.con,
                bridge_view,
                hint["source_to_bridge_bridge_column"],
            )
            if bridge_view_exists else False
        )

        bridge_col_2_exists = (
            duckdb_column_exists(
                fiben_data_bundle.con,
                bridge_view,
                hint["bridge_to_target_bridge_column"],
            )
            if bridge_view_exists else False
        )

        target_column_exists = (
            duckdb_column_exists(fiben_data_bundle.con, target_view, hint["target_column"])
            if target_view_exists else False
        )

        bridge_columns_exist = bridge_col_1_exists and bridge_col_2_exists

        is_valid = (
            source_view_exists
            and bridge_view_exists
            and target_view_exists
            and source_column_exists
            and bridge_columns_exist
            and target_column_exists
        )

        hint_validation_rows.append({
            "relationship_name": relationship_name,
            "hint_type": hint_type,
            "source_entity": hint["source_entity"],
            "target_entity": hint["target_entity"],
            "source_view": source_view,
            "target_view": target_view,
            "source_column": hint["source_column"],
            "target_column": hint["target_column"],
            "bridge_view": bridge_view,
            "source_view_exists": source_view_exists,
            "target_view_exists": target_view_exists,
            "bridge_view_exists": bridge_view_exists,
            "source_column_exists": source_column_exists,
            "target_column_exists": target_column_exists,
            "bridge_columns_exist": bridge_columns_exist,
            "confidence": hint["confidence"],
            "is_valid": is_valid,
            "notes": hint["notes"],
        })

    else:
        hint_validation_rows.append({
            "relationship_name": relationship_name,
            "hint_type": hint_type,
            "source_entity": hint.get("source_entity"),
            "target_entity": hint.get("target_entity"),
            "source_view": hint.get("source_view"),
            "target_view": hint.get("target_view"),
            "source_column": hint.get("source_column"),
            "target_column": hint.get("target_column"),
            "bridge_view": hint.get("bridge_view"),
            "source_view_exists": None,
            "target_view_exists": None,
            "bridge_view_exists": None,
            "source_column_exists": None,
            "target_column_exists": None,
            "bridge_columns_exist": None,
            "confidence": hint.get("confidence"),
            "is_valid": False,
            "notes": "Unsupported hint type.",
        })


fiben_relationship_join_hints_df = pd.DataFrame(hint_validation_rows)

invalid_join_hints_df = fiben_relationship_join_hints_df[
    fiben_relationship_join_hints_df["is_valid"] == False
].copy()


# ---------------------------------------------------------
# 5. Display validation
# ---------------------------------------------------------

print("Manual FIBEN relationship join hints were created.")

print("\nJoin hints validation:")
display(fiben_relationship_join_hints_df)

if not invalid_join_hints_df.empty:
    print("\nWarning: some join hints are invalid and must be reviewed.")
    display(invalid_join_hints_df)
else:
    print("\nAll manual join hints are structurally valid.")


# ---------------------------------------------------------
# 6. Save BLOCK V6C outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    fiben_relationship_join_hints_df,
    "variables/block_v06c/fiben_relationship_join_hints.csv",
)

save_dataframe(
    invalid_join_hints_df,
    "variables/block_v06c/invalid_relationship_join_hints.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "relationship_join_hints": FIBEN_MANUAL_RELATIONSHIP_JOIN_HINTS,
        "n_join_hints": len(FIBEN_MANUAL_RELATIONSHIP_JOIN_HINTS),
        "n_invalid_join_hints": len(invalid_join_hints_df),
        "output_files": {
            "join_hints_csv": "variables/block_v06c/fiben_relationship_join_hints.csv",
            "invalid_join_hints_csv": "variables/block_v06c/invalid_relationship_join_hints.csv",
        },
    },
    "variables/block_v06c/fiben_relationship_join_hints_metadata.json",
)

write_execution_log(
    block_name="BLOCK V6C — MANUAL FIBEN RELATIONSHIP JOIN HINTS",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_join_hints": len(FIBEN_MANUAL_RELATIONSHIP_JOIN_HINTS),
        "n_invalid_join_hints": len(invalid_join_hints_df),
        "join_hints_csv": "variables/block_v06c/fiben_relationship_join_hints.csv",
        "invalid_join_hints_csv": "variables/block_v06c/invalid_relationship_join_hints.csv",
        "metadata_json": "variables/block_v06c/fiben_relationship_join_hints_metadata.json",
    },
)


# ---------------------------------------------------------
# 7. Update README section for BLOCK V6C
# ---------------------------------------------------------

block_v06c_readme_body = f"""
This block defines manual join hints for the FIBEN conceptual relationships.

Main variable created:

    FIBEN_MANUAL_RELATIONSHIP_JOIN_HINTS

Additional variables created:

    fiben_relationship_join_hints_df
    invalid_join_hints_df

Number of manual join hints:

    {len(FIBEN_MANUAL_RELATIONSHIP_JOIN_HINTS)}

Number of invalid join hints:

    {len(invalid_join_hints_df)}

Generated reproducibility files:

    variables/block_v06c/fiben_relationship_join_hints.csv
    variables/block_v06c/invalid_relationship_join_hints.csv
    variables/block_v06c/fiben_relationship_join_hints_metadata.json

Important notes:

    Most joins are direct foreign-key-like relationships.

    The relationship corporation_has_listed_security is treated as a bridge
    relationship through Security:

        Corporation -> Security -> ListedSecurity

    BuyTransaction and SellTransaction are currently modeled as subtype aliases
    over the SecuritiesTransaction physical table.

    Some low-confidence hints still need empirical validation through observed
    join matches.
"""

update_readme_section(
    section_title="BLOCK V6C — Manual FIBEN Relationship Join Hints",
    section_body=block_v06c_readme_body,
)

Manual FIBEN relationship join hints were created.

Join hints validation:


,relationship_name,hint_type,source_entity,target_entity,source_view,target_view,source_column,target_column,bridge_view,source_view_exists,target_view_exists,bridge_view_exists,source_column_exists,target_column_exists,bridge_columns_exist,confidence,is_valid,notes
0,corporation_has_industry,direct,Corporation,Industry,fiben_corporations,fiben_industries,ISCLASSIFIEDBY,INDUSTRYSECTORCLASSIFIERID,None,True,True,None,True,True,None,high,True,Corporation points to IndustrySectorClassifier...
1,corporation_has_country,direct,Corporation,Country,fiben_corporations,fiben_countries,ISDOMICILEDIN,COUNTRYID,None,True,True,None,True,True,None,high,True,Corporation points to Country through ISDOMICI...
2,corporation_has_listed_security,bridge,Corporation,ListedSecurity,fiben_corporations,fiben_listed_securities,CORPORATIONID,LISTEDSECURITYID,fiben_securities,True,True,True,True,True,True,medium,True,Derived path Corporation -> Security -> Listed...
3,listed_security_represents_security,direct,ListedSecurity,Security,fiben_listed_securities,fiben_securities,LISTEDSECURITYID,ISTRADEDON,None,True,True,None,True,True,None,medium,True,Security appears to point to ListedSecurity th...
4,person_owns_financial_service_account,direct,Person,FinancialServiceAccount,fiben_persons,fiben_financial_service_accounts,PERSONID,ISOWNEDBY,None,True,True,None,True,True,None,high,True,FinancialServiceAccount points to Person throu...
5,account_has_holding,direct,FinancialServiceAccount,Holding,fiben_financial_service_accounts,fiben_holdings,FINANCIALSERVICEACCOUNTID,ISHELDBY,None,True,True,None,True,True,None,high,True,Holding points to FinancialServiceAccount thro...
6,holding_refers_to_listed_security,direct,Holding,ListedSecurity,fiben_holdings,fiben_listed_securities,REFERSTO,LISTEDSECURITYID,None,True,True,None,True,True,None,high,True,Holding points to ListedSecurity through REFER...
7,account_records_transaction,direct,FinancialServiceAccount,Transaction,fiben_financial_service_accounts,fiben_transactions,FINANCIALSERVICEACCOUNTID,ISFACILITATEDBY,None,True,True,None,True,True,None,high,True,SecuritiesTransaction points to FinancialServi...
8,transaction_refers_to_listed_security,direct,Transaction,ListedSecurity,fiben_transactions,fiben_listed_securities,REFERSTO,LISTEDSECURITYID,None,True,True,None,True,True,None,high,True,SecuritiesTransaction points to ListedSecurity...
9,buy_transaction_is_transaction,subtype_alias,BuyTransaction,Transaction,fiben_buy_transactions,fiben_transactions,SECURITIESTRANSACTIONID,SECURITIESTRANSACTIONID,None,True,True,None,True,True,None,medium,True,BuyTransaction is currently a conceptual alias...



All manual join hints are structurally valid.
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06c/fiben_relationship_join_hints.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06c/invalid_relationship_join_hints.csv
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06c/fiben_relationship_join_hints_metadata.json
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/logs/block_v6c___manual_fiben_relationship_join_hints_execution_log.json
Updated README section: BLOCK V6C — Manual FIBEN Relationship Join Hints
README path: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/README_FIBEN_Implementation_Framework.md


In [66]:
fiben_relationship_join_hints_df[
    ["relationship_name", "hint_type", "is_valid", "confidence", "notes"]
]

,relationship_name,hint_type,is_valid,confidence,notes
0,corporation_has_industry,direct,True,high,Corporation points to IndustrySectorClassifier...
1,corporation_has_country,direct,True,high,Corporation points to Country through ISDOMICI...
2,corporation_has_listed_security,bridge,True,medium,Derived path Corporation -> Security -> Listed...
3,listed_security_represents_security,direct,True,medium,Security appears to point to ListedSecurity th...
4,person_owns_financial_service_account,direct,True,high,FinancialServiceAccount points to Person throu...
5,account_has_holding,direct,True,high,Holding points to FinancialServiceAccount thro...
6,holding_refers_to_listed_security,direct,True,high,Holding points to ListedSecurity through REFER...
7,account_records_transaction,direct,True,high,SecuritiesTransaction points to FinancialServi...
8,transaction_refers_to_listed_security,direct,True,high,SecuritiesTransaction points to ListedSecurity...
9,buy_transaction_is_transaction,subtype_alias,True,medium,BuyTransaction is currently a conceptual alias...


In [67]:
# =========================================================
# BLOCK V6 — OBSERVED CARDINALITY WITH MANUAL FIBEN JOIN HINTS
# =========================================================
#
# Purpose:
# This block computes the observed cardinality profile for the
# FIBEN conceptual relationships using manual entity/view overrides
# and manual relationship join hints.
#
# This replaces the previous light-only automatic version.
#
# Inputs:
# - FIBEN_ENTITY_VIEW_OVERRIDES
# - FIBEN_MANUAL_RELATIONSHIP_JOIN_HINTS
# - conceptual_relationships_df
# - fiben_data_bundle
#
# Main outputs preserved for downstream compatibility:
# - observed_cardinality_df
# - relationship_cardinality_df
# - cardinality_observed_df
#
# Additional outputs:
# - observed_cardinality_summary_df
# - uncomputed_cardinality_df
# - low_confidence_cardinality_df
#
# Important:
# Some relationships are direct joins.
# Some relationships use a bridge table.
# Some relationships are conceptual subtype aliases.
# =========================================================

import pandas as pd
from typing import Optional


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "fiben_data_bundle",
    "fiben_scenario",
    "conceptual_relationships_df",
    "FIBEN_ENTITY_VIEW_OVERRIDES",
    "FIBEN_MANUAL_RELATIONSHIP_JOIN_HINTS",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first BLOCK V6B and BLOCK V6C."
        )


# ---------------------------------------------------------
# 2. Configuration
# ---------------------------------------------------------
#
# Start with a row limit for safety.
# Later, after validation, set this to None to profile the full dataset.
# ---------------------------------------------------------

FIBEN_CARDINALITY_ROW_LIMIT = 200_000


# ---------------------------------------------------------
# 3. SQL helpers
# ---------------------------------------------------------

def sql_identifier(name: str) -> str:
    """
    Quote a DuckDB identifier safely.
    """
    return '"' + str(name).replace('"', '""') + '"'


def limited_view_sql(view_name: str, row_limit: Optional[int]) -> str:
    """
    Return a DuckDB subquery with an optional row limit.
    """
    if row_limit is None:
        return f"(SELECT * FROM {view_name})"

    return f"(SELECT * FROM {view_name} LIMIT {int(row_limit)})"


def duckdb_view_exists(con, view_name: str) -> bool:
    """
    Return True if a DuckDB view can be queried.
    """
    try:
        con.execute(f"SELECT * FROM {view_name} LIMIT 1").df()
        return True
    except Exception:
        return False


def duckdb_column_exists(con, view_name: str, column_name: str) -> bool:
    """
    Return True if a column exists in a DuckDB view.
    """
    try:
        columns = con.execute(f"DESCRIBE {view_name}").df()["column_name"].tolist()
        return column_name in columns
    except Exception:
        return False


# ---------------------------------------------------------
# 4. Cardinality classification helper
# ---------------------------------------------------------

def classify_observed_cardinality(stats: dict) -> str:
    """
    Classify the observed cardinality based on join statistics.

    This is an empirical classification over the selected data scope.
    """
    joined_rows = stats.get("joined_rows", 0)
    joined_source = stats.get("joined_distinct_source_keys", 0)
    joined_target = stats.get("joined_distinct_target_keys", 0)

    if joined_rows is None or joined_rows == 0:
        return "no_observed_matches"

    if joined_source in [None, 0] or joined_target in [None, 0]:
        return "unknown"

    avg_targets_per_source = joined_rows / joined_source
    avg_sources_per_target = joined_rows / joined_target

    source_many = avg_targets_per_source > 1.05
    target_many = avg_sources_per_target > 1.05

    if not source_many and not target_many:
        return "observed_1_to_1"

    if source_many and not target_many:
        return "observed_1_to_N"

    if not source_many and target_many:
        return "observed_N_to_1"

    return "observed_N_to_M"


# ---------------------------------------------------------
# 5. Direct join statistics
# ---------------------------------------------------------

def compute_direct_join_statistics(
    con,
    source_view: str,
    target_view: str,
    source_column: str,
    target_column: str,
    row_limit: Optional[int],
) -> dict:
    """
    Compute observed cardinality statistics for a direct relationship.

    Direct relationship form:

        source_view.source_column = target_view.target_column
    """
    source_sql = limited_view_sql(source_view, row_limit)
    target_sql = limited_view_sql(target_view, row_limit)

    sql = f"""
    WITH
    s AS (
        SELECT {sql_identifier(source_column)} AS source_key
        FROM {source_sql}
        WHERE {sql_identifier(source_column)} IS NOT NULL
    ),
    t AS (
        SELECT {sql_identifier(target_column)} AS target_key
        FROM {target_sql}
        WHERE {sql_identifier(target_column)} IS NOT NULL
    ),
    joined AS (
        SELECT
            s.source_key,
            t.target_key
        FROM s
        INNER JOIN t
            ON s.source_key = t.target_key
    )
    SELECT
        (SELECT COUNT(*) FROM s) AS source_non_null_rows,
        (SELECT COUNT(*) FROM t) AS target_non_null_rows,
        (SELECT COUNT(DISTINCT source_key) FROM s) AS source_distinct_keys,
        (SELECT COUNT(DISTINCT target_key) FROM t) AS target_distinct_keys,
        COUNT(*) AS joined_rows,
        COUNT(DISTINCT source_key) AS joined_distinct_source_keys,
        COUNT(DISTINCT target_key) AS joined_distinct_target_keys
    FROM joined;
    """

    row = con.execute(sql).df().iloc[0].to_dict()

    return {
        k: (None if pd.isna(v) else v)
        for k, v in row.items()
    }


# ---------------------------------------------------------
# 6. Bridge join statistics
# ---------------------------------------------------------

def compute_bridge_join_statistics(
    con,
    source_view: str,
    bridge_view: str,
    target_view: str,
    source_column: str,
    source_to_bridge_bridge_column: str,
    bridge_to_target_bridge_column: str,
    target_column: str,
    row_limit: Optional[int],
) -> dict:
    """
    Compute observed cardinality statistics for a bridge relationship.

    Bridge relationship form:

        source_view.source_column = bridge_view.source_to_bridge_bridge_column
        bridge_view.bridge_to_target_bridge_column = target_view.target_column
    """
    source_sql = limited_view_sql(source_view, row_limit)
    bridge_sql = limited_view_sql(bridge_view, row_limit)
    target_sql = limited_view_sql(target_view, row_limit)

    sql = f"""
    WITH
    s AS (
        SELECT {sql_identifier(source_column)} AS source_key
        FROM {source_sql}
        WHERE {sql_identifier(source_column)} IS NOT NULL
    ),
    b AS (
        SELECT
            {sql_identifier(source_to_bridge_bridge_column)} AS bridge_source_key,
            {sql_identifier(bridge_to_target_bridge_column)} AS bridge_target_key
        FROM {bridge_sql}
        WHERE {sql_identifier(source_to_bridge_bridge_column)} IS NOT NULL
          AND {sql_identifier(bridge_to_target_bridge_column)} IS NOT NULL
    ),
    t AS (
        SELECT {sql_identifier(target_column)} AS target_key
        FROM {target_sql}
        WHERE {sql_identifier(target_column)} IS NOT NULL
    ),
    joined AS (
        SELECT
            s.source_key,
            t.target_key
        FROM s
        INNER JOIN b
            ON s.source_key = b.bridge_source_key
        INNER JOIN t
            ON b.bridge_target_key = t.target_key
    )
    SELECT
        (SELECT COUNT(*) FROM s) AS source_non_null_rows,
        (SELECT COUNT(*) FROM t) AS target_non_null_rows,
        (SELECT COUNT(*) FROM b) AS bridge_non_null_rows,
        (SELECT COUNT(DISTINCT source_key) FROM s) AS source_distinct_keys,
        (SELECT COUNT(DISTINCT target_key) FROM t) AS target_distinct_keys,
        COUNT(*) AS joined_rows,
        COUNT(DISTINCT source_key) AS joined_distinct_source_keys,
        COUNT(DISTINCT target_key) AS joined_distinct_target_keys
    FROM joined;
    """

    row = con.execute(sql).df().iloc[0].to_dict()

    return {
        k: (None if pd.isna(v) else v)
        for k, v in row.items()
    }


# ---------------------------------------------------------
# 7. Compute observed cardinality for all conceptual relationships
# ---------------------------------------------------------

observed_cardinality_rows = []

for _, rel in conceptual_relationships_df.iterrows():
    relationship_name = rel["name"]
    source_entity = rel["source"]
    target_entity = rel["target"]
    semantic_type = rel.get("semantic_type", "unspecified")
    edge_id = rel.get(
        "edge_id",
        f"{source_entity} -- {target_entity} [{relationship_name}]"
    )

    hint = FIBEN_MANUAL_RELATIONSHIP_JOIN_HINTS.get(relationship_name)

    if hint is None:
        observed_cardinality_rows.append({
            "edge_id": edge_id,
            "relationship_name": relationship_name,
            "semantic_type": semantic_type,
            "source_entity": source_entity,
            "target_entity": target_entity,
            "hint_type": None,
            "source_view": FIBEN_ENTITY_VIEW_OVERRIDES.get(source_entity),
            "target_view": FIBEN_ENTITY_VIEW_OVERRIDES.get(target_entity),
            "bridge_view": None,
            "selected_source_column": None,
            "selected_target_column": None,
            "bridge_source_column": None,
            "bridge_target_column": None,
            "hint_confidence": None,
            "observed_cardinality": "not_computed",
            "computation_status": "missing_manual_join_hint",
            "row_limit": FIBEN_CARDINALITY_ROW_LIMIT,
            "source_non_null_rows": None,
            "target_non_null_rows": None,
            "bridge_non_null_rows": None,
            "source_distinct_keys": None,
            "target_distinct_keys": None,
            "joined_rows": None,
            "joined_distinct_source_keys": None,
            "joined_distinct_target_keys": None,
            "notes": "No manual join hint was provided for this relationship.",
        })
        continue

    hint_type = hint["hint_type"]

    try:
        if hint_type in ["direct", "subtype_alias"]:
            source_view = hint["source_view"]
            target_view = hint["target_view"]
            source_column = hint["source_column"]
            target_column = hint["target_column"]

            missing = []

            if not duckdb_view_exists(fiben_data_bundle.con, source_view):
                missing.append(f"missing source view: {source_view}")

            if not duckdb_view_exists(fiben_data_bundle.con, target_view):
                missing.append(f"missing target view: {target_view}")

            if not missing:
                if not duckdb_column_exists(fiben_data_bundle.con, source_view, source_column):
                    missing.append(f"missing source column: {source_column}")

                if not duckdb_column_exists(fiben_data_bundle.con, target_view, target_column):
                    missing.append(f"missing target column: {target_column}")

            if missing:
                observed_cardinality_rows.append({
                    "edge_id": edge_id,
                    "relationship_name": relationship_name,
                    "semantic_type": semantic_type,
                    "source_entity": source_entity,
                    "target_entity": target_entity,
                    "hint_type": hint_type,
                    "source_view": source_view,
                    "target_view": target_view,
                    "bridge_view": None,
                    "selected_source_column": source_column,
                    "selected_target_column": target_column,
                    "bridge_source_column": None,
                    "bridge_target_column": None,
                    "hint_confidence": hint.get("confidence"),
                    "observed_cardinality": "not_computed",
                    "computation_status": "invalid_hint: " + "; ".join(missing),
                    "row_limit": FIBEN_CARDINALITY_ROW_LIMIT,
                    "source_non_null_rows": None,
                    "target_non_null_rows": None,
                    "bridge_non_null_rows": None,
                    "source_distinct_keys": None,
                    "target_distinct_keys": None,
                    "joined_rows": None,
                    "joined_distinct_source_keys": None,
                    "joined_distinct_target_keys": None,
                    "notes": hint.get("notes"),
                })
                continue

            stats = compute_direct_join_statistics(
                con=fiben_data_bundle.con,
                source_view=source_view,
                target_view=target_view,
                source_column=source_column,
                target_column=target_column,
                row_limit=FIBEN_CARDINALITY_ROW_LIMIT,
            )

            observed_cardinality = classify_observed_cardinality(stats)

            observed_cardinality_rows.append({
                "edge_id": edge_id,
                "relationship_name": relationship_name,
                "semantic_type": semantic_type,
                "source_entity": source_entity,
                "target_entity": target_entity,
                "hint_type": hint_type,
                "source_view": source_view,
                "target_view": target_view,
                "bridge_view": None,
                "selected_source_column": source_column,
                "selected_target_column": target_column,
                "bridge_source_column": None,
                "bridge_target_column": None,
                "hint_confidence": hint.get("confidence"),
                "observed_cardinality": observed_cardinality,
                "computation_status": "computed",
                "row_limit": FIBEN_CARDINALITY_ROW_LIMIT,
                "bridge_non_null_rows": None,
                **stats,
                "notes": hint.get("notes"),
            })

        elif hint_type == "bridge":
            source_view = hint["source_view"]
            bridge_view = hint["bridge_view"]
            target_view = hint["target_view"]

            source_column = hint["source_column"]
            bridge_source_column = hint["source_to_bridge_bridge_column"]
            bridge_target_column = hint["bridge_to_target_bridge_column"]
            target_column = hint["target_column"]

            missing = []

            for view_label, view_name in [
                ("source view", source_view),
                ("bridge view", bridge_view),
                ("target view", target_view),
            ]:
                if not duckdb_view_exists(fiben_data_bundle.con, view_name):
                    missing.append(f"missing {view_label}: {view_name}")

            if not missing:
                if not duckdb_column_exists(fiben_data_bundle.con, source_view, source_column):
                    missing.append(f"missing source column: {source_column}")

                if not duckdb_column_exists(fiben_data_bundle.con, bridge_view, bridge_source_column):
                    missing.append(f"missing bridge source column: {bridge_source_column}")

                if not duckdb_column_exists(fiben_data_bundle.con, bridge_view, bridge_target_column):
                    missing.append(f"missing bridge target column: {bridge_target_column}")

                if not duckdb_column_exists(fiben_data_bundle.con, target_view, target_column):
                    missing.append(f"missing target column: {target_column}")

            if missing:
                observed_cardinality_rows.append({
                    "edge_id": edge_id,
                    "relationship_name": relationship_name,
                    "semantic_type": semantic_type,
                    "source_entity": source_entity,
                    "target_entity": target_entity,
                    "hint_type": hint_type,
                    "source_view": source_view,
                    "target_view": target_view,
                    "bridge_view": bridge_view,
                    "selected_source_column": source_column,
                    "selected_target_column": target_column,
                    "bridge_source_column": bridge_source_column,
                    "bridge_target_column": bridge_target_column,
                    "hint_confidence": hint.get("confidence"),
                    "observed_cardinality": "not_computed",
                    "computation_status": "invalid_hint: " + "; ".join(missing),
                    "row_limit": FIBEN_CARDINALITY_ROW_LIMIT,
                    "source_non_null_rows": None,
                    "target_non_null_rows": None,
                    "bridge_non_null_rows": None,
                    "source_distinct_keys": None,
                    "target_distinct_keys": None,
                    "joined_rows": None,
                    "joined_distinct_source_keys": None,
                    "joined_distinct_target_keys": None,
                    "notes": hint.get("notes"),
                })
                continue

            stats = compute_bridge_join_statistics(
                con=fiben_data_bundle.con,
                source_view=source_view,
                bridge_view=bridge_view,
                target_view=target_view,
                source_column=source_column,
                source_to_bridge_bridge_column=bridge_source_column,
                bridge_to_target_bridge_column=bridge_target_column,
                target_column=target_column,
                row_limit=FIBEN_CARDINALITY_ROW_LIMIT,
            )

            observed_cardinality = classify_observed_cardinality(stats)

            observed_cardinality_rows.append({
                "edge_id": edge_id,
                "relationship_name": relationship_name,
                "semantic_type": semantic_type,
                "source_entity": source_entity,
                "target_entity": target_entity,
                "hint_type": hint_type,
                "source_view": source_view,
                "target_view": target_view,
                "bridge_view": bridge_view,
                "selected_source_column": source_column,
                "selected_target_column": target_column,
                "bridge_source_column": bridge_source_column,
                "bridge_target_column": bridge_target_column,
                "hint_confidence": hint.get("confidence"),
                "observed_cardinality": observed_cardinality,
                "computation_status": "computed",
                "row_limit": FIBEN_CARDINALITY_ROW_LIMIT,
                **stats,
                "notes": hint.get("notes"),
            })

        else:
            observed_cardinality_rows.append({
                "edge_id": edge_id,
                "relationship_name": relationship_name,
                "semantic_type": semantic_type,
                "source_entity": source_entity,
                "target_entity": target_entity,
                "hint_type": hint_type,
                "source_view": hint.get("source_view"),
                "target_view": hint.get("target_view"),
                "bridge_view": hint.get("bridge_view"),
                "selected_source_column": hint.get("source_column"),
                "selected_target_column": hint.get("target_column"),
                "bridge_source_column": hint.get("source_to_bridge_bridge_column"),
                "bridge_target_column": hint.get("bridge_to_target_bridge_column"),
                "hint_confidence": hint.get("confidence"),
                "observed_cardinality": "not_computed",
                "computation_status": f"unsupported_hint_type: {hint_type}",
                "row_limit": FIBEN_CARDINALITY_ROW_LIMIT,
                "source_non_null_rows": None,
                "target_non_null_rows": None,
                "bridge_non_null_rows": None,
                "source_distinct_keys": None,
                "target_distinct_keys": None,
                "joined_rows": None,
                "joined_distinct_source_keys": None,
                "joined_distinct_target_keys": None,
                "notes": hint.get("notes"),
            })

    except Exception as e:
        observed_cardinality_rows.append({
            "edge_id": edge_id,
            "relationship_name": relationship_name,
            "semantic_type": semantic_type,
            "source_entity": source_entity,
            "target_entity": target_entity,
            "hint_type": hint_type,
            "source_view": hint.get("source_view"),
            "target_view": hint.get("target_view"),
            "bridge_view": hint.get("bridge_view"),
            "selected_source_column": hint.get("source_column"),
            "selected_target_column": hint.get("target_column"),
            "bridge_source_column": hint.get("source_to_bridge_bridge_column"),
            "bridge_target_column": hint.get("bridge_to_target_bridge_column"),
            "hint_confidence": hint.get("confidence"),
            "observed_cardinality": "not_computed",
            "computation_status": f"computation_failed: {type(e).__name__}: {e}",
            "row_limit": FIBEN_CARDINALITY_ROW_LIMIT,
            "source_non_null_rows": None,
            "target_non_null_rows": None,
            "bridge_non_null_rows": None,
            "source_distinct_keys": None,
            "target_distinct_keys": None,
            "joined_rows": None,
            "joined_distinct_source_keys": None,
            "joined_distinct_target_keys": None,
            "notes": hint.get("notes"),
        })


# ---------------------------------------------------------
# 8. Main output aliases
# ---------------------------------------------------------

observed_cardinality_df = pd.DataFrame(observed_cardinality_rows)

relationship_cardinality_df = observed_cardinality_df.copy()
cardinality_observed_df = observed_cardinality_df.copy()


# ---------------------------------------------------------
# 9. Summary tables
# ---------------------------------------------------------

observed_cardinality_summary_df = (
    observed_cardinality_df
    .groupby(
        ["observed_cardinality", "computation_status", "hint_confidence"],
        dropna=False,
    )
    .agg(
        n_relationships=("relationship_name", "count"),
        relationships=("relationship_name", lambda s: sorted(set(s))),
    )
    .reset_index()
    .sort_values(["observed_cardinality", "computation_status", "hint_confidence"])
    .reset_index(drop=True)
)

uncomputed_cardinality_df = observed_cardinality_df[
    observed_cardinality_df["computation_status"] != "computed"
].copy()

low_confidence_cardinality_df = observed_cardinality_df[
    observed_cardinality_df["hint_confidence"].isin(["low", "medium"])
].copy()

no_match_cardinality_df = observed_cardinality_df[
    observed_cardinality_df["observed_cardinality"] == "no_observed_matches"
].copy()


# ---------------------------------------------------------
# 10. Display outputs
# ---------------------------------------------------------

print("Observed cardinality with manual FIBEN join hints completed.")

print("\nObserved cardinality by relationship:")
display(observed_cardinality_df)

print("\nObserved cardinality summary:")
display(observed_cardinality_summary_df)

if not uncomputed_cardinality_df.empty:
    print("\nRelationships not computed:")
    display(uncomputed_cardinality_df)
else:
    print("\nAll relationships were computed.")

if not no_match_cardinality_df.empty:
    print("\nRelationships computed but with no observed matches:")
    display(no_match_cardinality_df)

if not low_confidence_cardinality_df.empty:
    print("\nLow or medium confidence relationships to review:")
    display(low_confidence_cardinality_df)


# ---------------------------------------------------------
# 11. Save BLOCK V6 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    observed_cardinality_df,
    "variables/block_v06/observed_cardinality_by_relationship.csv",
)

save_dataframe(
    relationship_cardinality_df,
    "variables/block_v06/relationship_cardinality.csv",
)

save_dataframe(
    cardinality_observed_df,
    "variables/block_v06/cardinality_observed.csv",
)

save_dataframe(
    observed_cardinality_summary_df,
    "variables/block_v06/observed_cardinality_summary.csv",
)

save_dataframe(
    uncomputed_cardinality_df,
    "variables/block_v06/uncomputed_cardinality_relationships.csv",
)

save_dataframe(
    low_confidence_cardinality_df,
    "variables/block_v06/low_confidence_cardinality_relationships.csv",
)

save_dataframe(
    no_match_cardinality_df,
    "variables/block_v06/no_match_cardinality_relationships.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "row_limit": FIBEN_CARDINALITY_ROW_LIMIT,
        "n_relationships": len(observed_cardinality_df),
        "n_computed_relationships": int(
            (observed_cardinality_df["computation_status"] == "computed").sum()
        ),
        "n_uncomputed_relationships": len(uncomputed_cardinality_df),
        "n_low_or_medium_confidence_relationships": len(low_confidence_cardinality_df),
        "n_no_match_relationships": len(no_match_cardinality_df),
        "observed_cardinality_counts": (
            observed_cardinality_df["observed_cardinality"]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "computation_status_counts": (
            observed_cardinality_df["computation_status"]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "output_files": {
            "observed_cardinality_csv": "variables/block_v06/observed_cardinality_by_relationship.csv",
            "relationship_cardinality_csv": "variables/block_v06/relationship_cardinality.csv",
            "cardinality_observed_csv": "variables/block_v06/cardinality_observed.csv",
            "observed_cardinality_summary_csv": "variables/block_v06/observed_cardinality_summary.csv",
            "uncomputed_cardinality_csv": "variables/block_v06/uncomputed_cardinality_relationships.csv",
            "low_confidence_cardinality_csv": "variables/block_v06/low_confidence_cardinality_relationships.csv",
            "no_match_cardinality_csv": "variables/block_v06/no_match_cardinality_relationships.csv",
        },
    },
    "variables/block_v06/observed_cardinality_metadata.json",
)

write_execution_log(
    block_name="BLOCK V6 — OBSERVED CARDINALITY WITH MANUAL FIBEN JOIN HINTS",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "row_limit": FIBEN_CARDINALITY_ROW_LIMIT,
        "n_relationships": len(observed_cardinality_df),
        "n_computed_relationships": int(
            (observed_cardinality_df["computation_status"] == "computed").sum()
        ),
        "n_uncomputed_relationships": len(uncomputed_cardinality_df),
        "n_low_or_medium_confidence_relationships": len(low_confidence_cardinality_df),
        "n_no_match_relationships": len(no_match_cardinality_df),
        "observed_cardinality_csv": "variables/block_v06/observed_cardinality_by_relationship.csv",
        "relationship_cardinality_csv": "variables/block_v06/relationship_cardinality.csv",
        "cardinality_observed_csv": "variables/block_v06/cardinality_observed.csv",
        "observed_cardinality_summary_csv": "variables/block_v06/observed_cardinality_summary.csv",
        "uncomputed_cardinality_csv": "variables/block_v06/uncomputed_cardinality_relationships.csv",
        "low_confidence_cardinality_csv": "variables/block_v06/low_confidence_cardinality_relationships.csv",
        "no_match_cardinality_csv": "variables/block_v06/no_match_cardinality_relationships.csv",
        "metadata_json": "variables/block_v06/observed_cardinality_metadata.json",
    },
)


# ---------------------------------------------------------
# 12. Update README section for BLOCK V6
# ---------------------------------------------------------

block_v06_readme_body = f"""
This block computes the observed cardinality profile for the FIBEN conceptual
relationships using manual entity/view overrides and manual join hints.

Main variables created:

    observed_cardinality_df
    relationship_cardinality_df
    cardinality_observed_df

Additional variables created:

    observed_cardinality_summary_df
    uncomputed_cardinality_df
    low_confidence_cardinality_df
    no_match_cardinality_df

Number of conceptual relationships processed:

    {len(observed_cardinality_df)}

Number of computed relationships:

    {int((observed_cardinality_df["computation_status"] == "computed").sum())}

Number of uncomputed relationships:

    {len(uncomputed_cardinality_df)}

Number of low or medium confidence relationships:

    {len(low_confidence_cardinality_df)}

Number of computed relationships with no observed matches:

    {len(no_match_cardinality_df)}

Row limit used per table:

    {FIBEN_CARDINALITY_ROW_LIMIT}

Generated reproducibility files:

    variables/block_v06/observed_cardinality_by_relationship.csv
    variables/block_v06/relationship_cardinality.csv
    variables/block_v06/cardinality_observed.csv
    variables/block_v06/observed_cardinality_summary.csv
    variables/block_v06/uncomputed_cardinality_relationships.csv
    variables/block_v06/low_confidence_cardinality_relationships.csv
    variables/block_v06/no_match_cardinality_relationships.csv
    variables/block_v06/observed_cardinality_metadata.json

Methodological meaning:

    This block estimates the observed cardinality pattern of each conceptual
    relationship using the physical FIBEN data available in DuckDB.

Important notes:

    The relationship corporation_has_listed_security is computed as a bridge
    relationship through Security.

    BuyTransaction and SellTransaction are currently subtype aliases over the
    SecuritiesTransaction table.

    Low-confidence and no-match relationships should be reviewed before the
    final benchmark execution.
"""

update_readme_section(
    section_title="BLOCK V6 — Observed Cardinality with Manual FIBEN Join Hints",
    section_body=block_v06_readme_body,
)

Observed cardinality with manual FIBEN join hints completed.

Observed cardinality by relationship:


,edge_id,relationship_name,semantic_type,source_entity,target_entity,hint_type,source_view,target_view,bridge_view,selected_source_column,...,row_limit,bridge_non_null_rows,source_non_null_rows,target_non_null_rows,source_distinct_keys,target_distinct_keys,joined_rows,joined_distinct_source_keys,joined_distinct_target_keys,notes
0,Corporation -- Industry [corporation_has_indus...,corporation_has_industry,descriptor,Corporation,Industry,direct,fiben_corporations,fiben_industries,None,ISCLASSIFIEDBY,...,200000,NaN,2324,452,358,452,2324,358,358,Corporation points to IndustrySectorClassifier...
1,Corporation -- Country [corporation_has_country],corporation_has_country,descriptor,Corporation,Country,direct,fiben_corporations,fiben_countries,None,ISDOMICILEDIN,...,200000,NaN,2324,249,20,249,2324,20,20,Corporation points to Country through ISDOMICI...
2,Corporation -- ListedSecurity [corporation_has...,corporation_has_listed_security,association,Corporation,ListedSecurity,bridge,fiben_corporations,fiben_listed_securities,fiben_securities,CORPORATIONID,...,200000,2745.0,2324,2745,2324,2745,0,0,0,Derived path Corporation -> Security -> Listed...
3,ListedSecurity -- Security [listed_security_re...,listed_security_represents_security,association,ListedSecurity,Security,direct,fiben_listed_securities,fiben_securities,None,LISTEDSECURITYID,...,200000,NaN,2745,2745,2745,3,0,0,0,Security appears to point to ListedSecurity th...
4,Person -- FinancialServiceAccount [person_owns...,person_owns_financial_service_account,ownership,Person,FinancialServiceAccount,direct,fiben_persons,fiben_financial_service_accounts,None,PERSONID,...,200000,NaN,50100,97270,50100,50000,97270,50000,50000,FinancialServiceAccount points to Person throu...
5,FinancialServiceAccount -- Holding [account_ha...,account_has_holding,containment,FinancialServiceAccount,Holding,direct,fiben_financial_service_accounts,fiben_holdings,None,FINANCIALSERVICEACCOUNTID,...,200000,NaN,97270,200000,97270,36445,200000,36445,36445,Holding points to FinancialServiceAccount thro...
6,Holding -- ListedSecurity [holding_refers_to_l...,holding_refers_to_listed_security,association,Holding,ListedSecurity,direct,fiben_holdings,fiben_listed_securities,None,REFERSTO,...,200000,NaN,200000,2745,2745,2745,200000,2745,2745,Holding points to ListedSecurity through REFER...
7,FinancialServiceAccount -- Transaction [accoun...,account_records_transaction,containment,FinancialServiceAccount,Transaction,direct,fiben_financial_service_accounts,fiben_transactions,None,FINANCIALSERVICEACCOUNTID,...,200000,NaN,97270,200000,97270,84838,200000,84838,84838,SecuritiesTransaction points to FinancialServi...
8,Transaction -- ListedSecurity [transaction_ref...,transaction_refers_to_listed_security,association,Transaction,ListedSecurity,direct,fiben_transactions,fiben_listed_securities,None,REFERSTO,...,200000,NaN,200000,2745,2745,2745,200000,2745,2745,SecuritiesTransaction points to ListedSecurity...
9,BuyTransaction -- Transaction [buy_transaction...,buy_transaction_is_transaction,subtype,BuyTransaction,Transaction,subtype_alias,fiben_buy_transactions,fiben_transactions,None,SECURITIESTRANSACTIONID,...,200000,NaN,200000,200000,200000,200000,200000,200000,200000,BuyTransaction is currently a conceptual alias...



Observed cardinality summary:


,observed_cardinality,computation_status,hint_confidence,n_relationships,relationships
0,no_observed_matches,computed,low,2,"[financial_report_contains_disclosure, report_..."
1,no_observed_matches,computed,medium,2,"[corporation_has_listed_security, listed_secur..."
2,observed_1_to_1,computed,medium,2,"[buy_transaction_is_transaction, sell_transact..."
3,observed_N_to_M,computed,high,9,"[account_has_holding, account_records_transact..."



All relationships were computed.

Relationships computed but with no observed matches:


,edge_id,relationship_name,semantic_type,source_entity,target_entity,hint_type,source_view,target_view,bridge_view,selected_source_column,...,row_limit,bridge_non_null_rows,source_non_null_rows,target_non_null_rows,source_distinct_keys,target_distinct_keys,joined_rows,joined_distinct_source_keys,joined_distinct_target_keys,notes
2,Corporation -- ListedSecurity [corporation_has...,corporation_has_listed_security,association,Corporation,ListedSecurity,bridge,fiben_corporations,fiben_listed_securities,fiben_securities,CORPORATIONID,...,200000,2745.0,2324,2745,2324,2745,0,0,0,Derived path Corporation -> Security -> Listed...
3,ListedSecurity -- Security [listed_security_re...,listed_security_represents_security,association,ListedSecurity,Security,direct,fiben_listed_securities,fiben_securities,None,LISTEDSECURITYID,...,200000,NaN,2745,2745,2745,3,0,0,0,Security appears to point to ListedSecurity th...
13,ReportElement -- StatementElement [report_elem...,report_element_has_statement_element,containment,ReportElement,StatementElement,direct,fiben_report_elements,fiben_statement_elements,None,ELEMENTSOFFINANCIALREPORTID,...,200000,NaN,200000,200000,200000,200000,0,0,0,Tentative mapping. Needs validation with obser...
14,FinancialReport -- Disclosure [financial_repor...,financial_report_contains_disclosure,containment,FinancialReport,Disclosure,direct,fiben_reports,fiben_disclosures,None,HASUNIQUEIDENTIFIER,...,200000,NaN,48301,200000,48301,56,0,0,0,Tentative mapping because Disclosure has no ob...



Low or medium confidence relationships to review:


,edge_id,relationship_name,semantic_type,source_entity,target_entity,hint_type,source_view,target_view,bridge_view,selected_source_column,...,row_limit,bridge_non_null_rows,source_non_null_rows,target_non_null_rows,source_distinct_keys,target_distinct_keys,joined_rows,joined_distinct_source_keys,joined_distinct_target_keys,notes
2,Corporation -- ListedSecurity [corporation_has...,corporation_has_listed_security,association,Corporation,ListedSecurity,bridge,fiben_corporations,fiben_listed_securities,fiben_securities,CORPORATIONID,...,200000,2745.0,2324,2745,2324,2745,0,0,0,Derived path Corporation -> Security -> Listed...
3,ListedSecurity -- Security [listed_security_re...,listed_security_represents_security,association,ListedSecurity,Security,direct,fiben_listed_securities,fiben_securities,None,LISTEDSECURITYID,...,200000,NaN,2745,2745,2745,3,0,0,0,Security appears to point to ListedSecurity th...
9,BuyTransaction -- Transaction [buy_transaction...,buy_transaction_is_transaction,subtype,BuyTransaction,Transaction,subtype_alias,fiben_buy_transactions,fiben_transactions,None,SECURITIESTRANSACTIONID,...,200000,NaN,200000,200000,200000,200000,200000,200000,200000,BuyTransaction is currently a conceptual alias...
10,SellTransaction -- Transaction [sell_transacti...,sell_transaction_is_transaction,subtype,SellTransaction,Transaction,subtype_alias,fiben_sell_transactions,fiben_transactions,None,SECURITIESTRANSACTIONID,...,200000,NaN,200000,200000,200000,200000,200000,200000,200000,SellTransaction is currently a conceptual alia...
13,ReportElement -- StatementElement [report_elem...,report_element_has_statement_element,containment,ReportElement,StatementElement,direct,fiben_report_elements,fiben_statement_elements,None,ELEMENTSOFFINANCIALREPORTID,...,200000,NaN,200000,200000,200000,200000,0,0,0,Tentative mapping. Needs validation with obser...
14,FinancialReport -- Disclosure [financial_repor...,financial_report_contains_disclosure,containment,FinancialReport,Disclosure,direct,fiben_reports,fiben_disclosures,None,HASUNIQUEIDENTIFIER,...,200000,NaN,48301,200000,48301,56,0,0,0,Tentative mapping because Disclosure has no ob...


Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06/observed_cardinality_by_relationship.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06/relationship_cardinality.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06/cardinality_observed.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06/observed_cardinality_summary.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06/uncomputed_cardinality_relationships.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v06/low_confidence_cardinality_relationships.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework

In [68]:
observed_cardinality_df[
    [
        "relationship_name",
        "hint_type",
        "hint_confidence",
        "observed_cardinality",
        "computation_status",
        "joined_rows",
    ]
]

,relationship_name,hint_type,hint_confidence,observed_cardinality,computation_status,joined_rows
0,corporation_has_industry,direct,high,observed_N_to_M,computed,2324
1,corporation_has_country,direct,high,observed_N_to_M,computed,2324
2,corporation_has_listed_security,bridge,medium,no_observed_matches,computed,0
3,listed_security_represents_security,direct,medium,no_observed_matches,computed,0
4,person_owns_financial_service_account,direct,high,observed_N_to_M,computed,97270
5,account_has_holding,direct,high,observed_N_to_M,computed,200000
6,holding_refers_to_listed_security,direct,high,observed_N_to_M,computed,200000
7,account_records_transaction,direct,high,observed_N_to_M,computed,200000
8,transaction_refers_to_listed_security,direct,high,observed_N_to_M,computed,200000
9,buy_transaction_is_transaction,subtype_alias,medium,observed_1_to_1,computed,200000


In [69]:
uncomputed_cardinality_df

,edge_id,relationship_name,semantic_type,source_entity,target_entity,hint_type,source_view,target_view,bridge_view,selected_source_column,...,row_limit,bridge_non_null_rows,source_non_null_rows,target_non_null_rows,source_distinct_keys,target_distinct_keys,joined_rows,joined_distinct_source_keys,joined_distinct_target_keys,notes


In [70]:
# =========================================================
# BLOCK V7 — SEMANTIC CLASSIFICATION OF RELATIONSHIPS
# =========================================================
#
# Purpose:
# This block consolidates the semantic classification of the
# conceptual relationships in the FIBEN scenario.
#
# The semantic type was already defined in INSTANCE 1 for each
# conceptual relationship.
#
# This block organizes that information and integrates it with:
# - conceptual relationship metadata;
# - observed cardinality from BLOCK V6;
# - join hint confidence;
# - computation status.
#
# Main outputs preserved for downstream compatibility:
# - relationship_semantics_df
# - semantic_relationships_df
# - relationship_semantic_profile_df
#
# Additional outputs:
# - semantic_type_summary_df
# - semantic_cardinality_summary_df
# - edge_semantic_type_by_edge_id
# - edge_semantic_type_by_relationship_name
#
# Methodological role:
# The semantic profile helps later blocks identify which kinds of
# relationships are touched by each query:
# - descriptor;
# - association;
# - ownership;
# - containment;
# - subtype.
# =========================================================

import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "conceptual_relationships_df",
    "observed_cardinality_df",
    "fiben_scenario",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first INSTANCE 1 and BLOCKS V1 and V6."
        )


# ---------------------------------------------------------
# 2. Prepare the base relationship semantic table
# ---------------------------------------------------------

relationship_semantics_df = conceptual_relationships_df.copy()

if "semantic_type" not in relationship_semantics_df.columns:
    relationship_semantics_df["semantic_type"] = "unspecified"

if "notes" not in relationship_semantics_df.columns:
    relationship_semantics_df["notes"] = ""

if "edge_id" not in relationship_semantics_df.columns:
    relationship_semantics_df["edge_id"] = relationship_semantics_df.apply(
        lambda row: f'{row["source"]} -- {row["target"]} [{row["name"]}]',
        axis=1,
    )

relationship_semantics_df["semantic_type"] = (
    relationship_semantics_df["semantic_type"]
    .fillna("unspecified")
    .astype(str)
)

relationship_semantics_df["notes"] = (
    relationship_semantics_df["notes"]
    .fillna("")
    .astype(str)
)


# ---------------------------------------------------------
# 3. Add observed cardinality information from BLOCK V6
# ---------------------------------------------------------

cardinality_cols_to_keep = [
    "relationship_name",
    "observed_cardinality",
    "computation_status",
    "hint_type",
    "hint_confidence",
    "joined_rows",
    "source_non_null_rows",
    "target_non_null_rows",
    "source_distinct_keys",
    "target_distinct_keys",
    "joined_distinct_source_keys",
    "joined_distinct_target_keys",
]

available_cardinality_cols = [
    col for col in cardinality_cols_to_keep
    if col in observed_cardinality_df.columns
]

cardinality_merge_df = observed_cardinality_df[
    available_cardinality_cols
].copy()

relationship_semantics_df = relationship_semantics_df.merge(
    cardinality_merge_df,
    left_on="name",
    right_on="relationship_name",
    how="left",
)

if "relationship_name" in relationship_semantics_df.columns:
    relationship_semantics_df = relationship_semantics_df.drop(
        columns=["relationship_name"]
    )


# ---------------------------------------------------------
# 4. Add semantic flags
# ---------------------------------------------------------
#
# These boolean flags make later filtering easier.
# ---------------------------------------------------------

relationship_semantics_df["is_descriptor"] = (
    relationship_semantics_df["semantic_type"] == "descriptor"
)

relationship_semantics_df["is_association"] = (
    relationship_semantics_df["semantic_type"] == "association"
)

relationship_semantics_df["is_ownership"] = (
    relationship_semantics_df["semantic_type"] == "ownership"
)

relationship_semantics_df["is_containment"] = (
    relationship_semantics_df["semantic_type"] == "containment"
)

relationship_semantics_df["is_subtype"] = (
    relationship_semantics_df["semantic_type"] == "subtype"
)


# ---------------------------------------------------------
# 5. Create compatibility aliases
# ---------------------------------------------------------

semantic_relationships_df = relationship_semantics_df.copy()
relationship_semantic_profile_df = relationship_semantics_df.copy()


# ---------------------------------------------------------
# 6. Create lookup dictionaries
# ---------------------------------------------------------

edge_semantic_type_by_edge_id = dict(
    zip(
        relationship_semantics_df["edge_id"],
        relationship_semantics_df["semantic_type"],
    )
)

edge_semantic_type_by_relationship_name = dict(
    zip(
        relationship_semantics_df["name"],
        relationship_semantics_df["semantic_type"],
    )
)


# ---------------------------------------------------------
# 7. Build summary tables
# ---------------------------------------------------------

semantic_type_summary_df = (
    relationship_semantics_df
    .groupby("semantic_type", dropna=False)
    .agg(
        n_relationships=("name", "count"),
        relationships=("name", lambda s: sorted(set(s))),
        source_entities=("source", lambda s: sorted(set(s))),
        target_entities=("target", lambda s: sorted(set(s))),
    )
    .reset_index()
    .sort_values("semantic_type")
    .reset_index(drop=True)
)

semantic_cardinality_summary_df = (
    relationship_semantics_df
    .groupby(
        ["semantic_type", "observed_cardinality", "hint_confidence"],
        dropna=False,
    )
    .agg(
        n_relationships=("name", "count"),
        relationships=("name", lambda s: sorted(set(s))),
    )
    .reset_index()
    .sort_values(["semantic_type", "observed_cardinality", "hint_confidence"])
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 8. Validate semantic coverage
# ---------------------------------------------------------

missing_semantic_type_df = relationship_semantics_df[
    relationship_semantics_df["semantic_type"].isna()
    | (relationship_semantics_df["semantic_type"] == "")
    | (relationship_semantics_df["semantic_type"] == "unspecified")
].copy()

missing_cardinality_merge_df = relationship_semantics_df[
    relationship_semantics_df["observed_cardinality"].isna()
].copy()


# ---------------------------------------------------------
# 9. Display outputs
# ---------------------------------------------------------

print("Semantic classification of FIBEN relationships completed.")

print("\nRelationship semantic profile:")
display(relationship_semantics_df)

print("\nSemantic type summary:")
display(semantic_type_summary_df)

print("\nSemantic cardinality summary:")
display(semantic_cardinality_summary_df)

if not missing_semantic_type_df.empty:
    print("\nWarning: some relationships have missing or unspecified semantic type.")
    display(missing_semantic_type_df)
else:
    print("\nAll relationships have a semantic type.")

if not missing_cardinality_merge_df.empty:
    print("\nWarning: some relationships did not receive cardinality information.")
    display(missing_cardinality_merge_df)
else:
    print("\nAll relationships were integrated with observed cardinality information.")


# ---------------------------------------------------------
# 10. Save BLOCK V7 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    relationship_semantics_df,
    "variables/block_v07/relationship_semantics.csv",
)

save_dataframe(
    semantic_relationships_df,
    "variables/block_v07/semantic_relationships.csv",
)

save_dataframe(
    relationship_semantic_profile_df,
    "variables/block_v07/relationship_semantic_profile.csv",
)

save_dataframe(
    semantic_type_summary_df,
    "variables/block_v07/semantic_type_summary.csv",
)

save_dataframe(
    semantic_cardinality_summary_df,
    "variables/block_v07/semantic_cardinality_summary.csv",
)

save_dataframe(
    missing_semantic_type_df,
    "variables/block_v07/missing_semantic_type_relationships.csv",
)

save_dataframe(
    missing_cardinality_merge_df,
    "variables/block_v07/missing_cardinality_merge_relationships.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "n_relationships": len(relationship_semantics_df),
        "semantic_types": sorted(
            relationship_semantics_df["semantic_type"].dropna().unique().tolist()
        ),
        "semantic_type_counts": (
            relationship_semantics_df["semantic_type"]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "n_missing_semantic_type": len(missing_semantic_type_df),
        "n_missing_cardinality_merge": len(missing_cardinality_merge_df),
        "edge_semantic_type_by_edge_id": edge_semantic_type_by_edge_id,
        "edge_semantic_type_by_relationship_name": edge_semantic_type_by_relationship_name,
        "output_files": {
            "relationship_semantics_csv": "variables/block_v07/relationship_semantics.csv",
            "semantic_relationships_csv": "variables/block_v07/semantic_relationships.csv",
            "relationship_semantic_profile_csv": "variables/block_v07/relationship_semantic_profile.csv",
            "semantic_type_summary_csv": "variables/block_v07/semantic_type_summary.csv",
            "semantic_cardinality_summary_csv": "variables/block_v07/semantic_cardinality_summary.csv",
            "missing_semantic_type_csv": "variables/block_v07/missing_semantic_type_relationships.csv",
            "missing_cardinality_merge_csv": "variables/block_v07/missing_cardinality_merge_relationships.csv",
        },
    },
    "variables/block_v07/relationship_semantics_metadata.json",
)

write_execution_log(
    block_name="BLOCK V7 — SEMANTIC CLASSIFICATION OF RELATIONSHIPS",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_relationships": len(relationship_semantics_df),
        "semantic_types": sorted(
            relationship_semantics_df["semantic_type"].dropna().unique().tolist()
        ),
        "n_missing_semantic_type": len(missing_semantic_type_df),
        "n_missing_cardinality_merge": len(missing_cardinality_merge_df),
        "relationship_semantics_csv": "variables/block_v07/relationship_semantics.csv",
        "semantic_relationships_csv": "variables/block_v07/semantic_relationships.csv",
        "relationship_semantic_profile_csv": "variables/block_v07/relationship_semantic_profile.csv",
        "semantic_type_summary_csv": "variables/block_v07/semantic_type_summary.csv",
        "semantic_cardinality_summary_csv": "variables/block_v07/semantic_cardinality_summary.csv",
        "metadata_json": "variables/block_v07/relationship_semantics_metadata.json",
    },
)


# ---------------------------------------------------------
# 11. Update README section for BLOCK V7
# ---------------------------------------------------------

block_v07_readme_body = f"""
This block consolidates the semantic classification of the FIBEN conceptual
relationships.

Main variables created:

    relationship_semantics_df
    semantic_relationships_df
    relationship_semantic_profile_df

Additional variables created:

    semantic_type_summary_df
    semantic_cardinality_summary_df
    edge_semantic_type_by_edge_id
    edge_semantic_type_by_relationship_name
    missing_semantic_type_df
    missing_cardinality_merge_df

Number of relationships processed:

    {len(relationship_semantics_df)}

Semantic relationship types:

    {sorted(relationship_semantics_df["semantic_type"].dropna().unique().tolist())}

Number of relationships with missing semantic type:

    {len(missing_semantic_type_df)}

Number of relationships without cardinality merge:

    {len(missing_cardinality_merge_df)}

Generated reproducibility files:

    variables/block_v07/relationship_semantics.csv
    variables/block_v07/semantic_relationships.csv
    variables/block_v07/relationship_semantic_profile.csv
    variables/block_v07/semantic_type_summary.csv
    variables/block_v07/semantic_cardinality_summary.csv
    variables/block_v07/missing_semantic_type_relationships.csv
    variables/block_v07/missing_cardinality_merge_relationships.csv
    variables/block_v07/relationship_semantics_metadata.json

Methodological meaning:

    This block assigns each conceptual relationship to a semantic category.

    These semantic categories are later used to summarize which kinds of
    relationships are touched by each workload query.

Categories used in the FIBEN scenario:

    descriptor
    association
    ownership
    containment
    subtype
"""

update_readme_section(
    section_title="BLOCK V7 — Semantic Classification of Relationships",
    section_body=block_v07_readme_body,
)

Semantic classification of FIBEN relationships completed.

Relationship semantic profile:


,source,target,name,semantic_type,notes,edge_id,source_target_pair,observed_cardinality,computation_status,hint_type,...,target_non_null_rows,source_distinct_keys,target_distinct_keys,joined_distinct_source_keys,joined_distinct_target_keys,is_descriptor,is_association,is_ownership,is_containment,is_subtype
0,Corporation,Industry,corporation_has_industry,descriptor,Connects a corporation to its industry classif...,Corporation -- Industry [corporation_has_indus...,Corporation -> Industry,observed_N_to_M,computed,direct,...,452,358,452,358,358,True,False,False,False,False
1,Corporation,Country,corporation_has_country,descriptor,Connects a corporation to its country.,Corporation -- Country [corporation_has_country],Corporation -> Country,observed_N_to_M,computed,direct,...,249,20,249,20,20,True,False,False,False,False
2,Corporation,ListedSecurity,corporation_has_listed_security,association,Connects a corporation to listed securities.,Corporation -- ListedSecurity [corporation_has...,Corporation -> ListedSecurity,no_observed_matches,computed,bridge,...,2745,2324,2745,0,0,False,True,False,False,False
3,ListedSecurity,Security,listed_security_represents_security,association,Connects a listed security to the underlying s...,ListedSecurity -- Security [listed_security_re...,ListedSecurity -> Security,no_observed_matches,computed,direct,...,2745,2745,3,0,0,False,True,False,False,False
4,Person,FinancialServiceAccount,person_owns_financial_service_account,ownership,Connects a person to a financial service account.,Person -- FinancialServiceAccount [person_owns...,Person -> FinancialServiceAccount,observed_N_to_M,computed,direct,...,97270,50100,50000,50000,50000,False,False,True,False,False
5,FinancialServiceAccount,Holding,account_has_holding,containment,Connects an account to the holdings inside it.,FinancialServiceAccount -- Holding [account_ha...,FinancialServiceAccount -> Holding,observed_N_to_M,computed,direct,...,200000,97270,36445,36445,36445,False,False,False,True,False
6,Holding,ListedSecurity,holding_refers_to_listed_security,association,Connects a holding to the listed security bein...,Holding -- ListedSecurity [holding_refers_to_l...,Holding -> ListedSecurity,observed_N_to_M,computed,direct,...,2745,2745,2745,2745,2745,False,True,False,False,False
7,FinancialServiceAccount,Transaction,account_records_transaction,containment,Connects an account to its transactions.,FinancialServiceAccount -- Transaction [accoun...,FinancialServiceAccount -> Transaction,observed_N_to_M,computed,direct,...,200000,97270,84838,84838,84838,False,False,False,True,False
8,Transaction,ListedSecurity,transaction_refers_to_listed_security,association,Connects a transaction to the listed security ...,Transaction -- ListedSecurity [transaction_ref...,Transaction -> ListedSecurity,observed_N_to_M,computed,direct,...,2745,2745,2745,2745,2745,False,True,False,False,False
9,BuyTransaction,Transaction,buy_transaction_is_transaction,subtype,Models buy transactions as a subtype of transa...,BuyTransaction -- Transaction [buy_transaction...,BuyTransaction -> Transaction,observed_1_to_1,computed,subtype_alias,...,200000,200000,200000,200000,200000,False,False,False,False,True



Semantic type summary:


,semantic_type,n_relationships,relationships,source_entities,target_entities
0,association,4,"[corporation_has_listed_security, holding_refe...","[Corporation, Holding, ListedSecurity, Transac...","[ListedSecurity, Security]"
1,containment,6,"[account_has_holding, account_records_transact...","[Corporation, FinancialReport, FinancialServic...","[Disclosure, FinancialReport, Holding, ReportE..."
2,descriptor,2,"[corporation_has_country, corporation_has_indu...",[Corporation],"[Country, Industry]"
3,ownership,1,[person_owns_financial_service_account],[Person],[FinancialServiceAccount]
4,subtype,2,"[buy_transaction_is_transaction, sell_transact...","[BuyTransaction, SellTransaction]",[Transaction]



Semantic cardinality summary:


,semantic_type,observed_cardinality,hint_confidence,n_relationships,relationships
0,association,no_observed_matches,medium,2,"[corporation_has_listed_security, listed_secur..."
1,association,observed_N_to_M,high,2,"[holding_refers_to_listed_security, transactio..."
2,containment,no_observed_matches,low,2,"[financial_report_contains_disclosure, report_..."
3,containment,observed_N_to_M,high,4,"[account_has_holding, account_records_transact..."
4,descriptor,observed_N_to_M,high,2,"[corporation_has_country, corporation_has_indu..."
5,ownership,observed_N_to_M,high,1,[person_owns_financial_service_account]
6,subtype,observed_1_to_1,medium,2,"[buy_transaction_is_transaction, sell_transact..."



All relationships have a semantic type.

All relationships were integrated with observed cardinality information.
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v07/relationship_semantics.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v07/semantic_relationships.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v07/relationship_semantic_profile.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v07/semantic_type_summary.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v07/semantic_cardinality_summary.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v07/missing_semantic_type_relation

In [71]:
relationship_semantics_df[
    ["name", "source", "target", "semantic_type", "observed_cardinality", "hint_confidence"]
]

,name,source,target,semantic_type,observed_cardinality,hint_confidence
0,corporation_has_industry,Corporation,Industry,descriptor,observed_N_to_M,high
1,corporation_has_country,Corporation,Country,descriptor,observed_N_to_M,high
2,corporation_has_listed_security,Corporation,ListedSecurity,association,no_observed_matches,medium
3,listed_security_represents_security,ListedSecurity,Security,association,no_observed_matches,medium
4,person_owns_financial_service_account,Person,FinancialServiceAccount,ownership,observed_N_to_M,high
5,account_has_holding,FinancialServiceAccount,Holding,containment,observed_N_to_M,high
6,holding_refers_to_listed_security,Holding,ListedSecurity,association,observed_N_to_M,high
7,account_records_transaction,FinancialServiceAccount,Transaction,containment,observed_N_to_M,high
8,transaction_refers_to_listed_security,Transaction,ListedSecurity,association,observed_N_to_M,high
9,buy_transaction_is_transaction,BuyTransaction,Transaction,subtype,observed_1_to_1,medium


In [72]:
semantic_type_summary_df

,semantic_type,n_relationships,relationships,source_entities,target_entities
0,association,4,"[corporation_has_listed_security, holding_refe...","[Corporation, Holding, ListedSecurity, Transac...","[ListedSecurity, Security]"
1,containment,6,"[account_has_holding, account_records_transact...","[Corporation, FinancialReport, FinancialServic...","[Disclosure, FinancialReport, Holding, ReportE..."
2,descriptor,2,"[corporation_has_country, corporation_has_indu...",[Corporation],"[Country, Industry]"
3,ownership,1,[person_owns_financial_service_account],[Person],[FinancialServiceAccount]
4,subtype,2,"[buy_transaction_is_transaction, sell_transact...","[BuyTransaction, SellTransaction]",[Transaction]


In [73]:
missing_semantic_type_df 
missing_cardinality_merge_df 

,source,target,name,semantic_type,notes,edge_id,source_target_pair,observed_cardinality,computation_status,hint_type,...,target_non_null_rows,source_distinct_keys,target_distinct_keys,joined_distinct_source_keys,joined_distinct_target_keys,is_descriptor,is_association,is_ownership,is_containment,is_subtype


In [74]:
# =========================================================
# BLOCK V8 — TYPES OF RELATIONSHIPS TOUCHED BY EACH QUERY
# =========================================================
#
# Purpose:
# This block identifies which semantic relationship types are
# touched by each FIBEN workload query.
#
# It combines:
# - Rc selected edges from BLOCK V2;
# - semantic relationship classification from BLOCK V7;
# - workload metadata from BLOCK V0.
#
# Main outputs preserved for downstream compatibility:
# - relationship_types_by_query_df
# - query_relationship_types_df
# - query_semantic_profile_df
#
# Additional outputs:
# - query_relationship_edges_df
# - query_semantic_type_counts_df
# - query_semantic_flags_df
#
# Methodological role:
# This block describes the semantic profile of each query.
# Later blocks use this information to build the analytical matrix
# and to infer document configuration classes.
# =========================================================

import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "workload_conceptual_df",
    "rc_df",
    "rc_selected_edges_long_df",
    "relationship_semantics_df",
    "fiben_scenario",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first BLOCKS V0, V2, and V7."
        )


# ---------------------------------------------------------
# 2. Prepare relationship semantic lookup
# ---------------------------------------------------------

semantic_lookup_cols = [
    "edge_id",
    "name",
    "source",
    "target",
    "semantic_type",
    "observed_cardinality",
    "hint_confidence",
]

available_semantic_lookup_cols = [
    col for col in semantic_lookup_cols
    if col in relationship_semantics_df.columns
]

relationship_semantic_lookup_df = relationship_semantics_df[
    available_semantic_lookup_cols
].copy()


# ---------------------------------------------------------
# 3. Join Rc selected edges with relationship semantics
# ---------------------------------------------------------
#
# rc_selected_edges_long_df contains the edges selected to connect
# all entities touched by each query.
#
# We join it with relationship_semantics_df to discover the semantic
# type of each selected edge.
# ---------------------------------------------------------

if rc_selected_edges_long_df.empty:
    query_relationship_edges_df = pd.DataFrame(
        columns=[
            "query_name",
            "generic_class",
            "query_family",
            "edge_position",
            "edge_id",
            "name",
            "source",
            "target",
            "semantic_type",
            "observed_cardinality",
            "hint_confidence",
        ]
    )
else:
    query_relationship_edges_df = rc_selected_edges_long_df.merge(
        relationship_semantic_lookup_df,
        on="edge_id",
        how="left",
    )


# ---------------------------------------------------------
# 4. Identify queries with no selected edges
# ---------------------------------------------------------
#
# Single-entity queries, such as Q1, have Rc = 0.
# They do not touch conceptual relationships.
# ---------------------------------------------------------

queries_with_edges = set(query_relationship_edges_df["query_name"].tolist())

all_query_names = set(workload_conceptual_df["query_name"].tolist())

queries_without_edges = sorted(all_query_names - queries_with_edges)

queries_without_edges_df = workload_conceptual_df[
    workload_conceptual_df["query_name"].isin(queries_without_edges)
].copy()

queries_without_edges_df["relationship_status"] = "no_relationship_edges"
queries_without_edges_df["semantic_types_touched"] = [[] for _ in range(len(queries_without_edges_df))]
queries_without_edges_df["n_relationship_edges"] = 0
queries_without_edges_df["n_semantic_types"] = 0


# ---------------------------------------------------------
# 5. Summarize semantic relationship types by query
# ---------------------------------------------------------

if query_relationship_edges_df.empty:
    query_semantic_type_counts_df = pd.DataFrame(
        columns=[
            "query_name",
            "generic_class",
            "query_family",
            "semantic_type",
            "n_edges",
            "edge_ids",
            "relationships",
        ]
    )
else:
    query_semantic_type_counts_df = (
        query_relationship_edges_df
        .groupby(
            ["query_name", "generic_class", "query_family", "semantic_type"],
            dropna=False,
        )
        .agg(
            n_edges=("edge_id", "count"),
            edge_ids=("edge_id", lambda s: list(s)),
            relationships=("name", lambda s: sorted(set(s.dropna()))),
        )
        .reset_index()
        .sort_values(["query_name", "semantic_type"])
        .reset_index(drop=True)
    )


# ---------------------------------------------------------
# 6. Create compact semantic profile per query
# ---------------------------------------------------------

semantic_profile_rows = []

for _, workload_row in workload_conceptual_df.iterrows():
    query_name = workload_row["query_name"]

    q_edges_df = query_relationship_edges_df[
        query_relationship_edges_df["query_name"] == query_name
    ].copy()

    if q_edges_df.empty:
        semantic_types = []
        relationship_names = []
        edge_ids = []
        n_edges = 0
    else:
        semantic_types = sorted(
            q_edges_df["semantic_type"].dropna().unique().tolist()
        )
        relationship_names = sorted(
            q_edges_df["name"].dropna().unique().tolist()
        )
        edge_ids = q_edges_df["edge_id"].dropna().tolist()
        n_edges = len(q_edges_df)

    semantic_profile_rows.append({
        "query_name": query_name,
        "generic_class": workload_row.get("generic_class", None),
        "query_family": workload_row.get("query_family", None),
        "query_type": workload_row.get("query_type", None),
        "entities_touched": workload_row.get("entities_touched", []),
        "n_entities_touched": workload_row.get("n_entities_touched", None),
        "relationship_status": (
            "has_relationship_edges" if n_edges > 0 else "no_relationship_edges"
        ),
        "n_relationship_edges": n_edges,
        "semantic_types_touched": semantic_types,
        "n_semantic_types": len(semantic_types),
        "relationship_names_touched": relationship_names,
        "edge_ids_touched": edge_ids,
    })

query_semantic_profile_df = pd.DataFrame(semantic_profile_rows)


# ---------------------------------------------------------
# 7. Add semantic flags and counts
# ---------------------------------------------------------

known_semantic_types = sorted(
    relationship_semantics_df["semantic_type"].dropna().unique().tolist()
)

for semantic_type in known_semantic_types:
    flag_col = f"touches_{semantic_type}"
    count_col = f"n_{semantic_type}_edges"

    query_semantic_profile_df[flag_col] = (
        query_semantic_profile_df["semantic_types_touched"]
        .apply(lambda values: semantic_type in values)
    )

    semantic_count_lookup = (
        query_semantic_type_counts_df[
            query_semantic_type_counts_df["semantic_type"] == semantic_type
        ]
        .set_index("query_name")["n_edges"]
        .to_dict()
        if not query_semantic_type_counts_df.empty
        else {}
    )

    query_semantic_profile_df[count_col] = (
        query_semantic_profile_df["query_name"]
        .map(semantic_count_lookup)
        .fillna(0)
        .astype(int)
    )


query_semantic_flags_df = query_semantic_profile_df[
    ["query_name", "generic_class", "query_family"]
    + [f"touches_{st}" for st in known_semantic_types]
    + [f"n_{st}_edges" for st in known_semantic_types]
].copy()


# ---------------------------------------------------------
# 8. Compatibility aliases
# ---------------------------------------------------------

relationship_types_by_query_df = query_semantic_profile_df.copy()
query_relationship_types_df = query_semantic_profile_df.copy()


# ---------------------------------------------------------
# 9. Validation tables
# ---------------------------------------------------------

missing_semantic_for_edges_df = query_relationship_edges_df[
    query_relationship_edges_df["semantic_type"].isna()
].copy()

query_without_relationship_edges_df = query_semantic_profile_df[
    query_semantic_profile_df["n_relationship_edges"] == 0
].copy()


# ---------------------------------------------------------
# 10. Display outputs
# ---------------------------------------------------------

print("Relationship types touched by each query were identified successfully.")

print("\nQuery relationship edges:")
display(query_relationship_edges_df)

print("\nSemantic type counts by query:")
display(query_semantic_type_counts_df)

print("\nQuery semantic profile:")
display(query_semantic_profile_df)

print("\nQuery semantic flags:")
display(query_semantic_flags_df)

if not missing_semantic_for_edges_df.empty:
    print("\nWarning: some selected edges do not have semantic classification.")
    display(missing_semantic_for_edges_df)
else:
    print("\nAll selected query edges have semantic classification.")

if not query_without_relationship_edges_df.empty:
    print("\nQueries without relationship edges, usually Rc = 0:")
    display(
        query_without_relationship_edges_df[
            ["query_name", "generic_class", "query_family", "n_relationship_edges"]
        ]
    )


# ---------------------------------------------------------
# 11. Save BLOCK V8 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    query_relationship_edges_df,
    "variables/block_v08/query_relationship_edges.csv",
)

save_dataframe(
    query_semantic_type_counts_df,
    "variables/block_v08/query_semantic_type_counts.csv",
)

save_dataframe(
    query_semantic_profile_df,
    "variables/block_v08/query_semantic_profile.csv",
)

save_dataframe(
    query_semantic_flags_df,
    "variables/block_v08/query_semantic_flags.csv",
)

save_dataframe(
    relationship_types_by_query_df,
    "variables/block_v08/relationship_types_by_query.csv",
)

save_dataframe(
    query_relationship_types_df,
    "variables/block_v08/query_relationship_types.csv",
)

save_dataframe(
    missing_semantic_for_edges_df,
    "variables/block_v08/missing_semantic_for_edges.csv",
)

save_dataframe(
    query_without_relationship_edges_df,
    "variables/block_v08/query_without_relationship_edges.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "n_queries": len(query_semantic_profile_df),
        "known_semantic_types": known_semantic_types,
        "n_query_relationship_edges": len(query_relationship_edges_df),
        "n_queries_without_relationship_edges": len(query_without_relationship_edges_df),
        "n_edges_missing_semantic_classification": len(missing_semantic_for_edges_df),
        "output_files": {
            "query_relationship_edges_csv": "variables/block_v08/query_relationship_edges.csv",
            "query_semantic_type_counts_csv": "variables/block_v08/query_semantic_type_counts.csv",
            "query_semantic_profile_csv": "variables/block_v08/query_semantic_profile.csv",
            "query_semantic_flags_csv": "variables/block_v08/query_semantic_flags.csv",
            "relationship_types_by_query_csv": "variables/block_v08/relationship_types_by_query.csv",
            "query_relationship_types_csv": "variables/block_v08/query_relationship_types.csv",
            "missing_semantic_for_edges_csv": "variables/block_v08/missing_semantic_for_edges.csv",
            "query_without_relationship_edges_csv": "variables/block_v08/query_without_relationship_edges.csv",
        },
    },
    "variables/block_v08/query_relationship_types_metadata.json",
)

write_execution_log(
    block_name="BLOCK V8 — TYPES OF RELATIONSHIPS TOUCHED BY EACH QUERY",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_queries": len(query_semantic_profile_df),
        "known_semantic_types": known_semantic_types,
        "n_query_relationship_edges": len(query_relationship_edges_df),
        "n_queries_without_relationship_edges": len(query_without_relationship_edges_df),
        "n_edges_missing_semantic_classification": len(missing_semantic_for_edges_df),
        "query_relationship_edges_csv": "variables/block_v08/query_relationship_edges.csv",
        "query_semantic_type_counts_csv": "variables/block_v08/query_semantic_type_counts.csv",
        "query_semantic_profile_csv": "variables/block_v08/query_semantic_profile.csv",
        "query_semantic_flags_csv": "variables/block_v08/query_semantic_flags.csv",
        "metadata_json": "variables/block_v08/query_relationship_types_metadata.json",
    },
)


# ---------------------------------------------------------
# 12. Update README section for BLOCK V8
# ---------------------------------------------------------

block_v08_readme_body = f"""
This block identifies which semantic relationship types are touched by each
FIBEN workload query.

Main variables created:

    relationship_types_by_query_df
    query_relationship_types_df
    query_semantic_profile_df

Additional variables created:

    query_relationship_edges_df
    query_semantic_type_counts_df
    query_semantic_flags_df
    missing_semantic_for_edges_df
    query_without_relationship_edges_df

Number of queries processed:

    {len(query_semantic_profile_df)}

Known semantic relationship types:

    {known_semantic_types}

Number of query relationship edges:

    {len(query_relationship_edges_df)}

Number of queries without relationship edges:

    {len(query_without_relationship_edges_df)}

Number of selected edges missing semantic classification:

    {len(missing_semantic_for_edges_df)}

Generated reproducibility files:

    variables/block_v08/query_relationship_edges.csv
    variables/block_v08/query_semantic_type_counts.csv
    variables/block_v08/query_semantic_profile.csv
    variables/block_v08/query_semantic_flags.csv
    variables/block_v08/relationship_types_by_query.csv
    variables/block_v08/query_relationship_types.csv
    variables/block_v08/missing_semantic_for_edges.csv
    variables/block_v08/query_without_relationship_edges.csv
    variables/block_v08/query_relationship_types_metadata.json

Methodological meaning:

    This block builds the semantic profile of each query by combining the
    relationships selected in Rc(Qi) with the semantic classification of each
    conceptual edge.

    The resulting profile indicates whether a query touches descriptor,
    association, ownership, containment, or subtype relationships.

Important note:

    Queries with Rc = 0, such as local lookup queries over a single entity,
    do not touch relationship edges.
"""

update_readme_section(
    section_title="BLOCK V8 — Types of Relationships Touched by Each Query",
    section_body=block_v08_readme_body,
)

Relationship types touched by each query were identified successfully.

Query relationship edges:


,query_name,generic_class,query_family,edge_position,edge_id,name,source,target,semantic_type,observed_cardinality,hint_confidence
0,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,1,Corporation -- Industry [corporation_has_indus...,corporation_has_industry,Corporation,Industry,descriptor,observed_N_to_M,high
1,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,2,Corporation -- Country [corporation_has_country],corporation_has_country,Corporation,Country,descriptor,observed_N_to_M,high
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,3,Corporation -- ListedSecurity [corporation_has...,corporation_has_listed_security,Corporation,ListedSecurity,association,no_observed_matches,medium
3,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,1,ListedSecurity -- Security [listed_security_re...,listed_security_represents_security,ListedSecurity,Security,association,no_observed_matches,medium
4,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,2,FinancialServiceAccount -- Holding [account_ha...,account_has_holding,FinancialServiceAccount,Holding,containment,observed_N_to_M,high
5,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,3,Holding -- ListedSecurity [holding_refers_to_l...,holding_refers_to_listed_security,Holding,ListedSecurity,association,observed_N_to_M,high
6,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,1,Corporation -- ListedSecurity [corporation_has...,corporation_has_listed_security,Corporation,ListedSecurity,association,no_observed_matches,medium
7,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,2,Person -- FinancialServiceAccount [person_owns...,person_owns_financial_service_account,Person,FinancialServiceAccount,ownership,observed_N_to_M,high
8,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,3,FinancialServiceAccount -- Holding [account_ha...,account_has_holding,FinancialServiceAccount,Holding,containment,observed_N_to_M,high
9,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,4,Holding -- ListedSecurity [holding_refers_to_l...,holding_refers_to_listed_security,Holding,ListedSecurity,association,observed_N_to_M,high



Semantic type counts by query:


,query_name,generic_class,query_family,semantic_type,n_edges,edge_ids,relationships
0,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,association,1,[Holding -- ListedSecurity [holding_refers_to_...,[holding_refers_to_listed_security]
1,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,containment,2,[FinancialServiceAccount -- Holding [account_h...,"[account_has_holding, account_records_transact..."
2,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,ownership,1,[Person -- FinancialServiceAccount [person_own...,[person_owns_financial_service_account]
3,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,subtype,1,[BuyTransaction -- Transaction [buy_transactio...,[buy_transaction_is_transaction]
4,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,association,1,[Corporation -- ListedSecurity [corporation_ha...,[corporation_has_listed_security]
5,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,descriptor,2,[Corporation -- Industry [corporation_has_indu...,"[corporation_has_country, corporation_has_indu..."
6,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,association,2,[ListedSecurity -- Security [listed_security_r...,"[holding_refers_to_listed_security, listed_sec..."
7,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,containment,1,[FinancialServiceAccount -- Holding [account_h...,[account_has_holding]
8,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,association,2,[Corporation -- ListedSecurity [corporation_ha...,"[corporation_has_listed_security, holding_refe..."
9,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,containment,1,[FinancialServiceAccount -- Holding [account_h...,[account_has_holding]



Query semantic profile:


...


Query semantic flags:


,query_name,generic_class,query_family,touches_association,touches_containment,touches_descriptor,touches_ownership,touches_subtype,n_association_edges,n_containment_edges,n_descriptor_edges,n_ownership_edges,n_subtype_edges
0,Q1_CompanyProfileIBM,QG1,Local lookup,False,False,False,False,False,0,0,0,0,0
1,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,True,False,True,False,False,1,0,2,0,0
2,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,True,True,False,False,False,2,1,0,0,0
3,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,True,True,False,True,False,2,1,0,1,0
4,Q5_ReportsAndMetricDataOfCompany,QG5,Containment retrieval,False,True,False,False,False,0,4,0,0,0
5,Q6_TechUSListedSecuritiesWithHighLastTradedValue,QG6,Filtered search / recommendation,True,False,True,False,False,1,0,2,0,0
6,Q7_PersonsWhoBoughtMoreIBMThanSold,QG7,Aggregation / ranking,True,True,False,True,True,2,1,0,1,2
7,Q8_IBMTransactionsBelowAverageSellingPrice,QG8,Aggregation / ranking,True,False,False,False,True,2,0,0,0,1
8,Q9_PersonsWhoBoughtAndSoldSameStock,QG9,Associative retrieval + comparison,True,True,False,True,True,1,1,0,1,2
9,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,True,True,False,True,True,1,2,0,1,1



All selected query edges have semantic classification.

Queries without relationship edges, usually Rc = 0:


,query_name,generic_class,query_family,n_relationship_edges
0,Q1_CompanyProfileIBM,QG1,Local lookup,0


Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v08/query_relationship_edges.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v08/query_semantic_type_counts.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v08/query_semantic_profile.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v08/query_semantic_flags.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v08/relationship_types_by_query.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v08/query_relationship_types.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v08/m

In [75]:
# =========================================================
# BLOCK V9 — STRUCTURAL ANALYSIS OF EDGES BY DEPTH
# =========================================================
#
# Purpose:
# This block analyzes the structural position of conceptual
# relationships by depth from the selected root of each query.
#
# It uses:
# - selected roots from BLOCK V4;
# - distances and paths from BLOCK V5;
# - semantic relationship information from BLOCK V7;
# - query semantic profile from BLOCK V8.
#
# Main outputs preserved for downstream compatibility:
# - edge_depth_df
# - structural_edges_by_depth_df
# - query_edge_depth_profile_df
#
# Additional outputs:
# - query_depth_summary_df
# - query_edges_with_semantics_df
# - max_depth_by_query_df
#
# Methodological role:
# This block prepares the structural information needed to compute
# which entities and relationships are reachable from a selected
# root at a given document depth.
#
# Later blocks use this information to compute:
# - Re(Qi, er, D);
# - DeltaR;
# - DeltaRratio.
# =========================================================

import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "distance_df",
    "selected_root_df",
    "selected_root_by_query",
    "relationship_semantics_df",
    "query_semantic_profile_df",
    "fiben_scenario",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first BLOCKS V4, V5, V7, and V8."
        )


# ---------------------------------------------------------
# 2. Prepare edge semantic lookup
# ---------------------------------------------------------

edge_semantic_lookup_cols = [
    "edge_id",
    "name",
    "source",
    "target",
    "semantic_type",
    "observed_cardinality",
    "hint_type",
    "hint_confidence",
]

available_edge_semantic_lookup_cols = [
    col for col in edge_semantic_lookup_cols
    if col in relationship_semantics_df.columns
]

edge_semantic_lookup_df = relationship_semantics_df[
    available_edge_semantic_lookup_cols
].copy()


# ---------------------------------------------------------
# 3. Expand path edges by query/entity
# ---------------------------------------------------------
#
# distance_df contains one row per query/entity pair.
# Each row has:
# - path_entities;
# - path_edge_ids;
# - path_relationships.
#
# This section expands each path so that every edge in the path
# becomes one row.
# ---------------------------------------------------------

edge_depth_rows = []

for _, row in distance_df.iterrows():
    query_name = row["query_name"]
    generic_class = row.get("generic_class", None)
    query_family = row.get("query_family", None)
    query_type = row.get("query_type", None)
    selected_root = row["selected_root"]
    target_entity = row["entity"]
    distance_value = row["D"]
    distance_status = row["distance_status"]

    path_entities = row.get("path_entities", [])
    path_edge_ids = row.get("path_edge_ids", [])
    path_relationships = row.get("path_relationships", [])

    if not isinstance(path_entities, list):
        path_entities = []

    if not isinstance(path_edge_ids, list):
        path_edge_ids = []

    if not isinstance(path_relationships, list):
        path_relationships = []

    # Root entity has D = 0 and no edge path.
    if distance_value == 0 or len(path_edge_ids) == 0:
        continue

    for edge_position, edge_id in enumerate(path_edge_ids, start=1):
        from_entity = (
            path_entities[edge_position - 1]
            if edge_position - 1 < len(path_entities)
            else None
        )
        to_entity = (
            path_entities[edge_position]
            if edge_position < len(path_entities)
            else None
        )
        relationship_name = (
            path_relationships[edge_position - 1]
            if edge_position - 1 < len(path_relationships)
            else None
        )

        edge_depth_rows.append({
            "query_name": query_name,
            "generic_class": generic_class,
            "query_family": query_family,
            "query_type": query_type,
            "selected_root": selected_root,
            "target_entity": target_entity,
            "target_distance_D": distance_value,
            "distance_status": distance_status,
            "edge_position_from_root": edge_position,
            "edge_depth": edge_position,
            "from_entity_on_path": from_entity,
            "to_entity_on_path": to_entity,
            "edge_id": edge_id,
            "relationship_name_on_path": relationship_name,
        })


edge_depth_df = pd.DataFrame(edge_depth_rows)


# ---------------------------------------------------------
# 4. Add semantic information to each edge
# ---------------------------------------------------------

if edge_depth_df.empty:
    query_edges_with_semantics_df = pd.DataFrame(
        columns=[
            "query_name",
            "generic_class",
            "query_family",
            "query_type",
            "selected_root",
            "target_entity",
            "target_distance_D",
            "edge_depth",
            "edge_id",
            "semantic_type",
            "observed_cardinality",
            "hint_confidence",
        ]
    )
else:
    query_edges_with_semantics_df = edge_depth_df.merge(
        edge_semantic_lookup_df,
        on="edge_id",
        how="left",
        suffixes=("", "_semantic"),
    )


# ---------------------------------------------------------
# 5. Build unique edge-depth profile per query
# ---------------------------------------------------------
#
# The same edge may appear in multiple paths for the same query.
# For structural analysis, we keep the minimum depth where the edge
# appears from the selected root.
# ---------------------------------------------------------

if query_edges_with_semantics_df.empty:
    structural_edges_by_depth_df = pd.DataFrame(
        columns=[
            "query_name",
            "generic_class",
            "query_family",
            "query_type",
            "selected_root",
            "edge_id",
            "min_edge_depth",
            "max_edge_depth",
            "n_path_occurrences",
            "relationship_name",
            "semantic_type",
            "observed_cardinality",
            "hint_confidence",
            "entities_reached_by_edge",
        ]
    )
else:
    # Ensure a clean relationship name exists.
    if "name" in query_edges_with_semantics_df.columns:
        query_edges_with_semantics_df["relationship_name"] = (
            query_edges_with_semantics_df["name"]
        )
    else:
        query_edges_with_semantics_df["relationship_name"] = (
            query_edges_with_semantics_df["relationship_name_on_path"]
        )

    structural_edges_by_depth_df = (
        query_edges_with_semantics_df
        .groupby(
            [
                "query_name",
                "generic_class",
                "query_family",
                "query_type",
                "selected_root",
                "edge_id",
                "relationship_name",
                "semantic_type",
                "observed_cardinality",
                "hint_confidence",
            ],
            dropna=False,
        )
        .agg(
            min_edge_depth=("edge_depth", "min"),
            max_edge_depth=("edge_depth", "max"),
            n_path_occurrences=("edge_id", "count"),
            entities_reached_by_edge=("to_entity_on_path", lambda s: sorted(set(s.dropna()))),
            target_entities_using_edge=("target_entity", lambda s: sorted(set(s.dropna()))),
        )
        .reset_index()
        .sort_values(["query_name", "min_edge_depth", "edge_id"])
        .reset_index(drop=True)
    )


# ---------------------------------------------------------
# 6. Build depth summary by query
# ---------------------------------------------------------

if structural_edges_by_depth_df.empty:
    query_depth_summary_df = pd.DataFrame(
        columns=[
            "query_name",
            "generic_class",
            "query_family",
            "query_type",
            "selected_root",
            "max_edge_depth",
            "n_unique_edges",
            "semantic_types_by_depth",
        ]
    )
else:
    query_depth_summary_df = (
        structural_edges_by_depth_df
        .groupby(
            ["query_name", "generic_class", "query_family", "query_type", "selected_root"],
            dropna=False,
        )
        .agg(
            max_edge_depth=("min_edge_depth", "max"),
            n_unique_edges=("edge_id", "nunique"),
            semantic_types=("semantic_type", lambda s: sorted(set(s.dropna()))),
            relationship_names=("relationship_name", lambda s: sorted(set(s.dropna()))),
        )
        .reset_index()
        .sort_values(["query_name"])
        .reset_index(drop=True)
    )


# ---------------------------------------------------------
# 7. Build entity depth summary by query
# ---------------------------------------------------------

entity_depth_summary_df = (
    distance_df
    .groupby(
        ["query_name", "generic_class", "query_family", "query_type", "selected_root", "D"],
        dropna=False,
    )
    .agg(
        n_entities_at_depth=("entity", "count"),
        entities_at_depth=("entity", lambda s: sorted(set(s.dropna()))),
    )
    .reset_index()
    .sort_values(["query_name", "D"])
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 8. Build compact query edge-depth profile
# ---------------------------------------------------------

query_edge_depth_profile_rows = []

for _, row in selected_root_df.iterrows():
    query_name = row["query_name"]
    q_edges = structural_edges_by_depth_df[
        structural_edges_by_depth_df["query_name"] == query_name
    ].copy()

    q_entities = distance_df[
        distance_df["query_name"] == query_name
    ].copy()

    if q_edges.empty:
        max_edge_depth = 0
        n_unique_edges = 0
        edge_depths = []
        semantic_types = []
        relationship_names = []
    else:
        max_edge_depth = int(q_edges["min_edge_depth"].max())
        n_unique_edges = int(q_edges["edge_id"].nunique())
        edge_depths = sorted(q_edges["min_edge_depth"].dropna().unique().tolist())
        semantic_types = sorted(q_edges["semantic_type"].dropna().unique().tolist())
        relationship_names = sorted(q_edges["relationship_name"].dropna().unique().tolist())

    if q_entities.empty or q_entities["D"].dropna().empty:
        max_entity_depth = None
    else:
        max_entity_depth = int(q_entities["D"].dropna().max())

    query_edge_depth_profile_rows.append({
        "query_name": query_name,
        "generic_class": row.get("generic_class", None),
        "query_family": row.get("query_family", None),
        "query_type": row.get("query_type", None),
        "selected_root": row["selected_root"],
        "n_entities_touched": row.get("n_entities_touched", None),
        "max_entity_depth": max_entity_depth,
        "max_edge_depth": max_edge_depth,
        "n_unique_edges": n_unique_edges,
        "edge_depths": edge_depths,
        "semantic_types": semantic_types,
        "relationship_names": relationship_names,
    })


query_edge_depth_profile_df = pd.DataFrame(query_edge_depth_profile_rows)


# Compatibility alias expected by some notebook versions.
max_depth_by_query_df = query_edge_depth_profile_df[
    [
        "query_name",
        "generic_class",
        "query_family",
        "query_type",
        "selected_root",
        "max_entity_depth",
        "max_edge_depth",
        "n_unique_edges",
    ]
].copy()


# ---------------------------------------------------------
# 9. Validation tables
# ---------------------------------------------------------

edges_missing_semantics_df = query_edges_with_semantics_df[
    query_edges_with_semantics_df["semantic_type"].isna()
].copy() if not query_edges_with_semantics_df.empty else pd.DataFrame()

queries_with_no_edges_df = query_edge_depth_profile_df[
    query_edge_depth_profile_df["n_unique_edges"] == 0
].copy()


# ---------------------------------------------------------
# 10. Display outputs
# ---------------------------------------------------------

print("Structural analysis of edges by depth completed.")

print("\nExpanded edge depth table:")
display(edge_depth_df)

print("\nEdges with semantics:")
display(query_edges_with_semantics_df)

print("\nStructural edges by depth:")
display(structural_edges_by_depth_df)

print("\nEntity depth summary:")
display(entity_depth_summary_df)

print("\nQuery edge-depth profile:")
display(query_edge_depth_profile_df)

if not edges_missing_semantics_df.empty:
    print("\nWarning: some path edges are missing semantic information.")
    display(edges_missing_semantics_df)
else:
    print("\nAll path edges have semantic information.")

if not queries_with_no_edges_df.empty:
    print("\nQueries with no structural edges, usually local lookup queries:")
    display(
        queries_with_no_edges_df[
            ["query_name", "generic_class", "query_family", "selected_root"]
        ]
    )


# ---------------------------------------------------------
# 11. Save BLOCK V9 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    edge_depth_df,
    "variables/block_v09/edge_depth.csv",
)

save_dataframe(
    query_edges_with_semantics_df,
    "variables/block_v09/query_edges_with_semantics.csv",
)

save_dataframe(
    structural_edges_by_depth_df,
    "variables/block_v09/structural_edges_by_depth.csv",
)

save_dataframe(
    query_depth_summary_df,
    "variables/block_v09/query_depth_summary.csv",
)

save_dataframe(
    entity_depth_summary_df,
    "variables/block_v09/entity_depth_summary.csv",
)

save_dataframe(
    query_edge_depth_profile_df,
    "variables/block_v09/query_edge_depth_profile.csv",
)

save_dataframe(
    max_depth_by_query_df,
    "variables/block_v09/max_depth_by_query.csv",
)

save_dataframe(
    edges_missing_semantics_df,
    "variables/block_v09/edges_missing_semantics.csv",
)

save_dataframe(
    queries_with_no_edges_df,
    "variables/block_v09/queries_with_no_edges.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "n_edge_depth_rows": len(edge_depth_df),
        "n_structural_edges_by_depth": len(structural_edges_by_depth_df),
        "n_queries": len(query_edge_depth_profile_df),
        "n_queries_with_no_edges": len(queries_with_no_edges_df),
        "n_edges_missing_semantics": len(edges_missing_semantics_df),
        "max_entity_depth_global": (
            None
            if query_edge_depth_profile_df["max_entity_depth"].dropna().empty
            else int(query_edge_depth_profile_df["max_entity_depth"].dropna().max())
        ),
        "max_edge_depth_global": (
            None
            if query_edge_depth_profile_df["max_edge_depth"].dropna().empty
            else int(query_edge_depth_profile_df["max_edge_depth"].dropna().max())
        ),
        "output_files": {
            "edge_depth_csv": "variables/block_v09/edge_depth.csv",
            "query_edges_with_semantics_csv": "variables/block_v09/query_edges_with_semantics.csv",
            "structural_edges_by_depth_csv": "variables/block_v09/structural_edges_by_depth.csv",
            "query_depth_summary_csv": "variables/block_v09/query_depth_summary.csv",
            "entity_depth_summary_csv": "variables/block_v09/entity_depth_summary.csv",
            "query_edge_depth_profile_csv": "variables/block_v09/query_edge_depth_profile.csv",
            "max_depth_by_query_csv": "variables/block_v09/max_depth_by_query.csv",
            "edges_missing_semantics_csv": "variables/block_v09/edges_missing_semantics.csv",
            "queries_with_no_edges_csv": "variables/block_v09/queries_with_no_edges.csv",
        },
    },
    "variables/block_v09/structural_edges_by_depth_metadata.json",
)

write_execution_log(
    block_name="BLOCK V9 — STRUCTURAL ANALYSIS OF EDGES BY DEPTH",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_edge_depth_rows": len(edge_depth_df),
        "n_structural_edges_by_depth": len(structural_edges_by_depth_df),
        "n_queries": len(query_edge_depth_profile_df),
        "n_queries_with_no_edges": len(queries_with_no_edges_df),
        "n_edges_missing_semantics": len(edges_missing_semantics_df),
        "edge_depth_csv": "variables/block_v09/edge_depth.csv",
        "query_edges_with_semantics_csv": "variables/block_v09/query_edges_with_semantics.csv",
        "structural_edges_by_depth_csv": "variables/block_v09/structural_edges_by_depth.csv",
        "query_depth_summary_csv": "variables/block_v09/query_depth_summary.csv",
        "entity_depth_summary_csv": "variables/block_v09/entity_depth_summary.csv",
        "query_edge_depth_profile_csv": "variables/block_v09/query_edge_depth_profile.csv",
        "metadata_json": "variables/block_v09/structural_edges_by_depth_metadata.json",
    },
)


# ---------------------------------------------------------
# 12. Update README section for BLOCK V9
# ---------------------------------------------------------

max_entity_depth_global = (
    None
    if query_edge_depth_profile_df["max_entity_depth"].dropna().empty
    else int(query_edge_depth_profile_df["max_entity_depth"].dropna().max())
)

max_edge_depth_global = (
    None
    if query_edge_depth_profile_df["max_edge_depth"].dropna().empty
    else int(query_edge_depth_profile_df["max_edge_depth"].dropna().max())
)

block_v09_readme_body = f"""
This block analyzes structural edges by depth from the selected root of each
FIBEN workload query.

Main variables created:

    edge_depth_df
    structural_edges_by_depth_df
    query_edge_depth_profile_df

Additional variables created:

    query_edges_with_semantics_df
    query_depth_summary_df
    entity_depth_summary_df
    max_depth_by_query_df
    edges_missing_semantics_df
    queries_with_no_edges_df

Number of expanded edge-depth rows:

    {len(edge_depth_df)}

Number of unique structural query-edge rows:

    {len(structural_edges_by_depth_df)}

Number of queries processed:

    {len(query_edge_depth_profile_df)}

Maximum entity depth observed:

    {max_entity_depth_global}

Maximum edge depth observed:

    {max_edge_depth_global}

Number of queries with no edges:

    {len(queries_with_no_edges_df)}

Generated reproducibility files:

    variables/block_v09/edge_depth.csv
    variables/block_v09/query_edges_with_semantics.csv
    variables/block_v09/structural_edges_by_depth.csv
    variables/block_v09/query_depth_summary.csv
    variables/block_v09/entity_depth_summary.csv
    variables/block_v09/query_edge_depth_profile.csv
    variables/block_v09/max_depth_by_query.csv
    variables/block_v09/edges_missing_semantics.csv
    variables/block_v09/queries_with_no_edges.csv
    variables/block_v09/structural_edges_by_depth_metadata.json

Methodological meaning:

    This block identifies the depth at which each conceptual relationship
    appears from the selected root of a query.

    This structural depth information is required to compute which entities
    and relationships are reachable under a document embedding depth.

Important note:

    Queries with Rc = 0, such as local lookup over a single entity, may have
    no structural edges.
"""

update_readme_section(
    section_title="BLOCK V9 — Structural Analysis of Edges by Depth",
    section_body=block_v09_readme_body,
)

Structural analysis of edges by depth completed.

Expanded edge depth table:


,query_name,generic_class,query_family,query_type,selected_root,target_entity,target_distance_D,distance_status,edge_position_from_root,edge_depth,from_entity_on_path,to_entity_on_path,edge_id,relationship_name_on_path
0,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Corporation,Industry,1,connected,1,1,Corporation,Industry,Corporation -- Industry [corporation_has_indus...,corporation_has_industry
1,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Corporation,Country,1,connected,1,1,Corporation,Country,Corporation -- Country [corporation_has_country],corporation_has_country
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Corporation,ListedSecurity,1,connected,1,1,Corporation,ListedSecurity,Corporation -- ListedSecurity [corporation_has...,corporation_has_listed_security
3,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount,Holding,1,connected,1,1,FinancialServiceAccount,Holding,FinancialServiceAccount -- Holding [account_ha...,account_has_holding
4,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount,ListedSecurity,2,connected,1,1,FinancialServiceAccount,Holding,FinancialServiceAccount -- Holding [account_ha...,account_has_holding
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,Person,Transaction,2,connected,1,1,Person,FinancialServiceAccount,Person -- FinancialServiceAccount [person_owns...,person_owns_financial_service_account
71,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,Person,Transaction,2,connected,2,2,FinancialServiceAccount,Transaction,FinancialServiceAccount -- Transaction [accoun...,account_records_transaction
72,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,Person,BuyTransaction,3,connected,1,1,Person,FinancialServiceAccount,Person -- FinancialServiceAccount [person_owns...,person_owns_financial_service_account
73,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,Person,BuyTransaction,3,connected,2,2,FinancialServiceAccount,Transaction,FinancialServiceAccount -- Transaction [accoun...,account_records_transaction



Edges with semantics:


,query_name,generic_class,query_family,query_type,selected_root,target_entity,target_distance_D,distance_status,edge_position_from_root,edge_depth,...,edge_id,relationship_name_on_path,name,source,target,semantic_type,observed_cardinality,hint_type,hint_confidence,relationship_name
0,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Corporation,Industry,1,connected,1,1,...,Corporation -- Industry [corporation_has_indus...,corporation_has_industry,corporation_has_industry,Corporation,Industry,descriptor,observed_N_to_M,direct,high,corporation_has_industry
1,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Corporation,Country,1,connected,1,1,...,Corporation -- Country [corporation_has_country],corporation_has_country,corporation_has_country,Corporation,Country,descriptor,observed_N_to_M,direct,high,corporation_has_country
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Corporation,ListedSecurity,1,connected,1,1,...,Corporation -- ListedSecurity [corporation_has...,corporation_has_listed_security,corporation_has_listed_security,Corporation,ListedSecurity,association,no_observed_matches,bridge,medium,corporation_has_listed_security
3,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount,Holding,1,connected,1,1,...,FinancialServiceAccount -- Holding [account_ha...,account_has_holding,account_has_holding,FinancialServiceAccount,Holding,containment,observed_N_to_M,direct,high,account_has_holding
4,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount,ListedSecurity,2,connected,1,1,...,FinancialServiceAccount -- Holding [account_ha...,account_has_holding,account_has_holding,FinancialServiceAccount,Holding,containment,observed_N_to_M,direct,high,account_has_holding
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,Person,Transaction,2,connected,1,1,...,Person -- FinancialServiceAccount [person_owns...,person_owns_financial_service_account,person_owns_financial_service_account,Person,FinancialServiceAccount,ownership,observed_N_to_M,direct,high,person_owns_financial_service_account
71,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,Person,Transaction,2,connected,2,2,...,FinancialServiceAccount -- Transaction [accoun...,account_records_transaction,account_records_transaction,FinancialServiceAccount,Transaction,containment,observed_N_to_M,direct,high,account_records_transaction
72,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,Person,BuyTransaction,3,connected,1,1,...,Person -- FinancialServiceAccount [person_owns...,person_owns_financial_service_account,person_owns_financial_service_account,Person,FinancialServiceAccount,ownership,observed_N_to_M,direct,high,person_owns_financial_service_account
73,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,Person,BuyTransaction,3,connected,2,2,...,FinancialServiceAccount -- Transaction [accoun...,account_records_transaction,account_records_transaction,FinancialServiceAccount,Transaction,containment,observed_N_to_M,direct,high,account_records_transaction



Structural edges by depth:


...


Query edge-depth profile:


,query_name,generic_class,query_family,query_type,selected_root,n_entities_touched,max_entity_depth,max_edge_depth,n_unique_edges,edge_depths,semantic_types,relationship_names
0,Q1_CompanyProfileIBM,QG1,Local lookup,select,Corporation,1,0,0,0,[],[],[]
1,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Corporation,4,1,1,3,[1],"[association, descriptor]","[corporation_has_country, corporation_has_indu..."
2,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount,4,3,3,3,"[1, 2, 3]","[association, containment]","[account_has_holding, holding_refers_to_listed..."
3,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Person,5,4,4,4,"[1, 2, 3, 4]","[association, containment, ownership]","[account_has_holding, corporation_has_listed_s..."
4,Q5_ReportsAndMetricDataOfCompany,QG5,Containment retrieval,select,Corporation,5,3,3,4,"[1, 2, 3]",[containment],"[corporation_has_financial_report, financial_r..."
5,Q6_TechUSListedSecuritiesWithHighLastTradedValue,QG6,Filtered search / recommendation,select,ListedSecurity,4,2,2,3,"[1, 2]","[association, descriptor]","[corporation_has_country, corporation_has_indu..."
6,Q7_PersonsWhoBoughtMoreIBMThanSold,QG7,Aggregation / ranking,select,Person,7,4,4,7,"[1, 2, 3, 4]","[association, containment, ownership, subtype]","[account_has_holding, account_records_transact..."
7,Q8_IBMTransactionsBelowAverageSellingPrice,QG8,Aggregation / ranking,select,Transaction,4,2,2,3,"[1, 2]","[association, subtype]","[corporation_has_listed_security, sell_transac..."
8,Q9_PersonsWhoBoughtAndSoldSameStock,QG9,Associative retrieval + comparison,select,Person,6,3,3,6,"[1, 2, 3]","[association, containment, ownership, subtype]","[account_has_holding, account_records_transact..."
9,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,Person,6,3,3,5,"[1, 2, 3]","[association, containment, ownership, subtype]","[account_has_holding, account_records_transact..."



All path edges have semantic information.

Queries with no structural edges, usually local lookup queries:


,query_name,generic_class,query_family,selected_root
0,Q1_CompanyProfileIBM,QG1,Local lookup,Corporation


Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v09/edge_depth.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v09/query_edges_with_semantics.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v09/structural_edges_by_depth.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v09/query_depth_summary.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v09/entity_depth_summary.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v09/query_edge_depth_profile.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v09/max_depth_by_query.c

In [76]:
query_edge_depth_profile_df[
    [
        "query_name",
        "selected_root",
        "max_entity_depth",
        "max_edge_depth",
        "n_unique_edges",
        "semantic_types",
    ]
]

,query_name,selected_root,max_entity_depth,max_edge_depth,n_unique_edges,semantic_types
0,Q1_CompanyProfileIBM,Corporation,0,0,0,[]
1,Q2_CompanyWithIndustryCountryAndListedSecurities,Corporation,1,1,3,"[association, descriptor]"
2,Q3_SecuritiesHeldInEachFinancialServiceAccount,FinancialServiceAccount,3,3,3,"[association, containment]"
3,Q4_CompaniesReachedFromPersonThroughAccountHol...,Person,4,4,4,"[association, containment, ownership]"
4,Q5_ReportsAndMetricDataOfCompany,Corporation,3,3,4,[containment]
5,Q6_TechUSListedSecuritiesWithHighLastTradedValue,ListedSecurity,2,2,3,"[association, descriptor]"
6,Q7_PersonsWhoBoughtMoreIBMThanSold,Person,4,4,7,"[association, containment, ownership, subtype]"
7,Q8_IBMTransactionsBelowAverageSellingPrice,Transaction,2,2,3,"[association, subtype]"
8,Q9_PersonsWhoBoughtAndSoldSameStock,Person,3,3,6,"[association, containment, ownership, subtype]"
9,Q10_CreateAccountHoldingAndBuyTransaction,Person,3,3,5,"[association, containment, ownership, subtype]"


In [77]:
# =========================================================
# BLOCK V10 — CALCULATE Re(Qi, er, D) AND DeltaR
# =========================================================
#
# Purpose:
# This block computes Re(Qi, er, D), DeltaR, and DeltaRratio
# for each FIBEN workload query and each candidate document depth.
#
# Definitions:
#
# Rc(Qi):
#     Original number of conceptual relationships required to connect
#     all entities touched by query Qi.
#
# Re(Qi, er, D):
#     Number of relationships that remain external after choosing
#     selected root er and embedding entities up to depth D.
#
# DeltaR:
#     Rc - Re
#
# DeltaRratio:
#     DeltaR / Rc
#
# Methodological intuition:
#
# If a relationship edge is within the selected embedding depth D,
# it can be internalized in the document structure.
#
# If an edge is deeper than D, it remains external.
#
# Main outputs preserved for downstream compatibility:
# - re_df
# - delta_r_df
# - re_delta_df
#
# Additional outputs:
# - re_delta_summary_df
# - best_depth_by_query_df
# - depth_candidate_matrix_df
# =========================================================

import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "rc_df",
    "distance_df",
    "selected_root_df",
    "selected_root_by_query",
    "structural_edges_by_depth_df",
    "query_edge_depth_profile_df",
    "fiben_scenario",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first BLOCKS V2, V4, V5, and V9."
        )


# ---------------------------------------------------------
# 2. Configuration
# ---------------------------------------------------------
#
# Maximum depth tested by the methodology.
#
# If None, the block uses the maximum entity depth observed per query.
# If an integer is provided, the block tests depths from 0 to that value.
# ---------------------------------------------------------

FIBEN_MAX_DOCUMENT_DEPTH_TO_TEST = None


# ---------------------------------------------------------
# 3. Helper: safe list conversion
# ---------------------------------------------------------

def safe_list(value):
    """
    Convert a value into a list.

    This helper protects the block when a DataFrame cell contains:
    - list;
    - tuple;
    - missing value;
    - scalar.
    """
    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    if value is None:
        return []

    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    return [value]


# ---------------------------------------------------------
# 4. Helper: compute DeltaRratio safely
# ---------------------------------------------------------

def compute_delta_r_ratio(rc_value, delta_r_value):
    """
    Compute DeltaRratio safely.

    If Rc is 0, the ratio is defined as 0.0 because there are no
    relationships to reduce.
    """
    if rc_value is None:
        return None

    try:
        if pd.isna(rc_value):
            return None
    except Exception:
        pass

    if rc_value == 0:
        return 0.0

    return float(delta_r_value) / float(rc_value)


# ---------------------------------------------------------
# 5. Build Rc lookup
# ---------------------------------------------------------

rc_lookup = {}

for _, row in rc_df.iterrows():
    rc_lookup[row["query_name"]] = {
        "Rc": row["Rc"],
        "Rc_status": row["Rc_status"],
        "Rc_selected_edges": safe_list(row.get("Rc_selected_edges", [])),
    }


# ---------------------------------------------------------
# 6. Build distance lookup by query and depth
# ---------------------------------------------------------

query_entities_by_depth = {}

for query_name, group in distance_df.groupby("query_name"):
    depth_map = {}

    for _, row in group.iterrows():
        d_value = row["D"]

        if pd.isna(d_value):
            continue

        d_value = int(d_value)

        if d_value not in depth_map:
            depth_map[d_value] = []

        depth_map[d_value].append(row["entity"])

    query_entities_by_depth[query_name] = {
        depth: sorted(set(entities))
        for depth, entities in depth_map.items()
    }


# ---------------------------------------------------------
# 7. Build edge-depth lookup by query
# ---------------------------------------------------------

query_edges_depth_lookup = {}

if structural_edges_by_depth_df.empty:
    query_edges_depth_lookup = {}
else:
    for query_name, group in structural_edges_by_depth_df.groupby("query_name"):
        edge_rows = []

        for _, row in group.iterrows():
            edge_rows.append({
                "edge_id": row["edge_id"],
                "relationship_name": row.get("relationship_name"),
                "semantic_type": row.get("semantic_type"),
                "min_edge_depth": int(row["min_edge_depth"]),
                "observed_cardinality": row.get("observed_cardinality"),
                "hint_confidence": row.get("hint_confidence"),
            })

        query_edges_depth_lookup[query_name] = edge_rows


# ---------------------------------------------------------
# 8. Compute Re, DeltaR, and DeltaRratio
# ---------------------------------------------------------

re_delta_rows = []

for _, query_row in selected_root_df.iterrows():
    query_name = query_row["query_name"]
    generic_class = query_row.get("generic_class", None)
    query_family = query_row.get("query_family", None)
    query_type = query_row.get("query_type", None)
    selected_root = query_row["selected_root"]

    rc_info = rc_lookup.get(query_name, {})
    rc_value = rc_info.get("Rc")
    rc_status = rc_info.get("Rc_status")
    rc_selected_edges = rc_info.get("Rc_selected_edges", [])

    q_distance_rows = distance_df[
        distance_df["query_name"] == query_name
    ].copy()

    if q_distance_rows.empty or q_distance_rows["D"].dropna().empty:
        max_depth_for_query = 0
    else:
        max_depth_for_query = int(q_distance_rows["D"].dropna().max())

    if FIBEN_MAX_DOCUMENT_DEPTH_TO_TEST is not None:
        max_depth_to_test = int(FIBEN_MAX_DOCUMENT_DEPTH_TO_TEST)
    else:
        max_depth_to_test = max_depth_for_query

    q_edges = query_edges_depth_lookup.get(query_name, [])

    total_edges = 0 if rc_value is None or pd.isna(rc_value) else int(rc_value)

    for document_depth in range(0, max_depth_to_test + 1):
        reachable_entities = (
            q_distance_rows[
                q_distance_rows["D"].notna()
                & (q_distance_rows["D"] <= document_depth)
            ]["entity"]
            .dropna()
            .unique()
            .tolist()
        )

        reachable_entities = sorted(reachable_entities)

        covered_edges = [
            edge for edge in q_edges
            if edge["min_edge_depth"] <= document_depth
        ]

        remaining_edges = [
            edge for edge in q_edges
            if edge["min_edge_depth"] > document_depth
        ]

        covered_edge_ids = [edge["edge_id"] for edge in covered_edges]
        remaining_edge_ids = [edge["edge_id"] for edge in remaining_edges]

        covered_relationships = sorted(
            set(
                edge["relationship_name"]
                for edge in covered_edges
                if edge["relationship_name"] is not None
            )
        )

        remaining_relationships = sorted(
            set(
                edge["relationship_name"]
                for edge in remaining_edges
                if edge["relationship_name"] is not None
            )
        )

        covered_semantic_types = sorted(
            set(
                edge["semantic_type"]
                for edge in covered_edges
                if edge["semantic_type"] is not None
            )
        )

        remaining_semantic_types = sorted(
            set(
                edge["semantic_type"]
                for edge in remaining_edges
                if edge["semantic_type"] is not None
            )
        )

        # Re is the number of relationship edges that remain external.
        re_value = len(remaining_edge_ids)

        # For safety, if Rc is zero, Re must also be zero.
        if total_edges == 0:
            re_value = 0

        delta_r = total_edges - re_value
        delta_r_ratio = compute_delta_r_ratio(total_edges, delta_r)

        re_delta_rows.append({
            "query_name": query_name,
            "generic_class": generic_class,
            "query_family": query_family,
            "query_type": query_type,
            "selected_root": selected_root,
            "document_depth": document_depth,
            "max_depth_for_query": max_depth_for_query,
            "Rc": total_edges,
            "Rc_status": rc_status,
            "Re": re_value,
            "DeltaR": delta_r,
            "DeltaRratio": delta_r_ratio,
            "n_reachable_entities": len(reachable_entities),
            "reachable_entities": reachable_entities,
            "n_total_edges": total_edges,
            "n_covered_edges": len(covered_edge_ids),
            "n_remaining_edges": len(remaining_edge_ids),
            "covered_edge_ids": covered_edge_ids,
            "remaining_edge_ids": remaining_edge_ids,
            "covered_relationships": covered_relationships,
            "remaining_relationships": remaining_relationships,
            "covered_semantic_types": covered_semantic_types,
            "remaining_semantic_types": remaining_semantic_types,
            "is_full_coverage": re_value == 0,
        })


re_delta_df = pd.DataFrame(re_delta_rows)

# Compatibility aliases.
re_df = re_delta_df.copy()
delta_r_df = re_delta_df.copy()


# ---------------------------------------------------------
# 9. Build best-depth table
# ---------------------------------------------------------
#
# The best depth is the smallest depth that gives the maximum
# DeltaRratio for each query.
# ---------------------------------------------------------

best_depth_rows = []

for query_name, group in re_delta_df.groupby("query_name"):
    if group.empty:
        continue

    max_ratio = group["DeltaRratio"].max()

    best_group = (
        group[group["DeltaRratio"] == max_ratio]
        .sort_values("document_depth")
        .head(1)
    )

    best_row = best_group.iloc[0].to_dict()

    best_depth_rows.append({
        "query_name": best_row["query_name"],
        "generic_class": best_row["generic_class"],
        "query_family": best_row["query_family"],
        "query_type": best_row["query_type"],
        "selected_root": best_row["selected_root"],
        "best_document_depth": best_row["document_depth"],
        "max_depth_for_query": best_row["max_depth_for_query"],
        "Rc": best_row["Rc"],
        "Re_at_best_depth": best_row["Re"],
        "DeltaR_at_best_depth": best_row["DeltaR"],
        "DeltaRratio_at_best_depth": best_row["DeltaRratio"],
        "n_reachable_entities_at_best_depth": best_row["n_reachable_entities"],
        "reachable_entities_at_best_depth": best_row["reachable_entities"],
        "is_full_coverage_at_best_depth": best_row["is_full_coverage"],
    })

best_depth_by_query_df = pd.DataFrame(best_depth_rows)


# ---------------------------------------------------------
# 10. Build compact summary tables
# ---------------------------------------------------------

re_delta_summary_df = (
    re_delta_df
    .groupby(
        ["generic_class", "query_family", "document_depth"],
        dropna=False,
    )
    .agg(
        n_queries=("query_name", "nunique"),
        avg_Rc=("Rc", "mean"),
        avg_Re=("Re", "mean"),
        avg_DeltaR=("DeltaR", "mean"),
        avg_DeltaRratio=("DeltaRratio", "mean"),
        max_DeltaRratio=("DeltaRratio", "max"),
        n_full_coverage=("is_full_coverage", lambda s: int(s.sum())),
    )
    .reset_index()
    .sort_values(["generic_class", "document_depth"])
    .reset_index(drop=True)
)

depth_candidate_matrix_df = re_delta_df[
    [
        "query_name",
        "generic_class",
        "query_family",
        "query_type",
        "selected_root",
        "document_depth",
        "Rc",
        "Re",
        "DeltaR",
        "DeltaRratio",
        "n_reachable_entities",
        "n_covered_edges",
        "n_remaining_edges",
        "is_full_coverage",
    ]
].copy()


# ---------------------------------------------------------
# 11. Validation tables
# ---------------------------------------------------------

negative_delta_df = re_delta_df[
    re_delta_df["DeltaR"] < 0
].copy()

invalid_ratio_df = re_delta_df[
    re_delta_df["DeltaRratio"].isna()
].copy()

queries_without_full_coverage_df = (
    best_depth_by_query_df[
        best_depth_by_query_df["is_full_coverage_at_best_depth"] == False
    ].copy()
    if not best_depth_by_query_df.empty
    else pd.DataFrame()
)


# ---------------------------------------------------------
# 12. Display outputs
# ---------------------------------------------------------

print("Re(Qi, er, D), DeltaR, and DeltaRratio calculation completed.")

print("\nRe / DeltaR by query and depth:")
display(re_delta_df)

print("\nBest depth by query:")
display(best_depth_by_query_df)

print("\nDepth candidate matrix:")
display(depth_candidate_matrix_df)

print("\nSummary by generic class and depth:")
display(re_delta_summary_df)

if not negative_delta_df.empty:
    print("\nWarning: some rows have negative DeltaR.")
    display(negative_delta_df)
else:
    print("\nNo negative DeltaR values found.")

if not queries_without_full_coverage_df.empty:
    print("\nQueries without full coverage at best depth:")
    display(queries_without_full_coverage_df)
else:
    print("\nAll queries reach full coverage at their best selected depth.")


# ---------------------------------------------------------
# 13. Save BLOCK V10 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    re_delta_df,
    "variables/block_v10/re_delta_by_query_depth.csv",
)

save_dataframe(
    re_df,
    "variables/block_v10/re_df.csv",
)

save_dataframe(
    delta_r_df,
    "variables/block_v10/delta_r_df.csv",
)

save_dataframe(
    best_depth_by_query_df,
    "variables/block_v10/best_depth_by_query.csv",
)

save_dataframe(
    re_delta_summary_df,
    "variables/block_v10/re_delta_summary_by_class_depth.csv",
)

save_dataframe(
    depth_candidate_matrix_df,
    "variables/block_v10/depth_candidate_matrix.csv",
)

save_dataframe(
    negative_delta_df,
    "variables/block_v10/negative_delta_rows.csv",
)

save_dataframe(
    invalid_ratio_df,
    "variables/block_v10/invalid_ratio_rows.csv",
)

save_dataframe(
    queries_without_full_coverage_df,
    "variables/block_v10/queries_without_full_coverage.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "max_document_depth_to_test": FIBEN_MAX_DOCUMENT_DEPTH_TO_TEST,
        "n_re_delta_rows": len(re_delta_df),
        "n_queries": int(re_delta_df["query_name"].nunique()),
        "max_document_depth_observed": (
            None
            if re_delta_df["document_depth"].dropna().empty
            else int(re_delta_df["document_depth"].dropna().max())
        ),
        "n_negative_delta_rows": len(negative_delta_df),
        "n_invalid_ratio_rows": len(invalid_ratio_df),
        "n_queries_without_full_coverage": len(queries_without_full_coverage_df),
        "output_files": {
            "re_delta_csv": "variables/block_v10/re_delta_by_query_depth.csv",
            "re_df_csv": "variables/block_v10/re_df.csv",
            "delta_r_df_csv": "variables/block_v10/delta_r_df.csv",
            "best_depth_by_query_csv": "variables/block_v10/best_depth_by_query.csv",
            "re_delta_summary_csv": "variables/block_v10/re_delta_summary_by_class_depth.csv",
            "depth_candidate_matrix_csv": "variables/block_v10/depth_candidate_matrix.csv",
            "negative_delta_csv": "variables/block_v10/negative_delta_rows.csv",
            "invalid_ratio_csv": "variables/block_v10/invalid_ratio_rows.csv",
            "queries_without_full_coverage_csv": "variables/block_v10/queries_without_full_coverage.csv",
        },
    },
    "variables/block_v10/re_delta_metadata.json",
)

write_execution_log(
    block_name="BLOCK V10 — CALCULATE Re(Qi, er, D) AND DeltaR",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_re_delta_rows": len(re_delta_df),
        "n_queries": int(re_delta_df["query_name"].nunique()),
        "n_negative_delta_rows": len(negative_delta_df),
        "n_invalid_ratio_rows": len(invalid_ratio_df),
        "n_queries_without_full_coverage": len(queries_without_full_coverage_df),
        "re_delta_csv": "variables/block_v10/re_delta_by_query_depth.csv",
        "re_df_csv": "variables/block_v10/re_df.csv",
        "delta_r_df_csv": "variables/block_v10/delta_r_df.csv",
        "best_depth_by_query_csv": "variables/block_v10/best_depth_by_query.csv",
        "re_delta_summary_csv": "variables/block_v10/re_delta_summary_by_class_depth.csv",
        "depth_candidate_matrix_csv": "variables/block_v10/depth_candidate_matrix.csv",
        "metadata_json": "variables/block_v10/re_delta_metadata.json",
    },
)


# ---------------------------------------------------------
# 14. Update README section for BLOCK V10
# ---------------------------------------------------------

max_document_depth_observed = (
    None
    if re_delta_df["document_depth"].dropna().empty
    else int(re_delta_df["document_depth"].dropna().max())
)

block_v10_readme_body = f"""
This block computes Re(Qi, er, D), DeltaR, and DeltaRratio for the FIBEN
workload.

Main variables created:

    re_df
    delta_r_df
    re_delta_df

Additional variables created:

    best_depth_by_query_df
    re_delta_summary_df
    depth_candidate_matrix_df
    negative_delta_df
    invalid_ratio_df
    queries_without_full_coverage_df

Number of Re/DeltaR rows:

    {len(re_delta_df)}

Number of queries processed:

    {int(re_delta_df["query_name"].nunique())}

Maximum document depth tested:

    {max_document_depth_observed}

Number of negative DeltaR rows:

    {len(negative_delta_df)}

Number of invalid DeltaRratio rows:

    {len(invalid_ratio_df)}

Generated reproducibility files:

    variables/block_v10/re_delta_by_query_depth.csv
    variables/block_v10/re_df.csv
    variables/block_v10/delta_r_df.csv
    variables/block_v10/best_depth_by_query.csv
    variables/block_v10/re_delta_summary_by_class_depth.csv
    variables/block_v10/depth_candidate_matrix.csv
    variables/block_v10/negative_delta_rows.csv
    variables/block_v10/invalid_ratio_rows.csv
    variables/block_v10/queries_without_full_coverage.csv
    variables/block_v10/re_delta_metadata.json

Methodological meaning:

    Rc(Qi) is the original number of conceptual relationships required by a query.

    Re(Qi, er, D) is the number of relationships that remain external after
    choosing root er and embedding entities up to document depth D.

    DeltaR = Rc - Re

    DeltaRratio = DeltaR / Rc

Interpretation:

    Higher DeltaRratio means that a larger portion of the original relational
    traversal can potentially be internalized by the document configuration.

Important note:

    For queries with Rc = 0, DeltaRratio is defined as 0.0 because there are
    no relationship edges to reduce.
"""

update_readme_section(
    section_title="BLOCK V10 — Calculate Re(Qi, er, D) and DeltaR",
    section_body=block_v10_readme_body,
)

Re(Qi, er, D), DeltaR, and DeltaRratio calculation completed.

Re / DeltaR by query and depth:


,query_name,generic_class,query_family,query_type,selected_root,document_depth,max_depth_for_query,Rc,Rc_status,Re,...,n_total_edges,n_covered_edges,n_remaining_edges,covered_edge_ids,remaining_edge_ids,covered_relationships,remaining_relationships,covered_semantic_types,remaining_semantic_types,is_full_coverage
0,Q1_CompanyProfileIBM,QG1,Local lookup,select,Corporation,0,0,0,single_or_no_terminal,0,...,0,0,0,[],[],[],[],[],[],True
1,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Corporation,0,1,3,connected,3,...,3,0,3,[],[Corporation -- Country [corporation_has_count...,[],"[corporation_has_country, corporation_has_indu...",[],"[association, descriptor]",False
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Corporation,1,1,3,connected,0,...,3,3,0,[Corporation -- Country [corporation_has_count...,[],"[corporation_has_country, corporation_has_indu...",[],"[association, descriptor]",[],True
3,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount,0,3,3,connected,3,...,3,0,3,[],[FinancialServiceAccount -- Holding [account_h...,[],"[account_has_holding, holding_refers_to_listed...",[],"[association, containment]",False
4,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount,1,3,3,connected,2,...,3,1,2,[FinancialServiceAccount -- Holding [account_h...,[Holding -- ListedSecurity [holding_refers_to_...,[account_has_holding],"[holding_refers_to_listed_security, listed_sec...",[containment],[association],False
5,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount,2,3,3,connected,1,...,3,2,1,[FinancialServiceAccount -- Holding [account_h...,[ListedSecurity -- Security [listed_security_r...,"[account_has_holding, holding_refers_to_listed...",[listed_security_represents_security],"[association, containment]",[association],False
6,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount,3,3,3,connected,0,...,3,3,0,[FinancialServiceAccount -- Holding [account_h...,[],"[account_has_holding, holding_refers_to_listed...",[],"[association, containment]",[],True
7,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Person,0,4,4,connected,4,...,4,0,4,[],[Person -- FinancialServiceAccount [person_own...,[],"[account_has_holding, corporation_has_listed_s...",[],"[association, containment, ownership]",False
8,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Person,1,4,4,connected,3,...,4,1,3,[Person -- FinancialServiceAccount [person_own...,[FinancialServiceAccount -- Holding [account_h...,[person_owns_financial_service_account],"[account_has_holding, corporation_has_listed_s...",[ownership],"[association, containment]",False
9,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Person,2,4,4,connected,2,...,4,2,2,[Person -- FinancialServiceAccount [person_own...,[Holding -- ListedSecurity [holding_refers_to_...,"[account_has_holding, person_owns_financial_se...","[corporation_has_listed_security, holding_refe...","[containment, ownership]",[association],False



Best depth by query:


,query_name,generic_class,query_family,query_type,selected_root,best_document_depth,max_depth_for_query,Rc,Re_at_best_depth,DeltaR_at_best_depth,DeltaRratio_at_best_depth,n_reachable_entities_at_best_depth,reachable_entities_at_best_depth,is_full_coverage_at_best_depth
0,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,Person,3,3,5,0,5,1.0,6,"[BuyTransaction, FinancialServiceAccount, Hold...",True
1,Q1_CompanyProfileIBM,QG1,Local lookup,select,Corporation,0,0,0,0,0,0.0,1,[Corporation],True
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Corporation,1,1,3,0,3,1.0,4,"[Corporation, Country, Industry, ListedSecurity]",True
3,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount,3,3,3,0,3,1.0,4,"[FinancialServiceAccount, Holding, ListedSecur...",True
4,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Person,4,4,4,0,4,1.0,5,"[Corporation, FinancialServiceAccount, Holding...",True
5,Q5_ReportsAndMetricDataOfCompany,QG5,Containment retrieval,select,Corporation,3,3,4,0,4,1.0,5,"[Corporation, Disclosure, FinancialReport, Rep...",True
6,Q6_TechUSListedSecuritiesWithHighLastTradedValue,QG6,Filtered search / recommendation,select,ListedSecurity,2,2,3,0,3,1.0,4,"[Corporation, Country, Industry, ListedSecurity]",True
7,Q7_PersonsWhoBoughtMoreIBMThanSold,QG7,Aggregation / ranking,select,Person,4,4,6,0,6,1.0,7,"[BuyTransaction, Corporation, FinancialService...",True
8,Q8_IBMTransactionsBelowAverageSellingPrice,QG8,Aggregation / ranking,select,Transaction,2,2,3,0,3,1.0,4,"[Corporation, ListedSecurity, SellTransaction,...",True
9,Q9_PersonsWhoBoughtAndSoldSameStock,QG9,Associative retrieval + comparison,select,Person,3,3,5,0,5,1.0,6,"[BuyTransaction, FinancialServiceAccount, List...",True



Depth candidate matrix:


,query_name,generic_class,query_family,query_type,selected_root,document_depth,Rc,Re,DeltaR,DeltaRratio,n_reachable_entities,n_covered_edges,n_remaining_edges,is_full_coverage
0,Q1_CompanyProfileIBM,QG1,Local lookup,select,Corporation,0,0,0,0,0.000000,1,0,0,True
1,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Corporation,0,3,3,0,0.000000,1,0,3,False
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Corporation,1,3,0,3,1.000000,4,3,0,True
3,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount,0,3,3,0,0.000000,1,0,3,False
4,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount,1,3,2,1,0.333333,2,1,2,False
5,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount,2,3,1,2,0.666667,3,2,1,False
6,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount,3,3,0,3,1.000000,4,3,0,True
7,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Person,0,4,4,0,0.000000,1,0,4,False
8,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Person,1,4,3,1,0.250000,2,1,3,False
9,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Person,2,4,2,2,0.500000,3,2,2,False



Summary by generic class and depth:


,generic_class,query_family,document_depth,n_queries,avg_Rc,avg_Re,avg_DeltaR,avg_DeltaRratio,max_DeltaRratio,n_full_coverage
0,QG1,Local lookup,0,1,0.0,0.0,0.0,0.000000,0.000000,1
1,QG10,Update / insertion with relationship creation,0,1,5.0,5.0,0.0,0.000000,0.000000,0
2,QG10,Update / insertion with relationship creation,1,1,5.0,4.0,1.0,0.200000,0.200000,0
3,QG10,Update / insertion with relationship creation,2,1,5.0,2.0,3.0,0.600000,0.600000,0
4,QG10,Update / insertion with relationship creation,3,1,5.0,0.0,5.0,1.000000,1.000000,1
5,QG2,Shallow rooted retrieval,0,1,3.0,3.0,0.0,0.000000,0.000000,0
6,QG2,Shallow rooted retrieval,1,1,3.0,0.0,3.0,1.000000,1.000000,1
7,QG3,Associative retrieval,0,1,3.0,3.0,0.0,0.000000,0.000000,0
8,QG3,Associative retrieval,1,1,3.0,2.0,1.0,0.333333,0.333333,0
9,QG3,Associative retrieval,2,1,3.0,1.0,2.0,0.666667,0.666667,0


,query_name,generic_class,query_family,query_type,selected_root,document_depth,max_depth_for_query,Rc,Rc_status,Re,...,n_total_edges,n_covered_edges,n_remaining_edges,covered_edge_ids,remaining_edge_ids,covered_relationships,remaining_relationships,covered_semantic_types,remaining_semantic_types,is_full_coverage
19,Q7_PersonsWhoBoughtMoreIBMThanSold,QG7,Aggregation / ranking,select,Person,0,4,6,connected,7,...,6,0,7,[],[Person -- FinancialServiceAccount [person_own...,[],"[account_has_holding, account_records_transact...",[],"[association, containment, ownership, subtype]",False
27,Q9_PersonsWhoBoughtAndSoldSameStock,QG9,Associative retrieval + comparison,select,Person,0,3,5,connected,6,...,5,0,6,[],[Person -- FinancialServiceAccount [person_own...,[],"[account_has_holding, account_records_transact...",[],"[association, containment, ownership, subtype]",False



All queries reach full coverage at their best selected depth.
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v10/re_delta_by_query_depth.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v10/re_df.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v10/delta_r_df.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v10/best_depth_by_query.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v10/re_delta_summary_by_class_depth.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v10/depth_candidate_matrix.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with datas

In [78]:
best_depth_by_query_df[
    [
        "query_name",
        "selected_root",
        "best_document_depth",
        "Rc",
        "Re_at_best_depth",
        "DeltaR_at_best_depth",
        "DeltaRratio_at_best_depth",
        "is_full_coverage_at_best_depth",
    ]
]

,query_name,selected_root,best_document_depth,Rc,Re_at_best_depth,DeltaR_at_best_depth,DeltaRratio_at_best_depth,is_full_coverage_at_best_depth
0,Q10_CreateAccountHoldingAndBuyTransaction,Person,3,5,0,5,1.0,True
1,Q1_CompanyProfileIBM,Corporation,0,0,0,0,0.0,True
2,Q2_CompanyWithIndustryCountryAndListedSecurities,Corporation,1,3,0,3,1.0,True
3,Q3_SecuritiesHeldInEachFinancialServiceAccount,FinancialServiceAccount,3,3,0,3,1.0,True
4,Q4_CompaniesReachedFromPersonThroughAccountHol...,Person,4,4,0,4,1.0,True
5,Q5_ReportsAndMetricDataOfCompany,Corporation,3,4,0,4,1.0,True
6,Q6_TechUSListedSecuritiesWithHighLastTradedValue,ListedSecurity,2,3,0,3,1.0,True
7,Q7_PersonsWhoBoughtMoreIBMThanSold,Person,4,6,0,6,1.0,True
8,Q8_IBMTransactionsBelowAverageSellingPrice,Transaction,2,3,0,3,1.0,True
9,Q9_PersonsWhoBoughtAndSoldSameStock,Person,3,5,0,5,1.0,True


In [79]:
# =========================================================
# BLOCK V11 — ASSEMBLE COMPACT MATRIX BY QUERY
# =========================================================
#
# Purpose:
# This block assembles a compact analytical matrix with one row
# per FIBEN workload query.
#
# It integrates the main variables computed so far:
# - workload metadata;
# - Rc;
# - selected root;
# - conceptual distance D;
# - best document depth;
# - Re;
# - DeltaR;
# - DeltaRratio;
# - semantic relationship profile;
# - observed cardinality profile.
#
# Main outputs preserved for downstream compatibility:
# - compact_matrix_by_query_df
# - compact_query_matrix_df
# - query_compact_matrix_df
#
# Methodological role:
# This compact matrix summarizes the current state of the methodology
# before generating the expanded analytical matrix by depth.
# =========================================================

import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "workload_conceptual_df",
    "rc_df",
    "selected_root_df",
    "distance_df",
    "distance_summary_by_query_df",
    "query_semantic_profile_df",
    "query_semantic_flags_df",
    "query_edge_depth_profile_df",
    "best_depth_by_query_df",
    "re_delta_df",
    "observed_cardinality_df",
    "fiben_scenario",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first BLOCKS V0–V10."
        )


# ---------------------------------------------------------
# 2. Helper functions
# ---------------------------------------------------------

def safe_unique_list(series):
    """
    Return a sorted list of unique non-null values.
    """
    values = []

    for value in series:
        if isinstance(value, list):
            values.extend(value)
        elif pd.notna(value):
            values.append(value)

    return sorted(set(values))


def safe_first(series, default=None):
    """
    Return the first non-null value from a pandas Series.
    """
    for value in series:
        try:
            if pd.notna(value):
                return value
        except Exception:
            if value is not None:
                return value

    return default


# ---------------------------------------------------------
# 3. Start from workload metadata
# ---------------------------------------------------------

compact_matrix_by_query_df = workload_conceptual_df[
    [
        "query_name",
        "generic_class",
        "query_family",
        "query_type",
        "entities_touched",
        "n_entities_touched",
        "abstract_query",
    ]
].copy()


# ---------------------------------------------------------
# 4. Add Rc variables
# ---------------------------------------------------------

rc_compact_df = rc_df[
    [
        "query_name",
        "Rc",
        "Rc_status",
        "Rc_selected_edges",
    ]
].copy()

compact_matrix_by_query_df = compact_matrix_by_query_df.merge(
    rc_compact_df,
    on="query_name",
    how="left",
)


# ---------------------------------------------------------
# 5. Add selected root variables
# ---------------------------------------------------------

selected_root_compact_df = selected_root_df[
    [
        "query_name",
        "selected_root",
        "selection_strategy",
        "selection_reason",
        "selected_root_is_candidate",
        "selected_root_is_touched",
    ]
].copy()

compact_matrix_by_query_df = compact_matrix_by_query_df.merge(
    selected_root_compact_df,
    on="query_name",
    how="left",
)


# ---------------------------------------------------------
# 6. Add distance summary variables
# ---------------------------------------------------------

distance_compact_df = distance_summary_by_query_df[
    [
        "query_name",
        "max_D",
        "min_D",
        "avg_D",
        "n_unreachable",
    ]
].copy()

distance_compact_df = distance_compact_df.rename(columns={
    "max_D": "max_entity_distance_D",
    "min_D": "min_entity_distance_D",
    "avg_D": "avg_entity_distance_D",
    "n_unreachable": "n_unreachable_entities",
})

compact_matrix_by_query_df = compact_matrix_by_query_df.merge(
    distance_compact_df,
    on="query_name",
    how="left",
)


# ---------------------------------------------------------
# 7. Add structural depth variables
# ---------------------------------------------------------

edge_depth_compact_df = query_edge_depth_profile_df[
    [
        "query_name",
        "max_entity_depth",
        "max_edge_depth",
        "n_unique_edges",
        "edge_depths",
        "semantic_types",
        "relationship_names",
    ]
].copy()

edge_depth_compact_df = edge_depth_compact_df.rename(columns={
    "semantic_types": "structural_semantic_types",
    "relationship_names": "structural_relationship_names",
})

compact_matrix_by_query_df = compact_matrix_by_query_df.merge(
    edge_depth_compact_df,
    on="query_name",
    how="left",
)


# ---------------------------------------------------------
# 8. Add best depth and DeltaR variables
# ---------------------------------------------------------

best_depth_compact_df = best_depth_by_query_df[
    [
        "query_name",
        "best_document_depth",
        "Re_at_best_depth",
        "DeltaR_at_best_depth",
        "DeltaRratio_at_best_depth",
        "n_reachable_entities_at_best_depth",
        "reachable_entities_at_best_depth",
        "is_full_coverage_at_best_depth",
    ]
].copy()

compact_matrix_by_query_df = compact_matrix_by_query_df.merge(
    best_depth_compact_df,
    on="query_name",
    how="left",
)


# ---------------------------------------------------------
# 9. Add final-depth variables
# ---------------------------------------------------------
#
# This captures the values at the maximum tested depth per query.
# It is useful to compare the best depth with full-depth behavior.
# ---------------------------------------------------------

final_depth_rows = []

for query_name, group in re_delta_df.groupby("query_name"):
    group_sorted = group.sort_values("document_depth")
    final_row = group_sorted.iloc[-1].to_dict()

    final_depth_rows.append({
        "query_name": query_name,
        "final_tested_depth": final_row["document_depth"],
        "Re_at_final_depth": final_row["Re"],
        "DeltaR_at_final_depth": final_row["DeltaR"],
        "DeltaRratio_at_final_depth": final_row["DeltaRratio"],
        "is_full_coverage_at_final_depth": final_row["is_full_coverage"],
    })

final_depth_compact_df = pd.DataFrame(final_depth_rows)

compact_matrix_by_query_df = compact_matrix_by_query_df.merge(
    final_depth_compact_df,
    on="query_name",
    how="left",
)


# ---------------------------------------------------------
# 10. Add semantic profile variables
# ---------------------------------------------------------

semantic_profile_cols = [
    "query_name",
    "relationship_status",
    "n_relationship_edges",
    "semantic_types_touched",
    "n_semantic_types",
    "relationship_names_touched",
    "edge_ids_touched",
]

available_semantic_profile_cols = [
    col for col in semantic_profile_cols
    if col in query_semantic_profile_df.columns
]

semantic_compact_df = query_semantic_profile_df[
    available_semantic_profile_cols
].copy()

compact_matrix_by_query_df = compact_matrix_by_query_df.merge(
    semantic_compact_df,
    on="query_name",
    how="left",
)


# ---------------------------------------------------------
# 11. Add semantic flag variables
# ---------------------------------------------------------

semantic_flag_cols = [
    col for col in query_semantic_flags_df.columns
    if col == "query_name"
    or col.startswith("touches_")
    or col.startswith("n_")
]

semantic_flags_compact_df = query_semantic_flags_df[
    semantic_flag_cols
].copy()

compact_matrix_by_query_df = compact_matrix_by_query_df.merge(
    semantic_flags_compact_df,
    on="query_name",
    how="left",
)


# ---------------------------------------------------------
# 12. Add observed cardinality profile by query
# ---------------------------------------------------------
#
# We map each query's selected Rc edges to their observed cardinality.
# ---------------------------------------------------------

edge_to_cardinality_df = observed_cardinality_df[
    [
        "edge_id",
        "relationship_name",
        "observed_cardinality",
        "hint_confidence",
        "computation_status",
    ]
].copy()

if "rc_selected_edges_long_df" in globals() and not rc_selected_edges_long_df.empty:
    query_cardinality_edges_df = rc_selected_edges_long_df.merge(
        edge_to_cardinality_df,
        on="edge_id",
        how="left",
    )

    query_cardinality_profile_df = (
        query_cardinality_edges_df
        .groupby("query_name", dropna=False)
        .agg(
            observed_cardinalities_touched=(
                "observed_cardinality",
                lambda s: sorted(set(s.dropna())),
            ),
            hint_confidences_touched=(
                "hint_confidence",
                lambda s: sorted(set(s.dropna())),
            ),
            n_low_confidence_edges=(
                "hint_confidence",
                lambda s: int(s.isin(["low"]).sum()),
            ),
            n_medium_confidence_edges=(
                "hint_confidence",
                lambda s: int(s.isin(["medium"]).sum()),
            ),
            n_high_confidence_edges=(
                "hint_confidence",
                lambda s: int(s.isin(["high"]).sum()),
            ),
            n_no_match_edges=(
                "observed_cardinality",
                lambda s: int((s == "no_observed_matches").sum()),
            ),
        )
        .reset_index()
    )
else:
    query_cardinality_edges_df = pd.DataFrame()
    query_cardinality_profile_df = pd.DataFrame(
        columns=[
            "query_name",
            "observed_cardinalities_touched",
            "hint_confidences_touched",
            "n_low_confidence_edges",
            "n_medium_confidence_edges",
            "n_high_confidence_edges",
            "n_no_match_edges",
        ]
    )

compact_matrix_by_query_df = compact_matrix_by_query_df.merge(
    query_cardinality_profile_df,
    on="query_name",
    how="left",
)


# ---------------------------------------------------------
# 13. Fill missing values for local queries
# ---------------------------------------------------------

list_columns = [
    "Rc_selected_edges",
    "edge_depths",
    "structural_semantic_types",
    "structural_relationship_names",
    "reachable_entities_at_best_depth",
    "semantic_types_touched",
    "relationship_names_touched",
    "edge_ids_touched",
    "observed_cardinalities_touched",
    "hint_confidences_touched",
]

for col in list_columns:
    if col in compact_matrix_by_query_df.columns:
        compact_matrix_by_query_df[col] = compact_matrix_by_query_df[col].apply(
            lambda v: [] if isinstance(v, float) and pd.isna(v) else v
        )

numeric_fill_zero_cols = [
    "Rc",
    "n_unreachable_entities",
    "max_entity_depth",
    "max_edge_depth",
    "n_unique_edges",
    "best_document_depth",
    "Re_at_best_depth",
    "DeltaR_at_best_depth",
    "DeltaRratio_at_best_depth",
    "n_reachable_entities_at_best_depth",
    "final_tested_depth",
    "Re_at_final_depth",
    "DeltaR_at_final_depth",
    "DeltaRratio_at_final_depth",
    "n_relationship_edges",
    "n_semantic_types",
    "n_low_confidence_edges",
    "n_medium_confidence_edges",
    "n_high_confidence_edges",
    "n_no_match_edges",
]

for col in numeric_fill_zero_cols:
    if col in compact_matrix_by_query_df.columns:
        compact_matrix_by_query_df[col] = compact_matrix_by_query_df[col].fillna(0)


# ---------------------------------------------------------
# 14. Add compact interpretation columns
# ---------------------------------------------------------

compact_matrix_by_query_df["has_relationship_reduction_potential"] = (
    compact_matrix_by_query_df["DeltaR_at_best_depth"] > 0
)

compact_matrix_by_query_df["has_full_coverage_at_best_depth"] = (
    compact_matrix_by_query_df["is_full_coverage_at_best_depth"] == True
)

compact_matrix_by_query_df["has_low_confidence_mapping"] = (
    compact_matrix_by_query_df["n_low_confidence_edges"] > 0
)

compact_matrix_by_query_df["has_no_match_cardinality"] = (
    compact_matrix_by_query_df["n_no_match_edges"] > 0
)

compact_matrix_by_query_df["compact_matrix_status"] = "assembled"


# ---------------------------------------------------------
# 15. Compatibility aliases
# ---------------------------------------------------------

compact_query_matrix_df = compact_matrix_by_query_df.copy()
query_compact_matrix_df = compact_matrix_by_query_df.copy()

# Useful alias for later analytical matrix blocks.
compact_analytical_matrix_base_df = compact_matrix_by_query_df.copy()


# ---------------------------------------------------------
# 16. Validation tables
# ---------------------------------------------------------

compact_matrix_missing_root_df = compact_matrix_by_query_df[
    compact_matrix_by_query_df["selected_root"].isna()
].copy()

compact_matrix_missing_rc_df = compact_matrix_by_query_df[
    compact_matrix_by_query_df["Rc"].isna()
].copy()

compact_matrix_missing_delta_df = compact_matrix_by_query_df[
    compact_matrix_by_query_df["DeltaRratio_at_best_depth"].isna()
].copy()


# ---------------------------------------------------------
# 17. Display outputs
# ---------------------------------------------------------

print("Compact matrix by query assembled successfully.")

print("\nCompact matrix by query:")
display(compact_matrix_by_query_df)

print("\nQuery cardinality profile:")
display(query_cardinality_profile_df)

if not compact_matrix_missing_root_df.empty:
    print("\nWarning: some queries are missing selected_root.")
    display(compact_matrix_missing_root_df)

if not compact_matrix_missing_rc_df.empty:
    print("\nWarning: some queries are missing Rc.")
    display(compact_matrix_missing_rc_df)

if not compact_matrix_missing_delta_df.empty:
    print("\nWarning: some queries are missing DeltaRratio.")
    display(compact_matrix_missing_delta_df)


# ---------------------------------------------------------
# 18. Save BLOCK V11 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    compact_matrix_by_query_df,
    "variables/block_v11/compact_matrix_by_query.csv",
)

save_dataframe(
    compact_query_matrix_df,
    "variables/block_v11/compact_query_matrix.csv",
)

save_dataframe(
    query_compact_matrix_df,
    "variables/block_v11/query_compact_matrix.csv",
)

save_dataframe(
    compact_analytical_matrix_base_df,
    "variables/block_v11/compact_analytical_matrix_base.csv",
)

save_dataframe(
    query_cardinality_edges_df,
    "variables/block_v11/query_cardinality_edges.csv",
)

save_dataframe(
    query_cardinality_profile_df,
    "variables/block_v11/query_cardinality_profile.csv",
)

save_dataframe(
    compact_matrix_missing_root_df,
    "variables/block_v11/compact_matrix_missing_root.csv",
)

save_dataframe(
    compact_matrix_missing_rc_df,
    "variables/block_v11/compact_matrix_missing_rc.csv",
)

save_dataframe(
    compact_matrix_missing_delta_df,
    "variables/block_v11/compact_matrix_missing_delta.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "n_queries": len(compact_matrix_by_query_df),
        "n_columns": len(compact_matrix_by_query_df.columns),
        "columns": compact_matrix_by_query_df.columns.tolist(),
        "n_queries_with_reduction_potential": int(
            compact_matrix_by_query_df["has_relationship_reduction_potential"].sum()
        ),
        "n_queries_with_full_coverage_at_best_depth": int(
            compact_matrix_by_query_df["has_full_coverage_at_best_depth"].sum()
        ),
        "n_queries_with_low_confidence_mapping": int(
            compact_matrix_by_query_df["has_low_confidence_mapping"].sum()
        ),
        "n_queries_with_no_match_cardinality": int(
            compact_matrix_by_query_df["has_no_match_cardinality"].sum()
        ),
        "output_files": {
            "compact_matrix_by_query_csv": "variables/block_v11/compact_matrix_by_query.csv",
            "compact_query_matrix_csv": "variables/block_v11/compact_query_matrix.csv",
            "query_compact_matrix_csv": "variables/block_v11/query_compact_matrix.csv",
            "compact_analytical_matrix_base_csv": "variables/block_v11/compact_analytical_matrix_base.csv",
            "query_cardinality_edges_csv": "variables/block_v11/query_cardinality_edges.csv",
            "query_cardinality_profile_csv": "variables/block_v11/query_cardinality_profile.csv",
            "missing_root_csv": "variables/block_v11/compact_matrix_missing_root.csv",
            "missing_rc_csv": "variables/block_v11/compact_matrix_missing_rc.csv",
            "missing_delta_csv": "variables/block_v11/compact_matrix_missing_delta.csv",
        },
    },
    "variables/block_v11/compact_matrix_by_query_metadata.json",
)

write_execution_log(
    block_name="BLOCK V11 — ASSEMBLE COMPACT MATRIX BY QUERY",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_queries": len(compact_matrix_by_query_df),
        "n_columns": len(compact_matrix_by_query_df.columns),
        "n_queries_with_reduction_potential": int(
            compact_matrix_by_query_df["has_relationship_reduction_potential"].sum()
        ),
        "n_queries_with_full_coverage_at_best_depth": int(
            compact_matrix_by_query_df["has_full_coverage_at_best_depth"].sum()
        ),
        "compact_matrix_by_query_csv": "variables/block_v11/compact_matrix_by_query.csv",
        "query_cardinality_profile_csv": "variables/block_v11/query_cardinality_profile.csv",
        "metadata_json": "variables/block_v11/compact_matrix_by_query_metadata.json",
    },
)


# ---------------------------------------------------------
# 19. Update README section for BLOCK V11
# ---------------------------------------------------------

block_v11_readme_body = f"""
This block assembles a compact analytical matrix with one row per FIBEN
workload query.

Main variables created:

    compact_matrix_by_query_df
    compact_query_matrix_df
    query_compact_matrix_df
    compact_analytical_matrix_base_df

Additional variables created:

    query_cardinality_edges_df
    query_cardinality_profile_df
    compact_matrix_missing_root_df
    compact_matrix_missing_rc_df
    compact_matrix_missing_delta_df

Number of queries:

    {len(compact_matrix_by_query_df)}

Number of columns in the compact matrix:

    {len(compact_matrix_by_query_df.columns)}

Number of queries with relationship reduction potential:

    {int(compact_matrix_by_query_df["has_relationship_reduction_potential"].sum())}

Number of queries with full coverage at best depth:

    {int(compact_matrix_by_query_df["has_full_coverage_at_best_depth"].sum())}

Number of queries with low-confidence physical mapping:

    {int(compact_matrix_by_query_df["has_low_confidence_mapping"].sum())}

Number of queries with no-match cardinality edges:

    {int(compact_matrix_by_query_df["has_no_match_cardinality"].sum())}

Generated reproducibility files:

    variables/block_v11/compact_matrix_by_query.csv
    variables/block_v11/compact_query_matrix.csv
    variables/block_v11/query_compact_matrix.csv
    variables/block_v11/compact_analytical_matrix_base.csv
    variables/block_v11/query_cardinality_edges.csv
    variables/block_v11/query_cardinality_profile.csv
    variables/block_v11/compact_matrix_missing_root.csv
    variables/block_v11/compact_matrix_missing_rc.csv
    variables/block_v11/compact_matrix_missing_delta.csv
    variables/block_v11/compact_matrix_by_query_metadata.json

Methodological meaning:

    This block consolidates the main query-level methodology variables into
    a compact matrix.

    The matrix integrates:
    - workload metadata;
    - selected root;
    - Rc;
    - D;
    - Re;
    - DeltaR;
    - DeltaRratio;
    - semantic relationship profile;
    - observed cardinality profile.

This compact matrix is the basis for the next analytical matrix blocks.
"""

update_readme_section(
    section_title="BLOCK V11 — Assemble Compact Matrix by Query",
    section_body=block_v11_readme_body,
)

Compact matrix by query assembled successfully.

Compact matrix by query:


,query_name,generic_class,query_family,query_type,entities_touched,n_entities_touched,abstract_query,Rc,Rc_status,Rc_selected_edges,...,hint_confidences_touched,n_low_confidence_edges,n_medium_confidence_edges,n_high_confidence_edges,n_no_match_edges,has_relationship_reduction_potential,has_full_coverage_at_best_depth,has_low_confidence_mapping,has_no_match_cardinality,compact_matrix_status
0,Q1_CompanyProfileIBM,QG1,Local lookup,select,[Corporation],1,Show the profile of company IBM.,0,single_or_no_terminal,[],...,[],0.0,0.0,0.0,0.0,False,True,False,False,assembled
1,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,"[Corporation, Industry, Country, ListedSecurity]",4,"Show IBM with its industry, country, and liste...",3,connected,[Corporation -- Industry [corporation_has_indu...,...,"[high, medium]",0.0,1.0,2.0,1.0,True,True,False,True,assembled
2,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,"[FinancialServiceAccount, Holding, ListedSecur...",4,Show the securities held in each financial ser...,3,connected,[ListedSecurity -- Security [listed_security_r...,...,"[high, medium]",0.0,1.0,2.0,1.0,True,True,False,True,assembled
3,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,"[Person, FinancialServiceAccount, Holding, Lis...",5,Show the companies reached from a person throu...,4,connected,[Corporation -- ListedSecurity [corporation_ha...,...,"[high, medium]",0.0,1.0,3.0,1.0,True,True,False,True,assembled
4,Q5_ReportsAndMetricDataOfCompany,QG5,Containment retrieval,select,"[Corporation, FinancialReport, ReportElement, ...",5,Show the financial reports of a company and th...,4,connected,[Corporation -- FinancialReport [corporation_h...,...,"[high, low]",2.0,0.0,2.0,2.0,True,True,True,True,assembled
5,Q6_TechUSListedSecuritiesWithHighLastTradedValue,QG6,Filtered search / recommendation,select,"[Corporation, Industry, Country, ListedSecurity]",4,Find listed securities of technology companies...,3,connected,[Corporation -- Industry [corporation_has_indu...,...,"[high, medium]",0.0,1.0,2.0,1.0,True,True,False,True,assembled
6,Q7_PersonsWhoBoughtMoreIBMThanSold,QG7,Aggregation / ranking,select,"[Person, FinancialServiceAccount, BuyTransacti...",7,Who bought more IBM stocks than they sold?,6,connected,[Corporation -- ListedSecurity [corporation_ha...,...,"[high, medium]",0.0,3.0,3.0,1.0,True,True,False,True,assembled
7,Q8_IBMTransactionsBelowAverageSellingPrice,QG8,Aggregation / ranking,select,"[Transaction, SellTransaction, ListedSecurity,...",4,Show me each transaction for IBM whose price i...,3,connected,[Corporation -- ListedSecurity [corporation_ha...,...,"[high, medium]",0.0,2.0,1.0,1.0,True,True,False,True,assembled
8,Q9_PersonsWhoBoughtAndSoldSameStock,QG9,Associative retrieval + comparison,select,"[Person, FinancialServiceAccount, BuyTransacti...",6,Show me everyone who bought and sold the same ...,5,connected,[Person -- FinancialServiceAccount [person_own...,...,"[high, medium]",0.0,2.0,3.0,0.0,True,True,False,False,assembled
9,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,"[Person, FinancialServiceAccount, Holding, Lis...",6,Create a financial service account for a perso...,5,connected,[Person -- FinancialServiceAccount [person_own...,...,"[high, medium]",0.0,1.0,4.0,0.0,True,True,False,False,assembled



Query cardinality profile:


,query_name,observed_cardinalities_touched,hint_confidences_touched,n_low_confidence_edges,n_medium_confidence_edges,n_high_confidence_edges,n_no_match_edges
0,Q10_CreateAccountHoldingAndBuyTransaction,"[observed_1_to_1, observed_N_to_M]","[high, medium]",0,1,4,0
1,Q2_CompanyWithIndustryCountryAndListedSecurities,"[no_observed_matches, observed_N_to_M]","[high, medium]",0,1,2,1
2,Q3_SecuritiesHeldInEachFinancialServiceAccount,"[no_observed_matches, observed_N_to_M]","[high, medium]",0,1,2,1
3,Q4_CompaniesReachedFromPersonThroughAccountHol...,"[no_observed_matches, observed_N_to_M]","[high, medium]",0,1,3,1
4,Q5_ReportsAndMetricDataOfCompany,"[no_observed_matches, observed_N_to_M]","[high, low]",2,0,2,2
5,Q6_TechUSListedSecuritiesWithHighLastTradedValue,"[no_observed_matches, observed_N_to_M]","[high, medium]",0,1,2,1
6,Q7_PersonsWhoBoughtMoreIBMThanSold,"[no_observed_matches, observed_1_to_1, observe...","[high, medium]",0,3,3,1
7,Q8_IBMTransactionsBelowAverageSellingPrice,"[no_observed_matches, observed_1_to_1, observe...","[high, medium]",0,2,1,1
8,Q9_PersonsWhoBoughtAndSoldSameStock,"[observed_1_to_1, observed_N_to_M]","[high, medium]",0,2,3,0


Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v11/compact_matrix_by_query.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v11/compact_query_matrix.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v11/query_compact_matrix.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v11/compact_analytical_matrix_base.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v11/query_cardinality_edges.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v11/query_cardinality_profile.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v11/com

In [80]:
# =========================================================
# BLOCK V12 — ANALYTICAL MATRIX OF FIBEN EXPANDED BY DEPTH
# =========================================================
#
# Purpose:
# This block builds the expanded analytical matrix for the
# FIBEN workload.
#
# Unlike BLOCK V11, which has one row per query, this block has
# one row per:
#
#     query + document_depth
#
# This expanded matrix is important because the methodology needs
# to compare the effect of different document embedding depths.
#
# Main outputs preserved for downstream compatibility:
# - analytical_matrix_expanded_by_depth_df
# - expanded_analytical_matrix_df
# - document_analytical_matrix_expanded_df
#
# Methodological role:
# The expanded matrix shows how Re, DeltaR, and DeltaRratio change
# as document depth increases.
# =========================================================

import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "re_delta_df",
    "compact_matrix_by_query_df",
    "query_semantic_flags_df",
    "query_semantic_profile_df",
    "best_depth_by_query_df",
    "fiben_scenario",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first BLOCKS V8, V10, and V11."
        )


# ---------------------------------------------------------
# 2. Start from Re/DeltaR by query and depth
# ---------------------------------------------------------
#
# re_delta_df already contains:
# - query_name;
# - selected_root;
# - document_depth;
# - Rc;
# - Re;
# - DeltaR;
# - DeltaRratio;
# - reachable entities;
# - covered and remaining edges.
# ---------------------------------------------------------

analytical_matrix_expanded_by_depth_df = re_delta_df.copy()


# ---------------------------------------------------------
# 3. Add compact query-level variables from BLOCK V11
# ---------------------------------------------------------
#
# We avoid duplicating depth-dependent variables from V10.
# The compact matrix contributes stable query-level metadata.
# ---------------------------------------------------------

query_level_cols = [
    "query_name",
    "abstract_query",
    "entities_touched",
    "n_entities_touched",
    "selection_strategy",
    "selection_reason",
    "max_entity_distance_D",
    "min_entity_distance_D",
    "avg_entity_distance_D",
    "n_unreachable_entities",
    "max_entity_depth",
    "max_edge_depth",
    "n_unique_edges",
    "relationship_status",
    "n_relationship_edges",
    "semantic_types_touched",
    "n_semantic_types",
    "relationship_names_touched",
    "observed_cardinalities_touched",
    "hint_confidences_touched",
    "n_low_confidence_edges",
    "n_medium_confidence_edges",
    "n_high_confidence_edges",
    "n_no_match_edges",
    "has_relationship_reduction_potential",
    "has_low_confidence_mapping",
    "has_no_match_cardinality",
]

available_query_level_cols = [
    col for col in query_level_cols
    if col in compact_matrix_by_query_df.columns
]

query_level_df = compact_matrix_by_query_df[
    available_query_level_cols
].copy()

analytical_matrix_expanded_by_depth_df = (
    analytical_matrix_expanded_by_depth_df
    .merge(
        query_level_df,
        on="query_name",
        how="left",
    )
)


# ---------------------------------------------------------
# 4. Add semantic flags from BLOCK V8
# ---------------------------------------------------------

semantic_flag_cols = [
    col for col in query_semantic_flags_df.columns
    if col == "query_name"
    or col.startswith("touches_")
    or col.startswith("n_")
]

semantic_flags_df = query_semantic_flags_df[
    semantic_flag_cols
].copy()

# Avoid duplicate generic metadata columns if present.
semantic_flags_df = semantic_flags_df.drop(
    columns=[
        col for col in ["generic_class", "query_family"]
        if col in semantic_flags_df.columns
    ],
    errors="ignore",
)

analytical_matrix_expanded_by_depth_df = (
    analytical_matrix_expanded_by_depth_df
    .merge(
        semantic_flags_df,
        on="query_name",
        how="left",
    )
)


# ---------------------------------------------------------
# 5. Add best-depth markers
# ---------------------------------------------------------

best_depth_marker_df = best_depth_by_query_df[
    [
        "query_name",
        "best_document_depth",
        "DeltaRratio_at_best_depth",
        "is_full_coverage_at_best_depth",
    ]
].copy()

analytical_matrix_expanded_by_depth_df = (
    analytical_matrix_expanded_by_depth_df
    .merge(
        best_depth_marker_df,
        on="query_name",
        how="left",
    )
)

analytical_matrix_expanded_by_depth_df["is_best_depth"] = (
    analytical_matrix_expanded_by_depth_df["document_depth"]
    == analytical_matrix_expanded_by_depth_df["best_document_depth"]
)


# ---------------------------------------------------------
# 6. Add normalized depth variables
# ---------------------------------------------------------

def safe_depth_ratio(document_depth, max_depth):
    """
    Compute document_depth / max_depth safely.

    If max_depth is zero, return 0.0.
    """
    if max_depth is None:
        return None

    try:
        if pd.isna(max_depth):
            return None
    except Exception:
        pass

    if max_depth == 0:
        return 0.0

    return float(document_depth) / float(max_depth)


analytical_matrix_expanded_by_depth_df["document_depth_ratio"] = (
    analytical_matrix_expanded_by_depth_df
    .apply(
        lambda row: safe_depth_ratio(
            row["document_depth"],
            row["max_depth_for_query"],
        ),
        axis=1,
    )
)


# ---------------------------------------------------------
# 7. Add interpretation labels
# ---------------------------------------------------------

def classify_depth_effect(row):
    """
    Classify the effect of a document depth based on DeltaRratio.
    """
    rc = row.get("Rc", 0)
    ratio = row.get("DeltaRratio", 0)

    if rc == 0:
        return "no_relationships_to_reduce"

    if ratio == 0:
        return "no_reduction"

    if ratio < 0.5:
        return "partial_reduction_low"

    if ratio < 1.0:
        return "partial_reduction_high"

    return "full_reduction"


def classify_depth_position(row):
    """
    Classify the document depth position inside the query depth range.
    """
    depth = row.get("document_depth", 0)
    max_depth = row.get("max_depth_for_query", 0)

    if max_depth == 0:
        return "single_entity_or_depth_zero"

    if depth == 0:
        return "root_only"

    if depth < max_depth:
        return "intermediate_depth"

    return "maximum_depth"


analytical_matrix_expanded_by_depth_df["depth_effect_label"] = (
    analytical_matrix_expanded_by_depth_df
    .apply(classify_depth_effect, axis=1)
)

analytical_matrix_expanded_by_depth_df["depth_position_label"] = (
    analytical_matrix_expanded_by_depth_df
    .apply(classify_depth_position, axis=1)
)


# ---------------------------------------------------------
# 8. Reorder important columns
# ---------------------------------------------------------

preferred_column_order = [
    "query_name",
    "generic_class",
    "query_family",
    "query_type",
    "abstract_query",
    "selected_root",
    "document_depth",
    "max_depth_for_query",
    "document_depth_ratio",
    "best_document_depth",
    "is_best_depth",
    "depth_position_label",
    "depth_effect_label",
    "Rc",
    "Re",
    "DeltaR",
    "DeltaRratio",
    "n_reachable_entities",
    "reachable_entities",
    "n_total_edges",
    "n_covered_edges",
    "n_remaining_edges",
    "is_full_coverage",
    "covered_relationships",
    "remaining_relationships",
    "covered_semantic_types",
    "remaining_semantic_types",
    "entities_touched",
    "n_entities_touched",
    "semantic_types_touched",
    "observed_cardinalities_touched",
    "hint_confidences_touched",
    "has_low_confidence_mapping",
    "has_no_match_cardinality",
]

existing_preferred_columns = [
    col for col in preferred_column_order
    if col in analytical_matrix_expanded_by_depth_df.columns
]

remaining_columns = [
    col for col in analytical_matrix_expanded_by_depth_df.columns
    if col not in existing_preferred_columns
]

analytical_matrix_expanded_by_depth_df = analytical_matrix_expanded_by_depth_df[
    existing_preferred_columns + remaining_columns
].copy()


# ---------------------------------------------------------
# 9. Compatibility aliases
# ---------------------------------------------------------

expanded_analytical_matrix_df = analytical_matrix_expanded_by_depth_df.copy()
document_analytical_matrix_expanded_df = analytical_matrix_expanded_by_depth_df.copy()

# Useful alias for later blocks.
document_variable_matrix_expanded_df = analytical_matrix_expanded_by_depth_df.copy()


# ---------------------------------------------------------
# 10. Summary tables
# ---------------------------------------------------------

expanded_matrix_summary_by_depth_df = (
    analytical_matrix_expanded_by_depth_df
    .groupby("document_depth", dropna=False)
    .agg(
        n_rows=("query_name", "count"),
        n_queries=("query_name", "nunique"),
        avg_Rc=("Rc", "mean"),
        avg_Re=("Re", "mean"),
        avg_DeltaR=("DeltaR", "mean"),
        avg_DeltaRratio=("DeltaRratio", "mean"),
        n_full_coverage=("is_full_coverage", lambda s: int(s.sum())),
        n_best_depth_rows=("is_best_depth", lambda s: int(s.sum())),
    )
    .reset_index()
    .sort_values("document_depth")
    .reset_index(drop=True)
)

expanded_matrix_summary_by_query_df = (
    analytical_matrix_expanded_by_depth_df
    .groupby(
        ["query_name", "generic_class", "query_family", "selected_root"],
        dropna=False,
    )
    .agg(
        n_depth_rows=("document_depth", "count"),
        max_depth=("document_depth", "max"),
        max_DeltaRratio=("DeltaRratio", "max"),
        max_DeltaR=("DeltaR", "max"),
        min_Re=("Re", "min"),
        has_full_coverage=("is_full_coverage", lambda s: bool(s.any())),
        best_depth=("best_document_depth", "first"),
    )
    .reset_index()
    .sort_values("query_name")
    .reset_index(drop=True)
)

expanded_matrix_best_depth_rows_df = (
    analytical_matrix_expanded_by_depth_df[
        analytical_matrix_expanded_by_depth_df["is_best_depth"] == True
    ]
    .copy()
    .sort_values(["query_name", "document_depth"])
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 11. Validation tables
# ---------------------------------------------------------

expanded_matrix_missing_values_df = (
    analytical_matrix_expanded_by_depth_df[
        analytical_matrix_expanded_by_depth_df[
            ["Rc", "Re", "DeltaR", "DeltaRratio"]
        ].isna().any(axis=1)
    ]
    .copy()
)

expanded_matrix_negative_delta_df = (
    analytical_matrix_expanded_by_depth_df[
        analytical_matrix_expanded_by_depth_df["DeltaR"] < 0
    ]
    .copy()
)

expanded_matrix_invalid_best_depth_df = (
    analytical_matrix_expanded_by_depth_df[
        analytical_matrix_expanded_by_depth_df["is_best_depth"].isna()
    ]
    .copy()
)


# ---------------------------------------------------------
# 12. Display outputs
# ---------------------------------------------------------

print("Expanded analytical matrix by depth assembled successfully.")

print("\nExpanded analytical matrix:")
display(analytical_matrix_expanded_by_depth_df)

print("\nSummary by document depth:")
display(expanded_matrix_summary_by_depth_df)

print("\nSummary by query:")
display(expanded_matrix_summary_by_query_df)

print("\nBest-depth rows:")
display(expanded_matrix_best_depth_rows_df)

if not expanded_matrix_missing_values_df.empty:
    print("\nWarning: some rows have missing analytical values.")
    display(expanded_matrix_missing_values_df)

if not expanded_matrix_negative_delta_df.empty:
    print("\nWarning: some rows have negative DeltaR.")
    display(expanded_matrix_negative_delta_df)

if expanded_matrix_missing_values_df.empty and expanded_matrix_negative_delta_df.empty:
    print("\nExpanded matrix validation passed.")


# ---------------------------------------------------------
# 13. Save BLOCK V12 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    analytical_matrix_expanded_by_depth_df,
    "variables/block_v12/analytical_matrix_expanded_by_depth.csv",
)

save_dataframe(
    expanded_analytical_matrix_df,
    "variables/block_v12/expanded_analytical_matrix.csv",
)

save_dataframe(
    document_analytical_matrix_expanded_df,
    "variables/block_v12/document_analytical_matrix_expanded.csv",
)

save_dataframe(
    document_variable_matrix_expanded_df,
    "variables/block_v12/document_variable_matrix_expanded.csv",
)

save_dataframe(
    expanded_matrix_summary_by_depth_df,
    "variables/block_v12/expanded_matrix_summary_by_depth.csv",
)

save_dataframe(
    expanded_matrix_summary_by_query_df,
    "variables/block_v12/expanded_matrix_summary_by_query.csv",
)

save_dataframe(
    expanded_matrix_best_depth_rows_df,
    "variables/block_v12/expanded_matrix_best_depth_rows.csv",
)

save_dataframe(
    expanded_matrix_missing_values_df,
    "variables/block_v12/expanded_matrix_missing_values.csv",
)

save_dataframe(
    expanded_matrix_negative_delta_df,
    "variables/block_v12/expanded_matrix_negative_delta.csv",
)

save_dataframe(
    expanded_matrix_invalid_best_depth_df,
    "variables/block_v12/expanded_matrix_invalid_best_depth.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "n_rows": len(analytical_matrix_expanded_by_depth_df),
        "n_queries": int(analytical_matrix_expanded_by_depth_df["query_name"].nunique()),
        "n_columns": len(analytical_matrix_expanded_by_depth_df.columns),
        "columns": analytical_matrix_expanded_by_depth_df.columns.tolist(),
        "document_depths": sorted(
            analytical_matrix_expanded_by_depth_df["document_depth"]
            .dropna()
            .unique()
            .tolist()
        ),
        "n_best_depth_rows": int(
            analytical_matrix_expanded_by_depth_df["is_best_depth"].sum()
        ),
        "n_full_coverage_rows": int(
            analytical_matrix_expanded_by_depth_df["is_full_coverage"].sum()
        ),
        "n_missing_value_rows": len(expanded_matrix_missing_values_df),
        "n_negative_delta_rows": len(expanded_matrix_negative_delta_df),
        "output_files": {
            "analytical_matrix_expanded_csv": "variables/block_v12/analytical_matrix_expanded_by_depth.csv",
            "expanded_analytical_matrix_csv": "variables/block_v12/expanded_analytical_matrix.csv",
            "document_analytical_matrix_expanded_csv": "variables/block_v12/document_analytical_matrix_expanded.csv",
            "document_variable_matrix_expanded_csv": "variables/block_v12/document_variable_matrix_expanded.csv",
            "summary_by_depth_csv": "variables/block_v12/expanded_matrix_summary_by_depth.csv",
            "summary_by_query_csv": "variables/block_v12/expanded_matrix_summary_by_query.csv",
            "best_depth_rows_csv": "variables/block_v12/expanded_matrix_best_depth_rows.csv",
            "missing_values_csv": "variables/block_v12/expanded_matrix_missing_values.csv",
            "negative_delta_csv": "variables/block_v12/expanded_matrix_negative_delta.csv",
            "invalid_best_depth_csv": "variables/block_v12/expanded_matrix_invalid_best_depth.csv",
        },
    },
    "variables/block_v12/analytical_matrix_expanded_metadata.json",
)

write_execution_log(
    block_name="BLOCK V12 — ANALYTICAL MATRIX OF FIBEN EXPANDED BY DEPTH",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_rows": len(analytical_matrix_expanded_by_depth_df),
        "n_queries": int(analytical_matrix_expanded_by_depth_df["query_name"].nunique()),
        "n_columns": len(analytical_matrix_expanded_by_depth_df.columns),
        "n_best_depth_rows": int(
            analytical_matrix_expanded_by_depth_df["is_best_depth"].sum()
        ),
        "n_full_coverage_rows": int(
            analytical_matrix_expanded_by_depth_df["is_full_coverage"].sum()
        ),
        "n_missing_value_rows": len(expanded_matrix_missing_values_df),
        "n_negative_delta_rows": len(expanded_matrix_negative_delta_df),
        "analytical_matrix_expanded_csv": "variables/block_v12/analytical_matrix_expanded_by_depth.csv",
        "summary_by_depth_csv": "variables/block_v12/expanded_matrix_summary_by_depth.csv",
        "summary_by_query_csv": "variables/block_v12/expanded_matrix_summary_by_query.csv",
        "metadata_json": "variables/block_v12/analytical_matrix_expanded_metadata.json",
    },
)


# ---------------------------------------------------------
# 14. Update README section for BLOCK V12
# ---------------------------------------------------------

block_v12_readme_body = f"""
This block builds the expanded analytical matrix for the FIBEN workload.

Unlike the compact matrix from BLOCK V11, this matrix has one row per:

    query + document_depth

Main variables created:

    analytical_matrix_expanded_by_depth_df
    expanded_analytical_matrix_df
    document_analytical_matrix_expanded_df
    document_variable_matrix_expanded_df

Additional variables created:

    expanded_matrix_summary_by_depth_df
    expanded_matrix_summary_by_query_df
    expanded_matrix_best_depth_rows_df
    expanded_matrix_missing_values_df
    expanded_matrix_negative_delta_df
    expanded_matrix_invalid_best_depth_df

Number of rows:

    {len(analytical_matrix_expanded_by_depth_df)}

Number of queries:

    {int(analytical_matrix_expanded_by_depth_df["query_name"].nunique())}

Number of columns:

    {len(analytical_matrix_expanded_by_depth_df.columns)}

Document depths represented:

    {sorted(analytical_matrix_expanded_by_depth_df["document_depth"].dropna().unique().tolist())}

Number of best-depth rows:

    {int(analytical_matrix_expanded_by_depth_df["is_best_depth"].sum())}

Number of full-coverage rows:

    {int(analytical_matrix_expanded_by_depth_df["is_full_coverage"].sum())}

Generated reproducibility files:

    variables/block_v12/analytical_matrix_expanded_by_depth.csv
    variables/block_v12/expanded_analytical_matrix.csv
    variables/block_v12/document_analytical_matrix_expanded.csv
    variables/block_v12/document_variable_matrix_expanded.csv
    variables/block_v12/expanded_matrix_summary_by_depth.csv
    variables/block_v12/expanded_matrix_summary_by_query.csv
    variables/block_v12/expanded_matrix_best_depth_rows.csv
    variables/block_v12/expanded_matrix_missing_values.csv
    variables/block_v12/expanded_matrix_negative_delta.csv
    variables/block_v12/expanded_matrix_invalid_best_depth.csv
    variables/block_v12/analytical_matrix_expanded_metadata.json

Methodological meaning:

    This block shows how the analytical variables change when the document
    embedding depth changes.

    It allows comparing Re, DeltaR, and DeltaRratio across depth values for
    each query.

Why this is important:

    The activation logic later uses these variables to infer which generic
    document configuration families are relevant for each query.
"""

update_readme_section(
    section_title="BLOCK V12 — Analytical Matrix of FIBEN Expanded by Depth",
    section_body=block_v12_readme_body,
)

Expanded analytical matrix by depth assembled successfully.

Expanded analytical matrix:


,query_name,generic_class,query_family,query_type,abstract_query,selected_root,document_depth,max_depth_for_query,document_depth_ratio,best_document_depth,...,touches_descriptor,touches_ownership,touches_subtype,n_association_edges,n_containment_edges,n_descriptor_edges,n_ownership_edges,n_subtype_edges,DeltaRratio_at_best_depth,is_full_coverage_at_best_depth
0,Q1_CompanyProfileIBM,QG1,Local lookup,select,Show the profile of company IBM.,Corporation,0,0,0.000000,0,...,False,False,False,0,0,0,0,0,0.0,True
1,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,"Show IBM with its industry, country, and liste...",Corporation,0,1,0.000000,1,...,True,False,False,1,0,2,0,0,1.0,True
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,"Show IBM with its industry, country, and liste...",Corporation,1,1,1.000000,1,...,True,False,False,1,0,2,0,0,1.0,True
3,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,Show the securities held in each financial ser...,FinancialServiceAccount,0,3,0.000000,3,...,False,False,False,2,1,0,0,0,1.0,True
4,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,Show the securities held in each financial ser...,FinancialServiceAccount,1,3,0.333333,3,...,False,False,False,2,1,0,0,0,1.0,True
5,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,Show the securities held in each financial ser...,FinancialServiceAccount,2,3,0.666667,3,...,False,False,False,2,1,0,0,0,1.0,True
6,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,Show the securities held in each financial ser...,FinancialServiceAccount,3,3,1.000000,3,...,False,False,False,2,1,0,0,0,1.0,True
7,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Show the companies reached from a person throu...,Person,0,4,0.000000,4,...,False,True,False,2,1,0,1,0,1.0,True
8,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Show the companies reached from a person throu...,Person,1,4,0.250000,4,...,False,True,False,2,1,0,1,0,1.0,True
9,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Show the companies reached from a person throu...,Person,2,4,0.500000,4,...,False,True,False,2,1,0,1,0,1.0,True



Summary by document depth:


,document_depth,n_rows,n_queries,avg_Rc,avg_Re,avg_DeltaR,avg_DeltaRratio,n_full_coverage,n_best_depth_rows
0,0,10,10,3.600,3.800000,-0.200000,-0.036667,1,1
1,1,9,9,4.000,2.888889,1.111111,0.337037,1,1
2,2,8,8,4.125,1.625000,2.500000,0.656250,2,2
3,3,6,6,4.500,0.333333,4.166667,0.930556,4,4
4,4,2,2,5.000,0.000000,5.000000,1.000000,2,2



Summary by query:


,query_name,generic_class,query_family,selected_root,n_depth_rows,max_depth,max_DeltaRratio,max_DeltaR,min_Re,has_full_coverage,best_depth
0,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,Person,4,3,1.0,5,0,True,3
1,Q1_CompanyProfileIBM,QG1,Local lookup,Corporation,1,0,0.0,0,0,True,0
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,Corporation,2,1,1.0,3,0,True,1
3,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,FinancialServiceAccount,4,3,1.0,3,0,True,3
4,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,Person,5,4,1.0,4,0,True,4
5,Q5_ReportsAndMetricDataOfCompany,QG5,Containment retrieval,Corporation,4,3,1.0,4,0,True,3
6,Q6_TechUSListedSecuritiesWithHighLastTradedValue,QG6,Filtered search / recommendation,ListedSecurity,3,2,1.0,3,0,True,2
7,Q7_PersonsWhoBoughtMoreIBMThanSold,QG7,Aggregation / ranking,Person,5,4,1.0,6,0,True,4
8,Q8_IBMTransactionsBelowAverageSellingPrice,QG8,Aggregation / ranking,Transaction,3,2,1.0,3,0,True,2
9,Q9_PersonsWhoBoughtAndSoldSameStock,QG9,Associative retrieval + comparison,Person,4,3,1.0,5,0,True,3



Best-depth rows:


,query_name,generic_class,query_family,query_type,abstract_query,selected_root,document_depth,max_depth_for_query,document_depth_ratio,best_document_depth,...,touches_descriptor,touches_ownership,touches_subtype,n_association_edges,n_containment_edges,n_descriptor_edges,n_ownership_edges,n_subtype_edges,DeltaRratio_at_best_depth,is_full_coverage_at_best_depth
0,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,Create a financial service account for a perso...,Person,3,3,1.0,3,...,False,True,True,1,2,0,1,1,1.0,True
1,Q1_CompanyProfileIBM,QG1,Local lookup,select,Show the profile of company IBM.,Corporation,0,0,0.0,0,...,False,False,False,0,0,0,0,0,0.0,True
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,"Show IBM with its industry, country, and liste...",Corporation,1,1,1.0,1,...,True,False,False,1,0,2,0,0,1.0,True
3,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,Show the securities held in each financial ser...,FinancialServiceAccount,3,3,1.0,3,...,False,False,False,2,1,0,0,0,1.0,True
4,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Show the companies reached from a person throu...,Person,4,4,1.0,4,...,False,True,False,2,1,0,1,0,1.0,True
5,Q5_ReportsAndMetricDataOfCompany,QG5,Containment retrieval,select,Show the financial reports of a company and th...,Corporation,3,3,1.0,3,...,False,False,False,0,4,0,0,0,1.0,True
6,Q6_TechUSListedSecuritiesWithHighLastTradedValue,QG6,Filtered search / recommendation,select,Find listed securities of technology companies...,ListedSecurity,2,2,1.0,2,...,True,False,False,1,0,2,0,0,1.0,True
7,Q7_PersonsWhoBoughtMoreIBMThanSold,QG7,Aggregation / ranking,select,Who bought more IBM stocks than they sold?,Person,4,4,1.0,4,...,False,True,True,2,1,0,1,2,1.0,True
8,Q8_IBMTransactionsBelowAverageSellingPrice,QG8,Aggregation / ranking,select,Show me each transaction for IBM whose price i...,Transaction,2,2,1.0,2,...,False,False,True,2,0,0,0,1,1.0,True
9,Q9_PersonsWhoBoughtAndSoldSameStock,QG9,Associative retrieval + comparison,select,Show me everyone who bought and sold the same ...,Person,3,3,1.0,3,...,False,True,True,1,1,0,1,2,1.0,True


,query_name,generic_class,query_family,query_type,abstract_query,selected_root,document_depth,max_depth_for_query,document_depth_ratio,best_document_depth,...,touches_descriptor,touches_ownership,touches_subtype,n_association_edges,n_containment_edges,n_descriptor_edges,n_ownership_edges,n_subtype_edges,DeltaRratio_at_best_depth,is_full_coverage_at_best_depth
19,Q7_PersonsWhoBoughtMoreIBMThanSold,QG7,Aggregation / ranking,select,Who bought more IBM stocks than they sold?,Person,0,4,0.0,4,...,False,True,True,2,1,0,1,2,1.0,True
27,Q9_PersonsWhoBoughtAndSoldSameStock,QG9,Associative retrieval + comparison,select,Show me everyone who bought and sold the same ...,Person,0,3,0.0,3,...,False,True,True,1,1,0,1,2,1.0,True


Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v12/analytical_matrix_expanded_by_depth.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v12/expanded_analytical_matrix.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v12/document_analytical_matrix_expanded.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v12/document_variable_matrix_expanded.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v12/expanded_matrix_summary_by_depth.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v12/expanded_matrix_summary_by_query.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/

In [81]:
# =========================================================
# BLOCK V13 — COMPACT ANALYTICAL MATRIX BY QUERY
# =========================================================
#
# Purpose:
# This block builds the compact analytical matrix for the
# FIBEN workload.
#
# The previous block created an expanded matrix with one row per:
#
#     query + document_depth
#
# This block selects one representative row per query, normally the
# best-depth row, and produces a compact query-level matrix.
#
# Main outputs preserved for downstream compatibility:
# - analytical_matrix_compact_by_query_df
# - compact_analytical_matrix_df
# - document_variable_matrix_df
#
# Additional outputs:
# - compact_best_depth_rows_df
# - compact_matrix_summary_df
# - compact_matrix_validation_df
#
# Methodological role:
# This matrix is the main query-level analytical table used by
# the next methodology blocks.
#
# Later blocks will add:
# - explicitly modified entities;
# - update volatility;
# - observed sharedness;
# - final activation variables.
# =========================================================

import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "analytical_matrix_expanded_by_depth_df",
    "compact_matrix_by_query_df",
    "expanded_matrix_best_depth_rows_df",
    "fiben_scenario",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first BLOCKS V11 and V12."
        )


# ---------------------------------------------------------
# 2. Select one best-depth row per query
# ---------------------------------------------------------
#
# Usually there is exactly one best-depth row per query.
# If there are multiple rows marked as best depth, we keep the
# smallest document_depth.
# ---------------------------------------------------------

if expanded_matrix_best_depth_rows_df.empty:
    raise ValueError(
        "expanded_matrix_best_depth_rows_df is empty. "
        "Run BLOCK V12 and check whether is_best_depth was computed correctly."
    )

compact_best_depth_rows_df = (
    expanded_matrix_best_depth_rows_df
    .sort_values(["query_name", "document_depth"])
    .groupby("query_name", as_index=False)
    .head(1)
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 3. Define the core compact analytical columns
# ---------------------------------------------------------
#
# These variables are central to the methodology.
# We keep both document_depth and best_document_depth for
# downstream compatibility.
# ---------------------------------------------------------

core_columns = [
    "query_name",
    "generic_class",
    "query_family",
    "query_type",
    "abstract_query",
    "selected_root",
    "document_depth",
    "best_document_depth",
    "max_depth_for_query",
    "document_depth_ratio",
    "depth_position_label",
    "depth_effect_label",
    "Rc",
    "Re",
    "DeltaR",
    "DeltaRratio",
    "n_reachable_entities",
    "reachable_entities",
    "n_total_edges",
    "n_covered_edges",
    "n_remaining_edges",
    "is_full_coverage",
    "covered_relationships",
    "remaining_relationships",
    "covered_semantic_types",
    "remaining_semantic_types",
    "entities_touched",
    "n_entities_touched",
    "semantic_types_touched",
    "observed_cardinalities_touched",
    "hint_confidences_touched",
    "has_low_confidence_mapping",
    "has_no_match_cardinality",
]

available_core_columns = [
    col for col in core_columns
    if col in compact_best_depth_rows_df.columns
]

analytical_matrix_compact_by_query_df = compact_best_depth_rows_df[
    available_core_columns
].copy()


# ---------------------------------------------------------
# 4. Add semantic flags
# ---------------------------------------------------------

semantic_flag_columns = [
    col for col in compact_best_depth_rows_df.columns
    if col.startswith("touches_") or col.startswith("n_")
]

# Avoid duplicating already selected columns.
semantic_flag_columns = [
    col for col in semantic_flag_columns
    if col not in analytical_matrix_compact_by_query_df.columns
]

analytical_matrix_compact_by_query_df = pd.concat(
    [
        analytical_matrix_compact_by_query_df,
        compact_best_depth_rows_df[semantic_flag_columns].copy(),
    ],
    axis=1,
)


# ---------------------------------------------------------
# 5. Add query-level fields from BLOCK V11 when useful
# ---------------------------------------------------------
#
# Some variables are easier to preserve from the compact matrix
# assembled in BLOCK V11.
# ---------------------------------------------------------

additional_query_level_columns = [
    "selection_strategy",
    "selection_reason",
    "selected_root_is_candidate",
    "selected_root_is_touched",
    "max_entity_distance_D",
    "min_entity_distance_D",
    "avg_entity_distance_D",
    "n_unreachable_entities",
    "max_entity_depth",
    "max_edge_depth",
    "n_unique_edges",
    "relationship_status",
    "n_relationship_edges",
    "n_semantic_types",
    "relationship_names_touched",
    "n_low_confidence_edges",
    "n_medium_confidence_edges",
    "n_high_confidence_edges",
    "n_no_match_edges",
    "has_relationship_reduction_potential",
    "has_full_coverage_at_best_depth",
]

available_additional_columns = [
    col for col in additional_query_level_columns
    if col in compact_matrix_by_query_df.columns
]

query_level_additions_df = compact_matrix_by_query_df[
    ["query_name"] + available_additional_columns
].copy()

# Avoid duplicate columns if already present.
duplicate_cols = [
    col for col in available_additional_columns
    if col in analytical_matrix_compact_by_query_df.columns
]

query_level_additions_df = query_level_additions_df.drop(
    columns=duplicate_cols,
    errors="ignore",
)

analytical_matrix_compact_by_query_df = analytical_matrix_compact_by_query_df.merge(
    query_level_additions_df,
    on="query_name",
    how="left",
)


# ---------------------------------------------------------
# 6. Create normalized aliases and interpretation variables
# ---------------------------------------------------------

analytical_matrix_compact_by_query_df["selected_document_depth"] = (
    analytical_matrix_compact_by_query_df["document_depth"]
)

analytical_matrix_compact_by_query_df["best_depth_is_full_coverage"] = (
    analytical_matrix_compact_by_query_df["is_full_coverage"] == True
)

analytical_matrix_compact_by_query_df["relationship_reduction_potential"] = (
    analytical_matrix_compact_by_query_df["DeltaR"] > 0
)

analytical_matrix_compact_by_query_df["relationship_reduction_strength"] = (
    analytical_matrix_compact_by_query_df["DeltaRratio"]
)

analytical_matrix_compact_by_query_df["compact_matrix_status"] = "assembled"


# ---------------------------------------------------------
# 7. Classify query-level document potential
# ---------------------------------------------------------

def classify_document_potential(row):
    """
    Classify the document design potential of a query.

    This classification is interpretive and based on DeltaRratio.
    """
    rc = row.get("Rc", 0)
    ratio = row.get("DeltaRratio", 0)

    if rc == 0:
        return "no_relationship_traversal"

    if ratio == 0:
        return "no_reduction_potential"

    if ratio < 0.5:
        return "low_reduction_potential"

    if ratio < 1.0:
        return "medium_reduction_potential"

    return "high_reduction_potential"


def classify_mapping_reliability(row):
    """
    Classify the reliability of the physical mapping used by the query.
    """
    has_low_confidence = bool(row.get("has_low_confidence_mapping", False))
    has_no_match = bool(row.get("has_no_match_cardinality", False))

    if has_no_match:
        return "needs_review_no_observed_match"

    if has_low_confidence:
        return "needs_review_low_confidence_mapping"

    return "ok"


analytical_matrix_compact_by_query_df["document_potential_class"] = (
    analytical_matrix_compact_by_query_df
    .apply(classify_document_potential, axis=1)
)

analytical_matrix_compact_by_query_df["physical_mapping_reliability"] = (
    analytical_matrix_compact_by_query_df
    .apply(classify_mapping_reliability, axis=1)
)


# ---------------------------------------------------------
# 8. Reorder final compact matrix columns
# ---------------------------------------------------------

preferred_column_order = [
    "query_name",
    "generic_class",
    "query_family",
    "query_type",
    "abstract_query",
    "selected_root",
    "document_depth",
    "selected_document_depth",
    "best_document_depth",
    "max_depth_for_query",
    "document_depth_ratio",
    "depth_position_label",
    "depth_effect_label",
    "Rc",
    "Re",
    "DeltaR",
    "DeltaRratio",
    "relationship_reduction_potential",
    "relationship_reduction_strength",
    "document_potential_class",
    "is_full_coverage",
    "best_depth_is_full_coverage",
    "n_reachable_entities",
    "reachable_entities",
    "n_total_edges",
    "n_covered_edges",
    "n_remaining_edges",
    "covered_relationships",
    "remaining_relationships",
    "covered_semantic_types",
    "remaining_semantic_types",
    "entities_touched",
    "n_entities_touched",
    "semantic_types_touched",
    "observed_cardinalities_touched",
    "hint_confidences_touched",
    "has_low_confidence_mapping",
    "has_no_match_cardinality",
    "physical_mapping_reliability",
    "compact_matrix_status",
]

existing_preferred_columns = [
    col for col in preferred_column_order
    if col in analytical_matrix_compact_by_query_df.columns
]

remaining_columns = [
    col for col in analytical_matrix_compact_by_query_df.columns
    if col not in existing_preferred_columns
]

analytical_matrix_compact_by_query_df = analytical_matrix_compact_by_query_df[
    existing_preferred_columns + remaining_columns
].copy()


# ---------------------------------------------------------
# 9. Compatibility aliases
# ---------------------------------------------------------

compact_analytical_matrix_df = analytical_matrix_compact_by_query_df.copy()
query_analytical_matrix_df = analytical_matrix_compact_by_query_df.copy()

# Important downstream alias.
# Later blocks will add update volatility and sharedness to this matrix.
document_variable_matrix_df = analytical_matrix_compact_by_query_df.copy()

# Another explicit base alias for later integration blocks.
document_variable_matrix_base_df = analytical_matrix_compact_by_query_df.copy()


# ---------------------------------------------------------
# 10. Summary tables
# ---------------------------------------------------------

compact_matrix_summary_df = (
    analytical_matrix_compact_by_query_df
    .groupby(
        [
            "generic_class",
            "query_family",
            "document_potential_class",
            "physical_mapping_reliability",
        ],
        dropna=False,
    )
    .agg(
        n_queries=("query_name", "count"),
        avg_Rc=("Rc", "mean"),
        avg_Re=("Re", "mean"),
        avg_DeltaR=("DeltaR", "mean"),
        avg_DeltaRratio=("DeltaRratio", "mean"),
        max_DeltaRratio=("DeltaRratio", "max"),
        n_full_coverage=("is_full_coverage", lambda s: int(s.sum())),
    )
    .reset_index()
    .sort_values(["generic_class", "document_potential_class"])
    .reset_index(drop=True)
)

compact_matrix_by_potential_df = (
    analytical_matrix_compact_by_query_df
    .groupby("document_potential_class", dropna=False)
    .agg(
        n_queries=("query_name", "count"),
        queries=("query_name", lambda s: sorted(set(s))),
        avg_DeltaRratio=("DeltaRratio", "mean"),
        avg_document_depth=("document_depth", "mean"),
    )
    .reset_index()
    .sort_values("document_potential_class")
    .reset_index(drop=True)
)

compact_matrix_by_root_df = (
    analytical_matrix_compact_by_query_df
    .groupby("selected_root", dropna=False)
    .agg(
        n_queries=("query_name", "count"),
        queries=("query_name", lambda s: sorted(set(s))),
        avg_Rc=("Rc", "mean"),
        avg_DeltaRratio=("DeltaRratio", "mean"),
        avg_document_depth=("document_depth", "mean"),
    )
    .reset_index()
    .sort_values(["n_queries", "selected_root"], ascending=[False, True])
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 11. Validation tables
# ---------------------------------------------------------

compact_matrix_validation_rows = []

required_final_columns = [
    "query_name",
    "selected_root",
    "document_depth",
    "Rc",
    "Re",
    "DeltaR",
    "DeltaRratio",
    "document_potential_class",
]

for col in required_final_columns:
    compact_matrix_validation_rows.append({
        "check_name": f"column_exists_{col}",
        "status": "passed" if col in analytical_matrix_compact_by_query_df.columns else "failed",
        "details": col,
    })

missing_required_values_df = analytical_matrix_compact_by_query_df[
    analytical_matrix_compact_by_query_df[
        [
            col for col in required_final_columns
            if col in analytical_matrix_compact_by_query_df.columns
        ]
    ].isna().any(axis=1)
].copy()

compact_matrix_validation_rows.append({
    "check_name": "no_missing_required_values",
    "status": "passed" if missing_required_values_df.empty else "failed",
    "details": f"missing_rows={len(missing_required_values_df)}",
})

negative_delta_rows_df = analytical_matrix_compact_by_query_df[
    analytical_matrix_compact_by_query_df["DeltaR"] < 0
].copy()

compact_matrix_validation_rows.append({
    "check_name": "no_negative_delta",
    "status": "passed" if negative_delta_rows_df.empty else "failed",
    "details": f"negative_delta_rows={len(negative_delta_rows_df)}",
})

duplicate_query_rows_df = (
    analytical_matrix_compact_by_query_df[
        analytical_matrix_compact_by_query_df.duplicated("query_name", keep=False)
    ]
    .copy()
)

compact_matrix_validation_rows.append({
    "check_name": "one_row_per_query",
    "status": "passed" if duplicate_query_rows_df.empty else "failed",
    "details": f"duplicate_query_rows={len(duplicate_query_rows_df)}",
})

compact_matrix_validation_df = pd.DataFrame(compact_matrix_validation_rows)


# ---------------------------------------------------------
# 12. Display outputs
# ---------------------------------------------------------

print("Compact analytical matrix by query assembled successfully.")

print("\nCompact analytical matrix by query:")
display(analytical_matrix_compact_by_query_df)

print("\nSummary by class, potential, and reliability:")
display(compact_matrix_summary_df)

print("\nSummary by document potential:")
display(compact_matrix_by_potential_df)

print("\nSummary by selected root:")
display(compact_matrix_by_root_df)

print("\nValidation:")
display(compact_matrix_validation_df)

if not missing_required_values_df.empty:
    print("\nWarning: some compact matrix rows have missing required values.")
    display(missing_required_values_df)

if not negative_delta_rows_df.empty:
    print("\nWarning: some compact matrix rows have negative DeltaR.")
    display(negative_delta_rows_df)

if not duplicate_query_rows_df.empty:
    print("\nWarning: some queries appear more than once.")
    display(duplicate_query_rows_df)


# ---------------------------------------------------------
# 13. Save BLOCK V13 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    analytical_matrix_compact_by_query_df,
    "variables/block_v13/analytical_matrix_compact_by_query.csv",
)

save_dataframe(
    compact_analytical_matrix_df,
    "variables/block_v13/compact_analytical_matrix.csv",
)

save_dataframe(
    query_analytical_matrix_df,
    "variables/block_v13/query_analytical_matrix.csv",
)

save_dataframe(
    document_variable_matrix_df,
    "variables/block_v13/document_variable_matrix.csv",
)

save_dataframe(
    document_variable_matrix_base_df,
    "variables/block_v13/document_variable_matrix_base.csv",
)

save_dataframe(
    compact_best_depth_rows_df,
    "variables/block_v13/compact_best_depth_rows.csv",
)

save_dataframe(
    compact_matrix_summary_df,
    "variables/block_v13/compact_matrix_summary.csv",
)

save_dataframe(
    compact_matrix_by_potential_df,
    "variables/block_v13/compact_matrix_by_potential.csv",
)

save_dataframe(
    compact_matrix_by_root_df,
    "variables/block_v13/compact_matrix_by_root.csv",
)

save_dataframe(
    compact_matrix_validation_df,
    "variables/block_v13/compact_matrix_validation.csv",
)

save_dataframe(
    missing_required_values_df,
    "variables/block_v13/missing_required_values.csv",
)

save_dataframe(
    negative_delta_rows_df,
    "variables/block_v13/negative_delta_rows.csv",
)

save_dataframe(
    duplicate_query_rows_df,
    "variables/block_v13/duplicate_query_rows.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "n_queries": len(analytical_matrix_compact_by_query_df),
        "n_columns": len(analytical_matrix_compact_by_query_df.columns),
        "columns": analytical_matrix_compact_by_query_df.columns.tolist(),
        "document_potential_counts": (
            analytical_matrix_compact_by_query_df["document_potential_class"]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "physical_mapping_reliability_counts": (
            analytical_matrix_compact_by_query_df["physical_mapping_reliability"]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "n_validation_failed": int(
            (compact_matrix_validation_df["status"] == "failed").sum()
        ),
        "output_files": {
            "analytical_matrix_compact_csv": "variables/block_v13/analytical_matrix_compact_by_query.csv",
            "compact_analytical_matrix_csv": "variables/block_v13/compact_analytical_matrix.csv",
            "query_analytical_matrix_csv": "variables/block_v13/query_analytical_matrix.csv",
            "document_variable_matrix_csv": "variables/block_v13/document_variable_matrix.csv",
            "document_variable_matrix_base_csv": "variables/block_v13/document_variable_matrix_base.csv",
            "compact_best_depth_rows_csv": "variables/block_v13/compact_best_depth_rows.csv",
            "compact_matrix_summary_csv": "variables/block_v13/compact_matrix_summary.csv",
            "compact_matrix_by_potential_csv": "variables/block_v13/compact_matrix_by_potential.csv",
            "compact_matrix_by_root_csv": "variables/block_v13/compact_matrix_by_root.csv",
            "compact_matrix_validation_csv": "variables/block_v13/compact_matrix_validation.csv",
            "missing_required_values_csv": "variables/block_v13/missing_required_values.csv",
            "negative_delta_rows_csv": "variables/block_v13/negative_delta_rows.csv",
            "duplicate_query_rows_csv": "variables/block_v13/duplicate_query_rows.csv",
        },
    },
    "variables/block_v13/analytical_matrix_compact_metadata.json",
)

write_execution_log(
    block_name="BLOCK V13 — COMPACT ANALYTICAL MATRIX BY QUERY",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_queries": len(analytical_matrix_compact_by_query_df),
        "n_columns": len(analytical_matrix_compact_by_query_df.columns),
        "n_validation_failed": int(
            (compact_matrix_validation_df["status"] == "failed").sum()
        ),
        "analytical_matrix_compact_csv": "variables/block_v13/analytical_matrix_compact_by_query.csv",
        "document_variable_matrix_csv": "variables/block_v13/document_variable_matrix.csv",
        "compact_matrix_summary_csv": "variables/block_v13/compact_matrix_summary.csv",
        "metadata_json": "variables/block_v13/analytical_matrix_compact_metadata.json",
    },
)


# ---------------------------------------------------------
# 14. Update README section for BLOCK V13
# ---------------------------------------------------------

block_v13_readme_body = f"""
This block builds the compact analytical matrix for the FIBEN workload.

Unlike BLOCK V12, which has one row per query and document depth, this block
keeps one representative row per query. The selected representative row is
normally the best-depth row.

Main variables created:

    analytical_matrix_compact_by_query_df
    compact_analytical_matrix_df
    document_variable_matrix_df

Additional variables created:

    query_analytical_matrix_df
    document_variable_matrix_base_df
    compact_best_depth_rows_df
    compact_matrix_summary_df
    compact_matrix_by_potential_df
    compact_matrix_by_root_df
    compact_matrix_validation_df

Number of queries:

    {len(analytical_matrix_compact_by_query_df)}

Number of columns:

    {len(analytical_matrix_compact_by_query_df.columns)}

Number of failed validation checks:

    {int((compact_matrix_validation_df["status"] == "failed").sum())}

Generated reproducibility files:

    variables/block_v13/analytical_matrix_compact_by_query.csv
    variables/block_v13/compact_analytical_matrix.csv
    variables/block_v13/query_analytical_matrix.csv
    variables/block_v13/document_variable_matrix.csv
    variables/block_v13/document_variable_matrix_base.csv
    variables/block_v13/compact_best_depth_rows.csv
    variables/block_v13/compact_matrix_summary.csv
    variables/block_v13/compact_matrix_by_potential.csv
    variables/block_v13/compact_matrix_by_root.csv
    variables/block_v13/compact_matrix_validation.csv
    variables/block_v13/analytical_matrix_compact_metadata.json

Methodological meaning:

    This block produces the main compact query-level analytical matrix.

    It keeps the best-depth result per query and preserves the key variables:
    - selected root;
    - document depth;
    - Rc;
    - Re;
    - DeltaR;
    - DeltaRratio;
    - semantic profile;
    - cardinality profile;
    - physical mapping reliability.

Why this is important:

    The next blocks add update-specific variables and sharedness variables
    to document_variable_matrix_df.
"""

update_readme_section(
    section_title="BLOCK V13 — Compact Analytical Matrix by Query",
    section_body=block_v13_readme_body,
)

Compact analytical matrix by query assembled successfully.

Compact analytical matrix by query:


,query_name,generic_class,query_family,query_type,abstract_query,selected_root,document_depth,selected_document_depth,best_document_depth,max_depth_for_query,...,selected_root_is_touched,max_entity_distance_D,min_entity_distance_D,avg_entity_distance_D,max_entity_depth,max_edge_depth,relationship_status,relationship_names_touched,has_relationship_reduction_potential,has_full_coverage_at_best_depth
0,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,Create a financial service account for a perso...,Person,3,3,3,3,...,True,3,0,1.833333,3,3,has_relationship_edges,"[account_has_holding, account_records_transact...",True,True
1,Q1_CompanyProfileIBM,QG1,Local lookup,select,Show the profile of company IBM.,Corporation,0,0,0,0,...,True,0,0,0.000000,0,0,no_relationship_edges,[],False,True
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,"Show IBM with its industry, country, and liste...",Corporation,1,1,1,1,...,True,1,0,0.750000,1,1,has_relationship_edges,"[corporation_has_country, corporation_has_indu...",True,True
3,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,Show the securities held in each financial ser...,FinancialServiceAccount,3,3,3,3,...,True,3,0,1.500000,3,3,has_relationship_edges,"[account_has_holding, holding_refers_to_listed...",True,True
4,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Show the companies reached from a person throu...,Person,4,4,4,4,...,True,4,0,2.000000,4,4,has_relationship_edges,"[account_has_holding, corporation_has_listed_s...",True,True
5,Q5_ReportsAndMetricDataOfCompany,QG5,Containment retrieval,select,Show the financial reports of a company and th...,Corporation,3,3,3,3,...,True,3,0,1.600000,3,3,has_relationship_edges,"[corporation_has_financial_report, financial_r...",True,True
6,Q6_TechUSListedSecuritiesWithHighLastTradedValue,QG6,Filtered search / recommendation,select,Find listed securities of technology companies...,ListedSecurity,2,2,2,2,...,True,2,0,1.250000,2,2,has_relationship_edges,"[corporation_has_country, corporation_has_indu...",True,True
7,Q7_PersonsWhoBoughtMoreIBMThanSold,QG7,Aggregation / ranking,select,Who bought more IBM stocks than they sold?,Person,4,4,4,4,...,True,4,0,2.285714,4,4,has_relationship_edges,"[account_records_transaction, buy_transaction_...",True,True
8,Q8_IBMTransactionsBelowAverageSellingPrice,QG8,Aggregation / ranking,select,Show me each transaction for IBM whose price i...,Transaction,2,2,2,2,...,True,2,0,1.000000,2,2,has_relationship_edges,"[corporation_has_listed_security, sell_transac...",True,True
9,Q9_PersonsWhoBoughtAndSoldSameStock,QG9,Associative retrieval + comparison,select,Show me everyone who bought and sold the same ...,Person,3,3,3,3,...,True,3,0,2.000000,3,3,has_relationship_edges,"[account_records_transaction, buy_transaction_...",True,True



Summary by class, potential, and reliability:


,generic_class,query_family,document_potential_class,physical_mapping_reliability,n_queries,avg_Rc,avg_Re,avg_DeltaR,avg_DeltaRratio,max_DeltaRratio,n_full_coverage
0,QG1,Local lookup,no_relationship_traversal,ok,1,0.0,0.0,0.0,0.0,0.0,1
1,QG10,Update / insertion with relationship creation,high_reduction_potential,ok,1,5.0,0.0,5.0,1.0,1.0,1
2,QG2,Shallow rooted retrieval,high_reduction_potential,needs_review_no_observed_match,1,3.0,0.0,3.0,1.0,1.0,1
3,QG3,Associative retrieval,high_reduction_potential,needs_review_no_observed_match,1,3.0,0.0,3.0,1.0,1.0,1
4,QG4,Deep traversal,high_reduction_potential,needs_review_no_observed_match,1,4.0,0.0,4.0,1.0,1.0,1
5,QG5,Containment retrieval,high_reduction_potential,needs_review_no_observed_match,1,4.0,0.0,4.0,1.0,1.0,1
6,QG6,Filtered search / recommendation,high_reduction_potential,needs_review_no_observed_match,1,3.0,0.0,3.0,1.0,1.0,1
7,QG7,Aggregation / ranking,high_reduction_potential,needs_review_no_observed_match,1,6.0,0.0,6.0,1.0,1.0,1
8,QG8,Aggregation / ranking,high_reduction_potential,needs_review_no_observed_match,1,3.0,0.0,3.0,1.0,1.0,1
9,QG9,Associative retrieval + comparison,high_reduction_potential,ok,1,5.0,0.0,5.0,1.0,1.0,1



Summary by document potential:


...


Summary by selected root:


,selected_root,n_queries,queries,avg_Rc,avg_DeltaRratio,avg_document_depth
0,Person,4,"[Q10_CreateAccountHoldingAndBuyTransaction, Q4...",5.000000,1.000000,3.500000
1,Corporation,3,"[Q1_CompanyProfileIBM, Q2_CompanyWithIndustryC...",2.333333,0.666667,1.333333
2,FinancialServiceAccount,1,[Q3_SecuritiesHeldInEachFinancialServiceAccount],3.000000,1.000000,3.000000
3,ListedSecurity,1,[Q6_TechUSListedSecuritiesWithHighLastTradedVa...,3.000000,1.000000,2.000000
4,Transaction,1,[Q8_IBMTransactionsBelowAverageSellingPrice],3.000000,1.000000,2.000000



Validation:


,check_name,status,details
0,column_exists_query_name,passed,query_name
1,column_exists_selected_root,passed,selected_root
2,column_exists_document_depth,passed,document_depth
3,column_exists_Rc,passed,Rc
4,column_exists_Re,passed,Re
5,column_exists_DeltaR,passed,DeltaR
6,column_exists_DeltaRratio,passed,DeltaRratio
7,column_exists_document_potential_class,passed,document_potential_class
8,no_missing_required_values,passed,missing_rows=0
9,no_negative_delta,passed,negative_delta_rows=0


Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v13/analytical_matrix_compact_by_query.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v13/compact_analytical_matrix.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v13/query_analytical_matrix.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v13/document_variable_matrix.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v13/document_variable_matrix_base.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v13/compact_best_depth_rows.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variab

In [82]:
analytical_matrix_compact_by_query_df["query_name"].value_counts()

query_name
Q10_CreateAccountHoldingAndBuyTransaction                           1
Q1_CompanyProfileIBM                                                1
Q2_CompanyWithIndustryCountryAndListedSecurities                    1
Q3_SecuritiesHeldInEachFinancialServiceAccount                      1
Q4_CompaniesReachedFromPersonThroughAccountHoldingListedSecurity    1
Q5_ReportsAndMetricDataOfCompany                                    1
Q6_TechUSListedSecuritiesWithHighLastTradedValue                    1
Q7_PersonsWhoBoughtMoreIBMThanSold                                  1
Q8_IBMTransactionsBelowAverageSellingPrice                          1
Q9_PersonsWhoBoughtAndSoldSameStock                                 1
Name: count, dtype: int64

In [83]:
# =========================================================
# BLOCK V14 — ENTITIES EXPLICITLY MODIFIED BY QUERY
# =========================================================
#
# Purpose:
# This block identifies which conceptual entities are explicitly
# modified by each FIBEN workload query.
#
# Most FIBEN workload queries are read-only queries.
# The main write query is Q10:
#
#     Create a financial service account for a person,
#     add a holding for a listed security, and register
#     a buy transaction with price and quantity.
#
# This block separates:
# - created entities;
# - updated entities;
# - deleted entities;
# - referenced existing entities;
# - entities involved in relationship creation;
# - affected persistence-region entities.
#
# Main outputs preserved for downstream compatibility:
# - explicitly_modified_entities_df
# - modified_entities_by_query_df
# - query_modified_entities_df
#
# Additional outputs:
# - modified_entities_long_df
# - update_query_summary_df
# - relationship_creation_by_query_df
#
# Methodological role:
# This block prepares the update-related information required by
# the next blocks to compute update volatility by entity and query.
# =========================================================

import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "workload_conceptual_df",
    "document_variable_matrix_df",
    "fiben_scenario",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first INSTANCE 1 and BLOCKS V0–V13."
        )


# ---------------------------------------------------------
# 2. Define explicit modification specification
# ---------------------------------------------------------
#
# For read-only queries, all modification lists are empty.
#
# For Q10:
# - FinancialServiceAccount is created;
# - Holding is created;
# - BuyTransaction is created;
# - Transaction is included because BuyTransaction is modeled as a
#   conceptual subtype of Transaction;
# - Person and ListedSecurity are referenced existing entities;
# - relationship creation connects the new objects to the existing
#   Person and ListedSecurity.
#
# affected_persistence_region_entities is intentionally broader than
# created_entities. It represents document regions that may be affected
# depending on the selected document design.
# ---------------------------------------------------------

FIBEN_QUERY_MODIFICATION_SPEC = {
    "Q1_CompanyProfileIBM": {
        "operation_class": "read",
        "created_entities": [],
        "updated_entities": [],
        "deleted_entities": [],
        "referenced_existing_entities": ["Corporation"],
        "relationship_created_edges": [],
        "relationship_created_entities": [],
        "affected_persistence_region_entities": [],
        "notes": "Read-only local lookup over Corporation.",
    },

    "Q2_CompanyWithIndustryCountryAndListedSecurities": {
        "operation_class": "read",
        "created_entities": [],
        "updated_entities": [],
        "deleted_entities": [],
        "referenced_existing_entities": [
            "Corporation",
            "Industry",
            "Country",
            "ListedSecurity",
        ],
        "relationship_created_edges": [],
        "relationship_created_entities": [],
        "affected_persistence_region_entities": [],
        "notes": "Read-only shallow rooted retrieval.",
    },

    "Q3_SecuritiesHeldInEachFinancialServiceAccount": {
        "operation_class": "read",
        "created_entities": [],
        "updated_entities": [],
        "deleted_entities": [],
        "referenced_existing_entities": [
            "FinancialServiceAccount",
            "Holding",
            "ListedSecurity",
            "Security",
        ],
        "relationship_created_edges": [],
        "relationship_created_entities": [],
        "affected_persistence_region_entities": [],
        "notes": "Read-only associative retrieval over accounts and holdings.",
    },

    "Q4_CompaniesReachedFromPersonThroughAccountHoldingListedSecurity": {
        "operation_class": "read",
        "created_entities": [],
        "updated_entities": [],
        "deleted_entities": [],
        "referenced_existing_entities": [
            "Person",
            "FinancialServiceAccount",
            "Holding",
            "ListedSecurity",
            "Corporation",
        ],
        "relationship_created_edges": [],
        "relationship_created_entities": [],
        "affected_persistence_region_entities": [],
        "notes": "Read-only deep traversal from Person to Corporation.",
    },

    "Q5_ReportsAndMetricDataOfCompany": {
        "operation_class": "read",
        "created_entities": [],
        "updated_entities": [],
        "deleted_entities": [],
        "referenced_existing_entities": [
            "Corporation",
            "FinancialReport",
            "ReportElement",
            "StatementElement",
            "Disclosure",
        ],
        "relationship_created_edges": [],
        "relationship_created_entities": [],
        "affected_persistence_region_entities": [],
        "notes": "Read-only containment retrieval over financial reports and metric data.",
    },

    "Q6_TechUSListedSecuritiesWithHighLastTradedValue": {
        "operation_class": "read",
        "created_entities": [],
        "updated_entities": [],
        "deleted_entities": [],
        "referenced_existing_entities": [
            "Corporation",
            "Industry",
            "Country",
            "ListedSecurity",
        ],
        "relationship_created_edges": [],
        "relationship_created_entities": [],
        "affected_persistence_region_entities": [],
        "notes": "Read-only filtered search / recommendation query.",
    },

    "Q7_PersonsWhoBoughtMoreIBMThanSold": {
        "operation_class": "read",
        "created_entities": [],
        "updated_entities": [],
        "deleted_entities": [],
        "referenced_existing_entities": [
            "Person",
            "FinancialServiceAccount",
            "BuyTransaction",
            "SellTransaction",
            "Transaction",
            "ListedSecurity",
            "Corporation",
        ],
        "relationship_created_edges": [],
        "relationship_created_entities": [],
        "affected_persistence_region_entities": [],
        "notes": "Read-only aggregation comparing buy and sell activity.",
    },

    "Q8_IBMTransactionsBelowAverageSellingPrice": {
        "operation_class": "read",
        "created_entities": [],
        "updated_entities": [],
        "deleted_entities": [],
        "referenced_existing_entities": [
            "Transaction",
            "SellTransaction",
            "ListedSecurity",
            "Corporation",
        ],
        "relationship_created_edges": [],
        "relationship_created_entities": [],
        "affected_persistence_region_entities": [],
        "notes": "Read-only aggregation/comparison over IBM transactions.",
    },

    "Q9_PersonsWhoBoughtAndSoldSameStock": {
        "operation_class": "read",
        "created_entities": [],
        "updated_entities": [],
        "deleted_entities": [],
        "referenced_existing_entities": [
            "Person",
            "FinancialServiceAccount",
            "BuyTransaction",
            "SellTransaction",
            "Transaction",
            "ListedSecurity",
        ],
        "relationship_created_edges": [],
        "relationship_created_entities": [],
        "affected_persistence_region_entities": [],
        "notes": "Read-only associative retrieval plus comparison.",
    },

    "Q10_CreateAccountHoldingAndBuyTransaction": {
        "operation_class": "insert_with_relationship_creation",
        "created_entities": [
            "FinancialServiceAccount",
            "Holding",
            "BuyTransaction",
            "Transaction",
        ],
        "updated_entities": [],
        "deleted_entities": [],
        "referenced_existing_entities": [
            "Person",
            "ListedSecurity",
        ],
        "relationship_created_edges": [
            "person_owns_financial_service_account",
            "account_has_holding",
            "holding_refers_to_listed_security",
            "account_records_transaction",
            "transaction_refers_to_listed_security",
            "buy_transaction_is_transaction",
        ],
        "relationship_created_entities": [
            "Person",
            "FinancialServiceAccount",
            "Holding",
            "ListedSecurity",
            "BuyTransaction",
            "Transaction",
        ],
        "affected_persistence_region_entities": [
            "Person",
            "FinancialServiceAccount",
            "Holding",
            "ListedSecurity",
            "BuyTransaction",
            "Transaction",
        ],
        "notes": (
            "Insertion query. It creates a financial service account, a holding, "
            "and a buy transaction, while linking them to an existing person and "
            "listed security."
        ),
    },
}


# ---------------------------------------------------------
# 3. Validate that all workload queries have a modification spec
# ---------------------------------------------------------

workload_query_names = set(workload_conceptual_df["query_name"].tolist())
modification_spec_query_names = set(FIBEN_QUERY_MODIFICATION_SPEC.keys())

missing_modification_specs = sorted(
    workload_query_names - modification_spec_query_names
)

extra_modification_specs = sorted(
    modification_spec_query_names - workload_query_names
)

if missing_modification_specs:
    raise ValueError(
        "Some workload queries do not have a modification specification: "
        f"{missing_modification_specs}"
    )

if extra_modification_specs:
    print(
        "Warning: some modification specs do not correspond to workload queries:"
    )
    print(extra_modification_specs)


# ---------------------------------------------------------
# 4. Build query-level modification table
# ---------------------------------------------------------

modified_entities_rows = []

for _, row in workload_conceptual_df.iterrows():
    query_name = row["query_name"]
    spec = FIBEN_QUERY_MODIFICATION_SPEC[query_name]

    created_entities = spec["created_entities"]
    updated_entities = spec["updated_entities"]
    deleted_entities = spec["deleted_entities"]
    referenced_existing_entities = spec["referenced_existing_entities"]
    relationship_created_entities = spec["relationship_created_entities"]
    affected_persistence_region_entities = spec["affected_persistence_region_entities"]

    explicitly_modified_entities = sorted(
        set(created_entities + updated_entities + deleted_entities)
    )

    all_write_related_entities = sorted(
        set(
            explicitly_modified_entities
            + relationship_created_entities
            + affected_persistence_region_entities
        )
    )

    is_write_query = spec["operation_class"] != "read"

    modified_entities_rows.append({
        "query_name": query_name,
        "generic_class": row.get("generic_class", None),
        "query_family": row.get("query_family", None),
        "query_type": row.get("query_type", None),
        "operation_class": spec["operation_class"],
        "is_write_query": is_write_query,
        "created_entities": created_entities,
        "updated_entities": updated_entities,
        "deleted_entities": deleted_entities,
        "explicitly_modified_entities": explicitly_modified_entities,
        "referenced_existing_entities": referenced_existing_entities,
        "relationship_created_edges": spec["relationship_created_edges"],
        "relationship_created_entities": relationship_created_entities,
        "affected_persistence_region_entities": affected_persistence_region_entities,
        "all_write_related_entities": all_write_related_entities,
        "n_created_entities": len(created_entities),
        "n_updated_entities": len(updated_entities),
        "n_deleted_entities": len(deleted_entities),
        "n_explicitly_modified_entities": len(explicitly_modified_entities),
        "n_referenced_existing_entities": len(referenced_existing_entities),
        "n_relationship_created_edges": len(spec["relationship_created_edges"]),
        "n_relationship_created_entities": len(relationship_created_entities),
        "n_affected_persistence_region_entities": len(affected_persistence_region_entities),
        "n_all_write_related_entities": len(all_write_related_entities),
        "modification_notes": spec["notes"],
    })


modified_entities_by_query_df = pd.DataFrame(modified_entities_rows)

# Compatibility aliases.
explicitly_modified_entities_df = modified_entities_by_query_df.copy()
query_modified_entities_df = modified_entities_by_query_df.copy()


# ---------------------------------------------------------
# 5. Build long-format modified entity table
# ---------------------------------------------------------

long_rows = []

for _, row in modified_entities_by_query_df.iterrows():
    query_name = row["query_name"]

    entity_categories = {
        "created": row["created_entities"],
        "updated": row["updated_entities"],
        "deleted": row["deleted_entities"],
        "explicitly_modified": row["explicitly_modified_entities"],
        "referenced_existing": row["referenced_existing_entities"],
        "relationship_created": row["relationship_created_entities"],
        "affected_persistence_region": row["affected_persistence_region_entities"],
        "all_write_related": row["all_write_related_entities"],
    }

    for category, entities in entity_categories.items():
        for entity in entities:
            long_rows.append({
                "query_name": query_name,
                "generic_class": row["generic_class"],
                "query_family": row["query_family"],
                "query_type": row["query_type"],
                "operation_class": row["operation_class"],
                "is_write_query": row["is_write_query"],
                "entity": entity,
                "modification_category": category,
            })

modified_entities_long_df = pd.DataFrame(long_rows)


# ---------------------------------------------------------
# 6. Build relationship-creation table
# ---------------------------------------------------------

relationship_creation_rows = []

for _, row in modified_entities_by_query_df.iterrows():
    for edge_name in row["relationship_created_edges"]:
        relationship_creation_rows.append({
            "query_name": row["query_name"],
            "generic_class": row["generic_class"],
            "query_family": row["query_family"],
            "query_type": row["query_type"],
            "operation_class": row["operation_class"],
            "relationship_created_edge": edge_name,
        })

relationship_creation_by_query_df = pd.DataFrame(relationship_creation_rows)


# ---------------------------------------------------------
# 7. Add modification variables to document_variable_matrix_df
# ---------------------------------------------------------

modification_cols = [
    "query_name",
    "operation_class",
    "is_write_query",
    "created_entities",
    "updated_entities",
    "deleted_entities",
    "explicitly_modified_entities",
    "referenced_existing_entities",
    "relationship_created_edges",
    "relationship_created_entities",
    "affected_persistence_region_entities",
    "all_write_related_entities",
    "n_created_entities",
    "n_updated_entities",
    "n_deleted_entities",
    "n_explicitly_modified_entities",
    "n_referenced_existing_entities",
    "n_relationship_created_edges",
    "n_relationship_created_entities",
    "n_affected_persistence_region_entities",
    "n_all_write_related_entities",
    "modification_notes",
]

document_variable_matrix_with_modifications_df = document_variable_matrix_df.merge(
    modified_entities_by_query_df[modification_cols],
    on="query_name",
    how="left",
)

# Update the main matrix alias for downstream blocks.
document_variable_matrix_df = document_variable_matrix_with_modifications_df.copy()


# ---------------------------------------------------------
# 8. Summary tables
# ---------------------------------------------------------

update_query_summary_df = (
    modified_entities_by_query_df
    .groupby(["operation_class", "is_write_query"], dropna=False)
    .agg(
        n_queries=("query_name", "count"),
        queries=("query_name", lambda s: sorted(set(s))),
        avg_created_entities=("n_created_entities", "mean"),
        avg_explicitly_modified_entities=("n_explicitly_modified_entities", "mean"),
        avg_relationship_created_edges=("n_relationship_created_edges", "mean"),
        avg_affected_persistence_region_entities=(
            "n_affected_persistence_region_entities",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(["is_write_query", "operation_class"])
    .reset_index(drop=True)
)

modified_entity_frequency_df = (
    modified_entities_long_df
    .groupby(["entity", "modification_category"], dropna=False)
    .agg(
        n_queries=("query_name", "nunique"),
        queries=("query_name", lambda s: sorted(set(s))),
    )
    .reset_index()
    .sort_values(["modification_category", "n_queries", "entity"], ascending=[True, False, True])
    .reset_index(drop=True)
    if not modified_entities_long_df.empty
    else pd.DataFrame(
        columns=[
            "entity",
            "modification_category",
            "n_queries",
            "queries",
        ]
    )
)


# ---------------------------------------------------------
# 9. Validation tables
# ---------------------------------------------------------

write_queries_without_modified_entities_df = modified_entities_by_query_df[
    (modified_entities_by_query_df["is_write_query"] == True)
    & (modified_entities_by_query_df["n_all_write_related_entities"] == 0)
].copy()

read_queries_with_explicit_modifications_df = modified_entities_by_query_df[
    (modified_entities_by_query_df["is_write_query"] == False)
    & (modified_entities_by_query_df["n_explicitly_modified_entities"] > 0)
].copy()

missing_modification_merge_df = document_variable_matrix_df[
    document_variable_matrix_df["operation_class"].isna()
].copy()


# ---------------------------------------------------------
# 10. Display outputs
# ---------------------------------------------------------

print("Entities explicitly modified by query were identified successfully.")

print("\nModified entities by query:")
display(modified_entities_by_query_df)

print("\nModified entities in long format:")
display(modified_entities_long_df)

print("\nRelationship creation by query:")
display(relationship_creation_by_query_df)

print("\nUpdate query summary:")
display(update_query_summary_df)

print("\nModified entity frequency:")
display(modified_entity_frequency_df)

if not write_queries_without_modified_entities_df.empty:
    print("\nWarning: write queries without modified entities.")
    display(write_queries_without_modified_entities_df)

if not read_queries_with_explicit_modifications_df.empty:
    print("\nWarning: read queries with explicit modifications.")
    display(read_queries_with_explicit_modifications_df)

if not missing_modification_merge_df.empty:
    print("\nWarning: some rows in document_variable_matrix_df did not receive modification information.")
    display(missing_modification_merge_df)


# ---------------------------------------------------------
# 11. Save BLOCK V14 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    modified_entities_by_query_df,
    "variables/block_v14/modified_entities_by_query.csv",
)

save_dataframe(
    explicitly_modified_entities_df,
    "variables/block_v14/explicitly_modified_entities.csv",
)

save_dataframe(
    query_modified_entities_df,
    "variables/block_v14/query_modified_entities.csv",
)

save_dataframe(
    modified_entities_long_df,
    "variables/block_v14/modified_entities_long.csv",
)

save_dataframe(
    relationship_creation_by_query_df,
    "variables/block_v14/relationship_creation_by_query.csv",
)

save_dataframe(
    update_query_summary_df,
    "variables/block_v14/update_query_summary.csv",
)

save_dataframe(
    modified_entity_frequency_df,
    "variables/block_v14/modified_entity_frequency.csv",
)

save_dataframe(
    document_variable_matrix_df,
    "variables/block_v14/document_variable_matrix_with_modifications.csv",
)

save_dataframe(
    write_queries_without_modified_entities_df,
    "variables/block_v14/write_queries_without_modified_entities.csv",
)

save_dataframe(
    read_queries_with_explicit_modifications_df,
    "variables/block_v14/read_queries_with_explicit_modifications.csv",
)

save_dataframe(
    missing_modification_merge_df,
    "variables/block_v14/missing_modification_merge.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "query_modification_spec": FIBEN_QUERY_MODIFICATION_SPEC,
        "n_queries": len(modified_entities_by_query_df),
        "n_write_queries": int(modified_entities_by_query_df["is_write_query"].sum()),
        "n_read_queries": int((modified_entities_by_query_df["is_write_query"] == False).sum()),
        "write_queries": (
            modified_entities_by_query_df[
                modified_entities_by_query_df["is_write_query"] == True
            ]["query_name"].tolist()
        ),
        "n_modified_entity_long_rows": len(modified_entities_long_df),
        "n_relationship_creation_rows": len(relationship_creation_by_query_df),
        "output_files": {
            "modified_entities_by_query_csv": "variables/block_v14/modified_entities_by_query.csv",
            "explicitly_modified_entities_csv": "variables/block_v14/explicitly_modified_entities.csv",
            "query_modified_entities_csv": "variables/block_v14/query_modified_entities.csv",
            "modified_entities_long_csv": "variables/block_v14/modified_entities_long.csv",
            "relationship_creation_by_query_csv": "variables/block_v14/relationship_creation_by_query.csv",
            "update_query_summary_csv": "variables/block_v14/update_query_summary.csv",
            "modified_entity_frequency_csv": "variables/block_v14/modified_entity_frequency.csv",
            "document_variable_matrix_with_modifications_csv": "variables/block_v14/document_variable_matrix_with_modifications.csv",
            "write_queries_without_modified_entities_csv": "variables/block_v14/write_queries_without_modified_entities.csv",
            "read_queries_with_explicit_modifications_csv": "variables/block_v14/read_queries_with_explicit_modifications.csv",
            "missing_modification_merge_csv": "variables/block_v14/missing_modification_merge.csv",
        },
    },
    "variables/block_v14/modified_entities_metadata.json",
)

write_execution_log(
    block_name="BLOCK V14 — ENTITIES EXPLICITLY MODIFIED BY QUERY",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_queries": len(modified_entities_by_query_df),
        "n_write_queries": int(modified_entities_by_query_df["is_write_query"].sum()),
        "n_read_queries": int((modified_entities_by_query_df["is_write_query"] == False).sum()),
        "modified_entities_by_query_csv": "variables/block_v14/modified_entities_by_query.csv",
        "modified_entities_long_csv": "variables/block_v14/modified_entities_long.csv",
        "relationship_creation_by_query_csv": "variables/block_v14/relationship_creation_by_query.csv",
        "document_variable_matrix_with_modifications_csv": "variables/block_v14/document_variable_matrix_with_modifications.csv",
        "metadata_json": "variables/block_v14/modified_entities_metadata.json",
    },
)


# ---------------------------------------------------------
# 12. Update README section for BLOCK V14
# ---------------------------------------------------------

block_v14_readme_body = f"""
This block identifies which conceptual entities are explicitly modified by
each FIBEN workload query.

Main variables created:

    explicitly_modified_entities_df
    modified_entities_by_query_df
    query_modified_entities_df

Additional variables created:

    modified_entities_long_df
    relationship_creation_by_query_df
    update_query_summary_df
    modified_entity_frequency_df
    document_variable_matrix_with_modifications_df

Number of queries:

    {len(modified_entities_by_query_df)}

Number of write queries:

    {int(modified_entities_by_query_df["is_write_query"].sum())}

Number of read queries:

    {int((modified_entities_by_query_df["is_write_query"] == False).sum())}

Write query identified:

    Q10_CreateAccountHoldingAndBuyTransaction

Q10 created entities:

    FinancialServiceAccount
    Holding
    BuyTransaction
    Transaction

Q10 referenced existing entities:

    Person
    ListedSecurity

Q10 affected persistence-region entities:

    Person
    FinancialServiceAccount
    Holding
    ListedSecurity
    BuyTransaction
    Transaction

Generated reproducibility files:

    variables/block_v14/modified_entities_by_query.csv
    variables/block_v14/explicitly_modified_entities.csv
    variables/block_v14/query_modified_entities.csv
    variables/block_v14/modified_entities_long.csv
    variables/block_v14/relationship_creation_by_query.csv
    variables/block_v14/update_query_summary.csv
    variables/block_v14/modified_entity_frequency.csv
    variables/block_v14/document_variable_matrix_with_modifications.csv
    variables/block_v14/modified_entities_metadata.json

Methodological meaning:

    This block prepares the update-specific variables required to compute
    update volatility.

    It separates created entities from referenced existing entities because
    a document design may affect existing document roots even when the
    conceptual entity itself is not newly created.

Important note:

    BuyTransaction is modeled as a conceptual subtype of Transaction.
    Therefore, Q10 includes both BuyTransaction and Transaction as created
    write-related entities.
"""

update_readme_section(
    section_title="BLOCK V14 — Entities Explicitly Modified by Query",
    section_body=block_v14_readme_body,
)

Entities explicitly modified by query were identified successfully.

Modified entities by query:


,query_name,generic_class,query_family,query_type,operation_class,is_write_query,created_entities,updated_entities,deleted_entities,explicitly_modified_entities,...,n_created_entities,n_updated_entities,n_deleted_entities,n_explicitly_modified_entities,n_referenced_existing_entities,n_relationship_created_edges,n_relationship_created_entities,n_affected_persistence_region_entities,n_all_write_related_entities,modification_notes
0,Q1_CompanyProfileIBM,QG1,Local lookup,select,read,False,[],[],[],[],...,0,0,0,0,1,0,0,0,0,Read-only local lookup over Corporation.
1,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,read,False,[],[],[],[],...,0,0,0,0,4,0,0,0,0,Read-only shallow rooted retrieval.
2,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,read,False,[],[],[],[],...,0,0,0,0,4,0,0,0,0,Read-only associative retrieval over accounts ...
3,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,read,False,[],[],[],[],...,0,0,0,0,5,0,0,0,0,Read-only deep traversal from Person to Corpor...
4,Q5_ReportsAndMetricDataOfCompany,QG5,Containment retrieval,select,read,False,[],[],[],[],...,0,0,0,0,5,0,0,0,0,Read-only containment retrieval over financial...
5,Q6_TechUSListedSecuritiesWithHighLastTradedValue,QG6,Filtered search / recommendation,select,read,False,[],[],[],[],...,0,0,0,0,4,0,0,0,0,Read-only filtered search / recommendation query.
6,Q7_PersonsWhoBoughtMoreIBMThanSold,QG7,Aggregation / ranking,select,read,False,[],[],[],[],...,0,0,0,0,7,0,0,0,0,Read-only aggregation comparing buy and sell a...
7,Q8_IBMTransactionsBelowAverageSellingPrice,QG8,Aggregation / ranking,select,read,False,[],[],[],[],...,0,0,0,0,4,0,0,0,0,Read-only aggregation/comparison over IBM tran...
8,Q9_PersonsWhoBoughtAndSoldSameStock,QG9,Associative retrieval + comparison,select,read,False,[],[],[],[],...,0,0,0,0,6,0,0,0,0,Read-only associative retrieval plus comparison.
9,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,insert_with_relationship_creation,True,"[FinancialServiceAccount, Holding, BuyTransact...",[],[],"[BuyTransaction, FinancialServiceAccount, Hold...",...,4,0,0,4,2,6,6,6,6,Insertion query. It creates a financial servic...



Modified entities in long format:


,query_name,generic_class,query_family,query_type,operation_class,is_write_query,entity,modification_category
0,Q1_CompanyProfileIBM,QG1,Local lookup,select,read,False,Corporation,referenced_existing
1,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,read,False,Corporation,referenced_existing
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,read,False,Industry,referenced_existing
3,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,read,False,Country,referenced_existing
4,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,read,False,ListedSecurity,referenced_existing
...,...,...,...,...,...,...,...,...
63,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,insert_with_relationship_creation,True,FinancialServiceAccount,all_write_related
64,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,insert_with_relationship_creation,True,Holding,all_write_related
65,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,insert_with_relationship_creation,True,ListedSecurity,all_write_related
66,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,insert_with_relationship_creation,True,Person,all_write_related



Relationship creation by query:


...


Update query summary:


,operation_class,is_write_query,n_queries,queries,avg_created_entities,avg_explicitly_modified_entities,avg_relationship_created_edges,avg_affected_persistence_region_entities
0,read,False,9,"[Q1_CompanyProfileIBM, Q2_CompanyWithIndustryC...",0.0,0.0,0.0,0.0
1,insert_with_relationship_creation,True,1,[Q10_CreateAccountHoldingAndBuyTransaction],4.0,4.0,6.0,6.0



Modified entity frequency:


,entity,modification_category,n_queries,queries
0,BuyTransaction,affected_persistence_region,1,[Q10_CreateAccountHoldingAndBuyTransaction]
1,FinancialServiceAccount,affected_persistence_region,1,[Q10_CreateAccountHoldingAndBuyTransaction]
2,Holding,affected_persistence_region,1,[Q10_CreateAccountHoldingAndBuyTransaction]
3,ListedSecurity,affected_persistence_region,1,[Q10_CreateAccountHoldingAndBuyTransaction]
4,Person,affected_persistence_region,1,[Q10_CreateAccountHoldingAndBuyTransaction]
5,Transaction,affected_persistence_region,1,[Q10_CreateAccountHoldingAndBuyTransaction]
6,BuyTransaction,all_write_related,1,[Q10_CreateAccountHoldingAndBuyTransaction]
7,FinancialServiceAccount,all_write_related,1,[Q10_CreateAccountHoldingAndBuyTransaction]
8,Holding,all_write_related,1,[Q10_CreateAccountHoldingAndBuyTransaction]
9,ListedSecurity,all_write_related,1,[Q10_CreateAccountHoldingAndBuyTransaction]


Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v14/modified_entities_by_query.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v14/explicitly_modified_entities.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v14/query_modified_entities.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v14/modified_entities_long.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v14/relationship_creation_by_query.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v14/update_query_summary.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block

In [84]:
# =========================================================
# BLOCK V15 — CALCULATE UPDATE VOLATILITY BY ENTITY
# =========================================================
#
# Purpose:
# This block computes update volatility at the conceptual entity level.
#
# Update volatility estimates how much each conceptual entity is affected
# by write operations in the workload.
#
# The block uses the modification information produced in BLOCK V14.
#
# Main outputs preserved for downstream compatibility:
# - update_volatility_by_entity_df
# - entity_update_volatility_df
# - update_volatility_entity_df
#
# Additional outputs:
# - update_volatility_events_df
# - update_volatility_summary_df
# - entities_without_update_volatility_df
#
# Methodological role:
# The entity-level update volatility is later summarized by query and
# integrated into document_variable_matrix_df.
#
# Important:
# Derived categories such as explicitly_modified and all_write_related
# are not used directly in the weighted score to avoid double counting.
# =========================================================

import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "modified_entities_by_query_df",
    "modified_entities_long_df",
    "document_variable_matrix_df",
    "fiben_scenario",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first BLOCK V14 — ENTITIES EXPLICITLY MODIFIED BY QUERY."
        )


# ---------------------------------------------------------
# 2. Define volatility weights
# ---------------------------------------------------------
#
# These weights are methodological parameters.
#
# Interpretation:
# - created/updated/deleted entities have high write volatility;
# - relationship-created entities have medium volatility because
#   their persistence region may be touched by relationship insertion;
# - affected persistence-region entities have moderate volatility;
# - referenced existing entities have low volatility because they are
#   not necessarily modified, but may be affected depending on the
#   document design.
#
# Derived categories are excluded from scoring to avoid double counting:
# - explicitly_modified
# - all_write_related
# ---------------------------------------------------------

UPDATE_VOLATILITY_CATEGORY_WEIGHTS = {
    "created": 1.00,
    "updated": 1.00,
    "deleted": 1.00,
    "relationship_created": 0.60,
    "affected_persistence_region": 0.40,
    "referenced_existing": 0.15,
}

EXCLUDED_VOLATILITY_CATEGORIES = [
    "explicitly_modified",
    "all_write_related",
]


# ---------------------------------------------------------
# 3. Prepare scored update events
# ---------------------------------------------------------

if modified_entities_long_df.empty:
    update_volatility_events_df = pd.DataFrame(
        columns=[
            "query_name",
            "generic_class",
            "query_family",
            "query_type",
            "operation_class",
            "is_write_query",
            "entity",
            "modification_category",
            "category_weight",
            "weighted_update_event",
        ]
    )
else:
    update_volatility_events_df = modified_entities_long_df.copy()

    # Keep only categories that contribute to volatility scoring.
    update_volatility_events_df = update_volatility_events_df[
        ~update_volatility_events_df["modification_category"].isin(
            EXCLUDED_VOLATILITY_CATEGORIES
        )
    ].copy()

    # Add category weights.
    update_volatility_events_df["category_weight"] = (
        update_volatility_events_df["modification_category"]
        .map(UPDATE_VOLATILITY_CATEGORY_WEIGHTS)
        .fillna(0.0)
    )

    # Only write queries contribute to update volatility.
    update_volatility_events_df["weighted_update_event"] = (
        update_volatility_events_df["category_weight"]
        * update_volatility_events_df["is_write_query"].astype(float)
    )

    # Remove possible duplicates of the same query/entity/category.
    update_volatility_events_df = (
        update_volatility_events_df
        .drop_duplicates(
            subset=[
                "query_name",
                "entity",
                "modification_category",
            ]
        )
        .reset_index(drop=True)
    )


# ---------------------------------------------------------
# 4. Count workload-level denominators
# ---------------------------------------------------------

n_total_queries = int(modified_entities_by_query_df["query_name"].nunique())

n_write_queries = int(
    modified_entities_by_query_df[
        modified_entities_by_query_df["is_write_query"] == True
    ]["query_name"].nunique()
)

if n_write_queries == 0:
    n_write_queries_safe = 1
else:
    n_write_queries_safe = n_write_queries


# ---------------------------------------------------------
# 5. Aggregate volatility by entity
# ---------------------------------------------------------

if update_volatility_events_df.empty:
    entity_volatility_agg_df = pd.DataFrame(
        columns=[
            "entity",
            "weighted_update_events",
            "n_update_event_rows",
            "n_write_queries_affecting_entity",
            "n_queries_referencing_entity",
            "modification_categories",
            "write_queries_affecting_entity",
        ]
    )
else:
    entity_volatility_agg_df = (
        update_volatility_events_df
        .groupby("entity", dropna=False)
        .agg(
            weighted_update_events=("weighted_update_event", "sum"),
            n_update_event_rows=("modification_category", "count"),
            n_write_queries_affecting_entity=(
                "query_name",
                lambda s: int(
                    update_volatility_events_df.loc[
                        s.index
                    ].query("is_write_query == True")["query_name"].nunique()
                ),
            ),
            n_queries_referencing_entity=("query_name", "nunique"),
            modification_categories=(
                "modification_category",
                lambda s: sorted(set(s.dropna())),
            ),
            write_queries_affecting_entity=(
                "query_name",
                lambda s: sorted(
                    set(
                        update_volatility_events_df.loc[
                            s.index
                        ].query("is_write_query == True")["query_name"].tolist()
                    )
                ),
            ),
        )
        .reset_index()
    )


# ---------------------------------------------------------
# 6. Ensure every conceptual entity appears in the output
# ---------------------------------------------------------

all_entities_df = pd.DataFrame({
    "entity": fiben_scenario.entities
})

update_volatility_by_entity_df = all_entities_df.merge(
    entity_volatility_agg_df,
    on="entity",
    how="left",
)

# Fill missing values for entities without write activity.
update_volatility_by_entity_df["weighted_update_events"] = (
    update_volatility_by_entity_df["weighted_update_events"]
    .fillna(0.0)
)

update_volatility_by_entity_df["n_update_event_rows"] = (
    update_volatility_by_entity_df["n_update_event_rows"]
    .fillna(0)
    .astype(int)
)

update_volatility_by_entity_df["n_write_queries_affecting_entity"] = (
    update_volatility_by_entity_df["n_write_queries_affecting_entity"]
    .fillna(0)
    .astype(int)
)

update_volatility_by_entity_df["n_queries_referencing_entity"] = (
    update_volatility_by_entity_df["n_queries_referencing_entity"]
    .fillna(0)
    .astype(int)
)

for col in ["modification_categories", "write_queries_affecting_entity"]:
    update_volatility_by_entity_df[col] = (
        update_volatility_by_entity_df[col]
        .apply(lambda v: [] if isinstance(v, float) and pd.isna(v) else v)
    )


# ---------------------------------------------------------
# 7. Compute normalized volatility scores
# ---------------------------------------------------------

max_weighted_update_events = (
    update_volatility_by_entity_df["weighted_update_events"].max()
    if not update_volatility_by_entity_df.empty
    else 0
)

if max_weighted_update_events == 0:
    update_volatility_by_entity_df["weighted_update_volatility_norm"] = 0.0
else:
    update_volatility_by_entity_df["weighted_update_volatility_norm"] = (
        update_volatility_by_entity_df["weighted_update_events"]
        / max_weighted_update_events
    )

update_volatility_by_entity_df["write_query_coverage"] = (
    update_volatility_by_entity_df["n_write_queries_affecting_entity"]
    / n_write_queries_safe
)

# Main update volatility score.
#
# The score combines:
# - normalized weighted event intensity;
# - write-query coverage.
#
# With only one write query, coverage is useful but less discriminative.
# The weighted component remains the dominant term.
update_volatility_by_entity_df["update_volatility_score"] = (
    0.80 * update_volatility_by_entity_df["weighted_update_volatility_norm"]
    + 0.20 * update_volatility_by_entity_df["write_query_coverage"]
)

update_volatility_by_entity_df["has_update_volatility"] = (
    update_volatility_by_entity_df["update_volatility_score"] > 0
)


# ---------------------------------------------------------
# 8. Classify volatility level
# ---------------------------------------------------------

def classify_update_volatility_level(score):
    """
    Classify update volatility into qualitative levels.
    """
    if score == 0:
        return "none"

    if score < 0.25:
        return "low"

    if score < 0.60:
        return "medium"

    return "high"


update_volatility_by_entity_df["update_volatility_level"] = (
    update_volatility_by_entity_df["update_volatility_score"]
    .apply(classify_update_volatility_level)
)


# ---------------------------------------------------------
# 9. Add detailed category flags
# ---------------------------------------------------------

for category in UPDATE_VOLATILITY_CATEGORY_WEIGHTS.keys():
    flag_col = f"has_{category}_activity"

    update_volatility_by_entity_df[flag_col] = (
        update_volatility_by_entity_df["modification_categories"]
        .apply(lambda categories: category in categories)
    )


# ---------------------------------------------------------
# 10. Sort output
# ---------------------------------------------------------

update_volatility_by_entity_df = (
    update_volatility_by_entity_df
    .sort_values(
        ["update_volatility_score", "entity"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)


# Compatibility aliases.
entity_update_volatility_df = update_volatility_by_entity_df.copy()
update_volatility_entity_df = update_volatility_by_entity_df.copy()


# ---------------------------------------------------------
# 11. Build summary tables
# ---------------------------------------------------------

update_volatility_summary_df = (
    update_volatility_by_entity_df
    .groupby("update_volatility_level", dropna=False)
    .agg(
        n_entities=("entity", "count"),
        entities=("entity", lambda s: sorted(set(s))),
        avg_update_volatility_score=("update_volatility_score", "mean"),
        max_update_volatility_score=("update_volatility_score", "max"),
    )
    .reset_index()
    .sort_values("update_volatility_level")
    .reset_index(drop=True)
)

update_volatility_category_summary_df = (
    update_volatility_events_df
    .groupby("modification_category", dropna=False)
    .agg(
        n_event_rows=("entity", "count"),
        n_entities=("entity", "nunique"),
        entities=("entity", lambda s: sorted(set(s))),
        total_weighted_update_events=("weighted_update_event", "sum"),
    )
    .reset_index()
    .sort_values("modification_category")
    .reset_index(drop=True)
    if not update_volatility_events_df.empty
    else pd.DataFrame(
        columns=[
            "modification_category",
            "n_event_rows",
            "n_entities",
            "entities",
            "total_weighted_update_events",
        ]
    )
)

entities_without_update_volatility_df = update_volatility_by_entity_df[
    update_volatility_by_entity_df["has_update_volatility"] == False
].copy()

entities_with_update_volatility_df = update_volatility_by_entity_df[
    update_volatility_by_entity_df["has_update_volatility"] == True
].copy()


# ---------------------------------------------------------
# 12. Display outputs
# ---------------------------------------------------------

print("Update volatility by entity calculated successfully.")

print("\nUpdate volatility events:")
display(update_volatility_events_df)

print("\nUpdate volatility by entity:")
display(update_volatility_by_entity_df)

print("\nUpdate volatility summary:")
display(update_volatility_summary_df)

print("\nUpdate volatility by category:")
display(update_volatility_category_summary_df)

print("\nEntities with update volatility:")
display(entities_with_update_volatility_df)

print("\nEntities without update volatility:")
display(entities_without_update_volatility_df)


# ---------------------------------------------------------
# 13. Save BLOCK V15 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    update_volatility_by_entity_df,
    "variables/block_v15/update_volatility_by_entity.csv",
)

save_dataframe(
    entity_update_volatility_df,
    "variables/block_v15/entity_update_volatility.csv",
)

save_dataframe(
    update_volatility_entity_df,
    "variables/block_v15/update_volatility_entity.csv",
)

save_dataframe(
    update_volatility_events_df,
    "variables/block_v15/update_volatility_events.csv",
)

save_dataframe(
    update_volatility_summary_df,
    "variables/block_v15/update_volatility_summary.csv",
)

save_dataframe(
    update_volatility_category_summary_df,
    "variables/block_v15/update_volatility_category_summary.csv",
)

save_dataframe(
    entities_with_update_volatility_df,
    "variables/block_v15/entities_with_update_volatility.csv",
)

save_dataframe(
    entities_without_update_volatility_df,
    "variables/block_v15/entities_without_update_volatility.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "category_weights": UPDATE_VOLATILITY_CATEGORY_WEIGHTS,
        "excluded_categories": EXCLUDED_VOLATILITY_CATEGORIES,
        "n_total_queries": n_total_queries,
        "n_write_queries": n_write_queries,
        "n_entities": len(update_volatility_by_entity_df),
        "n_entities_with_update_volatility": len(entities_with_update_volatility_df),
        "n_entities_without_update_volatility": len(entities_without_update_volatility_df),
        "max_weighted_update_events": float(max_weighted_update_events),
        "output_files": {
            "update_volatility_by_entity_csv": "variables/block_v15/update_volatility_by_entity.csv",
            "entity_update_volatility_csv": "variables/block_v15/entity_update_volatility.csv",
            "update_volatility_entity_csv": "variables/block_v15/update_volatility_entity.csv",
            "update_volatility_events_csv": "variables/block_v15/update_volatility_events.csv",
            "update_volatility_summary_csv": "variables/block_v15/update_volatility_summary.csv",
            "update_volatility_category_summary_csv": "variables/block_v15/update_volatility_category_summary.csv",
            "entities_with_update_volatility_csv": "variables/block_v15/entities_with_update_volatility.csv",
            "entities_without_update_volatility_csv": "variables/block_v15/entities_without_update_volatility.csv",
        },
    },
    "variables/block_v15/update_volatility_by_entity_metadata.json",
)

write_execution_log(
    block_name="BLOCK V15 — CALCULATE UPDATE VOLATILITY BY ENTITY",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_total_queries": n_total_queries,
        "n_write_queries": n_write_queries,
        "n_entities": len(update_volatility_by_entity_df),
        "n_entities_with_update_volatility": len(entities_with_update_volatility_df),
        "n_entities_without_update_volatility": len(entities_without_update_volatility_df),
        "update_volatility_by_entity_csv": "variables/block_v15/update_volatility_by_entity.csv",
        "update_volatility_events_csv": "variables/block_v15/update_volatility_events.csv",
        "update_volatility_summary_csv": "variables/block_v15/update_volatility_summary.csv",
        "metadata_json": "variables/block_v15/update_volatility_by_entity_metadata.json",
    },
)


# ---------------------------------------------------------
# 14. Update README section for BLOCK V15
# ---------------------------------------------------------

block_v15_readme_body = f"""
This block calculates update volatility at the conceptual entity level.

Main variables created:

    update_volatility_by_entity_df
    entity_update_volatility_df
    update_volatility_entity_df

Additional variables created:

    update_volatility_events_df
    update_volatility_summary_df
    update_volatility_category_summary_df
    entities_with_update_volatility_df
    entities_without_update_volatility_df

Number of total queries:

    {n_total_queries}

Number of write queries:

    {n_write_queries}

Number of conceptual entities:

    {len(update_volatility_by_entity_df)}

Number of entities with update volatility:

    {len(entities_with_update_volatility_df)}

Number of entities without update volatility:

    {len(entities_without_update_volatility_df)}

Category weights used:

    created = 1.00
    updated = 1.00
    deleted = 1.00
    relationship_created = 0.60
    affected_persistence_region = 0.40
    referenced_existing = 0.15

Excluded derived categories:

    explicitly_modified
    all_write_related

Generated reproducibility files:

    variables/block_v15/update_volatility_by_entity.csv
    variables/block_v15/entity_update_volatility.csv
    variables/block_v15/update_volatility_entity.csv
    variables/block_v15/update_volatility_events.csv
    variables/block_v15/update_volatility_summary.csv
    variables/block_v15/update_volatility_category_summary.csv
    variables/block_v15/entities_with_update_volatility.csv
    variables/block_v15/entities_without_update_volatility.csv
    variables/block_v15/update_volatility_by_entity_metadata.json

Methodological meaning:

    Update volatility estimates how strongly each conceptual entity is affected
    by write operations in the workload.

    The score is based on weighted modification events and write-query coverage.

Important note:

    In the current FIBEN workload, Q10 is the main write query.
    Therefore, the volatility signal is concentrated around the entities
    affected by Q10: Person, FinancialServiceAccount, Holding, ListedSecurity,
    BuyTransaction, and Transaction.
"""

update_readme_section(
    section_title="BLOCK V15 — Calculate Update Volatility by Entity",
    section_body=block_v15_readme_body,
)

Update volatility by entity calculated successfully.

Update volatility events:


,query_name,generic_class,query_family,query_type,operation_class,is_write_query,entity,modification_category,category_weight,weighted_update_event
0,Q1_CompanyProfileIBM,QG1,Local lookup,select,read,False,Corporation,referenced_existing,0.15,0.00
1,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,read,False,Corporation,referenced_existing,0.15,0.00
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,read,False,Industry,referenced_existing,0.15,0.00
3,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,read,False,Country,referenced_existing,0.15,0.00
4,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,read,False,ListedSecurity,referenced_existing,0.15,0.00
5,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,read,False,FinancialServiceAccount,referenced_existing,0.15,0.00
6,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,read,False,Holding,referenced_existing,0.15,0.00
7,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,read,False,ListedSecurity,referenced_existing,0.15,0.00
8,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,read,False,Security,referenced_existing,0.15,0.00
9,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,read,False,Person,referenced_existing,0.15,0.00



Update volatility by entity:


,entity,weighted_update_events,n_update_event_rows,n_write_queries_affecting_entity,n_queries_referencing_entity,modification_categories,write_queries_affecting_entity,weighted_update_volatility_norm,write_query_coverage,update_volatility_score,has_update_volatility,update_volatility_level,has_created_activity,has_updated_activity,has_deleted_activity,has_relationship_created_activity,has_affected_persistence_region_activity,has_referenced_existing_activity
0,BuyTransaction,2.00,5,1,3,"[affected_persistence_region, created, referen...",[Q10_CreateAccountHoldingAndBuyTransaction],1.000,1.0,1.00,True,high,True,False,False,True,True,True
1,FinancialServiceAccount,2.00,7,1,5,"[affected_persistence_region, created, referen...",[Q10_CreateAccountHoldingAndBuyTransaction],1.000,1.0,1.00,True,high,True,False,False,True,True,True
2,Holding,2.00,5,1,3,"[affected_persistence_region, created, referen...",[Q10_CreateAccountHoldingAndBuyTransaction],1.000,1.0,1.00,True,high,True,False,False,True,True,True
3,Transaction,2.00,6,1,4,"[affected_persistence_region, created, referen...",[Q10_CreateAccountHoldingAndBuyTransaction],1.000,1.0,1.00,True,high,True,False,False,True,True,True
4,ListedSecurity,1.15,10,1,8,"[affected_persistence_region, referenced_exist...",[Q10_CreateAccountHoldingAndBuyTransaction],0.575,1.0,0.66,True,high,False,False,False,True,True,True
5,Person,1.15,6,1,4,"[affected_persistence_region, referenced_exist...",[Q10_CreateAccountHoldingAndBuyTransaction],0.575,1.0,0.66,True,high,False,False,False,True,True,True
6,Corporation,0.00,7,0,7,[referenced_existing],[],0.000,0.0,0.00,False,none,False,False,False,False,False,True
7,Country,0.00,2,0,2,[referenced_existing],[],0.000,0.0,0.00,False,none,False,False,False,False,False,True
8,Disclosure,0.00,1,0,1,[referenced_existing],[],0.000,0.0,0.00,False,none,False,False,False,False,False,True
9,FinancialReport,0.00,1,0,1,[referenced_existing],[],0.000,0.0,0.00,False,none,False,False,False,False,False,True



Update volatility summary:


...


Entities with update volatility:


,entity,weighted_update_events,n_update_event_rows,n_write_queries_affecting_entity,n_queries_referencing_entity,modification_categories,write_queries_affecting_entity,weighted_update_volatility_norm,write_query_coverage,update_volatility_score,has_update_volatility,update_volatility_level,has_created_activity,has_updated_activity,has_deleted_activity,has_relationship_created_activity,has_affected_persistence_region_activity,has_referenced_existing_activity
0,BuyTransaction,2.00,5,1,3,"[affected_persistence_region, created, referen...",[Q10_CreateAccountHoldingAndBuyTransaction],1.000,1.0,1.00,True,high,True,False,False,True,True,True
1,FinancialServiceAccount,2.00,7,1,5,"[affected_persistence_region, created, referen...",[Q10_CreateAccountHoldingAndBuyTransaction],1.000,1.0,1.00,True,high,True,False,False,True,True,True
2,Holding,2.00,5,1,3,"[affected_persistence_region, created, referen...",[Q10_CreateAccountHoldingAndBuyTransaction],1.000,1.0,1.00,True,high,True,False,False,True,True,True
3,Transaction,2.00,6,1,4,"[affected_persistence_region, created, referen...",[Q10_CreateAccountHoldingAndBuyTransaction],1.000,1.0,1.00,True,high,True,False,False,True,True,True
4,ListedSecurity,1.15,10,1,8,"[affected_persistence_region, referenced_exist...",[Q10_CreateAccountHoldingAndBuyTransaction],0.575,1.0,0.66,True,high,False,False,False,True,True,True
5,Person,1.15,6,1,4,"[affected_persistence_region, referenced_exist...",[Q10_CreateAccountHoldingAndBuyTransaction],0.575,1.0,0.66,True,high,False,False,False,True,True,True



Entities without update volatility:


,entity,weighted_update_events,n_update_event_rows,n_write_queries_affecting_entity,n_queries_referencing_entity,modification_categories,write_queries_affecting_entity,weighted_update_volatility_norm,write_query_coverage,update_volatility_score,has_update_volatility,update_volatility_level,has_created_activity,has_updated_activity,has_deleted_activity,has_relationship_created_activity,has_affected_persistence_region_activity,has_referenced_existing_activity
6,Corporation,0.0,7,0,7,[referenced_existing],[],0.0,0.0,0.0,False,none,False,False,False,False,False,True
7,Country,0.0,2,0,2,[referenced_existing],[],0.0,0.0,0.0,False,none,False,False,False,False,False,True
8,Disclosure,0.0,1,0,1,[referenced_existing],[],0.0,0.0,0.0,False,none,False,False,False,False,False,True
9,FinancialReport,0.0,1,0,1,[referenced_existing],[],0.0,0.0,0.0,False,none,False,False,False,False,False,True
10,Industry,0.0,2,0,2,[referenced_existing],[],0.0,0.0,0.0,False,none,False,False,False,False,False,True
11,ReportElement,0.0,1,0,1,[referenced_existing],[],0.0,0.0,0.0,False,none,False,False,False,False,False,True
12,Security,0.0,1,0,1,[referenced_existing],[],0.0,0.0,0.0,False,none,False,False,False,False,False,True
13,SellTransaction,0.0,3,0,3,[referenced_existing],[],0.0,0.0,0.0,False,none,False,False,False,False,False,True
14,StatementElement,0.0,1,0,1,[referenced_existing],[],0.0,0.0,0.0,False,none,False,False,False,False,False,True


Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v15/update_volatility_by_entity.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v15/entity_update_volatility.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v15/update_volatility_entity.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v15/update_volatility_events.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v15/update_volatility_summary.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v15/update_volatility_category_summary.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variab

In [85]:
update_volatility_by_entity_df[
    [
        "entity",
        "weighted_update_events",
        "write_query_coverage",
        "update_volatility_score",
        "update_volatility_level",
        "modification_categories",
    ]
]

,entity,weighted_update_events,write_query_coverage,update_volatility_score,update_volatility_level,modification_categories
0,BuyTransaction,2.00,1.0,1.00,high,"[affected_persistence_region, created, referen..."
1,FinancialServiceAccount,2.00,1.0,1.00,high,"[affected_persistence_region, created, referen..."
2,Holding,2.00,1.0,1.00,high,"[affected_persistence_region, created, referen..."
3,Transaction,2.00,1.0,1.00,high,"[affected_persistence_region, created, referen..."
4,ListedSecurity,1.15,1.0,0.66,high,"[affected_persistence_region, referenced_exist..."
5,Person,1.15,1.0,0.66,high,"[affected_persistence_region, referenced_exist..."
6,Corporation,0.00,0.0,0.00,none,[referenced_existing]
7,Country,0.00,0.0,0.00,none,[referenced_existing]
8,Disclosure,0.00,0.0,0.00,none,[referenced_existing]
9,FinancialReport,0.00,0.0,0.00,none,[referenced_existing]


In [86]:
# =========================================================
# BLOCK V16 — SUMMARIZE UPDATE VOLATILITY BY QUERY
# =========================================================
#
# Purpose:
# This block summarizes entity-level update volatility at the
# query level.
#
# The previous block calculated update volatility by entity.
# This block aggregates those scores for each workload query,
# using the entities touched by the query and the write-related
# entities when available.
#
# Main outputs preserved for downstream compatibility:
# - update_volatility_by_query_df
# - query_update_volatility_df
# - update_volatility_query_df
#
# Additional outputs:
# - query_entity_update_volatility_long_df
# - query_write_related_update_volatility_long_df
# - update_volatility_query_summary_df
#
# Methodological role:
# This block prepares update-volatility variables that will be
# integrated into document_variable_matrix_df in BLOCK V17.
# =========================================================

import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "workload_conceptual_df",
    "document_variable_matrix_df",
    "update_volatility_by_entity_df",
    "modified_entities_by_query_df",
    "fiben_scenario",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first BLOCKS V0, V14, and V15."
        )


# ---------------------------------------------------------
# 2. Helper functions
# ---------------------------------------------------------

def ensure_clean_list(value):
    """
    Convert a value into a clean Python list.

    This protects the block when a DataFrame cell contains:
    - a list;
    - a tuple;
    - a scalar;
    - a missing value.
    """
    if isinstance(value, list):
        return [v for v in value if pd.notna(v)]

    if isinstance(value, tuple):
        return [v for v in value if pd.notna(v)]

    if value is None:
        return []

    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    return [value]


def classify_query_update_volatility(score):
    """
    Classify query-level update volatility into qualitative levels.
    """
    if score == 0:
        return "none"

    if score < 0.25:
        return "low"

    if score < 0.60:
        return "medium"

    return "high"


# ---------------------------------------------------------
# 3. Build entity volatility lookup
# ---------------------------------------------------------

entity_volatility_cols = [
    "entity",
    "weighted_update_events",
    "weighted_update_volatility_norm",
    "write_query_coverage",
    "update_volatility_score",
    "update_volatility_level",
    "has_update_volatility",
    "modification_categories",
    "write_queries_affecting_entity",
]

available_entity_volatility_cols = [
    col for col in entity_volatility_cols
    if col in update_volatility_by_entity_df.columns
]

entity_volatility_lookup_df = update_volatility_by_entity_df[
    available_entity_volatility_cols
].copy()

entity_volatility_score_lookup = dict(
    zip(
        entity_volatility_lookup_df["entity"],
        entity_volatility_lookup_df["update_volatility_score"],
    )
)

entity_volatility_level_lookup = dict(
    zip(
        entity_volatility_lookup_df["entity"],
        entity_volatility_lookup_df["update_volatility_level"],
    )
)


# ---------------------------------------------------------
# 4. Build long table: query -> touched entity -> volatility
# ---------------------------------------------------------

query_entity_rows = []

for _, row in workload_conceptual_df.iterrows():
    query_name = row["query_name"]
    generic_class = row.get("generic_class", None)
    query_family = row.get("query_family", None)
    query_type = row.get("query_type", None)
    entities_touched = ensure_clean_list(row.get("entities_touched", []))

    for entity in entities_touched:
        score = entity_volatility_score_lookup.get(entity, 0.0)
        level = entity_volatility_level_lookup.get(entity, "none")

        query_entity_rows.append({
            "query_name": query_name,
            "generic_class": generic_class,
            "query_family": query_family,
            "query_type": query_type,
            "entity": entity,
            "entity_role_in_query": "touched_entity",
            "update_volatility_score": score,
            "update_volatility_level": level,
            "has_update_volatility": score > 0,
        })

query_entity_update_volatility_long_df = pd.DataFrame(query_entity_rows)


# ---------------------------------------------------------
# 5. Build long table: query -> write-related entity -> volatility
# ---------------------------------------------------------
#
# This table is especially relevant for write queries such as Q10.
# It uses all_write_related_entities from BLOCK V14.
# ---------------------------------------------------------

write_related_rows = []

for _, row in modified_entities_by_query_df.iterrows():
    query_name = row["query_name"]
    generic_class = row.get("generic_class", None)
    query_family = row.get("query_family", None)
    query_type = row.get("query_type", None)
    operation_class = row.get("operation_class", None)
    is_write_query = bool(row.get("is_write_query", False))
    all_write_related_entities = ensure_clean_list(
        row.get("all_write_related_entities", [])
    )

    for entity in all_write_related_entities:
        score = entity_volatility_score_lookup.get(entity, 0.0)
        level = entity_volatility_level_lookup.get(entity, "none")

        write_related_rows.append({
            "query_name": query_name,
            "generic_class": generic_class,
            "query_family": query_family,
            "query_type": query_type,
            "operation_class": operation_class,
            "is_write_query": is_write_query,
            "entity": entity,
            "entity_role_in_query": "write_related_entity",
            "update_volatility_score": score,
            "update_volatility_level": level,
            "has_update_volatility": score > 0,
        })

query_write_related_update_volatility_long_df = pd.DataFrame(write_related_rows)


# ---------------------------------------------------------
# 6. Aggregate update volatility by query over touched entities
# ---------------------------------------------------------

if query_entity_update_volatility_long_df.empty:
    touched_volatility_agg_df = pd.DataFrame(
        columns=[
            "query_name",
            "n_touched_entities_for_volatility",
            "n_touched_entities_with_update_volatility",
            "sum_touched_update_volatility",
            "avg_touched_update_volatility",
            "max_touched_update_volatility",
            "touched_entities_with_update_volatility",
        ]
    )
else:
    touched_volatility_agg_df = (
        query_entity_update_volatility_long_df
        .groupby("query_name", dropna=False)
        .agg(
            n_touched_entities_for_volatility=("entity", "count"),
            n_touched_entities_with_update_volatility=(
                "has_update_volatility",
                lambda s: int(s.sum()),
            ),
            sum_touched_update_volatility=("update_volatility_score", "sum"),
            avg_touched_update_volatility=("update_volatility_score", "mean"),
            max_touched_update_volatility=("update_volatility_score", "max"),
            touched_entities_with_update_volatility=(
                "entity",
                lambda s: sorted(
                    set(
                        query_entity_update_volatility_long_df.loc[
                            s.index
                        ].query("has_update_volatility == True")["entity"].tolist()
                    )
                ),
            ),
        )
        .reset_index()
    )


# ---------------------------------------------------------
# 7. Aggregate update volatility by query over write-related entities
# ---------------------------------------------------------

if query_write_related_update_volatility_long_df.empty:
    write_related_volatility_agg_df = pd.DataFrame(
        columns=[
            "query_name",
            "n_write_related_entities_for_volatility",
            "n_write_related_entities_with_update_volatility",
            "sum_write_related_update_volatility",
            "avg_write_related_update_volatility",
            "max_write_related_update_volatility",
            "write_related_entities_with_update_volatility",
        ]
    )
else:
    write_related_volatility_agg_df = (
        query_write_related_update_volatility_long_df
        .groupby("query_name", dropna=False)
        .agg(
            n_write_related_entities_for_volatility=("entity", "count"),
            n_write_related_entities_with_update_volatility=(
                "has_update_volatility",
                lambda s: int(s.sum()),
            ),
            sum_write_related_update_volatility=("update_volatility_score", "sum"),
            avg_write_related_update_volatility=("update_volatility_score", "mean"),
            max_write_related_update_volatility=("update_volatility_score", "max"),
            write_related_entities_with_update_volatility=(
                "entity",
                lambda s: sorted(
                    set(
                        query_write_related_update_volatility_long_df.loc[
                            s.index
                        ].query("has_update_volatility == True")["entity"].tolist()
                    )
                ),
            ),
        )
        .reset_index()
    )


# ---------------------------------------------------------
# 8. Build query-level update volatility table
# ---------------------------------------------------------

base_query_cols = [
    "query_name",
    "generic_class",
    "query_family",
    "query_type",
    "entities_touched",
    "n_entities_touched",
]

available_base_query_cols = [
    col for col in base_query_cols
    if col in workload_conceptual_df.columns
]

update_volatility_by_query_df = workload_conceptual_df[
    available_base_query_cols
].copy()

update_volatility_by_query_df = update_volatility_by_query_df.merge(
    modified_entities_by_query_df[
        [
            "query_name",
            "operation_class",
            "is_write_query",
            "created_entities",
            "referenced_existing_entities",
            "all_write_related_entities",
            "n_all_write_related_entities",
        ]
    ],
    on="query_name",
    how="left",
)

update_volatility_by_query_df = update_volatility_by_query_df.merge(
    touched_volatility_agg_df,
    on="query_name",
    how="left",
)

update_volatility_by_query_df = update_volatility_by_query_df.merge(
    write_related_volatility_agg_df,
    on="query_name",
    how="left",
)


# ---------------------------------------------------------
# 9. Fill missing volatility values
# ---------------------------------------------------------

numeric_zero_cols = [
    "n_touched_entities_for_volatility",
    "n_touched_entities_with_update_volatility",
    "sum_touched_update_volatility",
    "avg_touched_update_volatility",
    "max_touched_update_volatility",
    "n_write_related_entities_for_volatility",
    "n_write_related_entities_with_update_volatility",
    "sum_write_related_update_volatility",
    "avg_write_related_update_volatility",
    "max_write_related_update_volatility",
    "n_all_write_related_entities",
]

for col in numeric_zero_cols:
    if col in update_volatility_by_query_df.columns:
        update_volatility_by_query_df[col] = (
            update_volatility_by_query_df[col]
            .fillna(0)
        )

list_cols = [
    "touched_entities_with_update_volatility",
    "write_related_entities_with_update_volatility",
    "created_entities",
    "referenced_existing_entities",
    "all_write_related_entities",
]

for col in list_cols:
    if col in update_volatility_by_query_df.columns:
        update_volatility_by_query_df[col] = (
            update_volatility_by_query_df[col]
            .apply(lambda v: [] if isinstance(v, float) and pd.isna(v) else v)
        )


# ---------------------------------------------------------
# 10. Compute final query-level volatility score
# ---------------------------------------------------------
#
# For read queries, the query volatility is based on the entities touched
# by the query.
#
# For write queries, the query volatility also considers the write-related
# entities affected by the operation.
# ---------------------------------------------------------

def compute_query_update_volatility_score(row):
    """
    Compute one final update volatility score per query.

    Read queries:
        use average touched-entity volatility.

    Write queries:
        combine touched-entity volatility and write-related volatility.
    """
    touched_avg = float(row.get("avg_touched_update_volatility", 0) or 0)
    touched_max = float(row.get("max_touched_update_volatility", 0) or 0)

    write_avg = float(row.get("avg_write_related_update_volatility", 0) or 0)
    write_max = float(row.get("max_write_related_update_volatility", 0) or 0)

    is_write = bool(row.get("is_write_query", False))

    if is_write:
        return (
            0.35 * touched_avg
            + 0.25 * touched_max
            + 0.25 * write_avg
            + 0.15 * write_max
        )

    return (
        0.60 * touched_avg
        + 0.40 * touched_max
    )


update_volatility_by_query_df["query_update_volatility_score"] = (
    update_volatility_by_query_df
    .apply(compute_query_update_volatility_score, axis=1)
)

update_volatility_by_query_df["query_update_volatility_level"] = (
    update_volatility_by_query_df["query_update_volatility_score"]
    .apply(classify_query_update_volatility)
)

update_volatility_by_query_df["has_query_update_volatility"] = (
    update_volatility_by_query_df["query_update_volatility_score"] > 0
)


# ---------------------------------------------------------
# 11. Create compatibility aliases
# ---------------------------------------------------------

query_update_volatility_df = update_volatility_by_query_df.copy()
update_volatility_query_df = update_volatility_by_query_df.copy()


# ---------------------------------------------------------
# 12. Build summary tables
# ---------------------------------------------------------

update_volatility_query_summary_df = (
    update_volatility_by_query_df
    .groupby(
        ["query_update_volatility_level", "is_write_query"],
        dropna=False,
    )
    .agg(
        n_queries=("query_name", "count"),
        queries=("query_name", lambda s: sorted(set(s))),
        avg_query_update_volatility_score=(
            "query_update_volatility_score",
            "mean",
        ),
        max_query_update_volatility_score=(
            "query_update_volatility_score",
            "max",
        ),
    )
    .reset_index()
    .sort_values(
        ["is_write_query", "query_update_volatility_level"],
        ascending=[True, True],
    )
    .reset_index(drop=True)
)

update_volatility_by_generic_class_df = (
    update_volatility_by_query_df
    .groupby(["generic_class", "query_family"], dropna=False)
    .agg(
        n_queries=("query_name", "count"),
        avg_query_update_volatility_score=(
            "query_update_volatility_score",
            "mean",
        ),
        max_query_update_volatility_score=(
            "query_update_volatility_score",
            "max",
        ),
        n_queries_with_update_volatility=(
            "has_query_update_volatility",
            lambda s: int(s.sum()),
        ),
    )
    .reset_index()
    .sort_values(["generic_class", "query_family"])
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 13. Validation tables
# ---------------------------------------------------------

missing_query_update_volatility_df = update_volatility_by_query_df[
    update_volatility_by_query_df["query_update_volatility_score"].isna()
].copy()

write_queries_low_or_no_volatility_df = update_volatility_by_query_df[
    (update_volatility_by_query_df["is_write_query"] == True)
    & (update_volatility_by_query_df["query_update_volatility_score"] == 0)
].copy()

read_queries_with_volatility_df = update_volatility_by_query_df[
    (update_volatility_by_query_df["is_write_query"] == False)
    & (update_volatility_by_query_df["query_update_volatility_score"] > 0)
].copy()


# ---------------------------------------------------------
# 14. Display outputs
# ---------------------------------------------------------

print("Update volatility by query summarized successfully.")

print("\nQuery-entity update volatility long table:")
display(query_entity_update_volatility_long_df)

print("\nWrite-related update volatility long table:")
display(query_write_related_update_volatility_long_df)

print("\nUpdate volatility by query:")
display(update_volatility_by_query_df)

print("\nUpdate volatility query summary:")
display(update_volatility_query_summary_df)

print("\nUpdate volatility by generic class:")
display(update_volatility_by_generic_class_df)

if not missing_query_update_volatility_df.empty:
    print("\nWarning: some queries have missing update volatility scores.")
    display(missing_query_update_volatility_df)

if not write_queries_low_or_no_volatility_df.empty:
    print("\nWarning: some write queries have zero update volatility.")
    display(write_queries_low_or_no_volatility_df)

if not read_queries_with_volatility_df.empty:
    print("\nRead queries that touch volatile entities:")
    display(
        read_queries_with_volatility_df[
            [
                "query_name",
                "query_update_volatility_score",
                "query_update_volatility_level",
                "touched_entities_with_update_volatility",
            ]
        ]
    )


# ---------------------------------------------------------
# 15. Save BLOCK V16 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    update_volatility_by_query_df,
    "variables/block_v16/update_volatility_by_query.csv",
)

save_dataframe(
    query_update_volatility_df,
    "variables/block_v16/query_update_volatility.csv",
)

save_dataframe(
    update_volatility_query_df,
    "variables/block_v16/update_volatility_query.csv",
)

save_dataframe(
    query_entity_update_volatility_long_df,
    "variables/block_v16/query_entity_update_volatility_long.csv",
)

save_dataframe(
    query_write_related_update_volatility_long_df,
    "variables/block_v16/query_write_related_update_volatility_long.csv",
)

save_dataframe(
    update_volatility_query_summary_df,
    "variables/block_v16/update_volatility_query_summary.csv",
)

save_dataframe(
    update_volatility_by_generic_class_df,
    "variables/block_v16/update_volatility_by_generic_class.csv",
)

save_dataframe(
    missing_query_update_volatility_df,
    "variables/block_v16/missing_query_update_volatility.csv",
)

save_dataframe(
    write_queries_low_or_no_volatility_df,
    "variables/block_v16/write_queries_low_or_no_volatility.csv",
)

save_dataframe(
    read_queries_with_volatility_df,
    "variables/block_v16/read_queries_with_volatility.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "n_queries": len(update_volatility_by_query_df),
        "n_queries_with_update_volatility": int(
            update_volatility_by_query_df["has_query_update_volatility"].sum()
        ),
        "n_write_queries": int(
            update_volatility_by_query_df["is_write_query"].sum()
        ),
        "query_update_volatility_level_counts": (
            update_volatility_by_query_df["query_update_volatility_level"]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "max_query_update_volatility_score": float(
            update_volatility_by_query_df["query_update_volatility_score"].max()
        ),
        "output_files": {
            "update_volatility_by_query_csv": "variables/block_v16/update_volatility_by_query.csv",
            "query_update_volatility_csv": "variables/block_v16/query_update_volatility.csv",
            "update_volatility_query_csv": "variables/block_v16/update_volatility_query.csv",
            "query_entity_long_csv": "variables/block_v16/query_entity_update_volatility_long.csv",
            "query_write_related_long_csv": "variables/block_v16/query_write_related_update_volatility_long.csv",
            "query_summary_csv": "variables/block_v16/update_volatility_query_summary.csv",
            "by_generic_class_csv": "variables/block_v16/update_volatility_by_generic_class.csv",
            "missing_query_update_volatility_csv": "variables/block_v16/missing_query_update_volatility.csv",
            "write_queries_low_or_no_volatility_csv": "variables/block_v16/write_queries_low_or_no_volatility.csv",
            "read_queries_with_volatility_csv": "variables/block_v16/read_queries_with_volatility.csv",
        },
    },
    "variables/block_v16/update_volatility_by_query_metadata.json",
)

write_execution_log(
    block_name="BLOCK V16 — SUMMARIZE UPDATE VOLATILITY BY QUERY",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_queries": len(update_volatility_by_query_df),
        "n_queries_with_update_volatility": int(
            update_volatility_by_query_df["has_query_update_volatility"].sum()
        ),
        "n_write_queries": int(
            update_volatility_by_query_df["is_write_query"].sum()
        ),
        "update_volatility_by_query_csv": "variables/block_v16/update_volatility_by_query.csv",
        "query_entity_long_csv": "variables/block_v16/query_entity_update_volatility_long.csv",
        "query_summary_csv": "variables/block_v16/update_volatility_query_summary.csv",
        "metadata_json": "variables/block_v16/update_volatility_by_query_metadata.json",
    },
)


# ---------------------------------------------------------
# 16. Update README section for BLOCK V16
# ---------------------------------------------------------

block_v16_readme_body = f"""
This block summarizes update volatility at the query level.

The previous block calculated update volatility by conceptual entity.
This block aggregates those entity-level scores for each FIBEN workload query.

Main variables created:

    update_volatility_by_query_df
    query_update_volatility_df
    update_volatility_query_df

Additional variables created:

    query_entity_update_volatility_long_df
    query_write_related_update_volatility_long_df
    update_volatility_query_summary_df
    update_volatility_by_generic_class_df
    missing_query_update_volatility_df
    write_queries_low_or_no_volatility_df
    read_queries_with_volatility_df

Number of queries:

    {len(update_volatility_by_query_df)}

Number of queries with update volatility:

    {int(update_volatility_by_query_df["has_query_update_volatility"].sum())}

Number of write queries:

    {int(update_volatility_by_query_df["is_write_query"].sum())}

Maximum query update volatility score:

    {float(update_volatility_by_query_df["query_update_volatility_score"].max())}

Generated reproducibility files:

    variables/block_v16/update_volatility_by_query.csv
    variables/block_v16/query_update_volatility.csv
    variables/block_v16/update_volatility_query.csv
    variables/block_v16/query_entity_update_volatility_long.csv
    variables/block_v16/query_write_related_update_volatility_long.csv
    variables/block_v16/update_volatility_query_summary.csv
    variables/block_v16/update_volatility_by_generic_class.csv
    variables/block_v16/missing_query_update_volatility.csv
    variables/block_v16/write_queries_low_or_no_volatility.csv
    variables/block_v16/read_queries_with_volatility.csv
    variables/block_v16/update_volatility_by_query_metadata.json

Methodological meaning:

    Query-level update volatility estimates how sensitive each query is to
    write activity in the entities it touches.

    Read queries may still have non-zero volatility if they read entities that
    are affected by write operations elsewhere in the workload.

    Write queries combine touched-entity volatility and write-related entity
    volatility.

Important note:

    In the current FIBEN workload, Q10 is the only write query, but other
    read queries may still touch volatile entities such as Person,
    FinancialServiceAccount, Holding, ListedSecurity, BuyTransaction, and
    Transaction.
"""

update_readme_section(
    section_title="BLOCK V16 — Summarize Update Volatility by Query",
    section_body=block_v16_readme_body,
)

Update volatility by query summarized successfully.

Query-entity update volatility long table:


,query_name,generic_class,query_family,query_type,entity,entity_role_in_query,update_volatility_score,update_volatility_level,has_update_volatility
0,Q1_CompanyProfileIBM,QG1,Local lookup,select,Corporation,touched_entity,0.00,none,False
1,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Corporation,touched_entity,0.00,none,False
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Industry,touched_entity,0.00,none,False
3,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Country,touched_entity,0.00,none,False
4,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,ListedSecurity,touched_entity,0.66,high,True
5,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount,touched_entity,1.00,high,True
6,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,Holding,touched_entity,1.00,high,True
7,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,ListedSecurity,touched_entity,0.66,high,True
8,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,Security,touched_entity,0.00,none,False
9,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Person,touched_entity,0.66,high,True



Write-related update volatility long table:


,query_name,generic_class,query_family,query_type,operation_class,is_write_query,entity,entity_role_in_query,update_volatility_score,update_volatility_level,has_update_volatility
0,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,insert_with_relationship_creation,True,BuyTransaction,write_related_entity,1.00,high,True
1,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,insert_with_relationship_creation,True,FinancialServiceAccount,write_related_entity,1.00,high,True
2,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,insert_with_relationship_creation,True,Holding,write_related_entity,1.00,high,True
3,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,insert_with_relationship_creation,True,ListedSecurity,write_related_entity,0.66,high,True
4,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,insert_with_relationship_creation,True,Person,write_related_entity,0.66,high,True
5,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,insert_with_relationship_creation,True,Transaction,write_related_entity,1.00,high,True



Update volatility by query:


...


Update volatility by generic class:


,generic_class,query_family,n_queries,avg_query_update_volatility_score,max_query_update_volatility_score,n_queries_with_update_volatility
0,QG1,Local lookup,1,0.000000,0.000000,0
1,QG10,Update / insertion with relationship creation,1,0.932000,0.932000,1
2,QG2,Shallow rooted retrieval,1,0.363000,0.363000,1
3,QG3,Associative retrieval,1,0.799000,0.799000,1
4,QG4,Deep traversal,1,0.798400,0.798400,1
5,QG5,Containment retrieval,1,0.000000,0.000000,0
6,QG6,Filtered search / recommendation,1,0.363000,0.363000,1
7,QG7,Aggregation / ranking,1,0.770286,0.770286,1
8,QG8,Aggregation / ranking,1,0.649000,0.649000,1
9,QG9,Associative retrieval + comparison,1,0.832000,0.832000,1



Read queries that touch volatile entities:


,query_name,query_update_volatility_score,query_update_volatility_level,touched_entities_with_update_volatility
1,Q2_CompanyWithIndustryCountryAndListedSecurities,0.363000,medium,[ListedSecurity]
2,Q3_SecuritiesHeldInEachFinancialServiceAccount,0.799000,high,"[FinancialServiceAccount, Holding, ListedSecur..."
3,Q4_CompaniesReachedFromPersonThroughAccountHol...,0.798400,high,"[FinancialServiceAccount, Holding, ListedSecur..."
5,Q6_TechUSListedSecuritiesWithHighLastTradedValue,0.363000,medium,[ListedSecurity]
6,Q7_PersonsWhoBoughtMoreIBMThanSold,0.770286,high,"[BuyTransaction, FinancialServiceAccount, List..."
7,Q8_IBMTransactionsBelowAverageSellingPrice,0.649000,high,"[ListedSecurity, Transaction]"
8,Q9_PersonsWhoBoughtAndSoldSameStock,0.832000,high,"[BuyTransaction, FinancialServiceAccount, List..."


Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v16/update_volatility_by_query.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v16/query_update_volatility.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v16/update_volatility_query.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v16/query_entity_update_volatility_long.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v16/query_write_related_update_volatility_long.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v16/update_volatility_query_summary.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework wit

In [87]:
# =========================================================
# BLOCK V17 — INTEGRATE UPDATE VOLATILITY INTO THE ANALYTICAL MATRIX
# =========================================================
#
# Purpose:
# This block integrates query-level update volatility variables
# into the main analytical matrix:
#
#     document_variable_matrix_df
#
# Inputs:
# - document_variable_matrix_df from BLOCK V14 / V13;
# - update_volatility_by_query_df from BLOCK V16.
#
# Main outputs preserved for downstream compatibility:
# - document_variable_matrix_df
# - document_variable_matrix_with_update_volatility_df
# - analytical_matrix_with_update_volatility_df
#
# Additional outputs:
# - update_volatility_integration_check_df
# - update_volatility_matrix_summary_df
#
# Methodological role:
# This block enriches the analytical matrix with update-related
# variables that are later used by the activation logic.
#
# Important:
# The integration is query-level. Entity-level volatility was already
# summarized by query in BLOCK V16.
# =========================================================

import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "document_variable_matrix_df",
    "update_volatility_by_query_df",
    "update_volatility_by_entity_df",
    "fiben_scenario",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first BLOCKS V15 and V16."
        )


# ---------------------------------------------------------
# 2. Select update volatility columns to integrate
# ---------------------------------------------------------
#
# These variables summarize update volatility at query level.
# They are enough for the activation matrix.
# ---------------------------------------------------------

update_volatility_columns = [
    "query_name",
    "query_update_volatility_score",
    "query_update_volatility_level",
    "has_query_update_volatility",
    "n_touched_entities_for_volatility",
    "n_touched_entities_with_update_volatility",
    "sum_touched_update_volatility",
    "avg_touched_update_volatility",
    "max_touched_update_volatility",
    "touched_entities_with_update_volatility",
    "n_write_related_entities_for_volatility",
    "n_write_related_entities_with_update_volatility",
    "sum_write_related_update_volatility",
    "avg_write_related_update_volatility",
    "max_write_related_update_volatility",
    "write_related_entities_with_update_volatility",
]

available_update_volatility_columns = [
    col for col in update_volatility_columns
    if col in update_volatility_by_query_df.columns
]

update_volatility_integration_df = update_volatility_by_query_df[
    available_update_volatility_columns
].copy()


# ---------------------------------------------------------
# 3. Avoid duplicate columns before merging
# ---------------------------------------------------------
#
# If this block is re-executed, some update-volatility columns may
# already exist in document_variable_matrix_df.
#
# We remove them before merging again to avoid suffixes such as _x and _y.
# ---------------------------------------------------------

cols_to_replace = [
    col for col in available_update_volatility_columns
    if col != "query_name" and col in document_variable_matrix_df.columns
]

document_variable_matrix_without_old_update_df = (
    document_variable_matrix_df
    .drop(columns=cols_to_replace, errors="ignore")
    .copy()
)


# ---------------------------------------------------------
# 4. Merge update volatility into the analytical matrix
# ---------------------------------------------------------

document_variable_matrix_with_update_volatility_df = (
    document_variable_matrix_without_old_update_df
    .merge(
        update_volatility_integration_df,
        on="query_name",
        how="left",
    )
)


# ---------------------------------------------------------
# 5. Fill missing values safely
# ---------------------------------------------------------

numeric_update_cols = [
    "query_update_volatility_score",
    "n_touched_entities_for_volatility",
    "n_touched_entities_with_update_volatility",
    "sum_touched_update_volatility",
    "avg_touched_update_volatility",
    "max_touched_update_volatility",
    "n_write_related_entities_for_volatility",
    "n_write_related_entities_with_update_volatility",
    "sum_write_related_update_volatility",
    "avg_write_related_update_volatility",
    "max_write_related_update_volatility",
]

for col in numeric_update_cols:
    if col in document_variable_matrix_with_update_volatility_df.columns:
        document_variable_matrix_with_update_volatility_df[col] = (
            document_variable_matrix_with_update_volatility_df[col]
            .fillna(0)
        )

list_update_cols = [
    "touched_entities_with_update_volatility",
    "write_related_entities_with_update_volatility",
]

for col in list_update_cols:
    if col in document_variable_matrix_with_update_volatility_df.columns:
        document_variable_matrix_with_update_volatility_df[col] = (
            document_variable_matrix_with_update_volatility_df[col]
            .apply(lambda v: [] if isinstance(v, float) and pd.isna(v) else v)
        )

if "query_update_volatility_level" in document_variable_matrix_with_update_volatility_df.columns:
    document_variable_matrix_with_update_volatility_df["query_update_volatility_level"] = (
        document_variable_matrix_with_update_volatility_df["query_update_volatility_level"]
        .fillna("none")
    )

if "has_query_update_volatility" in document_variable_matrix_with_update_volatility_df.columns:
    document_variable_matrix_with_update_volatility_df["has_query_update_volatility"] = (
        document_variable_matrix_with_update_volatility_df["has_query_update_volatility"]
        .fillna(False)
        .astype(bool)
    )


# ---------------------------------------------------------
# 6. Add derived update-volatility interpretation variables
# ---------------------------------------------------------

def classify_update_embedding_risk(row):
    """
    Classify the interaction between document reduction potential
    and update volatility.

    A query with high DeltaRratio and high update volatility may produce
    good read benefits but also higher update-maintenance cost.
    """
    delta_ratio = float(row.get("DeltaRratio", 0) or 0)
    update_score = float(row.get("query_update_volatility_score", 0) or 0)

    if update_score == 0:
        return "no_update_pressure"

    if delta_ratio == 0:
        return "update_pressure_without_read_reduction"

    if delta_ratio >= 0.75 and update_score >= 0.60:
        return "high_read_gain_high_update_pressure"

    if delta_ratio >= 0.75 and update_score < 0.60:
        return "high_read_gain_manageable_update_pressure"

    if delta_ratio < 0.75 and update_score >= 0.60:
        return "limited_read_gain_high_update_pressure"

    return "moderate_tradeoff"


def classify_update_volatility_presence(row):
    """
    Compact binary label for update volatility presence.
    """
    if bool(row.get("has_query_update_volatility", False)):
        return "has_update_volatility"

    return "no_update_volatility"


document_variable_matrix_with_update_volatility_df["update_embedding_tradeoff_class"] = (
    document_variable_matrix_with_update_volatility_df
    .apply(classify_update_embedding_risk, axis=1)
)

document_variable_matrix_with_update_volatility_df["update_volatility_presence"] = (
    document_variable_matrix_with_update_volatility_df
    .apply(classify_update_volatility_presence, axis=1)
)


# ---------------------------------------------------------
# 7. Update main downstream aliases
# ---------------------------------------------------------

document_variable_matrix_df = document_variable_matrix_with_update_volatility_df.copy()

analytical_matrix_with_update_volatility_df = (
    document_variable_matrix_with_update_volatility_df.copy()
)

document_variable_matrix_update_enriched_df = (
    document_variable_matrix_with_update_volatility_df.copy()
)


# ---------------------------------------------------------
# 8. Build integration check table
# ---------------------------------------------------------

expected_query_names = set(document_variable_matrix_without_old_update_df["query_name"].tolist())
volatility_query_names = set(update_volatility_by_query_df["query_name"].tolist())
merged_query_names = set(document_variable_matrix_with_update_volatility_df["query_name"].tolist())

missing_from_volatility = sorted(expected_query_names - volatility_query_names)
extra_in_volatility = sorted(volatility_query_names - expected_query_names)
missing_after_merge = sorted(expected_query_names - merged_query_names)

update_volatility_integration_check_rows = [
    {
        "check_name": "all_matrix_queries_have_update_volatility_source",
        "status": "passed" if not missing_from_volatility else "failed",
        "details": missing_from_volatility,
    },
    {
        "check_name": "no_extra_update_volatility_queries",
        "status": "passed" if not extra_in_volatility else "warning",
        "details": extra_in_volatility,
    },
    {
        "check_name": "no_queries_lost_after_merge",
        "status": "passed" if not missing_after_merge else "failed",
        "details": missing_after_merge,
    },
    {
        "check_name": "row_count_preserved",
        "status": (
            "passed"
            if len(document_variable_matrix_without_old_update_df)
            == len(document_variable_matrix_with_update_volatility_df)
            else "failed"
        ),
        "details": {
            "before": len(document_variable_matrix_without_old_update_df),
            "after": len(document_variable_matrix_with_update_volatility_df),
        },
    },
]

update_volatility_integration_check_df = pd.DataFrame(
    update_volatility_integration_check_rows
)


# ---------------------------------------------------------
# 9. Build summary tables
# ---------------------------------------------------------

update_volatility_matrix_summary_df = (
    document_variable_matrix_with_update_volatility_df
    .groupby(
        [
            "query_update_volatility_level",
            "update_embedding_tradeoff_class",
        ],
        dropna=False,
    )
    .agg(
        n_queries=("query_name", "count"),
        queries=("query_name", lambda s: sorted(set(s))),
        avg_DeltaRratio=("DeltaRratio", "mean"),
        avg_query_update_volatility_score=(
            "query_update_volatility_score",
            "mean",
        ),
        max_query_update_volatility_score=(
            "query_update_volatility_score",
            "max",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "query_update_volatility_level",
            "update_embedding_tradeoff_class",
        ]
    )
    .reset_index(drop=True)
)

update_volatility_by_potential_summary_df = (
    document_variable_matrix_with_update_volatility_df
    .groupby(
        [
            "document_potential_class",
            "query_update_volatility_level",
        ],
        dropna=False,
    )
    .agg(
        n_queries=("query_name", "count"),
        queries=("query_name", lambda s: sorted(set(s))),
        avg_DeltaRratio=("DeltaRratio", "mean"),
        avg_query_update_volatility_score=(
            "query_update_volatility_score",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "document_potential_class",
            "query_update_volatility_level",
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 10. Validation tables
# ---------------------------------------------------------

missing_update_volatility_merge_df = (
    document_variable_matrix_with_update_volatility_df[
        document_variable_matrix_with_update_volatility_df[
            "query_update_volatility_score"
        ].isna()
    ].copy()
    if "query_update_volatility_score" in document_variable_matrix_with_update_volatility_df.columns
    else document_variable_matrix_with_update_volatility_df.copy()
)

failed_update_integration_checks_df = update_volatility_integration_check_df[
    update_volatility_integration_check_df["status"] == "failed"
].copy()


# ---------------------------------------------------------
# 11. Display outputs
# ---------------------------------------------------------

print("Update volatility integrated into the analytical matrix successfully.")

print("\nDocument variable matrix with update volatility:")
display(document_variable_matrix_with_update_volatility_df)

print("\nUpdate volatility integration checks:")
display(update_volatility_integration_check_df)

print("\nUpdate volatility matrix summary:")
display(update_volatility_matrix_summary_df)

print("\nUpdate volatility by document potential:")
display(update_volatility_by_potential_summary_df)

if not failed_update_integration_checks_df.empty:
    print("\nWarning: some integration checks failed.")
    display(failed_update_integration_checks_df)

if not missing_update_volatility_merge_df.empty:
    print("\nWarning: some rows are missing update volatility values.")
    display(missing_update_volatility_merge_df)


# ---------------------------------------------------------
# 12. Save BLOCK V17 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    document_variable_matrix_with_update_volatility_df,
    "variables/block_v17/document_variable_matrix_with_update_volatility.csv",
)

save_dataframe(
    analytical_matrix_with_update_volatility_df,
    "variables/block_v17/analytical_matrix_with_update_volatility.csv",
)

save_dataframe(
    document_variable_matrix_update_enriched_df,
    "variables/block_v17/document_variable_matrix_update_enriched.csv",
)

save_dataframe(
    document_variable_matrix_df,
    "variables/block_v17/document_variable_matrix.csv",
)

save_dataframe(
    update_volatility_integration_check_df,
    "variables/block_v17/update_volatility_integration_check.csv",
)

save_dataframe(
    update_volatility_matrix_summary_df,
    "variables/block_v17/update_volatility_matrix_summary.csv",
)

save_dataframe(
    update_volatility_by_potential_summary_df,
    "variables/block_v17/update_volatility_by_potential_summary.csv",
)

save_dataframe(
    missing_update_volatility_merge_df,
    "variables/block_v17/missing_update_volatility_merge.csv",
)

save_dataframe(
    failed_update_integration_checks_df,
    "variables/block_v17/failed_update_integration_checks.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "n_rows_before_merge": len(document_variable_matrix_without_old_update_df),
        "n_rows_after_merge": len(document_variable_matrix_with_update_volatility_df),
        "n_columns_after_merge": len(document_variable_matrix_with_update_volatility_df.columns),
        "n_queries_with_update_volatility": int(
            document_variable_matrix_with_update_volatility_df[
                "has_query_update_volatility"
            ].sum()
        ),
        "update_volatility_level_counts": (
            document_variable_matrix_with_update_volatility_df[
                "query_update_volatility_level"
            ]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "update_embedding_tradeoff_counts": (
            document_variable_matrix_with_update_volatility_df[
                "update_embedding_tradeoff_class"
            ]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "n_failed_integration_checks": len(failed_update_integration_checks_df),
        "output_files": {
            "document_variable_matrix_with_update_volatility_csv": "variables/block_v17/document_variable_matrix_with_update_volatility.csv",
            "analytical_matrix_with_update_volatility_csv": "variables/block_v17/analytical_matrix_with_update_volatility.csv",
            "document_variable_matrix_update_enriched_csv": "variables/block_v17/document_variable_matrix_update_enriched.csv",
            "document_variable_matrix_csv": "variables/block_v17/document_variable_matrix.csv",
            "integration_check_csv": "variables/block_v17/update_volatility_integration_check.csv",
            "matrix_summary_csv": "variables/block_v17/update_volatility_matrix_summary.csv",
            "by_potential_summary_csv": "variables/block_v17/update_volatility_by_potential_summary.csv",
            "missing_merge_csv": "variables/block_v17/missing_update_volatility_merge.csv",
            "failed_checks_csv": "variables/block_v17/failed_update_integration_checks.csv",
        },
    },
    "variables/block_v17/update_volatility_integration_metadata.json",
)

write_execution_log(
    block_name="BLOCK V17 — INTEGRATE UPDATE VOLATILITY INTO THE ANALYTICAL MATRIX",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_rows_before_merge": len(document_variable_matrix_without_old_update_df),
        "n_rows_after_merge": len(document_variable_matrix_with_update_volatility_df),
        "n_columns_after_merge": len(document_variable_matrix_with_update_volatility_df.columns),
        "n_failed_integration_checks": len(failed_update_integration_checks_df),
        "document_variable_matrix_with_update_volatility_csv": "variables/block_v17/document_variable_matrix_with_update_volatility.csv",
        "integration_check_csv": "variables/block_v17/update_volatility_integration_check.csv",
        "metadata_json": "variables/block_v17/update_volatility_integration_metadata.json",
    },
)


# ---------------------------------------------------------
# 13. Update README section for BLOCK V17
# ---------------------------------------------------------

block_v17_readme_body = f"""
This block integrates query-level update volatility variables into the main
analytical matrix.

Main variables created or updated:

    document_variable_matrix_df
    document_variable_matrix_with_update_volatility_df
    analytical_matrix_with_update_volatility_df

Additional variables created:

    document_variable_matrix_update_enriched_df
    update_volatility_integration_check_df
    update_volatility_matrix_summary_df
    update_volatility_by_potential_summary_df
    missing_update_volatility_merge_df
    failed_update_integration_checks_df

Rows before merge:

    {len(document_variable_matrix_without_old_update_df)}

Rows after merge:

    {len(document_variable_matrix_with_update_volatility_df)}

Columns after merge:

    {len(document_variable_matrix_with_update_volatility_df.columns)}

Queries with update volatility:

    {int(document_variable_matrix_with_update_volatility_df["has_query_update_volatility"].sum())}

Failed integration checks:

    {len(failed_update_integration_checks_df)}

Generated reproducibility files:

    variables/block_v17/document_variable_matrix_with_update_volatility.csv
    variables/block_v17/analytical_matrix_with_update_volatility.csv
    variables/block_v17/document_variable_matrix_update_enriched.csv
    variables/block_v17/document_variable_matrix.csv
    variables/block_v17/update_volatility_integration_check.csv
    variables/block_v17/update_volatility_matrix_summary.csv
    variables/block_v17/update_volatility_by_potential_summary.csv
    variables/block_v17/missing_update_volatility_merge.csv
    variables/block_v17/failed_update_integration_checks.csv
    variables/block_v17/update_volatility_integration_metadata.json

Methodological meaning:

    This block enriches the analytical matrix with update-volatility variables.

    These variables help identify whether a candidate document configuration
    may produce a read-performance benefit while also increasing update
    maintenance pressure.

Important interpretation:

    A query with high DeltaRratio and high update volatility may be a strong
    read-optimization candidate, but it may also require careful benchmark
    evaluation for update cost.
"""

update_readme_section(
    section_title="BLOCK V17 — Integrate Update Volatility into the Analytical Matrix",
    section_body=block_v17_readme_body,
)

Update volatility integrated into the analytical matrix successfully.

Document variable matrix with update volatility:


,query_name,generic_class,query_family,query_type,abstract_query,selected_root,document_depth,selected_document_depth,best_document_depth,max_depth_for_query,...,max_touched_update_volatility,touched_entities_with_update_volatility,n_write_related_entities_for_volatility,n_write_related_entities_with_update_volatility,sum_write_related_update_volatility,avg_write_related_update_volatility,max_write_related_update_volatility,write_related_entities_with_update_volatility,update_embedding_tradeoff_class,update_volatility_presence
0,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,Create a financial service account for a perso...,Person,3,3,3,3,...,1.00,"[BuyTransaction, FinancialServiceAccount, Hold...",6.0,6.0,5.32,0.886667,1.0,"[BuyTransaction, FinancialServiceAccount, Hold...",high_read_gain_high_update_pressure,has_update_volatility
1,Q1_CompanyProfileIBM,QG1,Local lookup,select,Show the profile of company IBM.,Corporation,0,0,0,0,...,0.00,[],0.0,0.0,0.00,0.000000,0.0,[],no_update_pressure,no_update_volatility
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,"Show IBM with its industry, country, and liste...",Corporation,1,1,1,1,...,0.66,[ListedSecurity],0.0,0.0,0.00,0.000000,0.0,[],high_read_gain_manageable_update_pressure,has_update_volatility
3,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,Show the securities held in each financial ser...,FinancialServiceAccount,3,3,3,3,...,1.00,"[FinancialServiceAccount, Holding, ListedSecur...",0.0,0.0,0.00,0.000000,0.0,[],high_read_gain_high_update_pressure,has_update_volatility
4,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Show the companies reached from a person throu...,Person,4,4,4,4,...,1.00,"[FinancialServiceAccount, Holding, ListedSecur...",0.0,0.0,0.00,0.000000,0.0,[],high_read_gain_high_update_pressure,has_update_volatility
5,Q5_ReportsAndMetricDataOfCompany,QG5,Containment retrieval,select,Show the financial reports of a company and th...,Corporation,3,3,3,3,...,0.00,[],0.0,0.0,0.00,0.000000,0.0,[],no_update_pressure,no_update_volatility
6,Q6_TechUSListedSecuritiesWithHighLastTradedValue,QG6,Filtered search / recommendation,select,Find listed securities of technology companies...,ListedSecurity,2,2,2,2,...,0.66,[ListedSecurity],0.0,0.0,0.00,0.000000,0.0,[],high_read_gain_manageable_update_pressure,has_update_volatility
7,Q7_PersonsWhoBoughtMoreIBMThanSold,QG7,Aggregation / ranking,select,Who bought more IBM stocks than they sold?,Person,4,4,4,4,...,1.00,"[BuyTransaction, FinancialServiceAccount, List...",0.0,0.0,0.00,0.000000,0.0,[],high_read_gain_high_update_pressure,has_update_volatility
8,Q8_IBMTransactionsBelowAverageSellingPrice,QG8,Aggregation / ranking,select,Show me each transaction for IBM whose price i...,Transaction,2,2,2,2,...,1.00,"[ListedSecurity, Transaction]",0.0,0.0,0.00,0.000000,0.0,[],high_read_gain_high_update_pressure,has_update_volatility
9,Q9_PersonsWhoBoughtAndSoldSameStock,QG9,Associative retrieval + comparison,select,Show me everyone who bought and sold the same ...,Person,3,3,3,3,...,1.00,"[BuyTransaction, FinancialServiceAccount, List...",0.0,0.0,0.00,0.000000,0.0,[],high_read_gain_high_update_pressure,has_update_volatility



Update volatility integration checks:


,check_name,status,details
0,all_matrix_queries_have_update_volatility_source,passed,[]
1,no_extra_update_volatility_queries,passed,[]
2,no_queries_lost_after_merge,passed,[]
3,row_count_preserved,passed,"{'before': 10, 'after': 10}"



Update volatility matrix summary:


,query_update_volatility_level,update_embedding_tradeoff_class,n_queries,queries,avg_DeltaRratio,avg_query_update_volatility_score,max_query_update_volatility_score
0,high,high_read_gain_high_update_pressure,6,"[Q10_CreateAccountHoldingAndBuyTransaction, Q3...",1.0,0.796781,0.932
1,medium,high_read_gain_manageable_update_pressure,2,[Q2_CompanyWithIndustryCountryAndListedSecurit...,1.0,0.363000,0.363
2,none,no_update_pressure,2,"[Q1_CompanyProfileIBM, Q5_ReportsAndMetricData...",0.5,0.000000,0.000



Update volatility by document potential:


,document_potential_class,query_update_volatility_level,n_queries,queries,avg_DeltaRratio,avg_query_update_volatility_score
0,high_reduction_potential,high,6,"[Q10_CreateAccountHoldingAndBuyTransaction, Q3...",1.0,0.796781
1,high_reduction_potential,medium,2,[Q2_CompanyWithIndustryCountryAndListedSecurit...,1.0,0.363000
2,high_reduction_potential,none,1,[Q5_ReportsAndMetricDataOfCompany],1.0,0.000000
3,no_relationship_traversal,none,1,[Q1_CompanyProfileIBM],0.0,0.000000


Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v17/document_variable_matrix_with_update_volatility.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v17/analytical_matrix_with_update_volatility.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v17/document_variable_matrix_update_enriched.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v17/document_variable_matrix.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v17/update_volatility_integration_check.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v17/update_volatility_matrix_summary.csv
Saved DataFrame: /home/jovyan/privado/framew

In [88]:
update_volatility_integration_check_df

,check_name,status,details
0,all_matrix_queries_have_update_volatility_source,passed,[]
1,no_extra_update_volatility_queries,passed,[]
2,no_queries_lost_after_merge,passed,[]
3,row_count_preserved,passed,"{'before': 10, 'after': 10}"


In [89]:
# =========================================================
# BLOCK V18 — EXTRACT SHAREDNESS OBSERVED IN THE FIBEN DATASET
# =========================================================
#
# Purpose:
# This block extracts observed sharedness from the physical FIBEN
# dataset using DuckDB and the manual relationship join hints.
#
# Sharedness measures how often a target entity is shared by multiple
# source entities.
#
# Example:
# If many Corporations point to the same Country, then Country has
# high observed sharedness in the relationship:
#
#     corporation_has_country
#
# Main outputs preserved for downstream compatibility:
# - observed_sharedness_df
# - relationship_sharedness_df
# - sharedness_observed_df
#
# Additional outputs:
# - query_sharedness_profile_df
# - query_edge_sharedness_df
# - sharedness_summary_df
#
# Methodological role:
# Sharedness is important for document design because embedding highly
# shared entities can create duplication and update-maintenance costs.
# =========================================================

import pandas as pd
from typing import Optional


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "fiben_data_bundle",
    "fiben_scenario",
    "conceptual_relationships_df",
    "FIBEN_MANUAL_RELATIONSHIP_JOIN_HINTS",
    "observed_cardinality_df",
    "rc_selected_edges_long_df",
    "document_variable_matrix_df",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first BLOCKS V6C, V6, V13, and V17."
        )


# ---------------------------------------------------------
# 2. Configuration
# ---------------------------------------------------------
#
# Start with the same limit used in cardinality extraction.
# Later, after the notebook is stable, this can be set to None.
# ---------------------------------------------------------

FIBEN_SHAREDNESS_ROW_LIMIT = 200_000


# ---------------------------------------------------------
# 3. SQL helper functions
# ---------------------------------------------------------

def sql_identifier(name: str) -> str:
    """
    Quote a DuckDB identifier safely.
    """
    return '"' + str(name).replace('"', '""') + '"'


def limited_view_sql(view_name: str, row_limit: Optional[int]) -> str:
    """
    Return a DuckDB subquery with an optional row limit.
    """
    if row_limit is None:
        return f"(SELECT * FROM {view_name})"

    return f"(SELECT * FROM {view_name} LIMIT {int(row_limit)})"


def duckdb_view_exists(con, view_name: str) -> bool:
    """
    Return True if a DuckDB view can be queried.
    """
    try:
        con.execute(f"SELECT * FROM {view_name} LIMIT 1").df()
        return True
    except Exception:
        return False


def duckdb_column_exists(con, view_name: str, column_name: str) -> bool:
    """
    Return True if a column exists in a DuckDB view.
    """
    try:
        cols = con.execute(f"DESCRIBE {view_name}").df()["column_name"].tolist()
        return column_name in cols
    except Exception:
        return False


# ---------------------------------------------------------
# 4. Sharedness classification
# ---------------------------------------------------------

def classify_observed_sharedness(score: float) -> str:
    """
    Convert observed sharedness score into a qualitative level.
    """
    if score == 0:
        return "none"

    if score < 0.25:
        return "low"

    if score < 0.60:
        return "medium"

    return "high"


def compute_combined_sharedness_score(stats: dict) -> float:
    """
    Compute a combined sharedness score.

    Components:
    - target_sharedness_ratio:
      fraction of target keys referenced by more than one source key;

    - duplicate_reference_pressure:
      fraction of joined rows that represent repeated target references.

    The score is bounded between 0 and 1.
    """
    target_sharedness_ratio = float(stats.get("target_sharedness_ratio", 0) or 0)
    duplicate_reference_pressure = float(
        stats.get("duplicate_reference_pressure", 0) or 0
    )

    score = (
        0.70 * target_sharedness_ratio
        + 0.30 * duplicate_reference_pressure
    )

    return max(0.0, min(1.0, score))


# ---------------------------------------------------------
# 5. Direct relationship sharedness
# ---------------------------------------------------------

def compute_direct_sharedness_statistics(
    con,
    source_view: str,
    target_view: str,
    source_column: str,
    target_column: str,
    row_limit: Optional[int],
) -> dict:
    """
    Compute sharedness statistics for a direct relationship.

    Relationship form:

        source_view.source_column = target_view.target_column

    The target side is considered shared when the same target key is
    connected to more than one distinct source key.
    """
    source_sql = limited_view_sql(source_view, row_limit)
    target_sql = limited_view_sql(target_view, row_limit)

    sql = f"""
    WITH
    s AS (
        SELECT {sql_identifier(source_column)} AS source_key
        FROM {source_sql}
        WHERE {sql_identifier(source_column)} IS NOT NULL
    ),
    t AS (
        SELECT {sql_identifier(target_column)} AS target_key
        FROM {target_sql}
        WHERE {sql_identifier(target_column)} IS NOT NULL
    ),
    joined AS (
        SELECT
            s.source_key,
            t.target_key
        FROM s
        INNER JOIN t
            ON s.source_key = t.target_key
    ),
    target_group AS (
        SELECT
            target_key,
            COUNT(*) AS joined_rows_for_target,
            COUNT(DISTINCT source_key) AS distinct_sources_for_target
        FROM joined
        GROUP BY target_key
    ),
    source_group AS (
        SELECT
            source_key,
            COUNT(*) AS joined_rows_for_source,
            COUNT(DISTINCT target_key) AS distinct_targets_for_source
        FROM joined
        GROUP BY source_key
    )
    SELECT
        (SELECT COUNT(*) FROM joined) AS joined_rows,
        (SELECT COUNT(DISTINCT source_key) FROM joined) AS joined_distinct_source_keys,
        (SELECT COUNT(DISTINCT target_key) FROM joined) AS joined_distinct_target_keys,

        (SELECT COUNT(*) FROM target_group) AS target_keys_in_join,
        (SELECT COUNT(*) FROM target_group WHERE distinct_sources_for_target > 1)
            AS target_keys_shared_by_multiple_sources,
        (SELECT AVG(distinct_sources_for_target) FROM target_group)
            AS avg_sources_per_target,
        (SELECT MAX(distinct_sources_for_target) FROM target_group)
            AS max_sources_per_target,

        (SELECT COUNT(*) FROM source_group) AS source_keys_in_join,
        (SELECT COUNT(*) FROM source_group WHERE distinct_targets_for_source > 1)
            AS source_keys_with_multiple_targets,
        (SELECT AVG(distinct_targets_for_source) FROM source_group)
            AS avg_targets_per_source,
        (SELECT MAX(distinct_targets_for_source) FROM source_group)
            AS max_targets_per_source;
    """

    row = con.execute(sql).df().iloc[0].to_dict()

    stats = {
        k: (None if pd.isna(v) else v)
        for k, v in row.items()
    }

    joined_rows = stats.get("joined_rows") or 0
    target_keys_in_join = stats.get("target_keys_in_join") or 0
    target_keys_shared = stats.get("target_keys_shared_by_multiple_sources") or 0

    if target_keys_in_join == 0:
        target_sharedness_ratio = 0.0
    else:
        target_sharedness_ratio = target_keys_shared / target_keys_in_join

    if joined_rows == 0:
        duplicate_reference_pressure = 0.0
    else:
        duplicate_reference_pressure = max(
            0.0,
            (joined_rows - target_keys_in_join) / joined_rows,
        )

    stats["target_sharedness_ratio"] = target_sharedness_ratio
    stats["duplicate_reference_pressure"] = duplicate_reference_pressure
    stats["observed_sharedness_score"] = compute_combined_sharedness_score(stats)
    stats["observed_sharedness_level"] = classify_observed_sharedness(
        stats["observed_sharedness_score"]
    )

    return stats


# ---------------------------------------------------------
# 6. Bridge relationship sharedness
# ---------------------------------------------------------

def compute_bridge_sharedness_statistics(
    con,
    source_view: str,
    bridge_view: str,
    target_view: str,
    source_column: str,
    source_to_bridge_bridge_column: str,
    bridge_to_target_bridge_column: str,
    target_column: str,
    row_limit: Optional[int],
) -> dict:
    """
    Compute sharedness statistics for a bridge relationship.

    Relationship form:

        source_view.source_column = bridge_view.source_to_bridge_bridge_column
        bridge_view.bridge_to_target_bridge_column = target_view.target_column
    """
    source_sql = limited_view_sql(source_view, row_limit)
    bridge_sql = limited_view_sql(bridge_view, row_limit)
    target_sql = limited_view_sql(target_view, row_limit)

    sql = f"""
    WITH
    s AS (
        SELECT {sql_identifier(source_column)} AS source_key
        FROM {source_sql}
        WHERE {sql_identifier(source_column)} IS NOT NULL
    ),
    b AS (
        SELECT
            {sql_identifier(source_to_bridge_bridge_column)} AS bridge_source_key,
            {sql_identifier(bridge_to_target_bridge_column)} AS bridge_target_key
        FROM {bridge_sql}
        WHERE {sql_identifier(source_to_bridge_bridge_column)} IS NOT NULL
          AND {sql_identifier(bridge_to_target_bridge_column)} IS NOT NULL
    ),
    t AS (
        SELECT {sql_identifier(target_column)} AS target_key
        FROM {target_sql}
        WHERE {sql_identifier(target_column)} IS NOT NULL
    ),
    joined AS (
        SELECT
            s.source_key,
            t.target_key
        FROM s
        INNER JOIN b
            ON s.source_key = b.bridge_source_key
        INNER JOIN t
            ON b.bridge_target_key = t.target_key
    ),
    target_group AS (
        SELECT
            target_key,
            COUNT(*) AS joined_rows_for_target,
            COUNT(DISTINCT source_key) AS distinct_sources_for_target
        FROM joined
        GROUP BY target_key
    ),
    source_group AS (
        SELECT
            source_key,
            COUNT(*) AS joined_rows_for_source,
            COUNT(DISTINCT target_key) AS distinct_targets_for_source
        FROM joined
        GROUP BY source_key
    )
    SELECT
        (SELECT COUNT(*) FROM joined) AS joined_rows,
        (SELECT COUNT(DISTINCT source_key) FROM joined) AS joined_distinct_source_keys,
        (SELECT COUNT(DISTINCT target_key) FROM joined) AS joined_distinct_target_keys,

        (SELECT COUNT(*) FROM target_group) AS target_keys_in_join,
        (SELECT COUNT(*) FROM target_group WHERE distinct_sources_for_target > 1)
            AS target_keys_shared_by_multiple_sources,
        (SELECT AVG(distinct_sources_for_target) FROM target_group)
            AS avg_sources_per_target,
        (SELECT MAX(distinct_sources_for_target) FROM target_group)
            AS max_sources_per_target,

        (SELECT COUNT(*) FROM source_group) AS source_keys_in_join,
        (SELECT COUNT(*) FROM source_group WHERE distinct_targets_for_source > 1)
            AS source_keys_with_multiple_targets,
        (SELECT AVG(distinct_targets_for_source) FROM source_group)
            AS avg_targets_per_source,
        (SELECT MAX(distinct_targets_for_source) FROM source_group)
            AS max_targets_per_source;
    """

    row = con.execute(sql).df().iloc[0].to_dict()

    stats = {
        k: (None if pd.isna(v) else v)
        for k, v in row.items()
    }

    joined_rows = stats.get("joined_rows") or 0
    target_keys_in_join = stats.get("target_keys_in_join") or 0
    target_keys_shared = stats.get("target_keys_shared_by_multiple_sources") or 0

    if target_keys_in_join == 0:
        target_sharedness_ratio = 0.0
    else:
        target_sharedness_ratio = target_keys_shared / target_keys_in_join

    if joined_rows == 0:
        duplicate_reference_pressure = 0.0
    else:
        duplicate_reference_pressure = max(
            0.0,
            (joined_rows - target_keys_in_join) / joined_rows,
        )

    stats["target_sharedness_ratio"] = target_sharedness_ratio
    stats["duplicate_reference_pressure"] = duplicate_reference_pressure
    stats["observed_sharedness_score"] = compute_combined_sharedness_score(stats)
    stats["observed_sharedness_level"] = classify_observed_sharedness(
        stats["observed_sharedness_score"]
    )

    return stats


# ---------------------------------------------------------
# 7. Compute observed sharedness for each conceptual relationship
# ---------------------------------------------------------

sharedness_rows = []

for _, rel in conceptual_relationships_df.iterrows():
    relationship_name = rel["name"]
    source_entity = rel["source"]
    target_entity = rel["target"]
    semantic_type = rel.get("semantic_type", "unspecified")
    edge_id = rel.get(
        "edge_id",
        f"{source_entity} -- {target_entity} [{relationship_name}]"
    )

    hint = FIBEN_MANUAL_RELATIONSHIP_JOIN_HINTS.get(relationship_name)

    if hint is None:
        sharedness_rows.append({
            "edge_id": edge_id,
            "relationship_name": relationship_name,
            "source_entity": source_entity,
            "target_entity": target_entity,
            "semantic_type": semantic_type,
            "hint_type": None,
            "source_view": None,
            "target_view": None,
            "bridge_view": None,
            "observed_sharedness_score": 0.0,
            "observed_sharedness_level": "not_computed",
            "sharedness_status": "missing_manual_join_hint",
            "row_limit": FIBEN_SHAREDNESS_ROW_LIMIT,
            "notes": "No manual join hint was provided.",
        })
        continue

    hint_type = hint["hint_type"]

    try:
        if hint_type in ["direct", "subtype_alias"]:
            source_view = hint["source_view"]
            target_view = hint["target_view"]
            source_column = hint["source_column"]
            target_column = hint["target_column"]

            missing = []

            if not duckdb_view_exists(fiben_data_bundle.con, source_view):
                missing.append(f"missing source view: {source_view}")

            if not duckdb_view_exists(fiben_data_bundle.con, target_view):
                missing.append(f"missing target view: {target_view}")

            if not missing:
                if not duckdb_column_exists(fiben_data_bundle.con, source_view, source_column):
                    missing.append(f"missing source column: {source_column}")

                if not duckdb_column_exists(fiben_data_bundle.con, target_view, target_column):
                    missing.append(f"missing target column: {target_column}")

            if missing:
                sharedness_rows.append({
                    "edge_id": edge_id,
                    "relationship_name": relationship_name,
                    "source_entity": source_entity,
                    "target_entity": target_entity,
                    "semantic_type": semantic_type,
                    "hint_type": hint_type,
                    "source_view": source_view,
                    "target_view": target_view,
                    "bridge_view": None,
                    "source_column": source_column,
                    "target_column": target_column,
                    "bridge_source_column": None,
                    "bridge_target_column": None,
                    "hint_confidence": hint.get("confidence"),
                    "observed_sharedness_score": 0.0,
                    "observed_sharedness_level": "not_computed",
                    "sharedness_status": "invalid_hint: " + "; ".join(missing),
                    "row_limit": FIBEN_SHAREDNESS_ROW_LIMIT,
                    "joined_rows": None,
                    "joined_distinct_source_keys": None,
                    "joined_distinct_target_keys": None,
                    "target_keys_in_join": None,
                    "target_keys_shared_by_multiple_sources": None,
                    "target_sharedness_ratio": None,
                    "duplicate_reference_pressure": None,
                    "avg_sources_per_target": None,
                    "max_sources_per_target": None,
                    "source_keys_with_multiple_targets": None,
                    "avg_targets_per_source": None,
                    "max_targets_per_source": None,
                    "notes": hint.get("notes"),
                })
                continue

            stats = compute_direct_sharedness_statistics(
                con=fiben_data_bundle.con,
                source_view=source_view,
                target_view=target_view,
                source_column=source_column,
                target_column=target_column,
                row_limit=FIBEN_SHAREDNESS_ROW_LIMIT,
            )

            sharedness_rows.append({
                "edge_id": edge_id,
                "relationship_name": relationship_name,
                "source_entity": source_entity,
                "target_entity": target_entity,
                "semantic_type": semantic_type,
                "hint_type": hint_type,
                "source_view": source_view,
                "target_view": target_view,
                "bridge_view": None,
                "source_column": source_column,
                "target_column": target_column,
                "bridge_source_column": None,
                "bridge_target_column": None,
                "hint_confidence": hint.get("confidence"),
                "sharedness_status": "computed",
                "row_limit": FIBEN_SHAREDNESS_ROW_LIMIT,
                **stats,
                "notes": hint.get("notes"),
            })

        elif hint_type == "bridge":
            source_view = hint["source_view"]
            bridge_view = hint["bridge_view"]
            target_view = hint["target_view"]

            source_column = hint["source_column"]
            bridge_source_column = hint["source_to_bridge_bridge_column"]
            bridge_target_column = hint["bridge_to_target_bridge_column"]
            target_column = hint["target_column"]

            missing = []

            for label, view_name in [
                ("source view", source_view),
                ("bridge view", bridge_view),
                ("target view", target_view),
            ]:
                if not duckdb_view_exists(fiben_data_bundle.con, view_name):
                    missing.append(f"missing {label}: {view_name}")

            if not missing:
                if not duckdb_column_exists(fiben_data_bundle.con, source_view, source_column):
                    missing.append(f"missing source column: {source_column}")

                if not duckdb_column_exists(fiben_data_bundle.con, bridge_view, bridge_source_column):
                    missing.append(f"missing bridge source column: {bridge_source_column}")

                if not duckdb_column_exists(fiben_data_bundle.con, bridge_view, bridge_target_column):
                    missing.append(f"missing bridge target column: {bridge_target_column}")

                if not duckdb_column_exists(fiben_data_bundle.con, target_view, target_column):
                    missing.append(f"missing target column: {target_column}")

            if missing:
                sharedness_rows.append({
                    "edge_id": edge_id,
                    "relationship_name": relationship_name,
                    "source_entity": source_entity,
                    "target_entity": target_entity,
                    "semantic_type": semantic_type,
                    "hint_type": hint_type,
                    "source_view": source_view,
                    "target_view": target_view,
                    "bridge_view": bridge_view,
                    "source_column": source_column,
                    "target_column": target_column,
                    "bridge_source_column": bridge_source_column,
                    "bridge_target_column": bridge_target_column,
                    "hint_confidence": hint.get("confidence"),
                    "observed_sharedness_score": 0.0,
                    "observed_sharedness_level": "not_computed",
                    "sharedness_status": "invalid_hint: " + "; ".join(missing),
                    "row_limit": FIBEN_SHAREDNESS_ROW_LIMIT,
                    "joined_rows": None,
                    "joined_distinct_source_keys": None,
                    "joined_distinct_target_keys": None,
                    "target_keys_in_join": None,
                    "target_keys_shared_by_multiple_sources": None,
                    "target_sharedness_ratio": None,
                    "duplicate_reference_pressure": None,
                    "avg_sources_per_target": None,
                    "max_sources_per_target": None,
                    "source_keys_with_multiple_targets": None,
                    "avg_targets_per_source": None,
                    "max_targets_per_source": None,
                    "notes": hint.get("notes"),
                })
                continue

            stats = compute_bridge_sharedness_statistics(
                con=fiben_data_bundle.con,
                source_view=source_view,
                bridge_view=bridge_view,
                target_view=target_view,
                source_column=source_column,
                source_to_bridge_bridge_column=bridge_source_column,
                bridge_to_target_bridge_column=bridge_target_column,
                target_column=target_column,
                row_limit=FIBEN_SHAREDNESS_ROW_LIMIT,
            )

            sharedness_rows.append({
                "edge_id": edge_id,
                "relationship_name": relationship_name,
                "source_entity": source_entity,
                "target_entity": target_entity,
                "semantic_type": semantic_type,
                "hint_type": hint_type,
                "source_view": source_view,
                "target_view": target_view,
                "bridge_view": bridge_view,
                "source_column": source_column,
                "target_column": target_column,
                "bridge_source_column": bridge_source_column,
                "bridge_target_column": bridge_target_column,
                "hint_confidence": hint.get("confidence"),
                "sharedness_status": "computed",
                "row_limit": FIBEN_SHAREDNESS_ROW_LIMIT,
                **stats,
                "notes": hint.get("notes"),
            })

        else:
            sharedness_rows.append({
                "edge_id": edge_id,
                "relationship_name": relationship_name,
                "source_entity": source_entity,
                "target_entity": target_entity,
                "semantic_type": semantic_type,
                "hint_type": hint_type,
                "source_view": hint.get("source_view"),
                "target_view": hint.get("target_view"),
                "bridge_view": hint.get("bridge_view"),
                "source_column": hint.get("source_column"),
                "target_column": hint.get("target_column"),
                "bridge_source_column": hint.get("source_to_bridge_bridge_column"),
                "bridge_target_column": hint.get("bridge_to_target_bridge_column"),
                "hint_confidence": hint.get("confidence"),
                "observed_sharedness_score": 0.0,
                "observed_sharedness_level": "not_computed",
                "sharedness_status": f"unsupported_hint_type: {hint_type}",
                "row_limit": FIBEN_SHAREDNESS_ROW_LIMIT,
                "joined_rows": None,
                "joined_distinct_source_keys": None,
                "joined_distinct_target_keys": None,
                "target_keys_in_join": None,
                "target_keys_shared_by_multiple_sources": None,
                "target_sharedness_ratio": None,
                "duplicate_reference_pressure": None,
                "avg_sources_per_target": None,
                "max_sources_per_target": None,
                "source_keys_with_multiple_targets": None,
                "avg_targets_per_source": None,
                "max_targets_per_source": None,
                "notes": hint.get("notes"),
            })

    except Exception as e:
        sharedness_rows.append({
            "edge_id": edge_id,
            "relationship_name": relationship_name,
            "source_entity": source_entity,
            "target_entity": target_entity,
            "semantic_type": semantic_type,
            "hint_type": hint_type,
            "source_view": hint.get("source_view"),
            "target_view": hint.get("target_view"),
            "bridge_view": hint.get("bridge_view"),
            "source_column": hint.get("source_column"),
            "target_column": hint.get("target_column"),
            "bridge_source_column": hint.get("source_to_bridge_bridge_column"),
            "bridge_target_column": hint.get("bridge_to_target_bridge_column"),
            "hint_confidence": hint.get("confidence"),
            "observed_sharedness_score": 0.0,
            "observed_sharedness_level": "not_computed",
            "sharedness_status": f"computation_failed: {type(e).__name__}: {e}",
            "row_limit": FIBEN_SHAREDNESS_ROW_LIMIT,
            "joined_rows": None,
            "joined_distinct_source_keys": None,
            "joined_distinct_target_keys": None,
            "target_keys_in_join": None,
            "target_keys_shared_by_multiple_sources": None,
            "target_sharedness_ratio": None,
            "duplicate_reference_pressure": None,
            "avg_sources_per_target": None,
            "max_sources_per_target": None,
            "source_keys_with_multiple_targets": None,
            "avg_targets_per_source": None,
            "max_targets_per_source": None,
            "notes": hint.get("notes"),
        })


# ---------------------------------------------------------
# 8. Main relationship-level outputs
# ---------------------------------------------------------

observed_sharedness_df = pd.DataFrame(sharedness_rows)

relationship_sharedness_df = observed_sharedness_df.copy()
sharedness_observed_df = observed_sharedness_df.copy()


# ---------------------------------------------------------
# 9. Build query-edge sharedness profile
# ---------------------------------------------------------

sharedness_edge_cols = [
    "edge_id",
    "relationship_name",
    "source_entity",
    "target_entity",
    "semantic_type",
    "observed_sharedness_score",
    "observed_sharedness_level",
    "target_sharedness_ratio",
    "duplicate_reference_pressure",
    "avg_sources_per_target",
    "max_sources_per_target",
    "hint_confidence",
    "sharedness_status",
]

available_sharedness_edge_cols = [
    col for col in sharedness_edge_cols
    if col in observed_sharedness_df.columns
]

edge_sharedness_lookup_df = observed_sharedness_df[
    available_sharedness_edge_cols
].copy()

if rc_selected_edges_long_df.empty:
    query_edge_sharedness_df = pd.DataFrame()
else:
    query_edge_sharedness_df = rc_selected_edges_long_df.merge(
        edge_sharedness_lookup_df,
        on="edge_id",
        how="left",
    )


# ---------------------------------------------------------
# 10. Aggregate sharedness by query
# ---------------------------------------------------------

if query_edge_sharedness_df.empty:
    query_sharedness_profile_df = pd.DataFrame(
        columns=[
            "query_name",
            "avg_observed_sharedness_score",
            "max_observed_sharedness_score",
            "sum_observed_sharedness_score",
            "n_edges_with_sharedness",
            "n_high_sharedness_edges",
            "shared_relationships",
            "shared_target_entities",
            "observed_sharedness_levels",
        ]
    )
else:
    query_sharedness_profile_df = (
        query_edge_sharedness_df
        .groupby("query_name", dropna=False)
        .agg(
            avg_observed_sharedness_score=(
                "observed_sharedness_score",
                "mean",
            ),
            max_observed_sharedness_score=(
                "observed_sharedness_score",
                "max",
            ),
            sum_observed_sharedness_score=(
                "observed_sharedness_score",
                "sum",
            ),
            n_edges_with_sharedness=(
                "observed_sharedness_score",
                lambda s: int((s > 0).sum()),
            ),
            n_high_sharedness_edges=(
                "observed_sharedness_level",
                lambda s: int((s == "high").sum()),
            ),
            shared_relationships=(
                "relationship_name",
                lambda s: sorted(set(s.dropna())),
            ),
            shared_target_entities=(
                "target_entity",
                lambda s: sorted(set(s.dropna())),
            ),
            observed_sharedness_levels=(
                "observed_sharedness_level",
                lambda s: sorted(set(s.dropna())),
            ),
        )
        .reset_index()
    )


# ---------------------------------------------------------
# 11. Ensure all queries appear in query_sharedness_profile_df
# ---------------------------------------------------------

all_queries_for_sharedness_df = document_variable_matrix_df[
    ["query_name"]
].drop_duplicates().copy()

query_sharedness_profile_df = all_queries_for_sharedness_df.merge(
    query_sharedness_profile_df,
    on="query_name",
    how="left",
)

numeric_sharedness_cols = [
    "avg_observed_sharedness_score",
    "max_observed_sharedness_score",
    "sum_observed_sharedness_score",
    "n_edges_with_sharedness",
    "n_high_sharedness_edges",
]

for col in numeric_sharedness_cols:
    if col in query_sharedness_profile_df.columns:
        query_sharedness_profile_df[col] = (
            query_sharedness_profile_df[col]
            .fillna(0)
        )

list_sharedness_cols = [
    "shared_relationships",
    "shared_target_entities",
    "observed_sharedness_levels",
]

for col in list_sharedness_cols:
    if col in query_sharedness_profile_df.columns:
        query_sharedness_profile_df[col] = (
            query_sharedness_profile_df[col]
            .apply(lambda v: [] if isinstance(v, float) and pd.isna(v) else v)
        )


# ---------------------------------------------------------
# 12. Classify query-level sharedness
# ---------------------------------------------------------

query_sharedness_profile_df["query_observed_sharedness_level"] = (
    query_sharedness_profile_df["max_observed_sharedness_score"]
    .apply(classify_observed_sharedness)
)

query_sharedness_profile_df["has_observed_sharedness"] = (
    query_sharedness_profile_df["max_observed_sharedness_score"] > 0
)


# ---------------------------------------------------------
# 13. Summary and validation tables
# ---------------------------------------------------------

sharedness_summary_df = (
    observed_sharedness_df
    .groupby(
        ["observed_sharedness_level", "semantic_type"],
        dropna=False,
    )
    .agg(
        n_relationships=("relationship_name", "count"),
        relationships=("relationship_name", lambda s: sorted(set(s))),
        avg_observed_sharedness_score=(
            "observed_sharedness_score",
            "mean",
        ),
        max_observed_sharedness_score=(
            "observed_sharedness_score",
            "max",
        ),
    )
    .reset_index()
    .sort_values(["observed_sharedness_level", "semantic_type"])
    .reset_index(drop=True)
)

query_sharedness_summary_df = (
    query_sharedness_profile_df
    .groupby("query_observed_sharedness_level", dropna=False)
    .agg(
        n_queries=("query_name", "count"),
        queries=("query_name", lambda s: sorted(set(s))),
        avg_max_observed_sharedness_score=(
            "max_observed_sharedness_score",
            "mean",
        ),
    )
    .reset_index()
    .sort_values("query_observed_sharedness_level")
    .reset_index(drop=True)
)

sharedness_not_computed_df = observed_sharedness_df[
    observed_sharedness_df["sharedness_status"] != "computed"
].copy()

high_sharedness_relationships_df = observed_sharedness_df[
    observed_sharedness_df["observed_sharedness_level"] == "high"
].copy()


# ---------------------------------------------------------
# 14. Display outputs
# ---------------------------------------------------------

print("Observed sharedness extracted successfully.")

print("\nObserved sharedness by relationship:")
display(observed_sharedness_df)

print("\nQuery-edge sharedness:")
display(query_edge_sharedness_df)

print("\nQuery sharedness profile:")
display(query_sharedness_profile_df)

print("\nSharedness summary:")
display(sharedness_summary_df)

print("\nQuery sharedness summary:")
display(query_sharedness_summary_df)

if not sharedness_not_computed_df.empty:
    print("\nWarning: some relationships did not receive sharedness statistics.")
    display(sharedness_not_computed_df)

if not high_sharedness_relationships_df.empty:
    print("\nHigh-sharedness relationships:")
    display(
        high_sharedness_relationships_df[
            [
                "relationship_name",
                "source_entity",
                "target_entity",
                "observed_sharedness_score",
                "target_sharedness_ratio",
                "max_sources_per_target",
            ]
        ]
    )


# ---------------------------------------------------------
# 15. Save BLOCK V18 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    observed_sharedness_df,
    "variables/block_v18/observed_sharedness_by_relationship.csv",
)

save_dataframe(
    relationship_sharedness_df,
    "variables/block_v18/relationship_sharedness.csv",
)

save_dataframe(
    sharedness_observed_df,
    "variables/block_v18/sharedness_observed.csv",
)

save_dataframe(
    query_edge_sharedness_df,
    "variables/block_v18/query_edge_sharedness.csv",
)

save_dataframe(
    query_sharedness_profile_df,
    "variables/block_v18/query_sharedness_profile.csv",
)

save_dataframe(
    sharedness_summary_df,
    "variables/block_v18/sharedness_summary.csv",
)

save_dataframe(
    query_sharedness_summary_df,
    "variables/block_v18/query_sharedness_summary.csv",
)

save_dataframe(
    sharedness_not_computed_df,
    "variables/block_v18/sharedness_not_computed.csv",
)

save_dataframe(
    high_sharedness_relationships_df,
    "variables/block_v18/high_sharedness_relationships.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "row_limit": FIBEN_SHAREDNESS_ROW_LIMIT,
        "n_relationships": len(observed_sharedness_df),
        "n_computed_relationships": int(
            (observed_sharedness_df["sharedness_status"] == "computed").sum()
        ),
        "n_not_computed_relationships": len(sharedness_not_computed_df),
        "n_high_sharedness_relationships": len(high_sharedness_relationships_df),
        "n_queries": int(query_sharedness_profile_df["query_name"].nunique()),
        "n_queries_with_observed_sharedness": int(
            query_sharedness_profile_df["has_observed_sharedness"].sum()
        ),
        "sharedness_level_counts": (
            observed_sharedness_df["observed_sharedness_level"]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "query_sharedness_level_counts": (
            query_sharedness_profile_df["query_observed_sharedness_level"]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "output_files": {
            "observed_sharedness_csv": "variables/block_v18/observed_sharedness_by_relationship.csv",
            "relationship_sharedness_csv": "variables/block_v18/relationship_sharedness.csv",
            "sharedness_observed_csv": "variables/block_v18/sharedness_observed.csv",
            "query_edge_sharedness_csv": "variables/block_v18/query_edge_sharedness.csv",
            "query_sharedness_profile_csv": "variables/block_v18/query_sharedness_profile.csv",
            "sharedness_summary_csv": "variables/block_v18/sharedness_summary.csv",
            "query_sharedness_summary_csv": "variables/block_v18/query_sharedness_summary.csv",
            "sharedness_not_computed_csv": "variables/block_v18/sharedness_not_computed.csv",
            "high_sharedness_relationships_csv": "variables/block_v18/high_sharedness_relationships.csv",
        },
    },
    "variables/block_v18/observed_sharedness_metadata.json",
)

write_execution_log(
    block_name="BLOCK V18 — EXTRACT SHAREDNESS OBSERVED IN THE FIBEN DATASET",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "row_limit": FIBEN_SHAREDNESS_ROW_LIMIT,
        "n_relationships": len(observed_sharedness_df),
        "n_computed_relationships": int(
            (observed_sharedness_df["sharedness_status"] == "computed").sum()
        ),
        "n_not_computed_relationships": len(sharedness_not_computed_df),
        "n_high_sharedness_relationships": len(high_sharedness_relationships_df),
        "n_queries_with_observed_sharedness": int(
            query_sharedness_profile_df["has_observed_sharedness"].sum()
        ),
        "observed_sharedness_csv": "variables/block_v18/observed_sharedness_by_relationship.csv",
        "query_sharedness_profile_csv": "variables/block_v18/query_sharedness_profile.csv",
        "metadata_json": "variables/block_v18/observed_sharedness_metadata.json",
    },
)


# ---------------------------------------------------------
# 16. Update README section for BLOCK V18
# ---------------------------------------------------------

block_v18_readme_body = f"""
This block extracts observed sharedness from the physical FIBEN dataset.

Sharedness measures how often a target entity is shared by multiple source
entities.

Main variables created:

    observed_sharedness_df
    relationship_sharedness_df
    sharedness_observed_df

Additional variables created:

    query_edge_sharedness_df
    query_sharedness_profile_df
    sharedness_summary_df
    query_sharedness_summary_df
    sharedness_not_computed_df
    high_sharedness_relationships_df

Number of relationships processed:

    {len(observed_sharedness_df)}

Number of computed relationships:

    {int((observed_sharedness_df["sharedness_status"] == "computed").sum())}

Number of relationships not computed:

    {len(sharedness_not_computed_df)}

Number of high-sharedness relationships:

    {len(high_sharedness_relationships_df)}

Number of queries with observed sharedness:

    {int(query_sharedness_profile_df["has_observed_sharedness"].sum())}

Row limit used per table:

    {FIBEN_SHAREDNESS_ROW_LIMIT}

Generated reproducibility files:

    variables/block_v18/observed_sharedness_by_relationship.csv
    variables/block_v18/relationship_sharedness.csv
    variables/block_v18/sharedness_observed.csv
    variables/block_v18/query_edge_sharedness.csv
    variables/block_v18/query_sharedness_profile.csv
    variables/block_v18/sharedness_summary.csv
    variables/block_v18/query_sharedness_summary.csv
    variables/block_v18/sharedness_not_computed.csv
    variables/block_v18/high_sharedness_relationships.csv
    variables/block_v18/observed_sharedness_metadata.json

Methodological meaning:

    Observed sharedness helps identify relationships where the target entity is
    reused by multiple source entities.

    This matters for document design because embedding highly shared entities
    can increase duplication and update-maintenance cost.

Examples in FIBEN:

    Many Corporations may share the same Country.
    Many Corporations may share the same Industry.
    Many Holdings may refer to the same ListedSecurity.
"""

update_readme_section(
    section_title="BLOCK V18 — Extract Sharedness Observed in the FIBEN Dataset",
    section_body=block_v18_readme_body,
)

Observed sharedness extracted successfully.

Observed sharedness by relationship:


,edge_id,relationship_name,source_entity,target_entity,semantic_type,hint_type,source_view,target_view,bridge_view,source_column,...,max_sources_per_target,source_keys_in_join,source_keys_with_multiple_targets,avg_targets_per_source,max_targets_per_source,target_sharedness_ratio,duplicate_reference_pressure,observed_sharedness_score,observed_sharedness_level,notes
0,Corporation -- Industry [corporation_has_indus...,corporation_has_industry,Corporation,Industry,descriptor,direct,fiben_corporations,fiben_industries,None,ISCLASSIFIEDBY,...,1.0,358.0,0.0,1.0,1.0,0.0,0.845955,0.253787,medium,Corporation points to IndustrySectorClassifier...
1,Corporation -- Country [corporation_has_country],corporation_has_country,Corporation,Country,descriptor,direct,fiben_corporations,fiben_countries,None,ISDOMICILEDIN,...,1.0,20.0,0.0,1.0,1.0,0.0,0.991394,0.297418,medium,Corporation points to Country through ISDOMICI...
2,Corporation -- ListedSecurity [corporation_has...,corporation_has_listed_security,Corporation,ListedSecurity,association,bridge,fiben_corporations,fiben_listed_securities,fiben_securities,CORPORATIONID,...,NaN,0.0,0.0,NaN,NaN,0.0,0.000000,0.000000,none,Derived path Corporation -> Security -> Listed...
3,ListedSecurity -- Security [listed_security_re...,listed_security_represents_security,ListedSecurity,Security,association,direct,fiben_listed_securities,fiben_securities,None,LISTEDSECURITYID,...,NaN,0.0,0.0,NaN,NaN,0.0,0.000000,0.000000,none,Security appears to point to ListedSecurity th...
4,Person -- FinancialServiceAccount [person_owns...,person_owns_financial_service_account,Person,FinancialServiceAccount,ownership,direct,fiben_persons,fiben_financial_service_accounts,None,PERSONID,...,1.0,50000.0,0.0,1.0,1.0,0.0,0.485967,0.145790,low,FinancialServiceAccount points to Person throu...
5,FinancialServiceAccount -- Holding [account_ha...,account_has_holding,FinancialServiceAccount,Holding,containment,direct,fiben_financial_service_accounts,fiben_holdings,None,FINANCIALSERVICEACCOUNTID,...,1.0,36445.0,0.0,1.0,1.0,0.0,0.817775,0.245333,low,Holding points to FinancialServiceAccount thro...
6,Holding -- ListedSecurity [holding_refers_to_l...,holding_refers_to_listed_security,Holding,ListedSecurity,association,direct,fiben_holdings,fiben_listed_securities,None,REFERSTO,...,1.0,2745.0,0.0,1.0,1.0,0.0,0.986275,0.295882,medium,Holding points to ListedSecurity through REFER...
7,FinancialServiceAccount -- Transaction [accoun...,account_records_transaction,FinancialServiceAccount,Transaction,containment,direct,fiben_financial_service_accounts,fiben_transactions,None,FINANCIALSERVICEACCOUNTID,...,1.0,84838.0,0.0,1.0,1.0,0.0,0.575810,0.172743,low,SecuritiesTransaction points to FinancialServi...
8,Transaction -- ListedSecurity [transaction_ref...,transaction_refers_to_listed_security,Transaction,ListedSecurity,association,direct,fiben_transactions,fiben_listed_securities,None,REFERSTO,...,1.0,2745.0,0.0,1.0,1.0,0.0,0.986275,0.295882,medium,SecuritiesTransaction points to ListedSecurity...
9,BuyTransaction -- Transaction [buy_transaction...,buy_transaction_is_transaction,BuyTransaction,Transaction,subtype,subtype_alias,fiben_buy_transactions,fiben_transactions,None,SECURITIESTRANSACTIONID,...,1.0,200000.0,0.0,1.0,1.0,0.0,0.000000,0.000000,none,BuyTransaction is currently a conceptual alias...



Query-edge sharedness:


,query_name,generic_class,query_family,edge_position,edge_id,relationship_name,source_entity,target_entity,semantic_type,observed_sharedness_score,observed_sharedness_level,target_sharedness_ratio,duplicate_reference_pressure,avg_sources_per_target,max_sources_per_target,hint_confidence,sharedness_status
0,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,1,Corporation -- Industry [corporation_has_indus...,corporation_has_industry,Corporation,Industry,descriptor,0.253787,medium,0.0,0.845955,1.0,1.0,high,computed
1,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,2,Corporation -- Country [corporation_has_country],corporation_has_country,Corporation,Country,descriptor,0.297418,medium,0.0,0.991394,1.0,1.0,high,computed
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,3,Corporation -- ListedSecurity [corporation_has...,corporation_has_listed_security,Corporation,ListedSecurity,association,0.000000,none,0.0,0.000000,NaN,NaN,medium,computed
3,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,1,ListedSecurity -- Security [listed_security_re...,listed_security_represents_security,ListedSecurity,Security,association,0.000000,none,0.0,0.000000,NaN,NaN,medium,computed
4,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,2,FinancialServiceAccount -- Holding [account_ha...,account_has_holding,FinancialServiceAccount,Holding,containment,0.245333,low,0.0,0.817775,1.0,1.0,high,computed
5,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,3,Holding -- ListedSecurity [holding_refers_to_l...,holding_refers_to_listed_security,Holding,ListedSecurity,association,0.295882,medium,0.0,0.986275,1.0,1.0,high,computed
6,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,1,Corporation -- ListedSecurity [corporation_has...,corporation_has_listed_security,Corporation,ListedSecurity,association,0.000000,none,0.0,0.000000,NaN,NaN,medium,computed
7,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,2,Person -- FinancialServiceAccount [person_owns...,person_owns_financial_service_account,Person,FinancialServiceAccount,ownership,0.145790,low,0.0,0.485967,1.0,1.0,high,computed
8,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,3,FinancialServiceAccount -- Holding [account_ha...,account_has_holding,FinancialServiceAccount,Holding,containment,0.245333,low,0.0,0.817775,1.0,1.0,high,computed
9,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,4,Holding -- ListedSecurity [holding_refers_to_l...,holding_refers_to_listed_security,Holding,ListedSecurity,association,0.295882,medium,0.0,0.986275,1.0,1.0,high,computed



Query sharedness profile:


...


Sharedness summary:


,observed_sharedness_level,semantic_type,n_relationships,relationships,avg_observed_sharedness_score,max_observed_sharedness_score
0,low,containment,2,"[account_has_holding, account_records_transact...",0.209038,0.245333
1,low,ownership,1,[person_owns_financial_service_account],0.145790,0.145790
2,medium,association,2,"[holding_refers_to_listed_security, transactio...",0.295882,0.295882
3,medium,containment,2,"[corporation_has_financial_report, financial_r...",0.290093,0.294621
4,medium,descriptor,2,"[corporation_has_country, corporation_has_indu...",0.275602,0.297418
5,none,association,2,"[corporation_has_listed_security, listed_secur...",0.000000,0.000000
6,none,containment,2,"[financial_report_contains_disclosure, report_...",0.000000,0.000000
7,none,subtype,2,"[buy_transaction_is_transaction, sell_transact...",0.000000,0.000000



Query sharedness summary:


,query_observed_sharedness_level,n_queries,queries,avg_max_observed_sharedness_score
0,medium,9,"[Q10_CreateAccountHoldingAndBuyTransaction, Q2...",0.296084
1,none,1,[Q1_CompanyProfileIBM],0.000000


Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v18/observed_sharedness_by_relationship.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v18/relationship_sharedness.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v18/sharedness_observed.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v18/query_edge_sharedness.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v18/query_sharedness_profile.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v18/sharedness_summary.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v18/quer

In [90]:
sharedness_not_computed_df

,edge_id,relationship_name,source_entity,target_entity,semantic_type,hint_type,source_view,target_view,bridge_view,source_column,...,max_sources_per_target,source_keys_in_join,source_keys_with_multiple_targets,avg_targets_per_source,max_targets_per_source,target_sharedness_ratio,duplicate_reference_pressure,observed_sharedness_score,observed_sharedness_level,notes


In [91]:
# =========================================================
# BLOCK V19 — INTEGRATE OBSERVED SHAREDNESS INTO THE ANALYTICAL MATRIX
# =========================================================
#
# Purpose:
# This block integrates observed sharedness variables into the
# main analytical matrix:
#
#     document_variable_matrix_df
#
# Inputs:
# - document_variable_matrix_df from BLOCK V17;
# - query_sharedness_profile_df from BLOCK V18;
# - observed_sharedness_df from BLOCK V18.
#
# Main outputs preserved for downstream compatibility:
# - document_variable_matrix_df
# - document_variable_matrix_with_sharedness_df
# - analytical_matrix_with_sharedness_df
#
# Additional outputs:
# - sharedness_integration_check_df
# - sharedness_matrix_summary_df
# - sharedness_tradeoff_summary_df
#
# Methodological role:
# Sharedness helps identify when embedding related entities may
# cause duplication because the same target entity is reused by
# many source entities.
#
# Example:
# If many Corporations share the same Country, embedding Country
# inside each Corporation document may duplicate Country data.
# =========================================================

import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "document_variable_matrix_df",
    "query_sharedness_profile_df",
    "observed_sharedness_df",
    "fiben_scenario",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first BLOCK V18 — EXTRACT SHAREDNESS OBSERVED IN THE FIBEN DATASET."
        )


# ---------------------------------------------------------
# 2. Select sharedness columns to integrate
# ---------------------------------------------------------
#
# These variables summarize observed sharedness at query level.
# ---------------------------------------------------------

sharedness_columns = [
    "query_name",
    "avg_observed_sharedness_score",
    "max_observed_sharedness_score",
    "sum_observed_sharedness_score",
    "n_edges_with_sharedness",
    "n_high_sharedness_edges",
    "shared_relationships",
    "shared_target_entities",
    "observed_sharedness_levels",
    "query_observed_sharedness_level",
    "has_observed_sharedness",
]

available_sharedness_columns = [
    col for col in sharedness_columns
    if col in query_sharedness_profile_df.columns
]

sharedness_integration_df = query_sharedness_profile_df[
    available_sharedness_columns
].copy()


# ---------------------------------------------------------
# 3. Avoid duplicate columns before merging
# ---------------------------------------------------------
#
# If this block is executed more than once, sharedness columns may
# already exist in document_variable_matrix_df.
#
# We remove them before merging again to avoid _x and _y suffixes.
# ---------------------------------------------------------

cols_to_replace = [
    col for col in available_sharedness_columns
    if col != "query_name" and col in document_variable_matrix_df.columns
]

document_variable_matrix_without_old_sharedness_df = (
    document_variable_matrix_df
    .drop(columns=cols_to_replace, errors="ignore")
    .copy()
)


# ---------------------------------------------------------
# 4. Merge sharedness into the analytical matrix
# ---------------------------------------------------------

document_variable_matrix_with_sharedness_df = (
    document_variable_matrix_without_old_sharedness_df
    .merge(
        sharedness_integration_df,
        on="query_name",
        how="left",
    )
)


# ---------------------------------------------------------
# 5. Fill missing sharedness values safely
# ---------------------------------------------------------

numeric_sharedness_cols = [
    "avg_observed_sharedness_score",
    "max_observed_sharedness_score",
    "sum_observed_sharedness_score",
    "n_edges_with_sharedness",
    "n_high_sharedness_edges",
]

for col in numeric_sharedness_cols:
    if col in document_variable_matrix_with_sharedness_df.columns:
        document_variable_matrix_with_sharedness_df[col] = (
            document_variable_matrix_with_sharedness_df[col]
            .fillna(0)
        )

list_sharedness_cols = [
    "shared_relationships",
    "shared_target_entities",
    "observed_sharedness_levels",
]

for col in list_sharedness_cols:
    if col in document_variable_matrix_with_sharedness_df.columns:
        document_variable_matrix_with_sharedness_df[col] = (
            document_variable_matrix_with_sharedness_df[col]
            .apply(lambda v: [] if isinstance(v, float) and pd.isna(v) else v)
        )

if "query_observed_sharedness_level" in document_variable_matrix_with_sharedness_df.columns:
    document_variable_matrix_with_sharedness_df["query_observed_sharedness_level"] = (
        document_variable_matrix_with_sharedness_df["query_observed_sharedness_level"]
        .fillna("none")
    )

if "has_observed_sharedness" in document_variable_matrix_with_sharedness_df.columns:
    document_variable_matrix_with_sharedness_df["has_observed_sharedness"] = (
        document_variable_matrix_with_sharedness_df["has_observed_sharedness"]
        .fillna(False)
        .astype(bool)
    )


# ---------------------------------------------------------
# 6. Add sharedness interpretation variables
# ---------------------------------------------------------

def classify_sharedness_embedding_risk(row):
    """
    Classify the interaction between document embedding potential
    and observed sharedness.

    High DeltaRratio means the document design may reduce reads.
    High sharedness means embedding may create duplication.
    """
    delta_ratio = float(row.get("DeltaRratio", 0) or 0)
    sharedness_score = float(row.get("max_observed_sharedness_score", 0) or 0)

    if sharedness_score == 0:
        return "no_sharedness_pressure"

    if delta_ratio == 0:
        return "sharedness_without_read_reduction"

    if delta_ratio >= 0.75 and sharedness_score >= 0.60:
        return "high_read_gain_high_duplication_risk"

    if delta_ratio >= 0.75 and sharedness_score < 0.60:
        return "high_read_gain_manageable_sharedness"

    if delta_ratio < 0.75 and sharedness_score >= 0.60:
        return "limited_read_gain_high_duplication_risk"

    return "moderate_sharedness_tradeoff"


def classify_combined_document_design_risk(row):
    """
    Combine read gain, update volatility, and sharedness into one
    qualitative interpretation.

    This is not the final activation class.
    It is an auxiliary interpretation for analysis.
    """
    delta_ratio = float(row.get("DeltaRratio", 0) or 0)
    update_score = float(row.get("query_update_volatility_score", 0) or 0)
    sharedness_score = float(row.get("max_observed_sharedness_score", 0) or 0)

    high_read_gain = delta_ratio >= 0.75
    high_update = update_score >= 0.60
    high_sharedness = sharedness_score >= 0.60

    if not high_read_gain and not high_update and not high_sharedness:
        return "low_overall_design_pressure"

    if high_read_gain and not high_update and not high_sharedness:
        return "strong_read_candidate_low_maintenance_pressure"

    if high_read_gain and high_update and high_sharedness:
        return "strong_read_candidate_high_update_and_duplication_pressure"

    if high_read_gain and high_update:
        return "strong_read_candidate_high_update_pressure"

    if high_read_gain and high_sharedness:
        return "strong_read_candidate_high_duplication_pressure"

    if high_update and high_sharedness:
        return "maintenance_pressure_without_strong_read_gain"

    if high_update:
        return "update_pressure"

    if high_sharedness:
        return "duplication_pressure"

    return "moderate_design_pressure"


def classify_sharedness_presence(row):
    """
    Compact binary label for observed sharedness presence.
    """
    if bool(row.get("has_observed_sharedness", False)):
        return "has_observed_sharedness"

    return "no_observed_sharedness"


document_variable_matrix_with_sharedness_df["sharedness_embedding_tradeoff_class"] = (
    document_variable_matrix_with_sharedness_df
    .apply(classify_sharedness_embedding_risk, axis=1)
)

document_variable_matrix_with_sharedness_df["combined_document_design_risk_class"] = (
    document_variable_matrix_with_sharedness_df
    .apply(classify_combined_document_design_risk, axis=1)
)

document_variable_matrix_with_sharedness_df["observed_sharedness_presence"] = (
    document_variable_matrix_with_sharedness_df
    .apply(classify_sharedness_presence, axis=1)
)


# ---------------------------------------------------------
# 7. Update main downstream aliases
# ---------------------------------------------------------

document_variable_matrix_df = document_variable_matrix_with_sharedness_df.copy()

analytical_matrix_with_sharedness_df = (
    document_variable_matrix_with_sharedness_df.copy()
)

document_variable_matrix_sharedness_enriched_df = (
    document_variable_matrix_with_sharedness_df.copy()
)


# ---------------------------------------------------------
# 8. Build integration check table
# ---------------------------------------------------------

expected_query_names = set(
    document_variable_matrix_without_old_sharedness_df["query_name"].tolist()
)

sharedness_query_names = set(
    query_sharedness_profile_df["query_name"].tolist()
)

merged_query_names = set(
    document_variable_matrix_with_sharedness_df["query_name"].tolist()
)

missing_from_sharedness = sorted(expected_query_names - sharedness_query_names)
extra_in_sharedness = sorted(sharedness_query_names - expected_query_names)
missing_after_merge = sorted(expected_query_names - merged_query_names)

sharedness_integration_check_rows = [
    {
        "check_name": "all_matrix_queries_have_sharedness_source",
        "status": "passed" if not missing_from_sharedness else "failed",
        "details": missing_from_sharedness,
    },
    {
        "check_name": "no_extra_sharedness_queries",
        "status": "passed" if not extra_in_sharedness else "warning",
        "details": extra_in_sharedness,
    },
    {
        "check_name": "no_queries_lost_after_merge",
        "status": "passed" if not missing_after_merge else "failed",
        "details": missing_after_merge,
    },
    {
        "check_name": "row_count_preserved",
        "status": (
            "passed"
            if len(document_variable_matrix_without_old_sharedness_df)
            == len(document_variable_matrix_with_sharedness_df)
            else "failed"
        ),
        "details": {
            "before": len(document_variable_matrix_without_old_sharedness_df),
            "after": len(document_variable_matrix_with_sharedness_df),
        },
    },
]

sharedness_integration_check_df = pd.DataFrame(
    sharedness_integration_check_rows
)


# ---------------------------------------------------------
# 9. Build summary tables
# ---------------------------------------------------------

sharedness_matrix_summary_df = (
    document_variable_matrix_with_sharedness_df
    .groupby(
        [
            "query_observed_sharedness_level",
            "sharedness_embedding_tradeoff_class",
        ],
        dropna=False,
    )
    .agg(
        n_queries=("query_name", "count"),
        queries=("query_name", lambda s: sorted(set(s))),
        avg_DeltaRratio=("DeltaRratio", "mean"),
        avg_max_observed_sharedness_score=(
            "max_observed_sharedness_score",
            "mean",
        ),
        avg_query_update_volatility_score=(
            "query_update_volatility_score",
            "mean",
        ) if "query_update_volatility_score" in document_variable_matrix_with_sharedness_df.columns else ("DeltaRratio", "mean"),
    )
    .reset_index()
    .sort_values(
        [
            "query_observed_sharedness_level",
            "sharedness_embedding_tradeoff_class",
        ]
    )
    .reset_index(drop=True)
)

sharedness_tradeoff_summary_df = (
    document_variable_matrix_with_sharedness_df
    .groupby(
        [
            "document_potential_class",
            "query_observed_sharedness_level",
            "query_update_volatility_level",
            "combined_document_design_risk_class",
        ],
        dropna=False,
    )
    .agg(
        n_queries=("query_name", "count"),
        queries=("query_name", lambda s: sorted(set(s))),
        avg_DeltaRratio=("DeltaRratio", "mean"),
        avg_sharedness_score=("max_observed_sharedness_score", "mean"),
    )
    .reset_index()
    .sort_values(
        [
            "document_potential_class",
            "query_observed_sharedness_level",
            "query_update_volatility_level",
        ]
    )
    .reset_index(drop=True)
)

sharedness_by_root_summary_df = (
    document_variable_matrix_with_sharedness_df
    .groupby(
        [
            "selected_root",
            "query_observed_sharedness_level",
        ],
        dropna=False,
    )
    .agg(
        n_queries=("query_name", "count"),
        queries=("query_name", lambda s: sorted(set(s))),
        avg_DeltaRratio=("DeltaRratio", "mean"),
        avg_sharedness_score=("max_observed_sharedness_score", "mean"),
    )
    .reset_index()
    .sort_values(["selected_root", "query_observed_sharedness_level"])
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 10. Validation tables
# ---------------------------------------------------------

missing_sharedness_merge_df = (
    document_variable_matrix_with_sharedness_df[
        document_variable_matrix_with_sharedness_df[
            "max_observed_sharedness_score"
        ].isna()
    ].copy()
    if "max_observed_sharedness_score" in document_variable_matrix_with_sharedness_df.columns
    else document_variable_matrix_with_sharedness_df.copy()
)

failed_sharedness_integration_checks_df = sharedness_integration_check_df[
    sharedness_integration_check_df["status"] == "failed"
].copy()

high_read_gain_high_sharedness_df = document_variable_matrix_with_sharedness_df[
    document_variable_matrix_with_sharedness_df[
        "sharedness_embedding_tradeoff_class"
    ].isin([
        "high_read_gain_high_duplication_risk",
        "strong_read_candidate_high_duplication_pressure",
    ])
].copy()


# ---------------------------------------------------------
# 11. Display outputs
# ---------------------------------------------------------

print("Observed sharedness integrated into the analytical matrix successfully.")

print("\nDocument variable matrix with observed sharedness:")
display(document_variable_matrix_with_sharedness_df)

print("\nSharedness integration checks:")
display(sharedness_integration_check_df)

print("\nSharedness matrix summary:")
display(sharedness_matrix_summary_df)

print("\nSharedness tradeoff summary:")
display(sharedness_tradeoff_summary_df)

print("\nSharedness by selected root:")
display(sharedness_by_root_summary_df)

if not failed_sharedness_integration_checks_df.empty:
    print("\nWarning: some sharedness integration checks failed.")
    display(failed_sharedness_integration_checks_df)

if not missing_sharedness_merge_df.empty:
    print("\nWarning: some rows are missing sharedness values.")
    display(missing_sharedness_merge_df)

if not high_read_gain_high_sharedness_df.empty:
    print("\nQueries with high read gain and high sharedness pressure:")
    display(
        high_read_gain_high_sharedness_df[
            [
                "query_name",
                "selected_root",
                "DeltaRratio",
                "max_observed_sharedness_score",
                "query_observed_sharedness_level",
                "shared_target_entities",
                "sharedness_embedding_tradeoff_class",
            ]
        ]
    )


# ---------------------------------------------------------
# 12. Save BLOCK V19 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    document_variable_matrix_with_sharedness_df,
    "variables/block_v19/document_variable_matrix_with_sharedness.csv",
)

save_dataframe(
    analytical_matrix_with_sharedness_df,
    "variables/block_v19/analytical_matrix_with_sharedness.csv",
)

save_dataframe(
    document_variable_matrix_sharedness_enriched_df,
    "variables/block_v19/document_variable_matrix_sharedness_enriched.csv",
)

save_dataframe(
    document_variable_matrix_df,
    "variables/block_v19/document_variable_matrix.csv",
)

save_dataframe(
    sharedness_integration_check_df,
    "variables/block_v19/sharedness_integration_check.csv",
)

save_dataframe(
    sharedness_matrix_summary_df,
    "variables/block_v19/sharedness_matrix_summary.csv",
)

save_dataframe(
    sharedness_tradeoff_summary_df,
    "variables/block_v19/sharedness_tradeoff_summary.csv",
)

save_dataframe(
    sharedness_by_root_summary_df,
    "variables/block_v19/sharedness_by_root_summary.csv",
)

save_dataframe(
    missing_sharedness_merge_df,
    "variables/block_v19/missing_sharedness_merge.csv",
)

save_dataframe(
    failed_sharedness_integration_checks_df,
    "variables/block_v19/failed_sharedness_integration_checks.csv",
)

save_dataframe(
    high_read_gain_high_sharedness_df,
    "variables/block_v19/high_read_gain_high_sharedness_queries.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "n_rows_before_merge": len(document_variable_matrix_without_old_sharedness_df),
        "n_rows_after_merge": len(document_variable_matrix_with_sharedness_df),
        "n_columns_after_merge": len(document_variable_matrix_with_sharedness_df.columns),
        "n_queries_with_observed_sharedness": int(
            document_variable_matrix_with_sharedness_df[
                "has_observed_sharedness"
            ].sum()
        ),
        "sharedness_level_counts": (
            document_variable_matrix_with_sharedness_df[
                "query_observed_sharedness_level"
            ]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "sharedness_embedding_tradeoff_counts": (
            document_variable_matrix_with_sharedness_df[
                "sharedness_embedding_tradeoff_class"
            ]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "combined_document_design_risk_counts": (
            document_variable_matrix_with_sharedness_df[
                "combined_document_design_risk_class"
            ]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "n_failed_integration_checks": len(failed_sharedness_integration_checks_df),
        "output_files": {
            "document_variable_matrix_with_sharedness_csv": "variables/block_v19/document_variable_matrix_with_sharedness.csv",
            "analytical_matrix_with_sharedness_csv": "variables/block_v19/analytical_matrix_with_sharedness.csv",
            "document_variable_matrix_sharedness_enriched_csv": "variables/block_v19/document_variable_matrix_sharedness_enriched.csv",
            "document_variable_matrix_csv": "variables/block_v19/document_variable_matrix.csv",
            "integration_check_csv": "variables/block_v19/sharedness_integration_check.csv",
            "matrix_summary_csv": "variables/block_v19/sharedness_matrix_summary.csv",
            "tradeoff_summary_csv": "variables/block_v19/sharedness_tradeoff_summary.csv",
            "by_root_summary_csv": "variables/block_v19/sharedness_by_root_summary.csv",
            "missing_merge_csv": "variables/block_v19/missing_sharedness_merge.csv",
            "failed_checks_csv": "variables/block_v19/failed_sharedness_integration_checks.csv",
            "high_read_gain_high_sharedness_csv": "variables/block_v19/high_read_gain_high_sharedness_queries.csv",
        },
    },
    "variables/block_v19/sharedness_integration_metadata.json",
)

write_execution_log(
    block_name="BLOCK V19 — INTEGRATE OBSERVED SHAREDNESS INTO THE ANALYTICAL MATRIX",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_rows_before_merge": len(document_variable_matrix_without_old_sharedness_df),
        "n_rows_after_merge": len(document_variable_matrix_with_sharedness_df),
        "n_columns_after_merge": len(document_variable_matrix_with_sharedness_df.columns),
        "n_failed_integration_checks": len(failed_sharedness_integration_checks_df),
        "document_variable_matrix_with_sharedness_csv": "variables/block_v19/document_variable_matrix_with_sharedness.csv",
        "sharedness_integration_check_csv": "variables/block_v19/sharedness_integration_check.csv",
        "metadata_json": "variables/block_v19/sharedness_integration_metadata.json",
    },
)


# ---------------------------------------------------------
# 13. Update README section for BLOCK V19
# ---------------------------------------------------------

block_v19_readme_body = f"""
This block integrates observed sharedness variables into the main analytical
matrix.

Main variables created or updated:

    document_variable_matrix_df
    document_variable_matrix_with_sharedness_df
    analytical_matrix_with_sharedness_df

Additional variables created:

    document_variable_matrix_sharedness_enriched_df
    sharedness_integration_check_df
    sharedness_matrix_summary_df
    sharedness_tradeoff_summary_df
    sharedness_by_root_summary_df
    missing_sharedness_merge_df
    failed_sharedness_integration_checks_df
    high_read_gain_high_sharedness_df

Rows before merge:

    {len(document_variable_matrix_without_old_sharedness_df)}

Rows after merge:

    {len(document_variable_matrix_with_sharedness_df)}

Columns after merge:

    {len(document_variable_matrix_with_sharedness_df.columns)}

Queries with observed sharedness:

    {int(document_variable_matrix_with_sharedness_df["has_observed_sharedness"].sum())}

Failed integration checks:

    {len(failed_sharedness_integration_checks_df)}

Generated reproducibility files:

    variables/block_v19/document_variable_matrix_with_sharedness.csv
    variables/block_v19/analytical_matrix_with_sharedness.csv
    variables/block_v19/document_variable_matrix_sharedness_enriched.csv
    variables/block_v19/document_variable_matrix.csv
    variables/block_v19/sharedness_integration_check.csv
    variables/block_v19/sharedness_matrix_summary.csv
    variables/block_v19/sharedness_tradeoff_summary.csv
    variables/block_v19/sharedness_by_root_summary.csv
    variables/block_v19/missing_sharedness_merge.csv
    variables/block_v19/failed_sharedness_integration_checks.csv
    variables/block_v19/high_read_gain_high_sharedness_queries.csv
    variables/block_v19/sharedness_integration_metadata.json

Methodological meaning:

    Observed sharedness helps identify possible duplication pressure in
    document configurations.

    If a target entity is shared by many source entities, embedding that
    target inside many documents may duplicate data.

Important interpretation:

    A query with high DeltaRratio and high sharedness may still be a good
    read-optimization candidate, but it requires careful benchmark evaluation
    because of possible duplication and update-maintenance cost.
"""

update_readme_section(
    section_title="BLOCK V19 — Integrate Observed Sharedness into the Analytical Matrix",
    section_body=block_v19_readme_body,
)

Observed sharedness integrated into the analytical matrix successfully.

Document variable matrix with observed sharedness:


,query_name,generic_class,query_family,query_type,abstract_query,selected_root,document_depth,selected_document_depth,best_document_depth,max_depth_for_query,...,n_edges_with_sharedness,n_high_sharedness_edges,shared_relationships,shared_target_entities,observed_sharedness_levels,query_observed_sharedness_level,has_observed_sharedness,sharedness_embedding_tradeoff_class,combined_document_design_risk_class,observed_sharedness_presence
0,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,Create a financial service account for a perso...,Person,3,3,3,3,...,4.0,0.0,"[account_has_holding, account_records_transact...","[FinancialServiceAccount, Holding, ListedSecur...","[low, medium, none]",medium,True,high_read_gain_manageable_sharedness,strong_read_candidate_high_update_pressure,has_observed_sharedness
1,Q1_CompanyProfileIBM,QG1,Local lookup,select,Show the profile of company IBM.,Corporation,0,0,0,0,...,0.0,0.0,[],[],[],none,False,no_sharedness_pressure,low_overall_design_pressure,no_observed_sharedness
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,"Show IBM with its industry, country, and liste...",Corporation,1,1,1,1,...,2.0,0.0,"[corporation_has_country, corporation_has_indu...","[Country, Industry, ListedSecurity]","[medium, none]",medium,True,high_read_gain_manageable_sharedness,strong_read_candidate_low_maintenance_pressure,has_observed_sharedness
3,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,Show the securities held in each financial ser...,FinancialServiceAccount,3,3,3,3,...,2.0,0.0,"[account_has_holding, holding_refers_to_listed...","[Holding, ListedSecurity, Security]","[low, medium, none]",medium,True,high_read_gain_manageable_sharedness,strong_read_candidate_high_update_pressure,has_observed_sharedness
4,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Show the companies reached from a person throu...,Person,4,4,4,4,...,3.0,0.0,"[account_has_holding, corporation_has_listed_s...","[FinancialServiceAccount, Holding, ListedSecur...","[low, medium, none]",medium,True,high_read_gain_manageable_sharedness,strong_read_candidate_high_update_pressure,has_observed_sharedness
5,Q5_ReportsAndMetricDataOfCompany,QG5,Containment retrieval,select,Show the financial reports of a company and th...,Corporation,3,3,3,3,...,2.0,0.0,"[corporation_has_financial_report, financial_r...","[Disclosure, FinancialReport, ReportElement, S...","[medium, none]",medium,True,high_read_gain_manageable_sharedness,strong_read_candidate_low_maintenance_pressure,has_observed_sharedness
6,Q6_TechUSListedSecuritiesWithHighLastTradedValue,QG6,Filtered search / recommendation,select,Find listed securities of technology companies...,ListedSecurity,2,2,2,2,...,2.0,0.0,"[corporation_has_country, corporation_has_indu...","[Country, Industry, ListedSecurity]","[medium, none]",medium,True,high_read_gain_manageable_sharedness,strong_read_candidate_low_maintenance_pressure,has_observed_sharedness
7,Q7_PersonsWhoBoughtMoreIBMThanSold,QG7,Aggregation / ranking,select,Who bought more IBM stocks than they sold?,Person,4,4,4,4,...,3.0,0.0,"[account_records_transaction, buy_transaction_...","[FinancialServiceAccount, ListedSecurity, Tran...","[low, medium, none]",medium,True,high_read_gain_manageable_sharedness,strong_read_candidate_high_update_pressure,has_observed_sharedness
8,Q8_IBMTransactionsBelowAverageSellingPrice,QG8,Aggregation / ranking,select,Show me each transaction for IBM whose price i...,Transaction,2,2,2,2,...,1.0,0.0,"[corporation_has_listed_security, sell_transac...","[ListedSecurity, Transaction]","[medium, none]",medium,True,high_read_gain_manageable_sharedness,strong_read_candidate_high_update_pressure,has_observed_sharedness
9,Q9_PersonsWhoBoughtAndSoldSameStock,QG9,Associative retrieval + comparison,select,Show me everyone who bought and sold the same ...,Person,3,3,3,3,...,3.0,0.0,"[account_records_


Sharedness integration checks:


,check_name,status,details
0,all_matrix_queries_have_sharedness_source,passed,[]
1,no_extra_sharedness_queries,passed,[]
2,no_queries_lost_after_merge,passed,[]
3,row_count_preserved,passed,"{'before': 10, 'after': 10}"



Sharedness matrix summary:


...


Sharedness tradeoff summary:


,document_potential_class,query_observed_sharedness_level,query_update_volatility_level,combined_document_design_risk_class,n_queries,queries,avg_DeltaRratio,avg_sharedness_score
0,high_reduction_potential,medium,high,strong_read_candidate_high_update_pressure,6,"[Q10_CreateAccountHoldingAndBuyTransaction, Q3...",1.0,0.295882
1,high_reduction_potential,medium,medium,strong_read_candidate_low_maintenance_pressure,2,[Q2_CompanyWithIndustryCountryAndListedSecurit...,1.0,0.297418
2,high_reduction_potential,medium,none,strong_read_candidate_low_maintenance_pressure,1,[Q5_ReportsAndMetricDataOfCompany],1.0,0.294621
3,no_relationship_traversal,none,none,low_overall_design_pressure,1,[Q1_CompanyProfileIBM],0.0,0.000000



Sharedness by selected root:


,selected_root,query_observed_sharedness_level,n_queries,queries,avg_DeltaRratio,avg_sharedness_score
0,Corporation,medium,2,[Q2_CompanyWithIndustryCountryAndListedSecurit...,1.0,0.296020
1,Corporation,none,1,[Q1_CompanyProfileIBM],0.0,0.000000
2,FinancialServiceAccount,medium,1,[Q3_SecuritiesHeldInEachFinancialServiceAccount],1.0,0.295882
3,ListedSecurity,medium,1,[Q6_TechUSListedSecuritiesWithHighLastTradedVa...,1.0,0.297418
4,Person,medium,4,"[Q10_CreateAccountHoldingAndBuyTransaction, Q4...",1.0,0.295882
5,Transaction,medium,1,[Q8_IBMTransactionsBelowAverageSellingPrice],1.0,0.295882


Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v19/document_variable_matrix_with_sharedness.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v19/analytical_matrix_with_sharedness.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v19/document_variable_matrix_sharedness_enriched.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v19/document_variable_matrix.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v19/sharedness_integration_check.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v19/sharedness_matrix_summary.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs

In [92]:
sharedness_integration_check_df

,check_name,status,details
0,all_matrix_queries_have_sharedness_source,passed,[]
1,no_extra_sharedness_queries,passed,[]
2,no_queries_lost_after_merge,passed,[]
3,row_count_preserved,passed,"{'before': 10, 'after': 10}"


In [93]:
# =========================================================
# BLOCK V20 — FINAL DOCUMENT VARIABLE MATRIX FOR ACTIVATION
# =========================================================
#
# Purpose:
# This block consolidates the final document variable matrix
# used as input for the generic activation logic G0–G9.
#
# Input:
# - document_variable_matrix_df from BLOCK V19.
#
# Main outputs preserved for downstream compatibility:
# - final_document_variable_matrix_df
# - document_variable_matrix_for_activation_df
# - activation_input_matrix_df
#
# Additional outputs:
# - activation_feature_matrix_df
# - activation_numeric_features_df
# - activation_boolean_features_df
# - activation_matrix_summary_df
#
# Methodological role:
# This block creates a clean and stable activation-ready matrix.
# It does not activate G0–G9 yet.
# It only prepares the variables that the activation logic will use.
# =========================================================

import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "document_variable_matrix_df",
    "fiben_scenario",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first BLOCK V19 — INTEGRATE OBSERVED SHAREDNESS INTO THE ANALYTICAL MATRIX."
        )


# ---------------------------------------------------------
# 2. Start from the current analytical matrix
# ---------------------------------------------------------

final_document_variable_matrix_df = document_variable_matrix_df.copy()


# ---------------------------------------------------------
# 3. Helper functions
# ---------------------------------------------------------

def safe_numeric(value, default=0.0):
    """
    Convert a value to float safely.

    Missing values are replaced by default.
    """
    try:
        if pd.isna(value):
            return float(default)
    except Exception:
        pass

    try:
        return float(value)
    except Exception:
        return float(default)


def safe_bool(value, default=False):
    """
    Convert a value to bool safely.
    """
    try:
        if pd.isna(value):
            return bool(default)
    except Exception:
        pass

    return bool(value)


def ensure_list_value(value):
    """
    Convert missing values to empty lists.

    This keeps list-like analytical columns stable.
    """
    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    if value is None:
        return []

    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    return value


def classify_binary_pressure(score, threshold=0.60):
    """
    Convert a numeric score into a binary pressure flag.
    """
    return safe_numeric(score) >= threshold


def classify_strength_from_ratio(ratio):
    """
    Convert DeltaRratio into a qualitative read-gain level.
    """
    ratio = safe_numeric(ratio)

    if ratio == 0:
        return "none"

    if ratio < 0.25:
        return "low"

    if ratio < 0.60:
        return "medium"

    if ratio < 1.0:
        return "high"

    return "full"


# ---------------------------------------------------------
# 4. Normalize core numeric variables
# ---------------------------------------------------------

core_numeric_defaults = {
    "Rc": 0.0,
    "Re": 0.0,
    "DeltaR": 0.0,
    "DeltaRratio": 0.0,
    "document_depth": 0.0,
    "selected_document_depth": 0.0,
    "max_depth_for_query": 0.0,
    "document_depth_ratio": 0.0,
    "n_entities_touched": 0.0,
    "n_reachable_entities": 0.0,
    "n_total_edges": 0.0,
    "n_covered_edges": 0.0,
    "n_remaining_edges": 0.0,
    "query_update_volatility_score": 0.0,
    "avg_observed_sharedness_score": 0.0,
    "max_observed_sharedness_score": 0.0,
    "sum_observed_sharedness_score": 0.0,
    "n_edges_with_sharedness": 0.0,
    "n_high_sharedness_edges": 0.0,
}

for col, default in core_numeric_defaults.items():
    if col not in final_document_variable_matrix_df.columns:
        final_document_variable_matrix_df[col] = default

    final_document_variable_matrix_df[col] = (
        final_document_variable_matrix_df[col]
        .apply(lambda v: safe_numeric(v, default))
    )


# ---------------------------------------------------------
# 5. Normalize important boolean variables
# ---------------------------------------------------------

core_boolean_defaults = {
    "is_full_coverage": False,
    "relationship_reduction_potential": False,
    "has_low_confidence_mapping": False,
    "has_no_match_cardinality": False,
    "is_write_query": False,
    "has_query_update_volatility": False,
    "has_observed_sharedness": False,
}

for col, default in core_boolean_defaults.items():
    if col not in final_document_variable_matrix_df.columns:
        final_document_variable_matrix_df[col] = default

    final_document_variable_matrix_df[col] = (
        final_document_variable_matrix_df[col]
        .apply(lambda v: safe_bool(v, default))
    )


# ---------------------------------------------------------
# 6. Normalize important list-like variables
# ---------------------------------------------------------

list_like_columns = [
    "entities_touched",
    "reachable_entities",
    "covered_relationships",
    "remaining_relationships",
    "covered_semantic_types",
    "remaining_semantic_types",
    "semantic_types_touched",
    "observed_cardinalities_touched",
    "hint_confidences_touched",
    "created_entities",
    "referenced_existing_entities",
    "all_write_related_entities",
    "touched_entities_with_update_volatility",
    "write_related_entities_with_update_volatility",
    "shared_relationships",
    "shared_target_entities",
    "observed_sharedness_levels",
]

for col in list_like_columns:
    if col in final_document_variable_matrix_df.columns:
        final_document_variable_matrix_df[col] = (
            final_document_variable_matrix_df[col]
            .apply(ensure_list_value)
        )


# ---------------------------------------------------------
# 7. Ensure semantic touch columns exist
# ---------------------------------------------------------
#
# These boolean variables are important for the activation logic.
# If a semantic flag is missing, we reconstruct it from
# semantic_types_touched when possible.
# ---------------------------------------------------------

expected_semantic_types = [
    "descriptor",
    "association",
    "ownership",
    "containment",
    "subtype",
]

for semantic_type in expected_semantic_types:
    flag_col = f"touches_{semantic_type}"
    count_col = f"n_{semantic_type}_edges"

    if flag_col not in final_document_variable_matrix_df.columns:
        if "semantic_types_touched" in final_document_variable_matrix_df.columns:
            final_document_variable_matrix_df[flag_col] = (
                final_document_variable_matrix_df["semantic_types_touched"]
                .apply(
                    lambda values: (
                        semantic_type in values
                        if isinstance(values, list)
                        else False
                    )
                )
            )
        else:
            final_document_variable_matrix_df[flag_col] = False

    final_document_variable_matrix_df[flag_col] = (
        final_document_variable_matrix_df[flag_col]
        .apply(lambda v: safe_bool(v, False))
    )

    if count_col not in final_document_variable_matrix_df.columns:
        final_document_variable_matrix_df[count_col] = (
            final_document_variable_matrix_df[flag_col].astype(int)
        )

    final_document_variable_matrix_df[count_col] = (
        final_document_variable_matrix_df[count_col]
        .apply(lambda v: int(safe_numeric(v, 0)))
    )


# ---------------------------------------------------------
# 8. Create activation-ready derived variables
# ---------------------------------------------------------
#
# These variables are compact features for the activation logic.
# They are intentionally generic and not tied to MongoDB syntax yet.
# ---------------------------------------------------------

final_document_variable_matrix_df["activation_has_relationship_traversal"] = (
    final_document_variable_matrix_df["Rc"] > 0
)

final_document_variable_matrix_df["activation_has_read_reduction"] = (
    final_document_variable_matrix_df["DeltaR"] > 0
)

final_document_variable_matrix_df["activation_full_relationship_reduction"] = (
    final_document_variable_matrix_df["DeltaRratio"] >= 1.0
)

final_document_variable_matrix_df["activation_high_read_gain"] = (
    final_document_variable_matrix_df["DeltaRratio"] >= 0.75
)

final_document_variable_matrix_df["activation_medium_or_high_read_gain"] = (
    final_document_variable_matrix_df["DeltaRratio"] >= 0.50
)

final_document_variable_matrix_df["activation_has_depth_greater_than_zero"] = (
    final_document_variable_matrix_df["document_depth"] > 0
)

final_document_variable_matrix_df["activation_has_multi_entity_query"] = (
    final_document_variable_matrix_df["n_entities_touched"] > 1
)

final_document_variable_matrix_df["activation_has_remaining_external_edges"] = (
    final_document_variable_matrix_df["Re"] > 0
)

final_document_variable_matrix_df["activation_has_update_pressure"] = (
    final_document_variable_matrix_df["query_update_volatility_score"] > 0
)

final_document_variable_matrix_df["activation_high_update_pressure"] = (
    final_document_variable_matrix_df["query_update_volatility_score"] >= 0.60
)

final_document_variable_matrix_df["activation_has_sharedness_pressure"] = (
    final_document_variable_matrix_df["max_observed_sharedness_score"] > 0
)

final_document_variable_matrix_df["activation_high_sharedness_pressure"] = (
    final_document_variable_matrix_df["max_observed_sharedness_score"] >= 0.60
)

final_document_variable_matrix_df["activation_has_mapping_warning"] = (
    final_document_variable_matrix_df["has_low_confidence_mapping"]
    | final_document_variable_matrix_df["has_no_match_cardinality"]
)


# ---------------------------------------------------------
# 9. Create compact qualitative classes
# ---------------------------------------------------------

final_document_variable_matrix_df["activation_read_gain_level"] = (
    final_document_variable_matrix_df["DeltaRratio"]
    .apply(classify_strength_from_ratio)
)

final_document_variable_matrix_df["activation_update_pressure_level"] = (
    final_document_variable_matrix_df["query_update_volatility_score"]
    .apply(classify_strength_from_ratio)
)

final_document_variable_matrix_df["activation_sharedness_pressure_level"] = (
    final_document_variable_matrix_df["max_observed_sharedness_score"]
    .apply(classify_strength_from_ratio)
)


# ---------------------------------------------------------
# 10. Create activation input matrix
# ---------------------------------------------------------

activation_input_columns = [
    # Identification
    "query_name",
    "generic_class",
    "query_family",
    "query_type",
    "abstract_query",

    # Root and depth
    "selected_root",
    "document_depth",
    "selected_document_depth",
    "max_depth_for_query",
    "document_depth_ratio",

    # Relationship reduction variables
    "Rc",
    "Re",
    "DeltaR",
    "DeltaRratio",
    "activation_has_relationship_traversal",
    "activation_has_read_reduction",
    "activation_full_relationship_reduction",
    "activation_high_read_gain",
    "activation_medium_or_high_read_gain",
    "activation_has_remaining_external_edges",

    # Entity and path variables
    "n_entities_touched",
    "n_reachable_entities",
    "n_total_edges",
    "n_covered_edges",
    "n_remaining_edges",
    "entities_touched",
    "reachable_entities",

    # Semantic profile
    "touches_descriptor",
    "touches_association",
    "touches_ownership",
    "touches_containment",
    "touches_subtype",
    "n_descriptor_edges",
    "n_association_edges",
    "n_ownership_edges",
    "n_containment_edges",
    "n_subtype_edges",
    "semantic_types_touched",
    "covered_semantic_types",
    "remaining_semantic_types",

    # Update volatility
    "is_write_query",
    "query_update_volatility_score",
    "query_update_volatility_level",
    "has_query_update_volatility",
    "activation_has_update_pressure",
    "activation_high_update_pressure",
    "activation_update_pressure_level",

    # Observed sharedness
    "avg_observed_sharedness_score",
    "max_observed_sharedness_score",
    "query_observed_sharedness_level",
    "has_observed_sharedness",
    "activation_has_sharedness_pressure",
    "activation_high_sharedness_pressure",
    "activation_sharedness_pressure_level",
    "shared_target_entities",

    # Risk and interpretation
    "document_potential_class",
    "physical_mapping_reliability",
    "update_embedding_tradeoff_class",
    "sharedness_embedding_tradeoff_class",
    "combined_document_design_risk_class",
    "activation_read_gain_level",
    "activation_has_mapping_warning",
]

available_activation_input_columns = [
    col for col in activation_input_columns
    if col in final_document_variable_matrix_df.columns
]

activation_input_matrix_df = final_document_variable_matrix_df[
    available_activation_input_columns
].copy()


# ---------------------------------------------------------
# 11. Create numeric and boolean feature subsets
# ---------------------------------------------------------

activation_numeric_feature_columns = [
    "Rc",
    "Re",
    "DeltaR",
    "DeltaRratio",
    "document_depth",
    "selected_document_depth",
    "max_depth_for_query",
    "document_depth_ratio",
    "n_entities_touched",
    "n_reachable_entities",
    "n_total_edges",
    "n_covered_edges",
    "n_remaining_edges",
    "n_descriptor_edges",
    "n_association_edges",
    "n_ownership_edges",
    "n_containment_edges",
    "n_subtype_edges",
    "query_update_volatility_score",
    "avg_observed_sharedness_score",
    "max_observed_sharedness_score",
    "sum_observed_sharedness_score",
    "n_edges_with_sharedness",
    "n_high_sharedness_edges",
]

available_numeric_feature_columns = [
    col for col in activation_numeric_feature_columns
    if col in final_document_variable_matrix_df.columns
]

activation_numeric_features_df = final_document_variable_matrix_df[
    ["query_name"] + available_numeric_feature_columns
].copy()


activation_boolean_feature_columns = [
    "activation_has_relationship_traversal",
    "activation_has_read_reduction",
    "activation_full_relationship_reduction",
    "activation_high_read_gain",
    "activation_medium_or_high_read_gain",
    "activation_has_depth_greater_than_zero",
    "activation_has_multi_entity_query",
    "activation_has_remaining_external_edges",
    "activation_has_update_pressure",
    "activation_high_update_pressure",
    "activation_has_sharedness_pressure",
    "activation_high_sharedness_pressure",
    "activation_has_mapping_warning",
    "touches_descriptor",
    "touches_association",
    "touches_ownership",
    "touches_containment",
    "touches_subtype",
    "is_write_query",
    "has_query_update_volatility",
    "has_observed_sharedness",
]

available_boolean_feature_columns = [
    col for col in activation_boolean_feature_columns
    if col in final_document_variable_matrix_df.columns
]

activation_boolean_features_df = final_document_variable_matrix_df[
    ["query_name"] + available_boolean_feature_columns
].copy()


# ---------------------------------------------------------
# 12. Create final aliases
# ---------------------------------------------------------

document_variable_matrix_for_activation_df = activation_input_matrix_df.copy()
activation_feature_matrix_df = activation_input_matrix_df.copy()

# Update the main matrix to the final version.
document_variable_matrix_df = final_document_variable_matrix_df.copy()


# ---------------------------------------------------------
# 13. Summary tables
# ---------------------------------------------------------

activation_matrix_summary_df = (
    final_document_variable_matrix_df
    .groupby(
        [
            "activation_read_gain_level",
            "activation_update_pressure_level",
            "activation_sharedness_pressure_level",
        ],
        dropna=False,
    )
    .agg(
        n_queries=("query_name", "count"),
        queries=("query_name", lambda s: sorted(set(s))),
        avg_DeltaRratio=("DeltaRratio", "mean"),
        avg_update_volatility=("query_update_volatility_score", "mean"),
        avg_sharedness=("max_observed_sharedness_score", "mean"),
    )
    .reset_index()
    .sort_values(
        [
            "activation_read_gain_level",
            "activation_update_pressure_level",
            "activation_sharedness_pressure_level",
        ]
    )
    .reset_index(drop=True)
)

activation_by_generic_class_df = (
    final_document_variable_matrix_df
    .groupby(["generic_class", "query_family"], dropna=False)
    .agg(
        n_queries=("query_name", "count"),
        avg_Rc=("Rc", "mean"),
        avg_Re=("Re", "mean"),
        avg_DeltaRratio=("DeltaRratio", "mean"),
        avg_update_volatility=("query_update_volatility_score", "mean"),
        avg_sharedness=("max_observed_sharedness_score", "mean"),
        n_high_read_gain=("activation_high_read_gain", lambda s: int(s.sum())),
        n_high_update_pressure=("activation_high_update_pressure", lambda s: int(s.sum())),
        n_high_sharedness_pressure=("activation_high_sharedness_pressure", lambda s: int(s.sum())),
    )
    .reset_index()
    .sort_values(["generic_class", "query_family"])
    .reset_index(drop=True)
)

activation_read_update_sharedness_summary_df = (
    final_document_variable_matrix_df
    .groupby(
        [
            "activation_high_read_gain",
            "activation_high_update_pressure",
            "activation_high_sharedness_pressure",
        ],
        dropna=False,
    )
    .agg(
        n_queries=("query_name", "count"),
        queries=("query_name", lambda s: sorted(set(s))),
    )
    .reset_index()
    .sort_values(
        [
            "activation_high_read_gain",
            "activation_high_update_pressure",
            "activation_high_sharedness_pressure",
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 14. Validation tables
# ---------------------------------------------------------

required_activation_columns = [
    "query_name",
    "selected_root",
    "document_depth",
    "Rc",
    "Re",
    "DeltaR",
    "DeltaRratio",
    "query_update_volatility_score",
    "max_observed_sharedness_score",
]

activation_validation_rows = []

for col in required_activation_columns:
    activation_validation_rows.append({
        "check_name": f"column_exists_{col}",
        "status": "passed" if col in final_document_variable_matrix_df.columns else "failed",
        "details": col,
    })

activation_missing_required_values_df = final_document_variable_matrix_df[
    final_document_variable_matrix_df[
        [
            col for col in required_activation_columns
            if col in final_document_variable_matrix_df.columns
        ]
    ].isna().any(axis=1)
].copy()

activation_validation_rows.append({
    "check_name": "no_missing_required_values",
    "status": "passed" if activation_missing_required_values_df.empty else "failed",
    "details": f"missing_rows={len(activation_missing_required_values_df)}",
})

activation_duplicate_query_rows_df = final_document_variable_matrix_df[
    final_document_variable_matrix_df.duplicated("query_name", keep=False)
].copy()

activation_validation_rows.append({
    "check_name": "one_row_per_query",
    "status": "passed" if activation_duplicate_query_rows_df.empty else "failed",
    "details": f"duplicate_rows={len(activation_duplicate_query_rows_df)}",
})

activation_negative_delta_df = final_document_variable_matrix_df[
    final_document_variable_matrix_df["DeltaR"] < 0
].copy()

activation_validation_rows.append({
    "check_name": "no_negative_delta",
    "status": "passed" if activation_negative_delta_df.empty else "failed",
    "details": f"negative_delta_rows={len(activation_negative_delta_df)}",
})

activation_validation_df = pd.DataFrame(activation_validation_rows)

failed_activation_validation_df = activation_validation_df[
    activation_validation_df["status"] == "failed"
].copy()


# ---------------------------------------------------------
# 15. Display outputs
# ---------------------------------------------------------

print("Final document variable matrix for activation assembled successfully.")

print("\nFinal document variable matrix:")
display(final_document_variable_matrix_df)

print("\nActivation input matrix:")
display(activation_input_matrix_df)

print("\nActivation numeric features:")
display(activation_numeric_features_df)

print("\nActivation boolean features:")
display(activation_boolean_features_df)

print("\nActivation matrix summary:")
display(activation_matrix_summary_df)

print("\nActivation by generic class:")
display(activation_by_generic_class_df)

print("\nActivation validation:")
display(activation_validation_df)

if not failed_activation_validation_df.empty:
    print("\nWarning: some activation validation checks failed.")
    display(failed_activation_validation_df)


# ---------------------------------------------------------
# 16. Save BLOCK V20 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    final_document_variable_matrix_df,
    "variables/block_v20/final_document_variable_matrix.csv",
)

save_dataframe(
    document_variable_matrix_df,
    "variables/block_v20/document_variable_matrix.csv",
)

save_dataframe(
    document_variable_matrix_for_activation_df,
    "variables/block_v20/document_variable_matrix_for_activation.csv",
)

save_dataframe(
    activation_input_matrix_df,
    "variables/block_v20/activation_input_matrix.csv",
)

save_dataframe(
    activation_feature_matrix_df,
    "variables/block_v20/activation_feature_matrix.csv",
)

save_dataframe(
    activation_numeric_features_df,
    "variables/block_v20/activation_numeric_features.csv",
)

save_dataframe(
    activation_boolean_features_df,
    "variables/block_v20/activation_boolean_features.csv",
)

save_dataframe(
    activation_matrix_summary_df,
    "variables/block_v20/activation_matrix_summary.csv",
)

save_dataframe(
    activation_by_generic_class_df,
    "variables/block_v20/activation_by_generic_class.csv",
)

save_dataframe(
    activation_read_update_sharedness_summary_df,
    "variables/block_v20/activation_read_update_sharedness_summary.csv",
)

save_dataframe(
    activation_validation_df,
    "variables/block_v20/activation_validation.csv",
)

save_dataframe(
    failed_activation_validation_df,
    "variables/block_v20/failed_activation_validation.csv",
)

save_dataframe(
    activation_missing_required_values_df,
    "variables/block_v20/activation_missing_required_values.csv",
)

save_dataframe(
    activation_duplicate_query_rows_df,
    "variables/block_v20/activation_duplicate_query_rows.csv",
)

save_dataframe(
    activation_negative_delta_df,
    "variables/block_v20/activation_negative_delta.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "n_queries": len(final_document_variable_matrix_df),
        "n_columns_final_matrix": len(final_document_variable_matrix_df.columns),
        "n_columns_activation_input": len(activation_input_matrix_df.columns),
        "n_numeric_features": len(available_numeric_feature_columns),
        "n_boolean_features": len(available_boolean_feature_columns),
        "activation_read_gain_level_counts": (
            final_document_variable_matrix_df["activation_read_gain_level"]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "activation_update_pressure_level_counts": (
            final_document_variable_matrix_df["activation_update_pressure_level"]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "activation_sharedness_pressure_level_counts": (
            final_document_variable_matrix_df["activation_sharedness_pressure_level"]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "n_failed_activation_validation_checks": len(failed_activation_validation_df),
        "output_files": {
            "final_document_variable_matrix_csv": "variables/block_v20/final_document_variable_matrix.csv",
            "document_variable_matrix_csv": "variables/block_v20/document_variable_matrix.csv",
            "document_variable_matrix_for_activation_csv": "variables/block_v20/document_variable_matrix_for_activation.csv",
            "activation_input_matrix_csv": "variables/block_v20/activation_input_matrix.csv",
            "activation_feature_matrix_csv": "variables/block_v20/activation_feature_matrix.csv",
            "activation_numeric_features_csv": "variables/block_v20/activation_numeric_features.csv",
            "activation_boolean_features_csv": "variables/block_v20/activation_boolean_features.csv",
            "activation_matrix_summary_csv": "variables/block_v20/activation_matrix_summary.csv",
            "activation_by_generic_class_csv": "variables/block_v20/activation_by_generic_class.csv",
            "activation_validation_csv": "variables/block_v20/activation_validation.csv",
        },
    },
    "variables/block_v20/final_document_variable_matrix_metadata.json",
)

write_execution_log(
    block_name="BLOCK V20 — FINAL DOCUMENT VARIABLE MATRIX FOR ACTIVATION",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_queries": len(final_document_variable_matrix_df),
        "n_columns_final_matrix": len(final_document_variable_matrix_df.columns),
        "n_columns_activation_input": len(activation_input_matrix_df.columns),
        "n_failed_activation_validation_checks": len(failed_activation_validation_df),
        "final_document_variable_matrix_csv": "variables/block_v20/final_document_variable_matrix.csv",
        "activation_input_matrix_csv": "variables/block_v20/activation_input_matrix.csv",
        "activation_validation_csv": "variables/block_v20/activation_validation.csv",
        "metadata_json": "variables/block_v20/final_document_variable_matrix_metadata.json",
    },
)


# ---------------------------------------------------------
# 17. Update README section for BLOCK V20
# ---------------------------------------------------------

block_v20_readme_body = f"""
This block consolidates the final document variable matrix used as input for
the generic activation logic G0–G9.

Main variables created:

    final_document_variable_matrix_df
    document_variable_matrix_for_activation_df
    activation_input_matrix_df

Additional variables created:

    activation_feature_matrix_df
    activation_numeric_features_df
    activation_boolean_features_df
    activation_matrix_summary_df
    activation_by_generic_class_df
    activation_read_update_sharedness_summary_df
    activation_validation_df

Number of queries:

    {len(final_document_variable_matrix_df)}

Number of columns in the final matrix:

    {len(final_document_variable_matrix_df.columns)}

Number of columns in the activation input matrix:

    {len(activation_input_matrix_df.columns)}

Number of numeric activation features:

    {len(available_numeric_feature_columns)}

Number of boolean activation features:

    {len(available_boolean_feature_columns)}

Failed activation validation checks:

    {len(failed_activation_validation_df)}

Generated reproducibility files:

    variables/block_v20/final_document_variable_matrix.csv
    variables/block_v20/document_variable_matrix.csv
    variables/block_v20/document_variable_matrix_for_activation.csv
    variables/block_v20/activation_input_matrix.csv
    variables/block_v20/activation_feature_matrix.csv
    variables/block_v20/activation_numeric_features.csv
    variables/block_v20/activation_boolean_features.csv
    variables/block_v20/activation_matrix_summary.csv
    variables/block_v20/activation_by_generic_class.csv
    variables/block_v20/activation_read_update_sharedness_summary.csv
    variables/block_v20/activation_validation.csv
    variables/block_v20/final_document_variable_matrix_metadata.json

Methodological meaning:

    This block prepares the final activation-ready matrix.

    It consolidates:
    - Rc;
    - Re;
    - DeltaR;
    - DeltaRratio;
    - selected root;
    - document depth;
    - semantic relationship profile;
    - update volatility;
    - observed sharedness;
    - document design risk classes.

Important note:

    This block does not activate G0–G9 yet.
    It only prepares the stable input matrix for the activation logic.
"""

update_readme_section(
    section_title="BLOCK V20 — Final Document Variable Matrix for Activation",
    section_body=block_v20_readme_body,
)

Final document variable matrix for activation assembled successfully.

Final document variable matrix:


,query_name,generic_class,query_family,query_type,abstract_query,selected_root,document_depth,selected_document_depth,best_document_depth,max_depth_for_query,...,activation_has_multi_entity_query,activation_has_remaining_external_edges,activation_has_update_pressure,activation_high_update_pressure,activation_has_sharedness_pressure,activation_high_sharedness_pressure,activation_has_mapping_warning,activation_read_gain_level,activation_update_pressure_level,activation_sharedness_pressure_level
0,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,Create a financial service account for a perso...,Person,3.0,3.0,3,3.0,...,True,False,True,True,True,False,False,full,high,medium
1,Q1_CompanyProfileIBM,QG1,Local lookup,select,Show the profile of company IBM.,Corporation,0.0,0.0,0,0.0,...,False,False,False,False,False,False,False,none,none,none
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,"Show IBM with its industry, country, and liste...",Corporation,1.0,1.0,1,1.0,...,True,False,True,False,True,False,True,full,medium,medium
3,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,Show the securities held in each financial ser...,FinancialServiceAccount,3.0,3.0,3,3.0,...,True,False,True,True,True,False,True,full,high,medium
4,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Show the companies reached from a person throu...,Person,4.0,4.0,4,4.0,...,True,False,True,True,True,False,True,full,high,medium
5,Q5_ReportsAndMetricDataOfCompany,QG5,Containment retrieval,select,Show the financial reports of a company and th...,Corporation,3.0,3.0,3,3.0,...,True,False,False,False,True,False,True,full,none,medium
6,Q6_TechUSListedSecuritiesWithHighLastTradedValue,QG6,Filtered search / recommendation,select,Find listed securities of technology companies...,ListedSecurity,2.0,2.0,2,2.0,...,True,False,True,False,True,False,True,full,medium,medium
7,Q7_PersonsWhoBoughtMoreIBMThanSold,QG7,Aggregation / ranking,select,Who bought more IBM stocks than they sold?,Person,4.0,4.0,4,4.0,...,True,False,True,True,True,False,True,full,high,medium
8,Q8_IBMTransactionsBelowAverageSellingPrice,QG8,Aggregation / ranking,select,Show me each transaction for IBM whose price i...,Transaction,2.0,2.0,2,2.0,...,True,False,True,True,True,False,True,full,high,medium
9,Q9_PersonsWhoBoughtAndSoldSameStock,QG9,Associative retrieval + comparison,select,Show me everyone who bought and sold the same ...,Person,3.0,3.0,3,3.0,...,True,False,True,True,True,False,False,full,high,medium



Activation input matrix:


,query_name,generic_class,query_family,query_type,abstract_query,selected_root,document_depth,selected_document_depth,max_depth_for_query,document_depth_ratio,...,activation_high_sharedness_pressure,activation_sharedness_pressure_level,shared_target_entities,document_potential_class,physical_mapping_reliability,update_embedding_tradeoff_class,sharedness_embedding_tradeoff_class,combined_document_design_risk_class,activation_read_gain_level,activation_has_mapping_warning
0,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,Create a financial service account for a perso...,Person,3.0,3.0,3.0,1.0,...,False,medium,"[FinancialServiceAccount, Holding, ListedSecur...",high_reduction_potential,ok,high_read_gain_high_update_pressure,high_read_gain_manageable_sharedness,strong_read_candidate_high_update_pressure,full,False
1,Q1_CompanyProfileIBM,QG1,Local lookup,select,Show the profile of company IBM.,Corporation,0.0,0.0,0.0,0.0,...,False,none,[],no_relationship_traversal,ok,no_update_pressure,no_sharedness_pressure,low_overall_design_pressure,none,False
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,"Show IBM with its industry, country, and liste...",Corporation,1.0,1.0,1.0,1.0,...,False,medium,"[Country, Industry, ListedSecurity]",high_reduction_potential,needs_review_no_observed_match,high_read_gain_manageable_update_pressure,high_read_gain_manageable_sharedness,strong_read_candidate_low_maintenance_pressure,full,True
3,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,Show the securities held in each financial ser...,FinancialServiceAccount,3.0,3.0,3.0,1.0,...,False,medium,"[Holding, ListedSecurity, Security]",high_reduction_potential,needs_review_no_observed_match,high_read_gain_high_update_pressure,high_read_gain_manageable_sharedness,strong_read_candidate_high_update_pressure,full,True
4,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Show the companies reached from a person throu...,Person,4.0,4.0,4.0,1.0,...,False,medium,"[FinancialServiceAccount, Holding, ListedSecur...",high_reduction_potential,needs_review_no_observed_match,high_read_gain_high_update_pressure,high_read_gain_manageable_sharedness,strong_read_candidate_high_update_pressure,full,True
5,Q5_ReportsAndMetricDataOfCompany,QG5,Containment retrieval,select,Show the financial reports of a company and th...,Corporation,3.0,3.0,3.0,1.0,...,False,medium,"[Disclosure, FinancialReport, ReportElement, S...",high_reduction_potential,needs_review_no_observed_match,no_update_pressure,high_read_gain_manageable_sharedness,strong_read_candidate_low_maintenance_pressure,full,True
6,Q6_TechUSListedSecuritiesWithHighLastTradedValue,QG6,Filtered search / recommendation,select,Find listed securities of technology companies...,ListedSecurity,2.0,2.0,2.0,1.0,...,False,medium,"[Country, Industry, ListedSecurity]",high_reduction_potential,needs_review_no_observed_match,high_read_gain_manageable_update_pressure,high_read_gain_manageable_sharedness,strong_read_candidate_low_maintenance_pressure,full,True
7,Q7_PersonsWhoBoughtMoreIBMThanSold,QG7,Aggregation / ranking,select,Who bought more IBM stocks than they sold?,Person,4.0,4.0,4.0,1.0,...,False,medium,"[FinancialServiceAccount, ListedSecurity, Tran...",high_reduction_potential,needs_review_no_observed_match,high_read_gain_high_update_pressure,high_read_gain_manageable_sharedness,strong_read_candidate_high_update_pressure,full,True
8,Q8_IBMTransactionsBelowAverageSellingPrice,QG8,Aggregation / ranking,select,Show me each transaction for IBM whose price i...,Transaction,2.0,2.0,2.0,1.0,...,False,medium,"[ListedSecurity, Transaction]",high_reduction_potential,needs_review_no_observed_match,high_read_gain_high_update_pressure,high_read_gain_manageable_sharedness,strong_read_candidate_high_update_pressure,full,True
9,Q9_PersonsWhoBoughtAndSoldSameStock,QG9,Associative retrieval + comparison,select,


Activation numeric features:


...


Activation by generic class:


,generic_class,query_family,n_queries,avg_Rc,avg_Re,avg_DeltaRratio,avg_update_volatility,avg_sharedness,n_high_read_gain,n_high_update_pressure,n_high_sharedness_pressure
0,QG1,Local lookup,1,0.0,0.0,0.0,0.000000,0.000000,0,0,0
1,QG10,Update / insertion with relationship creation,1,5.0,0.0,1.0,0.932000,0.295882,1,1,0
2,QG2,Shallow rooted retrieval,1,3.0,0.0,1.0,0.363000,0.297418,1,0,0
3,QG3,Associative retrieval,1,3.0,0.0,1.0,0.799000,0.295882,1,1,0
4,QG4,Deep traversal,1,4.0,0.0,1.0,0.798400,0.295882,1,1,0
5,QG5,Containment retrieval,1,4.0,0.0,1.0,0.000000,0.294621,1,0,0
6,QG6,Filtered search / recommendation,1,3.0,0.0,1.0,0.363000,0.297418,1,0,0
7,QG7,Aggregation / ranking,1,6.0,0.0,1.0,0.770286,0.295882,1,1,0
8,QG8,Aggregation / ranking,1,3.0,0.0,1.0,0.649000,0.295882,1,1,0
9,QG9,Associative retrieval + comparison,1,5.0,0.0,1.0,0.832000,0.295882,1,1,0



Activation validation:


,check_name,status,details
0,column_exists_query_name,passed,query_name
1,column_exists_selected_root,passed,selected_root
2,column_exists_document_depth,passed,document_depth
3,column_exists_Rc,passed,Rc
4,column_exists_Re,passed,Re
5,column_exists_DeltaR,passed,DeltaR
6,column_exists_DeltaRratio,passed,DeltaRratio
7,column_exists_query_update_volatility_score,passed,query_update_volatility_score
8,column_exists_max_observed_sharedness_score,passed,max_observed_sharedness_score
9,no_missing_required_values,passed,missing_rows=0


Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v20/final_document_variable_matrix.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v20/document_variable_matrix.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v20/document_variable_matrix_for_activation.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v20/activation_input_matrix.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v20/activation_feature_matrix.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v20/activation_numeric_features.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fi

In [94]:
activation_validation_df

,check_name,status,details
0,column_exists_query_name,passed,query_name
1,column_exists_selected_root,passed,selected_root
2,column_exists_document_depth,passed,document_depth
3,column_exists_Rc,passed,Rc
4,column_exists_Re,passed,Re
5,column_exists_DeltaR,passed,DeltaR
6,column_exists_DeltaRratio,passed,DeltaRratio
7,column_exists_query_update_volatility_score,passed,query_update_volatility_score
8,column_exists_max_observed_sharedness_score,passed,max_observed_sharedness_score
9,no_missing_required_values,passed,missing_rows=0


In [95]:
#TRANSFORMAR MATRIZ ANALÓTICA EM CSV PARA ANÁLISE DOS RESULTADOS
analytical_matrix_final_df.to_csv("analytical_matrix_final_df.csv", index=False)

NameError: name 'analytical_matrix_final_df' is not defined

In [96]:
# =========================================================
# BLOCK V21 — APPLY GENERIC ACTIVATION LOGIC G0–G9
# =========================================================
#
# Purpose:
# This block applies the generic activation logic G0–G9 over
# the final document variable matrix produced in BLOCK V20.
#
# Input:
# - final_document_variable_matrix_df
# - activation_input_matrix_df
#
# Main outputs preserved for downstream compatibility:
# - g_class_activation_by_query_df
# - generic_activation_by_query_df
# - activation_g0_g9_df
#
# Additional outputs:
# - g_class_activation_long_df
# - g_class_activation_summary_df
# - g_class_rule_catalog_df
# - document_variable_matrix_with_activation_df
#
# Methodological role:
# The activation logic maps analytical variables into generic
# document configuration families.
#
# These families will guide the next block, where concrete MongoDB
# candidate configurations are instantiated.
# =========================================================

import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "final_document_variable_matrix_df",
    "activation_input_matrix_df",
    "fiben_scenario",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first BLOCK V20 — FINAL DOCUMENT VARIABLE MATRIX FOR ACTIVATION."
        )


# ---------------------------------------------------------
# 2. Helper functions
# ---------------------------------------------------------

def get_num(row, col, default=0.0):
    """
    Safely read a numeric value from a row.
    """
    value = row.get(col, default)

    try:
        if pd.isna(value):
            return float(default)
    except Exception:
        pass

    try:
        return float(value)
    except Exception:
        return float(default)


def get_bool(row, col, default=False):
    """
    Safely read a boolean value from a row.
    """
    value = row.get(col, default)

    try:
        if pd.isna(value):
            return bool(default)
    except Exception:
        pass

    return bool(value)


def get_list(row, col):
    """
    Safely read a list-like value from a row.
    """
    value = row.get(col, [])

    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    if value is None:
        return []

    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    return [value]


def contains_text(row, col, text):
    """
    Case-insensitive containment check for string columns.
    """
    value = str(row.get(col, "")).lower()
    return text.lower() in value


def activation_label(active):
    """
    Convert a boolean activation result into a readable label.
    """
    return "active" if active else "inactive"


# ---------------------------------------------------------
# 3. Define G0–G9 rule catalog
# ---------------------------------------------------------
#
# These are generic document-configuration classes.
#
# They are intentionally independent from concrete MongoDB syntax.
# The next block will translate active classes into concrete
# candidate configurations.
# ---------------------------------------------------------

GENERIC_G_CLASS_CATALOG = {
    "G0": {
        "name": "Root-only document",
        "description": (
            "Use a simple root document when the query does not require "
            "relationship traversal."
        ),
    },
    "G1": {
        "name": "Root with direct descriptors",
        "description": (
            "Embed or colocate descriptor attributes/entities directly attached "
            "to the selected root."
        ),
    },
    "G2": {
        "name": "Root with contained children",
        "description": (
            "Embed containment children when the query benefits from nested "
            "document retrieval."
        ),
    },
    "G3": {
        "name": "Root with associated references",
        "description": (
            "Represent associative relationships around the root using embedded "
            "summaries or references."
        ),
    },
    "G4": {
        "name": "Deep nested traversal document",
        "description": (
            "Use multi-level nesting or path-oriented documents for deep "
            "traversal workloads."
        ),
    },
    "G5": {
        "name": "Shared target reference strategy",
        "description": (
            "Prefer references or controlled duplication when target entities "
            "are highly shared."
        ),
    },
    "G6": {
        "name": "Aggregation-oriented document",
        "description": (
            "Use precomputed or read-optimized structures for aggregation and "
            "ranking queries."
        ),
    },
    "G7": {
        "name": "Update-aware document strategy",
        "description": (
            "Avoid excessive embedding when write/update volatility is relevant."
        ),
    },
    "G8": {
        "name": "Hybrid embedded-reference strategy",
        "description": (
            "Use embedding for covered relationships and references for remaining "
            "external relationships."
        ),
    },
    "G9": {
        "name": "Benchmark-required trade-off candidate",
        "description": (
            "Generate alternative configurations when read gain, update pressure, "
            "sharedness, or mapping risk create a design trade-off."
        ),
    },
}


g_class_rule_catalog_df = pd.DataFrame([
    {
        "g_class": g_class,
        "g_class_name": spec["name"],
        "description": spec["description"],
    }
    for g_class, spec in GENERIC_G_CLASS_CATALOG.items()
])


# ---------------------------------------------------------
# 4. Define activation rules
# ---------------------------------------------------------

def evaluate_g_classes(row):
    """
    Evaluate all G0–G9 activation rules for one query row.

    Returns
    -------
    list of dict
        One dictionary per G class with activation status,
        score, and evidence.
    """
    rc = get_num(row, "Rc")
    re_value = get_num(row, "Re")
    delta_r = get_num(row, "DeltaR")
    delta_ratio = get_num(row, "DeltaRratio")

    document_depth = get_num(row, "document_depth")
    max_depth = get_num(row, "max_depth_for_query")
    n_entities = get_num(row, "n_entities_touched")
    n_remaining_edges = get_num(row, "n_remaining_edges")

    update_score = get_num(row, "query_update_volatility_score")
    sharedness_score = get_num(row, "max_observed_sharedness_score")

    touches_descriptor = get_bool(row, "touches_descriptor")
    touches_association = get_bool(row, "touches_association")
    touches_ownership = get_bool(row, "touches_ownership")
    touches_containment = get_bool(row, "touches_containment")
    touches_subtype = get_bool(row, "touches_subtype")

    is_write_query = get_bool(row, "is_write_query")
    has_mapping_warning = get_bool(row, "activation_has_mapping_warning")
    has_sharedness = get_bool(row, "has_observed_sharedness")
    has_update = get_bool(row, "has_query_update_volatility")

    query_family = str(row.get("query_family", "")).lower()
    generic_class = str(row.get("generic_class", "")).lower()

    is_aggregation_query = (
        "aggregation" in query_family
        or "ranking" in query_family
        or generic_class in ["qg7", "qg8"]
    )

    is_deep_query = (
        "deep" in query_family
        or max_depth >= 3
        or document_depth >= 3
    )

    has_relationship_traversal = rc > 0
    has_read_reduction = delta_r > 0
    high_read_gain = delta_ratio >= 0.75
    full_read_reduction = delta_ratio >= 1.0
    high_update_pressure = update_score >= 0.60
    high_sharedness_pressure = sharedness_score >= 0.60
    has_remaining_external_edges = re_value > 0 or n_remaining_edges > 0

    results = []

    # G0 — Root-only document
    active = rc == 0 or not has_relationship_traversal
    results.append({
        "g_class": "G0",
        "is_active": active,
        "activation_score": 1.0 if active else 0.0,
        "activation_reason": (
            "The query does not require conceptual relationship traversal."
            if active
            else "The query requires at least one conceptual relationship."
        ),
        "evidence": {
            "Rc": rc,
            "DeltaRratio": delta_ratio,
        },
    })

    # G1 — Root with direct descriptors
    active = (
        touches_descriptor
        and document_depth >= 1
        and has_read_reduction
    )
    results.append({
        "g_class": "G1",
        "is_active": active,
        "activation_score": (
            0.40 + 0.60 * min(delta_ratio, 1.0)
            if active else 0.0
        ),
        "activation_reason": (
            "The query touches descriptor relationships that can be colocated "
            "near the selected root."
            if active
            else "Descriptor-oriented embedding is not a main driver."
        ),
        "evidence": {
            "touches_descriptor": touches_descriptor,
            "document_depth": document_depth,
            "DeltaRratio": delta_ratio,
        },
    })

    # G2 — Root with contained children
    active = (
        touches_containment
        and document_depth >= 1
        and has_read_reduction
    )
    results.append({
        "g_class": "G2",
        "is_active": active,
        "activation_score": (
            0.35 + 0.65 * min(delta_ratio, 1.0)
            if active else 0.0
        ),
        "activation_reason": (
            "The query touches containment relationships suitable for nested "
            "document structures."
            if active
            else "Containment embedding is not a main driver."
        ),
        "evidence": {
            "touches_containment": touches_containment,
            "document_depth": document_depth,
            "DeltaRratio": delta_ratio,
        },
    })

    # G3 — Root with associated references
    active = (
        touches_association
        and n_entities > 1
        and has_relationship_traversal
    )
    results.append({
        "g_class": "G3",
        "is_active": active,
        "activation_score": (
            0.30 + 0.50 * min(delta_ratio, 1.0) + 0.20 * int(has_remaining_external_edges)
            if active else 0.0
        ),
        "activation_reason": (
            "The query touches associative relationships that may require "
            "embedded summaries or references."
            if active
            else "Associative references are not central for this query."
        ),
        "evidence": {
            "touches_association": touches_association,
            "n_entities_touched": n_entities,
            "remaining_external_edges": has_remaining_external_edges,
        },
    })

    # G4 — Deep nested traversal document
    active = (
        is_deep_query
        and has_relationship_traversal
        and has_read_reduction
    )
    results.append({
        "g_class": "G4",
        "is_active": active,
        "activation_score": (
            0.30 + 0.70 * min(delta_ratio, 1.0)
            if active else 0.0
        ),
        "activation_reason": (
            "The query has a deep traversal profile and benefits from a "
            "multi-level document structure."
            if active
            else "The query does not require a deep document structure."
        ),
        "evidence": {
            "query_family": row.get("query_family"),
            "max_depth_for_query": max_depth,
            "document_depth": document_depth,
            "DeltaRratio": delta_ratio,
        },
    })

    # G5 — Shared target reference strategy
    active = (
        has_sharedness
        and sharedness_score > 0
        and touches_association
    )
    results.append({
        "g_class": "G5",
        "is_active": active,
        "activation_score": (
            min(1.0, 0.20 + sharedness_score)
            if active else 0.0
        ),
        "activation_reason": (
            "The query touches shared target entities; reference-oriented design "
            "or controlled duplication should be considered."
            if active
            else "Observed sharedness is not a main design pressure."
        ),
        "evidence": {
            "max_observed_sharedness_score": sharedness_score,
            "touches_association": touches_association,
            "shared_target_entities": get_list(row, "shared_target_entities"),
        },
    })

    # G6 — Aggregation-oriented document
    active = (
        is_aggregation_query
        and has_relationship_traversal
    )
    results.append({
        "g_class": "G6",
        "is_active": active,
        "activation_score": (
            0.40 + 0.60 * min(delta_ratio, 1.0)
            if active else 0.0
        ),
        "activation_reason": (
            "The query has an aggregation/ranking profile and may benefit from "
            "read-optimized or precomputed document structures."
            if active
            else "Aggregation-oriented configuration is not central."
        ),
        "evidence": {
            "query_family": row.get("query_family"),
            "generic_class": row.get("generic_class"),
            "DeltaRratio": delta_ratio,
        },
    })

    # G7 — Update-aware document strategy
    active = (
        is_write_query
        or has_update
        or update_score > 0
    )
    results.append({
        "g_class": "G7",
        "is_active": active,
        "activation_score": (
            min(1.0, 0.20 + update_score)
            if active else 0.0
        ),
        "activation_reason": (
            "The query touches update-volatile entities or is itself a write query; "
            "embedding decisions should consider update maintenance."
            if active
            else "Update volatility is not relevant for this query."
        ),
        "evidence": {
            "is_write_query": is_write_query,
            "query_update_volatility_score": update_score,
            "query_update_volatility_level": row.get("query_update_volatility_level"),
        },
    })

    # G8 — Hybrid embedded-reference strategy
    active = (
        has_relationship_traversal
        and has_read_reduction
        and has_remaining_external_edges
    )
    results.append({
        "g_class": "G8",
        "is_active": active,
        "activation_score": (
            0.50 + 0.30 * min(delta_ratio, 1.0) + 0.20 * int(has_remaining_external_edges)
            if active else 0.0
        ),
        "activation_reason": (
            "The selected document depth covers some relationships but leaves "
            "others external; a hybrid embedding/reference design is appropriate."
            if active
            else "The query does not require a hybrid because either there is no "
                 "reduction or no remaining external edge."
        ),
        "evidence": {
            "Re": re_value,
            "DeltaR": delta_r,
            "DeltaRratio": delta_ratio,
            "n_remaining_edges": n_remaining_edges,
        },
    })

    # G9 — Benchmark-required trade-off candidate
    active = (
        has_mapping_warning
        or (
            high_read_gain
            and (
                high_update_pressure
                or high_sharedness_pressure
            )
        )
        or (
            full_read_reduction
            and (
                update_score > 0
                or sharedness_score > 0
            )
        )
    )
    results.append({
        "g_class": "G9",
        "is_active": active,
        "activation_score": (
            min(
                1.0,
                0.30
                + 0.30 * min(delta_ratio, 1.0)
                + 0.20 * min(update_score, 1.0)
                + 0.20 * min(sharedness_score, 1.0)
                + 0.10 * int(has_mapping_warning),
            )
            if active else 0.0
        ),
        "activation_reason": (
            "The query has a design trade-off that should be validated through "
            "benchmark alternatives."
            if active
            else "No strong benchmark-required trade-off was detected."
        ),
        "evidence": {
            "DeltaRratio": delta_ratio,
            "query_update_volatility_score": update_score,
            "max_observed_sharedness_score": sharedness_score,
            "activation_has_mapping_warning": has_mapping_warning,
        },
    })

    return results


# ---------------------------------------------------------
# 5. Apply activation rules
# ---------------------------------------------------------

activation_long_rows = []
activation_by_query_rows = []

for _, row in final_document_variable_matrix_df.iterrows():
    query_name = row["query_name"]
    query_results = evaluate_g_classes(row)

    active_results = [
        result for result in query_results
        if result["is_active"]
    ]

    active_g_classes = [
        result["g_class"]
        for result in active_results
    ]

    inactive_g_classes = [
        result["g_class"]
        for result in query_results
        if not result["is_active"]
    ]

    active_g_class_names = [
        GENERIC_G_CLASS_CATALOG[g]["name"]
        for g in active_g_classes
    ]

    for result in query_results:
        g_class = result["g_class"]

        activation_long_rows.append({
            "query_name": query_name,
            "generic_class": row.get("generic_class"),
            "query_family": row.get("query_family"),
            "query_type": row.get("query_type"),
            "selected_root": row.get("selected_root"),
            "document_depth": row.get("document_depth"),
            "g_class": g_class,
            "g_class_name": GENERIC_G_CLASS_CATALOG[g_class]["name"],
            "g_class_description": GENERIC_G_CLASS_CATALOG[g_class]["description"],
            "is_active": result["is_active"],
            "activation_status": activation_label(result["is_active"]),
            "activation_score": result["activation_score"],
            "activation_reason": result["activation_reason"],
            "activation_evidence": result["evidence"],
            "DeltaRratio": row.get("DeltaRratio"),
            "query_update_volatility_score": row.get("query_update_volatility_score"),
            "max_observed_sharedness_score": row.get("max_observed_sharedness_score"),
        })

    if active_results:
        primary_result = sorted(
            active_results,
            key=lambda r: (-r["activation_score"], r["g_class"]),
        )[0]

        primary_activation_class = primary_result["g_class"]
        primary_activation_name = GENERIC_G_CLASS_CATALOG[
            primary_activation_class
        ]["name"]
        primary_activation_score = primary_result["activation_score"]
        primary_activation_reason = primary_result["activation_reason"]
    else:
        primary_activation_class = "NONE"
        primary_activation_name = "No active generic class"
        primary_activation_score = 0.0
        primary_activation_reason = "No activation rule was satisfied."

    activation_by_query_rows.append({
        "query_name": query_name,
        "generic_class": row.get("generic_class"),
        "query_family": row.get("query_family"),
        "query_type": row.get("query_type"),
        "selected_root": row.get("selected_root"),
        "document_depth": row.get("document_depth"),
        "Rc": row.get("Rc"),
        "Re": row.get("Re"),
        "DeltaR": row.get("DeltaR"),
        "DeltaRratio": row.get("DeltaRratio"),
        "query_update_volatility_score": row.get("query_update_volatility_score"),
        "max_observed_sharedness_score": row.get("max_observed_sharedness_score"),
        "active_g_classes": active_g_classes,
        "active_g_class_names": active_g_class_names,
        "inactive_g_classes": inactive_g_classes,
        "n_active_g_classes": len(active_g_classes),
        "primary_activation_class": primary_activation_class,
        "primary_activation_name": primary_activation_name,
        "primary_activation_score": primary_activation_score,
        "primary_activation_reason": primary_activation_reason,
        "activation_has_g0": "G0" in active_g_classes,
        "activation_has_g1": "G1" in active_g_classes,
        "activation_has_g2": "G2" in active_g_classes,
        "activation_has_g3": "G3" in active_g_classes,
        "activation_has_g4": "G4" in active_g_classes,
        "activation_has_g5": "G5" in active_g_classes,
        "activation_has_g6": "G6" in active_g_classes,
        "activation_has_g7": "G7" in active_g_classes,
        "activation_has_g8": "G8" in active_g_classes,
        "activation_has_g9": "G9" in active_g_classes,
    })


g_class_activation_long_df = pd.DataFrame(activation_long_rows)
g_class_activation_by_query_df = pd.DataFrame(activation_by_query_rows)


# ---------------------------------------------------------
# 6. Compatibility aliases
# ---------------------------------------------------------

generic_activation_by_query_df = g_class_activation_by_query_df.copy()
activation_g0_g9_df = g_class_activation_by_query_df.copy()
g_class_activation_df = g_class_activation_by_query_df.copy()


# ---------------------------------------------------------
# 7. Merge activation results into the document variable matrix
# ---------------------------------------------------------

activation_merge_cols = [
    "query_name",
    "active_g_classes",
    "active_g_class_names",
    "n_active_g_classes",
    "primary_activation_class",
    "primary_activation_name",
    "primary_activation_score",
    "primary_activation_reason",
    "activation_has_g0",
    "activation_has_g1",
    "activation_has_g2",
    "activation_has_g3",
    "activation_has_g4",
    "activation_has_g5",
    "activation_has_g6",
    "activation_has_g7",
    "activation_has_g8",
    "activation_has_g9",
]

document_variable_matrix_with_activation_df = (
    final_document_variable_matrix_df
    .drop(
        columns=[
            col for col in activation_merge_cols
            if col != "query_name" and col in final_document_variable_matrix_df.columns
        ],
        errors="ignore",
    )
    .merge(
        g_class_activation_by_query_df[activation_merge_cols],
        on="query_name",
        how="left",
    )
)

# Update main matrix alias for downstream blocks.
document_variable_matrix_df = document_variable_matrix_with_activation_df.copy()


# ---------------------------------------------------------
# 8. Summary tables
# ---------------------------------------------------------

g_class_activation_summary_df = (
    g_class_activation_long_df[
        g_class_activation_long_df["is_active"] == True
    ]
    .groupby(["g_class", "g_class_name"], dropna=False)
    .agg(
        n_queries=("query_name", "nunique"),
        queries=("query_name", lambda s: sorted(set(s))),
        avg_activation_score=("activation_score", "mean"),
        max_activation_score=("activation_score", "max"),
        avg_DeltaRratio=("DeltaRratio", "mean"),
        avg_update_volatility=("query_update_volatility_score", "mean"),
        avg_sharedness=("max_observed_sharedness_score", "mean"),
    )
    .reset_index()
    .sort_values(["g_class"])
    .reset_index(drop=True)
)

primary_activation_summary_df = (
    g_class_activation_by_query_df
    .groupby(["primary_activation_class", "primary_activation_name"], dropna=False)
    .agg(
        n_queries=("query_name", "count"),
        queries=("query_name", lambda s: sorted(set(s))),
        avg_primary_activation_score=("primary_activation_score", "mean"),
        avg_DeltaRratio=("DeltaRratio", "mean"),
        avg_update_volatility=("query_update_volatility_score", "mean"),
        avg_sharedness=("max_observed_sharedness_score", "mean"),
    )
    .reset_index()
    .sort_values(["primary_activation_class"])
    .reset_index(drop=True)
)

activation_by_generic_class_summary_df = (
    g_class_activation_by_query_df
    .groupby(["generic_class", "query_family"], dropna=False)
    .agg(
        n_queries=("query_name", "count"),
        primary_classes=(
            "primary_activation_class",
            lambda s: sorted(set(s.dropna())),
        ),
        active_class_sets=(
            "active_g_classes",
            lambda s: [list(v) for v in s],
        ),
        avg_n_active_g_classes=("n_active_g_classes", "mean"),
        avg_DeltaRratio=("DeltaRratio", "mean"),
    )
    .reset_index()
    .sort_values(["generic_class", "query_family"])
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 9. Validation tables
# ---------------------------------------------------------

queries_without_activation_df = g_class_activation_by_query_df[
    g_class_activation_by_query_df["n_active_g_classes"] == 0
].copy()

activation_missing_primary_df = g_class_activation_by_query_df[
    g_class_activation_by_query_df["primary_activation_class"].isna()
].copy()

activation_merge_missing_df = document_variable_matrix_with_activation_df[
    document_variable_matrix_with_activation_df["primary_activation_class"].isna()
].copy()

activation_validation_rows = [
    {
        "check_name": "every_query_has_at_least_one_active_class",
        "status": "passed" if queries_without_activation_df.empty else "failed",
        "details": f"queries_without_activation={len(queries_without_activation_df)}",
    },
    {
        "check_name": "primary_activation_defined",
        "status": "passed" if activation_missing_primary_df.empty else "failed",
        "details": f"missing_primary_rows={len(activation_missing_primary_df)}",
    },
    {
        "check_name": "activation_merge_complete",
        "status": "passed" if activation_merge_missing_df.empty else "failed",
        "details": f"missing_after_merge={len(activation_merge_missing_df)}",
    },
    {
        "check_name": "row_count_preserved",
        "status": (
            "passed"
            if len(document_variable_matrix_with_activation_df)
            == len(final_document_variable_matrix_df)
            else "failed"
        ),
        "details": {
            "before": len(final_document_variable_matrix_df),
            "after": len(document_variable_matrix_with_activation_df),
        },
    },
]

activation_logic_validation_df = pd.DataFrame(activation_validation_rows)

failed_activation_logic_validation_df = activation_logic_validation_df[
    activation_logic_validation_df["status"] == "failed"
].copy()


# ---------------------------------------------------------
# 10. Display outputs
# ---------------------------------------------------------

print("Generic activation logic G0–G9 applied successfully.")

print("\nG-class rule catalog:")
display(g_class_rule_catalog_df)

print("\nActivation by query:")
display(g_class_activation_by_query_df)

print("\nActivation long table:")
display(g_class_activation_long_df)

print("\nG-class activation summary:")
display(g_class_activation_summary_df)

print("\nPrimary activation summary:")
display(primary_activation_summary_df)

print("\nActivation by generic class:")
display(activation_by_generic_class_summary_df)

print("\nActivation validation:")
display(activation_logic_validation_df)

if not failed_activation_logic_validation_df.empty:
    print("\nWarning: some activation validation checks failed.")
    display(failed_activation_logic_validation_df)

if not queries_without_activation_df.empty:
    print("\nQueries without activation:")
    display(queries_without_activation_df)


# ---------------------------------------------------------
# 11. Save BLOCK V21 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    g_class_rule_catalog_df,
    "variables/block_v21/g_class_rule_catalog.csv",
)

save_dataframe(
    g_class_activation_by_query_df,
    "variables/block_v21/g_class_activation_by_query.csv",
)

save_dataframe(
    generic_activation_by_query_df,
    "variables/block_v21/generic_activation_by_query.csv",
)

save_dataframe(
    activation_g0_g9_df,
    "variables/block_v21/activation_g0_g9.csv",
)

save_dataframe(
    g_class_activation_long_df,
    "variables/block_v21/g_class_activation_long.csv",
)

save_dataframe(
    g_class_activation_summary_df,
    "variables/block_v21/g_class_activation_summary.csv",
)

save_dataframe(
    primary_activation_summary_df,
    "variables/block_v21/primary_activation_summary.csv",
)

save_dataframe(
    activation_by_generic_class_summary_df,
    "variables/block_v21/activation_by_generic_class_summary.csv",
)

save_dataframe(
    document_variable_matrix_with_activation_df,
    "variables/block_v21/document_variable_matrix_with_activation.csv",
)

save_dataframe(
    document_variable_matrix_df,
    "variables/block_v21/document_variable_matrix.csv",
)

save_dataframe(
    activation_logic_validation_df,
    "variables/block_v21/activation_logic_validation.csv",
)

save_dataframe(
    failed_activation_logic_validation_df,
    "variables/block_v21/failed_activation_logic_validation.csv",
)

save_dataframe(
    queries_without_activation_df,
    "variables/block_v21/queries_without_activation.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "g_class_catalog": GENERIC_G_CLASS_CATALOG,
        "n_queries": len(g_class_activation_by_query_df),
        "n_activation_long_rows": len(g_class_activation_long_df),
        "n_queries_without_activation": len(queries_without_activation_df),
        "n_failed_activation_logic_validation_checks": len(failed_activation_logic_validation_df),
        "primary_activation_counts": (
            g_class_activation_by_query_df["primary_activation_class"]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "g_class_activation_counts": (
            g_class_activation_long_df[
                g_class_activation_long_df["is_active"] == True
            ]["g_class"]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "output_files": {
            "g_class_rule_catalog_csv": "variables/block_v21/g_class_rule_catalog.csv",
            "g_class_activation_by_query_csv": "variables/block_v21/g_class_activation_by_query.csv",
            "generic_activation_by_query_csv": "variables/block_v21/generic_activation_by_query.csv",
            "activation_g0_g9_csv": "variables/block_v21/activation_g0_g9.csv",
            "g_class_activation_long_csv": "variables/block_v21/g_class_activation_long.csv",
            "g_class_activation_summary_csv": "variables/block_v21/g_class_activation_summary.csv",
            "primary_activation_summary_csv": "variables/block_v21/primary_activation_summary.csv",
            "activation_by_generic_class_summary_csv": "variables/block_v21/activation_by_generic_class_summary.csv",
            "document_variable_matrix_with_activation_csv": "variables/block_v21/document_variable_matrix_with_activation.csv",
            "activation_logic_validation_csv": "variables/block_v21/activation_logic_validation.csv",
            "queries_without_activation_csv": "variables/block_v21/queries_without_activation.csv",
        },
    },
    "variables/block_v21/g_class_activation_metadata.json",
)

write_execution_log(
    block_name="BLOCK V21 — APPLY GENERIC ACTIVATION LOGIC G0–G9",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_queries": len(g_class_activation_by_query_df),
        "n_activation_long_rows": len(g_class_activation_long_df),
        "n_queries_without_activation": len(queries_without_activation_df),
        "n_failed_activation_logic_validation_checks": len(failed_activation_logic_validation_df),
        "g_class_activation_by_query_csv": "variables/block_v21/g_class_activation_by_query.csv",
        "g_class_activation_long_csv": "variables/block_v21/g_class_activation_long.csv",
        "document_variable_matrix_with_activation_csv": "variables/block_v21/document_variable_matrix_with_activation.csv",
        "activation_logic_validation_csv": "variables/block_v21/activation_logic_validation.csv",
        "metadata_json": "variables/block_v21/g_class_activation_metadata.json",
    },
)


# ---------------------------------------------------------
# 12. Update README section for BLOCK V21
# ---------------------------------------------------------

block_v21_readme_body = f"""
This block applies the generic activation logic G0-G9 over the final document
variable matrix.

Main variables created:

    g_class_activation_by_query_df
    generic_activation_by_query_df
    activation_g0_g9_df

Additional variables created:

    g_class_activation_long_df
    g_class_activation_summary_df
    primary_activation_summary_df
    activation_by_generic_class_summary_df
    document_variable_matrix_with_activation_df
    activation_logic_validation_df

Number of queries:

    {len(g_class_activation_by_query_df)}

Number of activation rows:

    {len(g_class_activation_long_df)}

Number of queries without activation:

    {len(queries_without_activation_df)}

Failed activation validation checks:

    {len(failed_activation_logic_validation_df)}

Generic activation classes:

    G0 - Root-only document
    G1 - Root with direct descriptors
    G2 - Root with contained children
    G3 - Root with associated references
    G4 - Deep nested traversal document
    G5 - Shared target reference strategy
    G6 - Aggregation-oriented document
    G7 - Update-aware document strategy
    G8 - Hybrid embedded-reference strategy
    G9 - Benchmark-required trade-off candidate

Generated reproducibility files:

    variables/block_v21/g_class_rule_catalog.csv
    variables/block_v21/g_class_activation_by_query.csv
    variables/block_v21/generic_activation_by_query.csv
    variables/block_v21/activation_g0_g9.csv
    variables/block_v21/g_class_activation_long.csv
    variables/block_v21/g_class_activation_summary.csv
    variables/block_v21/primary_activation_summary.csv
    variables/block_v21/activation_by_generic_class_summary.csv
    variables/block_v21/document_variable_matrix_with_activation.csv
    variables/block_v21/activation_logic_validation.csv
    variables/block_v21/g_class_activation_metadata.json

Methodological meaning:

    This block transforms analytical variables into generic document
    configuration families.

    These classes are not yet concrete MongoDB schemas.
    They identify which families of document designs should be instantiated
    and benchmarked.

Important interpretation:

    More than one G class may be active for the same query.

    The primary activation class is selected using the highest activation score,
    but all active classes should be considered when generating candidate
    MongoDB configurations.
"""

update_readme_section(
    section_title="BLOCK V21 — Apply Generic Activation Logic G0-G9",
    section_body=block_v21_readme_body,
)

Generic activation logic G0–G9 applied successfully.

G-class rule catalog:


,g_class,g_class_name,description
0,G0,Root-only document,Use a simple root document when the query does...
1,G1,Root with direct descriptors,Embed or colocate descriptor attributes/entiti...
2,G2,Root with contained children,Embed containment children when the query bene...
3,G3,Root with associated references,Represent associative relationships around the...
4,G4,Deep nested traversal document,Use multi-level nesting or path-oriented docum...
5,G5,Shared target reference strategy,Prefer references or controlled duplication wh...
6,G6,Aggregation-oriented document,Use precomputed or read-optimized structures f...
7,G7,Update-aware document strategy,Avoid excessive embedding when write/update vo...
8,G8,Hybrid embedded-reference strategy,Use embedding for covered relationships and re...
9,G9,Benchmark-required trade-off candidate,Generate alternative configurations when read ...



Activation by query:


,query_name,generic_class,query_family,query_type,selected_root,document_depth,Rc,Re,DeltaR,DeltaRratio,...,activation_has_g0,activation_has_g1,activation_has_g2,activation_has_g3,activation_has_g4,activation_has_g5,activation_has_g6,activation_has_g7,activation_has_g8,activation_has_g9
0,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,Person,3.0,5.0,0.0,5.0,1.0,...,False,False,True,True,True,True,False,True,False,True
1,Q1_CompanyProfileIBM,QG1,Local lookup,select,Corporation,0.0,0.0,0.0,0.0,0.0,...,True,False,False,False,False,False,False,False,False,False
2,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,Corporation,1.0,3.0,0.0,3.0,1.0,...,False,True,False,True,False,True,False,True,False,True
3,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,FinancialServiceAccount,3.0,3.0,0.0,3.0,1.0,...,False,False,True,True,True,True,False,True,False,True
4,Q4_CompaniesReachedFromPersonThroughAccountHol...,QG4,Deep traversal,select,Person,4.0,4.0,0.0,4.0,1.0,...,False,False,True,True,True,True,False,True,False,True
5,Q5_ReportsAndMetricDataOfCompany,QG5,Containment retrieval,select,Corporation,3.0,4.0,0.0,4.0,1.0,...,False,False,True,False,True,False,False,False,False,True
6,Q6_TechUSListedSecuritiesWithHighLastTradedValue,QG6,Filtered search / recommendation,select,ListedSecurity,2.0,3.0,0.0,3.0,1.0,...,False,True,False,True,False,True,False,True,False,True
7,Q7_PersonsWhoBoughtMoreIBMThanSold,QG7,Aggregation / ranking,select,Person,4.0,6.0,0.0,6.0,1.0,...,False,False,True,True,True,True,True,True,False,True
8,Q8_IBMTransactionsBelowAverageSellingPrice,QG8,Aggregation / ranking,select,Transaction,2.0,3.0,0.0,3.0,1.0,...,False,False,False,True,False,True,True,True,False,True
9,Q9_PersonsWhoBoughtAndSoldSameStock,QG9,Associative retrieval + comparison,select,Person,3.0,5.0,0.0,5.0,1.0,...,False,False,True,True,True,True,False,True,False,True



Activation long table:


...


Activation by generic class:


,generic_class,query_family,n_queries,primary_classes,active_class_sets,avg_n_active_g_classes,avg_DeltaRratio
0,QG1,Local lookup,1,[G0],[[G0]],1.0,0.0
1,QG10,Update / insertion with relationship creation,1,[G2],"[[G2, G3, G4, G5, G7, G9]]",6.0,1.0
2,QG2,Shallow rooted retrieval,1,[G1],"[[G1, G3, G5, G7, G9]]",5.0,1.0
3,QG3,Associative retrieval,1,[G2],"[[G2, G3, G4, G5, G7, G9]]",6.0,1.0
4,QG4,Deep traversal,1,[G2],"[[G2, G3, G4, G5, G7, G9]]",6.0,1.0
5,QG5,Containment retrieval,1,[G2],"[[G2, G4, G9]]",3.0,1.0
6,QG6,Filtered search / recommendation,1,[G1],"[[G1, G3, G5, G7, G9]]",5.0,1.0
7,QG7,Aggregation / ranking,1,[G2],"[[G2, G3, G4, G5, G6, G7, G9]]",7.0,1.0
8,QG8,Aggregation / ranking,1,[G6],"[[G3, G5, G6, G7, G9]]",5.0,1.0
9,QG9,Associative retrieval + comparison,1,[G2],"[[G2, G3, G4, G5, G7, G9]]",6.0,1.0



Activation validation:


,check_name,status,details
0,every_query_has_at_least_one_active_class,passed,queries_without_activation=0
1,primary_activation_defined,passed,missing_primary_rows=0
2,activation_merge_complete,passed,missing_after_merge=0
3,row_count_preserved,passed,"{'before': 10, 'after': 10}"


Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v21/g_class_rule_catalog.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v21/g_class_activation_by_query.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v21/generic_activation_by_query.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v21/activation_g0_g9.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v21/g_class_activation_long.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v21/g_class_activation_summary.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v21/prima

In [97]:
activation_logic_validation_df

,check_name,status,details
0,every_query_has_at_least_one_active_class,passed,queries_without_activation=0
1,primary_activation_defined,passed,missing_primary_rows=0
2,activation_merge_complete,passed,missing_after_merge=0
3,row_count_preserved,passed,"{'before': 10, 'after': 10}"


In [98]:
# =========================================================
# BLOCK V22 — INSTANTIATE CONCRETE MONGODB CONFIGURATION CANDIDATES
# =========================================================
#
# Purpose:
# This block instantiates concrete MongoDB configuration candidates
# from the active generic classes G0-G9.
#
# Input:
# - g_class_activation_by_query_df from BLOCK V21;
# - g_class_activation_long_df from BLOCK V21;
# - document_variable_matrix_df with activation results;
# - relationship_semantics_df.
#
# Main outputs:
# - mongodb_configuration_candidates_df
# - concrete_mongodb_candidates_df
# - mongo_candidate_configurations_df
#
# Additional outputs:
# - mongodb_configuration_candidates_long_df
# - mongodb_candidate_summary_df
# - mongodb_candidate_by_query_df
# - mongodb_candidate_validation_df
#
# Methodological role:
# This block translates generic activation classes into concrete
# MongoDB design candidates such as:
# - root-only collection;
# - embedded descriptor document;
# - embedded containment document;
# - reference-based shared target design;
# - hybrid embedded/reference design;
# - aggregation-oriented materialized collection;
# - update-aware normalized design.
#
# Important:
# This block does not load data into MongoDB.
# This block does not run benchmark queries.
# It only creates the candidate configuration catalog.
# =========================================================

import re
import json
import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "g_class_activation_by_query_df",
    "g_class_activation_long_df",
    "document_variable_matrix_df",
    "relationship_semantics_df",
    "fiben_scenario",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first BLOCK V21 — APPLY GENERIC ACTIVATION LOGIC G0-G9."
        )


# ---------------------------------------------------------
# 2. Helper functions
# ---------------------------------------------------------

def safe_list(value):
    """
    Convert a value into a clean Python list.

    This protects the block when a cell contains:
    - list;
    - tuple;
    - scalar;
    - missing value.
    """
    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    if value is None:
        return []

    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    return [value]


def safe_float(value, default=0.0):
    """
    Convert a value to float safely.
    """
    try:
        if pd.isna(value):
            return float(default)
    except Exception:
        pass

    try:
        return float(value)
    except Exception:
        return float(default)


def safe_bool(value, default=False):
    """
    Convert a value to bool safely.
    """
    try:
        if pd.isna(value):
            return bool(default)
    except Exception:
        pass

    return bool(value)


def snake_case(value):
    """
    Convert a name to snake_case.
    """
    value = str(value)
    value = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", value)
    value = re.sub(r"[^a-zA-Z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value)
    return value.strip("_").lower()


def make_collection_name(root_entity, suffix=None):
    """
    Create a stable MongoDB collection name.
    """
    base = snake_case(root_entity)

    if not base.endswith("s"):
        base = base + "s"

    if suffix:
        return f"{base}_{snake_case(suffix)}"

    return base


def make_candidate_id(query_name, g_class, variant):
    """
    Create a stable candidate configuration identifier.
    """
    return f"{snake_case(query_name)}__{snake_case(g_class)}__{snake_case(variant)}"


def remove_duplicates_preserve_order(values):
    """
    Remove duplicates while preserving order.
    """
    result = []
    seen = set()

    for value in values:
        if value not in seen:
            result.append(value)
            seen.add(value)

    return result


def get_relationship_rows_by_names(relationship_names):
    """
    Return relationship semantic rows matching relationship names.
    """
    names = safe_list(relationship_names)

    if not names:
        return relationship_semantics_df.iloc[0:0].copy()

    return relationship_semantics_df[
        relationship_semantics_df["name"].isin(names)
    ].copy()


def extract_target_entities_for_relationships(relationship_names, semantic_filter=None):
    """
    Extract target entities from relationship names.

    If semantic_filter is provided, only relationships with that semantic_type
    are used.
    """
    rel_df = get_relationship_rows_by_names(relationship_names)

    if semantic_filter is not None:
        rel_df = rel_df[
            rel_df["semantic_type"].isin(safe_list(semantic_filter))
        ].copy()

    if rel_df.empty:
        return []

    return remove_duplicates_preserve_order(rel_df["target"].dropna().tolist())


def extract_source_entities_for_relationships(relationship_names, semantic_filter=None):
    """
    Extract source entities from relationship names.

    If semantic_filter is provided, only relationships with that semantic_type
    are used.
    """
    rel_df = get_relationship_rows_by_names(relationship_names)

    if semantic_filter is not None:
        rel_df = rel_df[
            rel_df["semantic_type"].isin(safe_list(semantic_filter))
        ].copy()

    if rel_df.empty:
        return []

    return remove_duplicates_preserve_order(rel_df["source"].dropna().tolist())


def get_entity_volatility_level(entity):
    """
    Return update volatility level for one entity when available.
    """
    if "update_volatility_by_entity_df" not in globals():
        return "unknown"

    match = update_volatility_by_entity_df[
        update_volatility_by_entity_df["entity"] == entity
    ]

    if match.empty:
        return "unknown"

    return match.iloc[0].get("update_volatility_level", "unknown")


def get_entity_volatility_score(entity):
    """
    Return update volatility score for one entity when available.
    """
    if "update_volatility_by_entity_df" not in globals():
        return 0.0

    match = update_volatility_by_entity_df[
        update_volatility_by_entity_df["entity"] == entity
    ]

    if match.empty:
        return 0.0

    return safe_float(match.iloc[0].get("update_volatility_score", 0.0))


def filter_volatile_entities(entities, minimum_score=0.01):
    """
    Return entities with non-zero update volatility.
    """
    return [
        entity for entity in safe_list(entities)
        if get_entity_volatility_score(entity) >= minimum_score
    ]


def build_candidate_json(
    candidate_id,
    query_name,
    g_class,
    root_entity,
    collection_name,
    embedded_entities,
    referenced_entities,
    materialized_entities,
    design_pattern,
    rationale,
):
    """
    Build a JSON-like specification for one MongoDB candidate.
    """
    return {
        "candidate_id": candidate_id,
        "query_name": query_name,
        "g_class": g_class,
        "root_entity": root_entity,
        "collection_name": collection_name,
        "design_pattern": design_pattern,
        "embedded_entities": safe_list(embedded_entities),
        "referenced_entities": safe_list(referenced_entities),
        "materialized_entities": safe_list(materialized_entities),
        "rationale": rationale,
    }


# ---------------------------------------------------------
# 3. Candidate instantiation logic by G class
# ---------------------------------------------------------

def instantiate_candidate_for_g_class(row, g_class):
    """
    Instantiate one concrete MongoDB candidate from one active G class.

    Returns
    -------
    dict
        One candidate configuration row.
    """
    query_name = row["query_name"]
    root_entity = row.get("selected_root")
    query_family = row.get("query_family")
    query_type = row.get("query_type")
    generic_class = row.get("generic_class")

    entities_touched = safe_list(row.get("entities_touched", []))
    reachable_entities = safe_list(row.get("reachable_entities", []))
    covered_relationships = safe_list(row.get("covered_relationships", []))
    remaining_relationships = safe_list(row.get("remaining_relationships", []))
    shared_target_entities = safe_list(row.get("shared_target_entities", []))

    delta_ratio = safe_float(row.get("DeltaRratio", 0.0))
    update_score = safe_float(row.get("query_update_volatility_score", 0.0))
    sharedness_score = safe_float(row.get("max_observed_sharedness_score", 0.0))

    volatile_entities = filter_volatile_entities(entities_touched)

    embedded_entities = []
    referenced_entities = []
    materialized_entities = []
    design_pattern = None
    variant = None
    rationale = None
    benchmark_priority = "secondary"

    if g_class == "G0":
        variant = "root_only"
        design_pattern = "root_only_collection"
        collection_name = make_collection_name(root_entity)
        embedded_entities = []
        referenced_entities = []
        materialized_entities = []
        rationale = (
            "Use a simple root collection because the query has no relationship "
            "traversal or no relevant relationship-reduction need."
        )
        benchmark_priority = "control"

    elif g_class == "G1":
        variant = "embed_descriptors"
        design_pattern = "embedded_descriptors"
        collection_name = make_collection_name(root_entity, "with_descriptors")
        embedded_entities = extract_target_entities_for_relationships(
            covered_relationships,
            semantic_filter=["descriptor"],
        )
        referenced_entities = []
        materialized_entities = []
        rationale = (
            "Embed descriptor entities close to the root because descriptor "
            "relationships are covered by the selected document depth."
        )
        benchmark_priority = "primary"

    elif g_class == "G2":
        variant = "embed_containment_children"
        design_pattern = "embedded_containment"
        collection_name = make_collection_name(root_entity, "with_containment")
        embedded_entities = extract_target_entities_for_relationships(
            covered_relationships,
            semantic_filter=["containment"],
        )
        referenced_entities = []
        materialized_entities = []
        rationale = (
            "Embed containment children because the query follows containment "
            "paths that can be represented as nested arrays or subdocuments."
        )
        benchmark_priority = "primary"

    elif g_class == "G3":
        variant = "association_references"
        design_pattern = "association_references"
        collection_name = make_collection_name(root_entity, "with_references")
        embedded_entities = []
        referenced_entities = extract_target_entities_for_relationships(
            covered_relationships + remaining_relationships,
            semantic_filter=["association", "ownership", "subtype"],
        )
        materialized_entities = []
        rationale = (
            "Represent associative, ownership, or subtype links through references "
            "or small embedded summaries because these links may connect reusable "
            "or independently managed entities."
        )
        benchmark_priority = "primary"

    elif g_class == "G4":
        variant = "deep_nested_path"
        design_pattern = "deep_nested_document"
        collection_name = make_collection_name(root_entity, "deep_nested")
        embedded_entities = [
            entity for entity in reachable_entities
            if entity != root_entity
        ]
        referenced_entities = extract_target_entities_for_relationships(
            remaining_relationships
        )
        materialized_entities = []
        rationale = (
            "Create a deep nested document following the query path because the "
            "query reaches entities at multiple conceptual depths."
        )
        benchmark_priority = "primary"

    elif g_class == "G5":
        variant = "shared_targets_as_references"
        design_pattern = "shared_target_reference_strategy"
        collection_name = make_collection_name(root_entity, "shared_refs")
        embedded_entities = [
            entity for entity in reachable_entities
            if entity != root_entity and entity not in shared_target_entities
        ]
        referenced_entities = shared_target_entities
        materialized_entities = []
        rationale = (
            "Use references for shared target entities to avoid excessive "
            "duplication when the same target is reused by many source entities."
        )
        benchmark_priority = "primary"

    elif g_class == "G6":
        variant = "materialized_query_view"
        design_pattern = "aggregation_materialized_collection"
        collection_name = make_collection_name(root_entity, f"{query_name}_materialized")
        embedded_entities = []
        referenced_entities = []
        materialized_entities = entities_touched
        rationale = (
            "Create a materialized query-oriented collection because the query "
            "has aggregation, ranking, or comparison behavior."
        )
        benchmark_priority = "primary"

    elif g_class == "G7":
        variant = "update_aware_references"
        design_pattern = "update_aware_reference_design"
        collection_name = make_collection_name(root_entity, "update_aware")
        embedded_entities = [
            entity for entity in reachable_entities
            if entity != root_entity and entity not in volatile_entities
        ]
        referenced_entities = volatile_entities
        materialized_entities = []
        rationale = (
            "Prefer references for update-volatile entities to reduce write "
            "amplification and document maintenance cost."
        )
        benchmark_priority = "secondary_affected"

    elif g_class == "G8":
        variant = "hybrid_embedded_reference"
        design_pattern = "hybrid_embedded_reference"
        collection_name = make_collection_name(root_entity, "hybrid")
        embedded_entities = [
            entity for entity in reachable_entities
            if entity != root_entity and entity not in shared_target_entities
        ]
        referenced_entities = remove_duplicates_preserve_order(
            extract_target_entities_for_relationships(remaining_relationships)
            + shared_target_entities
        )
        materialized_entities = []
        rationale = (
            "Use a hybrid design because some relationships are covered by the "
            "selected depth while others remain external."
        )
        benchmark_priority = "primary"

    elif g_class == "G9":
        variant = "tradeoff_benchmark_candidate"
        design_pattern = "benchmark_tradeoff_alternative"
        collection_name = make_collection_name(root_entity, "tradeoff")
        embedded_entities = [
            entity for entity in reachable_entities
            if entity != root_entity
            and entity not in shared_target_entities
            and entity not in volatile_entities
        ]
        referenced_entities = remove_duplicates_preserve_order(
            shared_target_entities + volatile_entities
        )
        materialized_entities = []
        rationale = (
            "Generate a benchmark-required trade-off candidate because the query "
            "has read-reduction potential together with update pressure, "
            "sharedness pressure, or mapping uncertainty."
        )
        benchmark_priority = "secondary_affected"

    else:
        variant = "unknown"
        design_pattern = "unknown"
        collection_name = make_collection_name(root_entity, "unknown")
        embedded_entities = []
        referenced_entities = []
        materialized_entities = []
        rationale = "Unknown G class."
        benchmark_priority = "review"

    embedded_entities = remove_duplicates_preserve_order(embedded_entities)
    referenced_entities = remove_duplicates_preserve_order([
        entity for entity in referenced_entities
        if entity not in embedded_entities
    ])
    materialized_entities = remove_duplicates_preserve_order(materialized_entities)

    candidate_id = make_candidate_id(query_name, g_class, variant)

    candidate_spec = build_candidate_json(
        candidate_id=candidate_id,
        query_name=query_name,
        g_class=g_class,
        root_entity=root_entity,
        collection_name=collection_name,
        embedded_entities=embedded_entities,
        referenced_entities=referenced_entities,
        materialized_entities=materialized_entities,
        design_pattern=design_pattern,
        rationale=rationale,
    )

    return {
        "candidate_id": candidate_id,
        "query_name": query_name,
        "generic_class": generic_class,
        "query_family": query_family,
        "query_type": query_type,
        "g_class": g_class,
        "root_entity": root_entity,
        "collection_name": collection_name,
        "variant": variant,
        "design_pattern": design_pattern,
        "benchmark_priority": benchmark_priority,
        "embedded_entities": embedded_entities,
        "referenced_entities": referenced_entities,
        "materialized_entities": materialized_entities,
        "n_embedded_entities": len(embedded_entities),
        "n_referenced_entities": len(referenced_entities),
        "n_materialized_entities": len(materialized_entities),
        "DeltaRratio": delta_ratio,
        "query_update_volatility_score": update_score,
        "max_observed_sharedness_score": sharedness_score,
        "physical_mapping_reliability": row.get("physical_mapping_reliability"),
        "combined_document_design_risk_class": row.get("combined_document_design_risk_class"),
        "rationale": rationale,
        "candidate_spec": candidate_spec,
        "candidate_spec_json": json.dumps(candidate_spec, ensure_ascii=False),
    }


# ---------------------------------------------------------
# 4. Instantiate candidates for all active G classes
# ---------------------------------------------------------

candidate_rows = []

matrix_by_query = {
    row["query_name"]: row
    for _, row in document_variable_matrix_df.iterrows()
}

for _, activation_row in g_class_activation_by_query_df.iterrows():
    query_name = activation_row["query_name"]
    active_g_classes = safe_list(activation_row.get("active_g_classes", []))

    if query_name not in matrix_by_query:
        continue

    matrix_row = matrix_by_query[query_name]

    for g_class in active_g_classes:
        candidate_rows.append(
            instantiate_candidate_for_g_class(
                row=matrix_row,
                g_class=g_class,
            )
        )

mongodb_configuration_candidates_df = pd.DataFrame(candidate_rows)


# ---------------------------------------------------------
# 5. Add control candidates
# ---------------------------------------------------------
#
# Every query should have at least one control candidate.
# The control candidate represents the normalized/reference baseline.
# ---------------------------------------------------------

control_candidate_rows = []

for _, row in document_variable_matrix_df.iterrows():
    query_name = row["query_name"]
    root_entity = row.get("selected_root")
    entities_touched = safe_list(row.get("entities_touched", []))

    candidate_id = make_candidate_id(query_name, "CONTROL", "normalized_reference_baseline")

    candidate_spec = build_candidate_json(
        candidate_id=candidate_id,
        query_name=query_name,
        g_class="CONTROL",
        root_entity=root_entity,
        collection_name=make_collection_name(root_entity, "normalized_baseline"),
        embedded_entities=[],
        referenced_entities=[
            entity for entity in entities_touched
            if entity != root_entity
        ],
        materialized_entities=[],
        design_pattern="normalized_reference_baseline",
        rationale=(
            "Control configuration. Keep related entities as independent "
            "collections connected by references."
        ),
    )

    control_candidate_rows.append({
        "candidate_id": candidate_id,
        "query_name": query_name,
        "generic_class": row.get("generic_class"),
        "query_family": row.get("query_family"),
        "query_type": row.get("query_type"),
        "g_class": "CONTROL",
        "root_entity": root_entity,
        "collection_name": make_collection_name(root_entity, "normalized_baseline"),
        "variant": "normalized_reference_baseline",
        "design_pattern": "normalized_reference_baseline",
        "benchmark_priority": "control",
        "embedded_entities": [],
        "referenced_entities": [
            entity for entity in entities_touched
            if entity != root_entity
        ],
        "materialized_entities": [],
        "n_embedded_entities": 0,
        "n_referenced_entities": max(0, len(entities_touched) - 1),
        "n_materialized_entities": 0,
        "DeltaRratio": safe_float(row.get("DeltaRratio", 0.0)),
        "query_update_volatility_score": safe_float(row.get("query_update_volatility_score", 0.0)),
        "max_observed_sharedness_score": safe_float(row.get("max_observed_sharedness_score", 0.0)),
        "physical_mapping_reliability": row.get("physical_mapping_reliability"),
        "combined_document_design_risk_class": row.get("combined_document_design_risk_class"),
        "rationale": candidate_spec["rationale"],
        "candidate_spec": candidate_spec,
        "candidate_spec_json": json.dumps(candidate_spec, ensure_ascii=False),
    })

control_candidates_df = pd.DataFrame(control_candidate_rows)

mongodb_configuration_candidates_df = pd.concat(
    [
        mongodb_configuration_candidates_df,
        control_candidates_df,
    ],
    ignore_index=True,
)


# ---------------------------------------------------------
# 6. Remove duplicated candidate IDs
# ---------------------------------------------------------

mongodb_configuration_candidates_df = (
    mongodb_configuration_candidates_df
    .drop_duplicates(subset=["candidate_id"])
    .sort_values(["query_name", "benchmark_priority", "g_class", "variant"])
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 7. Compatibility aliases
# ---------------------------------------------------------

concrete_mongodb_candidates_df = mongodb_configuration_candidates_df.copy()
mongo_candidate_configurations_df = mongodb_configuration_candidates_df.copy()
mongodb_configuration_candidates_long_df = mongodb_configuration_candidates_df.copy()


# ---------------------------------------------------------
# 8. Add candidate information back to the document matrix
# ---------------------------------------------------------

candidate_profile_by_query_df = (
    mongodb_configuration_candidates_df
    .groupby("query_name", dropna=False)
    .agg(
        n_mongodb_candidates=("candidate_id", "count"),
        candidate_ids=("candidate_id", lambda s: sorted(set(s))),
        candidate_design_patterns=("design_pattern", lambda s: sorted(set(s))),
        candidate_g_classes=("g_class", lambda s: sorted(set(s))),
        n_primary_candidates=(
            "benchmark_priority",
            lambda s: int((s == "primary").sum()),
        ),
        n_secondary_affected_candidates=(
            "benchmark_priority",
            lambda s: int((s == "secondary_affected").sum()),
        ),
        n_control_candidates=(
            "benchmark_priority",
            lambda s: int((s == "control").sum()),
        ),
    )
    .reset_index()
)

document_variable_matrix_with_candidates_df = (
    document_variable_matrix_df
    .drop(
        columns=[
            col for col in [
                "n_mongodb_candidates",
                "candidate_ids",
                "candidate_design_patterns",
                "candidate_g_classes",
                "n_primary_candidates",
                "n_secondary_affected_candidates",
                "n_control_candidates",
            ]
            if col in document_variable_matrix_df.columns
        ],
        errors="ignore",
    )
    .merge(
        candidate_profile_by_query_df,
        on="query_name",
        how="left",
    )
)

document_variable_matrix_df = document_variable_matrix_with_candidates_df.copy()


# ---------------------------------------------------------
# 9. Summary tables
# ---------------------------------------------------------

mongodb_candidate_summary_df = (
    mongodb_configuration_candidates_df
    .groupby(["g_class", "design_pattern", "benchmark_priority"], dropna=False)
    .agg(
        n_candidates=("candidate_id", "count"),
        queries=("query_name", lambda s: sorted(set(s))),
        avg_embedded_entities=("n_embedded_entities", "mean"),
        avg_referenced_entities=("n_referenced_entities", "mean"),
        avg_DeltaRratio=("DeltaRratio", "mean"),
        avg_update_volatility=("query_update_volatility_score", "mean"),
        avg_sharedness=("max_observed_sharedness_score", "mean"),
    )
    .reset_index()
    .sort_values(["g_class", "design_pattern", "benchmark_priority"])
    .reset_index(drop=True)
)

mongodb_candidate_by_query_df = (
    mongodb_configuration_candidates_df
    .groupby("query_name", dropna=False)
    .agg(
        n_candidates=("candidate_id", "count"),
        candidate_ids=("candidate_id", lambda s: sorted(set(s))),
        design_patterns=("design_pattern", lambda s: sorted(set(s))),
        g_classes=("g_class", lambda s: sorted(set(s))),
        benchmark_priorities=("benchmark_priority", lambda s: sorted(set(s))),
    )
    .reset_index()
    .sort_values("query_name")
    .reset_index(drop=True)
)

mongodb_candidate_priority_summary_df = (
    mongodb_configuration_candidates_df
    .groupby("benchmark_priority", dropna=False)
    .agg(
        n_candidates=("candidate_id", "count"),
        queries=("query_name", lambda s: sorted(set(s))),
        design_patterns=("design_pattern", lambda s: sorted(set(s))),
    )
    .reset_index()
    .sort_values("benchmark_priority")
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 10. Validation tables
# ---------------------------------------------------------

queries_without_candidates_df = document_variable_matrix_df[
    document_variable_matrix_df["n_mongodb_candidates"].isna()
    | (document_variable_matrix_df["n_mongodb_candidates"] == 0)
].copy()

queries_without_control_candidate_df = mongodb_candidate_by_query_df[
    ~mongodb_candidate_by_query_df["benchmark_priorities"]
    .apply(lambda values: "control" in values)
].copy()

candidate_without_root_df = mongodb_configuration_candidates_df[
    mongodb_configuration_candidates_df["root_entity"].isna()
].copy()

candidate_duplicate_ids_df = mongodb_configuration_candidates_df[
    mongodb_configuration_candidates_df.duplicated("candidate_id", keep=False)
].copy()

mongodb_candidate_validation_rows = [
    {
        "check_name": "every_query_has_candidates",
        "status": "passed" if queries_without_candidates_df.empty else "failed",
        "details": f"queries_without_candidates={len(queries_without_candidates_df)}",
    },
    {
        "check_name": "every_query_has_control_candidate",
        "status": "passed" if queries_without_control_candidate_df.empty else "failed",
        "details": f"queries_without_control={len(queries_without_control_candidate_df)}",
    },
    {
        "check_name": "every_candidate_has_root",
        "status": "passed" if candidate_without_root_df.empty else "failed",
        "details": f"candidates_without_root={len(candidate_without_root_df)}",
    },
    {
        "check_name": "candidate_ids_are_unique",
        "status": "passed" if candidate_duplicate_ids_df.empty else "failed",
        "details": f"duplicated_candidate_ids={len(candidate_duplicate_ids_df)}",
    },
]

mongodb_candidate_validation_df = pd.DataFrame(
    mongodb_candidate_validation_rows
)

failed_mongodb_candidate_validation_df = mongodb_candidate_validation_df[
    mongodb_candidate_validation_df["status"] == "failed"
].copy()


# ---------------------------------------------------------
# 11. Display outputs
# ---------------------------------------------------------

print("Concrete MongoDB configuration candidates instantiated successfully.")

print("\nMongoDB configuration candidates:")
display(mongodb_configuration_candidates_df)

print("\nMongoDB candidate summary:")
display(mongodb_candidate_summary_df)

print("\nMongoDB candidate by query:")
display(mongodb_candidate_by_query_df)

print("\nMongoDB candidate priority summary:")
display(mongodb_candidate_priority_summary_df)

print("\nCandidate validation:")
display(mongodb_candidate_validation_df)

if not failed_mongodb_candidate_validation_df.empty:
    print("\nWarning: some MongoDB candidate validation checks failed.")
    display(failed_mongodb_candidate_validation_df)


# ---------------------------------------------------------
# 12. Save BLOCK V22 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    mongodb_configuration_candidates_df,
    "variables/block_v22/mongodb_configuration_candidates.csv",
)

save_dataframe(
    concrete_mongodb_candidates_df,
    "variables/block_v22/concrete_mongodb_candidates.csv",
)

save_dataframe(
    mongo_candidate_configurations_df,
    "variables/block_v22/mongo_candidate_configurations.csv",
)

save_dataframe(
    mongodb_configuration_candidates_long_df,
    "variables/block_v22/mongodb_configuration_candidates_long.csv",
)

save_dataframe(
    control_candidates_df,
    "variables/block_v22/control_candidates.csv",
)

save_dataframe(
    candidate_profile_by_query_df,
    "variables/block_v22/candidate_profile_by_query.csv",
)

save_dataframe(
    document_variable_matrix_with_candidates_df,
    "variables/block_v22/document_variable_matrix_with_candidates.csv",
)

save_dataframe(
    document_variable_matrix_df,
    "variables/block_v22/document_variable_matrix.csv",
)

save_dataframe(
    mongodb_candidate_summary_df,
    "variables/block_v22/mongodb_candidate_summary.csv",
)

save_dataframe(
    mongodb_candidate_by_query_df,
    "variables/block_v22/mongodb_candidate_by_query.csv",
)

save_dataframe(
    mongodb_candidate_priority_summary_df,
    "variables/block_v22/mongodb_candidate_priority_summary.csv",
)

save_dataframe(
    mongodb_candidate_validation_df,
    "variables/block_v22/mongodb_candidate_validation.csv",
)

save_dataframe(
    failed_mongodb_candidate_validation_df,
    "variables/block_v22/failed_mongodb_candidate_validation.csv",
)

save_dataframe(
    queries_without_candidates_df,
    "variables/block_v22/queries_without_candidates.csv",
)

save_dataframe(
    queries_without_control_candidate_df,
    "variables/block_v22/queries_without_control_candidate.csv",
)

save_dataframe(
    candidate_without_root_df,
    "variables/block_v22/candidate_without_root.csv",
)

save_dataframe(
    candidate_duplicate_ids_df,
    "variables/block_v22/candidate_duplicate_ids.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "n_candidates": len(mongodb_configuration_candidates_df),
        "n_queries": int(mongodb_configuration_candidates_df["query_name"].nunique()),
        "n_control_candidates": int(
            (mongodb_configuration_candidates_df["benchmark_priority"] == "control").sum()
        ),
        "n_primary_candidates": int(
            (mongodb_configuration_candidates_df["benchmark_priority"] == "primary").sum()
        ),
        "n_secondary_affected_candidates": int(
            (mongodb_configuration_candidates_df["benchmark_priority"] == "secondary_affected").sum()
        ),
        "design_pattern_counts": (
            mongodb_configuration_candidates_df["design_pattern"]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "g_class_counts": (
            mongodb_configuration_candidates_df["g_class"]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "n_failed_validation_checks": len(failed_mongodb_candidate_validation_df),
        "output_files": {
            "mongodb_configuration_candidates_csv": "variables/block_v22/mongodb_configuration_candidates.csv",
            "concrete_mongodb_candidates_csv": "variables/block_v22/concrete_mongodb_candidates.csv",
            "mongo_candidate_configurations_csv": "variables/block_v22/mongo_candidate_configurations.csv",
            "mongodb_configuration_candidates_long_csv": "variables/block_v22/mongodb_configuration_candidates_long.csv",
            "control_candidates_csv": "variables/block_v22/control_candidates.csv",
            "candidate_profile_by_query_csv": "variables/block_v22/candidate_profile_by_query.csv",
            "document_variable_matrix_with_candidates_csv": "variables/block_v22/document_variable_matrix_with_candidates.csv",
            "mongodb_candidate_summary_csv": "variables/block_v22/mongodb_candidate_summary.csv",
            "mongodb_candidate_by_query_csv": "variables/block_v22/mongodb_candidate_by_query.csv",
            "mongodb_candidate_priority_summary_csv": "variables/block_v22/mongodb_candidate_priority_summary.csv",
            "mongodb_candidate_validation_csv": "variables/block_v22/mongodb_candidate_validation.csv",
        },
    },
    "variables/block_v22/mongodb_configuration_candidates_metadata.json",
)

write_execution_log(
    block_name="BLOCK V22 — INSTANTIATE CONCRETE MONGODB CONFIGURATION CANDIDATES",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_candidates": len(mongodb_configuration_candidates_df),
        "n_queries": int(mongodb_configuration_candidates_df["query_name"].nunique()),
        "n_control_candidates": int(
            (mongodb_configuration_candidates_df["benchmark_priority"] == "control").sum()
        ),
        "n_primary_candidates": int(
            (mongodb_configuration_candidates_df["benchmark_priority"] == "primary").sum()
        ),
        "n_secondary_affected_candidates": int(
            (mongodb_configuration_candidates_df["benchmark_priority"] == "secondary_affected").sum()
        ),
        "n_failed_validation_checks": len(failed_mongodb_candidate_validation_df),
        "mongodb_configuration_candidates_csv": "variables/block_v22/mongodb_configuration_candidates.csv",
        "mongodb_candidate_summary_csv": "variables/block_v22/mongodb_candidate_summary.csv",
        "mongodb_candidate_validation_csv": "variables/block_v22/mongodb_candidate_validation.csv",
        "metadata_json": "variables/block_v22/mongodb_configuration_candidates_metadata.json",
    },
)


# ---------------------------------------------------------
# 13. Update README section for BLOCK V22
# ---------------------------------------------------------

block_v22_readme_body = f"""
This block instantiates concrete MongoDB configuration candidates from the
active generic classes G0-G9.

Main variables created:

    mongodb_configuration_candidates_df
    concrete_mongodb_candidates_df
    mongo_candidate_configurations_df

Additional variables created:

    mongodb_configuration_candidates_long_df
    control_candidates_df
    candidate_profile_by_query_df
    document_variable_matrix_with_candidates_df
    mongodb_candidate_summary_df
    mongodb_candidate_by_query_df
    mongodb_candidate_priority_summary_df
    mongodb_candidate_validation_df

Number of MongoDB candidates:

    {len(mongodb_configuration_candidates_df)}

Number of queries with candidates:

    {int(mongodb_configuration_candidates_df["query_name"].nunique())}

Number of control candidates:

    {int((mongodb_configuration_candidates_df["benchmark_priority"] == "control").sum())}

Number of primary candidates:

    {int((mongodb_configuration_candidates_df["benchmark_priority"] == "primary").sum())}

Number of secondary affected candidates:

    {int((mongodb_configuration_candidates_df["benchmark_priority"] == "secondary_affected").sum())}

Failed validation checks:

    {len(failed_mongodb_candidate_validation_df)}

Generated candidate design patterns:

    {sorted(mongodb_configuration_candidates_df["design_pattern"].dropna().unique().tolist())}

Generated reproducibility files:

    variables/block_v22/mongodb_configuration_candidates.csv
    variables/block_v22/concrete_mongodb_candidates.csv
    variables/block_v22/mongo_candidate_configurations.csv
    variables/block_v22/mongodb_configuration_candidates_long.csv
    variables/block_v22/control_candidates.csv
    variables/block_v22/candidate_profile_by_query.csv
    variables/block_v22/document_variable_matrix_with_candidates.csv
    variables/block_v22/mongodb_candidate_summary.csv
    variables/block_v22/mongodb_candidate_by_query.csv
    variables/block_v22/mongodb_candidate_priority_summary.csv
    variables/block_v22/mongodb_candidate_validation.csv
    variables/block_v22/mongodb_configuration_candidates_metadata.json

Methodological meaning:

    This block translates generic activation classes into concrete MongoDB
    candidate design patterns.

    These candidates are still logical/physical design specifications.
    They are not loaded into MongoDB yet.

Important note:

    Every query receives a control candidate using a normalized reference
    baseline. This allows later benchmark comparison between the baseline
    and the generated document-oriented alternatives.
"""

update_readme_section(
    section_title="BLOCK V22 — Instantiate Concrete MongoDB Configuration Candidates",
    section_body=block_v22_readme_body,
)

Concrete MongoDB configuration candidates instantiated successfully.

MongoDB configuration candidates:


,candidate_id,query_name,generic_class,query_family,query_type,g_class,root_entity,collection_name,variant,design_pattern,...,n_referenced_entities,n_materialized_entities,DeltaRratio,query_update_volatility_score,max_observed_sharedness_score,physical_mapping_reliability,combined_document_design_risk_class,rationale,candidate_spec,candidate_spec_json
0,q10_create_account_holding_and_buy_transaction...,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,CONTROL,Person,persons_normalized_baseline,normalized_reference_baseline,normalized_reference_baseline,...,5,0,1.0,0.932000,0.295882,ok,strong_read_candidate_high_update_pressure,Control configuration. Keep related entities a...,{'candidate_id': 'q10_create_account_holding_a...,"{""candidate_id"": ""q10_create_account_holding_a..."
1,q10_create_account_holding_and_buy_transaction...,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,G2,Person,persons_with_containment,embed_containment_children,embedded_containment,...,0,0,1.0,0.932000,0.295882,ok,strong_read_candidate_high_update_pressure,Embed containment children because the query f...,{'candidate_id': 'q10_create_account_holding_a...,"{""candidate_id"": ""q10_create_account_holding_a..."
2,q10_create_account_holding_and_buy_transaction...,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,G3,Person,persons_with_references,association_references,association_references,...,3,0,1.0,0.932000,0.295882,ok,strong_read_candidate_high_update_pressure,"Represent associative, ownership, or subtype l...",{'candidate_id': 'q10_create_account_holding_a...,"{""candidate_id"": ""q10_create_account_holding_a..."
3,q10_create_account_holding_and_buy_transaction...,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,G4,Person,persons_deep_nested,deep_nested_path,deep_nested_document,...,0,0,1.0,0.932000,0.295882,ok,strong_read_candidate_high_update_pressure,Create a deep nested document following the qu...,{'candidate_id': 'q10_create_account_holding_a...,"{""candidate_id"": ""q10_create_account_holding_a..."
4,q10_create_account_holding_and_buy_transaction...,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,G5,Person,persons_shared_refs,shared_targets_as_references,shared_target_reference_strategy,...,4,0,1.0,0.932000,0.295882,ok,strong_read_candidate_high_update_pressure,Use references for shared target entities to a...,{'candidate_id': 'q10_create_account_holding_a...,"{""candidate_id"": ""q10_create_account_holding_a..."
5,q10_create_account_holding_and_buy_transaction...,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,G7,Person,persons_update_aware,update_aware_references,update_aware_reference_design,...,6,0,1.0,0.932000,0.295882,ok,strong_read_candidate_high_update_pressure,Prefer references for update-volatile entities...,{'candidate_id': 'q10_create_account_holding_a...,"{""candidate_id"": ""q10_create_account_holding_a..."
6,q10_create_account_holding_and_buy_transaction...,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,G9,Person,persons_tradeoff,tradeoff_benchmark_candidate,benchmark_tradeoff_alternative,...,6,0,1.0,0.932000,0.295882,ok,strong_read_candidate_high_update_pressure,Generate a benchmark-required trade-off candid...,{'candidate_id': 'q10_create_account_holding_a...,"{""candidate_id"": ""q10_create_account_holding_a..."
7,q1_company_profile_ibm__control__normalized_re...,Q1_CompanyProfileIBM,QG1,Local lookup,select,CONTROL,Corporation,corporations_normalized_baseline,normalized_reference_baseline,normalized_reference_baseline,...,0,0,0.0,0.000000,0.000000,ok,low_overall_design_pressure,Control configuration. Keep related entities a...,{'candidate_id': 'q1_company_profile_ibm__co


MongoDB candidate summary:


,g_class,design_pattern,benchmark_priority,n_candidates,queries,avg_embedded_entities,avg_referenced_entities,avg_DeltaRratio,avg_update_volatility,avg_sharedness
0,CONTROL,normalized_reference_baseline,control,10,"[Q10_CreateAccountHoldingAndBuyTransaction, Q1...",0.000000,3.600,0.9,0.550669,0.266475
1,G0,root_only_collection,control,1,[Q1_CompanyProfileIBM],0.000000,0.000,0.0,0.000000,0.000000
2,G1,embedded_descriptors,primary,2,[Q2_CompanyWithIndustryCountryAndListedSecurit...,2.000000,0.000,1.0,0.363000,0.297418
3,G2,embedded_containment,primary,6,"[Q10_CreateAccountHoldingAndBuyTransaction, Q3...",2.000000,0.000,1.0,0.688614,0.295672
4,G3,association_references,primary,8,"[Q10_CreateAccountHoldingAndBuyTransaction, Q2...",0.000000,2.125,1.0,0.688336,0.296266
5,G4,deep_nested_document,primary,6,"[Q10_CreateAccountHoldingAndBuyTransaction, Q3...",4.500000,0.000,1.0,0.688614,0.295672
6,G5,shared_target_reference_strategy,primary,8,"[Q10_CreateAccountHoldingAndBuyTransaction, Q2...",1.250000,3.000,1.0,0.688336,0.296266
7,G6,aggregation_materialized_collection,primary,2,"[Q7_PersonsWhoBoughtMoreIBMThanSold, Q8_IBMTra...",0.000000,0.000,1.0,0.709643,0.295882
8,G7,update_aware_reference_design,secondary_affected,8,"[Q10_CreateAccountHoldingAndBuyTransaction, Q2...",1.500000,3.375,1.0,0.688336,0.296266
9,G9,benchmark_tradeoff_alternative,secondary_affected,9,"[Q10_CreateAccountHoldingAndBuyTransaction, Q2...",0.777778,4.000,1.0,0.611854,0.296084



MongoDB candidate by query:


...


MongoDB candidate priority summary:


,benchmark_priority,n_candidates,queries,design_patterns
0,control,11,"[Q10_CreateAccountHoldingAndBuyTransaction, Q1...","[normalized_reference_baseline, root_only_coll..."
1,primary,32,"[Q10_CreateAccountHoldingAndBuyTransaction, Q2...","[aggregation_materialized_collection, associat..."
2,secondary_affected,17,"[Q10_CreateAccountHoldingAndBuyTransaction, Q2...","[benchmark_tradeoff_alternative, update_aware_..."



Candidate validation:


,check_name,status,details
0,every_query_has_candidates,passed,queries_without_candidates=0
1,every_query_has_control_candidate,passed,queries_without_control=0
2,every_candidate_has_root,passed,candidates_without_root=0
3,candidate_ids_are_unique,passed,duplicated_candidate_ids=0


Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v22/mongodb_configuration_candidates.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v22/concrete_mongodb_candidates.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v22/mongo_candidate_configurations.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v22/mongodb_configuration_candidates_long.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v22/control_candidates.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v22/candidate_profile_by_query.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset 

In [99]:
mongodb_candidate_validation_df

,check_name,status,details
0,every_query_has_candidates,passed,queries_without_candidates=0
1,every_query_has_control_candidate,passed,queries_without_control=0
2,every_candidate_has_root,passed,candidates_without_root=0
3,candidate_ids_are_unique,passed,duplicated_candidate_ids=0


In [100]:
# =========================================================
# BLOCK V23 — SELECT PRIMARY, SECONDARY AFFECTED, AND CONTROL CONFIGURATIONS
# =========================================================
#
# Purpose:
# This block selects MongoDB configuration candidates for the
# benchmark execution plan.
#
# It organizes the candidates generated in BLOCK V22 into:
#
# - primary:
#   main configurations expected to benefit the target query;
#
# - secondary_affected:
#   configurations that may benefit reads but require attention
#   because of update volatility, sharedness, or trade-offs;
#
# - control:
#   baseline normalized/reference configurations used for comparison.
#
# Main outputs:
# - benchmark_configuration_selection_df
# - primary_configurations_df
# - secondary_affected_configurations_df
# - control_configurations_df
#
# Additional outputs:
# - benchmark_execution_plan_df
# - benchmark_selection_summary_df
# - benchmark_selection_validation_df
#
# Methodological role:
# This block prepares the final configuration catalog that will later
# be consumed by the external load and benchmark script executed on
# the local/server environment.
#
# Important:
# This block does not load MongoDB.
# This block does not run benchmark queries.
# =========================================================

import pandas as pd
import json


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "mongodb_configuration_candidates_df",
    "mongodb_candidate_by_query_df",
    "document_variable_matrix_df",
    "g_class_activation_by_query_df",
    "fiben_scenario",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first BLOCK V22 — INSTANTIATE CONCRETE MONGODB CONFIGURATION CANDIDATES."
        )


# ---------------------------------------------------------
# 2. Helper functions
# ---------------------------------------------------------

def safe_list(value):
    """
    Convert a value into a clean Python list.
    """
    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    if value is None:
        return []

    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    return [value]


def safe_float(value, default=0.0):
    """
    Convert a value to float safely.
    """
    try:
        if pd.isna(value):
            return float(default)
    except Exception:
        pass

    try:
        return float(value)
    except Exception:
        return float(default)


def safe_int(value, default=0):
    """
    Convert a value to int safely.
    """
    try:
        if pd.isna(value):
            return int(default)
    except Exception:
        pass

    try:
        return int(value)
    except Exception:
        return int(default)


def safe_bool(value, default=False):
    """
    Convert a value to bool safely.
    """
    try:
        if pd.isna(value):
            return bool(default)
    except Exception:
        pass

    return bool(value)


def normalize_benchmark_group(row):
    """
    Convert candidate priority and G class into the final benchmark group.

    CONTROL candidates always remain control.

    G0 is treated as a primary generated candidate when it is not the
    explicit CONTROL baseline. This is useful for local lookup queries
    where a root-only document can be the main document design.
    """
    g_class = str(row.get("g_class", ""))
    priority = str(row.get("benchmark_priority", ""))

    if g_class == "CONTROL":
        return "control"

    if priority == "secondary_affected":
        return "secondary_affected"

    if priority == "primary":
        return "primary"

    if g_class == "G0":
        return "primary"

    if priority == "control":
        return "primary"

    return "secondary_affected"


def compute_candidate_selection_score(row):
    """
    Compute a score used to order candidates inside each query/group.

    The score favors:
    - high DeltaRratio;
    - primary candidates;
    - reasonable embedding;
    - lower update/sharedness pressure for primary candidates.
    """
    delta_ratio = safe_float(row.get("DeltaRratio", 0.0))
    update_score = safe_float(row.get("query_update_volatility_score", 0.0))
    sharedness_score = safe_float(row.get("max_observed_sharedness_score", 0.0))
    n_embedded = safe_float(row.get("n_embedded_entities", 0.0))
    n_referenced = safe_float(row.get("n_referenced_entities", 0.0))

    benchmark_group = row.get("final_benchmark_group", row.get("benchmark_priority", ""))

    group_bonus = {
        "primary": 0.30,
        "secondary_affected": 0.15,
        "control": 0.00,
    }.get(benchmark_group, 0.00)

    embedding_signal = min(n_embedded / 5.0, 1.0)
    reference_signal = min(n_referenced / 5.0, 1.0)

    maintenance_penalty = 0.20 * update_score + 0.20 * sharedness_score

    if benchmark_group == "control":
        return 0.0

    score = (
        group_bonus
        + 0.45 * delta_ratio
        + 0.20 * embedding_signal
        + 0.05 * reference_signal
        - maintenance_penalty
    )

    return max(0.0, round(score, 6))


def classify_selection_role(row):
    """
    Add a readable role for benchmark interpretation.
    """
    group = row.get("final_benchmark_group")
    g_class = row.get("g_class")

    if group == "control":
        return "baseline_comparison"

    if group == "primary":
        return "main_candidate"

    if group == "secondary_affected":
        return "tradeoff_candidate"

    return "review_candidate"


def candidate_is_generated(row):
    """
    Return True if the candidate comes from G0-G9 activation,
    not from the explicit CONTROL baseline.
    """
    return str(row.get("g_class")) != "CONTROL"


# ---------------------------------------------------------
# 3. Start from all MongoDB candidates
# ---------------------------------------------------------

benchmark_configuration_selection_df = mongodb_configuration_candidates_df.copy()

benchmark_configuration_selection_df["final_benchmark_group"] = (
    benchmark_configuration_selection_df
    .apply(normalize_benchmark_group, axis=1)
)

benchmark_configuration_selection_df["is_generated_candidate"] = (
    benchmark_configuration_selection_df
    .apply(candidate_is_generated, axis=1)
)

benchmark_configuration_selection_df["selection_role"] = (
    benchmark_configuration_selection_df
    .apply(classify_selection_role, axis=1)
)

benchmark_configuration_selection_df["candidate_selection_score"] = (
    benchmark_configuration_selection_df
    .apply(compute_candidate_selection_score, axis=1)
)


# ---------------------------------------------------------
# 4. Add query-level activation and risk metadata
# ---------------------------------------------------------

query_metadata_cols = [
    "query_name",
    "active_g_classes",
    "primary_activation_class",
    "primary_activation_name",
    "primary_activation_score",
    "DeltaRratio",
    "query_update_volatility_score",
    "max_observed_sharedness_score",
    "combined_document_design_risk_class",
    "document_potential_class",
    "physical_mapping_reliability",
]

available_query_metadata_cols = [
    col for col in query_metadata_cols
    if col in document_variable_matrix_df.columns
]

query_metadata_df = document_variable_matrix_df[
    available_query_metadata_cols
].copy()

# Avoid duplicate columns before merge.
cols_to_replace = [
    col for col in available_query_metadata_cols
    if col != "query_name" and col in benchmark_configuration_selection_df.columns
]

benchmark_configuration_selection_df = (
    benchmark_configuration_selection_df
    .drop(columns=cols_to_replace, errors="ignore")
    .merge(
        query_metadata_df,
        on="query_name",
        how="left",
    )
)


# ---------------------------------------------------------
# 5. Select benchmark subsets
# ---------------------------------------------------------
#
# We keep:
# - all primary candidates;
# - all secondary affected candidates;
# - all control candidates.
#
# If later the candidate set becomes too large, a cap can be added here.
# ---------------------------------------------------------

primary_configurations_df = benchmark_configuration_selection_df[
    benchmark_configuration_selection_df["final_benchmark_group"] == "primary"
].copy()

secondary_affected_configurations_df = benchmark_configuration_selection_df[
    benchmark_configuration_selection_df["final_benchmark_group"] == "secondary_affected"
].copy()

control_configurations_df = benchmark_configuration_selection_df[
    benchmark_configuration_selection_df["final_benchmark_group"] == "control"
].copy()


# ---------------------------------------------------------
# 6. Create benchmark execution plan
# ---------------------------------------------------------
#
# This plan is ordered to make later execution easier:
# 1. control configurations first;
# 2. primary configurations;
# 3. secondary affected configurations.
# ---------------------------------------------------------

benchmark_group_order = {
    "control": 0,
    "primary": 1,
    "secondary_affected": 2,
    "review": 3,
}

benchmark_configuration_selection_df["benchmark_group_order"] = (
    benchmark_configuration_selection_df["final_benchmark_group"]
    .map(benchmark_group_order)
    .fillna(9)
    .astype(int)
)

benchmark_execution_plan_df = (
    benchmark_configuration_selection_df
    .sort_values(
        [
            "query_name",
            "benchmark_group_order",
            "candidate_selection_score",
            "g_class",
            "variant",
        ],
        ascending=[True, True, False, True, True],
    )
    .reset_index(drop=True)
)

benchmark_execution_plan_df["benchmark_plan_order"] = range(
    1,
    len(benchmark_execution_plan_df) + 1,
)

benchmark_execution_plan_df["should_load_configuration"] = True
benchmark_execution_plan_df["should_run_read_benchmark"] = True

benchmark_execution_plan_df["should_run_update_benchmark"] = (
    benchmark_execution_plan_df["final_benchmark_group"].isin(
        ["secondary_affected", "primary"]
    )
    & (
        benchmark_execution_plan_df["query_update_volatility_score"]
        .apply(lambda v: safe_float(v) > 0)
    )
)

benchmark_execution_plan_df["benchmark_notes"] = benchmark_execution_plan_df.apply(
    lambda row: (
        "Control baseline for comparison."
        if row["final_benchmark_group"] == "control"
        else (
            "Primary candidate generated from activation logic."
            if row["final_benchmark_group"] == "primary"
            else "Secondary affected candidate; evaluate read benefit and maintenance cost."
        )
    ),
    axis=1,
)


# ---------------------------------------------------------
# 7. Build compact query-level selection profile
# ---------------------------------------------------------

benchmark_selection_by_query_df = (
    benchmark_execution_plan_df
    .groupby("query_name", dropna=False)
    .agg(
        n_selected_configurations=("candidate_id", "count"),
        selected_candidate_ids=("candidate_id", lambda s: sorted(set(s))),
        selected_design_patterns=("design_pattern", lambda s: sorted(set(s))),
        selected_g_classes=("g_class", lambda s: sorted(set(s))),
        n_primary_configurations=(
            "final_benchmark_group",
            lambda s: int((s == "primary").sum()),
        ),
        n_secondary_affected_configurations=(
            "final_benchmark_group",
            lambda s: int((s == "secondary_affected").sum()),
        ),
        n_control_configurations=(
            "final_benchmark_group",
            lambda s: int((s == "control").sum()),
        ),
        max_candidate_selection_score=("candidate_selection_score", "max"),
    )
    .reset_index()
    .sort_values("query_name")
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 8. Add selection profile back to document variable matrix
# ---------------------------------------------------------

selection_profile_cols_to_replace = [
    "n_selected_configurations",
    "selected_candidate_ids",
    "selected_design_patterns",
    "selected_g_classes",
    "n_primary_configurations",
    "n_secondary_affected_configurations",
    "n_control_configurations",
    "max_candidate_selection_score",
]

document_variable_matrix_with_benchmark_selection_df = (
    document_variable_matrix_df
    .drop(columns=selection_profile_cols_to_replace, errors="ignore")
    .merge(
        benchmark_selection_by_query_df,
        on="query_name",
        how="left",
    )
)

document_variable_matrix_df = document_variable_matrix_with_benchmark_selection_df.copy()


# ---------------------------------------------------------
# 9. Summary tables
# ---------------------------------------------------------

benchmark_selection_summary_df = (
    benchmark_execution_plan_df
    .groupby(
        [
            "final_benchmark_group",
            "design_pattern",
            "g_class",
        ],
        dropna=False,
    )
    .agg(
        n_configurations=("candidate_id", "count"),
        queries=("query_name", lambda s: sorted(set(s))),
        avg_selection_score=("candidate_selection_score", "mean"),
        avg_DeltaRratio=("DeltaRratio", "mean"),
        avg_update_volatility=("query_update_volatility_score", "mean"),
        avg_sharedness=("max_observed_sharedness_score", "mean"),
    )
    .reset_index()
    .sort_values(
        [
            "final_benchmark_group",
            "design_pattern",
            "g_class",
        ]
    )
    .reset_index(drop=True)
)

benchmark_group_summary_df = (
    benchmark_execution_plan_df
    .groupby("final_benchmark_group", dropna=False)
    .agg(
        n_configurations=("candidate_id", "count"),
        n_queries=("query_name", "nunique"),
        candidate_ids=("candidate_id", lambda s: sorted(set(s))),
        design_patterns=("design_pattern", lambda s: sorted(set(s))),
    )
    .reset_index()
    .sort_values("final_benchmark_group")
    .reset_index(drop=True)
)

benchmark_selection_by_query_summary_df = (
    benchmark_selection_by_query_df
    .groupby(
        [
            "n_primary_configurations",
            "n_secondary_affected_configurations",
            "n_control_configurations",
        ],
        dropna=False,
    )
    .agg(
        n_queries=("query_name", "count"),
        queries=("query_name", lambda s: sorted(set(s))),
    )
    .reset_index()
    .sort_values(
        [
            "n_primary_configurations",
            "n_secondary_affected_configurations",
            "n_control_configurations",
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 10. Validation tables
# ---------------------------------------------------------

queries_without_primary_df = benchmark_selection_by_query_df[
    benchmark_selection_by_query_df["n_primary_configurations"] == 0
].copy()

queries_without_control_df = benchmark_selection_by_query_df[
    benchmark_selection_by_query_df["n_control_configurations"] == 0
].copy()

queries_without_any_selected_config_df = benchmark_selection_by_query_df[
    benchmark_selection_by_query_df["n_selected_configurations"] == 0
].copy()

duplicated_selected_candidate_ids_df = benchmark_execution_plan_df[
    benchmark_execution_plan_df.duplicated("candidate_id", keep=False)
].copy()

selected_candidates_without_collection_df = benchmark_execution_plan_df[
    benchmark_execution_plan_df["collection_name"].isna()
    | (benchmark_execution_plan_df["collection_name"].astype(str).str.len() == 0)
].copy()

benchmark_selection_validation_rows = [
    {
        "check_name": "every_query_has_selected_configuration",
        "status": "passed" if queries_without_any_selected_config_df.empty else "failed",
        "details": f"queries_without_any_selected_config={len(queries_without_any_selected_config_df)}",
    },
    {
        "check_name": "every_query_has_control_configuration",
        "status": "passed" if queries_without_control_df.empty else "failed",
        "details": f"queries_without_control={len(queries_without_control_df)}",
    },
    {
        "check_name": "every_query_has_primary_configuration",
        "status": "passed" if queries_without_primary_df.empty else "warning",
        "details": f"queries_without_primary={len(queries_without_primary_df)}",
    },
    {
        "check_name": "selected_candidate_ids_are_unique",
        "status": "passed" if duplicated_selected_candidate_ids_df.empty else "failed",
        "details": f"duplicated_candidate_ids={len(duplicated_selected_candidate_ids_df)}",
    },
    {
        "check_name": "every_selected_candidate_has_collection_name",
        "status": "passed" if selected_candidates_without_collection_df.empty else "failed",
        "details": f"candidates_without_collection={len(selected_candidates_without_collection_df)}",
    },
    {
        "check_name": "row_count_matches_selected_candidates",
        "status": (
            "passed"
            if len(benchmark_execution_plan_df)
            == len(benchmark_configuration_selection_df)
            else "failed"
        ),
        "details": {
            "selection_rows": len(benchmark_configuration_selection_df),
            "execution_plan_rows": len(benchmark_execution_plan_df),
        },
    },
]

benchmark_selection_validation_df = pd.DataFrame(
    benchmark_selection_validation_rows
)

failed_benchmark_selection_validation_df = benchmark_selection_validation_df[
    benchmark_selection_validation_df["status"] == "failed"
].copy()

warning_benchmark_selection_validation_df = benchmark_selection_validation_df[
    benchmark_selection_validation_df["status"] == "warning"
].copy()


# ---------------------------------------------------------
# 11. Display outputs
# ---------------------------------------------------------

print("Primary, secondary affected, and control configurations selected successfully.")

print("\nBenchmark configuration selection:")
display(benchmark_configuration_selection_df)

print("\nPrimary configurations:")
display(primary_configurations_df)

print("\nSecondary affected configurations:")
display(secondary_affected_configurations_df)

print("\nControl configurations:")
display(control_configurations_df)

print("\nBenchmark execution plan:")
display(benchmark_execution_plan_df)

print("\nBenchmark selection by query:")
display(benchmark_selection_by_query_df)

print("\nBenchmark selection summary:")
display(benchmark_selection_summary_df)

print("\nBenchmark group summary:")
display(benchmark_group_summary_df)

print("\nBenchmark selection validation:")
display(benchmark_selection_validation_df)

if not failed_benchmark_selection_validation_df.empty:
    print("\nWarning: some benchmark selection checks failed.")
    display(failed_benchmark_selection_validation_df)

if not warning_benchmark_selection_validation_df.empty:
    print("\nSelection warnings:")
    display(warning_benchmark_selection_validation_df)

if not queries_without_primary_df.empty:
    print("\nQueries without primary configuration:")
    display(queries_without_primary_df)


# ---------------------------------------------------------
# 12. Save BLOCK V23 outputs for reproducibility
# ---------------------------------------------------------

save_dataframe(
    benchmark_configuration_selection_df,
    "variables/block_v23/benchmark_configuration_selection.csv",
)

save_dataframe(
    primary_configurations_df,
    "variables/block_v23/primary_configurations.csv",
)

save_dataframe(
    secondary_affected_configurations_df,
    "variables/block_v23/secondary_affected_configurations.csv",
)

save_dataframe(
    control_configurations_df,
    "variables/block_v23/control_configurations.csv",
)

save_dataframe(
    benchmark_execution_plan_df,
    "variables/block_v23/benchmark_execution_plan.csv",
)

save_dataframe(
    benchmark_selection_by_query_df,
    "variables/block_v23/benchmark_selection_by_query.csv",
)

save_dataframe(
    document_variable_matrix_with_benchmark_selection_df,
    "variables/block_v23/document_variable_matrix_with_benchmark_selection.csv",
)

save_dataframe(
    document_variable_matrix_df,
    "variables/block_v23/document_variable_matrix.csv",
)

save_dataframe(
    benchmark_selection_summary_df,
    "variables/block_v23/benchmark_selection_summary.csv",
)

save_dataframe(
    benchmark_group_summary_df,
    "variables/block_v23/benchmark_group_summary.csv",
)

save_dataframe(
    benchmark_selection_by_query_summary_df,
    "variables/block_v23/benchmark_selection_by_query_summary.csv",
)

save_dataframe(
    benchmark_selection_validation_df,
    "variables/block_v23/benchmark_selection_validation.csv",
)

save_dataframe(
    failed_benchmark_selection_validation_df,
    "variables/block_v23/failed_benchmark_selection_validation.csv",
)

save_dataframe(
    warning_benchmark_selection_validation_df,
    "variables/block_v23/warning_benchmark_selection_validation.csv",
)

save_dataframe(
    queries_without_primary_df,
    "variables/block_v23/queries_without_primary.csv",
)

save_dataframe(
    queries_without_control_df,
    "variables/block_v23/queries_without_control.csv",
)

save_dataframe(
    queries_without_any_selected_config_df,
    "variables/block_v23/queries_without_any_selected_config.csv",
)

save_dataframe(
    duplicated_selected_candidate_ids_df,
    "variables/block_v23/duplicated_selected_candidate_ids.csv",
)

save_dataframe(
    selected_candidates_without_collection_df,
    "variables/block_v23/selected_candidates_without_collection.csv",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": fiben_data_bundle.scale_label,
        "n_selected_configurations": len(benchmark_execution_plan_df),
        "n_queries": int(benchmark_execution_plan_df["query_name"].nunique()),
        "n_primary_configurations": len(primary_configurations_df),
        "n_secondary_affected_configurations": len(secondary_affected_configurations_df),
        "n_control_configurations": len(control_configurations_df),
        "benchmark_group_counts": (
            benchmark_execution_plan_df["final_benchmark_group"]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "design_pattern_counts": (
            benchmark_execution_plan_df["design_pattern"]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "n_failed_validation_checks": len(failed_benchmark_selection_validation_df),
        "n_warning_validation_checks": len(warning_benchmark_selection_validation_df),
        "output_files": {
            "benchmark_configuration_selection_csv": "variables/block_v23/benchmark_configuration_selection.csv",
            "primary_configurations_csv": "variables/block_v23/primary_configurations.csv",
            "secondary_affected_configurations_csv": "variables/block_v23/secondary_affected_configurations.csv",
            "control_configurations_csv": "variables/block_v23/control_configurations.csv",
            "benchmark_execution_plan_csv": "variables/block_v23/benchmark_execution_plan.csv",
            "benchmark_selection_by_query_csv": "variables/block_v23/benchmark_selection_by_query.csv",
            "document_variable_matrix_with_benchmark_selection_csv": "variables/block_v23/document_variable_matrix_with_benchmark_selection.csv",
            "benchmark_selection_summary_csv": "variables/block_v23/benchmark_selection_summary.csv",
            "benchmark_group_summary_csv": "variables/block_v23/benchmark_group_summary.csv",
            "benchmark_selection_validation_csv": "variables/block_v23/benchmark_selection_validation.csv",
        },
    },
    "variables/block_v23/benchmark_configuration_selection_metadata.json",
)

write_execution_log(
    block_name="BLOCK V23 — SELECT PRIMARY, SECONDARY AFFECTED, AND CONTROL CONFIGURATIONS",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "n_selected_configurations": len(benchmark_execution_plan_df),
        "n_queries": int(benchmark_execution_plan_df["query_name"].nunique()),
        "n_primary_configurations": len(primary_configurations_df),
        "n_secondary_affected_configurations": len(secondary_affected_configurations_df),
        "n_control_configurations": len(control_configurations_df),
        "n_failed_validation_checks": len(failed_benchmark_selection_validation_df),
        "benchmark_execution_plan_csv": "variables/block_v23/benchmark_execution_plan.csv",
        "primary_configurations_csv": "variables/block_v23/primary_configurations.csv",
        "secondary_affected_configurations_csv": "variables/block_v23/secondary_affected_configurations.csv",
        "control_configurations_csv": "variables/block_v23/control_configurations.csv",
        "metadata_json": "variables/block_v23/benchmark_configuration_selection_metadata.json",
    },
)


# ---------------------------------------------------------
# 13. Update README section for BLOCK V23
# ---------------------------------------------------------

block_v23_readme_body = f"""
This block selects the MongoDB configuration candidates that will be used by
the benchmark execution plan.

Main variables created:

    benchmark_configuration_selection_df
    primary_configurations_df
    secondary_affected_configurations_df
    control_configurations_df

Additional variables created:

    benchmark_execution_plan_df
    benchmark_selection_by_query_df
    document_variable_matrix_with_benchmark_selection_df
    benchmark_selection_summary_df
    benchmark_group_summary_df
    benchmark_selection_validation_df

Number of selected configurations:

    {len(benchmark_execution_plan_df)}

Number of queries:

    {int(benchmark_execution_plan_df["query_name"].nunique())}

Number of primary configurations:

    {len(primary_configurations_df)}

Number of secondary affected configurations:

    {len(secondary_affected_configurations_df)}

Number of control configurations:

    {len(control_configurations_df)}

Failed validation checks:

    {len(failed_benchmark_selection_validation_df)}

Warning validation checks:

    {len(warning_benchmark_selection_validation_df)}

Generated reproducibility files:

    variables/block_v23/benchmark_configuration_selection.csv
    variables/block_v23/primary_configurations.csv
    variables/block_v23/secondary_affected_configurations.csv
    variables/block_v23/control_configurations.csv
    variables/block_v23/benchmark_execution_plan.csv
    variables/block_v23/benchmark_selection_by_query.csv
    variables/block_v23/document_variable_matrix_with_benchmark_selection.csv
    variables/block_v23/document_variable_matrix.csv
    variables/block_v23/benchmark_selection_summary.csv
    variables/block_v23/benchmark_group_summary.csv
    variables/block_v23/benchmark_selection_validation.csv
    variables/block_v23/benchmark_configuration_selection_metadata.json

Methodological meaning:

    This block separates the generated MongoDB candidates into benchmark
    groups.

    The control group contains normalized/reference baselines.

    The primary group contains the main candidate configurations expected to
    improve read behavior for the target query.

    The secondary_affected group contains candidates that may improve reads
    but should be evaluated carefully because of update volatility, sharedness,
    or document-design trade-offs.

Important note:

    This block still does not execute MongoDB loads or benchmark queries.

    It only produces the benchmark execution plan that the external server-side
    script will use later.
"""

update_readme_section(
    section_title="BLOCK V23 — Select Primary, Secondary Affected, and Control Configurations",
    section_body=block_v23_readme_body,
)

Primary, secondary affected, and control configurations selected successfully.

Benchmark configuration selection:


,candidate_id,query_name,generic_class,query_family,query_type,g_class,root_entity,collection_name,variant,design_pattern,...,primary_activation_class,primary_activation_name,primary_activation_score,DeltaRratio,query_update_volatility_score,max_observed_sharedness_score,combined_document_design_risk_class,document_potential_class,physical_mapping_reliability,benchmark_group_order
0,q10_create_account_holding_and_buy_transaction...,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,CONTROL,Person,persons_normalized_baseline,normalized_reference_baseline,normalized_reference_baseline,...,G2,Root with contained children,1.0,1.0,0.932000,0.295882,strong_read_candidate_high_update_pressure,high_reduction_potential,ok,0
1,q10_create_account_holding_and_buy_transaction...,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,G2,Person,persons_with_containment,embed_containment_children,embedded_containment,...,G2,Root with contained children,1.0,1.0,0.932000,0.295882,strong_read_candidate_high_update_pressure,high_reduction_potential,ok,1
2,q10_create_account_holding_and_buy_transaction...,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,G3,Person,persons_with_references,association_references,association_references,...,G2,Root with contained children,1.0,1.0,0.932000,0.295882,strong_read_candidate_high_update_pressure,high_reduction_potential,ok,1
3,q10_create_account_holding_and_buy_transaction...,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,G4,Person,persons_deep_nested,deep_nested_path,deep_nested_document,...,G2,Root with contained children,1.0,1.0,0.932000,0.295882,strong_read_candidate_high_update_pressure,high_reduction_potential,ok,1
4,q10_create_account_holding_and_buy_transaction...,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,G5,Person,persons_shared_refs,shared_targets_as_references,shared_target_reference_strategy,...,G2,Root with contained children,1.0,1.0,0.932000,0.295882,strong_read_candidate_high_update_pressure,high_reduction_potential,ok,1
5,q10_create_account_holding_and_buy_transaction...,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,G7,Person,persons_update_aware,update_aware_references,update_aware_reference_design,...,G2,Root with contained children,1.0,1.0,0.932000,0.295882,strong_read_candidate_high_update_pressure,high_reduction_potential,ok,2
6,q10_create_account_holding_and_buy_transaction...,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,G9,Person,persons_tradeoff,tradeoff_benchmark_candidate,benchmark_tradeoff_alternative,...,G2,Root with contained children,1.0,1.0,0.932000,0.295882,strong_read_candidate_high_update_pressure,high_reduction_potential,ok,2
7,q1_company_profile_ibm__control__normalized_re...,Q1_CompanyProfileIBM,QG1,Local lookup,select,CONTROL,Corporation,corporations_normalized_baseline,normalized_reference_baseline,normalized_reference_baseline,...,G0,Root-only document,1.0,0.0,0.000000,0.000000,low_overall_design_pressure,no_relationship_traversal,ok,0
8,q1_company_profile_ibm__g0__root_only,Q1_CompanyProfileIBM,QG1,Local lookup,select,G0,Corporation,corporations,root_only,root_only_collection,...,G0,Root-only document,1.0,0.0,0.000000,0.000000,low_overall_design_pressure,no_relationship_traversal,ok,1
9,q2_company_with_industry_country_and_listed_se...,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,CONTROL,Corporation,corporations_normalized_baseline,normalized_reference_baseline,normalized_reference_baseline,...,G1,Root with direct descriptors,1.0,1.0,0.363000,0.297418,strong_read_candidate_low_maintenance_pressure,high_reduction_potential,needs_review_no_observed_match,0



Primary configurations:


,candidate_id,query_name,generic_class,query_family,query_type,g_class,root_entity,collection_name,variant,design_pattern,...,active_g_classes,primary_activation_class,primary_activation_name,primary_activation_score,DeltaRratio,query_update_volatility_score,max_observed_sharedness_score,combined_document_design_risk_class,document_potential_class,physical_mapping_reliability
1,q10_create_account_holding_and_buy_transaction...,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,G2,Person,persons_with_containment,embed_containment_children,embedded_containment,...,"[G2, G3, G4, G5, G7, G9]",G2,Root with contained children,1.0,1.0,0.932000,0.295882,strong_read_candidate_high_update_pressure,high_reduction_potential,ok
2,q10_create_account_holding_and_buy_transaction...,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,G3,Person,persons_with_references,association_references,association_references,...,"[G2, G3, G4, G5, G7, G9]",G2,Root with contained children,1.0,1.0,0.932000,0.295882,strong_read_candidate_high_update_pressure,high_reduction_potential,ok
3,q10_create_account_holding_and_buy_transaction...,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,G4,Person,persons_deep_nested,deep_nested_path,deep_nested_document,...,"[G2, G3, G4, G5, G7, G9]",G2,Root with contained children,1.0,1.0,0.932000,0.295882,strong_read_candidate_high_update_pressure,high_reduction_potential,ok
4,q10_create_account_holding_and_buy_transaction...,Q10_CreateAccountHoldingAndBuyTransaction,QG10,Update / insertion with relationship creation,insert,G5,Person,persons_shared_refs,shared_targets_as_references,shared_target_reference_strategy,...,"[G2, G3, G4, G5, G7, G9]",G2,Root with contained children,1.0,1.0,0.932000,0.295882,strong_read_candidate_high_update_pressure,high_reduction_potential,ok
8,q1_company_profile_ibm__g0__root_only,Q1_CompanyProfileIBM,QG1,Local lookup,select,G0,Corporation,corporations,root_only,root_only_collection,...,[G0],G0,Root-only document,1.0,0.0,0.000000,0.000000,low_overall_design_pressure,no_relationship_traversal,ok
10,q2_company_with_industry_country_and_listed_se...,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,G1,Corporation,corporations_with_descriptors,embed_descriptors,embedded_descriptors,...,"[G1, G3, G5, G7, G9]",G1,Root with direct descriptors,1.0,1.0,0.363000,0.297418,strong_read_candidate_low_maintenance_pressure,high_reduction_potential,needs_review_no_observed_match
11,q2_company_with_industry_country_and_listed_se...,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,G3,Corporation,corporations_with_references,association_references,association_references,...,"[G1, G3, G5, G7, G9]",G1,Root with direct descriptors,1.0,1.0,0.363000,0.297418,strong_read_candidate_low_maintenance_pressure,high_reduction_potential,needs_review_no_observed_match
12,q2_company_with_industry_country_and_listed_se...,Q2_CompanyWithIndustryCountryAndListedSecurities,QG2,Shallow rooted retrieval,select,G5,Corporation,corporations_shared_refs,shared_targets_as_references,shared_target_reference_strategy,...,"[G1, G3, G5, G7, G9]",G1,Root with direct descriptors,1.0,1.0,0.363000,0.297418,strong_read_candidate_low_maintenance_pressure,high_reduction_potential,needs_review_no_observed_match
16,q3_securities_held_in_each_financial_service_a...,Q3_SecuritiesHeldInEachFinancialServiceAccount,QG3,Associative retrieval,select,G2,FinancialServiceAccount,financial_service_accounts_with_containment,embed_containment_children,embedded_containment,...,"[G2, G3, G4, G5, G7, G9]",G2,Root with contained children,1.0,1.0,0.799000,0.295882,strong_read_candidate_high_update_pressure,high_reduction_potential,needs_review_no_observed_match
17,q3_securities_held_in_each_financial_service_a...,Q3_SecuritiesHeldInEachFinancialServiceAccoun


Secondary affected configurations:


...


Benchmark group summary:


,final_benchmark_group,n_configurations,n_queries,candidate_ids,design_patterns
0,control,10,10,[q10_create_account_holding_and_buy_transactio...,[normalized_reference_baseline]
1,primary,33,10,[q10_create_account_holding_and_buy_transactio...,"[aggregation_materialized_collection, associat..."
2,secondary_affected,17,9,[q10_create_account_holding_and_buy_transactio...,"[benchmark_tradeoff_alternative, update_aware_..."



Benchmark selection validation:


,check_name,status,details
0,every_query_has_selected_configuration,passed,queries_without_any_selected_config=0
1,every_query_has_control_configuration,passed,queries_without_control=0
2,every_query_has_primary_configuration,passed,queries_without_primary=0
3,selected_candidate_ids_are_unique,passed,duplicated_candidate_ids=0
4,every_selected_candidate_has_collection_name,passed,candidates_without_collection=0
5,row_count_matches_selected_candidates,passed,"{'selection_rows': 60, 'execution_plan_rows': 60}"


Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v23/benchmark_configuration_selection.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v23/primary_configurations.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v23/secondary_affected_configurations.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v23/control_configurations.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v23/benchmark_execution_plan.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v23/benchmark_selection_by_query.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/va

In [101]:
benchmark_selection_validation_df

,check_name,status,details
0,every_query_has_selected_configuration,passed,queries_without_any_selected_config=0
1,every_query_has_control_configuration,passed,queries_without_control=0
2,every_query_has_primary_configuration,passed,queries_without_primary=0
3,selected_candidate_ids_are_unique,passed,duplicated_candidate_ids=0
4,every_selected_candidate_has_collection_name,passed,candidates_without_collection=0
5,row_count_matches_selected_candidates,passed,"{'selection_rows': 60, 'execution_plan_rows': 60}"


In [102]:
# =========================================================
# BLOCK V24 — EXPORT BENCHMARK CONFIGURATION ARTIFACTS
# =========================================================
#
# Purpose:
# This block exports benchmark configuration artifacts generated
# by the methodology.
#
# These artifacts are intended to be consumed later by an external
# server-side Python script that will:
#
# - create MongoDB collections;
# - load FIBEN data according to each candidate configuration;
# - execute benchmark queries;
# - collect execution time and result statistics.
#
# Important:
# This block does not connect to MongoDB.
# This block does not load data.
# This block does not execute benchmark queries.
#
# Main outputs:
# - benchmark_artifacts_dir
# - benchmark_manifest
# - exported_artifacts_df
# - benchmark_export_validation_df
#
# Main exported files:
# - benchmark_execution_plan.csv
# - benchmark_execution_plan.json
# - mongodb_candidate_specs.json
# - query_to_candidate_mapping.json
# - benchmark_groups_summary.csv
# - benchmark_manifest.json
# =========================================================

from pathlib import Path
import json
import pandas as pd


# ---------------------------------------------------------
# 1. Basic checks
# ---------------------------------------------------------

required_names = [
    "benchmark_execution_plan_df",
    "benchmark_configuration_selection_df",
    "benchmark_selection_by_query_df",
    "primary_configurations_df",
    "secondary_affected_configurations_df",
    "control_configurations_df",
    "mongodb_configuration_candidates_df",
    "document_variable_matrix_df",
    "fiben_scenario",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. "
            "Run first BLOCK V23 — SELECT PRIMARY, SECONDARY AFFECTED, AND CONTROL CONFIGURATIONS."
        )


# ---------------------------------------------------------
# 2. Define artifact directories
# ---------------------------------------------------------

if "FIBEN_PROJECT_OUTPUT_DIR" in globals():
    project_output_dir = Path(FIBEN_PROJECT_OUTPUT_DIR)
else:
    project_output_dir = Path(".")

block_v24_variables_dir = project_output_dir / "variables" / "block_v24"

benchmark_artifacts_dir = (
    project_output_dir
    / "benchmark_artifacts"
    / "fiben_mongodb_configurations"
)

block_v24_variables_dir.mkdir(parents=True, exist_ok=True)
benchmark_artifacts_dir.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------
# 3. Helper functions
# ---------------------------------------------------------

def make_json_safe(value):
    """
    Convert Python, pandas, and numpy-like values into JSON-safe values.
    """
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    if isinstance(value, dict):
        return {
            str(k): make_json_safe(v)
            for k, v in value.items()
        }

    if isinstance(value, list):
        return [make_json_safe(v) for v in value]

    if isinstance(value, tuple):
        return [make_json_safe(v) for v in value]

    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            pass

    if isinstance(value, Path):
        return str(value)

    return value


def dataframe_to_json_records(df: pd.DataFrame) -> list:
    """
    Convert a DataFrame into JSON-safe records.
    """
    records = df.to_dict(orient="records")
    return [make_json_safe(record) for record in records]


def dataframe_for_csv_export(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert list/dict columns to JSON strings before CSV export.

    This makes the CSV easier to reload in a server-side script.
    """
    export_df = df.copy()

    for col in export_df.columns:
        export_df[col] = export_df[col].apply(
            lambda value: (
                json.dumps(make_json_safe(value), ensure_ascii=False)
                if isinstance(value, (list, tuple, dict))
                else value
            )
        )

    return export_df


def write_json_artifact(obj, path: Path):
    """
    Write a JSON artifact with indentation.
    """
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            make_json_safe(obj),
            f,
            ensure_ascii=False,
            indent=2,
        )


def write_csv_artifact(df: pd.DataFrame, path: Path):
    """
    Write a CSV artifact after converting complex columns to JSON strings.
    """
    path.parent.mkdir(parents=True, exist_ok=True)

    export_df = dataframe_for_csv_export(df)
    export_df.to_csv(path, index=False)


def safe_list(value):
    """
    Convert a value into a clean list.
    """
    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    if value is None:
        return []

    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    return [value]


def parse_candidate_spec(row):
    """
    Return the candidate specification as a dictionary.

    Priority:
    1. candidate_spec if it is already a dict;
    2. candidate_spec_json if available;
    3. build a minimal candidate specification from row fields.
    """
    candidate_spec = row.get("candidate_spec")

    if isinstance(candidate_spec, dict):
        return make_json_safe(candidate_spec)

    candidate_spec_json = row.get("candidate_spec_json")

    if isinstance(candidate_spec_json, str) and candidate_spec_json.strip():
        try:
            return make_json_safe(json.loads(candidate_spec_json))
        except Exception:
            pass

    return {
        "candidate_id": row.get("candidate_id"),
        "query_name": row.get("query_name"),
        "g_class": row.get("g_class"),
        "root_entity": row.get("root_entity"),
        "collection_name": row.get("collection_name"),
        "design_pattern": row.get("design_pattern"),
        "embedded_entities": safe_list(row.get("embedded_entities")),
        "referenced_entities": safe_list(row.get("referenced_entities")),
        "materialized_entities": safe_list(row.get("materialized_entities")),
        "rationale": row.get("rationale"),
    }


def relative_artifact_path(path: Path) -> str:
    """
    Return path relative to the project output directory when possible.
    """
    try:
        return str(path.relative_to(project_output_dir))
    except Exception:
        return str(path)


# ---------------------------------------------------------
# 4. Build clean benchmark execution plan artifacts
# ---------------------------------------------------------

benchmark_execution_plan_export_df = benchmark_execution_plan_df.copy()

# Keep a stable set of useful columns first.
preferred_plan_columns = [
    "benchmark_plan_order",
    "query_name",
    "candidate_id",
    "final_benchmark_group",
    "selection_role",
    "benchmark_priority",
    "g_class",
    "design_pattern",
    "variant",
    "root_entity",
    "collection_name",
    "embedded_entities",
    "referenced_entities",
    "materialized_entities",
    "n_embedded_entities",
    "n_referenced_entities",
    "n_materialized_entities",
    "DeltaRratio",
    "query_update_volatility_score",
    "max_observed_sharedness_score",
    "candidate_selection_score",
    "should_load_configuration",
    "should_run_read_benchmark",
    "should_run_update_benchmark",
    "benchmark_notes",
    "candidate_spec_json",
]

existing_preferred_plan_columns = [
    col for col in preferred_plan_columns
    if col in benchmark_execution_plan_export_df.columns
]

remaining_plan_columns = [
    col for col in benchmark_execution_plan_export_df.columns
    if col not in existing_preferred_plan_columns
]

benchmark_execution_plan_export_df = benchmark_execution_plan_export_df[
    existing_preferred_plan_columns + remaining_plan_columns
].copy()


# ---------------------------------------------------------
# 5. Build MongoDB candidate specification JSON
# ---------------------------------------------------------

mongodb_candidate_specs = []

for _, row in benchmark_execution_plan_df.iterrows():
    row_dict = row.to_dict()

    candidate_spec = parse_candidate_spec(row_dict)

    candidate_spec["benchmark"] = {
        "benchmark_plan_order": row_dict.get("benchmark_plan_order"),
        "final_benchmark_group": row_dict.get("final_benchmark_group"),
        "selection_role": row_dict.get("selection_role"),
        "candidate_selection_score": row_dict.get("candidate_selection_score"),
        "should_load_configuration": row_dict.get("should_load_configuration"),
        "should_run_read_benchmark": row_dict.get("should_run_read_benchmark"),
        "should_run_update_benchmark": row_dict.get("should_run_update_benchmark"),
        "benchmark_notes": row_dict.get("benchmark_notes"),
    }

    candidate_spec["metrics_context"] = {
        "DeltaRratio": row_dict.get("DeltaRratio"),
        "query_update_volatility_score": row_dict.get("query_update_volatility_score"),
        "max_observed_sharedness_score": row_dict.get("max_observed_sharedness_score"),
        "physical_mapping_reliability": row_dict.get("physical_mapping_reliability"),
        "combined_document_design_risk_class": row_dict.get("combined_document_design_risk_class"),
    }

    mongodb_candidate_specs.append(candidate_spec)

mongodb_candidate_specs_by_candidate_id = {
    spec["candidate_id"]: spec
    for spec in mongodb_candidate_specs
}


# ---------------------------------------------------------
# 6. Build query-to-candidate mapping JSON
# ---------------------------------------------------------

query_to_candidate_mapping = {}

for query_name, group in benchmark_execution_plan_df.groupby("query_name"):
    query_to_candidate_mapping[query_name] = {
        "all_candidates": group["candidate_id"].tolist(),
        "control_candidates": group[
            group["final_benchmark_group"] == "control"
        ]["candidate_id"].tolist(),
        "primary_candidates": group[
            group["final_benchmark_group"] == "primary"
        ]["candidate_id"].tolist(),
        "secondary_affected_candidates": group[
            group["final_benchmark_group"] == "secondary_affected"
        ]["candidate_id"].tolist(),
        "read_benchmark_candidates": group[
            group["should_run_read_benchmark"] == True
        ]["candidate_id"].tolist(),
        "update_benchmark_candidates": group[
            group["should_run_update_benchmark"] == True
        ]["candidate_id"].tolist(),
        "design_patterns": sorted(group["design_pattern"].dropna().unique().tolist()),
        "g_classes": sorted(group["g_class"].dropna().unique().tolist()),
    }


# ---------------------------------------------------------
# 7. Build benchmark group mapping JSON
# ---------------------------------------------------------

benchmark_group_mapping = {}

for group_name, group in benchmark_execution_plan_df.groupby("final_benchmark_group"):
    benchmark_group_mapping[group_name] = {
        "candidate_ids": group["candidate_id"].tolist(),
        "query_names": sorted(group["query_name"].dropna().unique().tolist()),
        "design_patterns": sorted(group["design_pattern"].dropna().unique().tolist()),
        "n_candidates": int(len(group)),
        "n_queries": int(group["query_name"].nunique()),
    }


# ---------------------------------------------------------
# 8. Build workload query metadata artifact
# ---------------------------------------------------------

if "workload_conceptual_df" in globals():
    workload_query_metadata_df = workload_conceptual_df.copy()
else:
    workload_query_metadata_df = document_variable_matrix_df[
        [
            col for col in [
                "query_name",
                "generic_class",
                "query_family",
                "query_type",
                "abstract_query",
                "entities_touched",
            ]
            if col in document_variable_matrix_df.columns
        ]
    ].copy()

workload_query_metadata = dataframe_to_json_records(workload_query_metadata_df)


# ---------------------------------------------------------
# 9. Build benchmark manifest
# ---------------------------------------------------------

benchmark_manifest = {
    "artifact_type": "fiben_mongodb_benchmark_configuration_bundle",
    "scenario_name": fiben_scenario.name,
    "dataset_name": "FIBEN",
    "selected_scale": (
        fiben_data_bundle.scale_label
        if "fiben_data_bundle" in globals()
        else None
    ),
    "methodology_stage": "configuration_selection_completed",
    "description": (
        "Artifacts exported from the FIBEN methodology notebook. "
        "These files define MongoDB candidate configurations and benchmark groups. "
        "They are intended for a separate server-side load and benchmark script."
    ),
    "n_queries": int(benchmark_execution_plan_df["query_name"].nunique()),
    "n_selected_configurations": int(len(benchmark_execution_plan_df)),
    "n_control_configurations": int(
        (benchmark_execution_plan_df["final_benchmark_group"] == "control").sum()
    ),
    "n_primary_configurations": int(
        (benchmark_execution_plan_df["final_benchmark_group"] == "primary").sum()
    ),
    "n_secondary_affected_configurations": int(
        (benchmark_execution_plan_df["final_benchmark_group"] == "secondary_affected").sum()
    ),
    "benchmark_groups": benchmark_group_mapping,
    "expected_external_script_inputs": {
        "benchmark_execution_plan_csv": "benchmark_execution_plan.csv",
        "benchmark_execution_plan_json": "benchmark_execution_plan.json",
        "mongodb_candidate_specs_json": "mongodb_candidate_specs.json",
        "query_to_candidate_mapping_json": "query_to_candidate_mapping.json",
        "benchmark_groups_summary_csv": "benchmark_groups_summary.csv",
    },
    "important_note": (
        "This bundle does not contain the FIBEN data files. "
        "The external benchmark script must receive the dataset path separately."
    ),
}


# ---------------------------------------------------------
# 10. Define artifact paths
# ---------------------------------------------------------

artifact_paths = {
    "benchmark_execution_plan_csv": benchmark_artifacts_dir / "benchmark_execution_plan.csv",
    "benchmark_execution_plan_json": benchmark_artifacts_dir / "benchmark_execution_plan.json",
    "mongodb_candidate_specs_json": benchmark_artifacts_dir / "mongodb_candidate_specs.json",
    "mongodb_candidate_specs_by_candidate_id_json": benchmark_artifacts_dir / "mongodb_candidate_specs_by_candidate_id.json",
    "query_to_candidate_mapping_json": benchmark_artifacts_dir / "query_to_candidate_mapping.json",
    "benchmark_group_mapping_json": benchmark_artifacts_dir / "benchmark_group_mapping.json",
    "benchmark_groups_summary_csv": benchmark_artifacts_dir / "benchmark_groups_summary.csv",
    "benchmark_selection_by_query_csv": benchmark_artifacts_dir / "benchmark_selection_by_query.csv",
    "primary_configurations_csv": benchmark_artifacts_dir / "primary_configurations.csv",
    "secondary_affected_configurations_csv": benchmark_artifacts_dir / "secondary_affected_configurations.csv",
    "control_configurations_csv": benchmark_artifacts_dir / "control_configurations.csv",
    "workload_query_metadata_json": benchmark_artifacts_dir / "workload_query_metadata.json",
    "document_variable_matrix_csv": benchmark_artifacts_dir / "document_variable_matrix.csv",
    "benchmark_manifest_json": benchmark_artifacts_dir / "benchmark_manifest.json",
}


# ---------------------------------------------------------
# 11. Export benchmark artifacts
# ---------------------------------------------------------

write_csv_artifact(
    benchmark_execution_plan_export_df,
    artifact_paths["benchmark_execution_plan_csv"],
)

write_json_artifact(
    dataframe_to_json_records(benchmark_execution_plan_export_df),
    artifact_paths["benchmark_execution_plan_json"],
)

write_json_artifact(
    mongodb_candidate_specs,
    artifact_paths["mongodb_candidate_specs_json"],
)

write_json_artifact(
    mongodb_candidate_specs_by_candidate_id,
    artifact_paths["mongodb_candidate_specs_by_candidate_id_json"],
)

write_json_artifact(
    query_to_candidate_mapping,
    artifact_paths["query_to_candidate_mapping_json"],
)

write_json_artifact(
    benchmark_group_mapping,
    artifact_paths["benchmark_group_mapping_json"],
)

write_csv_artifact(
    benchmark_group_summary_df,
    artifact_paths["benchmark_groups_summary_csv"],
)

write_csv_artifact(
    benchmark_selection_by_query_df,
    artifact_paths["benchmark_selection_by_query_csv"],
)

write_csv_artifact(
    primary_configurations_df,
    artifact_paths["primary_configurations_csv"],
)

write_csv_artifact(
    secondary_affected_configurations_df,
    artifact_paths["secondary_affected_configurations_csv"],
)

write_csv_artifact(
    control_configurations_df,
    artifact_paths["control_configurations_csv"],
)

write_json_artifact(
    workload_query_metadata,
    artifact_paths["workload_query_metadata_json"],
)

write_csv_artifact(
    document_variable_matrix_df,
    artifact_paths["document_variable_matrix_csv"],
)

write_json_artifact(
    benchmark_manifest,
    artifact_paths["benchmark_manifest_json"],
)


# ---------------------------------------------------------
# 12. Create exported artifact inventory
# ---------------------------------------------------------

exported_artifact_rows = []

for artifact_name, path in artifact_paths.items():
    exists = path.exists()
    file_size_bytes = path.stat().st_size if exists else None

    exported_artifact_rows.append({
        "artifact_name": artifact_name,
        "path": str(path),
        "relative_path": relative_artifact_path(path),
        "exists": exists,
        "file_size_bytes": file_size_bytes,
    })

exported_artifacts_df = pd.DataFrame(exported_artifact_rows)


# ---------------------------------------------------------
# 13. Create external execution README
# ---------------------------------------------------------

external_readme_path = benchmark_artifacts_dir / "README_benchmark_artifacts.md"

external_readme_text = f"""# FIBEN MongoDB Benchmark Configuration Artifacts

This folder contains the MongoDB configuration artifacts generated by the methodology notebook.

## Dataset

Dataset: FIBEN  
Scenario: {fiben_scenario.name}  
Scale: {benchmark_manifest["selected_scale"]}

## Main files

- `benchmark_execution_plan.csv`: ordered list of configurations to load and benchmark.
- `benchmark_execution_plan.json`: JSON version of the execution plan.
- `mongodb_candidate_specs.json`: full MongoDB candidate configuration specifications.
- `mongodb_candidate_specs_by_candidate_id.json`: candidate specifications indexed by candidate id.
- `query_to_candidate_mapping.json`: mapping from each workload query to its selected candidates.
- `benchmark_group_mapping.json`: mapping from benchmark groups to candidate ids.
- `benchmark_groups_summary.csv`: summary by benchmark group.
- `benchmark_selection_by_query.csv`: compact query-to-selection summary.
- `document_variable_matrix.csv`: final analytical matrix with activation and benchmark-selection variables.
- `benchmark_manifest.json`: metadata manifest for this artifact bundle.

## Benchmark groups

- `control`: normalized/reference baseline.
- `primary`: main generated configurations expected to improve read behavior.
- `secondary_affected`: configurations that may improve reads but require careful evaluation because of update volatility, sharedness, or design trade-offs.

## Important note

These files do not contain the FIBEN dataset itself.

The external server-side script must receive:

1. the path to the FIBEN data files;
2. the path to this artifact folder;
3. the MongoDB connection string;
4. the benchmark output directory.

## Current counts

Number of queries: {benchmark_manifest["n_queries"]}

Number of selected configurations: {benchmark_manifest["n_selected_configurations"]}

Control configurations: {benchmark_manifest["n_control_configurations"]}

Primary configurations: {benchmark_manifest["n_primary_configurations"]}

Secondary affected configurations: {benchmark_manifest["n_secondary_affected_configurations"]}
"""

with open(external_readme_path, "w", encoding="utf-8") as f:
    f.write(external_readme_text)

exported_artifacts_df = pd.concat(
    [
        exported_artifacts_df,
        pd.DataFrame([
            {
                "artifact_name": "external_readme_md",
                "path": str(external_readme_path),
                "relative_path": relative_artifact_path(external_readme_path),
                "exists": external_readme_path.exists(),
                "file_size_bytes": (
                    external_readme_path.stat().st_size
                    if external_readme_path.exists()
                    else None
                ),
            }
        ]),
    ],
    ignore_index=True,
)


# ---------------------------------------------------------
# 14. Validation checks
# ---------------------------------------------------------

required_artifact_names = [
    "benchmark_execution_plan_csv",
    "benchmark_execution_plan_json",
    "mongodb_candidate_specs_json",
    "query_to_candidate_mapping_json",
    "benchmark_manifest_json",
]

missing_required_artifacts = exported_artifacts_df[
    exported_artifacts_df["artifact_name"].isin(required_artifact_names)
    & (exported_artifacts_df["exists"] == False)
].copy()

empty_required_artifacts = exported_artifacts_df[
    exported_artifacts_df["artifact_name"].isin(required_artifact_names)
    & (
        exported_artifacts_df["file_size_bytes"].isna()
        | (exported_artifacts_df["file_size_bytes"] == 0)
    )
].copy()

candidate_ids_in_plan = set(
    benchmark_execution_plan_df["candidate_id"].dropna().tolist()
)

candidate_ids_in_specs = set(
    spec["candidate_id"]
    for spec in mongodb_candidate_specs
    if spec.get("candidate_id") is not None
)

missing_specs_for_plan_candidates = sorted(
    candidate_ids_in_plan - candidate_ids_in_specs
)

extra_specs_not_in_plan = sorted(
    candidate_ids_in_specs - candidate_ids_in_plan
)

queries_in_plan = set(
    benchmark_execution_plan_df["query_name"].dropna().tolist()
)

queries_in_mapping = set(query_to_candidate_mapping.keys())

missing_query_mapping = sorted(
    queries_in_plan - queries_in_mapping
)

benchmark_export_validation_rows = [
    {
        "check_name": "required_artifacts_exist",
        "status": "passed" if missing_required_artifacts.empty else "failed",
        "details": f"missing_required_artifacts={len(missing_required_artifacts)}",
    },
    {
        "check_name": "required_artifacts_are_not_empty",
        "status": "passed" if empty_required_artifacts.empty else "failed",
        "details": f"empty_required_artifacts={len(empty_required_artifacts)}",
    },
    {
        "check_name": "candidate_specs_cover_execution_plan",
        "status": "passed" if not missing_specs_for_plan_candidates else "failed",
        "details": missing_specs_for_plan_candidates,
    },
    {
        "check_name": "no_extra_candidate_specs",
        "status": "passed" if not extra_specs_not_in_plan else "warning",
        "details": extra_specs_not_in_plan,
    },
    {
        "check_name": "query_mapping_covers_execution_plan",
        "status": "passed" if not missing_query_mapping else "failed",
        "details": missing_query_mapping,
    },
    {
        "check_name": "manifest_created",
        "status": "passed" if artifact_paths["benchmark_manifest_json"].exists() else "failed",
        "details": str(artifact_paths["benchmark_manifest_json"]),
    },
]

benchmark_export_validation_df = pd.DataFrame(
    benchmark_export_validation_rows
)

failed_benchmark_export_validation_df = benchmark_export_validation_df[
    benchmark_export_validation_df["status"] == "failed"
].copy()

warning_benchmark_export_validation_df = benchmark_export_validation_df[
    benchmark_export_validation_df["status"] == "warning"
].copy()


# ---------------------------------------------------------
# 15. Save BLOCK V24 variables for notebook reproducibility
# ---------------------------------------------------------

save_dataframe(
    exported_artifacts_df,
    "variables/block_v24/exported_artifacts.csv",
)

save_dataframe(
    benchmark_execution_plan_export_df,
    "variables/block_v24/benchmark_execution_plan_export.csv",
)

save_dataframe(
    benchmark_export_validation_df,
    "variables/block_v24/benchmark_export_validation.csv",
)

save_dataframe(
    failed_benchmark_export_validation_df,
    "variables/block_v24/failed_benchmark_export_validation.csv",
)

save_dataframe(
    warning_benchmark_export_validation_df,
    "variables/block_v24/warning_benchmark_export_validation.csv",
)

save_json(
    benchmark_manifest,
    "variables/block_v24/benchmark_manifest.json",
)

save_json(
    {
        "scenario_name": fiben_scenario.name,
        "dataset_name": "FIBEN",
        "selected_scale": benchmark_manifest["selected_scale"],
        "benchmark_artifacts_dir": str(benchmark_artifacts_dir),
        "n_exported_artifacts": len(exported_artifacts_df),
        "n_selected_configurations": len(benchmark_execution_plan_df),
        "n_candidate_specs": len(mongodb_candidate_specs),
        "n_queries_in_mapping": len(query_to_candidate_mapping),
        "n_failed_validation_checks": len(failed_benchmark_export_validation_df),
        "n_warning_validation_checks": len(warning_benchmark_export_validation_df),
        "exported_artifacts": exported_artifacts_df.to_dict(orient="records"),
    },
    "variables/block_v24/benchmark_export_metadata.json",
)

write_execution_log(
    block_name="BLOCK V24 — EXPORT BENCHMARK CONFIGURATION ARTIFACTS",
    notes={
        "status": "completed",
        "scenario_name": fiben_scenario.name,
        "benchmark_artifacts_dir": str(benchmark_artifacts_dir),
        "n_exported_artifacts": len(exported_artifacts_df),
        "n_selected_configurations": len(benchmark_execution_plan_df),
        "n_candidate_specs": len(mongodb_candidate_specs),
        "n_failed_validation_checks": len(failed_benchmark_export_validation_df),
        "n_warning_validation_checks": len(warning_benchmark_export_validation_df),
        "exported_artifacts_csv": "variables/block_v24/exported_artifacts.csv",
        "benchmark_export_validation_csv": "variables/block_v24/benchmark_export_validation.csv",
        "benchmark_manifest_json": "variables/block_v24/benchmark_manifest.json",
        "metadata_json": "variables/block_v24/benchmark_export_metadata.json",
    },
)


# ---------------------------------------------------------
# 16. Update README section for BLOCK V24
# ---------------------------------------------------------

block_v24_readme_body = f"""
This block exports benchmark configuration artifacts for the external
server-side load and benchmark script.

Main variables created:

    benchmark_artifacts_dir
    benchmark_manifest
    exported_artifacts_df
    benchmark_export_validation_df

Artifact directory:

    {benchmark_artifacts_dir}

Number of exported artifacts:

    {len(exported_artifacts_df)}

Number of selected configurations:

    {len(benchmark_execution_plan_df)}

Number of candidate specs:

    {len(mongodb_candidate_specs)}

Number of queries in query-to-candidate mapping:

    {len(query_to_candidate_mapping)}

Failed validation checks:

    {len(failed_benchmark_export_validation_df)}

Warning validation checks:

    {len(warning_benchmark_export_validation_df)}

Main exported files:

    benchmark_execution_plan.csv
    benchmark_execution_plan.json
    mongodb_candidate_specs.json
    mongodb_candidate_specs_by_candidate_id.json
    query_to_candidate_mapping.json
    benchmark_group_mapping.json
    benchmark_groups_summary.csv
    benchmark_selection_by_query.csv
    primary_configurations.csv
    secondary_affected_configurations.csv
    control_configurations.csv
    workload_query_metadata.json
    document_variable_matrix.csv
    benchmark_manifest.json
    README_benchmark_artifacts.md

Generated reproducibility files:

    variables/block_v24/exported_artifacts.csv
    variables/block_v24/benchmark_execution_plan_export.csv
    variables/block_v24/benchmark_export_validation.csv
    variables/block_v24/failed_benchmark_export_validation.csv
    variables/block_v24/warning_benchmark_export_validation.csv
    variables/block_v24/benchmark_manifest.json
    variables/block_v24/benchmark_export_metadata.json

Methodological meaning:

    This block closes the notebook-side methodology pipeline.

    It exports the selected MongoDB configuration candidates and benchmark
    execution plan to files that can be consumed by a separate execution script.

Important note:

    The exported artifacts do not include the FIBEN dataset files.

    The benchmark runner must receive the dataset path and MongoDB connection
    information separately.
"""

update_readme_section(
    section_title="BLOCK V24 — Export Benchmark Configuration Artifacts",
    section_body=block_v24_readme_body,
)


# ---------------------------------------------------------
# 17. Display outputs
# ---------------------------------------------------------

print("Benchmark configuration artifacts exported successfully.")

print("\nBenchmark artifact directory:")
print(benchmark_artifacts_dir)

print("\nExported artifacts:")
display(exported_artifacts_df)

print("\nBenchmark export validation:")
display(benchmark_export_validation_df)

if not failed_benchmark_export_validation_df.empty:
    print("\nWarning: some export validation checks failed.")
    display(failed_benchmark_export_validation_df)

if not warning_benchmark_export_validation_df.empty:
    print("\nExport validation warnings:")
    display(warning_benchmark_export_validation_df)

print("\nMain files ready for external benchmark script:")
for key in [
    "benchmark_execution_plan_csv",
    "mongodb_candidate_specs_json",
    "query_to_candidate_mapping_json",
    "benchmark_manifest_json",
]:
    print(f"- {artifact_paths[key]}")

Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v24/exported_artifacts.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v24/benchmark_execution_plan_export.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v24/benchmark_export_validation.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v24/failed_benchmark_export_validation.csv
Saved DataFrame: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v24/warning_benchmark_export_validation.csv
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/block_v24/benchmark_manifest.json
Saved JSON: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/variables/

/tmp/ipykernel_581042/1433105864.py:99: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  if pd.isna(value):
/tmp/ipykernel_581042/1433105864.py:99: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  if pd.isna(value):
/tmp/ipykernel_581042/1433105864.py:99: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  if pd.isna(value):


,artifact_name,path,relative_path,exists,file_size_bytes
0,benchmark_execution_plan_csv,/home/jovyan/privado/framework evaluation appr...,benchmark_artifacts/fiben_mongodb_configuratio...,True,125892
1,benchmark_execution_plan_json,/home/jovyan/privado/framework evaluation appr...,benchmark_artifacts/fiben_mongodb_configuratio...,True,200981
2,mongodb_candidate_specs_json,/home/jovyan/privado/framework evaluation appr...,benchmark_artifacts/fiben_mongodb_configuratio...,True,83412
3,mongodb_candidate_specs_by_candidate_id_json,/home/jovyan/privado/framework evaluation appr...,benchmark_artifacts/fiben_mongodb_configuratio...,True,88440
4,query_to_candidate_mapping_json,/home/jovyan/privado/framework evaluation appr...,benchmark_artifacts/fiben_mongodb_configuratio...,True,26626
5,benchmark_group_mapping_json,/home/jovyan/privado/framework evaluation appr...,benchmark_artifacts/fiben_mongodb_configuratio...,True,7710
6,benchmark_groups_summary_csv,/home/jovyan/privado/framework evaluation appr...,benchmark_artifacts/fiben_mongodb_configuratio...,True,5614
7,benchmark_selection_by_query_csv,/home/jovyan/privado/framework evaluation appr...,benchmark_artifacts/fiben_mongodb_configuratio...,True,8527
8,primary_configurations_csv,/home/jovyan/privado/framework evaluation appr...,benchmark_artifacts/fiben_mongodb_configuratio...,True,65751
9,secondary_affected_configurations_csv,/home/jovyan/privado/framework evaluation appr...,benchmark_artifacts/fiben_mongodb_configuratio...,True,36768



Benchmark export validation:


,check_name,status,details
0,required_artifacts_exist,passed,missing_required_artifacts=0
1,required_artifacts_are_not_empty,passed,empty_required_artifacts=0
2,candidate_specs_cover_execution_plan,passed,[]
3,no_extra_candidate_specs,passed,[]
4,query_mapping_covers_execution_plan,passed,[]
5,manifest_created,passed,/home/jovyan/privado/framework evaluation appr...



Main files ready for external benchmark script:
- /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/benchmark_artifacts/fiben_mongodb_configurations/benchmark_execution_plan.csv
- /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/benchmark_artifacts/fiben_mongodb_configurations/mongodb_candidate_specs.json
- /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/benchmark_artifacts/fiben_mongodb_configurations/query_to_candidate_mapping.json
- /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/benchmark_artifacts/fiben_mongodb_configurations/benchmark_manifest.json


In [103]:
benchmark_export_validation_df

,check_name,status,details
0,required_artifacts_exist,passed,missing_required_artifacts=0
1,required_artifacts_are_not_empty,passed,empty_required_artifacts=0
2,candidate_specs_cover_execution_plan,passed,[]
3,no_extra_candidate_specs,passed,[]
4,query_mapping_covers_execution_plan,passed,[]
5,manifest_created,passed,/home/jovyan/privado/framework evaluation appr...


In [104]:
print(benchmark_artifacts_dir)

/home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/benchmark_artifacts/fiben_mongodb_configurations


In [ ]:
# =========================================================
# BLOCK V22C — COMPARISON MATRIX BY FAMILY
# prepares comparative analysis of results by:
# - structural family
# - scale factor
# - competing configurations
# =========================================================

import pandas as pd
import itertools

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "mongo_experiment_catalog_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first o bloco V22."
        )

# ---------------------------------------------------------
# 2) Helpers
# ---------------------------------------------------------
def as_list(x):
    return x if isinstance(x, list) else []

def normalize_list(values):
    seen = []
    for v in values:
        if isinstance(v, list):
            for x in v:
                if x not in seen:
                    seen.append(x)
        else:
            if v not in seen:
                seen.append(v)
    return seen

def list_to_str(values):
    if isinstance(values, list):
        return ", ".join(str(v) for v in values)
    return str(values)

def family_readable_label(family_name):
    mapping = {
        "root_local_family": "WatchItem rasa",
        "associative_family": "Associativa",
        "containment_family": "Containment"
    }
    return mapping.get(family_name, family_name)

def comparison_question_for_family(family_name):
    mapping = {
        "root_local_family": (
            "Quanto cada política rasa melhora consultas centradas na raiz "
            "e qual o custo em queries afetadas?"
        ),
        "associative_family": (
            "Vale mais embutimento parcial, snapshot ou full embedding "
            "na região associativa WatchItem–Role–Person?"
        ),
        "containment_family": (
            "Containment baseline, reduzido ou completo: qual traz o melhor "
            "trade-off na relação Series–Episode?"
        )
    }
    return mapping.get(family_name, "Comparar variantes estruturais da mesma família.")

# ---------------------------------------------------------
# 3) Matriz agregada por família e scale factor
#    Uma linha = um grupo de comparação
# ---------------------------------------------------------
family_scale_rows = []

group_cols = ["benchmark_family", "scale_label"]

for keys, grp in mongo_experiment_catalog_df.groupby(group_cols):
    benchmark_family, scale_label = keys

    grp = grp.sort_values("config_name").reset_index(drop=True)

    config_names = grp["config_name"].tolist()
    experiment_ids = grp["experiment_id"].tolist()
    mongo_db_names = grp["mongo_db_name"].tolist()
    activated_classes = grp["activated_class"].tolist()

    primary_queries = normalize_list(grp["primary_queries"].tolist())
    secondary_queries = normalize_list(grp["secondary_affected_queries"].tolist())
    control_queries = normalize_list(grp["control_queries"].tolist())

    family_scale_rows.append({
        "benchmark_family": benchmark_family,
        "family_label": family_readable_label(benchmark_family),
        "scale_label": scale_label,
        "n_configurations_in_group": len(config_names),
        "config_names": config_names,
        "experiment_ids": experiment_ids,
        "mongo_db_names": mongo_db_names,
        "activated_classes": activated_classes,
        "primary_queries_union": primary_queries,
        "secondary_affected_queries_union": secondary_queries,
        "control_queries_union": control_queries,
        "n_primary_queries_union": len(primary_queries),
        "n_secondary_affected_queries_union": len(secondary_queries),
        "n_control_queries_union": len(control_queries),
        "comparison_question": comparison_question_for_family(benchmark_family)
    })

family_scale_comparison_df = pd.DataFrame(family_scale_rows).sort_values(
    ["scale_label", "benchmark_family"]
).reset_index(drop=True)

print("Matriz de comparação por família e scale factor:")
display(family_scale_comparison_df)

# ---------------------------------------------------------
# 4) Comparações par-a-par dentro de cada família e escala
#    Útil para análises mais finas de trade-off
# ---------------------------------------------------------
pairwise_rows = []

for _, row in family_scale_comparison_df.iterrows():
    benchmark_family = row["benchmark_family"]
    family_label = row["family_label"]
    scale_label = row["scale_label"]

    config_names = as_list(row["config_names"])
    experiment_ids = as_list(row["experiment_ids"])
    mongo_db_names = as_list(row["mongo_db_names"])
    activated_classes = as_list(row["activated_classes"])

    config_records = list(zip(config_names, experiment_ids, mongo_db_names, activated_classes))

    for (cfg_a, exp_a, db_a, cls_a), (cfg_b, exp_b, db_b, cls_b) in itertools.combinations(config_records, 2):
        pairwise_rows.append({
            "benchmark_family": benchmark_family,
            "family_label": family_label,
            "scale_label": scale_label,
            "config_a": cfg_a,
            "config_b": cfg_b,
            "experiment_id_a": exp_a,
            "experiment_id_b": exp_b,
            "mongo_db_a": db_a,
            "mongo_db_b": db_b,
            "activated_class_a": cls_a,
            "activated_class_b": cls_b,
            "comparison_goal": (
                f"Comparar {cfg_a} versus {cfg_b} dentro da família "
                f"{family_label} em {scale_label}."
            )
        })

pairwise_family_comparison_df = pd.DataFrame(pairwise_rows)

print("\nComparações par-a-par dentro de cada família e scale factor:")
display(pairwise_family_comparison_df)

# ---------------------------------------------------------
# 5) Versão amigável para visualização
# ---------------------------------------------------------
family_scale_comparison_view_df = family_scale_comparison_df.copy()

for col in [
    "config_names",
    "experiment_ids",
    "mongo_db_names",
    "activated_classes",
    "primary_queries_union",
    "secondary_affected_queries_union",
    "control_queries_union"
]:
    family_scale_comparison_view_df[col] = family_scale_comparison_view_df[col].apply(list_to_str)

print("\nVersão amigável da matriz de comparação por família:")
display(family_scale_comparison_view_df)

# ---------------------------------------------------------
# 6) Ordem sugerida de comparação analítica
#    dentro de cada scale factor:
#    1) família root_local
#    2) família associativa
#    3) família containment
# ---------------------------------------------------------
family_order = {
    "root_local_family": 1,
    "associative_family": 2,
    "containment_family": 3
}

scale_order = {
    "sf0.25": 1,
    "sf0.5": 2,
    "sf1": 3
}

comparison_execution_plan_df = family_scale_comparison_df.copy()
comparison_execution_plan_df["scale_priority"] = comparison_execution_plan_df["scale_label"].map(scale_order)
comparison_execution_plan_df["family_priority"] = comparison_execution_plan_df["benchmark_family"].map(family_order)

comparison_execution_plan_df = comparison_execution_plan_df.sort_values(
    ["scale_priority", "family_priority"]
).reset_index(drop=True)

comparison_execution_plan_df["comparison_execution_order"] = range(
    1, len(comparison_execution_plan_df) + 1
)

comparison_execution_plan_summary_df = comparison_execution_plan_df[
    [
        "comparison_execution_order",
        "scale_label",
        "family_label",
        "n_configurations_in_group",
        "n_primary_queries_union",
        "n_secondary_affected_queries_union",
        "n_control_queries_union",
        "comparison_question"
    ]
].copy()

print("\nPlano sugerido de comparação dos resultados por família:")
display(comparison_execution_plan_summary_df)

In [ ]:
# =========================================================
# BLOCK V22D — BENCHMARK EXECUTION TEMPLATE
# one row per:
# experimento × query × query_group × run_phase × repetition
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "mongo_experiment_catalog_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first o bloco V22."
        )

# ---------------------------------------------------------
# 2) Parâmetros globais do benchmark
# ---------------------------------------------------------
BENCHMARK_REPETITIONS = 10

# Fases sugeridas pela metodologia:
# - cold: primeira execução / ambiente frio
# - hot: execução subsequente / cache aquecido
BENCHMARK_PHASES = [
    {"run_phase": "cold", "phase_order": 1},
    {"run_phase": "hot",  "phase_order": 2},
]

benchmark_phases_df = pd.DataFrame(BENCHMARK_PHASES)

# ---------------------------------------------------------
# 3) Helpers
# ---------------------------------------------------------
def as_list(x):
    return x if isinstance(x, list) else []

def normalize_list(values):
    seen = []
    for v in values:
        if isinstance(v, list):
            for x in v:
                if x not in seen:
                    seen.append(x)
        else:
            if v not in seen:
                seen.append(v)
    return seen

def list_to_str(values):
    if isinstance(values, list):
        return ", ".join(str(v) for v in values)
    return str(values)

# ---------------------------------------------------------
# 4) Expandir o catálogo de experimentos em linhas de execução
# ---------------------------------------------------------
benchmark_template_rows = []

for _, exp_row in mongo_experiment_catalog_df.iterrows():
    experiment_id = exp_row["experiment_id"]

    primary_queries = normalize_list(exp_row.get("primary_queries", []))
    secondary_queries = normalize_list(exp_row.get("secondary_affected_queries", []))
    control_queries = normalize_list(exp_row.get("control_queries", []))

    query_groups = [
        ("primary", primary_queries),
        ("secondary_affected", secondary_queries),
        ("control", control_queries),
    ]

    for query_group, query_list in query_groups:
        for query_name in query_list:
            for _, phase_row in benchmark_phases_df.iterrows():
                run_phase = phase_row["run_phase"]
                phase_order = phase_row["phase_order"]

                for repetition in range(1, BENCHMARK_REPETITIONS + 1):
                    benchmark_template_rows.append({
                        # identidade da execução
                        "benchmark_run_id": (
                            f"{experiment_id}__{query_group}__{query_name}__"
                            f"{run_phase}__r{repetition:02d}"
                        ),
                        "experiment_id": experiment_id,
                        "config_name": exp_row["config_name"],
                        "activated_class": exp_row["activated_class"],
                        "benchmark_family": exp_row["benchmark_family"],

                        # escala
                        "scale_factor": exp_row["scale_factor"],
                        "scale_label": exp_row["scale_label"],

                        # banco / configuração física
                        "mongo_db_name": exp_row["mongo_db_name"],
                        "selected_root": exp_row["selected_root"],
                        "primary_collection": exp_row["primary_collection"],
                        "embedding_depth": exp_row["embedding_depth"],
                        "embedding_variant": exp_row["embedding_variant"],
                        "embedded_entities": exp_row["embedded_entities"],
                        "snapshot_entities": exp_row["snapshot_entities"],
                        "referenced_entities": exp_row["referenced_entities"],

                        # query executada
                        "query_name": query_name,
                        "query_group": query_group,   # primary / secondary_affected / control
                        "run_phase": run_phase,       # cold / hot
                        "phase_order": phase_order,
                        "repetition": repetition,

                        # status operacional
                        "execution_status": "planned",
                        "error_message": None,

                        # métricas brutas a preencher depois
                        "start_ts": None,
                        "end_ts": None,
                        "latency_ms": None,
                        "success": None,
                        "documents_returned": None,
                        "documents_written": None,
                        "bytes_read_estimate": None,
                        "bytes_written_estimate": None,

                        # metadados de interpretação
                        "experiment_goal": exp_row["experiment_goal"]
                    })

benchmark_execution_template_df = pd.DataFrame(benchmark_template_rows)

# ---------------------------------------------------------
# 5) Ordenação
# ---------------------------------------------------------
query_group_order = {
    "primary": 1,
    "secondary_affected": 2,
    "control": 3
}

benchmark_execution_template_df["query_group_order"] = (
    benchmark_execution_template_df["query_group"].map(query_group_order)
)

benchmark_execution_template_df = benchmark_execution_template_df.sort_values(
    [
        "scale_factor",
        "config_name",
        "query_group_order",
        "query_name",
        "phase_order",
        "repetition"
    ]
).reset_index(drop=True)

# ---------------------------------------------------------
# 6) Visões resumidas
# ---------------------------------------------------------
benchmark_execution_plan_summary_df = (
    benchmark_execution_template_df
    .groupby(
        [
            "experiment_id",
            "config_name",
            "activated_class",
            "benchmark_family",
            "scale_label",
            "query_group",
            "run_phase"
        ],
        as_index=False
    )
    .agg(
        n_rows=("benchmark_run_id", "count"),
        n_distinct_queries=("query_name", "nunique"),
        min_repetition=("repetition", "min"),
        max_repetition=("repetition", "max")
    )
    .sort_values(
        ["scale_label", "config_name", "query_group", "run_phase"]
    )
    .reset_index(drop=True)
)

benchmark_query_inventory_df = (
    benchmark_execution_template_df
    .groupby(
        [
            "experiment_id",
            "query_group"
        ],
        as_index=False
    )
    .agg(
        query_names=("query_name", lambda s: sorted(s.unique().tolist())),
        n_distinct_queries=("query_name", "nunique")
    )
    .sort_values(["experiment_id", "query_group"])
    .reset_index(drop=True)
)

benchmark_query_inventory_view_df = benchmark_query_inventory_df.copy()
benchmark_query_inventory_view_df["query_names"] = benchmark_query_inventory_view_df["query_names"].apply(list_to_str)

# ---------------------------------------------------------
# 7) Tabela de métricas agregadas (vazia, pronta para uso depois)
#    Esta tabela é o molde do resultado consolidado do benchmark.
# ---------------------------------------------------------
benchmark_result_aggregate_template_df = pd.DataFrame(columns=[
    "experiment_id",
    "config_name",
    "activated_class",
    "benchmark_family",
    "scale_label",
    "query_name",
    "query_group",
    "run_phase",
    "n_runs",
    "n_success_runs",
    "avg_latency_ms",
    "median_latency_ms",
    "p95_latency_ms",
    "p99_latency_ms",
    "min_latency_ms",
    "max_latency_ms",
    "std_latency_ms",
    "avg_documents_returned",
    "avg_documents_written",
    "avg_bytes_read_estimate",
    "avg_bytes_written_estimate"
])

# ---------------------------------------------------------
# 8) Display
# ---------------------------------------------------------
print("Template detalhado da execução do benchmark:")
display(benchmark_execution_template_df)

print("\nSummary do plano de execução por experimento / grupo / fase:")
display(benchmark_execution_plan_summary_df)

print("\nInventário de queries por experimento:")
display(benchmark_query_inventory_view_df)

print("\nTemplate da tabela agregada de resultados:")
display(benchmark_result_aggregate_template_df)

In [ ]:
#salvar todos os resultafos das matrizes geradas em uma pasta
import os
import pandas as pd

pasta_saida = "/home/jovyan/privado/framework evaluation approachs/framework with dataset imdb oficial/backup_variaveis"
os.makedirs(pasta_saida, exist_ok=True)

for nome, valor in list(globals().items()):
    if isinstance(valor, pd.DataFrame):
        caminho = os.path.join(pasta_saida, f"{nome}.csv")
        valor.to_csv(caminho, index=False)
        print(f"Salvo: {caminho}")

## Part B — Optional MongoDB validation / focal benchmark blocks

These blocks are **optional** and are meant for manual validation of a specific MongoDB payload / experiment. They depend on a running MongoDB instance and, in some cases, on variables created outside the core notebook. The original `V26` block was omitted here because it depends on `V25`, which is not present in the provided notebook.

### Optional environment notes

- Example SSH tunnel from the original notebook:

```bash
ssh -N -L 57017:127.0.0.1:27018 hudson@150.162.57.138
```

- Install PyMongo if needed:

```bash
pip install pymongo
```

- Optional connectivity checks for PostgreSQL/Cassandra/MongoDB existed in the original notebook and can be run separately if needed.

In [ ]:
# =========================================================
# BLOCK V27 — MONGODB CONNECTION AND UTILITIES
# =========================================================

from pathlib import Path

try:
    from pymongo import MongoClient
    HAS_PYMONGO = True
except ImportError:
    HAS_PYMONGO = False
    MongoClient = None


# ---------------------------------------------------------
# 1) Função para abrir conexão com MongoDB
# ---------------------------------------------------------
def mongo_connect_basic(host="127.0.0.1", port=57017, username="mongo", password="mongo", auth_source="admin"):
    """
    Abre conexão com MongoDB.

    Ajuste host/port se você estiver usando:
    - Docker com porta diferente
    - túnel SSH
    - autenticação
    """
    if not HAS_PYMONGO:
        raise ImportError(
            "pymongo não está instalado. Rode: pip install pymongo"
        )

    if username and password:
        client = MongoClient(
            host=host,
            port=port,
            username=username,
            password=password,
            authSource=auth_source,
            serverSelectionTimeoutMS=5000
        )
    else:
        client = MongoClient(
            host=host,
            port=port,
            serverSelectionTimeoutMS=5000
        )

    # força teste de conexão
    client.admin.command("ping")
    return client


# ---------------------------------------------------------
# 2) Função para apagar um database inteiro, se existir
# ---------------------------------------------------------
def drop_mongo_database_if_exists(client, db_name: str):
    """
    Remove o database inteiro, se ele existir.
    Útil para recomeçar o experimento limpo.
    """
    existing_dbs = client.list_database_names()

    if db_name in existing_dbs:
        client.drop_database(db_name)
        print(f"Database removido: {db_name}")
    else:
        print(f"Database ainda não existia: {db_name}")


# ---------------------------------------------------------
# 3) Função para contar documentos de todas as coleções
# ---------------------------------------------------------
def verify_mongo_collection_counts(db, collection_names):
    """
    Retorna um DataFrame com a contagem de documentos por coleção.
    """
    rows = []

    for collection_name in collection_names:
        count = db[collection_name].count_documents({})
        rows.append({
            "collection_name": collection_name,
            "document_count": count
        })

    return pd.DataFrame(rows).sort_values("collection_name").reset_index(drop=True)

In [ ]:
# =========================================================
# BLOCK V28 — LOAD THE M1 PAYLOAD INTO MONGODB
# =========================================================

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "mongo_payload_m1",
    "mongo_connect_basic",
    "drop_mongo_database_if_exists",
    "verify_mongo_collection_counts"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Parâmetros de conexão
# Ajuste aqui se sua porta do MongoDB não for 27017
# ---------------------------------------------------------
MONGO_HOST = "127.0.0.1"
MONGO_PORT = 57017
MONGO_USERNAME = "mongo"
MONGO_PASSWORD = "mongo"
MONGO_AUTH_SOURCE = "admin"

# ---------------------------------------------------------
# 3) Conectar
# ---------------------------------------------------------
mongo_client = mongo_connect_basic(
    host=MONGO_HOST,
    port=MONGO_PORT,
    username=MONGO_USERNAME,
    password=MONGO_PASSWORD,
    auth_source=MONGO_AUTH_SOURCE
)

print("Conexão com MongoDB: OK")

# ---------------------------------------------------------
# 4) Select database do experimento
# ---------------------------------------------------------
target_db_name = mongo_payload_m1.mongo_db_name
print("Database alvo:", target_db_name)

# ---------------------------------------------------------
# 5) Apagar database anterior, se existir
# Isso garante repetibilidade do experimento.
# ---------------------------------------------------------
drop_mongo_database_if_exists(mongo_client, target_db_name)

# ---------------------------------------------------------
# 6) Create database e carregar as coleções
# ---------------------------------------------------------
mongo_db = mongo_client[target_db_name]

for collection_name, documents in mongo_payload_m1.collections.items():
    if len(documents) > 0:
        mongo_db[collection_name].insert_many(documents)
        print(f"Coleção carregada: {collection_name} ({len(documents)} documentos)")
    else:
        print(f"Coleção vazia ignorada: {collection_name}")

print("\nCarga MongoDB finalizada.")

In [ ]:
# =========================================================
# BLOCK V29 — CREATE INDEXES FOR M1-FULL
# =========================================================

# ---------------------------------------------------------
# 1) Basic check
# ---------------------------------------------------------
if "mongo_db" not in globals():
    raise NameError(
        "The object 'mongo_db' is not defined. Run first o bloco V28."
    )

# ---------------------------------------------------------
# 2) Índices da coleção principal watchitems
# M1 foi pensado principalmente para Q2 e Q4
# ---------------------------------------------------------

# Busca por título (Q2)
mongo_db["watchitems"].create_index([("title", 1)])

# Filtro por gênero, tipo e ano (Q4)
mongo_db["watchitems"].create_index([
    ("genre_id", 1),
    ("item_type", 1),
    ("release_year", 1)
])

# Campo auxiliar de subtipo
mongo_db["watchitems"].create_index([("document_subtype", 1)])

# ---------------------------------------------------------
# 3) Índices das demais coleções do domínio
# Isso ajuda a manter o banco full mais realista e também
# prepara futuras consultas/benchmarks.
# ---------------------------------------------------------
mongo_db["users"].create_index([("user_id", 1)], unique=True)
mongo_db["persons"].create_index([("person_id", 1)], unique=True)
mongo_db["genres"].create_index([("genre_id", 1)], unique=True)
mongo_db["movies"].create_index([("watchitem_id", 1)], unique=True)
mongo_db["series"].create_index([("watchitem_id", 1)], unique=True)
mongo_db["episodes"].create_index([("watchitem_id", 1)], unique=True)
mongo_db["roles"].create_index([("role_id", 1)], unique=True)
mongo_db["rates"].create_index([("rate_id", 1)], unique=True)

# Índices auxiliares úteis
mongo_db["roles"].create_index([
    ("watchitem_id", 1),
    ("role_type", 1),
    ("person_id", 1)
])

mongo_db["rates"].create_index([
    ("user_id", 1),
    ("watchitem_id", 1)
])

print("Índices do M1-full criados com sucesso.")

In [ ]:
# =========================================================
# BLOCK V30 — VALIDATE THE M1 LOAD
# =========================================================

# ---------------------------------------------------------
# 1) Basic check
# ---------------------------------------------------------
if "mongo_db" not in globals():
    raise NameError(
        "The object 'mongo_db' is not defined. Run first o bloco V28."
    )

# ---------------------------------------------------------
# 2) Verificar contagem por coleção
# ---------------------------------------------------------
m1_counts_df = verify_mongo_collection_counts(
    mongo_db,
    collection_names=mongo_payload_m1.collection_names()
)

print("Contagens no MongoDB para o experimento M1:")
display(m1_counts_df)

# ---------------------------------------------------------
# 3) Verificar um documento de exemplo
# ---------------------------------------------------------
sample_watchitem_doc = mongo_db["watchitems"].find_one({}, {"_id": 0})

print("Exemplo de documento salvo no MongoDB (watchitems):")
display(pd.DataFrame([sample_watchitem_doc]))

In [ ]:
# =========================================================
# BLOCK V31 — TEST QUERY FOR M1
# =========================================================

# ---------------------------------------------------------
# 1) Basic check
# ---------------------------------------------------------
if "mongo_db" not in globals():
    raise NameError(
        "The object 'mongo_db' is not defined. Run first o bloco V28."
    )

# ---------------------------------------------------------
# 2) Teste simples no estilo Q4
# Escolhe um documento qualquer e usa seus próprios valores
# ---------------------------------------------------------
sample_for_q4 = mongo_db["watchitems"].find_one(
    {},
    {
        "_id": 0,
        "genre_id": 1,
        "item_type": 1,
        "release_year": 1
    }
)

if sample_for_q4 is None:
    raise ValueError("Nenhum documento encontrado em watchitems.")

genre_id = sample_for_q4["genre_id"]
item_type = sample_for_q4["item_type"]
center_year = sample_for_q4["release_year"]

year_low = center_year - 10
year_high = center_year + 10

query = {
    "genre_id": genre_id,
    "item_type": item_type,
    "release_year": {
        "$gte": year_low,
        "$lte": year_high
    }
}

q4_result = list(
    mongo_db["watchitems"].find(
        query,
        {
            "_id": 0,
            "watchitem_id": 1,
            "title": 1,
            "release_year": 1,
            "avg_rating": 1,
            "genre": 1,
            "document_subtype": 1
        }
    )
)

print("Parâmetros usados no teste tipo Q4:")
print({
    "genre_id": genre_id,
    "item_type": item_type,
    "year_low": year_low,
    "year_high": year_high
})

print(f"\nNúmero de documentos retornados: {len(q4_result)}")

if len(q4_result) > 0:
    display(pd.DataFrame(q4_result[:10]))

In [ ]:
# =========================================================
# BLOCK V32 — BENCHMARK CONNECTION FOR M1
# =========================================================

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "mongo_connect_basic",
    "active_experiment_row"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Parâmetros de conexão
# Ajuste aqui se seu Mongo estiver em outra porta
# ---------------------------------------------------------
MONGO_HOST = "127.0.0.1"
MONGO_PORT = 57017
MONGO_USERNAME = "mongo"
MONGO_PASSWORD = "mongo"
MONGO_AUTH_SOURCE = "admin"

# ---------------------------------------------------------
# 3) Funções utilitárias para benchmark
# ---------------------------------------------------------
def open_mongo_benchmark_connection(
    host=MONGO_HOST,
    port=MONGO_PORT,
    username=MONGO_USERNAME,
    password=MONGO_PASSWORD,
    auth_source=MONGO_AUTH_SOURCE,
    db_name=None
):
    """
    Abre conexão Mongo e retorna:
    - client
    - db
    """
    client = mongo_connect_basic(
        host=host,
        port=port,
        username=username,
        password=password,
        auth_source=auth_source
    )

    if db_name is None:
        raise ValueError("db_name precisa ser informado.")

    db = client[db_name]
    return client, db


def close_mongo_benchmark_connection(client):
    """
    Fecha a conexão Mongo de forma limpa.
    """
    try:
        if client is not None:
            client.close()
    except Exception:
        pass

# ---------------------------------------------------------
# 4) Abrir uma conexão de teste
# ---------------------------------------------------------
mongo_bench_client, mongo_bench_db = open_mongo_benchmark_connection(
    db_name=active_experiment_row["mongo_db_name"]
)

print("Conexão de benchmark MongoDB aberta com sucesso.")
print("Database:", active_experiment_row["mongo_db_name"])

In [ ]:
# =========================================================
# BLOCK V33 — MONGODB QUERIES FOR M1
# =========================================================

# ---------------------------------------------------------
# Q2 — SimpleSearch no M1
# Search by title directly in the watchitems collection
# ---------------------------------------------------------
def run_q2_simple_search_m1_mongodb(mongo_db, title: str):
    cursor = mongo_db["watchitems"].find(
        {"title": str(title)},
        {
            "_id": 0,
            "watchitem_id": 1,
            "title": 1,
            "release_year": 1,
            "avg_rating": 1,
            "genre": 1,
            "document_subtype": 1,
            "movie_data": 1,
            "series_data": 1,
            "episode_data": 1
        }
    )
    return list(cursor)


# ---------------------------------------------------------
# Q4 — Recommendation no M1
# Filtra diretamente em watchitems por:
# - genre_id
# - item_type
# - release_year em faixa
# ---------------------------------------------------------
def run_q4_recommendation_m1_mongodb(
    mongo_db,
    genre_id: int,
    item_type: str,
    year_low: int,
    year_high: int
):
    query = {
        "genre_id": int(genre_id),
        "item_type": str(item_type),
        "release_year": {
            "$gte": int(year_low),
            "$lte": int(year_high)
        }
    }

    cursor = mongo_db["watchitems"].find(
        query,
        {
            "_id": 0,
            "watchitem_id": 1,
            "title": 1,
            "release_year": 1,
            "avg_rating": 1,
            "genre": 1,
            "document_subtype": 1
        }
    )

    return list(cursor)


# ---------------------------------------------------------
# Teste rápido das funções
# ---------------------------------------------------------
sample_title = mongo_bench_db["watchitems"].find_one({}, {"_id": 0, "title": 1})["title"]

q2_test_result = run_q2_simple_search_m1_mongodb(
    mongo_bench_db,
    title=sample_title
)

print("Teste rápido da Q2 no M1:")
print("Título usado:", sample_title)
print("Número de documentos retornados:", len(q2_test_result))

sample_q4 = mongo_bench_db["watchitems"].find_one(
    {},
    {"_id": 0, "genre_id": 1, "item_type": 1, "release_year": 1}
)

q4_test_result = run_q4_recommendation_m1_mongodb(
    mongo_bench_db,
    genre_id=sample_q4["genre_id"],
    item_type=sample_q4["item_type"],
    year_low=sample_q4["release_year"] - 10,
    year_high=sample_q4["release_year"] + 10
)

print("\nTeste rápido da Q4 no M1:")
print("Parâmetros usados:", sample_q4)
print("Número de documentos retornados:", len(q4_test_result))

In [ ]:
# =========================================================
# BLOCK V34B — MINI-BENCHMARK COM VALIDAÇÃO AUTOMÁTICA
# =========================================================

import time
import random
import pandas as pd

def run_m1_focal_benchmark_validated(
    mongo_db,
    parameter_pool,
    cold_repetitions=3,
    hot_repetitions=20,
    seed=42
):
    rng = random.Random(seed)
    results = []

    def append_result(query_name, phase, run_id, start_time, success, error_message):
        latency_ms = (time.perf_counter() - start_time) * 1000.0
        results.append({
            "experiment_id": active_experiment_row["experiment_id"],
            "mongo_db_name": active_experiment_row["mongo_db_name"],
            "query_name": query_name,
            "benchmark_phase": phase,
            "run_id": run_id,
            "latency_ms": latency_ms,
            "success": success,
            "error_message": error_message
        })

    # -----------------------------
    # COLD RUNS — Q2
    # -----------------------------
    for run_id in range(1, cold_repetitions + 1):
        title = rng.choice(parameter_pool["Q2_SimpleSearch"])

        start = time.perf_counter()
        success = True
        error_message = None

        local_client = None
        local_db = None
        try:
            local_client, local_db = open_mongo_benchmark_connection(
                db_name=active_experiment_row["mongo_db_name"]
            )
            _ = run_q2_simple_search_m1_mongodb(local_db, title)
        except Exception as e:
            success = False
            error_message = str(e)
        finally:
            close_mongo_benchmark_connection(local_client)

        append_result("Q2_SimpleSearch", "cold", run_id, start, success, error_message)

    # -----------------------------
    # COLD RUNS — Q4
    # -----------------------------
    for run_id in range(1, cold_repetitions + 1):
        row = rng.choice(parameter_pool["Q4_Recommendation"])

        start = time.perf_counter()
        success = True
        error_message = None

        local_client = None
        local_db = None
        try:
            local_client, local_db = open_mongo_benchmark_connection(
                db_name=active_experiment_row["mongo_db_name"]
            )
            _ = run_q4_recommendation_m1_mongodb(
                local_db,
                genre_id=row["genre_id"],
                item_type=row["item_type"],
                year_low=row["release_year"] - 10,
                year_high=row["release_year"] + 10
            )
        except Exception as e:
            success = False
            error_message = str(e)
        finally:
            close_mongo_benchmark_connection(local_client)

        append_result("Q4_Recommendation", "cold", run_id, start, success, error_message)

    # -----------------------------
    # HOT RUNS — Q2
    # -----------------------------
    for run_id in range(1, hot_repetitions + 1):
        title = rng.choice(parameter_pool["Q2_SimpleSearch"])

        start = time.perf_counter()
        success = True
        error_message = None

        try:
            _ = run_q2_simple_search_m1_mongodb(mongo_db, title)
        except Exception as e:
            success = False
            error_message = str(e)

        append_result("Q2_SimpleSearch", "hot", run_id, start, success, error_message)

    # -----------------------------
    # HOT RUNS — Q4
    # -----------------------------
    for run_id in range(1, hot_repetitions + 1):
        row = rng.choice(parameter_pool["Q4_Recommendation"])

        start = time.perf_counter()
        success = True
        error_message = None

        try:
            _ = run_q4_recommendation_m1_mongodb(
                mongo_db,
                genre_id=row["genre_id"],
                item_type=row["item_type"],
                year_low=row["release_year"] - 10,
                year_high=row["release_year"] + 10
            )
        except Exception as e:
            success = False
            error_message = str(e)

        append_result("Q4_Recommendation", "hot", run_id, start, success, error_message)

    benchmark_df = pd.DataFrame(results)

    # -----------------------------
    # VALIDAÇÃO AUTOMÁTICA
    # -----------------------------
    expected_counts = {
        ("Q2_SimpleSearch", "cold"): cold_repetitions,
        ("Q2_SimpleSearch", "hot"): hot_repetitions,
        ("Q4_Recommendation", "cold"): cold_repetitions,
        ("Q4_Recommendation", "hot"): hot_repetitions,
    }

    actual_counts = (
        benchmark_df
        .groupby(["query_name", "benchmark_phase"])
        .size()
        .to_dict()
    )

    for key, expected_value in expected_counts.items():
        actual_value = actual_counts.get(key, 0)
        if actual_value != expected_value:
            raise ValueError(
                f"Contagem inesperada para {key}: esperado={expected_value}, obtido={actual_value}"
            )

    return benchmark_df

In [ ]:
# =========================================================
# BLOCK V34A — DIAGNOSTICS AND CORRECT DISPLAY FOR THE MINI-BENCHMARK
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Basic check
# ---------------------------------------------------------
if "m1_benchmark_df" not in globals():
    raise NameError(
        "O DataFrame 'm1_benchmark_df' is not defined. "
        "Rode primeiro o bloco do mini-benchmark."
    )

# ---------------------------------------------------------
# 2) Sort explicitamente o resultado
# Isso ajuda a visualizar tudo de forma consistente
# ---------------------------------------------------------
m1_benchmark_df = m1_benchmark_df.sort_values(
    by=["query_name", "benchmark_phase", "run_id"]
).reset_index(drop=True)

# ---------------------------------------------------------
# 3) Contagem de execuções por query e fase
# Esse é o teste mais importante para saber se Q4 hot existe
# ---------------------------------------------------------
m1_counts_by_query_phase_df = (
    m1_benchmark_df
    .groupby(["query_name", "benchmark_phase"], as_index=False)
    .agg(
        n_runs=("run_id", "count"),
        n_success=("success", lambda s: int(s.sum())),
        n_failures=("success", lambda s: int((~s).sum()))
    )
    .sort_values(["query_name", "benchmark_phase"])
    .reset_index(drop=True)
)

print("Contagem de execuções por query e fase:")
display(m1_counts_by_query_phase_df)

# ---------------------------------------------------------
# 4) Mostrar explicitamente as linhas de Q4 hot
# ---------------------------------------------------------
m1_q4_hot_df = m1_benchmark_df[
    (m1_benchmark_df["query_name"] == "Q4_Recommendation") &
    (m1_benchmark_df["benchmark_phase"] == "hot")
].copy()

print("Linhas de Q4 hot:")
display(m1_q4_hot_df)

# ---------------------------------------------------------
# 5) Summary estatístico simples por query e fase
# ---------------------------------------------------------
m1_summary_df = (
    m1_benchmark_df[m1_benchmark_df["success"] == True]
    .groupby(["query_name", "benchmark_phase"], as_index=False)
    .agg(
        runs=("latency_ms", "count"),
        avg_ms=("latency_ms", "mean"),
        median_ms=("latency_ms", "median"),
        min_ms=("latency_ms", "min"),
        max_ms=("latency_ms", "max")
    )
    .sort_values(["query_name", "benchmark_phase"])
    .reset_index(drop=True)
)

print("Summary estatístico do mini-benchmark:")
display(m1_summary_df)

# ---------------------------------------------------------
# 6) Mostrar tudo, se quiser inspecionar linha a linha
# ---------------------------------------------------------
print("Resultados brutos completos do mini-benchmark:")
display(m1_benchmark_df)

# ---------------------------------------------------------
# 7) Falhas, se existirem
# ---------------------------------------------------------
m1_failures_df = m1_benchmark_df[m1_benchmark_df["success"] == False].copy()

print("Falhas, se existirem:")
display(m1_failures_df)

In [ ]:
# =========================================================
# BLOCK V35 — EXPORT BENCHMARK RESULTS PER EXPERIMENT
# =========================================================

from pathlib import Path
import pandas as pd

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "m1_benchmark_df",
    "active_experiment_row"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Função para resumir estatísticas básicas
# ---------------------------------------------------------
def summarize_benchmark_basic(result_df: pd.DataFrame) -> pd.DataFrame:
    """
    Gera um resumo simples por query e fase.
    """
    if result_df.empty:
        return pd.DataFrame()

    ok_df = result_df[result_df["success"] == True].copy()

    if ok_df.empty:
        return pd.DataFrame()

    summary_df = (
        ok_df
        .groupby(["query_name", "benchmark_phase"], as_index=False)
        .agg(
            runs=("latency_ms", "count"),
            avg_ms=("latency_ms", "mean"),
            median_ms=("latency_ms", "median"),
            min_ms=("latency_ms", "min"),
            max_ms=("latency_ms", "max"),
            std_ms=("latency_ms", "std")
        )
        .sort_values(["query_name", "benchmark_phase"])
        .reset_index(drop=True)
    )

    summary_df["std_ms"] = summary_df["std_ms"].fillna(0.0)
    return summary_df


# ---------------------------------------------------------
# 3) Função para contar execuções por query e fase
# ---------------------------------------------------------
def build_counts_by_query_phase(result_df: pd.DataFrame) -> pd.DataFrame:
    """
    Conta quantas execuções ocorreram por query e fase,
    incluindo sucessos e falhas.
    """
    if result_df.empty:
        return pd.DataFrame()

    counts_df = (
        result_df
        .groupby(["query_name", "benchmark_phase"], as_index=False)
        .agg(
            n_runs=("run_id", "count"),
            n_success=("success", lambda s: int(s.sum())),
            n_failures=("success", lambda s: int((~s).sum()))
        )
        .sort_values(["query_name", "benchmark_phase"])
        .reset_index(drop=True)
    )

    return counts_df


# ---------------------------------------------------------
# 4) Define diretório do experimento
# Cada experimento ganha sua própria pasta
# ---------------------------------------------------------
experiment_id = active_experiment_row["experiment_id"]

experiment_results_dir = Path("exports/mongodb_benchmarks") / experiment_id
experiment_results_dir.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 5) Preparar DataFrames para exportação
# ---------------------------------------------------------
benchmark_raw_df = m1_benchmark_df.copy().sort_values(
    by=["query_name", "benchmark_phase", "run_id"]
).reset_index(drop=True)

benchmark_summary_df = summarize_benchmark_basic(benchmark_raw_df)

benchmark_failures_df = benchmark_raw_df[
    benchmark_raw_df["success"] == False
].copy().reset_index(drop=True)

benchmark_counts_df = build_counts_by_query_phase(benchmark_raw_df)

# ---------------------------------------------------------
# 6) Define nomes dos arquivos com prefixo do experimento
# ---------------------------------------------------------
raw_csv_path = experiment_results_dir / f"{experiment_id}_raw.csv"
summary_csv_path = experiment_results_dir / f"{experiment_id}_summary.csv"
failures_csv_path = experiment_results_dir / f"{experiment_id}_failures.csv"
counts_csv_path = experiment_results_dir / f"{experiment_id}_counts_by_query_phase.csv"

# ---------------------------------------------------------
# 7) Salvar os CSVs
# ---------------------------------------------------------
benchmark_raw_df.to_csv(raw_csv_path, index=False)
benchmark_summary_df.to_csv(summary_csv_path, index=False)
benchmark_failures_df.to_csv(failures_csv_path, index=False)
benchmark_counts_df.to_csv(counts_csv_path, index=False)

# ---------------------------------------------------------
# 8) Mostrar os caminhos gerados
# ---------------------------------------------------------
print("Arquivos CSV do benchmark exportados com sucesso:")
print(raw_csv_path)
print(summary_csv_path)
print(failures_csv_path)
print(counts_csv_path)

# ---------------------------------------------------------
# 9) Mostrar prévia dos dados exportados
# ---------------------------------------------------------
print("\nSummary estatístico exportado:")
display(benchmark_summary_df)

print("Contagem por query e fase exportada:")
display(benchmark_counts_df)

print("Falhas exportadas:")
display(benchmark_failures_df)